# Shift-Scan: Research Notebook and State-Passing Validation

**Clean research release** · Mamba-2 130M · FineWeb-Edu · Ternary projections and dyadic decay

This release preserves successful experiment cells and saved outputs, removes cells whose recorded execution ended in an error, and adds one consolidated held-out integration check.

## Execution status and scope

- Historical successful outputs are retained as recorded evidence; they are not a claim that every cell has been rerun in a fresh environment.
- The final integration check requires the original Kaggle CUDA/Triton environment, saved 20-capture validation dataset, and the setup cells that define `SCALE`, `build_reference_and_modes`, and `shift_scan_state_passing_i32_kernel`.
- The final integration check is intentionally unexecuted in this release. Run it in that environment before reporting a pass.
- This validation covers the Mamba-2 state-passing recurrence, not every multiplication or the complete fused SSD scan.

## Clean-release policy

Failed exploratory executions were removed rather than hidden by clearing their error outputs. Successful experiment outputs and analysis cells remain unchanged.

In [4]:
import importlib.util
import torch
import sys

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

spec = importlib.util.find_spec("mamba_ssm")
print("mamba_ssm installed:", spec is not None)

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
CUDA: 13.0
mamba_ssm installed: False


In [6]:
import os
import importlib.util

for key in [
    "MAMBA_FORCE_BUILD",
    "MAMBA_KEEP_CUDA_BUILD",
    "MAMBA_SKIP_CUDA_BUILD",
]:
    print(key, "=", repr(os.environ.get(key)))

print("mamba_ssm installed:", importlib.util.find_spec("mamba_ssm") is not None)

MAMBA_FORCE_BUILD = None
MAMBA_KEEP_CUDA_BUILD = None
MAMBA_SKIP_CUDA_BUILD = None
mamba_ssm installed: False


In [7]:
import os
import sys
import subprocess
import importlib.util

env = os.environ.copy()

# These are the important flags for mamba-ssm v2.3.2.post1
env["MAMBA_SKIP_CUDA_BUILD"] = "TRUE"
env["MAMBA_FORCE_BUILD"] = "TRUE"

# Do not opt into any CUDA extension build
env.pop("MAMBA_KEEP_CUDA_BUILD", None)

cmd = [
    sys.executable, "-m", "pip", "install",
    "-v",
    "--no-build-isolation",
    "--no-cache-dir",
    "mamba-ssm==2.3.2.post1"
]

print("Installing Mamba core package without selective_scan_cuda...")
result = subprocess.run(cmd, env=env)

print("\nInstallation exit code:", result.returncode)

if result.returncode == 0:
    print("mamba_ssm installed:", importlib.util.find_spec("mamba_ssm") is not None)
else:
    print("Installation failed. Share the final 40–60 lines of the error.")

Installing Mamba core package without selective_scan_cuda...
Using pip 26.2.2 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)
  Preparing metadata (pyproject.toml): started


  Running command Preparing metadata (pyproject.toml)
  /usr/local/lib/python3.13/dist-packages/wheel/bdist_wheel.py:4: FutureWarning: The 'wheel' package is no longer the canonical location of the 'bdist_wheel' command, and will be removed in a future release. Please update to setuptools v70.1 or later which contains an integrated version of this command.
    warn(
  /usr/local/lib/python3.13/dist-packages/setuptools/config/_apply_pyprojecttoml.py:82: SetuptoolsDeprecationWarning: `project.license` as a TOML table is deprecated
  !!

          ********************************************************************************
          Please use a simple string containing a SPDX expression for `project.license`. You can also use `project.license-files`. (Both options available on setuptools>=77.0.0).

          By 2027-Feb-18, you need to update your project and remove deprecated calls
          or your builds will no longer be supported.

          See https://packaging.python.org/en/l

  Preparing metadata (pyproject.toml): finished with status 'done'
  Obtaining dependency information for tilelang==0.1.8 from https://files.pythonhosted.org/packages/5d/0b/96ba853aa9e4795020d183e0ca832e9e37d82d4f7f48896241323d1b5ece/tilelang-0.1.8-cp38-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata
  Obtaining dependency information for apache-tvm-ffi<=0.1.9 from https://files.pythonhosted.org/packages/70/ef/5402da5d37f5270fd88ea0348acca78dba9be8bdbf6c2bcae0935eb03ef1/apache_tvm_ffi-0.1.9-cp312-abi3-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata
  Obtaining dependency information for quack-kernels>=0.3.4 from https://files.pythonhosted.org/packages/73/d8/f5e56c38b286308d115a171e3c5c44f24e9d03980c3cfd22588df40261fa/quack_kernels-0.6.5-py3-none-any.whl.metadata
  Obtaining dependency information for torch-c-dlpack-ext from https://files.pythonhosted.org/packages/2d/68/f434b48700f3e04f33882f54d8d3910327b935f55e14ec49da7d607bf470/torch_c_dlpack_ext-0.1.5-cp313-cp3

  Running command Building wheel for mamba-ssm (pyproject.toml)
  /usr/local/lib/python3.13/dist-packages/wheel/bdist_wheel.py:4: FutureWarning: The 'wheel' package is no longer the canonical location of the 'bdist_wheel' command, and will be removed in a future release. Please update to setuptools v70.1 or later which contains an integrated version of this command.
    warn(
  /usr/local/lib/python3.13/dist-packages/setuptools/config/_apply_pyprojecttoml.py:82: SetuptoolsDeprecationWarning: `project.license` as a TOML table is deprecated
  !!

          ********************************************************************************
          Please use a simple string containing a SPDX expression for `project.license`. You can also use `project.license-files`. (Both options available on setuptools>=77.0.0).

          By 2027-Feb-18, you need to update your project and remove deprecated calls
          or your builds will no longer be supported.

          See https://packaging.pytho

  Created wheel for mamba-ssm: filename=mamba_ssm-2.3.2.post1-py3-none-any.whl size=211266 sha256=b08e3c8cf3019990521bac9e67c70b431bb100fb2e15d6032f343c7f9447a392
  Stored in directory: /tmp/pip-ephem-wheel-cache-v9zlskyl/wheels/7f/68/c4/588f5c6c8950032840738453efc028cde9be35603c72d662ca
Successfully built mamba-ssm
  changing mode of /usr/local/bin/tvm-ffi-config to 755
  changing mode of /usr/local/bin/tvm-ffi-stubgen to 755
  changing mode of /usr/local/bin/run-iket to 755


Installation exit code: 0
mamba_ssm installed: True


In [8]:
import sys
import types

# Mamba-1 CUDA extension is not installed.
# Mamba-2 uses its separate Triton SSD scan path.
selective_scan_stub = types.ModuleType("selective_scan_cuda")

def unavailable_mamba1_extension(*args, **kwargs):
    raise RuntimeError(
        "The Mamba-1 selective_scan_cuda extension is unavailable. "
        "This experiment must use the Mamba-2 Triton SSD path."
    )

def stub_getattr(name):
    if name.startswith("__"):
        raise AttributeError(name)
    return unavailable_mamba1_extension

selective_scan_stub.__getattr__ = stub_getattr
sys.modules["selective_scan_cuda"] = selective_scan_stub

print("Guarded Mamba-1 import placeholder registered.")

Guarded Mamba-1 import placeholder registered.


In [9]:
import torch
import triton
import triton.language as tl

@triton.jit
def double_kernel(X, Y, N: tl.constexpr):
    i = tl.program_id(0) * 256 + tl.arange(0, 256)
    x = tl.load(X + i, i < N, 0)
    tl.store(Y + i, 2.0 * x, i < N)

x = torch.randn(1024, device="cuda")
y = torch.empty_like(x)

double_kernel[(4,)](x, y, 1024)
torch.cuda.synchronize()

print("Triton:", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("Triton GPU kernel: PASS")
print("Maximum error:", (y - 2*x).abs().max().item())

Triton: 3.6.0
GPU: Tesla T4
Triton GPU kernel: PASS
Maximum error: 0.0


In [10]:
import inspect
import mamba_ssm.modules.mamba2 as mamba2_module

from mamba_ssm.modules.mamba2 import Mamba2
from mamba_ssm.ops.triton.ssd_combined import mamba_chunk_scan_combined

print("=" * 70)
print("1. MAMBA-2 FORWARD METHOD")
print("=" * 70)

forward_src = inspect.getsource(Mamba2.forward)
print(forward_src[:14000])

print("\n" + "=" * 70)
print("2. MAMBA-2 INITIALIZATION")
print("=" * 70)

init_src = inspect.getsource(Mamba2.__init__)
print(init_src[:10000])

print("\n" + "=" * 70)
print("3. SCAN FUNCTION LOCATION")
print("=" * 70)

print(inspect.getfile(mamba_chunk_scan_combined))
print("Scan signature:")
print(inspect.signature(mamba_chunk_scan_combined))

print("\n" + "=" * 70)
print("4. IMPORTANT MODULE SETTINGS")
print("=" * 70)

print("Mamba2 module:", mamba2_module.__file__)
print("Forward source available:", bool(forward_src))

1. MAMBA-2 FORWARD METHOD
    def forward(self, u, seqlen=None, seq_idx=None, cu_seqlens=None, inference_params=None):
        """
        u: (batch, seqlen, hidden_dim) if seqlen=None.
            If seqlen is not None, u is (batch * seqlen, hidden_dim). This is so that when we
            split u during sequence parallel, we split the batch * seqlen dimension
            (in case batch is small).
        Returns: same shape as u
        """
        seqlen_og = seqlen
        if seqlen is None:
            batch, seqlen, dim = u.shape
        else:
            batch_seqlen, dim = u.shape
            batch = batch_seqlen // seqlen

        conv_state, ssm_state = None, None
        if inference_params is not None:
            inference_batch = cu_seqlens.shape[0] - 1 if cu_seqlens is not None else batch
            conv_state, ssm_state = self._get_states_from_cache(inference_params, inference_batch)
            if inference_params.seqlen_offset > 0:
                # The states are up

In [11]:
import sys
import types
import importlib.machinery
import torch

# Register a safe placeholder only if the CUDA extension is absent.
existing = sys.modules.get("selective_scan_cuda")

if existing is None:
    stub = types.ModuleType("selective_scan_cuda")
    stub.__spec__ = importlib.machinery.ModuleSpec(
        "selective_scan_cuda", loader=None
    )

    def unavailable_extension(*args, **kwargs):
        raise RuntimeError(
            "Mamba-1 CUDA extension is unavailable. "
            "This experiment requires the Mamba-2 Triton scan."
        )

    def stub_getattr(name):
        if name.startswith("__"):
            raise AttributeError(name)
        return unavailable_extension

    stub.__getattr__ = stub_getattr
    sys.modules["selective_scan_cuda"] = stub

else:
    # Repair the placeholder created by the previous cell.
    if getattr(existing, "__spec__", None) is None:
        existing.__spec__ = importlib.machinery.ModuleSpec(
            "selective_scan_cuda", loader=None
        )

from mamba_ssm.models.mixer_seq_simple import MambaLMHeadModel

MODEL_ID = "state-spaces/mamba2-130m"

model = MambaLMHeadModel.from_pretrained(
    MODEL_ID,
    device="cuda",
    dtype=torch.float16
)

model.eval()

for layer in model.backbone.layers:
    if hasattr(layer.mixer, "use_mem_eff_path"):
        layer.mixer.use_mem_eff_path = False

print("Model loaded successfully")
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Layers:", len(model.backbone.layers))
print("Device:", next(model.parameters()).device)
print("Dtype:", next(model.parameters()).dtype)

config.json:   0%|          | 0.00/330 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  258MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Model loaded successfully
Parameters: 128989632
Layers: 24
Device: cuda:0
Dtype: torch.float16


In [12]:
import torch
import time

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

print("\nGPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 3), "GB")

print("GPU memory reserved:",
      round(torch.cuda.memory_reserved() / 1024**3, 3), "GB")

print("\nModel layers:", len(model.backbone.layers))

for i, layer in enumerate(model.backbone.layers[:3]):
    m = layer.mixer
    print(
        f"Layer {i}: "
        f"use_mem_eff_path={m.use_mem_eff_path}, "
        f"chunk_size={m.chunk_size}, "
        f"heads={m.nheads}"
    )

CUDA available: True
GPU: Tesla T4

GPU memory allocated: 0.245 GB
GPU memory reserved: 0.322 GB

Model layers: 24
Layer 0: use_mem_eff_path=False, chunk_size=256, heads=24
Layer 1: use_mem_eff_path=False, chunk_size=256, heads=24
Layer 2: use_mem_eff_path=False, chunk_size=256, heads=24


In [13]:
import inspect
from mamba_ssm.ops.triton.ssd_combined import mamba_chunk_scan_combined

print("Scan source:", inspect.getfile(mamba_chunk_scan_combined))
print("CUDA synchronized:", torch.cuda.is_available())
print("Triton scan function:", mamba_chunk_scan_combined)

Scan source: /usr/local/lib/python3.13/dist-packages/mamba_ssm/ops/triton/ssd_combined.py
CUDA synchronized: True
Triton scan function: <function mamba_chunk_scan_combined at 0x79be5c459c60>


In [14]:
for layer in model.backbone.layers:
    layer.mixer.use_mem_eff_path = True

print("Fused Mamba-2 path restored.")

for i, layer in enumerate(model.backbone.layers[:3]):
    print(i, layer.mixer.use_mem_eff_path)

Fused Mamba-2 path restored.
0 True
1 True
2 True


In [15]:
import mamba_ssm.modules.mamba2 as m2

print("causal_conv1d_fn:", m2.causal_conv1d_fn)
print("causal_conv1d available:", m2.causal_conv1d_fn is not None)
print("Triton scan available:", m2.mamba_chunk_scan_combined is not None)

causal_conv1d_fn: None
causal_conv1d available: False
Triton scan available: True


In [16]:
import inspect
from triton.runtime.autotuner import Autotuner

import mamba_ssm.ops.triton.ssd_chunk_scan as scan_mod
import mamba_ssm.ops.triton.ssd_chunk_state as state_mod
import mamba_ssm.ops.triton.ssd_state_passing as passing_mod
import mamba_ssm.ops.triton.ssd_combined as combined_mod

modules = [
    scan_mod,
    state_mod,
    passing_mod,
    combined_mod,
]

print("TRITON AUTOTUNER CONFIGURATION AUDIT")
print("=" * 60)

for module in modules:
    print("\nModule:", module.__name__)

    for name, obj in vars(module).items():
        if isinstance(obj, Autotuner):
            print(
                f"{name}: "
                f"{len(obj.configs)} candidate configurations"
            )

TRITON AUTOTUNER CONFIGURATION AUDIT

Module: mamba_ssm.ops.triton.ssd_chunk_scan
_chunk_scan_fwd_kernel: 11 candidate configurations
_chunk_scan_fwd_kernel_wip: 4 candidate configurations
_chunk_scan_bwd_dz_kernel: 4 candidate configurations
_chunk_scan_bwd_dstates_kernel: 9 candidate configurations
_chunk_scan_bwd_dc_kernel: 8 candidate configurations
_chunk_scan_bwd_dx_kernel: 9 candidate configurations
_chunk_scan_bwd_dcb_kernel: 6 candidate configurations
_chunk_scan_bwd_ddAcs_unstable_kernel: 4 candidate configurations
_chunk_scan_bwd_ddAcs_stable_kernel_old: 8 candidate configurations
_chunk_scan_bwd_ddAcs_stable_kernel: 6 candidate configurations
_chunk_scan_bwd_ddAcs_prev_kernel: 6 candidate configurations

Module: mamba_ssm.ops.triton.ssd_chunk_state
_chunk_cumsum_fwd_kernel: 7 candidate configurations
_chunk_cumsum_bwd_kernel: 7 candidate configurations
_chunk_state_fwd_kernel: 9 candidate configurations
_chunk_state_bwd_dx_kernel: 9 candidate configurations
_chunk_state_bwd

In [17]:
from triton.runtime.autotuner import Autotuner

import mamba_ssm.ops.triton.ssd_chunk_scan as scan_mod
import mamba_ssm.ops.triton.ssd_chunk_state as state_mod
import mamba_ssm.ops.triton.ssd_state_passing as passing_mod
import mamba_ssm.ops.triton.ssd_combined as combined_mod

forward_kernels = {
    "chunk_scan": [
        (scan_mod, "_chunk_scan_fwd_kernel"),
    ],
    "chunk_state": [
        (state_mod, "_chunk_cumsum_fwd_kernel"),
        (state_mod, "_chunk_state_fwd_kernel"),
    ],
    "state_passing": [
        (passing_mod, "_state_passing_fwd_kernel"),
    ],
    "combined": [
        (combined_mod, "_chunk_scan_chunk_state_bwd_dx_kernel"),
    ],
}

for group, entries in forward_kernels.items():
    print("\n" + "=" * 65)
    print(group.upper())

    for module, name in entries:
        kernel = getattr(module, name, None)

        if isinstance(kernel, Autotuner):
            print(f"\n{name}:")
            for i, config in enumerate(kernel.configs):
                print(
                    f"  [{i}] "
                    f"kwargs={config.kwargs}, "
                    f"warps={config.num_warps}, "
                    f"stages={config.num_stages}"
                )
        else:
            print(f"\n{name}: not found as an Autotuner")


CHUNK_SCAN

_chunk_scan_fwd_kernel:
  [0] kwargs={'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 256, 'BLOCK_SIZE_K': 64}, warps=8, stages=3
  [1] kwargs={'BLOCK_SIZE_M': 64, 'BLOCK_SIZE_N': 256, 'BLOCK_SIZE_K': 32}, warps=4, stages=4
  [2] kwargs={'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 128, 'BLOCK_SIZE_K': 32}, warps=4, stages=4
  [3] kwargs={'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 64, 'BLOCK_SIZE_K': 32}, warps=4, stages=4
  [4] kwargs={'BLOCK_SIZE_M': 64, 'BLOCK_SIZE_N': 128, 'BLOCK_SIZE_K': 32}, warps=4, stages=4
  [5] kwargs={'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 64, 'BLOCK_SIZE_K': 64}, warps=4, stages=4
  [6] kwargs={'BLOCK_SIZE_M': 64, 'BLOCK_SIZE_N': 128, 'BLOCK_SIZE_K': 64}, warps=4, stages=4
  [7] kwargs={'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 32, 'BLOCK_SIZE_K': 32}, warps=4, stages=4
  [8] kwargs={'BLOCK_SIZE_M': 64, 'BLOCK_SIZE_N': 32, 'BLOCK_SIZE_K': 32}, warps=2, stages=5
  [9] kwargs={'BLOCK_SIZE_M': 32, 'BLOCK_SIZE_N': 64, 'BLOCK_SIZE_K': 32}, warps=2, stages=5
  [10] kwargs={'BLOCK_S

In [20]:
import torch
import time

mixer = model.backbone.layers[0].mixer
mixer.use_mem_eff_path = False

x = torch.randn(
    1, 16, mixer.d_model,
    device="cuda",
    dtype=next(mixer.parameters()).dtype
)

print("Starting one-layer Mamba-2 test...")

try:
    torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.inference_mode():
        y = mixer(x)

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    print("\nONE-LAYER TEST: PASS")
    print("Elapsed:", round(elapsed, 3), "seconds")
    print("Input shape:", tuple(x.shape))
    print("Output shape:", tuple(y.shape))
    print("Finite:", bool(torch.isfinite(y).all()))

except Exception as e:
    print("\nONE-LAYER TEST: FAILED")
    print(type(e).__name__, str(e))

Starting one-layer Mamba-2 test...

ONE-LAYER TEST: FAILED
OutOfResources out of resource: shared memory, Required: 131072, Hardware limit: 65536. Reducing block sizes or `num_stages` may help.


In [22]:
import torch
import time

mixer = model.backbone.layers[0].mixer

x = torch.randn(
    1, 16, mixer.d_model,
    device="cuda",
    dtype=next(mixer.parameters()).dtype
)

try:
    torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.inference_mode():
        y = mixer(x)

    torch.cuda.synchronize()

    print("\nONE-LAYER TEST: PASS")
    print("Elapsed:", round(time.perf_counter() - start, 3), "seconds")
    print("Output shape:", tuple(y.shape))
    print("Finite:", bool(torch.isfinite(y).all()))

except Exception as e:
    print("\nONE-LAYER TEST: FAILED")
    print(type(e).__name__, str(e))


ONE-LAYER TEST: FAILED
OutOfResources out of resource: shared memory, Required: 131072, Hardware limit: 65536. Reducing block sizes or `num_stages` may help.


In [ ]:
for (module_name, kernel_name), configs in AUTOTUNE_BACKUP.items():
    module = __import__(module_name, fromlist=[kernel_name])
    getattr(module, kernel_name).configs = configs

print("Original autotuning configurations restored.")

In [ ]:
dt = captured["dt"].float()
A = captured["A"].float()
dt_bias = captured["dt_bias"].float()

dt_effective = torch.nn.functional.softplus(dt + dt_bias)

if captured["dt_limit"] is not None:
    dt_effective = dt_effective.clamp(
        min=captured["dt_limit"][0],
        max=captured["dt_limit"][1]
    )

decay = torch.exp(dt_effective * A)

print("ACTUAL MAMBA-2 DECAY AUDIT")
print("=" * 50)
print("Shape:", tuple(decay.shape))
print("Minimum:", decay.min().item())
print("Maximum:", decay.max().item())
print("Mean:", decay.mean().item())
print("Std:", decay.std().item())
print("Values below 0.5:", (decay < 0.5).float().mean().item())
print("Values above 0.99:", (decay > 0.99).float().mean().item())
print("All finite:", torch.isfinite(decay).all().item())

In [23]:
import numpy as np

# Reconstruct all valid exponent pairs.
# Each exponent is in [2, 14], matching the earlier Dyadic2 design.
DYADIC_PAIRS = [
    (k1, k2)
    for k1 in range(2, 15)
    for k2 in range(k1, 15)
]

DYADIC_GRID = [
    (2.0**(-k1) + 2.0**(-k2), k1, k2)
    for k1, k2 in DYADIC_PAIRS
]

grid_d = np.array([v[0] for v in DYADIC_GRID])
grid_a = 1.0 - grid_d

print("Grid entries:", len(grid_a))
print("Minimum decay:", grid_a.min())
print("Maximum decay:", grid_a.max())
print("Repeated exponent pairs:", sum(k1 == k2 for _, k1, k2 in DYADIC_GRID))

Grid entries: 91
Minimum decay: 0.5
Maximum decay: 0.9998779296875
Repeated exponent pairs: 13


In [26]:
import os

roots = [
    "/kaggle/input",
    "/kaggle/working"
]

for root in roots:
    print("\nDIRECTORY:", root)

    if not os.path.exists(root):
        continue

    for current, dirs, files in os.walk(root):
        depth = current.replace(root, "").count(os.sep)

        if depth > 2:
            dirs[:] = []
            continue

        for filename in files[:30]:
            print(os.path.join(current, filename))


DIRECTORY: /kaggle/input

DIRECTORY: /kaggle/working
/kaggle/working/.virtual_documents/__notebook_source__.ipynb


In [27]:
import torch
import importlib.util
import os

print("GPU:", torch.cuda.get_device_name(0))
print("Model loaded:", "model" in globals())
print("Model parameters:", sum(p.numel() for p in model.parameters()))

print("\nRequired packages:")
for package in ["datasets", "transformers", "huggingface_hub"]:
    print(package, importlib.util.find_spec(package) is not None)

print("\nKaggle input directories:")
print(os.listdir("/kaggle/input"))

GPU: Tesla T4
Model loaded: True
Model parameters: 128989632

Required packages:
datasets True
transformers True
huggingface_hub True

Kaggle input directories:
['datasets']


In [29]:
import torch

embedding_size = model.backbone.embedding.num_embeddings
output_size = model.lm_head.out_features

test = tokenizer(
    "The autumn sky reflected quietly across the lake.",
    return_tensors="pt"
)

max_token_id = int(test.input_ids.max())

print("Tokenizer vocab_size:", tokenizer.vocab_size)
print("Tokenizer len:", len(tokenizer))
print("Model embedding size:", embedding_size)
print("Model output size:", output_size)
print("Maximum test token ID:", max_token_id)

assert max_token_id < embedding_size, "Token ID exceeds embedding range!"

print("\nTOKEN ID RANGE CHECK: PASS")

Tokenizer vocab_size: 50254
Tokenizer len: 50277
Model embedding size: 50288
Model output size: 50288
Maximum test token ID: 25986

TOKEN ID RANGE CHECK: PASS


In [30]:
from datasets import load_dataset

print("Loading FineWeb-Edu streaming sample...")

dataset = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="sample-10BT",
    split="train",
    streaming=True
)

samples = []

for item in dataset.take(10):
    text = item.get("text", "").strip()
    if len(text) > 100:
        samples.append(text)

print("Documents loaded:", len(samples))

for i, sample in enumerate(samples[:3]):
    print(f"\nDOCUMENT {i+1}:")
    print(sample[:300])

Loading FineWeb-Edu streaming sample...


README.md:   0%|          | 0.00/26.4k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Documents loaded: 10

DOCUMENT 1:
The Independent Jane
For all the love, romance and scandal in Jane Austen’s books, what they are really about is freedom and independence. Independence of thought and the freedom to choose.
Elizabeth’s refusal of Mr. Collins offer of marriage showed an independence seldom seen in heroines of the day

DOCUMENT 2:
Taking Play Seriously
By ROBIN MARANTZ HENIG
Published: February 17, 2008
On a drizzly Tuesday night in late January, 200 people came out to hear a psychiatrist talk rhapsodically about play -- not just the intense, joyous play of children, but play for all people, at all ages, at all times. (All sp

DOCUMENT 3:
How do you get HIV?
HIV can be passed on when infected bodily fluid, such as blood or semen, is passed into an uninfected person. Semen is the liquid which is released from a man's penis during sex which carries sperm. It can be infected with HIV or AIDS when someone is HIV positive or is carrying t


In [31]:
import torch

encoded_samples = []

for sample in samples:
    tokens = tokenizer(
        sample,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    assert int(tokens.input_ids.max()) < embedding_size

    encoded_samples.append(tokens.input_ids)

print("Tokenized documents:", len(encoded_samples))
print("First sequence shape:", encoded_samples[0].shape)

print("\nFINEWEB TOKENIZATION: PASS")

Tokenized documents: 10
First sequence shape: torch.Size([1, 128])

FINEWEB TOKENIZATION: PASS


In [38]:
import inspect
import mamba_ssm.ops.triton.ssd_combined as combined
import mamba_ssm.ops.triton.ssd_chunk_scan as chunk_scan
import mamba_ssm.ops.triton.ssd_chunk_state as chunk_state
import mamba_ssm.ops.triton.ssd_state_passing as state_passing

modules = [
    ("combined", combined),
    ("chunk_scan", chunk_scan),
    ("chunk_state", chunk_state),
    ("state_passing", state_passing),
]

for module_name, module in modules:

    print("\n" + "=" * 70)
    print("MODULE:", module_name)
    print("=" * 70)

    for name, obj in vars(module).items():

        if callable(obj) and (
            "kernel" in name.lower()
            or "scan" in name.lower()
            or "state" in name.lower()
            or "cumsum" in name.lower()
        ):
            print(name, "|", type(obj).__name__)


MODULE: combined
_chunk_cumsum_fwd | function
_chunk_cumsum_bwd | function
_chunk_state_fwd | function
_chunk_state_bwd_db | function
_chunk_state_bwd_ddAcs_stable | function
chunk_state | function
chunk_state_ref | function
chunk_state_varlen | function
_state_passing_fwd | function
_state_passing_bwd | function
state_passing | function
state_passing_ref | function
_chunk_scan_fwd | function
_chunk_scan_bwd_dz | function
_chunk_scan_bwd_dstates | function
_chunk_scan_bwd_dC | function
_chunk_scan_bwd_dcb | function
_chunk_scan_bwd_ddAcs_stable | function
chunk_scan | function
chunk_scan_ref | function
_chunk_scan_bwd_ddAcs_prev | function
_chunk_scan_chunk_state_bwd_dx | function
_mamba_chunk_scan_combined_fwd | function
_mamba_chunk_scan_combined_bwd | function
selective_scan_bwd | function
MambaChunkScanCombinedFn | FunctionMeta
mamba_chunk_scan_combined | function
mamba_chunk_scan | function
ssd_chunk_scan_combined_ref | function
ssd_selective_scan | function
mamba_conv1d_scan_ref

In [39]:
for module_name, module in modules:

    print("\n" + "=" * 70)
    print("MODULE:", module_name)
    print("=" * 70)

    for name, obj in vars(module).items():

        if not callable(obj):
            continue

        if not any(
            word in name.lower()
            for word in ["fwd", "forward", "scan", "state", "cumsum"]
        ):
            continue

        try:
            source = inspect.getsource(obj)

            print("\nKERNEL/FUNCTION:", name)
            print(source[:5000])

        except (OSError, TypeError):
            continue


MODULE: combined

KERNEL/FUNCTION: custom_fwd
    def decorator(*args, **kwargs):
        if cuda_amp_deprecated:
            kwargs["device_type"] = "cuda"
        return dec(*args, **kwargs)


KERNEL/FUNCTION: _bmm_chunk_fwd
def _bmm_chunk_fwd(a, b, chunk_size, seq_idx=None, causal=False, output_dtype=None):
    """
    Argument:
        a: (batch, seqlen, k) or (batch, seqlen, ngroups, k)
        b: (batch, seqlen, k) or (batch, seqlen, ngroups, k)
        seq_idx: (batch, seqlen) or None. out[i, j] for seq_idx[i] != seq_idx[j] will be zeroed out.
        causal: if True, then out[i, j] for i > j will be arbitrary, only out[i, j] for i <= j are
            guaranteed to be correct.
    Return:
        out: (batch, nchunks, chunk_size, chunk_size) or (batch, nchunks, ngroups, chunk_size, chunk_size)
    """
    # Check constraints.
    has_groups = a.dim() == 4
    if not has_groups:
        batch, seqlen, k = a.shape
    else:
        batch, seqlen, ngroups, k = a.shape
    assert 

In [40]:
import inspect

forward_kernels = [
    ("Cumulative decay", chunk_state._chunk_cumsum_fwd),
    ("Within-chunk state", chunk_state._chunk_state_fwd),
    ("Within-chunk scan", chunk_scan._chunk_scan_fwd),
    ("Between-chunk state passing", state_passing._state_passing_fwd),
]

for name, fn in forward_kernels:

    print("\n" + "=" * 85)
    print(name)
    print("=" * 85)

    try:
        print("Signature:", inspect.signature(fn))
    except Exception as e:
        print("Signature unavailable:", e)

    try:
        source = inspect.getsource(fn)
        print(source[:9000])
    except Exception as e:
        print("Source unavailable:", e)


Cumulative decay
Signature: (dt, A, chunk_size, dt_bias=None, dt_softplus=False, dt_limit=(0.0, inf))
def _chunk_cumsum_fwd(dt, A, chunk_size, dt_bias=None, dt_softplus=False, dt_limit=(0.0, float("inf"))):
    batch, seqlen, nheads = dt.shape
    assert A.shape == (nheads,)
    if dt_bias is not None:
        assert dt_bias.shape == (nheads,)
    nchunks = math.ceil(seqlen / chunk_size)
    dt_out = torch.empty(batch, nheads, nchunks, chunk_size, device=dt.device, dtype=torch.float32)
    dA_cumsum = torch.empty(batch, nheads, nchunks, chunk_size, device=dt.device, dtype=torch.float32)
    grid_chunk_cs = lambda META: (batch, nchunks, triton.cdiv(nheads, META['BLOCK_SIZE_H']))
    with torch.cuda.device(dt.device.index):
        _chunk_cumsum_fwd_kernel[grid_chunk_cs](
            dt, A, dt_bias, dt_out, dA_cumsum,
            batch, seqlen, nheads, chunk_size,
            dt_limit[0], dt_limit[1],
            dt.stride(0), dt.stride(1), dt.stride(2),
            A.stride(0),
       

In [41]:
keywords = [
    "tl.exp",
    "exp(",
    "tl.cumsum",
    "cumsum",
    "dA",
    "A",
    "dt",
    "tl.sum",
    "state",
    "tl.dot",
]

for name, fn in forward_kernels:

    print("\n" + "=" * 85)
    print("DECAY AUDIT:", name)
    print("=" * 85)

    try:
        source = inspect.getsource(fn)
    except Exception:
        continue

    lines = source.splitlines()

    matching = [
        i for i, line in enumerate(lines)
        if any(word in line for word in keywords)
    ]

    shown = set()

    for i in matching:

        for j in range(max(0, i - 2), min(len(lines), i + 3)):

            if j not in shown:
                print(f"{j+1:04d}: {lines[j]}")
                shown.add(j)


DECAY AUDIT: Cumulative decay
0001: def _chunk_cumsum_fwd(dt, A, chunk_size, dt_bias=None, dt_softplus=False, dt_limit=(0.0, float("inf"))):
0002:     batch, seqlen, nheads = dt.shape
0003:     assert A.shape == (nheads,)
0004:     if dt_bias is not None:
0005:         assert dt_bias.shape == (nheads,)
0006:     nchunks = math.ceil(seqlen / chunk_size)
0007:     dt_out = torch.empty(batch, nheads, nchunks, chunk_size, device=dt.device, dtype=torch.float32)
0008:     dA_cumsum = torch.empty(batch, nheads, nchunks, chunk_size, device=dt.device, dtype=torch.float32)
0009:     grid_chunk_cs = lambda META: (batch, nchunks, triton.cdiv(nheads, META['BLOCK_SIZE_H']))
0010:     with torch.cuda.device(dt.device.index):
0011:         _chunk_cumsum_fwd_kernel[grid_chunk_cs](
0012:             dt, A, dt_bias, dt_out, dA_cumsum,
0013:             batch, seqlen, nheads, chunk_size,
0014:             dt_limit[0], dt_limit[1],
0015:             dt.stride(0), dt.stride(1), dt.stride(2),
0016:         

In [42]:
import inspect

kernel_candidates = [
    ("Cumsum Triton kernel", chunk_state, "_chunk_cumsum_fwd_kernel"),
    ("Chunk-state Triton kernel", chunk_state, "_chunk_state_fwd_kernel"),
    ("Chunk-scan Triton kernel", chunk_scan, "_chunk_scan_fwd_kernel"),
    ("State-passing Triton kernel", state_passing, "_state_passing_fwd_kernel"),
]

for label, module, kernel_name in kernel_candidates:

    print("\n" + "=" * 80)
    print(label)
    print("=" * 80)

    kernel = getattr(module, kernel_name, None)

    if kernel is None:
        print("Not exposed in this module.")

        # Search the combined module too
        kernel = getattr(combined, kernel_name, None)

    if kernel is None:
        print("Not exposed in combined module either.")
        continue

    print("Kernel type:", type(kernel))
    print("Kernel name:", kernel_name)

    try:
        print(inspect.getsource(kernel))
    except Exception as e:
        print("Source inspection error:", repr(e))

    try:
        print("\nTriton source:")
        print(kernel.src)
    except Exception:
        pass


Cumsum Triton kernel
Kernel type: <class 'triton.runtime.autotuner.Autotuner'>
Kernel name: _chunk_cumsum_fwd_kernel
Source inspection error: TypeError('module, class, method, function, traceback, frame, or code object was expected, got Autotuner')

Triton source:

Chunk-state Triton kernel
Kernel type: <class 'triton.runtime.autotuner.Autotuner'>
Kernel name: _chunk_state_fwd_kernel
Source inspection error: TypeError('module, class, method, function, traceback, frame, or code object was expected, got Autotuner')

Triton source:

Chunk-scan Triton kernel
Kernel type: <class 'triton.runtime.autotuner.Autotuner'>
Kernel name: _chunk_scan_fwd_kernel
Source inspection error: TypeError('module, class, method, function, traceback, frame, or code object was expected, got Autotuner')

Triton source:

State-passing Triton kernel
Kernel type: <class 'triton.runtime.autotuner.Autotuner'>
Kernel name: _state_passing_fwd_kernel
Source inspection error: TypeError('module, class, method, function, t

In [43]:
for module_name, module in modules:

    print("\nMODULE:", module_name)

    for name, obj in vars(module).items():

        if name.endswith("_kernel"):

            print(
                name,
                "|",
                type(obj).__name__,
                "|",
                getattr(obj, "__module__", "unknown")
            )


MODULE: combined
_chunk_scan_chunk_state_bwd_dx_kernel | Autotuner | triton.runtime.autotuner

MODULE: chunk_scan
_chunk_scan_fwd_kernel | Autotuner | triton.runtime.autotuner
_chunk_scan_bwd_dz_kernel | Autotuner | triton.runtime.autotuner
_chunk_scan_bwd_dstates_kernel | Autotuner | triton.runtime.autotuner
_chunk_scan_bwd_dc_kernel | Autotuner | triton.runtime.autotuner
_chunk_scan_bwd_dx_kernel | Autotuner | triton.runtime.autotuner
_chunk_scan_bwd_dcb_kernel | Autotuner | triton.runtime.autotuner
_chunk_scan_bwd_ddAcs_unstable_kernel | Autotuner | triton.runtime.autotuner
_chunk_scan_bwd_ddAcs_stable_kernel | Autotuner | triton.runtime.autotuner
_chunk_scan_bwd_ddAcs_prev_kernel | Autotuner | triton.runtime.autotuner

MODULE: chunk_state
_chunk_cumsum_fwd_kernel | Autotuner | triton.runtime.autotuner
_chunk_cumsum_bwd_kernel | Autotuner | triton.runtime.autotuner
_chunk_state_fwd_kernel | Autotuner | triton.runtime.autotuner
_chunk_state_bwd_dx_kernel | Autotuner | triton.runtime

In [44]:
import inspect

kernel_candidates = [
    ("Cumsum", chunk_state, "_chunk_cumsum_fwd_kernel"),
    ("Chunk state", chunk_state, "_chunk_state_fwd_kernel"),
    ("Chunk scan", chunk_scan, "_chunk_scan_fwd_kernel"),
    ("State passing", state_passing, "_state_passing_fwd_kernel"),
]

extracted_sources = {}

for label, module, kernel_name in kernel_candidates:

    print("\n" + "=" * 90)
    print(label.upper())
    print("=" * 90)

    autotuner = getattr(module, kernel_name)

    print("Autotuner:", type(autotuner))

    # Access the underlying Triton JIT function
    jit_kernel = autotuner.fn

    print("Underlying function:", type(jit_kernel))
    print("Function name:", getattr(jit_kernel, "__name__", "unknown"))

    try:
        source = jit_kernel.src
    except Exception:
        try:
            source = inspect.getsource(jit_kernel.fn)
        except Exception as e:
            print("Could not extract source:", repr(e))
            continue

    extracted_sources[label] = source

    print("Source length:", len(source), "characters")
    print(source)


CUMSUM
Autotuner: <class 'triton.runtime.autotuner.Autotuner'>
Underlying function: <class 'triton.runtime.jit.JITFunction'>
Function name: _chunk_cumsum_fwd_kernel
Source length: 2433 characters
def _chunk_cumsum_fwd_kernel(
    # Pointers to matrices
    dt_ptr, A_ptr, dt_bias_ptr, dt_out_ptr, dA_cumsum_ptr,
    # Matrix dimension
    batch, seqlen, nheads, chunk_size,
    dt_min, dt_max,
    # Strides
    stride_dt_batch, stride_dt_seqlen, stride_dt_head,
    stride_A_head,
    stride_dt_bias_head,
    stride_dt_out_batch, stride_dt_out_chunk, stride_dt_out_head, stride_dt_out_csize,
    stride_dA_cs_batch, stride_dA_cs_chunk, stride_dA_cs_head, stride_dA_cs_csize,
    # Meta-parameters
    DT_SOFTPLUS: tl.constexpr,
    HAS_DT_BIAS: tl.constexpr,
    BLOCK_SIZE_H: tl.constexpr, BLOCK_SIZE_CHUNK: tl.constexpr,
):
    pid_b = tl.program_id(axis=0)
    pid_c = tl.program_id(axis=1)
    pid_h = tl.program_id(axis=2)
    dt_ptr += pid_b * stride_dt_batch + pid_c * chunk_size * stride_d

In [45]:
keywords = [
    "exp",
    "exp2",
    "cumsum",
    "dA",
    "dt",
    "tl.sum",
    "tl.dot",
    "tl.load",
    "tl.store",
    "state",
    "cumprod",
]

for label, source in extracted_sources.items():

    print("\n" + "=" * 90)
    print("DECAY OPERATIONS:", label)
    print("=" * 90)

    lines = source.splitlines()

    matching = [
        i for i, line in enumerate(lines)
        if any(word in line.lower() for word in keywords)
    ]

    shown = set()

    for i in matching:

        for j in range(max(0, i - 2), min(len(lines), i + 4)):

            if j not in shown:
                print(f"{j + 1:04d}: {lines[j]}")
                shown.add(j)

        print("-" * 40)


DECAY OPERATIONS: Cumsum
0001: def _chunk_cumsum_fwd_kernel(
0002:     # Pointers to matrices
0003:     dt_ptr, A_ptr, dt_bias_ptr, dt_out_ptr, dA_cumsum_ptr,
0004:     # Matrix dimension
----------------------------------------
0005:     batch, seqlen, nheads, chunk_size,
0006:     dt_min, dt_max,
----------------------------------------
0007:     # Strides
0008:     stride_dt_batch, stride_dt_seqlen, stride_dt_head,
0009:     stride_A_head,
----------------------------------------
0010:     stride_dt_bias_head,
0011:     stride_dt_out_batch, stride_dt_out_chunk, stride_dt_out_head, stride_dt_out_csize,
----------------------------------------
0012:     stride_dA_cs_batch, stride_dA_cs_chunk, stride_dA_cs_head, stride_dA_cs_csize,
0013:     # Meta-parameters
----------------------------------------
0014:     DT_SOFTPLUS: tl.constexpr,
----------------------------------------
0015:     HAS_DT_BIAS: tl.constexpr,
0016:     BLOCK_SIZE_H: tl.constexpr, BLOCK_SIZE_CHUNK: tl.constexpr,
001

In [47]:
# ---------------------------------------------------------
# 2. Construct the original Dyadic2 coefficient grid
# ---------------------------------------------------------

pairs = [
    (k1, k2)
    for k1 in range(2, 15)
    for k2 in range(k1, 15)
]

grid_d = np.array([
    2.0**(-k1) + 2.0**(-k2)
    for k1, k2 in pairs
])

grid_a = 1.0 - grid_d

print("Grid size:", len(grid_a))
print("Grid range:", grid_a.min(), grid_a.max())


# ---------------------------------------------------------
# 3. Collect causal interval decay factors
# ---------------------------------------------------------

all_interval_weights = []
all_previous_state_weights = []
all_chunk_weights = []

for record in actual_scan_records:

    dA = record["dA"].numpy()

    # Shape: batch, sequence, heads
    dA_cs = np.cumsum(dA, axis=1)

    batch_size, seq_len, nheads = dA_cs.shape

    # Pairwise interval factors:
    # exp(prefix[m] - prefix[k])
    for b in range(batch_size):

        for h in range(nheads):

            prefix = dA_cs[b, :, h]

            differences = (
                prefix[:, None] - prefix[None, :]
            )

            causal_mask = np.tril(
                np.ones((seq_len, seq_len), dtype=bool)
            )

            weights = np.exp(
                np.minimum(differences[causal_mask], 0.0)
            )

            all_interval_weights.append(weights)

            # Previous-state factors: exp(prefix[m])
            all_previous_state_weights.append(
                np.exp(np.minimum(prefix, 0.0))
            )

            # Whole-chunk decay factor
            all_chunk_weights.append(
                np.exp(np.minimum(prefix[-1], 0.0))
            )


interval_values = np.concatenate(all_interval_weights)
previous_values = np.concatenate(all_previous_state_weights)
chunk_values = np.asarray(all_chunk_weights)

print("Interval factors:", len(interval_values))
print("Previous-state factors:", len(previous_values))
print("Chunk factors:", len(chunk_values))

Grid size: 91
Grid range: 0.5 0.9998779296875
Interval factors: 198144
Previous-state factors: 3072
Chunk factors: 24


In [49]:

import numpy as np

all_interval_weights = []
all_previous_weights = []
all_chunk_weights = []

block_size = 4

for record in actual_scan_records:
    dA = record["dA"].numpy()  # [batch, sequence, heads]
    batch_size, seq_len, n_heads = dA.shape

    prefix = np.cumsum(dA, axis=1, dtype=np.float32)

    # Previous-state factors
    prev = np.exp(np.minimum(prefix, 0.0))
    all_previous_weights.append(prev.reshape(-1))

    # Whole-chunk factors
    chunk = np.exp(np.minimum(prefix[:, -1, :], 0.0))
    all_chunk_weights.append(chunk.reshape(-1))

    # Causal interval factors, in small row blocks
    for start in range(0, seq_len, block_size):
        end = min(start + block_size, seq_len)

        # [batch, block_rows, sequence, heads]
        block_diff = (
            prefix[:, start:end, None, :]
            - prefix[:, None, :, :]
        )

        row_indices = np.arange(start, end)[:, None]
        col_indices = np.arange(seq_len)[None, :]
        causal = col_indices <= row_indices

        weights = np.exp(np.minimum(block_diff, 0.0))

        # Select only causal positions
        selected = weights[:, causal, :]
        all_interval_weights.append(selected.reshape(-1))

        del block_diff, weights, selected

all_interval_weights = np.concatenate(all_interval_weights)
all_previous_weights = np.concatenate(all_previous_weights)
all_chunk_weights = np.concatenate(all_chunk_weights)

print("Interval factors:", len(all_interval_weights))
print("Previous-state factors:", len(all_previous_weights))
print("Chunk factors:", len(all_chunk_weights))

print("Finite interval values:",
      np.isfinite(all_interval_weights).all())
print("Finite previous-state values:",
      np.isfinite(all_previous_weights).all())
print("Finite chunk values:",
      np.isfinite(all_chunk_weights).all())


Interval factors: 198144
Previous-state factors: 3072
Chunk factors: 24
Finite interval values: True
Finite previous-state values: True
Finite chunk values: True


In [50]:

import numpy as np
import pandas as pd

pairs = [
    (k1, k2)
    for k1 in range(2, 15)
    for k2 in range(k1, 15)
]

grid_a = np.array([
    1.0 - 2.0**(-k1) - 2.0**(-k2)
    for k1, k2 in pairs
], dtype=np.float64)

def evaluate_dyadic2(values, name):
    values = np.asarray(values, dtype=np.float64).ravel()

    # Values below the representable range are reported separately.
    fallback_mask = values < grid_a.min()
    eligible = values[~fallback_mask]

    if len(eligible) == 0:
        return {
            "factor_type": name,
            "count": len(values),
            "fallback_count": int(fallback_mask.sum()),
            "fallback_pct": 100.0,
            "MAE": np.nan,
            "RMSE": np.nan,
            "max_error": np.nan
        }

    indices = np.abs(
        eligible[:, None] - grid_a[None, :]
    ).argmin(axis=1)

    approximated = grid_a[indices]
    errors = approximated - eligible

    return {
        "factor_type": name,
        "count": len(values),
        "fallback_count": int(fallback_mask.sum()),
        "fallback_pct": 100.0 * fallback_mask.mean(),
        "MAE": np.mean(np.abs(errors)),
        "RMSE": np.sqrt(np.mean(errors**2)),
        "max_error": np.max(np.abs(errors))
    }

results = [
    evaluate_dyadic2(all_interval_weights, "Within-sequence intervals"),
    evaluate_dyadic2(all_previous_weights, "Previous-state factors"),
    evaluate_dyadic2(all_chunk_weights, "Whole-chunk factors")
]

results_df = pd.DataFrame(results)

display(results_df.round(8))


,factor_type,count,fallback_count,fallback_pct,MAE,RMSE,max_error
0,Within-sequence intervals,198144,131239,66.234153,0.015093,0.021851,0.062498
1,Previous-state factors,3072,2532,82.421875,0.016387,0.022969,0.062027
2,Whole-chunk factors,24,24,100.000000,NaN,NaN,NaN


In [51]:

import numpy as np
import pandas as pd

factor_sets = {
    "Within-sequence intervals": all_interval_weights,
    "Previous-state factors": all_previous_weights,
    "Whole-chunk factors": all_chunk_weights,
}

rows = []

for name, values in factor_sets.items():
    values = np.asarray(values, dtype=np.float64)

    rows.append({
        "factor_type": name,
        "min": values.min(),
        "p001": np.quantile(values, 0.001),
        "p01": np.quantile(values, 0.01),
        "p05": np.quantile(values, 0.05),
        "p10": np.quantile(values, 0.10),
        "p25": np.quantile(values, 0.25),
        "median": np.median(values),
        "p75": np.quantile(values, 0.75),
        "p90": np.quantile(values, 0.90),
        "p95": np.quantile(values, 0.95),
        "p99": np.quantile(values, 0.99),
        "max": values.max()
    })

distribution_df = pd.DataFrame(rows)
display(distribution_df.round(8))

# Additional coverage analysis
thresholds = [0.001, 0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]

coverage_rows = []

for name, values in factor_sets.items():
    values = np.asarray(values)

    row = {"factor_type": name}

    for threshold in thresholds:
        row[f"below_{threshold}"] = (
            100.0 * np.mean(values < threshold)
        )

    coverage_rows.append(row)

display(pd.DataFrame(coverage_rows).round(3))


,factor_type,min,p001,p01,p05,p10,p25,median,p75,p90,p95,p99,max
0,Within-sequence intervals,0.0,1.000000e-08,1.497000e-05,0.042788,0.068407,0.147584,0.331233,0.609998,0.837677,0.919750,1.000000,1.000000
1,Previous-state factors,0.0,0.000000e+00,6.000000e-08,0.025886,0.032317,0.057047,0.146476,0.377445,0.662593,0.811221,0.942834,0.972835
2,Whole-chunk factors,0.0,2.213000e-04,2.212960e-03,0.011609,0.022921,0.024961,0.026086,0.026551,0.029470,0.030349,0.034903,0.036217


,factor_type,below_0.001,below_0.01,below_0.05,below_0.1,below_0.2,below_0.3,below_0.4,below_0.5
0,Within-sequence intervals,1.933,2.580,6.310,16.345,33.307,46.411,57.108,66.234
1,Previous-state factors,2.897,3.385,21.549,39.714,58.268,69.010,76.628,82.422
2,Whole-chunk factors,4.167,8.333,100.000,100.000,100.000,100.000,100.000,100.000


In [52]:

import numpy as np
import pandas as pd

rows = []

for record_index, record in enumerate(actual_scan_records):
    dA = record["dA"].numpy().astype(np.float64)

    # Inspect original decay exponents
    abs_dA = np.abs(dA)

    # Reconstruct a small subset of interval factors in FP64
    batch_size, seq_len, n_heads = dA.shape

    sample_heads = min(4, n_heads)
    sample_batch = min(1, batch_size)

    sampled_dA = dA[:sample_batch, :, :sample_heads]

    prefix64 = np.cumsum(sampled_dA, axis=1)
    intervals64 = []

    for end in range(seq_len):
        for start in range(end + 1):
            log_weight = (
                prefix64[:, end, :]
                - prefix64[:, start, :]
            )
            intervals64.append(np.exp(np.minimum(log_weight, 0.0)))

    intervals64 = np.concatenate(
        [x.reshape(-1) for x in intervals64]
    )

    rows.append({
        "record": record_index,
        "dA_min": dA.min(),
        "dA_max": dA.max(),
        "dA_mean": dA.mean(),
        "dA_abs_p50": np.quantile(abs_dA, 0.50),
        "dA_abs_p90": np.quantile(abs_dA, 0.90),
        "dA_abs_p99": np.quantile(abs_dA, 0.99),
        "sampled_interval_zero_pct": 100 * np.mean(intervals64 == 0),
        "sampled_interval_below_1e-30_pct":
            100 * np.mean(intervals64 < 1e-30),
        "sampled_interval_min": intervals64.min(),
        "sampled_interval_median": np.median(intervals64),
    })

diagnostic_df = pd.DataFrame(rows)
display(diagnostic_df.round(10))

print("\nAggregate dA statistics:")
all_dA_abs = np.concatenate([
    np.abs(r["dA"].numpy().astype(np.float64)).ravel()
    for r in actual_scan_records
])

print("Mean |dA|:", all_dA_abs.mean())
print("Median |dA|:", np.median(all_dA_abs))
print("99th percentile |dA|:", np.quantile(all_dA_abs, 0.99))
print("Maximum |dA|:", all_dA_abs.max())


,record,dA_min,dA_max,dA_mean,dA_abs_p50,dA_abs_p90,dA_abs_p99,sampled_interval_zero_pct,sampled_interval_below_1e-30_pct,sampled_interval_min,sampled_interval_median
0,0,-0.314669,-0.015237,-0.034585,0.028504,0.033429,0.191819,0.0,0.0,0.010019,0.333508



Aggregate dA statistics:
Mean |dA|: 0.03458468924873159
Median |dA|: 0.028504355810582638
99th percentile |dA|: 0.19181902706623077
Maximum |dA|: 0.3146694004535675


In [53]:

import numpy as np
import pandas as pd

audit_rows = []

for i, record in enumerate(actual_scan_records):
    dA = record["dA"].numpy().astype(np.float64)

    audit_rows.append({
        "record": i,
        "shape": str(dA.shape),
        "min": dA.min(),
        "max": dA.max(),
        "positive_pct": 100 * np.mean(dA > 0),
        "zero_pct": 100 * np.mean(dA == 0),
        "nonfinite": int((~np.isfinite(dA)).sum()),
        "abs_p50": np.quantile(np.abs(dA), .50),
        "abs_p90": np.quantile(np.abs(dA), .90),
        "abs_p99": np.quantile(np.abs(dA), .99),
    })

audit_df = pd.DataFrame(audit_rows)

display(audit_df.head(10).round(8))
display(audit_df.tail(10).round(8))

print("Overall positive values:",
      sum((r["dA"].numpy() > 0).sum() for r in actual_scan_records))

print("Overall negative values:",
      sum((r["dA"].numpy() < 0).sum() for r in actual_scan_records))

print("Overall nonfinite values:",
      sum((~np.isfinite(r["dA"].numpy())).sum() for r in actual_scan_records))


,record,shape,min,max,positive_pct,zero_pct,nonfinite,abs_p50,abs_p90,abs_p99
0,0,"(1, 128, 24)",-0.314669,-0.015237,0.0,0.0,0,0.028504,0.033429,0.191819


,record,shape,min,max,positive_pct,zero_pct,nonfinite,abs_p50,abs_p90,abs_p99
0,0,"(1, 128, 24)",-0.314669,-0.015237,0.0,0.0,0,0.028504,0.033429,0.191819


Overall positive values: 0
Overall negative values: 3072
Overall nonfinite values: 0


In [54]:

record = actual_scan_records[0]

dA = record["dA"].numpy().astype(np.float64)

# One batch, first head
prefix = np.cumsum(dA[0, :, 0])

test_pairs = [
    (1, 0),
    (5, 0),
    (10, 5),
    (20, 10),
    (50, 25),
    (100, 50),
    (127, 126),
]

for m, k in test_pairs:
    log_weight = prefix[m] - prefix[k]
    weight = np.exp(min(log_weight, 0.0))

    print({
        "m": m,
        "k": k,
        "log_weight": float(log_weight),
        "weight_fp64": float(weight)
    })


{'m': 1, 'k': 0, 'log_weight': -0.03075673058629036, 'weight_fp64': 0.9697114455194062}
{'m': 5, 'k': 0, 'log_weight': -0.15316231548786163, 'weight_fp64': 0.8579904453706169}
{'m': 10, 'k': 5, 'log_weight': -0.1546580269932747, 'weight_fp64': 0.8567080984395381}
{'m': 20, 'k': 10, 'log_weight': -0.29113020561635494, 'weight_fp64': 0.7474183536147292}
{'m': 50, 'k': 25, 'log_weight': -0.7413078267127275, 'weight_fp64': 0.47649034104985805}
{'m': 100, 'k': 50, 'log_weight': -1.4690169990062714, 'weight_fp64': 0.23015161329101275}
{'m': 127, 'k': 126, 'log_weight': -0.0313541442155838, 'weight_fp64': 0.9691322996973997}


In [55]:

import numpy as np
import pandas as pd

# Gather actual nonnegative decay rates
rates = np.concatenate([
    -record["dA"].numpy().astype(np.float64).ravel()
    for record in actual_scan_records
])

# Candidate rate grid: two dyadic terms
rate_pairs = [
    (k1, k2)
    for k1 in range(1, 25)
    for k2 in range(k1, 25)
]

rate_grid = np.array([
    2.0**(-k1) + 2.0**(-k2)
    for k1, k2 in rate_pairs
])

# Nearest rate approximation
indices = np.abs(
    rates[:, None] - rate_grid[None, :]
).argmin(axis=1)

rate_approx = rate_grid[indices]
rate_error = rate_approx - rates

# Evaluate coefficient reconstruction too
a_true = np.exp(-rates)
a_approx = np.exp(-rate_approx)
coefficient_error = a_approx - a_true

summary = pd.DataFrame([
    {
        "metric": "Rate approximation",
        "MAE": np.mean(np.abs(rate_error)),
        "RMSE": np.sqrt(np.mean(rate_error**2)),
        "Max error": np.max(np.abs(rate_error)),
        "Mean signed error": np.mean(rate_error)
    },
    {
        "metric": "Reconstructed decay coefficient",
        "MAE": np.mean(np.abs(coefficient_error)),
        "RMSE": np.sqrt(np.mean(coefficient_error**2)),
        "Max error": np.max(np.abs(coefficient_error)),
        "Mean signed error": np.mean(coefficient_error)
    }
])

print("Rate grid size:", len(rate_grid))
print("Rate grid range:", rate_grid.min(), rate_grid.max())
display(summary)

# Rate approximation coverage
print("\nRate distribution:")
print(pd.Series(rates).describe(
    percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99]
))

# Inspect selected exponent pairs
selected_pairs = np.array(rate_pairs)[indices]

print("\nUnique selected rate representations:",
      len(np.unique(selected_pairs, axis=0)))


Rate grid size: 300
Rate grid range: 1.1920928955078125e-07 1.0


,metric,MAE,RMSE,Max error,Mean signed error
0,Rate approximation,0.002290,0.003061,0.030679,0.000599
1,Reconstructed decay coefficient,0.002188,0.002801,0.024263,-0.000574



Rate distribution:
count    3072.000000
mean        0.034585
std         0.030076
min         0.015237
1%          0.022568
5%          0.024582
10%         0.025589
25%         0.027147
50%         0.028504
75%         0.030326
90%         0.033429
95%         0.038658
99%         0.191819
max         0.314669
dtype: float64

Unique selected rate representations: 25


In [56]:

import numpy as np
import pandas as pd

rates = np.concatenate([
    -record["dA"].numpy().astype(np.float64).ravel()
    for record in actual_scan_records
])

# Include positive and negative exponent indices.
# d = 2^(-k1) + 2^(-k2)
k_values = range(-4, 25)

rate_pairs_extended = [
    (k1, k2)
    for k1 in k_values
    for k2 in k_values
    if k2 >= k1
]

rate_grid_extended = np.array([
    2.0**(-k1) + 2.0**(-k2)
    for k1, k2 in rate_pairs_extended
], dtype=np.float64)

grid_coefficients = np.exp(-rate_grid_extended)
true_coefficients = np.exp(-rates)

# Select nearest reconstructed coefficient in memory-safe batches.
batch_size = 8192
selected_rates = np.empty_like(rates)
selected_coefficients = np.empty_like(rates)

for start in range(0, len(rates), batch_size):
    end = min(start + batch_size, len(rates))

    target = true_coefficients[start:end]

    distances = np.abs(
        target[:, None] - grid_coefficients[None, :]
    )

    indices = distances.argmin(axis=1)

    selected_rates[start:end] = rate_grid_extended[indices]
    selected_coefficients[start:end] = grid_coefficients[indices]

rate_errors = selected_rates - rates
coefficient_errors = selected_coefficients - true_coefficients

# Baseline: previous grid, nearest rate selection
old_rate_grid = np.array([
    2.0**(-k1) + 2.0**(-k2)
    for k1 in range(1, 25)
    for k2 in range(k1, 25)
])

old_indices = np.abs(
    rates[:, None] - old_rate_grid[None, :]
).argmin(axis=1)

old_coefficients = np.exp(-old_rate_grid[old_indices])
old_errors = old_coefficients - true_coefficients

def metrics(errors):
    return {
        "MAE": np.mean(np.abs(errors)),
        "RMSE": np.sqrt(np.mean(errors**2)),
        "Max error": np.max(np.abs(errors)),
        "Mean signed error": np.mean(errors)
    }

comparison = pd.DataFrame([
    {
        "method": "Original rate-nearest",
        **metrics(old_errors)
    },
    {
        "method": "Extended rate-Dyadic2, coefficient-nearest",
        **metrics(coefficient_errors)
    }
])

print("Extended grid size:", len(rate_grid_extended))
print("Rate grid range:",
      rate_grid_extended.min(), rate_grid_extended.max())

display(comparison)

print("Unique selected exponent pairs:",
      len(np.unique(
          np.array(rate_pairs_extended)[
              np.array([
                  np.abs(v - rate_grid_extended).argmin()
                  for v in selected_rates[::100]
              ])
          ],
          axis=0
      )))

print("Selected rate range:",
      selected_rates.min(), selected_rates.max())


Extended grid size: 435
Rate grid range: 1.1920928955078125e-07 32.0


,method,MAE,RMSE,Max error,Mean signed error
0,Original rate-nearest,0.002188,0.002801,0.024263,-0.000574
1,"Extended rate-Dyadic2, coefficient-nearest",0.002188,0.002801,0.024263,-0.000602


Unique selected exponent pairs: 7
Selected rate range: 0.015625 0.3125


In [57]:

import numpy as np
import pandas as pd

# Reconstruct individual actual decay coefficients, layer by layer
layer_coefficients = {}

for layer_idx, record in enumerate(actual_scan_records):
    rates_layer = -record["dA"].numpy().astype(np.float64)
    layer_coefficients[layer_idx % 24] = (
        np.exp(-rates_layer).reshape(-1)
        if layer_idx % 24 not in layer_coefficients
        else np.concatenate([
            layer_coefficients[layer_idx % 24],
            np.exp(-rates_layer).reshape(-1)
        ])
    )

# Original coefficient grid
old_pairs = [
    (k1, k2)
    for k1 in range(2, 15)
    for k2 in range(k1, 15)
]

old_grid = np.array([
    1 - 2.0**(-k1) - 2.0**(-k2)
    for k1, k2 in old_pairs
])

# Extended rate grid and corresponding coefficients
rate_pairs = [
    (k1, k2)
    for k1 in range(-4, 25)
    for k2 in range(k1, 25)
]

rate_grid = np.array([
    2.0**(-k1) + 2.0**(-k2)
    for k1, k2 in rate_pairs
])

rate_coefficient_grid = np.exp(-rate_grid)

def quantize_batched(values, grid, batch_size=8192):
    result = np.empty_like(values, dtype=np.float64)

    for start in range(0, len(values), batch_size):
        end = min(start + batch_size, len(values))
        block = values[start:end]

        idx = np.abs(block[:, None] - grid[None, :]).argmin(axis=1)
        result[start:end] = grid[idx]

    return result

def summarize_errors(errors):
    return {
        "MAE": np.mean(np.abs(errors)),
        "RMSE": np.sqrt(np.mean(errors**2)),
        "Max error": np.max(np.abs(errors)),
        "Bias": np.mean(errors)
    }

layer_rows = []

for layer, values in sorted(layer_coefficients.items()):
    # Original grid with exact fallback below 0.5
    old_approx = values.copy()
    eligible = values >= old_grid.min()
    old_approx[eligible] = quantize_batched(
        values[eligible], old_grid
    )

    # Extended rate grid, nearest reconstructed coefficient
    new_approx = quantize_batched(
        values, rate_coefficient_grid
    )

    old_metrics = summarize_errors(old_approx - values)
    new_metrics = summarize_errors(new_approx - values)

    layer_rows.append({
        "layer": layer,
        "count": len(values),
        "old_fallback_pct": 100 * np.mean(~eligible),
        "old_MAE": old_metrics["MAE"],
        "new_MAE": new_metrics["MAE"],
        "old_RMSE": old_metrics["RMSE"],
        "new_RMSE": new_metrics["RMSE"],
        "old_max": old_metrics["Max error"],
        "new_max": new_metrics["Max error"]
    })

layer_comparison = pd.DataFrame(layer_rows)
display(layer_comparison.round(8))

# Global fair comparison
all_values = np.concatenate(list(layer_coefficients.values()))

old_global = all_values.copy()
eligible = all_values >= old_grid.min()
old_global[eligible] = quantize_batched(
    all_values[eligible], old_grid
)

new_global = quantize_batched(
    all_values, rate_coefficient_grid
)

global_comparison = pd.DataFrame([
    {"method": "Original coefficient-Dyadic2 + exact fallback",
     **summarize_errors(old_global - all_values)},
    {"method": "Extended rate-Dyadic2",
     **summarize_errors(new_global - all_values)}
])

print("GLOBAL COMPARISON")
display(global_comparison.round(9))

print("Original exact-fallback percentage:",
      round(100 * np.mean(~eligible), 4), "%")


,layer,count,old_fallback_pct,old_MAE,new_MAE,old_RMSE,new_RMSE,old_max,new_max
0,0,3072,0.0,0.002398,0.002188,0.003179,0.002801,0.030322,0.024263


GLOBAL COMPARISON


,method,MAE,RMSE,Max error,Bias
0,Original coefficient-Dyadic2 + exact fallback,0.002398,0.003179,0.030322,-0.000263
1,Extended rate-Dyadic2,0.002188,0.002801,0.024263,-0.000602


Original exact-fallback percentage: 0.0 %


In [58]:

import numpy as np
import pandas as pd

# The arrays are ordered by captured document and sequence.
# This is a provisional split; use a document-level split
# in the final experiment.

policy_rows = []
evaluation_errors = []

for layer, values in sorted(layer_coefficients.items()):
    values = np.asarray(values, dtype=np.float64)

    midpoint = len(values) // 2
    calibration = values[:midpoint]
    evaluation = values[midpoint:]

    # Original Dyadic2 + exact fallback
    old_cal = calibration.copy()
    old_eval = evaluation.copy()

    old_cal_mask = calibration >= old_grid.min()
    old_eval_mask = evaluation >= old_grid.min()

    old_cal[old_cal_mask] = quantize_batched(
        calibration[old_cal_mask], old_grid
    )
    old_eval[old_eval_mask] = quantize_batched(
        evaluation[old_eval_mask], old_grid
    )

    # Extended rate-Dyadic2
    rate_cal = quantize_batched(
        calibration, rate_coefficient_grid
    )
    rate_eval = quantize_batched(
        evaluation, rate_coefficient_grid
    )

    old_cal_mae = np.mean(np.abs(old_cal - calibration))
    rate_cal_mae = np.mean(np.abs(rate_cal - calibration))

    # Select method only from calibration data
    chosen = (
        "original" if old_cal_mae <= rate_cal_mae
        else "rate"
    )

    chosen_eval = old_eval if chosen == "original" else rate_eval

    policy_rows.append({
        "layer": layer,
        "calibration_old_MAE": old_cal_mae,
        "calibration_rate_MAE": rate_cal_mae,
        "selected_method": chosen,
        "evaluation_MAE": np.mean(
            np.abs(chosen_eval - evaluation)
        ),
        "evaluation_RMSE": np.sqrt(
            np.mean((chosen_eval - evaluation)**2)
        ),
        "evaluation_max": np.max(
            np.abs(chosen_eval - evaluation)
        )
    })

policy_df = pd.DataFrame(policy_rows)
display(policy_df.round(8))

print("Selected original:",
      (policy_df.selected_method == "original").sum())
print("Selected rate:",
      (policy_df.selected_method == "rate").sum())


,layer,calibration_old_MAE,calibration_rate_MAE,selected_method,evaluation_MAE,evaluation_RMSE,evaluation_max
0,0,0.00237,0.002151,rate,0.002225,0.002857,0.024263


Selected original: 0
Selected rate: 1


In [61]:

# SHIFT-SCAN: STAGE 1 — EXPERIMENT BASELINE AUDIT

import os
import sys
import json
import inspect
import platform
import importlib.metadata as metadata
import torch
import numpy as np

print("=" * 70)
print("SHIFT-SCAN EXPERIMENT BASELINE")
print("=" * 70)

# 1. Environment
print("\n[1] ENVIRONMENT")
print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)

for package in ["mamba-ssm", "triton", "transformers", "datasets"]:
    try:
        print(f"{package}:", metadata.version(package))
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

# 2. GPU
print("\n[2] GPU")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        prop = torch.cuda.get_device_properties(i)
        print({
            "device": i,
            "name": prop.name,
            "memory_GB": round(prop.total_memory / 1e9, 2),
            "compute_capability": f"{prop.major}.{prop.minor}"
        })

# 3. Model
print("\n[3] MODEL")

assert "model" in globals(), "Load the Mamba-2 model first."

parameter_count = sum(p.numel() for p in model.parameters())
model_device = next(model.parameters()).device
model_dtype = next(model.parameters()).dtype

print("Model class:", type(model).__name__)
print("Parameters:", f"{parameter_count:,}")
print("Device:", model_device)
print("Parameter dtype:", model_dtype)
print("Training mode:", model.training)

# 4. Mamba-2 layer configuration
print("\n[4] MAMBA-2 LAYER CONFIGURATION")

backbone = getattr(model, "backbone", None)
assert backbone is not None, "Could not locate model.backbone."

layers = getattr(backbone, "layers", None)
assert layers is not None, "Could not locate backbone.layers."

print("Number of layers:", len(layers))

layer_config_rows = []

for i, layer in enumerate(layers):
    mixer = layer.mixer

    row = {
        "layer": i,
        "use_mem_eff_path": getattr(
            mixer, "use_mem_eff_path", None
        ),
        "d_state": getattr(mixer, "d_state", None),
        "headdim": getattr(mixer, "headdim", None),
        "nheads": getattr(mixer, "nheads", None),
        "ngroups": getattr(mixer, "ngroups", None),
        "chunk_size": getattr(mixer, "chunk_size", None),
    }

    layer_config_rows.append(row)

for row in layer_config_rows:
    print(row)

assert all(
    row["use_mem_eff_path"] is False
    for row in layer_config_rows
), "Some layers are not using the intended fallback scan path."

# 5. Scan implementation
print("\n[5] SCAN IMPLEMENTATION")

import mamba_ssm.modules.mamba2 as mamba2_mod

scan_fn = mamba2_mod.mamba_chunk_scan_combined

print("Scan function:", scan_fn)
print("Scan module:", getattr(scan_fn, "__module__", None))

try:
    print("Scan signature:", inspect.signature(scan_fn))
except Exception as e:
    print("Signature unavailable:", repr(e))

# 6. Tokenizer and evaluation data
print("\n[6] DATA")

assert "tokenizer" in globals(), "Tokenizer is not loaded."
assert "encoded_samples" in globals(), "Tokenized samples are missing."

print("Tokenizer:", type(tokenizer).__name__)
print("Tokenizer length:", len(tokenizer))
print("Evaluation sequences:", len(encoded_samples))

for i, tokens in enumerate(encoded_samples[:3]):
    ids = tokens["input_ids"] if isinstance(tokens, dict) else tokens
    print(
        f"Sequence {i}: shape={tuple(ids.shape)}, "
        f"min_id={int(ids.min())}, max_id={int(ids.max())}"
    )

# 7. Existing capture audit
print("\n[7] EXISTING DECAY CAPTURE")

if "actual_scan_records" in globals():
    print("Captured scan records:", len(actual_scan_records))

    if len(actual_scan_records) > 0:
        sample = actual_scan_records[0]
        for key, value in sample.items():
            if torch.is_tensor(value):
                print(key, tuple(value.shape), value.dtype)
else:
    print("No existing actual_scan_records variable.")

# 8. Save baseline metadata
baseline_metadata = {
    "python": sys.version,
    "platform": platform.platform(),
    "pytorch": torch.__version__,
    "numpy": np.__version__,
    "parameter_count": int(parameter_count),
    "model_device": str(model_device),
    "model_dtype": str(model_dtype),
    "layer_count": len(layers),
    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available() else None
    ),
    "sequence_count": len(encoded_samples),
    "sequence_shapes": [
        list(
            (x["input_ids"] if isinstance(x, dict) else x).shape
        )
        for x in encoded_samples
    ],
    "fallback_scan_enabled": True,
}

os.makedirs("/kaggle/working/shiftscan_final", exist_ok=True)

with open(
    "/kaggle/working/shiftscan_final/baseline_metadata.json",
    "w"
) as f:
    json.dump(baseline_metadata, f, indent=2)

print("\nBASELINE AUDIT COMPLETE")
print("Saved: /kaggle/working/shiftscan_final/baseline_metadata.json")


SHIFT-SCAN EXPERIMENT BASELINE

[1] ENVIRONMENT
Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Platform: Linux-6.18.55+-x86_64-with-glibc2.39
PyTorch: 2.11.0+cu130
NumPy: 2.1.3
mamba-ssm: 2.3.2.post1
triton: 3.6.0
transformers: 5.20.2
datasets: 4.8.5

[2] GPU
CUDA available: True
{'device': 0, 'name': 'Tesla T4', 'memory_GB': 15.64, 'compute_capability': '7.5'}
{'device': 1, 'name': 'Tesla T4', 'memory_GB': 15.64, 'compute_capability': '7.5'}

[3] MODEL
Model class: MambaLMHeadModel
Parameters: 128,989,632
Device: cuda:0
Parameter dtype: torch.float16
Training mode: False

[4] MAMBA-2 LAYER CONFIGURATION
Number of layers: 24
{'layer': 0, 'use_mem_eff_path': False, 'd_state': 128, 'headdim': 64, 'nheads': 24, 'ngroups': 1, 'chunk_size': 256}
{'layer': 1, 'use_mem_eff_path': False, 'd_state': 128, 'headdim': 64, 'nheads': 24, 'ngroups': 1, 'chunk_size': 256}
{'layer': 2, 'use_mem_eff_path': False, 'd_state': 128, 'headdim': 64, 'nheads': 24, 'ngroups': 1, 'chunk_size': 256}
{

In [62]:
import inspect
import textwrap
import mamba_ssm.ops.triton.ssd_combined as ssd

print("=" * 75)
print("1. TRITON SCAN MODULE")
print("=" * 75)
print("Module file:", inspect.getfile(ssd))

# Find scan-related kernels and helpers defined in this module.
interesting = []
for name, obj in vars(ssd).items():
    if any(term in name.lower() for term in
           ["cumsum", "chunk_scan", "chunk_state", "state_passing"]):
        interesting.append((name, obj))

for name, obj in interesting:
    print(f"\n--- {name} ---")
    print("Type:", type(obj))
    target = getattr(obj, "fn", obj)
    try:
        print("Source file:", inspect.getfile(target))
        source = inspect.getsource(target)
        lines = source.splitlines()
        # Show the relevant arithmetic and its surrounding context.
        matches = [
            i for i, line in enumerate(lines)
            if any(term in line.lower() for term in
                   ["softplus", "dt_bias", "dt_limit", "dA", "cumsum",
                    "tl.exp", "tl.minimum", "tl.maximum"])
        ]
        shown = set()
        for i in matches:
            for j in range(max(0, i - 3), min(len(lines), i + 4)):
                if j not in shown:
                    print(f"{j+1:4d}: {lines[j]}")
                    shown.add(j)
            print("   ...")
    except Exception as e:
        print("Could not inspect source:", repr(e))

print("\n" + "=" * 75)
print("2. CAPTURE RECORD STRUCTURE")
print("=" * 75)

print("Number of records:", len(actual_scan_records))
if actual_scan_records:
    rec = actual_scan_records[0]
    print("Record type:", type(rec))
    if isinstance(rec, dict):
        print("Record keys:", list(rec.keys()))
        for key, value in rec.items():
            if hasattr(value, "shape"):
                print(
                    f"{key}: shape={tuple(value.shape)}, "
                    f"dtype={getattr(value, 'dtype', None)}, "
                    f"device={getattr(value, 'device', None)}"
                )
            else:
                print(f"{key}: {type(value).__name__} = {value}")
    else:
        print("Record preview:", repr(rec)[:2000])

print("\n" + "=" * 75)
print("3. CAPTURED DECAY SANITY CHECK")
print("=" * 75)

if isinstance(actual_scan_records[0], dict):
    rec = actual_scan_records[0]
    for key in ["dt", "dt_bias", "A", "dA", "decay", "dt_effective"]:
        if key in rec and hasattr(rec[key], "shape"):
            t = rec[key].detach().float()
            print(
                f"{key}: min={t.min().item():.8g}, "
                f"max={t.max().item():.8g}, "
                f"mean={t.mean().item():.8g}, "
                f"finite={torch.isfinite(t).all().item()}"
            )

1. TRITON SCAN MODULE
Module file: /usr/local/lib/python3.13/dist-packages/mamba_ssm/ops/triton/ssd_combined.py

--- _chunk_cumsum_fwd ---
Type: <class 'function'>
Source file: /usr/local/lib/python3.13/dist-packages/mamba_ssm/ops/triton/ssd_chunk_state.py
   1: def _chunk_cumsum_fwd(dt, A, chunk_size, dt_bias=None, dt_softplus=False, dt_limit=(0.0, float("inf"))):
   2:     batch, seqlen, nheads = dt.shape
   3:     assert A.shape == (nheads,)
   4:     if dt_bias is not None:
   ...
   5:         assert dt_bias.shape == (nheads,)
   6:     nchunks = math.ceil(seqlen / chunk_size)
   7:     dt_out = torch.empty(batch, nheads, nchunks, chunk_size, device=dt.device, dtype=torch.float32)
   ...
   8:     dA_cumsum = torch.empty(batch, nheads, nchunks, chunk_size, device=dt.device, dtype=torch.float32)
   ...
   9:     grid_chunk_cs = lambda META: (batch, nchunks, triton.cdiv(nheads, META['BLOCK_SIZE_H']))
  10:     with torch.cuda.device(dt.device.index):
  11:         _chunk_cumsum_fwd_

In [68]:

import numpy as np
import pandas as pd
import torch

print("=" * 75)
print("DECAY REPRESENTATION DIAGNOSTIC")
print("=" * 75)

# Inspect the first captured record to establish its stored fields.
record = actual_scan_records[0]

print("\nCaptured record keys:")
print(list(record.keys()))

for key, value in record.items():
    if torch.is_tensor(value):
        print(
            f"{key}: shape={tuple(value.shape)}, "
            f"dtype={value.dtype}, "
            f"min={value.float().min().item():.6g}, "
            f"max={value.float().max().item():.6g}"
        )
    elif isinstance(value, (int, float, str, bool)):
        print(f"{key}: {value}")

print("\nAudit dataframe columns:")
print(list(decay_audit_df.columns) if "decay_audit_df" in globals()
      else "decay_audit_df not found")

print("\nPlease also inspect the audit code's tensor construction:")
print("Which variables were used for captured_dA, kernel_dA, and expected_dA?")


DECAY REPRESENTATION DIAGNOSTIC

Captured record keys:
['dt', 'A', 'dA']
dt: shape=(1, 128, 24), dtype=torch.float32, min=0.0162768, max=0.110125
A: shape=(24,), dtype=torch.float32, min=-5.06356, max=-0.270728
dA: shape=(1, 128, 24), dtype=torch.float32, min=-0.314669, max=-0.0152368

Audit dataframe columns:
decay_audit_df not found

Please also inspect the audit code's tensor construction:
Which variables were used for captured_dA, kernel_dA, and expected_dA?


In [69]:

import torch
import torch.nn.functional as F
from mamba_ssm.ops.triton.ssd_chunk_state import _chunk_cumsum_fwd

record = actual_scan_records[0]
mixer = model.backbone.layers[0].mixer
device = next(model.parameters()).device

dt_raw = record["dt"].to(device).float()
A = record["A"].to(device).float()
captured_dA = record["dA"].to(device).float()
dt_bias = mixer.dt_bias.detach().to(device).float()

# Independently reconstruct the per-token increments.
dt_biased = dt_raw + dt_bias.view(1, 1, -1)
dt_expected = torch.where(
    dt_biased <= 20.0,
    F.softplus(dt_biased),
    dt_biased
).clamp(min=0.0)

expected_dA = dt_expected * A.view(1, 1, -1)

# Run the actual Triton cumsum wrapper.
with torch.no_grad():
    kernel_outputs = _chunk_cumsum_fwd(
        dt_raw,
        A,
        chunk_size=256,
        dt_bias=dt_bias,
        dt_softplus=True,
        dt_limit=(0.0, float("inf"))
    )

print("=" * 70)
print("ACTUAL TRITON OUTPUT INSPECTION")
print("=" * 70)

print("\nKernel return type:", type(kernel_outputs))
print("Number of returned values:", len(kernel_outputs))

for i, tensor in enumerate(kernel_outputs):
    print(f"\nOutput {i}:")
    print("Shape:", tuple(tensor.shape))
    print("Dtype:", tensor.dtype)
    print("Min:", tensor.float().min().item())
    print("Max:", tensor.float().max().item())
    print("Mean:", tensor.float().mean().item())
    print("First values:", tensor.float().flatten()[:10].cpu().tolist())

print("\nCaptured tensors:")
for name, tensor in [
    ("dt_raw", dt_raw),
    ("captured_dA", captured_dA),
    ("expected_dA", expected_dA),
]:
    print(
        name,
        "shape =", tuple(tensor.shape),
        "min =", tensor.min().item(),
        "max =", tensor.max().item(),
        "mean =", tensor.mean().item()
    )

print("\nDirect captured-vs-expected comparison:")
print("Max abs error:", (captured_dA - expected_dA).abs().max().item())
print("Mean abs error:", (captured_dA - expected_dA).abs().mean().item())

# Check the expected cumulative sum in the natural [batch, heads, time] layout.
expected_layout = expected_dA.transpose(1, 2)
expected_cumsum = torch.cumsum(expected_layout, dim=-1)

print("\nExpected cumulative sum:")
print("Shape:", tuple(expected_cumsum.shape))
print("Min:", expected_cumsum.min().item())
print("Max:", expected_cumsum.max().item())
print("First 10 values:", expected_cumsum[0, 0, :10].cpu().tolist())

# Compare each kernel output against the expected per-token or cumulative tensor
# only when the shapes actually match.
for i, tensor in enumerate(kernel_outputs):
    print(f"\nOutput {i} shape-match checks:")
    for name, target in [
        ("expected_dA", expected_dA),
        ("expected_layout", expected_layout),
        ("expected_cumsum", expected_cumsum),
    ]:
        if tensor.shape == target.shape:
            diff = (tensor.float() - target.float()).abs()
            print(
                f"  vs {name}: max={diff.max().item():.8e}, "
                f"mean={diff.mean().item():.8e}"
            )
        else:
            print(f"  vs {name}: shape mismatch {tuple(target.shape)}")


ACTUAL TRITON OUTPUT INSPECTION

Kernel return type: <class 'tuple'>
Number of returned values: 2

Output 0:
Shape: (1, 24, 1, 256)
Dtype: torch.float32
Min: -22.933218002319336
Max: -0.02556370198726654
Mean: -3.312887191772461
First values: [-0.025981320068240166, -0.051982227712869644, -0.07791000604629517, -0.1042582169175148, -0.13019026815891266, -0.15594899654388428, -0.18179664015769958, -0.20786483585834503, -0.23382148146629333, -0.25999218225479126]

Output 1:
Shape: (1, 24, 1, 256)
Dtype: torch.float32
Min: 0.0
Max: 0.11248598992824554
Mean: 0.028942391276359558
First values: [0.06712768226861954, 0.06717829406261444, 0.06698933988809586, 0.06807561963796616, 0.0670003816485405, 0.0665525570511818, 0.06678229570388794, 0.06735216081142426, 0.06706392019987106, 0.0676170289516449]

Captured tensors:
dt_raw shape = (1, 128, 24) min = 0.016276828944683075 max = 0.11012475937604904 mean = 0.059129685163497925
captured_dA shape = (1, 128, 24) min = -0.3146694004535675 max = -0.0

In [71]:

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd

device = next(model.parameters()).device
rows = []

for record_idx, record in enumerate(actual_scan_records):
    layer = record_idx % 24
    doc = record_idx // 24

    mixer = model.backbone.layers[layer].mixer

    dt_raw = record["dt"].to(device).float()
    A = record["A"].to(device).float()
    bias = mixer.dt_bias.detach().to(device).float()

    dt_biased = dt_raw + bias.view(1, 1, -1)
    dt_eff = torch.where(
        dt_biased <= 20,
        F.softplus(dt_biased),
        dt_biased
    ).clamp(min=0)

    dA = dt_eff * A.view(1, 1, -1)
    dA_layout = dA.transpose(1, 2)

    # Two independent cumulative references.
    cumsum_fp32 = torch.cumsum(dA_layout, dim=-1)
    cumsum_fp64 = torch.cumsum(
        dA_layout.double(), dim=-1
    ).float()

    with torch.no_grad():
        kernel_cumsum, kernel_dt = _chunk_cumsum_fwd(
            dt_raw, A, chunk_size=256,
            dt_bias=bias,
            dt_softplus=True,
            dt_limit=(0.0, float("inf"))
        )

    kernel_valid = kernel_cumsum[:, :, 0, :128]
    fp32_error = (kernel_valid - cumsum_fp32).abs()
    fp64_error = (kernel_valid - cumsum_fp64).abs()

    reference_scale = cumsum_fp64.abs().max().item()
    relative_max = (
        fp64_error.max().item() / max(reference_scale, 1e-30)
    )

    rows.append({
        "doc": doc,
        "layer": layer,
        "max_abs_dA": dA.abs().max().item(),
        "mean_abs_dA": dA.abs().mean().item(),
        "max_abs_cumsum": reference_scale,
        "kernel_vs_fp32_max": fp32_error.max().item(),
        "kernel_vs_fp64_max": fp64_error.max().item(),
        "kernel_vs_fp64_mean": fp64_error.mean().item(),
        "relative_to_max_cumsum": relative_max
    })

precision_df = pd.DataFrame(rows)

print("=" * 75)
print("CUMULATIVE PRECISION DIAGNOSTIC")
print("=" * 75)

print("\nLayer-level summary:")
summary = precision_df.groupby("layer").agg({
    "max_abs_dA": "max",
    "mean_abs_dA": "mean",
    "max_abs_cumsum": "max",
    "kernel_vs_fp32_max": "max",
    "kernel_vs_fp64_max": "max",
    "kernel_vs_fp64_mean": "mean",
    "relative_to_max_cumsum": "max"
})

print(summary.to_string(float_format=lambda x: f"{x:.6e}"))

print("\nWorst 12 calls against FP64 reference:")
print(
    precision_df.nlargest(12, "kernel_vs_fp64_max").to_string(
        index=False,
        float_format=lambda x: f"{x:.6e}"
    )
)

print("\nLayer 16 detailed values:")
print(
    precision_df[precision_df.layer == 16].to_string(
        index=False,
        float_format=lambda x: f"{x:.6e}"
    )
)


CUMULATIVE PRECISION DIAGNOSTIC

Layer-level summary:
        max_abs_dA  mean_abs_dA  max_abs_cumsum  kernel_vs_fp32_max  kernel_vs_fp64_max  kernel_vs_fp64_mean  relative_to_max_cumsum
layer                                                                                                                               
0     1.842509e-01 3.441633e-02    2.293321e+01        3.814697e-06        5.722046e-06         1.825623e-07            2.495091e-07

Worst 12 calls against FP64 reference:
 doc  layer   max_abs_dA  mean_abs_dA  max_abs_cumsum  kernel_vs_fp32_max  kernel_vs_fp64_max  kernel_vs_fp64_mean  relative_to_max_cumsum
   0      0 1.842509e-01 3.441633e-02    2.293321e+01        3.814697e-06        5.722046e-06         1.825623e-07            2.495091e-07

Layer 16 detailed values:
Empty DataFrame
Columns: [doc, layer, max_abs_dA, mean_abs_dA, max_abs_cumsum, kernel_vs_fp32_max, kernel_vs_fp64_max, kernel_vs_fp64_mean, relative_to_max_cumsum]
Index: []


In [74]:

import torch
import numpy as np
import pandas as pd
import json
import os

print("=" * 80)
print("MAMBA-2 CHECKPOINT INTEGRITY CHECK")
print("=" * 80)

print("\nModel configuration:")
config = model.config

for key in [
    "d_model", "n_layer", "n_layers", "d_state",
    "d_conv", "expand", "headdim", "nheads",
    "ngroups", "vocab_size", "chunk_size"
]:
    print(f"{key}: {getattr(config, key, 'not present')}")

print("\nBackbone configuration:")
backbone_config = getattr(model.backbone, "config", None)
print(backbone_config)

print("\nAll layer A_log parameters:")
rows = []

for layer_idx, block in enumerate(model.backbone.layers):
    mixer = block.mixer

    a_log = mixer.A_log.detach().float().cpu()
    A = -torch.exp(a_log)
    bias = mixer.dt_bias.detach().float().cpu()

    rows.append({
        "layer": layer_idx,
        "A_log_min": a_log.min().item(),
        "A_log_max": a_log.max().item(),
        "A_log_mean": a_log.mean().item(),
        "A_log_positive_heads": int((a_log > 0).sum()),
        "A_min": A.min().item(),
        "A_max": A.max().item(),
        "A_nonfinite": int((~torch.isfinite(A)).sum()),
        "bias_min": bias.min().item(),
        "bias_max": bias.max().item(),
        "A_log_dtype": str(mixer.A_log.dtype),
        "A_log_shape": tuple(mixer.A_log.shape)
    })

parameter_df = pd.DataFrame(rows)
print(parameter_df.to_string(index=False))

print("\nState-dict parameter names containing A_log:")
for key, value in model.state_dict().items():
    if "A_log" in key:
        print(
            key,
            "shape=", tuple(value.shape),
            "dtype=", value.dtype,
            "min=", value.float().min().item(),
            "max=", value.float().max().item()
        )

print("\nModel parameter count:")
print(sum(p.numel() for p in model.parameters()))

print("\nModel class:")
print(type(model))

print("\nModel config class:")
print(type(config))

print("\nModel config dictionary:")
try:
    print(config.to_dict())
except Exception as e:
    print("Could not convert config:", repr(e))


MAMBA-2 CHECKPOINT INTEGRITY CHECK

Model configuration:
d_model: 768
n_layer: 24
n_layers: not present
d_state: not present
d_conv: not present
expand: not present
headdim: not present
nheads: not present
ngroups: not present
vocab_size: 50277
chunk_size: not present

Backbone configuration:
None

All layer A_log parameters:
 layer  A_log_min  A_log_max  A_log_mean  A_log_positive_heads         A_min     A_max  A_nonfinite  bias_min  bias_max   A_log_dtype A_log_shape
     0  -1.306641   1.622070   -0.624115                     1     -5.063562 -0.270728            0 -3.357422 -2.238281 torch.float16       (24,)
     1  -4.082031  -0.300293   -2.036641                     0     -0.740601 -0.016873            0 -4.195312  4.031250 torch.float16       (24,)
     2  -5.718750   2.859375   -1.342287                     7    -17.450617 -0.003284            0 -8.656250  2.845703 torch.float16       (24,)
     3  -3.423828  -0.126709   -2.164154                     0     -0.880990 -0.032587  

In [76]:

import os
import glob
import json

print("=" * 80)
print("INSPECTING ALL CHECKPOINT FILES")
print("=" * 80)

for root, dirs, files in os.walk(snapshot):
    for filename in files:
        path = os.path.join(root, filename)
        print(
            filename,
            "| size:", os.path.getsize(path),
            "| symlink:", os.path.islink(path),
            "| target:", os.path.realpath(path)
        )

print("\nCheckpoint weight candidates:")

patterns = [
    "*.bin",
    "*.pt",
    "*.pth",
    "*.safetensors",
    "*.index.json"
]

weight_candidates = []

for pattern in patterns:
    matches = glob.glob(os.path.join(snapshot, pattern))
    weight_candidates.extend(matches)

for path in weight_candidates:
    print(os.path.basename(path))

print("\nSnapshot directory:", snapshot)


INSPECTING ALL CHECKPOINT FILES
pytorch_model.bin | size: 258059314 | symlink: True | target: /root/.cache/huggingface/hub/blobs/cb/cb257c10c0a07b29fe1e45b4b7eaeb3854e83ed9e2216f1f53363c5fa887b462
config.json | size: 330 | symlink: True | target: /root/.cache/huggingface/hub/models--state-spaces--mamba2-130m/blobs/24674500a4d40c8d0fc5232c41af977a1649a8fb

Checkpoint weight candidates:
pytorch_model.bin

Snapshot directory: /root/.cache/huggingface/hub/models--state-spaces--mamba2-130m/snapshots/3a5aea0c25d0fb43cc360e2c2aac82c26e3eed49


In [77]:

import torch
import pandas as pd

checkpoint_path = os.path.join(snapshot, "pytorch_model.bin")

print("Loading original checkpoint on CPU...")

# weights_only=False may be needed for older PyTorch checkpoint formats.
try:
    raw_checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )
except Exception as e:
    print("Safe loading failed:", repr(e))
    raise

print("Checkpoint loaded.")

# Handle common checkpoint dictionary formats
if "state_dict" in raw_checkpoint:
    original_state = raw_checkpoint["state_dict"]
elif "model" in raw_checkpoint:
    original_state = raw_checkpoint["model"]
else:
    original_state = raw_checkpoint

print("Number of original state-dict entries:", len(original_state))

# Extract original A_log tensors
original_a_logs = {
    key: value
    for key, value in original_state.items()
    if "A_log" in key
}

print("Original A_log tensor count:", len(original_a_logs))

comparison_rows = []

for layer_idx in range(len(model.backbone.layers)):
    key = f"backbone.layers.{layer_idx}.mixer.A_log"

    if key not in original_a_logs:
        print("MISSING:", key)
        continue

    original = original_a_logs[key].detach().float().cpu()

    loaded = (
        model.backbone.layers[layer_idx]
        .mixer.A_log.detach().float().cpu()
    )

    comparison_rows.append({
        "layer": layer_idx,
        "original_dtype": str(original_a_logs[key].dtype),
        "loaded_dtype": str(
            model.backbone.layers[layer_idx].mixer.A_log.dtype
        ),
        "original_min": original.min().item(),
        "original_max": original.max().item(),
        "loaded_min": loaded.min().item(),
        "loaded_max": loaded.max().item(),
        "max_abs_difference": (original - loaded).abs().max().item(),
        "mean_abs_difference": (original - loaded).abs().mean().item(),
        "exact_match": torch.equal(original, loaded)
    })

comparison_df = pd.DataFrame(comparison_rows)

print("\n" + "=" * 80)
print("ORIGINAL CHECKPOINT VS LOADED MODEL")
print("=" * 80)

print(comparison_df.to_string(index=False))

print("\nOverall comparison:")
if len(comparison_df):
    print(
        "Maximum parameter difference:",
        comparison_df["max_abs_difference"].max()
    )
    print(
        "Layers with exact matches:",
        int(comparison_df["exact_match"].sum()),
        "/",
        len(comparison_df)
    )

print("\nExtreme original checkpoint parameters:")

for layer_idx in [4, 6, 10, 14, 16, 19, 20, 22, 23]:
    key = f"backbone.layers.{layer_idx}.mixer.A_log"

    if key in original_a_logs:
        values = original_a_logs[key].float()
        print(f"\nLayer {layer_idx}:")
        print(values.tolist())


Loading original checkpoint on CPU...
Checkpoint loaded.
Number of original state-dict entries: 219
Original A_log tensor count: 24

ORIGINAL CHECKPOINT VS LOADED MODEL
 layer original_dtype  loaded_dtype  original_min  original_max  loaded_min  loaded_max  max_abs_difference  mean_abs_difference  exact_match
     0  torch.float16 torch.float16     -1.306641      1.622070   -1.306641    1.622070                 0.0                  0.0         True
     1  torch.float16 torch.float16     -4.082031     -0.300293   -4.082031   -0.300293                 0.0                  0.0         True
     2  torch.float16 torch.float16     -5.718750      2.859375   -5.718750    2.859375                 0.0                  0.0         True
     3  torch.float16 torch.float16     -3.423828     -0.126709   -3.423828   -0.126709                 0.0                  0.0         True
     4  torch.float16 torch.float16     -7.828125     10.500000   -7.828125   10.500000                 0.0              

In [78]:

import hashlib
import os
import torch
import pandas as pd

# 1. Verify cached file against its content-addressed blob name
blob_path = os.path.realpath(checkpoint_path)
expected_hash = os.path.basename(blob_path)

sha256 = hashlib.sha256()

with open(checkpoint_path, "rb") as f:
    for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
        sha256.update(chunk)

actual_hash = sha256.hexdigest()

print("CHECKPOINT FILE INTEGRITY")
print("Expected blob hash:", expected_hash)
print("Computed SHA-256:  ", actual_hash)
print("Hash matches:", actual_hash == expected_hash)

# 2. Compare original and loaded dt_bias parameters
rows = []

for layer_idx in range(len(model.backbone.layers)):
    key = f"backbone.layers.{layer_idx}.mixer.dt_bias"

    if key not in original_state:
        print("Missing original parameter:", key)
        continue

    original = original_state[key].detach().float().cpu()
    loaded = (
        model.backbone.layers[layer_idx]
        .mixer.dt_bias.detach().float().cpu()
    )

    rows.append({
        "layer": layer_idx,
        "original_dtype": str(original_state[key].dtype),
        "loaded_dtype": str(
            model.backbone.layers[layer_idx].mixer.dt_bias.dtype
        ),
        "max_abs_difference": (original - loaded).abs().max().item(),
        "exact_match": torch.equal(original, loaded)
    })

dt_bias_df = pd.DataFrame(rows)

print("\nDT_BIAS CHECKPOINT COMPARISON")
print(dt_bias_df.to_string(index=False))

print("\nExact matches:",
      int(dt_bias_df["exact_match"].sum()), "/", len(dt_bias_df))


CHECKPOINT FILE INTEGRITY
Expected blob hash: cb257c10c0a07b29fe1e45b4b7eaeb3854e83ed9e2216f1f53363c5fa887b462
Computed SHA-256:   f786aff903d8b6f2cef67ff2b1db06a9f44e7780340e6ea6d3dead16b52fc501
Hash matches: False

DT_BIAS CHECKPOINT COMPARISON
 layer original_dtype  loaded_dtype  max_abs_difference  exact_match
     0  torch.float16 torch.float16                 0.0         True
     1  torch.float16 torch.float16                 0.0         True
     2  torch.float16 torch.float16                 0.0         True
     3  torch.float16 torch.float16                 0.0         True
     4  torch.float16 torch.float16                 0.0         True
     5  torch.float16 torch.float16                 0.0         True
     6  torch.float16 torch.float16                 0.0         True
     7  torch.float16 torch.float16                 0.0         True
     8  torch.float16 torch.float16                 0.0         True
     9  torch.float16 torch.float16                 0.0        

In [79]:

import os
import glob
import hashlib

print("=" * 75)
print("HUGGING FACE CACHE METADATA")
print("=" * 75)

print("Checkpoint path:", checkpoint_path)
print("Is symlink:", os.path.islink(checkpoint_path))
print("Symlink target:", os.readlink(checkpoint_path))
print("Resolved path:", os.path.realpath(checkpoint_path))
print("File size:", os.path.getsize(checkpoint_path))
print("Cache identifier:", os.path.basename(os.path.realpath(checkpoint_path)))

# Inspect nearby cache metadata
metadata_root = os.path.expanduser("~/.cache/huggingface")

for folder in [
    os.path.join(metadata_root, "hub", "models--state-spaces--mamba2-130m"),
    os.path.join(metadata_root, "download")
]:
    print("\nDirectory:", folder)

    if os.path.isdir(folder):
        for root, dirs, files in os.walk(folder):
            for filename in files:
                if (
                    "pytorch_model" in filename
                    or filename.endswith(".metadata")
                    or filename == "main"
                ):
                    path = os.path.join(root, filename)
                    print(path)

                    if filename.endswith(".metadata") or filename == "main":
                        try:
                            with open(path, "r") as f:
                                print("Contents:", f.read()[:1000])
                        except Exception as e:
                            print("Could not read:", repr(e))

# Check the resolved file's SHA-256 independently
h = hashlib.sha256()

with open(os.path.realpath(checkpoint_path), "rb") as f:
    for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
        h.update(chunk)

print("\nComputed SHA-256:", h.hexdigest())


HUGGING FACE CACHE METADATA
Checkpoint path: /root/.cache/huggingface/hub/models--state-spaces--mamba2-130m/snapshots/3a5aea0c25d0fb43cc360e2c2aac82c26e3eed49/pytorch_model.bin
Is symlink: True
Symlink target: ../../blobs/f786aff903d8b6f2cef67ff2b1db06a9f44e7780340e6ea6d3dead16b52fc501
Resolved path: /root/.cache/huggingface/hub/blobs/cb/cb257c10c0a07b29fe1e45b4b7eaeb3854e83ed9e2216f1f53363c5fa887b462
File size: 258059314
Cache identifier: cb257c10c0a07b29fe1e45b4b7eaeb3854e83ed9e2216f1f53363c5fa887b462

Directory: /root/.cache/huggingface/hub/models--state-spaces--mamba2-130m
/root/.cache/huggingface/hub/models--state-spaces--mamba2-130m/refs/main
Contents: 3a5aea0c25d0fb43cc360e2c2aac82c26e3eed49
/root/.cache/huggingface/hub/models--state-spaces--mamba2-130m/snapshots/3a5aea0c25d0fb43cc360e2c2aac82c26e3eed49/pytorch_model.bin

Directory: /root/.cache/huggingface/download

Computed SHA-256: f786aff903d8b6f2cef67ff2b1db06a9f44e7780340e6ea6d3dead16b52fc501


In [81]:

import inspect
import mamba_ssm.modules.mamba2 as mamba2_mod

scan_fn = mamba2_mod.mamba_chunk_scan_combined

print("Scan function:", scan_fn)
print("Module:", scan_fn.__module__)

try:
    print("\nFunction signature:")
    print(inspect.signature(scan_fn))
except Exception as e:
    print("Signature unavailable:", repr(e))

print("\nFunction source:")
try:
    print(inspect.getsource(scan_fn)[:12000])
except Exception as e:
    print("Source unavailable:", repr(e))

print("\nCumulative-scan implementation:")
try:
    print(
        inspect.getsource(mamba2_mod._chunk_cumsum_fwd)[:12000]
    )
except Exception as e:
    print("Could not inspect _chunk_cumsum_fwd:", repr(e))

print("\nRelevant module functions:")
for name in dir(mamba2_mod):
    if "cumsum" in name.lower() or "chunk_scan" in name.lower():
        obj = getattr(mamba2_mod, name)
        print(name, type(obj))


Scan function: <function mamba_chunk_scan_combined at 0x79be5c459c60>
Module: mamba_ssm.ops.triton.ssd_combined

Function signature:
(x, dt, A, B, C, chunk_size, D=None, z=None, dt_bias=None, initial_states=None, seq_idx=None, cu_seqlens=None, dt_softplus=False, dt_limit=(0.0, inf), return_final_states=False, return_varlen_states=False)

Function source:
def mamba_chunk_scan_combined(x, dt, A, B, C, chunk_size, D=None, z=None, dt_bias=None, initial_states=None, seq_idx=None, cu_seqlens=None, dt_softplus=False, dt_limit=(0.0, float("inf")), return_final_states=False, return_varlen_states=False):
    """
    Argument:
        x: (batch, seqlen, nheads, headdim)
        dt: (batch, seqlen, nheads)
        A: (nheads)
        B: (batch, seqlen, ngroups, dstate)
        C: (batch, seqlen, ngroups, dstate)
        chunk_size: int
        D: (nheads, headdim) or (nheads,)
        z: (batch, seqlen, nheads, headdim)
        dt_bias: (nheads,)
        initial_states: (batch, nheads, headdim, ds

In [82]:

import inspect
import mamba_ssm.modules.mamba2 as mamba2_mod
import mamba_ssm.ops.triton.ssd_combined as ssd_mod

print("=" * 75)
print("1. CUSTOM AUTOGRAD FUNCTION")
print("=" * 75)

scan_class = ssd_mod.MambaChunkScanCombinedFn

for method_name in ["forward", "backward"]:
    method = getattr(scan_class, method_name, None)
    print(f"\n--- {method_name} ---")
    if method is not None:
        try:
            print(inspect.getsource(method)[:18000])
        except Exception as e:
            print("Source unavailable:", repr(e))

print("\n" + "=" * 75)
print("2. MAMBA-2 FORWARD CALL SITE")
print("=" * 75)

try:
    source = inspect.getsource(mamba2_mod.Mamba2.forward)
    lines = source.splitlines()

    matches = [
        i for i, line in enumerate(lines)
        if "mamba_chunk_scan_combined" in line
    ]

    for i in matches:
        start = max(0, i - 18)
        end = min(len(lines), i + 25)
        print("\n".join(
            f"{j+1:4d}: {lines[j]}"
            for j in range(start, end)
        ))
except Exception as e:
    print("Could not inspect Mamba2.forward:", repr(e))

print("\n" + "=" * 75)
print("3. EFFECTIVE DECAY SEMANTICS")
print("=" * 75)

print("Scan function module:", mamba2_mod.mamba_chunk_scan_combined.__module__)
print("Scan function is same object:",
      mamba2_mod.mamba_chunk_scan_combined is ssd_mod.mamba_chunk_scan_combined)

print("\nModel scan-related configuration:")
for layer_idx, block in enumerate(model.backbone.layers[:2]):
    mixer = block.mixer
    print(
        f"Layer {layer_idx}: "
        f"nheads={getattr(mixer, 'nheads', None)}, "
        f"headdim={getattr(mixer, 'headdim', None)}, "
        f"ngroups={getattr(mixer, 'ngroups', None)}, "
        f"chunk_size={getattr(mixer, 'chunk_size', None)}"
    )


1. CUSTOM AUTOGRAD FUNCTION

--- forward ---
    @staticmethod
    def forward(ctx, x, dt, A, B, C, chunk_size, D=None, z=None, dt_bias=None, initial_states=None, seq_idx=None, cu_seqlens=None, dt_softplus=False, dt_limit=(0.0, float("inf")), return_final_states=False, return_varlen_states=False):
        ctx.dt_dtype = dt.dtype
        if not return_varlen_states:
            cu_seqlens = None
        else:
            assert cu_seqlens is not None, "cu_seqlens must be provided if return_varlen_states is True"
        out, out_x, dt_out, dA_cumsum, states, final_states, *rest = _mamba_chunk_scan_combined_fwd(x, dt, A, B, C, chunk_size, D=D, z=z, dt_bias=dt_bias, initial_states=initial_states, seq_idx=seq_idx, cu_seqlens=cu_seqlens, dt_softplus=dt_softplus, dt_limit=dt_limit)
        ctx.save_for_backward(out if z is None else out_x, x, dt, dA_cumsum, A, B, C, D, z, dt_bias, initial_states, seq_idx)
        ctx.dt_softplus = dt_softplus
        ctx.chunk_size = chunk_size
        ctx.d

In [83]:

import inspect
import mamba_ssm.ops.triton.ssd_combined as ssd

print("=" * 75)
print("SSD COMBINED MODULE")
print("=" * 75)

print("Module file:", ssd.__file__)

# Inspect the actual forward implementation.
for name in [
    "_mamba_chunk_scan_combined_fwd",
    "MambaChunkScanCombinedFn",
]:
    obj = getattr(ssd, name, None)

    print(f"\n{'=' * 75}")
    print(name, ":", obj)

    if obj is None:
        continue

    try:
        print(inspect.getsource(obj)[:22000])
    except Exception as e:
        print("Source inspection failed:", repr(e))

# Identify the actual cumulative-scan function references.
print("\n" + "=" * 75)
print("CUMULATIVE-SCAN REFERENCES")
print("=" * 75)

fwd = getattr(ssd, "_mamba_chunk_scan_combined_fwd", None)

if fwd is not None:
    for name, obj in fwd.__globals__.items():
        if "cumsum" in name.lower() or "chunk" in name.lower():
            if callable(obj):
                print("\nName:", name)
                print("Object:", obj)
                try:
                    print("Module:", obj.__module__)
                    print("Source file:", inspect.getsourcefile(obj))
                    print(inspect.getsource(obj)[:12000])
                except Exception as e:
                    print("Inspection:", repr(e))


SSD COMBINED MODULE
Module file: /usr/local/lib/python3.13/dist-packages/mamba_ssm/ops/triton/ssd_combined.py

_mamba_chunk_scan_combined_fwd : <function _mamba_chunk_scan_combined_fwd at 0x79be5c4599e0>
def _mamba_chunk_scan_combined_fwd(x, dt, A, B, C, chunk_size, D=None, z=None, dt_bias=None, initial_states=None, seq_idx=None, cu_seqlens=None, dt_softplus=False, dt_limit=(0.0, float("inf"))):
    batch, seqlen, nheads, headdim = x.shape
    _, _, ngroups, dstate = B.shape
    assert nheads % ngroups == 0
    assert B.shape == (batch, seqlen, ngroups, dstate)
    assert x.shape == (batch, seqlen, nheads, headdim)
    assert dt.shape == (batch, seqlen, nheads)
    assert A.shape == (nheads,)
    assert C.shape == B.shape
    if z is not None:
        assert z.shape == x.shape
    if D is not None:
        assert D.shape == (nheads, headdim) or D.shape == (nheads,)
    if seq_idx is not None:
        assert seq_idx.shape == (batch, seqlen)
    if B.stride(-1) != 1:
        B = B.contig

In [ ]:
import inspect
import mamba_ssm.ops.triton.ssd_combined as ssd_combined

print("Module:", ssd_combined.__name__)

print("\nFunction signature:")
print(inspect.signature(ssd_combined._chunk_cumsum_fwd))

print("\nFunction source:")
print(inspect.getsource(ssd_combined._chunk_cumsum_fwd))

print("\nForward function uses this exact global:")
print(
    ssd_combined._mamba_chunk_scan_combined_fwd
        .__globals__["_chunk_cumsum_fwd"]
    is ssd_combined._chunk_cumsum_fwd
)

In [89]:

import inspect
import mamba_ssm.ops.triton.ssd_chunk_scan as scan_mod

kernel = scan_mod._chunk_scan_fwd_kernel
jit_fn = kernel.fn

print("JIT FUNCTION TYPE:", type(jit_fn))
print("JIT FUNCTION NAME:", getattr(jit_fn, "__name__", None))

print("\nSOURCE FILE:")
print(getattr(jit_fn, "src", "Source unavailable")[:5000]
      if isinstance(getattr(jit_fn, "src", None), str)
      else "No source string available")

print("\nAUTOTUNER CONFIGURATION:")
for cfg in kernel.configs:
    print(
        cfg.kwargs,
        "warps:", cfg.num_warps,
        "stages:", cfg.num_stages
    )

print("\nPRUNING KEYS:", kernel.keys)


JIT FUNCTION TYPE: <class 'triton.runtime.jit.JITFunction'>
JIT FUNCTION NAME: _chunk_scan_fwd_kernel

SOURCE FILE:
def _chunk_scan_fwd_kernel(
    # Pointers to matrices
    cb_ptr, x_ptr, z_ptr, out_ptr, out_x_ptr, dt_ptr, dA_cumsum_ptr, seq_idx_ptr, C_ptr, prev_states_ptr, D_ptr,
    # Matrix dimensions
    chunk_size, hdim, dstate,
    batch, seqlen, nheads_ngroups_ratio,
    # Strides
    stride_cb_batch, stride_cb_chunk, stride_cb_head, stride_cb_csize_m, stride_cb_csize_k,
    stride_x_batch, stride_x_seqlen, stride_x_head, stride_x_hdim,
    stride_z_batch, stride_z_seqlen, stride_z_head, stride_z_hdim,
    stride_out_batch, stride_out_seqlen, stride_out_head, stride_out_hdim,
    stride_dt_batch, stride_dt_chunk, stride_dt_head, stride_dt_csize,
    stride_dA_cs_batch, stride_dA_cs_chunk, stride_dA_cs_head, stride_dA_cs_csize,
    stride_seq_idx_batch, stride_seq_idx_seqlen,
    stride_C_batch, stride_C_seqlen, stride_C_head, stride_C_dstate,
    stride_states_batch, stride_st

In [90]:

import triton
import torch
import gc

kernel = scan_mod._chunk_scan_fwd_kernel

# Keep a backup so the original configuration can be restored.
if not hasattr(kernel, "_shiftscan_original_configs"):
    kernel._shiftscan_original_configs = list(kernel.configs)

# Smaller tile configuration for Tesla T4.
kernel.configs = [
    triton.Config(
        {
            "BLOCK_SIZE_M": 32,
            "BLOCK_SIZE_N": 64,
            "BLOCK_SIZE_K": 32,
        },
        num_warps=2,
        num_stages=2,
    )
]

# Clear cached autotuning choices.
kernel.cache.clear()
kernel.best_config = None

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print("New chunk-scan configuration:")
for cfg in kernel.configs:
    print(
        cfg.kwargs,
        "warps:", cfg.num_warps,
        "stages:", cfg.num_stages
    )

print("GPU:", torch.cuda.get_device_name(0))
print("Shared memory:", torch.cuda.get_device_properties(0).shared_memory_per_block)


New chunk-scan configuration:
{'BLOCK_SIZE_M': 32, 'BLOCK_SIZE_N': 64, 'BLOCK_SIZE_K': 32} warps: 2 stages: 2
GPU: Tesla T4
Shared memory: 49152


In [91]:

model.eval()

test_input = encoded_eval[0].to("cuda")

torch.cuda.synchronize()

with torch.inference_mode():
    test_output = model(test_input)

torch.cuda.synchronize()

print("Baseline forward pass: SUCCESS")

if hasattr(test_output, "logits"):
    logits = test_output.logits
else:
    logits = test_output

print("Output shape:", logits.shape)
print("All finite:", torch.isfinite(logits).all().item())
print("Logit mean:", logits.float().mean().item())
print("Logit std:", logits.float().std().item())


Baseline forward pass: SUCCESS
Output shape: torch.Size([1, 128, 50288])
All finite: True
Logit mean: 2.667745590209961
Logit std: 53.464900970458984


In [92]:

import torch
import torch.nn.functional as F

model.eval()

# Save baseline output from the successful forward pass.
baseline_logits = (
    test_output.logits.detach().clone()
    if hasattr(test_output, "logits")
    else test_output.detach().clone()
)

original_fn = combined_mod._chunk_cumsum_fwd

# Confirm that the model's forward path uses this same function.
assert (
    combined_mod._mamba_chunk_scan_combined_fwd.__globals__["_chunk_cumsum_fwd"]
    is original_fn
), "Unexpected cumsum function reference."

roundtrip_stats = {
    "calls": 0,
    "max_dt_error": 0.0,
}

def roundtrip_cumsum(dt, A, chunk_size, dt_bias=None,
                     dt_softplus=False,
                     dt_limit=(0.0, float("inf"))):

    # First calculate the exact original effective dt and cumsum.
    original_cumsum, original_dt = original_fn(
        dt, A, chunk_size,
        dt_bias=dt_bias,
        dt_softplus=dt_softplus,
        dt_limit=dt_limit
    )

    batch, seqlen, nheads = dt.shape

    # Reconstruct dA from effective dt.
    A_fp32 = A.float().view(1, nheads, 1, 1)
    dA = original_dt.float() * A_fp32

    # Convert reconstructed dA back to effective dt.
    reconstructed_dt = (
        dA.permute(0, 2, 3, 1)
        .reshape(batch, -1, nheads)[:, :seqlen, :]
        / A.float().view(1, 1, nheads)
    ).contiguous()

    dt_error = (
        reconstructed_dt.float()
        - dt_effective_for_check(original_dt, batch, seqlen, nheads)
    ).abs().max().item()

    roundtrip_stats["calls"] += 1
    roundtrip_stats["max_dt_error"] = max(
        roundtrip_stats["max_dt_error"], dt_error
    )

    # Original Triton kernel computes the reconstructed cumsum.
    reconstructed_cumsum, _ = original_fn(
        reconstructed_dt, A, chunk_size,
        dt_bias=None,
        dt_softplus=False,
        dt_limit=(0.0, float("inf"))
    )

    # Preserve original dt output for the downstream scan.
    return reconstructed_cumsum, original_dt


def dt_effective_for_check(original_dt, batch, seqlen, nheads):
    return (
        original_dt.permute(0, 2, 3, 1)
        .reshape(batch, -1, nheads)[:, :seqlen, :]
    )


try:
    combined_mod._chunk_cumsum_fwd = roundtrip_cumsum

    with torch.inference_mode():
        roundtrip_output = model(test_input)

    roundtrip_logits = (
        roundtrip_output.logits
        if hasattr(roundtrip_output, "logits")
        else roundtrip_output
    )

    torch.cuda.synchronize()

finally:
    combined_mod._chunk_cumsum_fwd = original_fn

# Compare outputs.
diff = (
    roundtrip_logits.float() - baseline_logits.float()
)

print("ROUND-TRIP CONTROL")
print("Cumsum calls:", roundtrip_stats["calls"])
print("Maximum effective-dt reconstruction error:",
      roundtrip_stats["max_dt_error"])
print("Logit MAE:", diff.abs().mean().item())
print("Logit RMSE:", diff.square().mean().sqrt().item())
print("Maximum logit error:", diff.abs().max().item())
print("Baseline function restored:",
      combined_mod._chunk_cumsum_fwd is original_fn)
print("All round-trip logits finite:",
      torch.isfinite(roundtrip_logits).all().item())


ROUND-TRIP CONTROL
Cumsum calls: 24
Maximum effective-dt reconstruction error: 9.5367431640625e-07
Logit MAE: 0.04960436001420021
Logit RMSE: 0.07377486675977707
Maximum logit error: 0.3125
Baseline function restored: True
All round-trip logits finite: True


In [93]:

import torch

assert combined_mod._chunk_cumsum_fwd is original_fn

model.eval()

with torch.inference_mode():
    repeat_output = model(test_input)

repeat_logits = (
    repeat_output.logits
    if hasattr(repeat_output, "logits")
    else repeat_output
)

torch.cuda.synchronize()

repeat_diff = repeat_logits.float() - baseline_logits.float()

print("BASELINE REPEATABILITY TEST")
print("Logit MAE:", repeat_diff.abs().mean().item())
print("Logit RMSE:", repeat_diff.square().mean().sqrt().item())
print("Maximum logit error:", repeat_diff.abs().max().item())
print("All finite:", torch.isfinite(repeat_logits).all().item())


BASELINE REPEATABILITY TEST
Logit MAE: 0.0
Logit RMSE: 0.0
Maximum logit error: 0.0
All finite: True


In [94]:

import torch

original_fn = combined_mod._chunk_cumsum_fwd

diagnostic = {
    "calls": 0,
    "dt_max_errors": [],
    "cumsum_max_errors": [],
    "cumsum_mean_errors": [],
}

def diagnostic_cumsum(dt, A, chunk_size, dt_bias=None,
                      dt_softplus=False,
                      dt_limit=(0.0, float("inf"))):

    original_cumsum, original_dt = original_fn(
        dt, A, chunk_size,
        dt_bias=dt_bias,
        dt_softplus=dt_softplus,
        dt_limit=dt_limit
    )

    batch, seqlen, nheads = dt.shape

    A_fp32 = A.float().view(1, nheads, 1, 1)
    dA = original_dt.float() * A_fp32

    reconstructed_dt = (
        dA.permute(0, 2, 3, 1)
        .reshape(batch, -1, nheads)[:, :seqlen, :]
        / A.float().view(1, 1, nheads)
    ).contiguous()

    original_dt_seq = (
        original_dt.permute(0, 2, 3, 1)
        .reshape(batch, -1, nheads)[:, :seqlen, :]
    )

    dt_error = (
        reconstructed_dt - original_dt_seq
    ).abs()

    reconstructed_cumsum, _ = original_fn(
        reconstructed_dt, A, chunk_size,
        dt_bias=None,
        dt_softplus=False,
        dt_limit=(0.0, float("inf"))
    )

    cumsum_error = (
        reconstructed_cumsum.float() - original_cumsum.float()
    ).abs()

    diagnostic["calls"] += 1
    diagnostic["dt_max_errors"].append(dt_error.max().item())
    diagnostic["cumsum_max_errors"].append(cumsum_error.max().item())
    diagnostic["cumsum_mean_errors"].append(cumsum_error.mean().item())

    return reconstructed_cumsum, original_dt


try:
    combined_mod._chunk_cumsum_fwd = diagnostic_cumsum

    with torch.inference_mode():
        diagnostic_output = model(test_input)

    diagnostic_logits = (
        diagnostic_output.logits
        if hasattr(diagnostic_output, "logits")
        else diagnostic_output
    )

    torch.cuda.synchronize()

finally:
    combined_mod._chunk_cumsum_fwd = original_fn

logit_error = (
    diagnostic_logits.float() - baseline_logits.float()
).abs()

print("CUMULATIVE-DECAY DIAGNOSTIC")
print("Calls:", diagnostic["calls"])
print("Maximum dt error:", max(diagnostic["dt_max_errors"]))
print("Maximum cumsum error:", max(diagnostic["cumsum_max_errors"]))
print("Mean cumsum error across calls:",
      sum(diagnostic["cumsum_mean_errors"]) / len(diagnostic["cumsum_mean_errors"]))
print("Maximum cumsum error across calls:",
      max(diagnostic["cumsum_max_errors"]))
print("Logit MAE:", logit_error.mean().item())
print("Logit RMSE:", logit_error.square().mean().sqrt().item())
print("Maximum logit error:", logit_error.max().item())
print("Original function restored:",
      combined_mod._chunk_cumsum_fwd is original_fn)


CUMULATIVE-DECAY DIAGNOSTIC
Calls: 24
Maximum dt error: 9.5367431640625e-07
Maximum cumsum error: 0.0625
Mean cumsum error across calls: 2.8027093961953873e-06
Maximum cumsum error across calls: 0.0625
Logit MAE: 0.04960436001420021
Logit RMSE: 0.07377486675977707
Maximum logit error: 0.3125
Original function restored: True


In [96]:

import torch

# Capture original and reconstructed cumsums for each layer.
cumsum_audit = []

def audit_roundtrip_cumsum(
    dt, A, chunk_size,
    dt_bias=None,
    dt_softplus=False,
    dt_limit=(0.0, float("inf"))
):
    original_cumsum, original_dt = original_fn(
        dt, A, chunk_size,
        dt_bias=dt_bias,
        dt_softplus=dt_softplus,
        dt_limit=dt_limit
    )

    A_fp32 = A.float().view(1, -1, 1, 1)
    dA = original_dt.float() * A_fp32

    batch, seqlen, nheads = dt.shape

    reconstructed_dt = (
        dA.permute(0, 2, 3, 1)
        .reshape(batch, -1, nheads)[:, :seqlen, :]
        / A.float().view(1, 1, nheads)
    ).contiguous()

    reconstructed_cumsum, _ = original_fn(
        reconstructed_dt,
        A,
        chunk_size,
        dt_bias=None,
        dt_softplus=False,
        dt_limit=(0.0, float("inf"))
    )

    err = (
        reconstructed_cumsum.float()
        - original_cumsum.float()
    ).abs()

    ref = original_cumsum.float().abs()

    cumsum_audit.append({
        "max_abs": err.max().item(),
        "mean_abs": err.mean().item(),
        "max_reference": ref.max().item(),
        "relative_max": (
            err.max() / ref.max().clamp_min(1e-30)
        ).item(),
        "relative_mean": (
            err / ref.clamp_min(1e-6)
        ).mean().item()
    })

    return reconstructed_cumsum, original_dt


try:
    combined_mod._chunk_cumsum_fwd = audit_roundtrip_cumsum

    with torch.inference_mode():
        _ = model(test_input)

    torch.cuda.synchronize()

finally:
    combined_mod._chunk_cumsum_fwd = original_fn

print("CUMULATIVE-DECAY ROUND-TRIP AUDIT")
print("Calls:", len(cumsum_audit))

for key in cumsum_audit[0]:
    values = [x[key] for x in cumsum_audit]
    print(
        key,
        "mean:", sum(values) / len(values),
        "maximum:", max(values)
    )

print("Original function restored:",
      combined_mod._chunk_cumsum_fwd is original_fn)


CUMULATIVE-DECAY ROUND-TRIP AUDIT
Calls: 24
max_abs mean: 0.0031003554662068686 maximum: 0.0625
mean_abs mean: 2.8027093961953873e-06 maximum: 2.5531589926686138e-05
max_reference mean: 83665.72508386771 maximum: 944887.125
relative_max mean: 5.941755577000609e-08 maximum: 1.2324368015015352e-07
relative_mean mean: 5.21799488339969e-09 maximum: 1.0626257207491108e-08
Original function restored: True


In [97]:

import torch

model.eval()
original_fn = combined_mod._chunk_cumsum_fwd

# Save the original baseline logits.
baseline = baseline_logits.detach().clone()

perturbation_results = {}

def make_perturbed_cumsum(epsilon):
    def patched(
        dt, A, chunk_size,
        dt_bias=None,
        dt_softplus=False,
        dt_limit=(0.0, float("inf"))
    ):
        cumsum, dt_out = original_fn(
            dt, A, chunk_size,
            dt_bias=dt_bias,
            dt_softplus=dt_softplus,
            dt_limit=dt_limit
        )

        # Add a small, controlled perturbation to valid cumsum entries.
        batch, seqlen, nheads = dt.shape
        valid_chunks = (seqlen + chunk_size - 1) // chunk_size

        cumsum = cumsum.clone()
        for chunk in range(valid_chunks):
            start = chunk * chunk_size
            valid = min(chunk_size, seqlen - start)
            cumsum[:, :, chunk, :valid] += epsilon

        return cumsum, dt_out

    return patched


for eps in [1e-7, 1e-6, 1e-5]:
    try:
        combined_mod._chunk_cumsum_fwd = make_perturbed_cumsum(eps)

        with torch.inference_mode():
            out = model(test_input)

        logits = out.logits if hasattr(out, "logits") else out
        diff = logits.float() - baseline.float()

        perturbation_results[eps] = {
            "mae": diff.abs().mean().item(),
            "rmse": diff.square().mean().sqrt().item(),
            "max": diff.abs().max().item()
        }

    finally:
        combined_mod._chunk_cumsum_fwd = original_fn

print("CONTROLLED CUMSUM PERTURBATION RESULTS")
for eps, result in perturbation_results.items():
    print(
        f"epsilon={eps:.0e}",
        result
    )

print("Original function restored:",
      combined_mod._chunk_cumsum_fwd is original_fn)


CONTROLLED CUMSUM PERTURBATION RESULTS
epsilon=1e-07 {'mae': 0.06336431950330734, 'rmse': 0.09133598953485489, 'max': 0.40625}
epsilon=1e-06 {'mae': 0.06127922981977463, 'rmse': 0.09294730424880981, 'max': 0.453125}
epsilon=1e-05 {'mae': 0.05421562120318413, 'rmse': 0.07914099842309952, 'max': 0.46875}
Original function restored: True


In [98]:

import inspect
import mamba_ssm.ops.triton.ssd_chunk_scan as scan_mod
import mamba_ssm.ops.triton.ssd_chunk_state as state_mod

print("CHUNK SCAN SOURCE")
print(scan_mod._chunk_scan_fwd_kernel.fn.src[:14000])

print("\nCHUNK STATE MODULE:")
print(state_mod.__name__)

print("\nAVAILABLE STATE KERNELS:")
print([
    name for name in dir(state_mod)
    if "fwd_kernel" in name.lower()
])


CHUNK SCAN SOURCE
def _chunk_scan_fwd_kernel(
    # Pointers to matrices
    cb_ptr, x_ptr, z_ptr, out_ptr, out_x_ptr, dt_ptr, dA_cumsum_ptr, seq_idx_ptr, C_ptr, prev_states_ptr, D_ptr,
    # Matrix dimensions
    chunk_size, hdim, dstate,
    batch, seqlen, nheads_ngroups_ratio,
    # Strides
    stride_cb_batch, stride_cb_chunk, stride_cb_head, stride_cb_csize_m, stride_cb_csize_k,
    stride_x_batch, stride_x_seqlen, stride_x_head, stride_x_hdim,
    stride_z_batch, stride_z_seqlen, stride_z_head, stride_z_hdim,
    stride_out_batch, stride_out_seqlen, stride_out_head, stride_out_hdim,
    stride_dt_batch, stride_dt_chunk, stride_dt_head, stride_dt_csize,
    stride_dA_cs_batch, stride_dA_cs_chunk, stride_dA_cs_head, stride_dA_cs_csize,
    stride_seq_idx_batch, stride_seq_idx_seqlen,
    stride_C_batch, stride_C_seqlen, stride_C_head, stride_C_dstate,
    stride_states_batch, stride_states_chunk, stride_states_head, stride_states_hdim, stride_states_dstate,
    stride_D_head,
    #

In [100]:

import mamba_ssm.ops.triton.ssd_chunk_state as state_mod
import mamba_ssm.ops.triton.ssd_state_passing as passing_mod

def print_kernel_source(module, kernel_name):
    kernel = getattr(module, kernel_name)

    print("\n" + "=" * 75)
    print("KERNEL:", kernel_name)
    print("TYPE:", type(kernel))
    print("=" * 75)

    jit_fn = getattr(kernel, "fn", kernel)
    source = getattr(jit_fn, "src", None)

    if source is not None:
        print(source)
    else:
        print("Source unavailable.")
        print("Available attributes:", [
            x for x in dir(kernel)
            if not x.startswith("__")
        ])

    if hasattr(kernel, "configs"):
        print("\nAUTOTUNER CONFIGURATIONS:")
        for cfg in kernel.configs:
            print(
                cfg.kwargs,
                "warps:", cfg.num_warps,
                "stages:", cfg.num_stages
            )

print_kernel_source(
    state_mod,
    "_chunk_state_fwd_kernel"
)

print("\nSTATE-PASSING MODULE:", passing_mod.__name__)

passing_kernels = [
    name for name in dir(passing_mod)
    if "fwd_kernel" in name.lower()
]

print("Available forward kernels:", passing_kernels)

for name in passing_kernels:
    print_kernel_source(passing_mod, name)



KERNEL: _chunk_state_fwd_kernel
TYPE: <class 'triton.runtime.autotuner.Autotuner'>
def _chunk_state_fwd_kernel(
    # Pointers to matrices
    x_ptr, b_ptr, states_ptr, dt_ptr, dA_cumsum_ptr, seq_idx_ptr,
    # Matrix dimensions
    hdim, dstate, chunk_size,
    batch, seqlen, nheads_ngroups_ratio,
    # Strides
    stride_x_batch, stride_x_seqlen, stride_x_head, stride_x_hdim,
    stride_b_batch, stride_b_seqlen, stride_b_head, stride_b_dstate,
    stride_states_batch, stride_states_chunk, stride_states_head, stride_states_hdim, stride_states_dstate,
    stride_dt_batch, stride_dt_chunk, stride_dt_head, stride_dt_csize,
    stride_dA_cs_batch, stride_dA_cs_chunk, stride_dA_cs_head, stride_dA_cs_csize,
    stride_seq_idx_batch, stride_seq_idx_seqlen,
    # Meta-parameters
    HAS_SEQ_IDX: tl.constexpr,
    BLOCK_SIZE_M: tl.constexpr, BLOCK_SIZE_N: tl.constexpr, BLOCK_SIZE_K: tl.constexpr,
):
    pid_bc = tl.program_id(axis=1)
    pid_c = pid_bc // batch
    pid_b = pid_bc - pid_c * ba

In [102]:

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import mamba_ssm.ops.triton.ssd_combined as ssd_combined

# ============================================================
# 1. CONFIGURATION
# ============================================================

N_DOCS = 100
MAX_LENGTH = 128
BOOTSTRAP_SAMPLES = 20000
RANDOM_SEED = 42

device = "cuda"
model.eval()

# ============================================================
# 2. LOAD 100 DOCUMENTS
# ============================================================

# Reuse the existing streaming FineWeb-Edu dataset.
# Skip very short or empty documents.

encoded_eval = []

for item in dataset:
    text = item.get("text", "").strip()

    if len(text) < 100:
        continue

    tokens = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    ids = tokens.input_ids

    # Ensure at least two tokens for next-token evaluation.
    if ids.shape[1] < 2:
        continue

    if int(ids.max()) >= model.config.vocab_size:
        raise ValueError("Token ID exceeds model vocabulary size.")

    encoded_eval.append(ids)

    if len(encoded_eval) >= N_DOCS:
        break

if len(encoded_eval) < N_DOCS:
    raise RuntimeError(
        f"Only obtained {len(encoded_eval)} usable documents."
    )

print("Evaluation documents:", len(encoded_eval))
print("Token lengths:", [x.shape[1] for x in encoded_eval[:10]])

# ============================================================
# 3. DYADIC2 GRID
# ============================================================

original_chunk_cumsum = ssd_combined._chunk_cumsum_fwd

pairs = [
    (k1, k2)
    for k1 in range(2, 15)
    for k2 in range(k1, 15)
]

grid_d = torch.tensor(
    [2.0**(-k1) + 2.0**(-k2) for k1, k2 in pairs],
    dtype=torch.float32,
    device=device
)

grid_a = 1.0 - grid_d
log_grid_d = torch.log(grid_d)

# ============================================================
# 4. QUANTIZED CUMSUM INTERVENTION
# ============================================================

run_stats = []

def quantized_chunk_cumsum(
    dt,
    A,
    chunk_size,
    dt_bias=None,
    dt_softplus=False,
    dt_limit=(0.0, float("inf"))
):
    # Preserve original dt processing exactly.
    _, dt_out = original_chunk_cumsum(
        dt,
        A,
        chunk_size,
        dt_bias=dt_bias,
        dt_softplus=dt_softplus,
        dt_limit=dt_limit
    )

    batch, seqlen, nheads = dt.shape
    _, _, nchunks, _ = dt_out.shape

    # Original individual log-decay.
    dA = (
        dt_out.float()
        * A.float().view(1, nheads, 1, 1)
    )

    decay = torch.exp(dA)
    distance = 1.0 - decay

    # Exclude padded positions.
    positions = (
        torch.arange(nchunks, device=device)[:, None] * chunk_size
        + torch.arange(chunk_size, device=device)[None, :]
    )

    valid = (positions < seqlen).view(
        1, 1, nchunks, chunk_size
    )

    # Quantize only coefficients inside the positive grid.
    eligible = (
        valid
        & torch.isfinite(decay)
        & (decay > 0)
        & (distance >= grid_d.min())
        & (distance <= grid_d.max())
    )

    decay_q = decay.clone()

    if eligible.any():
        selected_d = distance[eligible]

        nearest = torch.abs(
            torch.log(selected_d[:, None])
            - log_grid_d[None, :]
        ).argmin(dim=-1)

        decay_q[eligible] = grid_a[nearest]

    # Exact fallback outside representable range.
    dA_q = dA.clone()

    if eligible.any():
        dA_q[eligible] = torch.log(decay_q[eligible])

    # Padded increments must remain zero.
    dA_q = torch.where(
        valid,
        dA_q,
        torch.zeros_like(dA_q)
    )

    dA_cumsum_q = torch.cumsum(dA_q, dim=-1)

    total = int(valid.sum().item() * batch * nheads)
    quantized_count = int(eligible.sum().item())

    run_stats.append({
        "total": total,
        "quantized": quantized_count,
        "fallback": total - quantized_count
    })

    return dA_cumsum_q, dt_out

# ============================================================
# 5. MODEL EVALUATION
# ============================================================

def extract_logits(output):
    if hasattr(output, "logits"):
        return output.logits
    if isinstance(output, (tuple, list)):
        return output[0]
    return output

def evaluate_one(ids):
    ids = ids.to(device)

    with torch.no_grad():
        logits = extract_logits(model(ids))

        predictions = logits[:, :-1, :].float()
        targets = ids[:, 1:]

        loss_sum = F.cross_entropy(
            predictions.reshape(-1, predictions.shape[-1]),
            targets.reshape(-1),
            reduction="sum"
        )

        token_count = targets.numel()

        result = {
            "ce": loss_sum.item() / token_count,
            "loss_sum": loss_sum.item(),
            "tokens": token_count
        }

    del logits
    return result

results = []

# ============================================================
# 6. BASELINE
# ============================================================

print("\nRunning FP baseline...")

for i, ids in enumerate(encoded_eval):
    result = evaluate_one(ids)

    results.append({
        "document": i,
        "baseline_ce": result["ce"],
        "baseline_loss_sum": result["loss_sum"],
        "tokens": result["tokens"]
    })

    if (i + 1) % 10 == 0:
        print(f"Baseline: {i + 1}/{N_DOCS}")

# ============================================================
# 7. DYADIC2
# ============================================================

print("\nRunning Dyadic2 intervention...")

try:
    ssd_combined._chunk_cumsum_fwd = quantized_chunk_cumsum

    for i, ids in enumerate(encoded_eval):
        result = evaluate_one(ids)

        results[i]["dyadic2_ce"] = result["ce"]
        results[i]["dyadic2_loss_sum"] = result["loss_sum"]

        if (i + 1) % 10 == 0:
            print(f"Dyadic2: {i + 1}/{N_DOCS}")

finally:
    ssd_combined._chunk_cumsum_fwd = original_chunk_cumsum

# ============================================================
# 8. PAIRED RESULTS
# ============================================================

df = pd.DataFrame(results)

df["delta_ce"] = (
    df["dyadic2_ce"] - df["baseline_ce"]
)

total_tokens = df["tokens"].sum()

baseline_total_ce = (
    df["baseline_loss_sum"].sum() / total_tokens
)

dyadic2_total_ce = (
    df["dyadic2_loss_sum"].sum() / total_tokens
)

delta_ce = dyadic2_total_ce - baseline_total_ce

baseline_ppl = float(np.exp(baseline_total_ce))
dyadic2_ppl = float(np.exp(dyadic2_total_ce))

# ============================================================
# 9. DOCUMENT-LEVEL BOOTSTRAP CI
# ============================================================

differences = df["delta_ce"].to_numpy()

rng = np.random.default_rng(RANDOM_SEED)

bootstrap_means = np.empty(BOOTSTRAP_SAMPLES)

for b in range(BOOTSTRAP_SAMPLES):
    sample = rng.choice(
        differences,
        size=len(differences),
        replace=True
    )
    bootstrap_means[b] = sample.mean()

ci_low, ci_high = np.percentile(
    bootstrap_means,
    [2.5, 97.5]
)

# ============================================================
# 10. FALLBACK STATISTICS
# ============================================================

total_decay = sum(x["total"] for x in run_stats)
quantized_decay = sum(x["quantized"] for x in run_stats)
fallback_decay = sum(x["fallback"] for x in run_stats)

fallback_pct = (
    100.0 * fallback_decay / total_decay
    if total_decay else float("nan")
)

quantized_pct = (
    100.0 * quantized_decay / total_decay
    if total_decay else float("nan")
)

# ============================================================
# 11. FINAL REPORT
# ============================================================

print("\n" + "=" * 65)
print("SHIFT-SCAN: 100-DOCUMENT DYADIC2 EVALUATION")
print("=" * 65)

print("\nPER-DOCUMENT RESULTS")
display(df[[
    "document",
    "tokens",
    "baseline_ce",
    "dyadic2_ce",
    "delta_ce"
]])

print("\nAGGREGATE RESULTS")

print("Documents:", len(df))
print("Total evaluated tokens:", total_tokens)

print("Baseline CE:", baseline_total_ce)
print("Dyadic2 CE:", dyadic2_total_ce)
print("Delta CE:", delta_ce)

print("Baseline PPL:", baseline_ppl)
print("Dyadic2 PPL:", dyadic2_ppl)

print(
    "Relative PPL change (%):",
    100.0 * (dyadic2_ppl / baseline_ppl - 1.0)
)

print("\nPAIRED DOCUMENT BOOTSTRAP")
print("Mean document delta CE:", differences.mean())
print("95% CI lower:", ci_low)
print("95% CI upper:", ci_high)

print("Documents improved:", int((differences < 0).sum()))
print("Documents worsened:", int((differences > 0).sum()))
print("Documents unchanged:", int((differences == 0).sum()))

print("\nDECAY QUANTIZATION")
print("Total coefficients:", total_decay)
print("Quantized:", quantized_decay)
print("Exact fallback:", fallback_decay)
print("Quantized percentage:", quantized_pct)
print("Fallback percentage:", fallback_pct)

print("\nFUNCTION RESTORED:",
      ssd_combined._chunk_cumsum_fwd is original_chunk_cumsum)

# Save results.
output_path = "/kaggle/working/shiftscan_100doc_dyadic2.csv"
df.to_csv(output_path, index=False)

print("\nSaved:", output_path)


Evaluation documents: 100
Token lengths: [128, 128, 128, 128, 128, 128, 128, 128, 128, 128]

Running FP baseline...
Baseline: 10/100
Baseline: 20/100
Baseline: 30/100
Baseline: 40/100
Baseline: 50/100
Baseline: 60/100
Baseline: 70/100
Baseline: 80/100
Baseline: 90/100
Baseline: 100/100

Running Dyadic2 intervention...
Dyadic2: 10/100
Dyadic2: 20/100
Dyadic2: 30/100
Dyadic2: 40/100
Dyadic2: 50/100
Dyadic2: 60/100
Dyadic2: 70/100
Dyadic2: 80/100
Dyadic2: 90/100
Dyadic2: 100/100

SHIFT-SCAN: 100-DOCUMENT DYADIC2 EVALUATION

PER-DOCUMENT RESULTS


,document,tokens,baseline_ce,dyadic2_ce,delta_ce
0,0,127,3.710593,3.712081,0.001488
1,1,127,3.701580,3.701790,0.000210
2,2,127,2.721435,2.716581,-0.004854
3,3,127,3.373219,3.370663,-0.002557
4,4,127,3.581251,3.579661,-0.001591
...,...,...,...,...,...
95,95,127,4.452336,4.450315,-0.002021
96,96,127,3.641122,3.641380,0.000258
97,97,127,3.428078,3.429603,0.001525
98,98,127,3.349006,3.349560,0.000554



AGGREGATE RESULTS
Documents: 100
Total evaluated tokens: 12633
Baseline CE: 3.2179452036580782
Dyadic2 CE: 3.2182494735506397
Delta CE: 0.00030426989256149284
Baseline PPL: 24.976745292087468
Dyadic2 PPL: 24.98434611998693
Relative PPL change (%): 0.030431618734039212

PAIRED DOCUMENT BOOTSTRAP
Mean document delta CE: 0.0003201210719869341
95% CI lower: -0.00024289066646334678
95% CI upper: 0.0009064532839440247
Documents improved: 48
Documents worsened: 52
Documents unchanged: 0

DECAY QUANTIZATION
Total coefficients: 7334208
Quantized: 5794978
Exact fallback: 1539230
Quantized percentage: 79.01300317634842
Fallback percentage: 20.98699682365158

FUNCTION RESTORED: True

Saved: /kaggle/working/shiftscan_100doc_dyadic2.csv


In [103]:

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import mamba_ssm.ops.triton.ssd_combined as ssd_combined

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

device = "cuda"
model.eval()

original_chunk_cumsum = ssd_combined._chunk_cumsum_fwd

pairs = [
    (k1, k2)
    for k1 in range(2, 15)
    for k2 in range(k1, 15)
]

grid_d = torch.tensor(
    [2.0**(-k1) + 2.0**(-k2) for k1, k2 in pairs],
    device=device,
    dtype=torch.float32
)

grid_a = 1.0 - grid_d
log_grid_d = torch.log(grid_d)

run_stats = []

# ------------------------------------------------------------
# TRITON-PRESERVING QUANTIZATION PATCH
# ------------------------------------------------------------

def quantized_triton_cumsum(
    dt,
    A,
    chunk_size,
    dt_bias=None,
    dt_softplus=False,
    dt_limit=(0.0, float("inf"))
):
    # Original effective dt and original Triton cumulative sum.
    original_cumsum, original_dt_out = original_chunk_cumsum(
        dt,
        A,
        chunk_size,
        dt_bias=dt_bias,
        dt_softplus=dt_softplus,
        dt_limit=dt_limit
    )

    batch, seqlen, nheads = dt.shape
    _, _, nchunks, _ = original_dt_out.shape

    # Reconstruct the exact effective log-decay increments.
    dA = (
        original_dt_out.float()
        * A.float().view(1, nheads, 1, 1)
    )

    decay = torch.exp(dA)
    distance = 1.0 - decay

    positions = (
        torch.arange(nchunks, device=device)[:, None] * chunk_size
        + torch.arange(chunk_size, device=device)[None, :]
    )

    valid = (positions < seqlen).view(
        1, 1, nchunks, chunk_size
    )

    eligible = (
        valid
        & torch.isfinite(decay)
        & (decay > 0)
        & (distance >= grid_d.min())
        & (distance <= grid_d.max())
    )

    dA_q = dA.clone()

    if eligible.any():
        selected_d = distance[eligible]

        nearest = torch.abs(
            torch.log(selected_d[:, None])
            - log_grid_d[None, :]
        ).argmin(dim=-1)

        # Replace only the eligible log-decay increments.
        dA_q[eligible] = torch.log(grid_a[nearest])

    # Preserve zero increments in padded positions.
    dA_q = torch.where(
        valid,
        dA_q,
        torch.zeros_like(dA_q)
    )

    # Convert quantized dA back to effective dt:
    # dA_q = dt_q * A
    dA_q_seq = (
        dA_q.permute(0, 2, 3, 1)
        .reshape(batch, nchunks * chunk_size, nheads)
        [:, :seqlen, :]
        .contiguous()
    )

    dt_q = (
        dA_q_seq
        / A.float().view(1, 1, nheads)
    ).contiguous()

    # Crucial: use the ORIGINAL Triton kernel to accumulate
    # the quantized increments.
    quantized_cumsum, _ = original_chunk_cumsum(
        dt_q,
        A,
        chunk_size,
        dt_bias=None,
        dt_softplus=False,
        dt_limit=(0.0, float("inf"))
    )

    total = int(valid.sum().item() * batch * nheads)
    quantized_count = int(eligible.sum().item())

    run_stats.append({
        "total": total,
        "quantized": quantized_count,
        "fallback": total - quantized_count
    })

    # Preserve original dt for chunk-state and chunk-scan.
    return quantized_cumsum, original_dt_out


# ------------------------------------------------------------
# MODEL OUTPUT AND LOSS
# ------------------------------------------------------------

def extract_logits(output):
    if hasattr(output, "logits"):
        return output.logits
    if isinstance(output, (tuple, list)):
        return output[0]
    return output


def evaluate_document(ids):
    ids = ids.to(device)

    with torch.no_grad():
        logits = extract_logits(model(ids))

        predictions = logits[:, :-1, :].float()
        targets = ids[:, 1:]

        loss_sum = F.cross_entropy(
            predictions.reshape(-1, predictions.shape[-1]),
            targets.reshape(-1),
            reduction="sum"
        )

        token_count = targets.numel()

        result = {
            "loss_sum": loss_sum.item(),
            "tokens": token_count,
            "ce": loss_sum.item() / token_count
        }

    del logits
    return result


# ------------------------------------------------------------
# BASELINE AND QUANTIZED EVALUATION
# ------------------------------------------------------------

new_results = []

print("Running original Triton baseline...")

for i, ids in enumerate(encoded_eval):
    r = evaluate_document(ids)

    new_results.append({
        "document": i,
        "tokens": r["tokens"],
        "baseline_ce": r["ce"],
        "baseline_loss_sum": r["loss_sum"]
    })

    if (i + 1) % 10 == 0:
        print(f"Baseline: {i+1}/100")


print("\nRunning Triton-preserving Dyadic2...")

try:
    ssd_combined._chunk_cumsum_fwd = quantized_triton_cumsum

    for i, ids in enumerate(encoded_eval):
        r = evaluate_document(ids)

        new_results[i]["dyadic2_ce"] = r["ce"]
        new_results[i]["dyadic2_loss_sum"] = r["loss_sum"]

        if (i + 1) % 10 == 0:
            print(f"Dyadic2: {i+1}/100")

finally:
    ssd_combined._chunk_cumsum_fwd = original_chunk_cumsum


# ------------------------------------------------------------
# AGGREGATE RESULTS
# ------------------------------------------------------------

df_triton = pd.DataFrame(new_results)

df_triton["delta_ce"] = (
    df_triton["dyadic2_ce"]
    - df_triton["baseline_ce"]
)

total_tokens = int(df_triton["tokens"].sum())

baseline_ce = (
    df_triton["baseline_loss_sum"].sum()
    / total_tokens
)

dyadic2_ce = (
    df_triton["dyadic2_loss_sum"].sum()
    / total_tokens
)

delta_ce = dyadic2_ce - baseline_ce

baseline_ppl = np.exp(baseline_ce)
dyadic2_ppl = np.exp(dyadic2_ce)

# Paired document bootstrap.
differences = df_triton["delta_ce"].to_numpy()

rng = np.random.default_rng(42)
B = 20000

boot = np.empty(B)

for b in range(B):
    idx = rng.integers(0, len(differences), len(differences))
    boot[b] = differences[idx].mean()

ci_low, ci_high = np.percentile(boot, [2.5, 97.5])

total = sum(s["total"] for s in run_stats)
quantized = sum(s["quantized"] for s in run_stats)
fallback = sum(s["fallback"] for s in run_stats)

print("\n" + "=" * 65)
print("TRITON-PRESERVING DYADIC2 RESULTS")
print("=" * 65)

display(df_triton[[
    "document",
    "tokens",
    "baseline_ce",
    "dyadic2_ce",
    "delta_ce"
]])

print("\nBASELINE CE:", baseline_ce)
print("DYADIC2 CE:", dyadic2_ce)
print("DELTA CE:", delta_ce)

print("\nBASELINE PPL:", baseline_ppl)
print("DYADIC2 PPL:", dyadic2_ppl)

print(
    "RELATIVE PPL CHANGE (%):",
    100 * (dyadic2_ppl / baseline_ppl - 1)
)

print("\nDOCUMENT-LEVEL BOOTSTRAP")
print("Mean delta CE:", differences.mean())
print("95% CI:", (ci_low, ci_high))

print("Improved:", int((differences < 0).sum()))
print("Worsened:", int((differences > 0).sum()))

print("\nQUANTIZATION")
print("Total coefficients:", total)
print("Quantized:", quantized)
print("Fallback:", fallback)
print("Quantized percentage:", 100 * quantized / total)
print("Fallback percentage:", 100 * fallback / total)

print(
    "\nFUNCTION RESTORED:",
    ssd_combined._chunk_cumsum_fwd is original_chunk_cumsum
)

save_path = (
    "/kaggle/working/"
    "shiftscan_100doc_dyadic2_triton_preserving.csv"
)

df_triton.to_csv(save_path, index=False)
print("\nSaved:", save_path)


Running original Triton baseline...
Baseline: 10/100
Baseline: 20/100
Baseline: 30/100
Baseline: 40/100
Baseline: 50/100
Baseline: 60/100
Baseline: 70/100
Baseline: 80/100
Baseline: 90/100
Baseline: 100/100

Running Triton-preserving Dyadic2...
Dyadic2: 10/100
Dyadic2: 20/100
Dyadic2: 30/100
Dyadic2: 40/100
Dyadic2: 50/100
Dyadic2: 60/100
Dyadic2: 70/100
Dyadic2: 80/100
Dyadic2: 90/100
Dyadic2: 100/100

TRITON-PRESERVING DYADIC2 RESULTS


,document,tokens,baseline_ce,dyadic2_ce,delta_ce
0,0,127,3.710593,3.712531,0.001938
1,1,127,3.701580,3.698869,-0.002710
2,2,127,2.721435,2.715167,-0.006268
3,3,127,3.373219,3.370400,-0.002820
4,4,127,3.581251,3.579799,-0.001452
...,...,...,...,...,...
95,95,127,4.452336,4.450405,-0.001931
96,96,127,3.641122,3.640760,-0.000363
97,97,127,3.428078,3.429870,0.001792
98,98,127,3.349006,3.349337,0.000331



BASELINE CE: 3.2179452036580782
DYADIC2 CE: 3.2183664769257443
DELTA CE: 0.0004212732676660558

BASELINE PPL: 24.976745292087468
DYADIC2 PPL: 24.987269543829644
RELATIVE PPL CHANGE (%): 0.042136201571096876

DOCUMENT-LEVEL BOOTSTRAP
Mean delta CE: 0.00043628673616800116
95% CI: (np.float64(-0.00014767387957402843), np.float64(0.001032727577930259))
Improved: 42
Worsened: 58

QUANTIZATION
Total coefficients: 7334208
Quantized: 5795010
Fallback: 1539198
Quantized percentage: 79.01343948794471
Fallback percentage: 20.986560512055288

FUNCTION RESTORED: True

Saved: /kaggle/working/shiftscan_100doc_dyadic2_triton_preserving.csv


In [104]:

import torch
import os

# Save corrected records separately from previous experiments
output_dir = "/kaggle/working/shiftscan_final"
os.makedirs(output_dir, exist_ok=True)

records = []

# Always use the original, unpatched Triton function
capture_original = combined_mod._chunk_cumsum_fwd

assert capture_original is original_fn, (
    "The original Triton cumsum function is not currently installed."
)

def capture_decay_records(dt, A, chunk_size, *args, **kwargs):
    result = capture_original(
        dt, A, chunk_size, *args, **kwargs
    )

    dA_cumsum, dt_out = result

    # dt_out is the effective dt after bias, softplus and limits
    # Its layout is [batch, heads, chunks, chunk_size].
    effective_dt = dt_out.detach().float()

    A_float = A.detach().float().reshape(1, -1, 1, 1)

    dA = effective_dt * A_float
    decay = torch.exp(dA)

    seq_len = dt.shape[1]
    valid = min(seq_len, chunk_size)

    records.append({
        "effective_dt": effective_dt[:, :, :, :valid].cpu(),
        "dA": dA[:, :, :, :valid].cpu(),
        "decay": decay[:, :, :, :valid].cpu(),
        "A": A.detach().float().cpu(),
        "seq_len": int(seq_len),
        "chunk_size": int(chunk_size),
    })

    return result


# Capture baseline forwards without changing their outputs
combined_mod._chunk_cumsum_fwd = capture_decay_records

try:
    with torch.inference_mode():
        for i, input_ids in enumerate(encoded_eval):
            _ = model(input_ids.to("cuda"))

            if (i + 1) % 10 == 0:
                print(f"Captured documents: {i+1}/{len(encoded_eval)}")

finally:
    combined_mod._chunk_cumsum_fwd = capture_original

# Verify restoration
print("\nOriginal function restored:",
      combined_mod._chunk_cumsum_fwd is original_fn)

save_path = os.path.join(
    output_dir,
    "corrected_effective_decay_records_100doc.pt"
)

torch.save(records, save_path)

print("Records captured:", len(records))
print("Saved:", save_path)
print("First record decay shape:", records[0]["decay"].shape)
print("First record decay range:",
      records[0]["decay"].min().item(),
      records[0]["decay"].max().item())


Captured documents: 10/100
Captured documents: 20/100
Captured documents: 30/100
Captured documents: 40/100
Captured documents: 50/100
Captured documents: 60/100
Captured documents: 70/100
Captured documents: 80/100
Captured documents: 90/100
Captured documents: 100/100

Original function restored: True
Records captured: 2400
Saved: /kaggle/working/shiftscan_final/corrected_effective_decay_records_100doc.pt
First record decay shape: torch.Size([1, 24, 1, 128])
First record decay range: 0.7300302386283875 0.9848787188529968


In [106]:

import torch
import math
import os
import pandas as pd

records_path = (
    "/kaggle/working/shiftscan_final/"
    "corrected_effective_decay_records_100doc.pt"
)

records = torch.load(
    records_path,
    map_location="cpu",
    weights_only=False
)

# Original Dyadic2 grid
pairs = [
    (k1, k2)
    for k1 in range(2, 15)
    for k2 in range(k1, 15)
]

grid_d = torch.tensor(
    [2.0**(-k1) + 2.0**(-k2) for k1, k2 in pairs],
    dtype=torch.float64
)

log_grid_d = torch.log(grid_d)
D_MIN = grid_d.min()
D_MAX = grid_d.max()

def quantize_dyadic2_with_fallback(decay):
    decay = decay.to(torch.float64)
    d = 1.0 - decay

    eligible = (
        torch.isfinite(decay)
        & (decay > 0.0)
        & (d >= D_MIN)
        & (d <= D_MAX)
    )

    safe_d = torch.where(eligible, d, torch.ones_like(d))

    distances = torch.abs(
        torch.log(safe_d).unsqueeze(-1) - log_grid_d
    )

    indices = distances.argmin(dim=-1)
    quantized = 1.0 - grid_d[indices]

    output = torch.where(eligible, quantized, decay)

    return output, eligible


LAGS = [1, 2, 4, 8, 16, 32, 64, 96, 127]

# Separate accumulators for each lag
stats = {
    method: {
        lag: {
            "count": 0,
            "abs_sum": 0.0,
            "sq_sum": 0.0,
            "max_error": 0.0,
            "fallback_count": 0
        }
        for lag in LAGS
    }
    for method in ["composed", "direct"]
}

token_fallback_count = 0
token_count = 0

# ---------------------------------------------------------
# Process all records, including variable-length documents
# ---------------------------------------------------------
for record_index, record in enumerate(records):

    dA = record["dA"].to(torch.float64)
    decay = record["decay"].to(torch.float64)

    # Remove batch and chunk dimensions without assuming
    # a fixed sequence length.
    dA = dA.reshape(-1, dA.shape[-1])
    decay = decay.reshape(-1, decay.shape[-1])

    assert dA.shape == decay.shape, (
        f"Shape mismatch at record {record_index}: "
        f"{dA.shape} vs {decay.shape}"
    )

    nheads, seq_len = dA.shape

    token_q, token_eligible = (
        quantize_dyadic2_with_fallback(decay)
    )

    token_fallback_count += (
        (~token_eligible).sum().item()
    )
    token_count += token_eligible.numel()

    # Prefix sums of exact and quantized log-decays
    exact_prefix = torch.cat(
        [
            torch.zeros((nheads, 1), dtype=torch.float64),
            torch.cumsum(dA, dim=-1)
        ],
        dim=-1
    )

    quantized_prefix = torch.cat(
        [
            torch.zeros((nheads, 1), dtype=torch.float64),
            torch.cumsum(torch.log(token_q), dim=-1)
        ],
        dim=-1
    )

    fallback_prefix = torch.cat(
        [
            torch.zeros((nheads, 1), dtype=torch.int32),
            torch.cumsum(
                (~token_eligible).to(torch.int32),
                dim=-1
            )
        ],
        dim=-1
    )

    for lag in LAGS:

        if lag >= seq_len:
            continue

        exact_interval = torch.exp(
            exact_prefix[:, lag:]
            - exact_prefix[:, :-lag]
        )

        composed_interval = torch.exp(
            quantized_prefix[:, lag:]
            - quantized_prefix[:, :-lag]
        )

        composed_fallback = (
            fallback_prefix[:, lag:]
            - fallback_prefix[:, :-lag]
        ) > 0

        direct_interval, direct_eligible = (
            quantize_dyadic2_with_fallback(exact_interval)
        )

        direct_fallback = ~direct_eligible

        for method, approx, fallback_mask in [
            ("composed", composed_interval, composed_fallback),
            ("direct", direct_interval, direct_fallback)
        ]:

            error = torch.abs(approx - exact_interval)

            st = stats[method][lag]

            st["count"] += error.numel()
            st["abs_sum"] += error.sum().item()
            st["sq_sum"] += (error ** 2).sum().item()
            st["max_error"] = max(
                st["max_error"],
                error.max().item()
            )
            st["fallback_count"] += (
                fallback_mask.sum().item()
            )

    if (record_index + 1) % 200 == 0:
        print(
            f"Processed {record_index+1}/{len(records)} "
            f"layer calls"
        )

# ---------------------------------------------------------
# Print per-lag results
# ---------------------------------------------------------
rows = []

print("\n" + "=" * 85)
print("VARIABLE-LENGTH INTERVAL DYADIC2 RESULTS")
print("=" * 85)

for lag in LAGS:

    for method in ["composed", "direct"]:

        st = stats[method][lag]

        if st["count"] == 0:
            continue

        mae = st["abs_sum"] / st["count"]
        rmse = math.sqrt(st["sq_sum"] / st["count"])
        fallback_pct = (
            st["fallback_count"] / st["count"] * 100
        )

        rows.append({
            "lag": lag,
            "method": method,
            "count": st["count"],
            "MAE": mae,
            "RMSE": rmse,
            "max_error": st["max_error"],
            "fallback_pct": fallback_pct
        })

        print(
            f"Lag {lag:3d} | {method:8s} | "
            f"MAE {mae:.8f} | "
            f"RMSE {rmse:.8f} | "
            f"Max {st['max_error']:.8f} | "
            f"Fallback {fallback_pct:.3f}%"
        )

print("\nTOKEN-LEVEL FALLBACK")
print(
    f"{token_fallback_count}/{token_count} "
    f"({100*token_fallback_count/token_count:.3f}%)"
)

# Save results
out_dir = "/kaggle/working/shiftscan_final"
os.makedirs(out_dir, exist_ok=True)

result_df = pd.DataFrame(rows)

save_path = os.path.join(
    out_dir,
    "dyadic2_interval_comparison_100doc_variable_length.csv"
)

result_df.to_csv(save_path, index=False)

print("\nSaved:", save_path)
print("Records processed:", len(records))


Processed 200/2400 layer calls
Processed 400/2400 layer calls
Processed 600/2400 layer calls
Processed 800/2400 layer calls
Processed 1000/2400 layer calls
Processed 1200/2400 layer calls
Processed 1400/2400 layer calls
Processed 1600/2400 layer calls
Processed 1800/2400 layer calls
Processed 2000/2400 layer calls
Processed 2200/2400 layer calls
Processed 2400/2400 layer calls

VARIABLE-LENGTH INTERVAL DYADIC2 RESULTS
Lag   1 | composed | MAE nan | RMSE nan | Max 0.06698705 | Fallback 20.991%
Lag   1 | direct   | MAE 0.00523695 | RMSE 0.01199383 | Max 0.06698705 | Fallback 20.991%
Lag   2 | composed | MAE nan | RMSE nan | Max 0.07128058 | Fallback 26.624%
Lag   2 | direct   | MAE 0.00450769 | RMSE 0.01108007 | Max 0.06698658 | Fallback 34.414%
Lag   4 | composed | MAE nan | RMSE nan | Max 0.06646713 | Fallback 33.166%
Lag   4 | direct   | MAE 0.00372753 | RMSE 0.00967227 | Max 0.06698706 | Fallback 46.871%
Lag   8 | composed | MAE nan | RMSE nan | Max 0.06223275 | Fallback 39.704%
Lag 

In [107]:

import torch
import math
import pandas as pd
import os

# Load records
records_path = (
    "/kaggle/working/shiftscan_final/"
    "corrected_effective_decay_records_100doc.pt"
)
records = torch.load(
    records_path, map_location="cpu", weights_only=False
)

# Dyadic2 grid
pairs = [
    (k1, k2)
    for k1 in range(2, 15)
    for k2 in range(k1, 15)
]
grid_d = torch.tensor(
    [2.0**(-k1) + 2.0**(-k2) for k1, k2 in pairs],
    dtype=torch.float64
)
log_grid_d = torch.log(grid_d)
D_MIN, D_MAX = grid_d.min(), grid_d.max()

def quantize_dyadic2(decay):
    decay = decay.to(torch.float64)
    d = 1.0 - decay

    eligible = (
        torch.isfinite(decay)
        & (decay > 0)
        & (d >= D_MIN)
        & (d <= D_MAX)
    )

    safe_d = torch.where(eligible, d, torch.ones_like(d))
    distances = torch.abs(
        torch.log(safe_d).unsqueeze(-1) - log_grid_d
    )
    idx = distances.argmin(dim=-1)
    q = 1.0 - grid_d[idx]

    return torch.where(eligible, q, decay), eligible

LAGS = [1, 2, 4, 8, 16, 32, 64, 96, 127]

methods = ["composed", "direct"]
stats = {
    method: {
        lag: {
            "all_abs": 0.0,
            "all_sq": 0.0,
            "all_n": 0,
            "all_max": 0.0,
            "eligible_abs": 0.0,
            "eligible_sq": 0.0,
            "eligible_n": 0,
            "eligible_max": 0.0,
            "fallback_n": 0
        }
        for lag in LAGS
    }
    for method in methods
}

token_fallback = 0
token_total = 0

for ri, record in enumerate(records):

    dA = record["dA"].to(torch.float64).reshape(
        -1, record["dA"].shape[-1]
    )
    decay = record["decay"].to(torch.float64).reshape(
        -1, record["decay"].shape[-1]
    )

    heads, length = dA.shape

    token_q, token_ok = quantize_dyadic2(decay)

    token_fallback += (~token_ok).sum().item()
    token_total += token_ok.numel()

    # Exact interval prefix
    exact_prefix = torch.cat([
        torch.zeros((heads, 1), dtype=torch.float64),
        torch.cumsum(dA, dim=-1)
    ], dim=-1)

    # Zero-safe quantized log prefix:
    # replace log(0) with 0 and track zeros separately.
    positive = token_q > 0
    safe_log = torch.where(
        positive,
        torch.log(token_q.clamp_min(1e-300)),
        torch.zeros_like(token_q)
    )

    q_log_prefix = torch.cat([
        torch.zeros((heads, 1), dtype=torch.float64),
        torch.cumsum(safe_log, dim=-1)
    ], dim=-1)

    zero_prefix = torch.cat([
        torch.zeros((heads, 1), dtype=torch.int32),
        torch.cumsum(
            (~positive).to(torch.int32), dim=-1
        )
    ], dim=-1)

    fallback_prefix = torch.cat([
        torch.zeros((heads, 1), dtype=torch.int32),
        torch.cumsum(
            (~token_ok).to(torch.int32), dim=-1
        )
    ], dim=-1)

    for lag in LAGS:

        if lag >= length:
            continue

        exact = torch.exp(
            exact_prefix[:, lag:] - exact_prefix[:, :-lag]
        )

        # Composed approximation, correctly handling zero factors
        log_product = (
            q_log_prefix[:, lag:] - q_log_prefix[:, :-lag]
        )
        zero_count = (
            zero_prefix[:, lag:] - zero_prefix[:, :-lag]
        )
        composed = torch.where(
            zero_count > 0,
            torch.zeros_like(log_product),
            torch.exp(log_product)
        )

        # A composed interval is fully quantized only if
        # every token in that interval was eligible.
        composed_ok = (
            fallback_prefix[:, lag:]
            - fallback_prefix[:, :-lag]
        ) == 0

        # Direct interval approximation
        direct, direct_ok = quantize_dyadic2(exact)

        for method, approx, eligible in [
            ("composed", composed, composed_ok),
            ("direct", direct, direct_ok)
        ]:
            error = torch.abs(approx - exact)
            st = stats[method][lag]

            st["all_abs"] += error.sum().item()
            st["all_sq"] += (error**2).sum().item()
            st["all_n"] += error.numel()
            st["all_max"] = max(
                st["all_max"], error.max().item()
            )

            st["fallback_n"] += (~eligible).sum().item()

            if eligible.any():
                e = error[eligible]
                st["eligible_abs"] += e.sum().item()
                st["eligible_sq"] += (e**2).sum().item()
                st["eligible_n"] += e.numel()
                st["eligible_max"] = max(
                    st["eligible_max"], e.max().item()
                )

    if (ri + 1) % 200 == 0:
        print(f"Processed {ri+1}/{len(records)}")

# Results
rows = []

print("\nZERO-SAFE INTERVAL DYADIC2 RESULTS")
print("=" * 100)

for lag in LAGS:
    for method in methods:
        st = stats[method][lag]
        if st["all_n"] == 0:
            continue

        overall_mae = st["all_abs"] / st["all_n"]
        overall_rmse = math.sqrt(st["all_sq"] / st["all_n"])
        fallback_pct = 100 * st["fallback_n"] / st["all_n"]

        eligible_mae = (
            st["eligible_abs"] / st["eligible_n"]
            if st["eligible_n"] else float("nan")
        )
        eligible_rmse = (
            math.sqrt(st["eligible_sq"] / st["eligible_n"])
            if st["eligible_n"] else float("nan")
        )

        row = {
            "lag": lag,
            "method": method,
            "count": st["all_n"],
            "overall_MAE": overall_mae,
            "overall_RMSE": overall_rmse,
            "overall_max": st["all_max"],
            "fallback_pct": fallback_pct,
            "quantized_only_MAE": eligible_mae,
            "quantized_only_RMSE": eligible_rmse,
            "quantized_only_max": st["eligible_max"]
        }
        rows.append(row)

        print(
            f"Lag {lag:3d} | {method:8s} | "
            f"overall MAE={overall_mae:.8f} | "
            f"quantized-only MAE={eligible_mae:.8f} | "
            f"fallback={fallback_pct:.2f}%"
        )

print("\nToken fallback:",
      token_fallback, "/", token_total,
      f"({100*token_fallback/token_total:.3f}%)")

out_dir = "/kaggle/working/shiftscan_final"
os.makedirs(out_dir, exist_ok=True)

result_df = pd.DataFrame(rows)
save_path = os.path.join(
    out_dir,
    "dyadic2_interval_comparison_100doc_zero_safe.csv"
)
result_df.to_csv(save_path, index=False)

print("\nSaved:", save_path)


Processed 200/2400
Processed 400/2400
Processed 600/2400
Processed 800/2400
Processed 1000/2400
Processed 1200/2400
Processed 1400/2400
Processed 1600/2400
Processed 1800/2400
Processed 2000/2400
Processed 2200/2400
Processed 2400/2400

ZERO-SAFE INTERVAL DYADIC2 RESULTS
Lag   1 | composed | overall MAE=0.00523696 | quantized-only MAE=0.00662831 | fallback=20.99%
Lag   1 | direct   | overall MAE=0.00523695 | quantized-only MAE=0.00662833 | fallback=20.99%
Lag   2 | composed | overall MAE=0.00566377 | quantized-only MAE=0.00668837 | fallback=26.62%
Lag   2 | direct   | overall MAE=0.00450769 | quantized-only MAE=0.00687290 | fallback=34.41%
Lag   4 | composed | overall MAE=0.00463221 | quantized-only MAE=0.00582842 | fallback=33.17%
Lag   4 | direct   | overall MAE=0.00372753 | quantized-only MAE=0.00701595 | fallback=46.87%
Lag   8 | composed | overall MAE=0.00302198 | quantized-only MAE=0.00449930 | fallback=39.70%
Lag   8 | direct   | overall MAE=0.00336279 | quantized-only MAE=0.007

In [108]:

import torch
import pandas as pd
import os

records_path = (
    "/kaggle/working/shiftscan_final/"
    "corrected_effective_decay_records_100doc.pt"
)
records = torch.load(
    records_path, map_location="cpu", weights_only=False
)

pairs = [
    (k1, k2)
    for k1 in range(2, 15)
    for k2 in range(k1, 15)
]
grid_d = torch.tensor(
    [2.0**(-k1) + 2.0**(-k2) for k1, k2 in pairs],
    dtype=torch.float64
)
log_grid_d = torch.log(grid_d)
D_MIN, D_MAX = grid_d.min(), grid_d.max()

def quantize_dyadic2(decay):
    decay = decay.to(torch.float64)
    d = 1.0 - decay
    eligible = (
        torch.isfinite(decay)
        & (decay > 0)
        & (d >= D_MIN)
        & (d <= D_MAX)
    )
    safe_d = torch.where(eligible, d, torch.ones_like(d))
    idx = torch.abs(
        torch.log(safe_d).unsqueeze(-1) - log_grid_d
    ).argmin(dim=-1)
    q = 1.0 - grid_d[idx]
    return torch.where(eligible, q, decay), eligible

LAGS = [1, 2, 4, 8, 16, 32, 64, 96, 127]

summary = {
    lag: {
        "total": 0,
        "composed_ok": 0,
        "direct_ok": 0,
        "common_ok": 0,
        "composed_common_abs": 0.0,
        "direct_common_abs": 0.0,
        "composed_common_sq": 0.0,
        "direct_common_sq": 0.0,
        "composed_common_max": 0.0,
        "direct_common_max": 0.0,
        "composed_all_abs": 0.0,
        "direct_all_abs": 0.0
    }
    for lag in LAGS
}

for ri, record in enumerate(records):

    dA = record["dA"].to(torch.float64).reshape(
        -1, record["dA"].shape[-1]
    )
    decay = record["decay"].to(torch.float64).reshape(
        -1, record["decay"].shape[-1]
    )

    heads, length = dA.shape

    token_q, token_ok = quantize_dyadic2(decay)

    # Exact interval prefix
    exact_prefix = torch.cat([
        torch.zeros((heads, 1), dtype=torch.float64),
        torch.cumsum(dA, dim=-1)
    ], dim=-1)

    # Zero-safe composed prefix
    positive = token_q > 0
    safe_log = torch.where(
        positive,
        torch.log(token_q.clamp_min(1e-300)),
        torch.zeros_like(token_q)
    )

    q_prefix = torch.cat([
        torch.zeros((heads, 1), dtype=torch.float64),
        torch.cumsum(safe_log, dim=-1)
    ], dim=-1)

    zero_prefix = torch.cat([
        torch.zeros((heads, 1), dtype=torch.int32),
        torch.cumsum((~positive).int(), dim=-1)
    ], dim=-1)

    fallback_prefix = torch.cat([
        torch.zeros((heads, 1), dtype=torch.int32),
        torch.cumsum((~token_ok).int(), dim=-1)
    ], dim=-1)

    for lag in LAGS:
        if lag >= length:
            continue

        exact = torch.exp(
            exact_prefix[:, lag:] - exact_prefix[:, :-lag]
        )

        composed = torch.where(
            (
                zero_prefix[:, lag:]
                - zero_prefix[:, :-lag]
            ) > 0,
            torch.zeros_like(exact),
            torch.exp(
                q_prefix[:, lag:] - q_prefix[:, :-lag]
            )
        )

        composed_ok = (
            fallback_prefix[:, lag:]
            - fallback_prefix[:, :-lag]
        ) == 0

        direct, direct_ok = quantize_dyadic2(exact)

        common = composed_ok & direct_ok

        e_comp = torch.abs(composed - exact)
        e_direct = torch.abs(direct - exact)

        st = summary[lag]
        st["total"] += exact.numel()
        st["composed_ok"] += composed_ok.sum().item()
        st["direct_ok"] += direct_ok.sum().item()
        st["common_ok"] += common.sum().item()

        st["composed_all_abs"] += e_comp.sum().item()
        st["direct_all_abs"] += e_direct.sum().item()

        if common.any():
            ec = e_comp[common]
            ed = e_direct[common]

            st["composed_common_abs"] += ec.sum().item()
            st["direct_common_abs"] += ed.sum().item()

            st["composed_common_sq"] += (ec**2).sum().item()
            st["direct_common_sq"] += (ed**2).sum().item()

            st["composed_common_max"] = max(
                st["composed_common_max"], ec.max().item()
            )
            st["direct_common_max"] = max(
                st["direct_common_max"], ed.max().item()
            )

    if (ri + 1) % 400 == 0:
        print(f"Processed {ri+1}/{len(records)}")

rows = []

print("\nPAIRED COMMON-ELIGIBILITY COMPARISON")
print("=" * 105)

for lag in LAGS:
    st = summary[lag]
    n = st["total"]
    nc = st["common_ok"]

    if n == 0:
        continue

    row = {
        "lag": lag,
        "total_intervals": n,
        "composed_coverage_pct": 100 * st["composed_ok"] / n,
        "direct_coverage_pct": 100 * st["direct_ok"] / n,
        "common_coverage_pct": 100 * nc / n,
        "composed_common_MAE": (
            st["composed_common_abs"] / nc if nc else float("nan")
        ),
        "direct_common_MAE": (
            st["direct_common_abs"] / nc if nc else float("nan")
        ),
        "composed_common_RMSE": (
            (st["composed_common_sq"] / nc)**0.5
            if nc else float("nan")
        ),
        "direct_common_RMSE": (
            (st["direct_common_sq"] / nc)**0.5
            if nc else float("nan")
        ),
        "composed_all_MAE": st["composed_all_abs"] / n,
        "direct_all_MAE": st["direct_all_abs"] / n
    }

    rows.append(row)

    print(
        f"Lag {lag:3d} | "
        f"common coverage={row['common_coverage_pct']:.2f}% | "
        f"composed MAE={row['composed_common_MAE']:.8f} | "
        f"direct MAE={row['direct_common_MAE']:.8f}"
    )

out_path = (
    "/kaggle/working/shiftscan_final/"
    "dyadic2_paired_common_eligibility.csv"
)

pd.DataFrame(rows).to_csv(out_path, index=False)
print("\nSaved:", out_path)


Processed 400/2400
Processed 800/2400
Processed 1200/2400
Processed 1600/2400
Processed 2000/2400
Processed 2400/2400

PAIRED COMMON-ELIGIBILITY COMPARISON
Lag   1 | common coverage=79.01% | composed MAE=0.00662833 | direct MAE=0.00662833
Lag   2 | common coverage=63.51% | composed MAE=0.00436527 | direct MAE=0.00709449
Lag   4 | common coverage=49.52% | composed MAE=0.00347749 | direct MAE=0.00750896
Lag   8 | common coverage=39.15% | composed MAE=0.00311986 | direct MAE=0.00850117
Lag  16 | common coverage=30.58% | composed MAE=0.00323760 | direct MAE=0.01048276
Lag  32 | common coverage=18.30% | composed MAE=0.00233582 | direct MAE=0.01182246
Lag  64 | common coverage=10.97% | composed MAE=0.00217417 | direct MAE=0.01690832
Lag  96 | common coverage=6.28% | composed MAE=0.00198092 | direct MAE=0.01772109
Lag 127 | common coverage=3.79% | composed MAE=0.00180733 | direct MAE=0.02231750

Saved: /kaggle/working/shiftscan_final/dyadic2_paired_common_eligibility.csv


In [109]:

# ============================================================
# SHIFT-SCAN: EXPANDED DYADIC GRID SWEEP
# ============================================================
# Tests:
#   1. One-term, two-term and three-term representations
#   2. Multiple exponent ranges
#   3. Overall and quantized-only approximation errors
#   4. Exact-fallback coverage
#   5. Per-layer and decay-range sensitivity
#
# Input:
#   corrected_effective_decay_records_100doc.pt
#
# Output:
#   expanded_grid_global_results.csv
#   expanded_grid_layer_results.csv
#   expanded_grid_decay_bucket_results.csv
#   expanded_grid_best_candidates.csv
# ============================================================

import os
import json
import itertools
import numpy as np
import pandas as pd
import torch
from datetime import datetime

# -----------------------------
# 1. CONFIGURATION
# -----------------------------

RECORDS_PATH = (
    "/kaggle/working/shiftscan_final/"
    "corrected_effective_decay_records_100doc.pt"
)

OUT_DIR = "/kaggle/working/shiftscan_final/expanded_grid_sweep"
os.makedirs(OUT_DIR, exist_ok=True)

assert os.path.exists(RECORDS_PATH), (
    f"Missing records file: {RECORDS_PATH}"
)

records = torch.load(
    RECORDS_PATH,
    map_location="cpu",
    weights_only=False
)

print("Loaded records:", len(records))

# Sweep configurations.
# Larger exponent ranges increase representation resolution.
GRID_CONFIGS = [
    {"name": "K1_14", "k_min": 1, "k_max": 14},
    {"name": "K1_16", "k_min": 1, "k_max": 16},
    {"name": "K1_18", "k_min": 1, "k_max": 18},
]

TERM_COUNTS = [1, 2, 3]

# -----------------------------
# 2. BUILD REPRESENTATION GRIDS
# -----------------------------

def build_grid(k_min, k_max, n_terms):
    exponent_tuples = list(
        itertools.combinations_with_replacement(
            range(k_min, k_max + 1),
            n_terms
        )
    )

    rows = []

    for exponents in exponent_tuples:
        d = sum(2.0 ** (-k) for k in exponents)
        q = 1.0 - d

        # Keep only positive decay coefficients.
        if 0.0 < q < 1.0:
            rows.append({
                "exponents": exponents,
                "d": d,
                "q": q
            })

    rows.sort(key=lambda x: x["d"])

    if not rows:
        raise ValueError("Empty quantization grid")

    return rows


grids = {}

for cfg in GRID_CONFIGS:
    for n_terms in TERM_COUNTS:
        key = f"{cfg['name']}_T{n_terms}"

        grids[key] = build_grid(
            cfg["k_min"],
            cfg["k_max"],
            n_terms
        )

        print(
            key,
            "| grid values:",
            len(grids[key]),
            "| min decay:",
            min(x["q"] for x in grids[key]),
            "| max decay:",
            max(x["q"] for x in grids[key])
        )

# -----------------------------
# 3. LOAD AND ORGANIZE RECORDS
# -----------------------------

# Convert records into CPU float64 arrays.
# Each record corresponds to one model layer call.
# Preserve variable sequence lengths.

layer_records = {}

for ri, record in enumerate(records):

    decay = (
        record["decay"]
        .detach()
        .cpu()
        .double()
        .numpy()
        .reshape(-1, record["decay"].shape[-1])
    )

    # In the current Mamba-2 setup, records are ordered
    # document-major, with 24 layers per document.
    layer_id = ri % 24

    layer_records.setdefault(layer_id, []).append(decay)

print("\nLayer record counts:")
for layer_id in sorted(layer_records):
    print(
        f"Layer {layer_id:02d}: "
        f"{len(layer_records[layer_id])} records"
    )

# -----------------------------
# 4. QUANTIZATION FUNCTION
# -----------------------------

def quantize_values(values, grid):
    """
    Quantize decay using nearest distance in log(1-a).

    Exact fallback is used outside the representable range.
    Returns:
        quantized values
        eligibility mask
    """

    values = np.asarray(values, dtype=np.float64)

    grid_d = np.array(
        [item["d"] for item in grid],
        dtype=np.float64
    )

    grid_q = np.array(
        [item["q"] for item in grid],
        dtype=np.float64
    )

    d = 1.0 - values

    eligible = (
        np.isfinite(values)
        & (values > 0.0)
        & (values < 1.0)
        & (d >= grid_d.min())
        & (d <= grid_d.max())
    )

    output = values.copy()

    if eligible.any():
        log_d = np.log(d[eligible])
        log_grid = np.log(grid_d)

        indices = np.abs(
            log_d[:, None] - log_grid[None, :]
        ).argmin(axis=1)

        output[eligible] = grid_q[indices]

    return output, eligible


# -----------------------------
# 5. ERROR METRICS
# -----------------------------

def calculate_metrics(reference, approximation, eligible):
    error = approximation - reference
    abs_error = np.abs(error)

    total = reference.size
    quantized_count = int(eligible.sum())
    fallback_count = total - quantized_count

    quantized_error = abs_error[eligible]

    return {
        "total": total,
        "quantized_count": quantized_count,
        "fallback_count": fallback_count,
        "coverage_pct": 100.0 * quantized_count / total,
        "fallback_pct": 100.0 * fallback_count / total,
        "overall_mae": float(abs_error.mean()),
        "overall_rmse": float(np.sqrt(np.mean(error**2))),
        "overall_max_abs": float(abs_error.max()),
        "quantized_only_mae": (
            float(quantized_error.mean())
            if quantized_count else np.nan
        ),
        "quantized_only_rmse": (
            float(np.sqrt(np.mean(quantized_error**2)))
            if quantized_count else np.nan
        ),
        "quantized_only_max_abs": (
            float(quantized_error.max())
            if quantized_count else np.nan
        ),
        "mean_signed_error": float(error.mean())
    }


# -----------------------------
# 6. GLOBAL, LAYER AND BUCKET SWEEP
# -----------------------------

global_rows = []
layer_rows = []
bucket_rows = []

decay_buckets = [
    (0.0, 0.5),
    (0.5, 0.7),
    (0.7, 0.8),
    (0.8, 0.9),
    (0.9, 0.95),
    (0.95, 0.99),
    (0.99, 1.000001)
]

for grid_name, grid in grids.items():

    print("\nEvaluating:", grid_name)

    all_reference = []
    all_approximation = []
    all_eligible = []

    # Keep per-layer metrics.
    for layer_id in sorted(layer_records):

        ref_parts = layer_records[layer_id]

        reference = np.concatenate(
            [x.reshape(-1) for x in ref_parts]
        )

        approximation, eligible = quantize_values(
            reference,
            grid
        )

        metrics = calculate_metrics(
            reference,
            approximation,
            eligible
        )

        layer_rows.append({
            "grid": grid_name,
            "layer": layer_id,
            **metrics
        })

        all_reference.append(reference)
        all_approximation.append(approximation)
        all_eligible.append(eligible)

    reference = np.concatenate(all_reference)
    approximation = np.concatenate(all_approximation)
    eligible = np.concatenate(all_eligible)

    metrics = calculate_metrics(
        reference,
        approximation,
        eligible
    )

    global_rows.append({
        "grid": grid_name,
        "terms": int(grid_name.split("_T")[-1]),
        "grid_size": len(grid),
        "k_min": min(
            min(x["exponents"])
            for x in grid
        ),
        "k_max": max(
            max(x["exponents"])
            for x in grid
        ),
        **metrics
    })

    # Decay-range analysis.
    for low, high in decay_buckets:

        mask = (
            (reference >= low)
            & (reference < high)
        )

        if not mask.any():
            continue

        bucket_metrics = calculate_metrics(
            reference[mask],
            approximation[mask],
            eligible[mask]
        )

        bucket_rows.append({
            "grid": grid_name,
            "decay_low": low,
            "decay_high": high,
            **bucket_metrics
        })

# -----------------------------
# 7. SAVE RESULTS
# -----------------------------

global_df = pd.DataFrame(global_rows)
layer_df = pd.DataFrame(layer_rows)
bucket_df = pd.DataFrame(bucket_rows)

global_path = os.path.join(
    OUT_DIR, "expanded_grid_global_results.csv"
)

layer_path = os.path.join(
    OUT_DIR, "expanded_grid_layer_results.csv"
)

bucket_path = os.path.join(
    OUT_DIR, "expanded_grid_decay_bucket_results.csv"
)

global_df.to_csv(global_path, index=False)
layer_df.to_csv(layer_path, index=False)
bucket_df.to_csv(bucket_path, index=False)

# -----------------------------
# 8. RANK CANDIDATES
# -----------------------------

# Do not rank only by MAE.
# Coverage and quantized-only error both matter.

global_df["candidate_score"] = (
    global_df["quantized_only_mae"]
    + 0.0005 * (100.0 - global_df["coverage_pct"])
)

ranked = global_df.sort_values(
    "candidate_score",
    ascending=True
).reset_index(drop=True)

ranked_path = os.path.join(
    OUT_DIR, "expanded_grid_best_candidates.csv"
)

ranked.to_csv(ranked_path, index=False)

# -----------------------------
# 9. PRINT RESULTS
# -----------------------------

print("\n" + "=" * 110)
print("GLOBAL EXPANDED GRID RESULTS")
print("=" * 110)

display_columns = [
    "grid",
    "grid_size",
    "coverage_pct",
    "fallback_pct",
    "overall_mae",
    "overall_rmse",
    "quantized_only_mae",
    "quantized_only_rmse",
    "quantized_only_max_abs"
]

print(
    global_df[display_columns]
    .sort_values("quantized_only_mae")
    .to_string(index=False)
)

print("\n" + "=" * 110)
print("TOP CANDIDATES")
print("=" * 110)

print(
    ranked[display_columns + ["candidate_score"]]
    .head(10)
    .to_string(index=False)
)

print("\n" + "=" * 110)
print("PER-LAYER RESULTS")
print("=" * 110)

for grid_name in grids:
    subset = layer_df[layer_df["grid"] == grid_name]

    print("\n", grid_name)
    print(
        subset[
            [
                "layer",
                "coverage_pct",
                "overall_mae",
                "quantized_only_mae",
                "quantized_only_rmse"
            ]
        ].to_string(index=False)
    )

# -----------------------------
# 10. SAVE EXPERIMENT METADATA
# -----------------------------

metadata = {
    "experiment": "expanded_dyadic_grid_sweep",
    "timestamp": datetime.utcnow().isoformat(),
    "records_path": RECORDS_PATH,
    "record_count": len(records),
    "layers_assumed": 24,
    "grid_configurations": GRID_CONFIGS,
    "term_counts": TERM_COUNTS,
    "grid_sizes": {
        key: len(value)
        for key, value in grids.items()
    },
    "metric_definitions": {
        "overall_mae": "Mean absolute error including exact fallbacks",
        "quantized_only_mae": "MAE only over eligible quantized values",
        "coverage_pct": "Percentage of coefficients represented by the grid",
        "candidate_score": (
            "Quantized-only MAE + 0.0005 * fallback percentage"
        )
    },
    "limitations": [
        "Coefficient-level experiment only",
        "Not a Triton performance benchmark",
        "Does not establish multiplication elimination",
        "Grid ranking score is a heuristic, not a validated objective",
        "Layer assignment assumes 24 consecutive calls per document"
    ]
}

metadata_path = os.path.join(
    OUT_DIR, "experiment_metadata.json"
)

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("\nSaved experiment files:")
print(global_path)
print(layer_path)
print(bucket_path)
print(ranked_path)
print(metadata_path)

print("\nEXPANDED GRID SWEEP COMPLETE")


Loaded records: 2400
K1_14_T1 | grid values: 14 | min decay: 0.5 | max decay: 0.99993896484375
K1_14_T2 | grid values: 104 | min decay: 0.25 | max decay: 0.9998779296875
K1_14_T3 | grid values: 545 | min decay: 0.125 | max decay: 0.99981689453125
K1_16_T1 | grid values: 16 | min decay: 0.5 | max decay: 0.9999847412109375
K1_16_T2 | grid values: 135 | min decay: 0.25 | max decay: 0.999969482421875
K1_16_T3 | grid values: 799 | min decay: 0.125 | max decay: 0.9999542236328125
K1_18_T1 | grid values: 18 | min decay: 0.5 | max decay: 0.9999961853027344
K1_18_T2 | grid values: 170 | min decay: 0.25 | max decay: 0.9999923706054688
K1_18_T3 | grid values: 1121 | min decay: 0.125 | max decay: 0.9999885559082031

Layer record counts:
Layer 00: 100 records
Layer 01: 100 records
Layer 02: 100 records
Layer 03: 100 records
Layer 04: 100 records
Layer 05: 100 records
Layer 06: 100 records
Layer 07: 100 records
Layer 08: 100 records
Layer 09: 100 records
Layer 10: 100 records
Layer 11: 100 records
L

/tmp/ipykernel_50/3420637663.py:457: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


In [110]:

# ============================================================
# SHIFT-SCAN: MODEL-LEVEL EXPANDED GRID EVALUATION
# ============================================================
# Compares:
#   1. Original Mamba-2
#   2. No-quantization round-trip control
#   3. K1_14_T2
#   4. K1_14_T3
#   5. K1_18_T3
#
# Measures:
#   - Token-weighted CE and perplexity
#   - Per-document CE
#   - Paired document bootstrap CI
#   - Quantization coverage and fallback
#   - Difference relative to round-trip control
#
# IMPORTANT:
#   This is a model-level numerical experiment.
#   It is NOT a multiplication-free Triton kernel benchmark.
# ============================================================

import os
import math
import itertools
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from mamba_ssm.ops.triton import ssd_combined as combined_mod

# -----------------------------
# 1. CHECK ENVIRONMENT
# -----------------------------

assert "model" in globals(), "Model variable not found."
assert "encoded_eval" in globals(), "encoded_eval not found."

assert len(encoded_eval) > 0, "No evaluation documents."

model.eval()

DEVICE = next(model.parameters()).device

OUT_DIR = (
    "/kaggle/working/shiftscan_final/"
    "expanded_grid_model_eval"
)
os.makedirs(OUT_DIR, exist_ok=True)

MAX_DOCS = min(100, len(encoded_eval))
BOOTSTRAP_SAMPLES = 20000
BOOTSTRAP_SEED = 2026

print("Device:", DEVICE)
print("Evaluation documents:", MAX_DOCS)

# -----------------------------
# 2. DEFINE QUANTIZATION GRIDS
# -----------------------------

def make_grid(k_min, k_max, terms):

    tuples = list(
        itertools.combinations_with_replacement(
            range(k_min, k_max + 1),
            terms
        )
    )

    grid = []

    for exponents in tuples:

        d = sum(2.0 ** (-k) for k in exponents)
        q = 1.0 - d

        if 0.0 < q < 1.0:
            grid.append((d, q, exponents))

    grid.sort(key=lambda x: x[0])

    return grid


CONFIGS = {
    "K1_14_T2": make_grid(1, 14, 2),
    "K1_14_T3": make_grid(1, 14, 3),
    "K1_18_T3": make_grid(1, 18, 3),
}

for name, grid in CONFIGS.items():
    print(
        name,
        "| values:", len(grid),
        "| decay range:",
        min(x[1] for x in grid),
        "to",
        max(x[1] for x in grid)
    )

# -----------------------------
# 3. SAVE ORIGINAL TRITON FUNCTION
# -----------------------------

original_cumsum = combined_mod._chunk_cumsum_fwd

# Verify that the forward implementation uses this function.
from mamba_ssm.ops.triton import ssd_combined

assert (
    ssd_combined._mamba_chunk_scan_combined_fwd
    .__globals__["_chunk_cumsum_fwd"]
    is original_cumsum
), "Unexpected _chunk_cumsum_fwd reference."

# -----------------------------
# 4. QUANTIZATION PATCH
# -----------------------------

def make_cumsum_patch(grid, stats, enabled=True):

    if grid is not None:

        grid_d = torch.tensor(
            [x[0] for x in grid],
            device=DEVICE,
            dtype=torch.float64
        )

        grid_q = torch.tensor(
            [x[1] for x in grid],
            device=DEVICE,
            dtype=torch.float64
        )

        log_grid_d = torch.log(grid_d)

        d_min = grid_d.min()
        d_max = grid_d.max()

    def patched_cumsum(
        dt,
        A,
        chunk_size,
        *args,
        **kwargs
    ):

        # First call: exact Mamba-2 dt semantics.
        # Includes dt_bias, softplus and dt_limit.
        dA_cumsum_exact, dt_out = original_cumsum(
            dt,
            A,
            chunk_size,
            *args,
            **kwargs
        )

        if not enabled:
            dA_quantized = None
            dt_quantized = None

        else:

            dt_effective = dt_out.float()

            A_float = A.detach().float().reshape(
                1, -1, 1, 1
            )

            if torch.any(A_float == 0):
                raise RuntimeError(
                    "Zero A value encountered; cannot reconstruct dt."
                )

            dA = dt_effective * A_float

            # Valid positions exclude chunk padding.
            batch, seqlen, nheads = dt.shape
            nchunks = dt_out.shape[2]

            chunk_ids = torch.arange(
                nchunks,
                device=DEVICE
            ).view(nchunks, 1)

            position_ids = torch.arange(
                chunk_size,
                device=DEVICE
            ).view(1, chunk_size)

            valid_positions = (
                chunk_ids * chunk_size + position_ids
            ) < seqlen

            valid_mask = valid_positions.view(
                1, 1, nchunks, chunk_size
            )

            decay = torch.exp(dA.double())

            finite = torch.isfinite(decay)

            eligible = (
                finite
                & (decay > 0)
                & (decay < 1)
                & ((1.0 - decay) >= d_min)
                & ((1.0 - decay) <= d_max)
                & valid_mask
            )

            # Nearest neighbor in log(1-decay) space.
            distance = 1.0 - decay

            safe_distance = torch.where(
                eligible,
                distance,
                torch.ones_like(distance)
            )

            log_distance = torch.log(safe_distance)

            right = torch.searchsorted(
                log_grid_d,
                log_distance.contiguous()
            )

            right = right.clamp(
                min=0,
                max=len(grid) - 1
            )

            left = (right - 1).clamp(
                min=0,
                max=len(grid) - 1
            )

            error_left = torch.abs(
                log_distance - log_grid_d[left]
            )

            error_right = torch.abs(
                log_distance - log_grid_d[right]
            )

            best = torch.where(
                error_left <= error_right,
                left,
                right
            )

            selected_decay = grid_q[best]

            # For exact fallback positions, preserve original dA.
            safe_selected = torch.where(
                eligible,
                selected_decay,
                torch.ones_like(selected_decay)
            )

            dA_quantized = torch.where(
                eligible,
                torch.log(safe_selected).float(),
                dA
            )

            # Reconstruct dt from quantized dA.
            dt_quantized_layout = (
                dA_quantized / A_float
            )

            # Convert [B,H,nchunks,chunk] to [B,seqlen,H].
            dt_quantized = (
                dt_quantized_layout
                .permute(0, 2, 3, 1)
                .reshape(
                    batch,
                    nchunks * chunk_size,
                    nheads
                )[:, :seqlen, :]
                .contiguous()
            )

            stats["total"] += int(valid_mask.sum().item())
            stats["quantized"] += int(eligible.sum().item())

        if not enabled:
            # Round-trip control:
            # reconstruct dt from exact effective dA.
            dt_effective = dt_out.float()

            A_float = A.detach().float().reshape(
                1, -1, 1, 1
            )

            dA_exact = dt_effective * A_float

            batch, seqlen, nheads = dt.shape
            nchunks = dt_out.shape[2]

            dt_quantized_layout = dA_exact / A_float

            dt_quantized = (
                dt_quantized_layout
                .permute(0, 2, 3, 1)
                .reshape(
                    batch,
                    nchunks * chunk_size,
                    nheads
                )[:, :seqlen, :]
                .contiguous()
            )

        # Second call: original Triton kernel computes
        # the cumulative sum of reconstructed/quantized dA.
        #
        # dt_bias=None and dt_softplus=False because dt_quantized
        # is already effective dt.
        dA_cumsum_quantized, _ = original_cumsum(
            dt_quantized,
            A,
            chunk_size,
            dt_bias=None,
            dt_softplus=False,
            dt_limit=(0.0, float("inf"))
        )

        return dA_cumsum_quantized, dt_out

    return patched_cumsum


# -----------------------------
# 5. MODEL EVALUATION FUNCTION
# -----------------------------

def evaluate_model():

    document_losses = []
    total_loss = 0.0
    total_tokens = 0

    with torch.inference_mode():

        for i in range(MAX_DOCS):

            input_ids = encoded_eval[i].to(DEVICE)

            output = model(input_ids)

            if isinstance(output, tuple):
                logits = output[0]
            elif hasattr(output, "logits"):
                logits = output.logits
            else:
                logits = output

            labels = input_ids[:, 1:]
            predictions = logits[:, :-1, :]

            loss_sum = F.cross_entropy(
                predictions.float().reshape(
                    -1, predictions.shape[-1]
                ),
                labels.reshape(-1),
                reduction="sum"
            )

            token_count = labels.numel()

            doc_ce = (
                loss_sum.item() / token_count
            )

            document_losses.append(doc_ce)

            total_loss += loss_sum.item()
            total_tokens += token_count

            if (i + 1) % 20 == 0:
                print(
                    f"Evaluated {i+1}/{MAX_DOCS}"
                )

    return {
        "document_losses": np.array(
            document_losses,
            dtype=np.float64
        ),
        "total_ce": total_loss / total_tokens,
        "ppl": math.exp(total_loss / total_tokens),
        "tokens": total_tokens
    }


# -----------------------------
# 6. RUN ORIGINAL BASELINE
# -----------------------------

results = {}
stats_by_method = {}

print("\nRunning ORIGINAL baseline...")

combined_mod._chunk_cumsum_fwd = original_cumsum

try:
    with torch.inference_mode():
        results["Baseline"] = evaluate_model()
finally:
    combined_mod._chunk_cumsum_fwd = original_cumsum

print(
    "Baseline CE:",
    results["Baseline"]["total_ce"]
)

print(
    "Baseline PPL:",
    results["Baseline"]["ppl"]
)

# -----------------------------
# 7. RUN ROUND-TRIP CONTROL
# -----------------------------

print("\nRunning round-trip control...")

control_stats = {
    "total": 0,
    "quantized": 0
}

combined_mod._chunk_cumsum_fwd = make_cumsum_patch(
    None,
    control_stats,
    enabled=False
)

try:
    results["RoundTrip_Control"] = evaluate_model()
finally:
    combined_mod._chunk_cumsum_fwd = original_cumsum

# -----------------------------
# 8. RUN QUANTIZATION CANDIDATES
# -----------------------------

for name, grid in CONFIGS.items():

    print("\n" + "=" * 70)
    print("Running:", name)
    print("=" * 70)

    stats = {
        "total": 0,
        "quantized": 0
    }

    combined_mod._chunk_cumsum_fwd = make_cumsum_patch(
        grid,
        stats,
        enabled=True
    )

    try:
        results[name] = evaluate_model()

    finally:
        combined_mod._chunk_cumsum_fwd = original_cumsum

    stats["fallback"] = (
        stats["total"] - stats["quantized"]
    )

    stats["coverage_pct"] = (
        100.0 * stats["quantized"] / stats["total"]
        if stats["total"] else 0.0
    )

    stats["fallback_pct"] = (
        100.0 * stats["fallback"] / stats["total"]
        if stats["total"] else 0.0
    )

    stats_by_method[name] = stats

    print("CE:", results[name]["total_ce"])
    print("PPL:", results[name]["ppl"])
    print("Coverage:", stats["coverage_pct"])
    print("Fallback:", stats["fallback_pct"])

# -----------------------------
# 9. PAIRED DOCUMENT BOOTSTRAP
# -----------------------------

baseline_docs = results["Baseline"]["document_losses"]
control_docs = results["RoundTrip_Control"]["document_losses"]

rng = np.random.default_rng(BOOTSTRAP_SEED)

def bootstrap_ci(values, samples=20000):

    values = np.asarray(values, dtype=np.float64)
    n = len(values)

    means = np.empty(samples)

    for i in range(samples):
        idx = rng.integers(0, n, size=n)
        means[i] = values[idx].mean()

    return (
        float(values.mean()),
        float(np.quantile(means, 0.025)),
        float(np.quantile(means, 0.975))
    )

summary_rows = []

for name, result in results.items():

    docs = result["document_losses"]

    delta_baseline = docs - baseline_docs
    delta_control = docs - control_docs

    mean_delta, ci_low, ci_high = bootstrap_ci(
        delta_baseline
    )

    control_mean, control_low, control_high = bootstrap_ci(
        delta_control
    )

    row = {
        "method": name,
        "mean_CE": result["total_ce"],
        "PPL": result["ppl"],
        "tokens": result["tokens"],
        "delta_CE_vs_baseline": (
            result["total_ce"]
            - results["Baseline"]["total_ce"]
        ),
        "relative_PPL_change_pct": (
            100.0 * (
                result["ppl"]
                / results["Baseline"]["ppl"] - 1.0
            )
        ),
        "paired_doc_delta_mean": mean_delta,
        "paired_doc_CI_low": ci_low,
        "paired_doc_CI_high": ci_high,
        "delta_CE_vs_roundtrip_control": (
            result["total_ce"]
            - results["RoundTrip_Control"]["total_ce"]
        ),
        "paired_control_delta_mean": control_mean,
        "paired_control_CI_low": control_low,
        "paired_control_CI_high": control_high
    }

    if name in stats_by_method:
        row.update(stats_by_method[name])

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

# -----------------------------
# 10. SAVE DOCUMENT-LEVEL RESULTS
# -----------------------------

summary_path = os.path.join(
    OUT_DIR,
    "expanded_grid_model_summary.csv"
)

summary_df.to_csv(summary_path, index=False)

doc_rows = []

for name, result in results.items():

    for i, loss in enumerate(result["document_losses"]):

        doc_rows.append({
            "method": name,
            "document_index": i,
            "document_CE": float(loss)
        })

doc_df = pd.DataFrame(doc_rows)

doc_path = os.path.join(
    OUT_DIR,
    "expanded_grid_document_losses.csv"
)

doc_df.to_csv(doc_path, index=False)

# -----------------------------
# 11. DISPLAY RESULTS
# -----------------------------

print("\n" + "=" * 110)
print("MODEL-LEVEL EXPANDED GRID RESULTS")
print("=" * 110)

cols = [
    "method",
    "mean_CE",
    "PPL",
    "delta_CE_vs_baseline",
    "relative_PPL_change_pct",
    "paired_doc_CI_low",
    "paired_doc_CI_high"
]

print(summary_df[cols].to_string(index=False))

print("\n" + "=" * 110)
print("QUANTIZATION COVERAGE")
print("=" * 110)

for name, stats in stats_by_method.items():

    print(
        f"{name}: "
        f"coverage={stats['coverage_pct']:.4f}% | "
        f"fallback={stats['fallback_pct']:.4f}% | "
        f"quantized={stats['quantized']:,} | "
        f"total={stats['total']:,}"
    )

print("\nRound-trip control CE:")
print(results["RoundTrip_Control"]["total_ce"])

print("\nRound-trip control PPL:")
print(results["RoundTrip_Control"]["ppl"])

print("\nOriginal function restored:")
print(
    combined_mod._chunk_cumsum_fwd
    is original_cumsum
)

print("\nSaved:")
print(summary_path)
print(doc_path)

print("\nMODEL-LEVEL GRID EVALUATION COMPLETE")


Device: cuda:0
Evaluation documents: 100
K1_14_T2 | values: 104 | decay range: 0.25 to 0.9998779296875
K1_14_T3 | values: 545 | decay range: 0.125 to 0.99981689453125
K1_18_T3 | values: 1121 | decay range: 0.125 to 0.9999885559082031

Running ORIGINAL baseline...
Evaluated 20/100
Evaluated 40/100
Evaluated 60/100
Evaluated 80/100
Evaluated 100/100
Baseline CE: 3.2179452036580782
Baseline PPL: 24.976745292087468

Running round-trip control...
Evaluated 20/100
Evaluated 40/100
Evaluated 60/100
Evaluated 80/100
Evaluated 100/100

Running: K1_14_T2
Evaluated 20/100
Evaluated 40/100
Evaluated 60/100
Evaluated 80/100
Evaluated 100/100
CE: 3.2182451325320467
PPL: 24.984237662711298
Coverage: 2126.5190188224824
Fallback: -2026.5190188224822

Running: K1_14_T3
Evaluated 20/100
Evaluated 40/100
Evaluated 60/100
Evaluated 80/100
Evaluated 100/100
CE: 3.217854760939309
PPL: 24.97448642948762
Coverage: 2168.824445666117
Fallback: -2068.824445666117

Running: K1_18_T3
Evaluated 20/100
Evaluated 40/1

In [111]:

import pandas as pd
import numpy as np

PATH = "/kaggle/working/shiftscan_final/expanded_grid_model_eval/expanded_grid_document_losses.csv"

df = pd.read_csv(PATH)

print("Columns:", df.columns.tolist())
print("Shape:", df.shape)
display(df.head())


Columns: ['method', 'document_index', 'document_CE']
Shape: (500, 3)


,method,document_index,document_CE
0,Baseline,0,3.710593
1,Baseline,1,3.701580
2,Baseline,2,2.721435
3,Baseline,3,3.373219
4,Baseline,4,3.581251


In [112]:

import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Load document-level results
# --------------------------------------------------
PATH = "/kaggle/working/shiftscan_final/expanded_grid_model_eval/expanded_grid_document_losses.csv"

df = pd.read_csv(PATH)

wide = df.pivot(
    index="document_index",
    columns="method",
    values="document_CE"
).sort_index()

print("Document count:", len(wide))
print("\nMethods:", wide.columns.tolist())

required = [
    "Baseline",
    "RoundTrip_Control",
    "K1_14_T2",
    "K1_14_T3",
    "K1_18_T3"
]

assert all(m in wide.columns for m in required)
assert not wide[required].isna().any().any()

# --------------------------------------------------
# 2. Paired bootstrap
# Negative difference = candidate has lower CE
# --------------------------------------------------
def paired_bootstrap(candidate, reference, n_boot=20000, seed=42):
    differences = (
        wide[candidate].to_numpy()
        - wide[reference].to_numpy()
    )

    rng = np.random.default_rng(seed)
    n = len(differences)

    boot_means = np.empty(n_boot)

    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot_means[i] = differences[idx].mean()

    low, high = np.percentile(boot_means, [2.5, 97.5])

    return {
        "candidate": candidate,
        "reference": reference,
        "mean_paired_delta_CE": differences.mean(),
        "CI95_low": low,
        "CI95_high": high,
        "candidate_better_docs": int((differences < 0).sum()),
        "reference_better_docs": int((differences > 0).sum()),
        "ties": int((differences == 0).sum()),
        "std_paired_delta": differences.std(ddof=1)
    }

# --------------------------------------------------
# 3. Compare each grid against BOTH controls
# --------------------------------------------------
candidates = ["K1_14_T2", "K1_14_T3", "K1_18_T3"]

results = []

for candidate in candidates:
    results.append(
        paired_bootstrap(candidate, "Baseline", seed=42)
    )
    results.append(
        paired_bootstrap(candidate, "RoundTrip_Control", seed=43)
    )

results_df = pd.DataFrame(results)

print("\nPAIRED DOCUMENT-LEVEL COMPARISONS")
display(results_df.round(8))

# --------------------------------------------------
# 4. Direct comparison between T3 candidates
# --------------------------------------------------
direct = paired_bootstrap(
    "K1_14_T3",
    "K1_18_T3",
    seed=44
)

print("\nK1_14_T3 vs K1_18_T3")
display(pd.DataFrame([direct]).round(8))

# --------------------------------------------------
# 5. Save results
# --------------------------------------------------
OUT = "/kaggle/working/shiftscan_final/expanded_grid_model_eval"

results_df.to_csv(
    f"{OUT}/paired_control_comparisons.csv",
    index=False
)

pd.DataFrame([direct]).to_csv(
    f"{OUT}/paired_t3_comparison.csv",
    index=False
)

print("\nSaved paired comparison results.")


Document count: 100

Methods: ['Baseline', 'K1_14_T2', 'K1_14_T3', 'K1_18_T3', 'RoundTrip_Control']

PAIRED DOCUMENT-LEVEL COMPARISONS


,candidate,reference,mean_paired_delta_CE,CI95_low,CI95_high,candidate_better_docs,reference_better_docs,ties,std_paired_delta
0,K1_14_T2,Baseline,0.000314,-0.000308,0.000936,48,52,0,0.003218
1,K1_14_T2,RoundTrip_Control,0.000226,-0.000367,0.000801,47,53,0,0.003006
2,K1_14_T3,Baseline,-0.000083,-0.000456,0.000285,48,52,0,0.001898
3,K1_14_T3,RoundTrip_Control,-0.000171,-0.000513,0.000166,51,49,0,0.001760
4,K1_18_T3,Baseline,-0.000034,-0.000373,0.000304,55,45,0,0.001740
5,K1_18_T3,RoundTrip_Control,-0.000122,-0.000432,0.000179,58,42,0,0.001574



K1_14_T3 vs K1_18_T3


,candidate,reference,mean_paired_delta_CE,CI95_low,CI95_high,candidate_better_docs,reference_better_docs,ties,std_paired_delta
0,K1_14_T3,K1_18_T3,-0.000049,-0.000331,0.00023,56,44,0,0.001453



Saved paired comparison results.


In [113]:

import os
import hashlib
import json
import torch
import pandas as pd
from datasets import load_dataset

# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------
SEED = 2026
N_HELDOUT = 500
MAX_LENGTH = 128

OUT_DIR = "/kaggle/working/shiftscan_final/heldout_500"
os.makedirs(OUT_DIR, exist_ok=True)

assert "encoded_eval" in globals(), "Original 100-document encoded_eval is missing."
assert "tokenizer" in globals(), "Tokenizer is missing."
assert "model" in globals(), "Model is missing."

print("Original evaluation documents:", len(encoded_eval))
print("Target held-out documents:", N_HELDOUT)

# --------------------------------------------------
# 2. Hash the original 100 token sequences
# --------------------------------------------------
def token_hash(input_ids):
    arr = input_ids.detach().cpu().long().contiguous().numpy()
    return hashlib.sha256(arr.tobytes()).hexdigest()

original_hashes = {
    token_hash(x)
    for x in encoded_eval
}

print("Unique original token hashes:", len(original_hashes))

# --------------------------------------------------
# 3. Stream FineWeb-Edu
# --------------------------------------------------
dataset = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="sample-10BT",
    split="train",
    streaming=True
)

heldout_tokens = []
metadata = []

accepted_index = 0
raw_index = 0

for item in dataset:
    raw_index += 1

    text = item.get("text", "").strip()

    if len(text) <= 100:
        continue

    # Assumes the original evaluation used the first
    # 100 documents satisfying this same text filter.
    if accepted_index < 100:
        accepted_index += 1
        continue

    accepted_index += 1

    encoded = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    input_ids = encoded.input_ids

    if input_ids.shape[1] < 2:
        continue

    h = token_hash(input_ids)

    heldout_tokens.append(input_ids)

    metadata.append({
        "heldout_index": len(heldout_tokens) - 1,
        "fineweb_accepted_index": accepted_index - 1,
        "raw_stream_index": raw_index - 1,
        "text_sha256": hashlib.sha256(
            text.encode("utf-8")
        ).hexdigest(),
        "token_sha256": h,
        "token_count": int(input_ids.shape[1]),
        "text_characters": len(text)
    })

    if len(heldout_tokens) % 100 == 0:
        print(f"Collected {len(heldout_tokens)}/{N_HELDOUT}")

    if len(heldout_tokens) >= N_HELDOUT:
        break

# --------------------------------------------------
# 4. Validate the held-out set
# --------------------------------------------------
assert len(heldout_tokens) == N_HELDOUT, (
    f"Expected {N_HELDOUT} documents, got {len(heldout_tokens)}"
)

heldout_hashes = [token_hash(x) for x in heldout_tokens]

overlap = set(heldout_hashes) & original_hashes

print("\nHELD-OUT VALIDATION")
print("Documents:", len(heldout_tokens))
print("Unique token sequences:", len(set(heldout_hashes)))
print("Overlap with original token sequences:", len(overlap))

assert len(overlap) == 0, (
    "Held-out token sequences overlap with the original evaluation. "
    "Do not proceed until the split is corrected."
)

# --------------------------------------------------
# 5. Save tokens, metadata and configuration
# --------------------------------------------------
torch.save(
    heldout_tokens,
    f"{OUT_DIR}/heldout_500_tokens.pt"
)

metadata_df = pd.DataFrame(metadata)
metadata_df.to_csv(
    f"{OUT_DIR}/heldout_500_metadata.csv",
    index=False
)

config = {
    "dataset": "HuggingFaceFW/fineweb-edu",
    "dataset_config": "sample-10BT",
    "documents": N_HELDOUT,
    "max_length": MAX_LENGTH,
    "original_documents_skipped": 100,
    "tokenizer_class": tokenizer.__class__.__name__,
    "model": "state-spaces/mamba2-130m",
    "seed": SEED,
    "unique_token_sequences": len(set(heldout_hashes)),
    "overlap_with_original": len(overlap)
}

with open(f"{OUT_DIR}/experiment_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("\nHeld-out set saved successfully.")
print("Directory:", OUT_DIR)
print("Metadata rows:", len(metadata_df))
print("Token length range:",
      metadata_df.token_count.min(),
      "to",
      metadata_df.token_count.max())
print("Mean token length:", metadata_df.token_count.mean())


Original evaluation documents: 100
Target held-out documents: 500
Unique original token hashes: 100


Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Collected 100/500
Collected 200/500
Collected 300/500
Collected 400/500
Collected 500/500

HELD-OUT VALIDATION
Documents: 500
Unique token sequences: 500
Overlap with original token sequences: 0

Held-out set saved successfully.
Directory: /kaggle/working/shiftscan_final/heldout_500
Metadata rows: 500
Token length range: 83 to 128
Mean token length: 127.438


In [114]:

import os
import torch
import pandas as pd

HELDOUT_DIR = "/kaggle/working/shiftscan_final/heldout_500"

heldout_tokens = torch.load(
    f"{HELDOUT_DIR}/heldout_500_tokens.pt",
    map_location="cpu",
    weights_only=True
)

metadata = pd.read_csv(
    f"{HELDOUT_DIR}/heldout_500_metadata.csv"
)

assert len(heldout_tokens) == 500
assert len(metadata) == 500

assert all(
    x.ndim == 2 and x.shape[0] == 1 and x.shape[1] >= 2
    for x in heldout_tokens
)

embedding_size = model.backbone.embedding.weight.shape[0]

assert max(
    int(x.max()) for x in heldout_tokens
) < embedding_size

print("Held-out evaluation ready.")
print("Documents:", len(heldout_tokens))
print("Embedding vocabulary:", embedding_size)
print("Device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)
print("Sequence length range:",
      min(x.shape[1] for x in heldout_tokens),
      max(x.shape[1] for x in heldout_tokens))
print("Mean sequence length:",
      sum(x.shape[1] for x in heldout_tokens) / len(heldout_tokens))


Held-out evaluation ready.
Documents: 500
Embedding vocabulary: 50288
Device: cuda:0
Model dtype: torch.float16
Sequence length range: 83 128
Mean sequence length: 127.438


In [115]:

import os
import json
import math
import itertools
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

# ============================================================
# 1. CONFIGURATION
# ============================================================

HELDOUT_DIR = "/kaggle/working/shiftscan_final/heldout_500"
OUT_DIR = f"{HELDOUT_DIR}/model_evaluation"
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = next(model.parameters()).device
model.eval()

heldout_tokens = torch.load(
    f"{HELDOUT_DIR}/heldout_500_tokens.pt",
    map_location="cpu",
    weights_only=True
)

assert len(heldout_tokens) == 500
assert all(x.shape[1] >= 2 for x in heldout_tokens)

print("Device:", DEVICE)
print("Documents:", len(heldout_tokens))
print("Model dtype:", next(model.parameters()).dtype)

# ============================================================
# 2. BUILD THE EXACT EXPANDED DYADIC GRIDS
# ============================================================

def make_grid(k_min, k_max, terms):
    """
    Construct unique decay values:
        a_q = 1 - sum_i 2^(-k_i)

    Repeated exponents are permitted.
    """
    pairs = itertools.combinations_with_replacement(
        range(k_min, k_max + 1), terms
    )

    rows = []

    for exponents in pairs:
        d = sum(2.0 ** (-k) for k in exponents)
        a = 1.0 - d

        if 0.0 < a < 1.0:
            rows.append((d, a, exponents))

    # Deduplicate identical decay values.
    unique = {}
    for d, a, exponents in rows:
        unique[round(d, 15)] = (d, a, exponents)

    rows = sorted(unique.values(), key=lambda z: z[0])

    return {
        "d": np.array([r[0] for r in rows], dtype=np.float64),
        "a": np.array([r[1] for r in rows], dtype=np.float64),
        "exponents": [r[2] for r in rows],
        "log_d": np.log(
            np.array([r[0] for r in rows], dtype=np.float64)
        )
    }


GRIDS = {
    "K1_14_T2": make_grid(1, 14, 2),
    "K1_14_T3": make_grid(1, 14, 3),
    "K1_18_T3": make_grid(1, 18, 3)
}

for name, grid in GRIDS.items():
    print(
        name,
        "| values:", len(grid["a"]),
        "| decay range:",
        float(grid["a"].min()),
        "to",
        float(grid["a"].max())
    )

# ============================================================
# 3. IMPORT THE ORIGINAL TRITON SSD CUMSUM FUNCTION
# ============================================================

import mamba_ssm.ops.triton.ssd_combined as combined_mod

original_cumsum = combined_mod._chunk_cumsum_fwd

assert callable(original_cumsum)

# ============================================================
# 4. CREATE TEMPORARY QUANTIZATION PATCH
# ============================================================

def make_cumsum_patch(mode, grid=None, stats=None):
    """
    mode:
        baseline is not patched
        roundtrip: reconstruct dt from dA without quantization
        quantized: quantize effective decay increments

    The original function is used to calculate effective dt,
    including dt_bias, softplus and dt_limit.

    The original Triton cumsum kernel then computes the
    cumulative quantized increments.
    """

    grid_d = None
    grid_log_d = None

    if grid is not None:
        grid_d = torch.tensor(
            grid["d"], device=DEVICE, dtype=torch.float32
        )
        grid_log_d = torch.log(grid_d)

    def patched_cumsum(*args, **kwargs):

        # Preserve the original call and semantics.
        result = original_cumsum(*args, **kwargs)

        original_dA_cumsum, original_dt_out = result

        dt_input = args[0] if len(args) > 0 else kwargs["dt"]
        A = args[1] if len(args) > 1 else kwargs["A"]
        chunk_size = (
            args[2] if len(args) > 2 else kwargs["chunk_size"]
        )

        # Effective dA increments, not the raw dt parameter.
        A_float = A.float()
        A_view = A_float.view(1, -1, 1, 1)

        dA = original_dt_out.float() * A_view

        if mode == "roundtrip":
            dA_new = dA

        elif mode == "quantized":

            decay = torch.exp(dA)
            distance = 1.0 - decay

            # Padded positions have dt_out == 0.
            valid = original_dt_out > 0

            min_d = grid_d[0]
            max_d = grid_d[-1]

            eligible = (
                valid
                & torch.isfinite(decay)
                & torch.isfinite(distance)
                & (distance >= min_d)
                & (distance <= max_d)
                & (distance > 0)
            )

            # Safe values for entries outside the quantizable range.
            safe_distance = torch.where(
                eligible,
                distance,
                torch.ones_like(distance)
            )

            log_distance = torch.log(safe_distance)

            # Nearest neighbour in log-distance space.
            flat = log_distance.reshape(-1)

            right = torch.searchsorted(
                grid_log_d, flat
            ).clamp(0, len(grid_log_d) - 1)

            left = (right - 1).clamp(0, len(grid_log_d) - 1)

            left_error = torch.abs(
                flat - grid_log_d[left]
            )
            right_error = torch.abs(
                flat - grid_log_d[right]
            )

            chosen = torch.where(
                left_error <= right_error,
                left,
                right
            )

            chosen_d = grid_d[chosen].reshape_as(distance)

            # Convert quantized decay back to log-decay increments.
            quantized_dA = torch.log1p(-chosen_d)

            dA_new = torch.where(
                eligible,
                quantized_dA,
                dA
            )

            # Count valid coefficients across ALL heads.
            if stats is not None:
                stats["total"] += int(valid.numel()) if valid.shape == original_dt_out.shape else int(valid.expand_as(decay).numel())
                stats["valid"] += int(valid.sum().item())
                stats["quantized"] += int(eligible.sum().item())

        else:
            raise ValueError(f"Unknown patch mode: {mode}")

        # Recover effective dt from dA = dt * A.
        # Mamba-2 A values are negative and nonzero.
        safe_A = torch.where(
            A_view != 0,
            A_view,
            torch.ones_like(A_view)
        )

        dt_quantized = dA_new / safe_A

        # Convert [B,H,nchunks,chunk] back to [B,L,H].
        B, H, n_chunks, chunk = dt_quantized.shape

        dt_sequence = (
            dt_quantized
            .permute(0, 2, 3, 1)
            .reshape(B, n_chunks * chunk, H)
        )

        original_length = dt_input.shape[1]
        dt_sequence = dt_sequence[:, :original_length, :].contiguous()

        # Run the original Triton cumsum with the reconstructed dt.
        # Bias/softplus must NOT be applied a second time.
        new_result = original_cumsum(
            dt_sequence,
            A,
            chunk_size,
            dt_bias=None,
            dt_softplus=False,
            dt_limit=(0.0, float("inf"))
        )

        new_cumsum, _ = new_result

        # Return original dt_out because downstream kernels use dt.
        return new_cumsum, original_dt_out

    return patched_cumsum


# ============================================================
# 5. MODEL EVALUATION
# ============================================================

def get_logits(output):
    if hasattr(output, "logits"):
        return output.logits

    if isinstance(output, (tuple, list)):
        return output[0]

    return output


def evaluate_documents():
    rows = []

    total_loss_sum = 0.0
    total_tokens = 0

    with torch.inference_mode():

        for index, cpu_ids in enumerate(
            tqdm(heldout_tokens, desc="Evaluating documents")
        ):

            input_ids = cpu_ids.to(DEVICE)

            output = model(input_ids)
            logits = get_logits(output)

            # Next-token prediction.
            pred = logits[:, :-1, :].float()
            target = input_ids[:, 1:]

            token_count = target.numel()

            loss_sum = F.cross_entropy(
                pred.reshape(-1, pred.shape[-1]),
                target.reshape(-1),
                reduction="sum"
            )

            doc_ce = float((loss_sum / token_count).item())

            total_loss_sum += float(loss_sum.item())
            total_tokens += token_count

            rows.append({
                "method": CURRENT_METHOD,
                "document_index": index,
                "document_CE": doc_ce,
                "target_tokens": token_count
            })

            del output, logits, pred, target, loss_sum

    mean_ce = total_loss_sum / total_tokens
    ppl = math.exp(mean_ce)

    return rows, mean_ce, ppl, total_tokens


# ============================================================
# 6. RUN BASELINE, CONTROLS AND CANDIDATES
# ============================================================

all_rows = []
summary = []

CURRENT_METHOD = "Baseline"

print("\nRunning original baseline...")
baseline_rows, baseline_ce, baseline_ppl, baseline_tokens = (
    evaluate_documents()
)

all_rows.extend(baseline_rows)

summary.append({
    "method": "Baseline",
    "mean_CE": baseline_ce,
    "PPL": baseline_ppl,
    "target_tokens": baseline_tokens
})

print("Baseline CE:", baseline_ce)
print("Baseline PPL:", baseline_ppl)

# Save baseline before other runs.
pd.DataFrame(all_rows).to_csv(
    f"{OUT_DIR}/heldout_document_losses_partial.csv",
    index=False
)

# Round-trip control.
CURRENT_METHOD = "RoundTrip_Control"

print("\nRunning round-trip control...")
combined_mod._chunk_cumsum_fwd = make_cumsum_patch(
    mode="roundtrip"
)

try:
    rows, ce, ppl, token_count = evaluate_documents()
finally:
    combined_mod._chunk_cumsum_fwd = original_cumsum

all_rows.extend(rows)

summary.append({
    "method": CURRENT_METHOD,
    "mean_CE": ce,
    "PPL": ppl,
    "target_tokens": token_count
})

print("Round-trip CE:", ce)
print("Round-trip PPL:", ppl)

# Quantization candidates.
coverage_results = []

for method_name, grid in GRIDS.items():

    print("\n" + "=" * 65)
    print("Running:", method_name)
    print("=" * 65)

    CURRENT_METHOD = method_name

    stats = {
        "total": 0,
        "valid": 0,
        "quantized": 0
    }

    combined_mod._chunk_cumsum_fwd = make_cumsum_patch(
        mode="quantized",
        grid=grid,
        stats=stats
    )

    try:
        rows, ce, ppl, token_count = evaluate_documents()
    finally:
        combined_mod._chunk_cumsum_fwd = original_cumsum

    all_rows.extend(rows)

    summary.append({
        "method": method_name,
        "mean_CE": ce,
        "PPL": ppl,
        "delta_CE_vs_baseline": ce - baseline_ce,
        "relative_PPL_change_pct": (
            (ppl / baseline_ppl) - 1
        ) * 100,
        "target_tokens": token_count
    })

    valid_total = stats["valid"]
    quantized_total = stats["quantized"]

    coverage = (
        100.0 * quantized_total / valid_total
        if valid_total else float("nan")
    )

    fallback = 100.0 - coverage

    coverage_results.append({
        "method": method_name,
        "grid_size": len(grid["a"]),
        "valid_coefficients": valid_total,
        "quantized_coefficients": quantized_total,
        "fallback_coefficients": valid_total - quantized_total,
        "coverage_pct": coverage,
        "fallback_pct": fallback
    })

    print("CE:", ce)
    print("PPL:", ppl)
    print("Valid coefficients:", valid_total)
    print("Quantized coefficients:", quantized_total)
    print("Coverage:", coverage, "%")
    print("Fallback:", fallback, "%")

    # Incremental saving.
    pd.DataFrame(all_rows).to_csv(
        f"{OUT_DIR}/heldout_document_losses_partial.csv",
        index=False
    )

# ============================================================
# 7. PAIRED DOCUMENT BOOTSTRAP
# ============================================================

document_df = pd.DataFrame(all_rows)

wide = document_df.pivot(
    index="document_index",
    columns="method",
    values="document_CE"
).sort_index()

assert len(wide) == 500
assert not wide.isna().any().any()

def paired_bootstrap(candidate, reference, n_boot=20000, seed=42):

    differences = (
        wide[candidate].to_numpy()
        - wide[reference].to_numpy()
    )

    rng = np.random.default_rng(seed)
    n = len(differences)

    boot_means = np.empty(n_boot)

    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot_means[i] = differences[idx].mean()

    low, high = np.percentile(
        boot_means, [2.5, 97.5]
    )

    return {
        "candidate": candidate,
        "reference": reference,
        "mean_paired_delta_CE": differences.mean(),
        "CI95_low": low,
        "CI95_high": high,
        "candidate_better_docs": int((differences < 0).sum()),
        "reference_better_docs": int((differences > 0).sum()),
        "ties": int((differences == 0).sum())
    }


paired_results = []

for candidate in GRIDS:

    paired_results.append(
        paired_bootstrap(candidate, "Baseline", seed=42)
    )

    paired_results.append(
        paired_bootstrap(candidate, "RoundTrip_Control", seed=43)
    )

# Direct T3 comparison.
paired_results.append(
    paired_bootstrap(
        "K1_14_T3",
        "K1_18_T3",
        seed=44
    )
)

# ============================================================
# 8. SAVE ALL RESULTS
# ============================================================

summary_df = pd.DataFrame(summary)
coverage_df = pd.DataFrame(coverage_results)
paired_df = pd.DataFrame(paired_results)

document_df.to_csv(
    f"{OUT_DIR}/heldout_document_losses.csv",
    index=False
)

summary_df.to_csv(
    f"{OUT_DIR}/heldout_model_summary.csv",
    index=False
)

coverage_df.to_csv(
    f"{OUT_DIR}/heldout_quantization_coverage.csv",
    index=False
)

paired_df.to_csv(
    f"{OUT_DIR}/heldout_paired_comparisons.csv",
    index=False
)

print("\n" + "=" * 75)
print("500-DOCUMENT HELD-OUT EVALUATION COMPLETE")
print("=" * 75)

print("\nMODEL SUMMARY")
display(summary_df.round(8))

print("\nQUANTIZATION COVERAGE")
display(coverage_df.round(5))

print("\nPAIRED DOCUMENT-LEVEL COMPARISONS")
display(paired_df.round(8))

print("\nOriginal cumsum function restored:",
      combined_mod._chunk_cumsum_fwd is original_cumsum)

print("\nSaved results to:", OUT_DIR)


Device: cuda:0
Documents: 500
Model dtype: torch.float16
K1_14_T2 | values: 104 | decay range: 0.25 to 0.9998779296875
K1_14_T3 | values: 467 | decay range: 0.125 to 0.99981689453125
K1_18_T3 | values: 985 | decay range: 0.125 to 0.9999885559082031

Running original baseline...


Evaluating documents:   0%|          | 0/500 [00:00<?, ?it/s]

Baseline CE: 3.1942716685492285
Baseline PPL: 24.392401467896843

Running round-trip control...


Evaluating documents:   0%|          | 0/500 [00:00<?, ?it/s]

Round-trip CE: 3.194266652523124
Round-trip PPL: 24.392279115281188

Running: K1_14_T2


Evaluating documents:   0%|          | 0/500 [00:00<?, ?it/s]

... (7 outputs left out)

,method,grid_size,valid_coefficients,quantized_coefficients,fallback_coefficients,coverage_pct,fallback_pct
0,K1_14_T2,104,36702143,32492144,4209999,88.52928,11.47072
1,K1_14_T3,467,36702143,33139987,3562156,90.29442,9.70558
2,K1_18_T3,985,36702143,33818311,2883832,92.14261,7.85739



PAIRED DOCUMENT-LEVEL COMPARISONS


,candidate,reference,mean_paired_delta_CE,CI95_low,CI95_high,candidate_better_docs,reference_better_docs,ties
0,K1_14_T2,Baseline,0.000376,0.000141,0.000613,223,277,0
1,K1_14_T2,RoundTrip_Control,0.000383,0.000141,0.000622,235,265,0
2,K1_14_T3,Baseline,-0.000066,-0.000205,0.000074,251,249,0
3,K1_14_T3,RoundTrip_Control,-0.000059,-0.000200,0.000085,259,241,0
4,K1_18_T3,Baseline,-0.000037,-0.000179,0.000106,254,246,0
5,K1_18_T3,RoundTrip_Control,-0.000030,-0.000171,0.000111,253,247,0
6,K1_14_T3,K1_18_T3,-0.000029,-0.000155,0.000096,258,242,0



Original cumsum function restored: True

Saved results to: /kaggle/working/shiftscan_final/heldout_500/model_evaluation


In [116]:

import os
import json
import itertools
import numpy as np
import pandas as pd

OLD_DIR = "/kaggle/working/shiftscan_final/expanded_grid_sweep"
NEW_DIR = "/kaggle/working/shiftscan_final/heldout_500/model_evaluation"

# Current evaluation grid-generation rule
def audit_grid(k_min, k_max, terms):
    values = set()

    for exponents in itertools.combinations_with_replacement(
        range(k_min, k_max + 1), terms
    ):
        d = sum(2.0 ** (-k) for k in exponents)
        a = 1.0 - d

        if 0 < a < 1:
            values.add(round(d, 15))

    return len(values)

current_counts = {
    "K1_14_T2": audit_grid(1, 14, 2),
    "K1_14_T3": audit_grid(1, 14, 3),
    "K1_18_T3": audit_grid(1, 18, 3)
}

print("CURRENT GRID COUNTS")
print(current_counts)

# Compare with previously saved coefficient sweep.
global_path = f"{OLD_DIR}/expanded_grid_global_results.csv"

if os.path.exists(global_path):
    old = pd.read_csv(global_path)

    print("\nPREVIOUS SWEEP GRID COUNTS")
    display(old[["grid", "grid_size", "coverage_pct"]])

else:
    print("Previous global results CSV not found.")

# Read original experiment metadata.
metadata_path = f"{OLD_DIR}/experiment_metadata.json"

if os.path.exists(metadata_path):
    with open(metadata_path) as f:
        old_metadata = json.load(f)

    print("\nPREVIOUS EXPERIMENT METADATA")
    print(json.dumps(old_metadata, indent=2)[:8000])

else:
    print("Previous experiment metadata not found.")

# Read current evaluation summary.
summary_path = f"{NEW_DIR}/heldout_model_summary.csv"

if os.path.exists(summary_path):
    print("\nCURRENT MODEL EVALUATION")
    display(pd.read_csv(summary_path))

print("\nGRID AUDIT COMPLETE")


CURRENT GRID COUNTS
{'K1_14_T2': 104, 'K1_14_T3': 467, 'K1_18_T3': 985}

PREVIOUS SWEEP GRID COUNTS


,grid,grid_size,coverage_pct
0,K1_14_T1,14,79.672965
1,K1_14_T2,104,88.604741
2,K1_14_T3,545,90.368094
3,K1_16_T1,16,80.172842
4,K1_16_T2,135,89.608898
5,K1_16_T3,799,91.818176
6,K1_18_T1,18,80.286024
7,K1_18_T2,170,89.843580
8,K1_18_T3,1121,92.184841



PREVIOUS EXPERIMENT METADATA
{
  "experiment": "expanded_dyadic_grid_sweep",
  "timestamp": "2026-11-15T05:28:53.260385",
  "records_path": "/kaggle/working/shiftscan_final/corrected_effective_decay_records_100doc.pt",
  "record_count": 2400,
  "layers_assumed": 24,
  "grid_configurations": [
    {
      "name": "K1_14",
      "k_min": 1,
      "k_max": 14
    },
    {
      "name": "K1_16",
      "k_min": 1,
      "k_max": 16
    },
    {
      "name": "K1_18",
      "k_min": 1,
      "k_max": 18
    }
  ],
  "term_counts": [
    1,
    2,
    3
  ],
  "grid_sizes": {
    "K1_14_T1": 14,
    "K1_14_T2": 104,
    "K1_14_T3": 545,
    "K1_16_T1": 16,
    "K1_16_T2": 135,
    "K1_16_T3": 799,
    "K1_18_T1": 18,
    "K1_18_T2": 170,
    "K1_18_T3": 1121
  },
  "metric_definitions": {
    "overall_mae": "Mean absolute error including exact fallbacks",
    "quantized_only_mae": "MAE only over eligible quantized values",
    "coverage_pct": "Percentage of coefficients represented by the gr

,method,mean_CE,PPL,target_tokens,delta_CE_vs_baseline,relative_PPL_change_pct
0,Baseline,3.194272,24.392401,63219,NaN,NaN
1,RoundTrip_Control,3.194267,24.392279,63219,NaN,NaN
2,K1_14_T2,3.194639,24.401359,63219,0.000367,0.036722
3,K1_14_T3,3.194211,24.390928,63219,-0.000060,-0.006041
4,K1_18_T3,3.194240,24.391638,63219,-0.000031,-0.003130



GRID AUDIT COMPLETE


In [117]:

import pandas as pd
import os

path = (
    "/kaggle/working/shiftscan_final/heldout_500/"
    "model_evaluation/heldout_document_losses.csv"
)

assert os.path.exists(path), f"File not found: {path}"

doc_df = pd.read_csv(path)

print("Rows:", len(doc_df))
print("\nAvailable columns:")
for col in doc_df.columns:
    print(" -", col)

print("\nFirst five rows:")
display(doc_df.head())

print("\nMissing values:")
display(doc_df.isna().sum())


Rows: 2500

Available columns:
 - method
 - document_index
 - document_CE
 - target_tokens

First five rows:


,method,document_index,document_CE,target_tokens
0,Baseline,0,2.952189,127
1,Baseline,1,3.666725,127
2,Baseline,2,3.462435,127
3,Baseline,3,2.823172,127
4,Baseline,4,2.971298,127



Missing values:


method            0
document_index    0
document_CE       0
target_tokens     0
dtype: int64

In [118]:

import numpy as np
import pandas as pd

# Load document-level results
path = (
    "/kaggle/working/shiftscan_final/heldout_500/"
    "model_evaluation/heldout_document_losses.csv"
)
df = pd.read_csv(path)

# Convert to aligned document-by-method matrices
ce = df.pivot(
    index="document_index",
    columns="method",
    values="document_CE"
).sort_index()

tokens = df.pivot(
    index="document_index",
    columns="method",
    values="target_tokens"
).sort_index()

# Validate paired evaluation
assert not ce.isna().any().any(), "Missing document-method CE values"
assert not tokens.isna().any().any(), "Missing token counts"
assert (tokens.nunique(axis=1) == 1).all(), (
    "Target token counts differ between methods for a document"
)

methods = list(ce.columns)
print("Methods:", methods)
print("Documents:", len(ce))

# Convert to NumPy arrays
ce = ce.to_numpy(dtype=np.float64)
tokens = tokens.iloc[:, 0].to_numpy(dtype=np.float64)

method_idx = {m: i for i, m in enumerate(methods)}

# Verify token-weighted global CE against the saved evaluation summary
global_ce = np.sum(ce * tokens[:, None], axis=0) / np.sum(tokens)

print("\nRecomputed global token-weighted CE:")
for m, value in zip(methods, global_ce):
    print(f"{m:20s}: {value:.10f}")

# Paired token-weighted bootstrap
N_BOOT = 20000
BATCH_SIZE = 1000
rng = np.random.default_rng(20261115)
n_docs = len(tokens)

comparisons = [
    ("RoundTrip_Control", "Baseline"),
    ("K1_14_T2", "Baseline"),
    ("K1_14_T2", "RoundTrip_Control"),
    ("K1_14_T3", "Baseline"),
    ("K1_14_T3", "RoundTrip_Control"),
    ("K1_18_T3", "Baseline"),
    ("K1_18_T3", "RoundTrip_Control"),
    ("K1_14_T3", "K1_18_T3"),
]

results = []

for candidate, reference in comparisons:
    ci = method_idx[candidate]
    ri = method_idx[reference]

    # Point estimate: difference between global token-weighted CE values
    candidate_global = np.sum(ce[:, ci] * tokens) / np.sum(tokens)
    reference_global = np.sum(ce[:, ri] * tokens) / np.sum(tokens)
    delta = candidate_global - reference_global

    boot_deltas = []

    # Resample documents in paired fashion, preserving token weights
    for start in range(0, N_BOOT, BATCH_SIZE):
        batch = min(BATCH_SIZE, N_BOOT - start)

        indices = rng.integers(
            0, n_docs, size=(batch, n_docs)
        )

        sampled_tokens = tokens[indices]
        denominator = sampled_tokens.sum(axis=1)

        candidate_ce = (
            ce[indices, ci] * sampled_tokens
        ).sum(axis=1) / denominator

        reference_ce = (
            ce[indices, ri] * sampled_tokens
        ).sum(axis=1) / denominator

        boot_deltas.append(candidate_ce - reference_ce)

    boot_deltas = np.concatenate(boot_deltas)

    low, high = np.percentile(boot_deltas, [2.5, 97.5])

    results.append({
        "candidate": candidate,
        "reference": reference,
        "delta_CE": delta,
        "CI95_low": low,
        "CI95_high": high,
        "relative_PPL_change_pct": 100 * np.expm1(delta),
        "PPL_change_CI_low_pct": 100 * np.expm1(low),
        "PPL_change_CI_high_pct": 100 * np.expm1(high),
        "bootstrap_mean": boot_deltas.mean(),
        "CI_excludes_zero": bool(low > 0 or high < 0)
    })

results_df = pd.DataFrame(results)

print("\nPAIRED TOKEN-WEIGHTED BOOTSTRAP RESULTS")
display(results_df.round(8))

# Save results
out_dir = (
    "/kaggle/working/shiftscan_final/"
    "heldout_500/model_evaluation"
)
out_path = f"{out_dir}/token_weighted_bootstrap_20000.csv"

results_df.to_csv(out_path, index=False)

print("\nSaved:", out_path)
print("Bootstrap resamples per comparison:", N_BOOT)


Methods: ['Baseline', 'K1_14_T2', 'K1_14_T3', 'K1_18_T3', 'RoundTrip_Control']
Documents: 500

Recomputed global token-weighted CE:
Baseline            : 3.1942716573
K1_14_T2            : 3.1946388084
K1_14_T3            : 3.1942112527
K1_18_T3            : 3.1942403641
RoundTrip_Control   : 3.1942666406

PAIRED TOKEN-WEIGHTED BOOTSTRAP RESULTS


,candidate,reference,delta_CE,CI95_low,CI95_high,relative_PPL_change_pct,PPL_change_CI_low_pct,PPL_change_CI_high_pct,bootstrap_mean,CI_excludes_zero
0,RoundTrip_Control,Baseline,-0.000005,-0.000127,0.000118,-0.000502,-0.012725,0.011831,-0.000005,False
1,K1_14_T2,Baseline,0.000367,0.000136,0.000597,0.036722,0.013630,0.059710,0.000368,True
2,K1_14_T2,RoundTrip_Control,0.000372,0.000137,0.000615,0.037224,0.013674,0.061471,0.000373,True
3,K1_14_T3,Baseline,-0.000060,-0.000197,0.000077,-0.006040,-0.019653,0.007692,-0.000061,False
4,K1_14_T3,RoundTrip_Control,-0.000055,-0.000198,0.000089,-0.005539,-0.019794,0.008892,-0.000055,False
5,K1_18_T3,Baseline,-0.000031,-0.000174,0.000108,-0.003129,-0.017422,0.010782,-0.000032,False
6,K1_18_T3,RoundTrip_Control,-0.000026,-0.000168,0.000114,-0.002628,-0.016843,0.011413,-0.000027,False
7,K1_14_T3,K1_18_T3,-0.000029,-0.000153,0.000093,-0.002911,-0.015310,0.009309,-0.000030,False



Saved: /kaggle/working/shiftscan_final/heldout_500/model_evaluation/token_weighted_bootstrap_20000.csv
Bootstrap resamples per comparison: 20000


In [119]:

import itertools
import pandas as pd

def inspect_grid(k_min, k_max, terms, name):
    raw = []

    for exponents in itertools.combinations_with_replacement(
        range(k_min, k_max + 1), terms
    ):
        d = sum(2.0 ** (-k) for k in exponents)
        a = 1.0 - d

        if 0.0 < a < 1.0:
            raw.append((round(d, 15), d, a, exponents))

    # Group different exponent representations of the same coefficient
    grouped = {}

    for rounded_d, d, a, exponents in raw:
        if rounded_d not in grouped:
            grouped[rounded_d] = {
                "grid": name,
                "decay": a,
                "decay_distance": d,
                "canonical_exponents": exponents,
                "representation_count": 0
            }

        grouped[rounded_d]["representation_count"] += 1

    unique = list(grouped.values())

    print(f"\n{name}")
    print("Valid exponent combinations:", len(raw))
    print("Unique numerical coefficients:", len(unique))
    print("Duplicate representations:", len(raw) - len(unique))

    return unique

configs = [
    (1, 14, 2, "K1_14_T2"),
    (1, 14, 3, "K1_14_T3"),
    (1, 18, 3, "K1_18_T3")
]

all_grids = []

for k_min, k_max, terms, name in configs:
    all_grids.extend(
        inspect_grid(k_min, k_max, terms, name)
    )

grid_df = pd.DataFrame(all_grids)

out_path = (
    "/kaggle/working/shiftscan_final/"
    "canonical_dyadic_grid_definitions.csv"
)

grid_df.to_csv(out_path, index=False)

print("\nCANONICAL GRID SUMMARY")
display(
    grid_df.groupby("grid").agg(
        unique_coefficients=("decay", "count"),
        total_representations=("representation_count", "sum"),
        duplicate_representations=(
            "representation_count",
            lambda x: x.sum() - len(x)
        )
    ).reset_index()
)

print("\nSaved canonical grid:", out_path)



K1_14_T2
Valid exponent combinations: 104
Unique numerical coefficients: 104
Duplicate representations: 0

K1_14_T3
Valid exponent combinations: 545
Unique numerical coefficients: 467
Duplicate representations: 78

K1_18_T3
Valid exponent combinations: 1121
Unique numerical coefficients: 985
Duplicate representations: 136

CANONICAL GRID SUMMARY


,grid,unique_coefficients,total_representations,duplicate_representations
0,K1_14_T2,104,104,0
1,K1_14_T3,467,545,78
2,K1_18_T3,985,1121,136



Saved canonical grid: /kaggle/working/shiftscan_final/canonical_dyadic_grid_definitions.csv


In [120]:

import pandas as pd

path = (
    "/kaggle/working/shiftscan_final/"
    "canonical_dyadic_grid_definitions.csv"
)

grid_df = pd.read_csv(path)

print("Shape:", grid_df.shape)
print("Columns:", grid_df.columns.tolist())
print("\nFirst 15 rows:")
display(grid_df.head(15))

print("\nGrid counts:")
if "grid" in grid_df.columns:
    display(grid_df.groupby("grid").size())

print("\nDuplicate numerical coefficients:")
possible_cols = [
    c for c in grid_df.columns
    if "coefficient" in c.lower()
    or c.lower() in ["a", "decay", "value"]
]
print("Possible coefficient columns:", possible_cols)


Shape: (1556, 5)
Columns: ['grid', 'decay', 'decay_distance', 'canonical_exponents', 'representation_count']

First 15 rows:


,grid,decay,decay_distance,canonical_exponents,representation_count
0,K1_14_T2,0.250000,0.750000,"(1, 2)",1
1,K1_14_T2,0.375000,0.625000,"(1, 3)",1
2,K1_14_T2,0.437500,0.562500,"(1, 4)",1
3,K1_14_T2,0.468750,0.531250,"(1, 5)",1
4,K1_14_T2,0.484375,0.515625,"(1, 6)",1
5,K1_14_T2,0.492188,0.507812,"(1, 7)",1
6,K1_14_T2,0.496094,0.503906,"(1, 8)",1
7,K1_14_T2,0.498047,0.501953,"(1, 9)",1
8,K1_14_T2,0.499023,0.500977,"(1, 10)",1
9,K1_14_T2,0.499512,0.500488,"(1, 11)",1



Grid counts:


grid
K1_14_T2    104
K1_14_T3    467
K1_18_T3    985
dtype: int64


Duplicate numerical coefficients:
Possible coefficient columns: ['decay']


In [121]:

import ast
import numpy as np
import pandas as pd

GRID_PATH = (
    "/kaggle/working/shiftscan_final/"
    "canonical_dyadic_grid_definitions.csv"
)

grid_df = pd.read_csv(GRID_PATH)

# Round signed integer division to nearest, ties away from zero.
def round_div_signed(numerator, denominator):
    numerator = np.asarray(numerator, dtype=np.int64)
    magnitude = np.abs(numerator)

    rounded = (
        magnitude + denominator // 2
    ) // denominator

    return np.where(numerator < 0, -rounded, rounded)


# Round signed right shift to nearest, ties away from zero.
def round_shift_signed(x, k):
    x = np.asarray(x, dtype=np.int64)
    return round_div_signed(x, 1 << int(k))


# Exact reference: round(H * (1 - sum(2^-k))).
# Uses integer arithmetic, avoiding floating-point ambiguity.
def reference_decay(H, exponents):
    H = np.asarray(H, dtype=np.int64)

    K = max(exponents)
    denominator = 1 << K

    numerator_coefficient = (
        denominator
        - sum(1 << (K - k) for k in exponents)
    )

    numerator = H * numerator_coefficient

    return round_div_signed(numerator, denominator)


# Proposed multiplication-reduced operation.
def shift_subtract_decay(H, exponents):
    H = np.asarray(H, dtype=np.int64)

    result = H.copy()

    for k in exponents:
        result -= round_shift_signed(H, k)

    return result


# Deterministic test values:
# random signed states plus boundary and small-magnitude cases.
rng = np.random.default_rng(20261115)

random_states = rng.integers(
    low=-(2**31),
    high=2**31,
    size=8192,
    dtype=np.int64
)

boundary_states = np.array([
    -(2**31), -(2**31) + 1,
    -1000000, -1001, -100, -17, -3, -2, -1,
    0, 1, 2, 3, 17, 100, 1001, 1000000,
    2**31 - 2, 2**31 - 1
], dtype=np.int64)

test_states = np.concatenate([
    random_states,
    boundary_states
])

results = []

for grid_name, group in grid_df.groupby("grid"):
    total = 0
    exact = 0
    abs_errors = []
    max_error = 0

    for _, row in group.iterrows():
        exponents = ast.literal_eval(
            str(row["canonical_exponents"])
        )

        reference = reference_decay(
            test_states, exponents
        )

        proposed = shift_subtract_decay(
            test_states, exponents
        )

        error = proposed - reference
        abs_error = np.abs(error)

        total += len(test_states)
        exact += np.count_nonzero(error == 0)
        abs_errors.append(abs_error)

        max_error = max(
            max_error,
            int(abs_error.max())
        )

    all_errors = np.concatenate(abs_errors)

    results.append({
        "grid": grid_name,
        "coefficients": len(group),
        "tested_operations": total,
        "exact_match_pct": 100 * exact / total,
        "mean_abs_integer_error": all_errors.mean(),
        "max_abs_integer_error": max_error,
        "error_gt_1_count": np.count_nonzero(all_errors > 1)
    })

results_df = pd.DataFrame(results)

print("SHIFT-SUBTRACT ARITHMETIC VALIDATION")
display(results_df.round(8))

print("\nError distribution by grid:")
for grid_name, group in grid_df.groupby("grid"):
    errors = []

    for _, row in group.iterrows():
        exponents = ast.literal_eval(
            str(row["canonical_exponents"])
        )

        ref = reference_decay(test_states, exponents)
        got = shift_subtract_decay(test_states, exponents)

        errors.extend(np.abs(got - ref).tolist())

    errors = np.asarray(errors)

    print(
        grid_name,
        {
            "zero_error_pct": round(
                100 * np.mean(errors == 0), 5
            ),
            "one_unit_error_pct": round(
                100 * np.mean(errors == 1), 5
            ),
            "greater_than_one_pct": round(
                100 * np.mean(errors > 1), 5
            )
        }
    )

out_path = (
    "/kaggle/working/shiftscan_final/"
    "shift_subtract_arithmetic_validation.csv"
)

results_df.to_csv(out_path, index=False)

print("\nSaved:", out_path)


SHIFT-SUBTRACT ARITHMETIC VALIDATION


,grid,coefficients,tested_operations,exact_match_pct,mean_abs_integer_error,max_abs_integer_error,error_gt_1_count
0,K1_14_T2,104,853944,72.957712,0.270423,1,0
1,K1_14_T3,467,3834537,64.542994,0.354570,1,0
2,K1_18_T3,985,8087835,65.217626,0.347824,1,0



Error distribution by grid:
K1_14_T2 {'zero_error_pct': np.float64(72.95771), 'one_unit_error_pct': np.float64(27.04229), 'greater_than_one_pct': np.float64(0.0)}
K1_14_T3 {'zero_error_pct': np.float64(64.54299), 'one_unit_error_pct': np.float64(35.45701), 'greater_than_one_pct': np.float64(0.0)}
K1_18_T3 {'zero_error_pct': np.float64(65.21763), 'one_unit_error_pct': np.float64(34.78237), 'greater_than_one_pct': np.float64(0.0)}

Saved: /kaggle/working/shiftscan_final/shift_subtract_arithmetic_validation.csv


In [124]:

import ast
import numpy as np
import pandas as pd
import time

GRID_PATH = (
    "/kaggle/working/shiftscan_final/"
    "canonical_dyadic_grid_definitions.csv"
)

grid_df = pd.read_csv(GRID_PATH)

F = 24
SCALE = 1 << F
SEQ_LEN = 8192
N_CHANNELS = 256
N_SEEDS = 10

def round_shift(x, k):
    x = np.asarray(x, dtype=np.int64)
    d = 1 << int(k)
    return np.sign(x) * ((np.abs(x) + d // 2) // d)

def round_reference(x, exponents):
    K = max(exponents)
    denominator = 1 << K
    numerator_coeff = denominator - sum(
        1 << (K-k) for k in exponents
    )
    numerator = x * numerator_coeff
    return (
        np.sign(numerator)
        * ((np.abs(numerator) + denominator//2) // denominator)
    )

def shift_decay(x, exponents):
    result = x.copy()
    for k in exponents:
        result -= round_shift(x, k)
    return result

rng = np.random.default_rng(20261115)
results = []

start_all = time.time()

for seed in range(42, 42 + N_SEEDS):
    rg = np.random.default_rng(seed)

    h0 = np.rint(
        rg.normal(0, 0.1, N_CHANNELS) * SCALE
    ).astype(np.int64)

    inputs = np.rint(
        rg.normal(0, 0.01, (SEQ_LEN, N_CHANNELS)) * SCALE
    ).astype(np.int64)

    for grid_name, group in grid_df.groupby("grid"):
        selected = rg.integers(0, len(group), N_CHANNELS)

        exponents = [
            ast.literal_eval(
                str(group.iloc[i]["canonical_exponents"])
            )
            for i in selected
        ]

        # Precompute the canonical exponent groups.
        ref_coeff = np.empty(N_CHANNELS, dtype=np.float64)
        for c, ks in enumerate(exponents):
            ref_coeff[c] = 1.0 - sum(2.0**(-k) for k in ks)

        h_ref = h0.copy()
        h_shift = h0.copy()
        max_error = 0

        for t in range(SEQ_LEN):
            # Exact rounded multiplication reference.
            # Coefficients are dyadic, so use integer numerator.
            h_ref_new = np.empty_like(h_ref)

            for ks in set(exponents):
                mask = np.array([e == ks for e in exponents])
                h_ref_new[mask] = round_reference(
                    h_ref[mask], ks
                )

            # Vectorized shift-subtract by exponent groups.
            h_shift_new = h_shift.copy()

            for k in sorted(set(k for e in exponents for k in e)):
                counts = np.array([
                    e.count(k) for e in exponents
                ])
                mask = counts > 0

                for _ in range(int(counts[mask].max())):
                    active = counts > _
                    h_shift_new[active] -= round_shift(
                        h_shift[active], k
                    )

            h_ref = h_ref_new + inputs[t]
            h_shift = h_shift_new + inputs[t]

            max_error = max(
                max_error,
                int(np.max(np.abs(h_shift - h_ref)))
            )

        error = h_shift - h_ref

        results.append({
            "seed": seed,
            "grid": grid_name,
            "final_mae_integer": np.abs(error).mean(),
            "final_rmse_integer": np.sqrt(np.mean(error**2)),
            "final_max_error_integer": np.abs(error).max(),
            "max_error_seen_integer": max_error,
            "final_mae_real": np.abs(error).mean() / SCALE
        })

    print(f"Completed seed {seed}")

results_df = pd.DataFrame(results)

print("\nFINAL RESULTS")
display(
    results_df.groupby("grid").agg({
        "final_mae_integer": ["mean", "max"],
        "final_rmse_integer": ["mean", "max"],
        "final_max_error_integer": ["mean", "max"],
        "max_error_seen_integer": ["mean", "max"],
        "final_mae_real": ["mean", "max"]
    }).round(8)
)

out = (
    "/kaggle/working/shiftscan_final/"
    "recurrent_rounding_accumulation_vectorized.csv"
)

results_df.to_csv(out, index=False)

print("\nSaved:", out)
print("Elapsed seconds:", round(time.time() - start_all, 2))


Completed seed 42
Completed seed 43
Completed seed 44
Completed seed 45
Completed seed 46
Completed seed 47
Completed seed 48
Completed seed 49
Completed seed 50
Completed seed 51

FINAL RESULTS


final_mae_integer           final_rmse_integer             \
                      mean       max               mean        max   
grid                                                                 
K1_14_T2          3.337109  3.957031           7.019098   9.073054   
K1_14_T3          2.042578  2.472656           4.476460   5.822572   
K1_18_T3          3.950000  5.007812          10.229263  14.846085   

         final_max_error_integer      max_error_seen_integer       \
                            mean  max                   mean  max   
grid                                                                
K1_14_T2                    45.1   60                   73.0  103   
K1_14_T3                    37.4   72                   58.5   81   
K1_18_T3                    89.8  126                  117.9  160   

         final_mae_real                
                   mean           max  
grid                                   
K1_14_T2   2.000000e-07  2.400000e-07  
K1_14_T3   1.200000e-07  1.500000e-07  
K1_18_T3   2.400000e-07  3.000000e-07


Saved: /kaggle/working/shiftscan_final/recurrent_rounding_accumulation_vectorized.csv
Elapsed seconds: 2199.43


In [125]:
print(results_df.to_string(index=False))
print("\nColumn names:")
print(results_df.columns.tolist())

 seed     grid  final_mae_integer  final_rmse_integer  final_max_error_integer  max_error_seen_integer  final_mae_real
   42 K1_14_T2           3.765625            9.073054                       60                     103    2.244487e-07
   42 K1_14_T3           2.472656            5.139841                       34                      81    1.473818e-07
   42 K1_18_T3           4.218750            9.971835                       62                     111    2.514571e-07
   43 K1_14_T2           3.355469            7.143889                       46                      59    2.000015e-07
   43 K1_14_T3           1.582031            2.844265                       16                      44    9.429641e-08
   43 K1_18_T3           4.085938           10.209906                       89                     108    2.435409e-07
   44 K1_14_T2           3.750000            7.350064                       44                      78    2.235174e-07
   44 K1_14_T3           1.718750            3.5

In [128]:

# ================================================================
# SHIFT-SCAN: CORRECTED MAMBA-2 SSD SOURCE AUDIT
# Does not modify model functions or Triton kernels.
# ================================================================

import os, sys, json, inspect, importlib, traceback
from datetime import datetime, timezone

import torch
import triton

ROOT = "/kaggle/working/shiftscan_final"
OUT = os.path.join(ROOT, "final_validation")
os.makedirs(OUT, exist_ok=True)

print("=" * 72)
print("CORRECTED MAMBA-2 SSD SOURCE AUDIT")
print("=" * 72)

# Correct package location.
MODULE_CANDIDATES = [
    "mamba_ssm.ops.triton.ssd_combined",
    "mamba_ssm.ops.triton.ssd_chunk_scan",
    "mamba_ssm.modules.mamba2"
]

modules = {}

for module_name in MODULE_CANDIDATES:
    try:
        mod = importlib.import_module(module_name)
        modules[module_name] = mod
        print("IMPORTED:", module_name)
    except Exception as e:
        print("UNAVAILABLE:", module_name, "|", repr(e))

if "mamba_ssm.ops.triton.ssd_combined" not in modules:
    raise RuntimeError(
        "Could not import mamba_ssm.ops.triton.ssd_combined. "
        "Please send the import errors above."
    )

ssd = modules["mamba_ssm.ops.triton.ssd_combined"]

audit = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "torch": torch.__version__,
    "triton": triton.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "modules": {},
    "functions": {},
    "live_bindings": {}
}

def get_source(obj):
    try:
        return inspect.getsource(obj)
    except Exception as e:
        return "SOURCE UNAVAILABLE: " + repr(e)

# Discover functions from the actual imported module.
wanted = [
    "_mamba_chunk_scan_combined_fwd",
    "mamba_chunk_scan_combined",
    "_chunk_cumsum_fwd",
    "_chunk_state_fwd",
    "_state_passing_fwd",
    "_chunk_scan_fwd"
]

for name in wanted:
    fn = getattr(ssd, name, None)

    print("\n" + "-" * 65)
    print("FUNCTION:", name)

    if fn is None:
        print("NOT FOUND in ssd_combined")
        continue

    try:
        signature = str(inspect.signature(fn))
    except Exception:
        signature = "Unavailable"

    source = get_source(fn)

    print("Signature:", signature)
    print("Source length:", len(source))

    matches = []
    for i, line in enumerate(source.splitlines(), 1):
        if any(term in line for term in [
            "exp(", "tl.exp", "dA_cumsum", "dA_cs",
            "scale", "decay", "dt", "state"
        ]):
            matches.append({"line": i, "text": line.strip()})

    for item in matches[:35]:
        print(f"L{item['line']}: {item['text']}")

    audit["functions"][name] = {
        "signature": signature,
        "source": source,
        "relevant_expressions": matches
    }

# Inspect Triton scan module, if present.
scan = modules.get("mamba_ssm.ops.triton.ssd_chunk_scan")

if scan is not None:
    print("\n" + "=" * 72)
    print("TRITON SCAN KERNELS")
    print("=" * 72)

    kernel_names = [
        "_chunk_scan_fwd_kernel",
        "_chunk_state_fwd_kernel",
        "_state_passing_fwd_kernel"
    ]

    for name in kernel_names:
        obj = getattr(scan, name, None)

        if obj is None:
            print(name, ": NOT FOUND")
            continue

        source = get_source(obj)
        print("\nKERNEL:", name)
        print("Source length:", len(source))

        matches = []
        for i, line in enumerate(source.splitlines(), 1):
            if any(term in line for term in [
                "tl.exp", "tl.dot", "dA_cs",
                "scale_m", "scale =", "cb *=", "states ="
            ]):
                matches.append({"line": i, "text": line.strip()})

        for item in matches[:45]:
            print(f"L{item['line']}: {item['text']}")

        audit["functions"]["kernel_" + name] = {
            "source": source,
            "relevant_expressions": matches
        }

# Inspect exact global bindings used by the combined forward.
forward = getattr(ssd, "_mamba_chunk_scan_combined_fwd", None)

if forward is not None:
    g = forward.__globals__

    print("\n" + "=" * 72)
    print("LIVE SSD FUNCTION BINDINGS")
    print("=" * 72)

    for name in [
        "_chunk_cumsum_fwd",
        "_chunk_state_fwd",
        "_state_passing_fwd",
        "_chunk_scan_fwd"
    ]:
        fn = g.get(name)

        print(name, ":", "FOUND" if fn is not None else "MISSING")

        audit["live_bindings"][name] = {
            "found": fn is not None,
            "module": getattr(fn, "__module__", None),
            "name": getattr(fn, "__name__", None),
            "object_id": id(fn) if fn is not None else None
        }

# Record model module path as an additional cross-check.
try:
    m2 = importlib.import_module("mamba_ssm.modules.mamba2")
    print("\nMamba2 class module:", m2.__file__)
    audit["modules"]["mamba2"] = m2.__file__
except Exception as e:
    print("Could not inspect mamba2 module:", repr(e))

for name, mod in modules.items():
    audit["modules"][name] = getattr(mod, "__file__", None)

path = os.path.join(OUT, "mamba2_ssd_integration_audit.json")

with open(path, "w") as f:
    json.dump(audit, f, indent=2, default=str)

print("\n" + "=" * 72)
print("AUDIT COMPLETE")
print("Saved:", path)
print("No model functions or kernels were modified.")
print("=" * 72)


CORRECTED MAMBA-2 SSD SOURCE AUDIT
IMPORTED: mamba_ssm.ops.triton.ssd_combined
IMPORTED: mamba_ssm.ops.triton.ssd_chunk_scan
IMPORTED: mamba_ssm.modules.mamba2

-----------------------------------------------------------------
FUNCTION: _mamba_chunk_scan_combined_fwd
Signature: (x, dt, A, B, C, chunk_size, D=None, z=None, dt_bias=None, initial_states=None, seq_idx=None, cu_seqlens=None, dt_softplus=False, dt_limit=(0.0, inf))
Source length: 3795
L1: def _mamba_chunk_scan_combined_fwd(x, dt, A, B, C, chunk_size, D=None, z=None, dt_bias=None, initial_states=None, seq_idx=None, cu_seqlens=None, dt_softplus=False, dt_limit=(0.0, float("inf"))):
L3: _, _, ngroups, dstate = B.shape
L5: assert B.shape == (batch, seqlen, ngroups, dstate)
L7: assert dt.shape == (batch, seqlen, nheads)
L26: if initial_states is not None:
L27: assert initial_states.shape == (batch, nheads, headdim, dstate)
L29: # dA_cumsum_tmp0, dt_tmp0 = _chunk_cumsum_fwd(dt[:, :147], A, chunk_size, dt_bias=dt_bias, dt_softplus=

In [129]:

# ================================================================
# SHIFT-SCAN: LOCATE THE LIVE STATE-PASSING TRITON KERNEL
# Read-only inspection. No monkeypatching.
# ================================================================

import os
import json
import inspect
import importlib

import torch
import triton

ssd = importlib.import_module("mamba_ssm.ops.triton.ssd_combined")

wrappers = {
    "_chunk_cumsum_fwd": ssd._chunk_cumsum_fwd,
    "_chunk_state_fwd": ssd._chunk_state_fwd,
    "_state_passing_fwd": ssd._state_passing_fwd,
    "_chunk_scan_fwd": ssd._chunk_scan_fwd,
}

kernel_names = [
    "_chunk_cumsum_fwd_kernel",
    "_chunk_state_fwd_kernel",
    "_state_passing_fwd_kernel",
    "_chunk_scan_fwd_kernel",
]

report = {}

def describe_kernel(kernel):
    result = {
        "type": str(type(kernel)),
        "module": getattr(kernel, "__module__", None),
        "name": getattr(kernel, "__name__", None),
        "signature": None,
        "source": None,
        "configs": None,
    }

    try:
        result["signature"] = str(inspect.signature(kernel))
    except Exception:
        pass

    # Triton JITFunction commonly exposes its source as .src.
    src = getattr(kernel, "src", None)

    if isinstance(src, str):
        result["source"] = src
    else:
        try:
            result["source"] = inspect.getsource(kernel)
        except Exception:
            try:
                result["source"] = inspect.getsource(kernel.fn)
            except Exception:
                result["source"] = None

    configs = getattr(kernel, "configs", None)
    if configs is not None:
        result["configs"] = [str(c) for c in configs]

    return result

print("=" * 72)
print("LIVE SSD KERNEL LOCATION")
print("=" * 72)

for wrapper_name, wrapper in wrappers.items():
    print("\nWRAPPER:", wrapper_name)

    g = wrapper.__globals__

    print("Wrapper module:", wrapper.__module__)
    print("Wrapper source file:", inspect.getsourcefile(wrapper))

    for kernel_name in kernel_names:
        kernel = g.get(kernel_name)

        if kernel is None:
            continue

        print("\n  FOUND KERNEL:", kernel_name)

        details = describe_kernel(kernel)

        print("  Type:", details["type"])
        print("  Module:", details["module"])
        print("  Signature:", details["signature"])
        print("  Source available:", details["source"] is not None)

        if details["source"]:
            print("\n  Relevant kernel source:")
            lines = details["source"].splitlines()

            for i, line in enumerate(lines, 1):
                if any(term in line.lower() for term in [
                    "tl.exp", "tl.load", "tl.store",
                    "dA", "state", "scale", "seq_idx"
                ]):
                    print(f"    L{i}: {line.strip()}")

        report[kernel_name] = details

# Specifically inspect state-passing wrapper globals.
state_wrapper = ssd._state_passing_fwd
state_globals = state_wrapper.__globals__

print("\n" + "=" * 72)
print("STATE-PASSING KERNEL BINDING")
print("=" * 72)

state_kernel = state_globals.get("_state_passing_fwd_kernel")

if state_kernel is None:
    print("Kernel not found in state-passing wrapper globals.")
else:
    print("Kernel object:", state_kernel)
    print("Kernel type:", type(state_kernel))
    print("Kernel module:", getattr(state_kernel, "__module__", None))
    print("Kernel name:", getattr(state_kernel, "__name__", None))

    src = getattr(state_kernel, "src", None)

    if isinstance(src, str):
        print("\nFull kernel source:\n")
        print(src)
    else:
        try:
            print("\nFull kernel source:\n")
            print(inspect.getsource(state_kernel.fn))
        except Exception as e:
            print("Could not retrieve source:", repr(e))

# Save report.
out_dir = "/kaggle/working/shiftscan_final/final_validation"
os.makedirs(out_dir, exist_ok=True)

out_path = os.path.join(
    out_dir,
    "state_passing_kernel_location.json"
)

with open(out_path, "w") as f:
    json.dump(report, f, indent=2, default=str)

print("\nSaved:", out_path)
print("\nNo functions or kernels were modified.")


LIVE SSD KERNEL LOCATION

WRAPPER: _chunk_cumsum_fwd
Wrapper module: mamba_ssm.ops.triton.ssd_chunk_state
Wrapper source file: /usr/local/lib/python3.13/dist-packages/mamba_ssm/ops/triton/ssd_chunk_state.py

  FOUND KERNEL: _chunk_cumsum_fwd_kernel
  Type: <class 'triton.runtime.autotuner.Autotuner'>
  Module: triton.runtime.autotuner
  Signature: None
  Source available: False

  FOUND KERNEL: _chunk_state_fwd_kernel
  Type: <class 'triton.runtime.autotuner.Autotuner'>
  Module: triton.runtime.autotuner
  Signature: None
  Source available: False

WRAPPER: _chunk_state_fwd
Wrapper module: mamba_ssm.ops.triton.ssd_chunk_state
Wrapper source file: /usr/local/lib/python3.13/dist-packages/mamba_ssm/ops/triton/ssd_chunk_state.py

  FOUND KERNEL: _chunk_cumsum_fwd_kernel
  Type: <class 'triton.runtime.autotuner.Autotuner'>
  Module: triton.runtime.autotuner
  Signature: None
  Source available: False

  FOUND KERNEL: _chunk_state_fwd_kernel
  Type: <class 'triton.runtime.autotuner.Autotuner

In [130]:

# ================================================================
# SHIFT-SCAN: EXTRACT UNDERLYING TRITON JIT SOURCE
# ================================================================

import inspect
import importlib
import json
import os

module_names = [
    "mamba_ssm.ops.triton.ssd_state_passing",
    "mamba_ssm.ops.triton.ssd_chunk_state",
    "mamba_ssm.ops.triton.ssd_chunk_scan",
]

output = {}

for module_name in module_names:
    mod = importlib.import_module(module_name)

    print("\n" + "=" * 72)
    print("MODULE:", module_name)
    print("FILE:", mod.__file__)
    print("=" * 72)

    for name in dir(mod):
        if not name.endswith("_kernel"):
            continue

        obj = getattr(mod, name)

        # Autotuner.fn is normally the underlying Triton JITFunction.
        jit_obj = getattr(obj, "fn", obj)

        src = getattr(jit_obj, "src", None)

        # Some Triton versions expose the original function as .fn.
        if not isinstance(src, str):
            nested = getattr(jit_obj, "fn", None)
            src = getattr(nested, "src", None)

        item = {
            "module": module_name,
            "wrapper_type": str(type(obj)),
            "jit_type": str(type(jit_obj)),
            "jit_name": getattr(jit_obj, "__name__", None),
            "source": src if isinstance(src, str) else None,
        }

        output[name] = item

        print("\nKERNEL:", name)
        print("Wrapper type:", item["wrapper_type"])
        print("Underlying type:", item["jit_type"])
        print("Underlying name:", item["jit_name"])

        if isinstance(src, str):
            print("\n--- FULL SOURCE ---")
            print(src)
            print("--- END SOURCE ---")
        else:
            print("Source not available through .src")

            # Last diagnostic: inspect available attributes.
            print(
                "Available attributes:",
                [
                    a for a in dir(jit_obj)
                    if not a.startswith("__")
                ][:60]
            )

# Save all extracted sources.
out_dir = "/kaggle/working/shiftscan_final/final_validation"
os.makedirs(out_dir, exist_ok=True)

path = os.path.join(out_dir, "extracted_triton_kernel_sources.json")

with open(path, "w") as f:
    json.dump(output, f, indent=2, default=str)

print("\nSaved source extraction:", path)



MODULE: mamba_ssm.ops.triton.ssd_state_passing
FILE: /usr/local/lib/python3.13/dist-packages/mamba_ssm/ops/triton/ssd_state_passing.py

KERNEL: _state_passing_bwd_kernel
Wrapper type: <class 'triton.runtime.autotuner.Autotuner'>
Underlying type: <class 'triton.runtime.jit.JITFunction'>
Underlying name: _state_passing_bwd_kernel

--- FULL SOURCE ---
def _state_passing_bwd_kernel(
    # Pointers to matrices
    dout_ptr, out_ptr, dA_cs_ptr, dfinal_states_ptr, seq_idx_ptr,
    dstates_ptr, ddA_cs_ptr, dinitstates_ptr, states_converted_ptr,
    # Matrix dimensions
    dim, nchunks, seqlen, chunk_size,
    # Strides
    stride_dout_batch, stride_dout_chunk, stride_dout_head, stride_dout_dim,
    stride_out_batch, stride_out_chunk, stride_out_head, stride_out_dim,
    stride_dA_cs_batch, stride_dA_cs_chunk, stride_dA_cs_head,
    stride_dfinal_states_batch, stride_dfinal_states_head, stride_dfinal_states_dim,
    stride_seq_idx_batch, stride_seq_idx_seqlen,
    stride_dstates_batch, stride_

In [131]:

import inspect
import importlib
import json
import os

ssd = importlib.import_module("mamba_ssm.ops.triton.ssd_combined")

print("=" * 80)
print("SHIFT-SCAN: STATE-PASSING KERNEL INSPECTION")
print("=" * 80)

names = [
    "_state_passing_fwd",
    "_state_passing_fwd_kernel",
    "_state_passing_bwd",
    "_state_passing_bwd_kernel",
]

inspection = {}

for name in names:
    obj = getattr(ssd, name, None)

    print(f"\n{'-' * 70}")
    print("NAME:", name)

    if obj is None:
        print("NOT FOUND in ssd_combined")
        continue

    print("TYPE:", type(obj))

    try:
        print("SIGNATURE:", inspect.signature(obj))
    except Exception as e:
        print("Signature unavailable:", repr(e))

    target = getattr(obj, "fn", obj)

    try:
        source = inspect.getsource(target)
        print("\nSOURCE:\n")
        print(source)
        inspection[name] = source
    except Exception as e:
        print("Source unavailable:", repr(e))

# Also locate state-passing references in the main forward wrapper.
print("\n" + "=" * 80)
print("REFERENCES IN MAIN SSD FORWARD")
print("=" * 80)

for name in [
    "_mamba_chunk_scan_combined_fwd",
    "mamba_chunk_scan_combined",
]:
    obj = getattr(ssd, name, None)
    if obj is None:
        continue

    target = getattr(obj, "fn", obj)

    try:
        source = inspect.getsource(target)
        lines = source.splitlines()

        for i, line in enumerate(lines):
            if "state_passing" in line or "states" in line and "fwd" in line:
                lo = max(0, i - 5)
                hi = min(len(lines), i + 9)
                print(f"\n--- {name}, lines {lo + 1}-{hi} ---")
                print("\n".join(lines[lo:hi]))
    except Exception as e:
        print(name, "source unavailable:", repr(e))

# Save inspection for reproducibility.
out_dir = "/kaggle/working/shiftscan_final/final_validation"
os.makedirs(out_dir, exist_ok=True)

out_path = os.path.join(
    out_dir,
    "state_passing_kernel_inspection.json"
)

with open(out_path, "w") as f:
    json.dump(inspection, f, indent=2)

print("\nSaved:", out_path)


SHIFT-SCAN: STATE-PASSING KERNEL INSPECTION

----------------------------------------------------------------------
NAME: _state_passing_fwd
TYPE: <class 'function'>
SIGNATURE: (states, dA_chunk_cumsum, initial_states=None, seq_idx=None, chunk_size=None, out_dtype=None)

SOURCE:

def _state_passing_fwd(states, dA_chunk_cumsum, initial_states=None, seq_idx=None, chunk_size=None,
                       out_dtype=None):
    batch, nchunks, nheads, dim = states.shape
    assert dA_chunk_cumsum.shape == (batch, nheads, nchunks)
    if initial_states is not None:
        assert initial_states.shape == (batch, nheads, dim)
    if seq_idx is not None:
        assert chunk_size is not None
        seqlen = seq_idx.shape[-1]
        assert seq_idx.shape == (batch, seqlen)
    out_dtype = states.dtype if out_dtype is None else out_dtype
    out = torch.empty((batch, nchunks, nheads, dim), device=states.device, dtype=out_dtype)
    final_states = torch.empty((batch, nheads, dim), device=states.dev

In [132]:

import inspect
import json
import torch
import numpy as np
from pathlib import Path

# Find the actual kernel through the function's global namespace.
sp_fn = ssd_combined._state_passing_fwd
sp_globals = sp_fn.__globals__

print("=" * 75)
print("ACTUAL STATE-PASSING KERNEL")
print("=" * 75)

kernel = sp_globals.get("_state_passing_fwd_kernel")

assert kernel is not None, (
    "Could not locate _state_passing_fwd_kernel in function globals."
)

print("Kernel object:", kernel)
print("Kernel type:", type(kernel))

try:
    print("\nKernel source:")
    print(inspect.getsource(kernel))
except Exception as e:
    print("Source unavailable:", repr(e))

try:
    print("\nKernel signature:")
    print(inspect.signature(kernel))
except Exception as e:
    print("Signature unavailable:", repr(e))

print("\nKernel module:", getattr(kernel, "__module__", None))

# Inspect the imported module where the kernel is actually defined.
kernel_module = inspect.getmodule(kernel)
print("Kernel module object:", kernel_module)

if kernel_module is not None:
    print("Kernel module file:", getattr(kernel_module, "__file__", None))


# ------------------------------------------------------------
# Capture actual inputs to state passing during a real forward.
# ------------------------------------------------------------

capture_records = []
original_state_passing = ssd_combined._state_passing_fwd

def capture_state_passing(
    states,
    dA_chunk_cumsum,
    initial_states=None,
    seq_idx=None,
    chunk_size=None,
    out_dtype=None
):
    record = {
        "states_shape": list(states.shape),
        "states_dtype": str(states.dtype),
        "states_min": float(states.detach().float().min().item()),
        "states_max": float(states.detach().float().max().item()),
        "states_mean": float(states.detach().float().mean().item()),
        "states_absmax": float(states.detach().float().abs().max().item()),

        "decay_log_shape": list(dA_chunk_cumsum.shape),
        "decay_log_dtype": str(dA_chunk_cumsum.dtype),
        "decay_log_min": float(
            dA_chunk_cumsum.detach().float().min().item()
        ),
        "decay_log_max": float(
            dA_chunk_cumsum.detach().float().max().item()
        ),

        "has_initial_states": initial_states is not None,
        "initial_states_shape": (
            list(initial_states.shape)
            if initial_states is not None else None
        ),
        "initial_states_dtype": (
            str(initial_states.dtype)
            if initial_states is not None else None
        ),
        "seq_idx_present": seq_idx is not None,
        "chunk_size": chunk_size,
        "out_dtype": str(out_dtype),
    }

    # The actual chunk-level decay factors.
    decay = torch.exp(dA_chunk_cumsum.detach().float())

    record.update({
        "decay_min": float(decay.min().item()),
        "decay_max": float(decay.max().item()),
        "decay_mean": float(decay.mean().item()),
        "decay_below_0_5_pct": float(
            (decay < 0.5).float().mean().item() * 100
        ),
        "decay_above_0_95_pct": float(
            (decay >= 0.95).float().mean().item() * 100
        ),
    })

    capture_records.append(record)

    return original_state_passing(
        states,
        dA_chunk_cumsum,
        initial_states=initial_states,
        seq_idx=seq_idx,
        chunk_size=chunk_size,
        out_dtype=out_dtype,
    )


ssd_combined._state_passing_fwd = capture_state_passing

try:
    # Use one of the already-tokenized evaluation examples.
    with torch.inference_mode():
        _ = model(input_ids=encoded_samples[0].to("cuda"))
finally:
    ssd_combined._state_passing_fwd = original_state_passing


print("\n" + "=" * 75)
print("REAL MODEL RUNTIME CAPTURE")
print("=" * 75)

print("Captured state-passing calls:", len(capture_records))

if capture_records:
    print("\nFirst call:")
    print(json.dumps(capture_records[0], indent=2))

    print("\nDistinct states shapes:")
    print(sorted({
        tuple(r["states_shape"]) for r in capture_records
    }))

    print("\nDistinct state dtypes:")
    print(sorted({
        r["states_dtype"] for r in capture_records
    }))

    print("\nDecay statistics across calls:")
    for key in [
        "decay_min",
        "decay_max",
        "decay_mean",
        "decay_below_0_5_pct",
        "decay_above_0_95_pct",
    ]:
        values = [r[key] for r in capture_records]
        print(
            f"{key:25s} "
            f"mean={np.mean(values):.8g} "
            f"min={np.min(values):.8g} "
            f"max={np.max(values):.8g}"
        )

# Save evidence for reproducibility.
out_dir = Path("/kaggle/working/shiftscan_final/state_passing")
out_dir.mkdir(parents=True, exist_ok=True)

with open(out_dir / "state_passing_runtime_capture.json", "w") as f:
    json.dump(capture_records, f, indent=2)

print("\nSaved:")
print(out_dir / "state_passing_runtime_capture.json")


ACTUAL STATE-PASSING KERNEL
Kernel object: <triton.runtime.autotuner.Autotuner object at 0x79be5c584150>
Kernel type: <class 'triton.runtime.autotuner.Autotuner'>

Kernel source:
Source unavailable: TypeError('module, class, method, function, traceback, frame, or code object was expected, got Autotuner')

Kernel signature:
Signature unavailable: TypeError('<triton.runtime.autotuner.Autotuner object at 0x79be5c584150> is not a callable object')

Kernel module: triton.runtime.autotuner
Kernel module object: <module 'triton.runtime.autotuner' from '/usr/local/lib/python3.13/dist-packages/triton/runtime/autotuner.py'>
Kernel module file: /usr/local/lib/python3.13/dist-packages/triton/runtime/autotuner.py

REAL MODEL RUNTIME CAPTURE
Captured state-passing calls: 24

First call:
{
  "states_shape": [
    1,
    1,
    24,
    8192
  ],
  "states_dtype": "torch.float32",
  "states_min": -5.016904830932617,
  "states_max": 35.60179138183594,
  "states_mean": 0.017262820154428482,
  "states_abs

In [133]:

import inspect
import json
import numpy as np
import torch
from pathlib import Path

# ------------------------------------------------------------
# 1. Inspect the underlying Triton JIT function
# ------------------------------------------------------------

kernel = ssd_combined._state_passing_fwd.__globals__[
    "_state_passing_fwd_kernel"
]

print("=" * 75)
print("UNDERLYING TRITON KERNEL")
print("=" * 75)

underlying_fn = getattr(kernel, "fn", None)

print("Autotuner:", type(kernel))
print("Underlying function:", type(underlying_fn))

if underlying_fn is not None:
    try:
        print("\nUnderlying signature:")
        print(inspect.signature(underlying_fn))
    except Exception as e:
        print("Signature unavailable:", repr(e))

    try:
        print("\nUnderlying source:")
        print(inspect.getsource(underlying_fn))
    except Exception as e:
        print("Source unavailable:", repr(e))

    print("\nJIT function name:", getattr(underlying_fn, "__name__", None))
    print("JIT function module:", getattr(underlying_fn, "__module__", None))


# ------------------------------------------------------------
# 2. Prepare a sequence longer than four 256-token chunks
# ------------------------------------------------------------

long_text = max(samples, key=lambda x: len(x))

long_encoded = tokenizer(
    long_text,
    truncation=True,
    max_length=1024,
    return_tensors="pt"
)

long_input_ids = long_encoded.input_ids.to("cuda")

seq_len = long_input_ids.shape[1]

print("\n" + "=" * 75)
print("LONG-SEQUENCE INPUT")
print("=" * 75)
print("Token count:", seq_len)
print("Chunk size:", model.config.chunk_size if hasattr(model.config, "chunk_size") else 256)
print("Expected chunks:", int(np.ceil(seq_len / 256)))

assert seq_len > 256, (
    "This document produced no more than 256 tokens. "
    "Choose a longer FineWeb-Edu document."
)


# ------------------------------------------------------------
# 3. Capture real state-passing tensors
# ------------------------------------------------------------

capture_records_long = []
original_state_passing = ssd_combined._state_passing_fwd

def capture_state_passing_long(
    states,
    dA_chunk_cumsum,
    initial_states=None,
    seq_idx=None,
    chunk_size=None,
    out_dtype=None
):
    states_f = states.detach().float()
    decay_log_f = dA_chunk_cumsum.detach().float()
    decay_f = torch.exp(decay_log_f)

    record = {
        "states_shape": list(states.shape),
        "states_dtype": str(states.dtype),
        "states_min": float(states_f.min().item()),
        "states_max": float(states_f.max().item()),
        "states_mean": float(states_f.mean().item()),
        "states_absmax": float(states_f.abs().max().item()),

        "nchunks": int(states.shape[1]),
        "nheads": int(states.shape[2]),
        "state_dim": int(states.shape[3]),

        "decay_log_shape": list(dA_chunk_cumsum.shape),
        "decay_log_min": float(decay_log_f.min().item()),
        "decay_log_max": float(decay_log_f.max().item()),

        "decay_min": float(decay_f.min().item()),
        "decay_max": float(decay_f.max().item()),
        "decay_mean": float(decay_f.mean().item()),

        "decay_below_0_5_pct": float(
            (decay_f < 0.5).float().mean().item() * 100
        ),
        "decay_below_0_25_pct": float(
            (decay_f < 0.25).float().mean().item() * 100
        ),
        "decay_above_0_5_pct": float(
            (decay_f >= 0.5).float().mean().item() * 100
        ),

        "has_initial_states": initial_states is not None,
        "initial_states_shape": (
            list(initial_states.shape)
            if initial_states is not None else None
        ),
        "seq_idx_present": seq_idx is not None,
        "chunk_size": chunk_size,
        "out_dtype": str(out_dtype),
    }

    capture_records_long.append(record)

    return original_state_passing(
        states,
        dA_chunk_cumsum,
        initial_states=initial_states,
        seq_idx=seq_idx,
        chunk_size=chunk_size,
        out_dtype=out_dtype,
    )


ssd_combined._state_passing_fwd = capture_state_passing_long

try:
    with torch.inference_mode():
        long_output = model(input_ids=long_input_ids)
finally:
    ssd_combined._state_passing_fwd = original_state_passing


# ------------------------------------------------------------
# 4. Report and save results
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("MULTI-CHUNK STATE-PASSING RESULTS")
print("=" * 75)

print("Captured calls:", len(capture_records_long))
print("Output shape:", tuple(long_output.logits.shape))
print("Output finite:", bool(torch.isfinite(long_output.logits).all()))

if capture_records_long:
    nchunks_values = sorted({
        r["nchunks"] for r in capture_records_long
    })

    print("Distinct chunk counts:", nchunks_values)

    print("\nFirst layer capture:")
    print(json.dumps(capture_records_long[0], indent=2))

    print("\nPer-layer chunk count and decay statistics:")
    for layer_idx, r in enumerate(capture_records_long):
        print(
            f"Layer {layer_idx:02d} | "
            f"chunks={r['nchunks']} | "
            f"decay min={r['decay_min']:.6g} | "
            f"max={r['decay_max']:.6g} | "
            f"mean={r['decay_mean']:.6g} | "
            f"below .5={r['decay_below_0_5_pct']:.2f}% | "
            f"state absmax={r['states_absmax']:.5g}"
        )

    assert all(r["nchunks"] > 1 for r in capture_records_long), (
        "At least one layer did not process multiple chunks."
    )

out_dir = Path("/kaggle/working/shiftscan_final/state_passing")
out_dir.mkdir(parents=True, exist_ok=True)

with open(out_dir / "state_passing_multichunk_capture.json", "w") as f:
    json.dump(capture_records_long, f, indent=2)

print("\nSaved:")
print(out_dir / "state_passing_multichunk_capture.json")

print("\nOriginal state-passing function restored:",
      ssd_combined._state_passing_fwd is original_state_passing)


UNDERLYING TRITON KERNEL
Autotuner: <class 'triton.runtime.autotuner.Autotuner'>
Underlying function: <class 'triton.runtime.jit.JITFunction'>

Underlying signature:
(*args, **kwargs)

Underlying source:
Source unavailable: TypeError('module, class, method, function, traceback, frame, or code object was expected, got JITFunction')

JIT function name: _state_passing_fwd_kernel
JIT function module: mamba_ssm.ops.triton.ssd_state_passing

LONG-SEQUENCE INPUT
Token count: 1024
Chunk size: 256
Expected chunks: 4

MULTI-CHUNK STATE-PASSING RESULTS
Captured calls: 24
Output shape: (1, 1024, 50288)
Output finite: True
Distinct chunk counts: [4]

First layer capture:
{
  "states_shape": [
    1,
    4,
    24,
    8192
  ],
  "states_dtype": "torch.float32",
  "states_min": -5.228885650634766,
  "states_max": 36.967952728271484,
  "states_mean": 0.01769118569791317,
  "states_absmax": 36.967952728271484,
  "nchunks": 4,
  "nheads": 24,
  "state_dim": 8192,
  "decay_log_shape": [
    1,
    24,


In [134]:

import torch
import numpy as np
import json
import inspect
from pathlib import Path

# ------------------------------------------------------------
# 1. Retrieve underlying Triton source
# ------------------------------------------------------------

kernel = ssd_combined._state_passing_fwd.__globals__[
    "_state_passing_fwd_kernel"
]
jit_fn = getattr(kernel, "fn", None)

print("=" * 75)
print("TRITON SOURCE INSPECTION")
print("=" * 75)

for obj_name, obj in [
    ("Autotuner", kernel),
    ("JITFunction", jit_fn),
]:
    print(f"\n{obj_name}:")

    if obj is None:
        continue

    for attr in ["src", "source", "arg_names", "constexprs"]:
        try:
            value = getattr(obj, attr)
            if attr in ["src", "source"]:
                print(f"\n{attr}:")
                print(value)
            else:
                print(f"{attr}:", value)
        except Exception as e:
            print(f"{attr}: unavailable ({type(e).__name__})")


# ------------------------------------------------------------
# 2. Capture actual state-passing tensors and outputs
# ------------------------------------------------------------

captured = []
original_fn = ssd_combined._state_passing_fwd

def capture_and_validate(
    states,
    dA_chunk_cumsum,
    initial_states=None,
    seq_idx=None,
    chunk_size=None,
    out_dtype=None
):
    # Preserve actual tensors before the original function runs.
    states_copy = states.detach().float().clone()
    decay_log_copy = dA_chunk_cumsum.detach().float().clone()

    initial_copy = (
        initial_states.detach().float().clone()
        if initial_states is not None else None
    )

    # Run the real Triton implementation.
    out, final_states = original_fn(
        states,
        dA_chunk_cumsum,
        initial_states=initial_states,
        seq_idx=seq_idx,
        chunk_size=chunk_size,
        out_dtype=out_dtype,
    )

    captured.append({
        "states": states_copy.cpu(),
        "decay_log": decay_log_copy.cpu(),
        "initial_states": (
            initial_copy.cpu() if initial_copy is not None else None
        ),
        "triton_out": out.detach().float().cpu().clone(),
        "triton_final": final_states.detach().float().cpu().clone(),
        "seq_idx_present": seq_idx is not None,
        "chunk_size": chunk_size,
        "out_dtype": str(out_dtype),
    })

    return out, final_states


ssd_combined._state_passing_fwd = capture_and_validate

try:
    with torch.inference_mode():
        _ = model(input_ids=long_input_ids)
finally:
    ssd_combined._state_passing_fwd = original_fn


# ------------------------------------------------------------
# 3. Reconstruct recurrence independently in FP32
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("FP32 RECURRENCE VALIDATION")
print("=" * 75)

validation_rows = []

for layer_idx, record in enumerate(captured):
    N = record["states"]                # [B, chunks, heads, dim]
    decay = torch.exp(record["decay_log"])  # [B, heads, chunks]
    initial = record["initial_states"]

    B, nchunks, nheads, dim = N.shape

    # This first validation intentionally handles the captured
    # no-initial-state, no-sequence-boundary case.
    assert initial is None, (
        "Initial states are present; recurrence needs that input."
    )
    assert not record["seq_idx_present"], (
        "Sequence IDs are present; segmented recurrence needs separate handling."
    )

    running = torch.zeros(
        (B, nheads, dim),
        dtype=torch.float32
    )

    reconstructed = []

    for c in range(nchunks):
        dc = decay[:, :, c].unsqueeze(-1)
        nc = N[:, c]

        running = dc * running + nc
        reconstructed.append(running.clone())

    reconstructed = torch.stack(reconstructed, dim=1)

    triton_out = record["triton_out"]

    # Triton output is cast to C.dtype (FP16 in this model).
    # Compare against the actual output after the same cast.
    reconstructed_cast = reconstructed.to(triton_out.dtype).float()

    diff = reconstructed_cast - triton_out

    row = {
        "layer": layer_idx,
        "chunks": nchunks,
        "state_dim": dim,
        "mae": float(diff.abs().mean()),
        "rmse": float(torch.sqrt((diff ** 2).mean())),
        "max_abs_error": float(diff.abs().max()),
        "triton_output_absmax": float(triton_out.abs().max()),
        "reconstructed_absmax": float(reconstructed.abs().max()),
        "final_state_mae": float(
            (running - record["triton_final"]).abs().mean()
        ),
        "final_state_max_abs_error": float(
            (running - record["triton_final"]).abs().max()
        ),
    }

    validation_rows.append(row)

    print(
        f"Layer {layer_idx:02d} | "
        f"MAE={row['mae']:.7g} | "
        f"RMSE={row['rmse']:.7g} | "
        f"Max={row['max_abs_error']:.7g} | "
        f"Final-state max={row['final_state_max_abs_error']:.7g}"
    )


# ------------------------------------------------------------
# 4. Aggregate validation
# ------------------------------------------------------------

maes = [r["mae"] for r in validation_rows]
rmses = [r["rmse"] for r in validation_rows]
maxes = [r["max_abs_error"] for r in validation_rows]

print("\nAggregate:")
print("Mean layer MAE:", np.mean(maes))
print("Mean layer RMSE:", np.mean(rmses))
print("Worst layer max error:", np.max(maxes))

# ------------------------------------------------------------
# 5. Save captured evidence
# ------------------------------------------------------------

out_dir = Path(
    "/kaggle/working/shiftscan_final/state_passing"
)
out_dir.mkdir(parents=True, exist_ok=True)

torch.save(
    captured,
    out_dir / "state_passing_raw_tensors.pt"
)

with open(
    out_dir / "state_passing_fp32_recurrence_validation.json", "w"
) as f:
    json.dump(validation_rows, f, indent=2)

print("\nSaved:")
print(out_dir / "state_passing_raw_tensors.pt")
print(out_dir / "state_passing_fp32_recurrence_validation.json")

print(
    "\nOriginal function restored:",
    ssd_combined._state_passing_fwd is original_fn
)


TRITON SOURCE INSPECTION

Autotuner:
src: unavailable (AttributeError)
source: unavailable (AttributeError)
arg_names: ['states_ptr', 'out_ptr', 'final_states_ptr', 'dA_cs_ptr', 'initstates_ptr', 'seq_idx_ptr', 'dim', 'nchunks', 'seqlen', 'chunk_size', 'stride_states_batch', 'stride_states_chunk', 'stride_states_head', 'stride_states_dim', 'stride_out_batch', 'stride_out_chunk', 'stride_out_head', 'stride_out_dim', 'stride_final_states_batch', 'stride_final_states_head', 'stride_final_states_dim', 'stride_dA_cs_batch', 'stride_dA_cs_chunk', 'stride_dA_cs_head', 'stride_initstates_batch', 'stride_initstates_head', 'stride_initstates_dim', 'stride_seq_idx_batch', 'stride_seq_idx_seqlen', 'HAS_INITSTATES', 'HAS_SEQ_IDX', 'BLOCK_SIZE']
constexprs: unavailable (AttributeError)

JITFunction:

src:
def _state_passing_fwd_kernel(
    # Pointers to matrices
    states_ptr, out_ptr, final_states_ptr, dA_cs_ptr, initstates_ptr, seq_idx_ptr,
    # Matrix dimensions
    dim, nchunks, seqlen, chunk_

In [135]:

import torch
import numpy as np
import json
from pathlib import Path

capture_path = Path(
    "/kaggle/working/shiftscan_final/state_passing/"
    "state_passing_raw_tensors.pt"
)

captured = torch.load(
    capture_path,
    map_location="cpu",
    weights_only=False
)

rows = []

for layer_idx, record in enumerate(captured):
    N = record["states"].float()
    decay = torch.exp(record["decay_log"].float())
    initial = record["initial_states"]

    triton_out = record["triton_out"].float()
    triton_final = record["triton_final"].float()

    B, nchunks, nheads, dim = N.shape

    if initial is None:
        running = torch.zeros(
            (B, nheads, dim), dtype=torch.float32
        )
    else:
        running = initial.float().clone()

    # The Triton output's first slot stores the initial state.
    reconstructed_out = [running.clone()]

    # Apply the actual chunk recurrence.
    for c in range(nchunks):
        scale = decay[:, :, c].unsqueeze(-1)
        running = scale * running + N[:, c]

        if c < nchunks - 1:
            reconstructed_out.append(running.clone())

    reconstructed_out = torch.stack(
        reconstructed_out, dim=1
    )

    # Compare intermediate outputs in their correct positions.
    intermediate_diff = reconstructed_out - triton_out

    # Compare final state separately.
    final_diff = running - triton_final

    row = {
        "layer": layer_idx,
        "chunks": nchunks,
        "intermediate_mae": float(intermediate_diff.abs().mean()),
        "intermediate_rmse": float(
            torch.sqrt((intermediate_diff ** 2).mean())
        ),
        "intermediate_max_abs": float(
            intermediate_diff.abs().max()
        ),
        "final_mae": float(final_diff.abs().mean()),
        "final_rmse": float(
            torch.sqrt((final_diff ** 2).mean())
        ),
        "final_max_abs": float(final_diff.abs().max()),
    }

    rows.append(row)

    print(
        f"Layer {layer_idx:02d} | "
        f"intermediate MAE={row['intermediate_mae']:.8g} | "
        f"intermediate max={row['intermediate_max_abs']:.8g} | "
        f"final max={row['final_max_abs']:.8g}"
    )

print("\n" + "=" * 70)
print("AGGREGATE CORRECTED VALIDATION")
print("=" * 70)

for key in [
    "intermediate_mae",
    "intermediate_rmse",
    "intermediate_max_abs",
    "final_mae",
    "final_rmse",
    "final_max_abs",
]:
    values = [r[key] for r in rows]
    print(
        f"{key:25s} "
        f"mean={np.mean(values):.9g} "
        f"max={np.max(values):.9g}"
    )

out_path = capture_path.parent / "corrected_recurrence_validation.json"

with open(out_path, "w") as f:
    json.dump(rows, f, indent=2)

print("\nSaved:", out_path)


Layer 00 | intermediate MAE=7.9019965e-06 | intermediate max=0.015533447 | final max=1.1920929e-07
Layer 01 | intermediate MAE=1.958481e-05 | intermediate max=0.0077171326 | final max=1.9073486e-06
Layer 02 | intermediate MAE=1.0699211e-05 | intermediate max=0.0077171326 | final max=1.9073486e-06
Layer 03 | intermediate MAE=4.6215555e-05 | intermediate max=0.031196594 | final max=7.6293945e-06
Layer 04 | intermediate MAE=5.3921012e-05 | intermediate max=0.029075623 | final max=9.5367432e-06
Layer 05 | intermediate MAE=3.536779e-05 | intermediate max=0.015037537 | final max=3.8146973e-06
Layer 06 | intermediate MAE=2.5693531e-05 | intermediate max=0.012466431 | final max=3.8146973e-06
Layer 07 | intermediate MAE=8.929817e-05 | intermediate max=0.030082703 | final max=7.6293945e-06
Layer 08 | intermediate MAE=3.4278037e-05 | intermediate max=0.030174255 | final max=4.7683716e-07
Layer 09 | intermediate MAE=2.7589791e-05 | intermediate max=0.031188965 | final max=7.6293945e-06
Layer 10 | 

In [136]:

import torch
import numpy as np
import json
from pathlib import Path

capture_path = Path(
    "/kaggle/working/shiftscan_final/state_passing/"
    "state_passing_raw_tensors.pt"
)

captured = torch.load(
    capture_path, map_location="cpu", weights_only=False
)

rows = []

for layer_idx, record in enumerate(captured):
    N = record["states"].float()
    decay = torch.exp(record["decay_log"].float())
    initial = record["initial_states"]

    triton_out = record["triton_out"].float()
    triton_final = record["triton_final"].float()

    B, nchunks, nheads, dim = N.shape

    running = (
        initial.float().clone()
        if initial is not None
        else torch.zeros((B, nheads, dim), dtype=torch.float32)
    )

    reconstructed_out = [running.clone()]

    for c in range(nchunks):
        running = (
            decay[:, :, c].unsqueeze(-1) * running
            + N[:, c]
        )

        if c < nchunks - 1:
            reconstructed_out.append(running.clone())

    reconstructed_out = torch.stack(reconstructed_out, dim=1)

    # Match the kernel's actual output dtype.
    reconstructed_cast = reconstructed_out.to(torch.float16).float()

    intermediate_diff = reconstructed_cast - triton_out
    final_diff = running - triton_final

    rows.append({
        "layer": layer_idx,
        "intermediate_mae_after_cast": float(
            intermediate_diff.abs().mean()
        ),
        "intermediate_rmse_after_cast": float(
            torch.sqrt((intermediate_diff ** 2).mean())
        ),
        "intermediate_max_after_cast": float(
            intermediate_diff.abs().max()
        ),
        "final_mae": float(final_diff.abs().mean()),
        "final_max": float(final_diff.abs().max()),
    })

print("Layer | intermediate MAE after FP16 cast | max | final max")
for r in rows:
    print(
        f"{r['layer']:02d} | "
        f"{r['intermediate_mae_after_cast']:.9g} | "
        f"{r['intermediate_max_after_cast']:.9g} | "
        f"{r['final_max']:.9g}"
    )

print("\nAggregate")
for key in [
    "intermediate_mae_after_cast",
    "intermediate_rmse_after_cast",
    "intermediate_max_after_cast",
    "final_max",
]:
    vals = [r[key] for r in rows]
    print(
        f"{key}: mean={np.mean(vals):.9g}, "
        f"max={np.max(vals):.9g}"
    )

out_path = capture_path.parent / "fp16_cast_recurrence_validation.json"
with open(out_path, "w") as f:
    json.dump(rows, f, indent=2)

print("\nSaved:", out_path)


Layer | intermediate MAE after FP16 cast | max | final max
00 | 0 | 0 | 1.1920929e-07
01 | 3.13699872e-10 | 0.000122070312 | 1.90734863e-06
02 | 5.56762358e-10 | 0.000244140625 | 1.90734863e-06
03 | 9.73917266e-11 | 6.10351562e-05 | 7.62939453e-06
04 | 1.26888153e-08 | 0.00390625 | 9.53674316e-06
05 | 2.10100848e-09 | 0.0009765625 | 3.81469727e-06
06 | 1.04788944e-09 | 0.00048828125 | 3.81469727e-06
07 | 3.63797881e-12 | 1.90734863e-06 | 7.62939453e-06
08 | 1.9705718e-11 | 1.52587891e-05 | 4.76837158e-07
09 | 8.00355338e-11 | 6.10351562e-05 | 7.62939453e-06
10 | 3.40583028e-09 | 0.001953125 | 3.81469727e-06
11 | 3.0316491e-13 | 2.38418579e-07 | 9.53674316e-07
12 | 6.2179123e-10 | 0.00048828125 | 1.90734863e-06
13 | 2.09411155e-10 | 6.10351562e-05 | 1.90734863e-06
14 | 1.80231538e-10 | 6.10351562e-05 | 4.76837158e-07
15 | 1.09973064e-09 | 0.00048828125 | 7.62939453e-06
16 | 8.38402514e-10 | 0.000244140625 | 2.38418579e-06
17 | 0 | 0 | 9.31322575e-10
18 | 3.63797881e-12 | 1.90734863e-06 

In [137]:

import ast
import re
import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path

# ------------------------------------------------------------
# 1. Load actual captured Mamba-2 tensors and canonical grid
# ------------------------------------------------------------

root = Path("/kaggle/working/shiftscan_final")
capture_path = root / "state_passing/state_passing_raw_tensors.pt"
grid_path = root / "canonical_dyadic_grid_definitions.csv"

captured = torch.load(
    capture_path,
    map_location="cpu",
    weights_only=False
)

grid_df = pd.read_csv(grid_path)
grid_df = grid_df[grid_df["grid"] == "K1_14_T3"].copy()

assert len(grid_df) > 0, "K1_14_T3 grid was not found."

def parse_exponents(value):
    if isinstance(value, (list, tuple, np.ndarray)):
        return tuple(int(x) for x in value)

    try:
        parsed = ast.literal_eval(str(value))
        return tuple(int(x) for x in parsed)
    except Exception:
        return tuple(
            int(x) for x in re.findall(r"\d+", str(value))
        )

grid_df["exponents"] = grid_df["canonical_exponents"].apply(
    parse_exponents
)

grid_df = grid_df.sort_values("decay").drop_duplicates(
    subset=["decay"]
)

grid_decay = grid_df["decay"].to_numpy(dtype=np.float64)
grid_exponents = list(grid_df["exponents"])

assert all(len(k) == 3 for k in grid_exponents), (
    "Expected three-term representations in K1_14_T3."
)

grid_d = 1.0 - grid_decay
log_grid_d = np.log(grid_d)

print("Canonical unique coefficients:", len(grid_decay))
print("Decay range:", grid_decay.min(), "to", grid_decay.max())
print("Exponent examples:", grid_exponents[:5])


# ------------------------------------------------------------
# 2. Quantize chunk decays using nearest log-distance
# ------------------------------------------------------------

def quantize_chunk_decays(decay_tensor):
    """
    decay_tensor: [B, heads, chunks]

    Quantize only when the decay lies within the canonical grid
    range. Otherwise preserve exact decay as a fallback.
    """
    decay_np = decay_tensor.numpy().astype(np.float64)

    q_decay = decay_np.copy()
    exponent_map = np.zeros(
        decay_np.shape + (3,), dtype=np.int32
    )

    eligible = (
        np.isfinite(decay_np)
        & (decay_np >= grid_decay.min())
        & (decay_np <= grid_decay.max())
    )

    flat_decay = decay_np.reshape(-1)
    flat_q = q_decay.reshape(-1)
    flat_exp = exponent_map.reshape(-1, 3)
    flat_eligible = eligible.reshape(-1)

    for i in np.flatnonzero(flat_eligible):
        a = flat_decay[i]
        d = 1.0 - a

        if d <= 0:
            flat_eligible[i] = False
            continue

        idx = int(np.argmin(np.abs(np.log(d) - log_grid_d)))

        flat_q[i] = grid_decay[idx]
        flat_exp[i] = grid_exponents[idx]

    return (
        torch.from_numpy(q_decay).float(),
        torch.from_numpy(eligible),
        torch.from_numpy(exponent_map),
    )


# ------------------------------------------------------------
# 3. Signed rounded right shift: ties away from zero
# ------------------------------------------------------------

def round_shift_signed(x, k):
    """
    Integer round(x / 2**k), ties away from zero.
    x is int64.
    """
    x = x.to(torch.int64)
    magnitude = torch.abs(x)
    rounded = (magnitude + (1 << (k - 1))) >> k
    return torch.where(x < 0, -rounded, rounded)


# ------------------------------------------------------------
# 4. Evaluate FP32 and fixed-point recurrences
# ------------------------------------------------------------

all_rows = []
summary_rows = []

for layer_idx, record in enumerate(captured):
    N = record["states"].float()            # [B,C,H,D]
    decay = torch.exp(record["decay_log"].float())  # [B,H,C]

    B, nchunks, nheads, dim = N.shape

    q_decay, eligible, exponent_map = quantize_chunk_decays(
        decay
    )

    fallback = ~eligible

    # FP32 reference recurrence
    ref_state = torch.zeros(
        (B, nheads, dim), dtype=torch.float32
    )

    # FP32 Dyadic2 coefficient recurrence
    dyadic_state = torch.zeros_like(ref_state)

    for c in range(nchunks):
        ref_state = (
            decay[:, :, c].unsqueeze(-1) * ref_state
            + N[:, c]
        )

        dyadic_state = (
            q_decay[:, :, c].unsqueeze(-1) * dyadic_state
            + N[:, c]
        )

    dyadic_error = dyadic_state - ref_state

    for F in [20, 21, 22]:
        scale = 1 << F

        fixed_state = torch.zeros(
            (B, nheads, dim), dtype=torch.int64
        )

        max_abs_integer_state = 0
        max_abs_input_integer = 0
        max_abs_integer_seen = 0

        for c in range(nchunks):
            # Quantize incoming chunk contribution to fixed point.
            N_int = torch.round(
                N[:, c].double() * scale
            ).to(torch.int64)

            max_abs_input_integer = max(
                max_abs_input_integer,
                int(N_int.abs().max().item())
            )

            next_state = fixed_state.clone()

            for b in range(B):
                for h in range(nheads):
                    if bool(eligible[b, h, c]):
                        # Dyadic2: a_q = 1 - sum(2**-k)
                        exponents = exponent_map[b, h, c].tolist()

                        value = fixed_state[b, h]

                        for k in exponents:
                            value = value - round_shift_signed(
                                fixed_state[b, h], int(k)
                            )

                        next_state[b, h] = value + N_int[b, h]

                    else:
                        # Exact-decay fallback, rounded back to fixed point.
                        product = (
                            decay[b, h, c].double()
                            * fixed_state[b, h].double()
                        )
                        next_state[b, h] = (
                            torch.round(product).to(torch.int64)
                            + N_int[b, h]
                        )

            fixed_state = next_state

            current_max = int(fixed_state.abs().max().item())
            max_abs_integer_seen = max(
                max_abs_integer_seen, current_max
            )

        fixed_real = fixed_state.double() / scale
        ref_final = ref_state.double()

        fixed_error = fixed_real - ref_final

        int32_limit = 2**31 - 1
        int32_safe = max_abs_integer_seen <= int32_limit

        # Nominal signed real range for int32 at this precision.
        nominal_real_limit = (2**31 - 1) / scale

        row = {
            "layer": layer_idx,
            "fraction_bits": F,
            "chunks": nchunks,
            "dyadic_coverage_pct": float(
                eligible.float().mean().item() * 100
            ),
            "fallback_pct": float(
                fallback.float().mean().item() * 100
            ),
            "fp32_dyadic_mae": float(dyadic_error.abs().mean()),
            "fp32_dyadic_rmse": float(
                torch.sqrt((dyadic_error ** 2).mean())
            ),
            "fp32_dyadic_max": float(dyadic_error.abs().max()),
            "fixed_mae": float(fixed_error.abs().mean()),
            "fixed_rmse": float(
                torch.sqrt((fixed_error ** 2).mean())
            ),
            "fixed_max": float(fixed_error.abs().max()),
            "max_abs_integer_state": max_abs_integer_seen,
            "max_abs_integer_input": max_abs_input_integer,
            "nominal_int32_real_limit": nominal_real_limit,
            "int32_safe": bool(int32_safe),
        }

        all_rows.append(row)

        print(
            f"Layer {layer_idx:02d} | F{F} | "
            f"coverage={row['dyadic_coverage_pct']:.1f}% | "
            f"FP32-Dyadic MAE={row['fp32_dyadic_mae']:.5g} | "
            f"Fixed MAE={row['fixed_mae']:.5g} | "
            f"Fixed RMSE={row['fixed_rmse']:.5g} | "
            f"Fixed max={row['fixed_max']:.5g} | "
            f"int32 safe={int32_safe}"
        )


# ------------------------------------------------------------
# 5. Aggregate by precision
# ------------------------------------------------------------

results_df = pd.DataFrame(all_rows)

print("\n" + "=" * 75)
print("AGGREGATE RESULTS")
print("=" * 75)

for F, group in results_df.groupby("fraction_bits"):
    summary = {
        "fraction_bits": int(F),
        "mean_coverage_pct": group["dyadic_coverage_pct"].mean(),
        "mean_fp32_dyadic_mae": group["fp32_dyadic_mae"].mean(),
        "mean_fp32_dyadic_rmse": group["fp32_dyadic_rmse"].mean(),
        "mean_fixed_mae": group["fixed_mae"].mean(),
        "mean_fixed_rmse": group["fixed_rmse"].mean(),
        "worst_layer_fixed_max": group["fixed_max"].max(),
        "max_integer_state": group["max_abs_integer_state"].max(),
        "int32_safe_all_layers": bool(group["int32_safe"].all()),
    }

    summary_rows.append(summary)

    print(json.dumps(summary, indent=2))


# ------------------------------------------------------------
# 6. Save all results
# ------------------------------------------------------------

out_dir = root / "state_passing/fixed_point_pilot"
out_dir.mkdir(parents=True, exist_ok=True)

results_df.to_csv(
    out_dir / "layer_fixed_point_results.csv",
    index=False
)

with open(out_dir / "fixed_point_summary.json", "w") as f:
    json.dump(summary_rows, f, indent=2)

print("\nSaved:")
print(out_dir / "layer_fixed_point_results.csv")
print(out_dir / "fixed_point_summary.json")


Canonical unique coefficients: 467
Decay range: 0.125 to 0.99981689453125
Exponent examples: [(1, 2, 3), (1, 2, 4), (1, 2, 5), (1, 2, 6), (1, 2, 7)]
Layer 00 | F20 | coverage=0.0% | FP32-Dyadic MAE=0 | Fixed MAE=3.1382e-07 | Fixed RMSE=3.8448e-07 | Fixed max=9.5367e-07 | int32 safe=True
Layer 00 | F21 | coverage=0.0% | FP32-Dyadic MAE=0 | Fixed MAE=1.5661e-07 | Fixed RMSE=1.9203e-07 | Fixed max=4.7684e-07 | int32 safe=True
Layer 00 | F22 | coverage=0.0% | FP32-Dyadic MAE=0 | Fixed MAE=7.8536e-08 | Fixed RMSE=9.6356e-08 | Fixed max=4.7684e-07 | int32 safe=True
Layer 01 | F20 | coverage=14.6% | FP32-Dyadic MAE=0.00096976 | Fixed MAE=0.00096996 | Fixed RMSE=0.00805 | Fixed max=0.40739 | int32 safe=True
Layer 01 | F21 | coverage=14.6% | FP32-Dyadic MAE=0.00096976 | Fixed MAE=0.00096986 | Fixed RMSE=0.00805 | Fixed max=0.40739 | int32 safe=True
Layer 01 | F22 | coverage=14.6% | FP32-Dyadic MAE=0.00096976 | Fixed MAE=0.00096981 | Fixed RMSE=0.00805 | Fixed max=0.40739 | int32 safe=True
Layer

In [138]:

import ast
import re
import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path

root = Path("/kaggle/working/shiftscan_final")

captured = torch.load(
    root / "state_passing/state_passing_raw_tensors.pt",
    map_location="cpu",
    weights_only=False
)

grid_df = pd.read_csv(
    root / "canonical_dyadic_grid_definitions.csv"
)
grid_df = grid_df[grid_df["grid"] == "K1_14_T3"].copy()

def parse_exp(x):
    try:
        return tuple(int(v) for v in ast.literal_eval(str(x)))
    except Exception:
        return tuple(int(v) for v in re.findall(r"\d+", str(x)))

grid_df["exponents"] = grid_df["canonical_exponents"].apply(parse_exp)
grid_df = grid_df.sort_values("decay").drop_duplicates("decay")

dyadic_a = grid_df["decay"].to_numpy(np.float64)
dyadic_exp = list(grid_df["exponents"])
dyadic_log_d = np.log(1.0 - dyadic_a)

# Direct powers of two for small chunk decays.
POW_MIN = 4
POW_MAX = 30
pow_k = np.arange(POW_MIN, POW_MAX + 1)
pow_a = np.exp2(-pow_k.astype(np.float64))

def hybrid_quantize(decay):
    """
    Modes:
      0 = exact fallback
      1 = three-term 1 - sum(2^-k)
      2 = direct power-of-two 2^-k
      3 = exact zero coefficient
    """
    a = decay.numpy().astype(np.float64)

    q = a.copy()
    mode = np.zeros(a.shape, dtype=np.int8)
    exps = np.zeros(a.shape + (3,), dtype=np.int32)
    shift_k = np.zeros(a.shape, dtype=np.int32)

    flat_a = a.reshape(-1)
    flat_q = q.reshape(-1)
    flat_mode = mode.reshape(-1)
    flat_exps = exps.reshape(-1, 3)
    flat_shift = shift_k.reshape(-1)

    for i, value in enumerate(flat_a):
        if not np.isfinite(value) or value < 0 or value > 1:
            continue

        if value == 0:
            flat_q[i] = 0.0
            flat_mode[i] = 3
            continue

        if value < 0.125:
            # Nearest power of two in log space.
            idx = int(np.argmin(np.abs(np.log(value) - np.log(pow_a))))
            candidate = pow_a[idx]

            if value >= pow_a.min():
                flat_q[i] = candidate
                flat_mode[i] = 2
                flat_shift[i] = int(pow_k[idx])
            # Otherwise exact fallback.
            continue

        # Original three-term Dyadic2 region.
        d = 1.0 - value
        idx = int(np.argmin(np.abs(np.log(d) - dyadic_log_d)))

        flat_q[i] = dyadic_a[idx]
        flat_mode[i] = 1
        flat_exps[i] = dyadic_exp[idx]

    return (
        torch.from_numpy(q).float(),
        torch.from_numpy(mode),
        torch.from_numpy(exps),
        torch.from_numpy(shift_k),
    )


# ------------------------------------------------------------
# Coefficient coverage and error
# ------------------------------------------------------------

layer_rows = []
all_abs_errors = []
all_sq_errors = []
all_valid_count = 0
all_quantized_count = 0
all_fallback_count = 0
all_zero_count = 0
all_dyadic_count = 0
all_pot_count = 0

# Keep per-layer quantized values for recurrence validation.
quantized_records = []

for layer_idx, record in enumerate(captured):
    decay = torch.exp(record["decay_log"].float())

    q, mode, exps, shift_k = hybrid_quantize(decay)

    error = q - decay
    abs_error = error.abs()

    quantized_mask = mode != 0
    fallback_mask = mode == 0

    count = decay.numel()
    qcount = int(quantized_mask.sum())
    fcount = int(fallback_mask.sum())

    all_valid_count += count
    all_quantized_count += qcount
    all_fallback_count += fcount
    all_zero_count += int((mode == 3).sum())
    all_dyadic_count += int((mode == 1).sum())
    all_pot_count += int((mode == 2).sum())

    all_abs_errors.extend(abs_error.flatten().tolist())
    all_sq_errors.extend((error ** 2).flatten().tolist())

    row = {
        "layer": layer_idx,
        "total_coefficients": count,
        "quantized_count": qcount,
        "fallback_count": fcount,
        "coverage_pct": 100 * qcount / count,
        "fallback_pct": 100 * fcount / count,
        "dyadic2_pct": 100 * float((mode == 1).float().mean()),
        "direct_pot_pct": 100 * float((mode == 2).float().mean()),
        "zero_pct": 100 * float((mode == 3).float().mean()),
        "coefficient_mae": float(abs_error.mean()),
        "coefficient_rmse": float(torch.sqrt((error ** 2).mean())),
        "coefficient_max_abs": float(abs_error.max()),
    }
    layer_rows.append(row)

    quantized_records.append({
        "q": q,
        "mode": mode,
        "exps": exps,
        "shift_k": shift_k,
    })


# ------------------------------------------------------------
# FP32 recurrence: exact versus hybrid coefficient
# ------------------------------------------------------------

recurrence_rows = []

for layer_idx, (record, qr) in enumerate(
    zip(captured, quantized_records)
):
    N = record["states"].float()
    decay = torch.exp(record["decay_log"].float())

    q = qr["q"]

    B, nchunks, nheads, dim = N.shape

    ref = torch.zeros((B, nheads, dim), dtype=torch.float32)
    hybrid = torch.zeros_like(ref)

    for c in range(nchunks):
        ref = decay[:, :, c].unsqueeze(-1) * ref + N[:, c]
        hybrid = q[:, :, c].unsqueeze(-1) * hybrid + N[:, c]

    diff = hybrid - ref

    recurrence_rows.append({
        "layer": layer_idx,
        "recurrence_mae": float(diff.abs().mean()),
        "recurrence_rmse": float(torch.sqrt((diff ** 2).mean())),
        "recurrence_max_abs": float(diff.abs().max()),
        "reference_absmax": float(ref.abs().max()),
        "hybrid_absmax": float(hybrid.abs().max()),
    })


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("=" * 75)
print("HYBRID COEFFICIENT COVERAGE")
print("=" * 75)

print("Total coefficients:", all_valid_count)
print("Quantized:", all_quantized_count)
print("Fallback:", all_fallback_count)
print("Overall coverage:", 100 * all_quantized_count / all_valid_count)
print("Fallback percentage:", 100 * all_fallback_count / all_valid_count)
print("Three-term Dyadic2:", all_dyadic_count)
print("Direct power-of-two:", all_pot_count)
print("Exact zero:", all_zero_count)

err = np.asarray(all_abs_errors)
sq = np.asarray(all_sq_errors)

print("\nCoefficient error:")
print("MAE:", err.mean())
print("RMSE:", np.sqrt(sq.mean()))
print("Max abs:", err.max())

print("\n" + "=" * 75)
print("PER-LAYER RESULTS")
print("=" * 75)

for row in layer_rows:
    print(
        f"Layer {row['layer']:02d} | "
        f"coverage={row['coverage_pct']:.2f}% | "
        f"Dyadic2={row['dyadic2_pct']:.1f}% | "
        f"PoT={row['direct_pot_pct']:.1f}% | "
        f"zero={row['zero_pct']:.1f}% | "
        f"fallback={row['fallback_pct']:.1f}% | "
        f"coef MAE={row['coefficient_mae']:.6g}"
    )

print("\n" + "=" * 75)
print("FOUR-CHUNK FP32 RECURRENCE ERROR")
print("=" * 75)

for row in recurrence_rows:
    print(
        f"Layer {row['layer']:02d} | "
        f"MAE={row['recurrence_mae']:.7g} | "
        f"RMSE={row['recurrence_rmse']:.7g} | "
        f"max={row['recurrence_max_abs']:.7g}"
    )

print("\nAggregate recurrence:")
for key in [
    "recurrence_mae",
    "recurrence_rmse",
    "recurrence_max_abs",
]:
    vals = [r[key] for r in recurrence_rows]
    print(
        f"{key}: mean={np.mean(vals):.9g}, "
        f"worst_layer={np.max(vals):.9g}"
    )


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

out_dir = root / "state_passing/hybrid_grid_pilot"
out_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame(layer_rows).to_csv(
    out_dir / "hybrid_layer_coverage.csv",
    index=False
)

pd.DataFrame(recurrence_rows).to_csv(
    out_dir / "hybrid_recurrence_errors.csv",
    index=False
)

summary = {
    "total_coefficients": all_valid_count,
    "quantized_count": all_quantized_count,
    "fallback_count": all_fallback_count,
    "coverage_pct": 100 * all_quantized_count / all_valid_count,
    "fallback_pct": 100 * all_fallback_count / all_valid_count,
    "dyadic2_count": all_dyadic_count,
    "direct_pot_count": all_pot_count,
    "exact_zero_count": all_zero_count,
    "coefficient_mae": float(err.mean()),
    "coefficient_rmse": float(np.sqrt(sq.mean())),
    "coefficient_max_abs": float(err.max()),
    "recurrence_mean_mae": float(np.mean([
        r["recurrence_mae"] for r in recurrence_rows
    ])),
    "recurrence_mean_rmse": float(np.mean([
        r["recurrence_rmse"] for r in recurrence_rows
    ])),
    "recurrence_worst_layer_max": float(np.max([
        r["recurrence_max_abs"] for r in recurrence_rows
    ])),
}

with open(out_dir / "hybrid_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nSaved results to:", out_dir)


HYBRID COEFFICIENT COVERAGE
Total coefficients: 2304
Quantized: 1767
Fallback: 537
Overall coverage: 76.69270833333333
Fallback percentage: 23.307291666666668
Three-term Dyadic2: 497
Direct power-of-two: 493
Exact zero: 777

Coefficient error:
MAE: 0.001914726979006223
RMSE: 0.0061038524377518555
Max abs: 0.06093733012676239

PER-LAYER RESULTS
Layer 00 | coverage=95.83% | Dyadic2=0.0% | PoT=95.8% | zero=0.0% | fallback=4.2% | coef MAE=0.000172269
Layer 01 | coverage=79.17% | Dyadic2=14.6% | PoT=14.6% | zero=50.0% | fallback=20.8% | coef MAE=0.00450954
Layer 02 | coverage=87.50% | Dyadic2=53.1% | PoT=13.5% | zero=20.8% | fallback=12.5% | coef MAE=0.00623172
Layer 03 | coverage=70.83% | Dyadic2=4.2% | PoT=29.2% | zero=37.5% | fallback=29.2% | coef MAE=0.00054162
Layer 04 | coverage=91.67% | Dyadic2=65.6% | PoT=13.5% | zero=12.5% | fallback=8.3% | coef MAE=0.00325415
Layer 05 | coverage=93.75% | Dyadic2=21.9% | PoT=9.4% | zero=62.5% | fallback=6.2% | coef MAE=0.00253155
Layer 06 | coverag

In [139]:
import os, json, torch

BASE = "/kaggle/working/shiftscan_final/state_passing"
RAW = os.path.join(BASE, "state_passing_raw_tensors.pt")
PILOT = os.path.join(BASE, "hybrid_grid_pilot")

print("=" * 72)
print("SAVED TENSOR STRUCTURE")
print("=" * 72)

obj = torch.load(RAW, map_location="cpu", weights_only=False)

def describe(x, path="root", depth=0):
    indent = "  " * depth

    if isinstance(x, dict):
        print(f"{indent}{path}: dict, {len(x)} keys")
        for k, v in x.items():
            describe(v, f"{path}.{k}", depth + 1)

    elif isinstance(x, (list, tuple)):
        print(f"{indent}{path}: {type(x).__name__}, length={len(x)}")
        if len(x):
            describe(x[0], f"{path}[0]", depth + 1)

    elif torch.is_tensor(x):
        print(
            f"{indent}{path}: Tensor "
            f"shape={tuple(x.shape)}, dtype={x.dtype}, "
            f"min={x.min().item():.6g}, max={x.max().item():.6g}"
        )

    else:
        print(f"{indent}{path}: {type(x).__name__} = {str(x)[:160]}")

describe(obj)

print("\n" + "=" * 72)
print("HYBRID PILOT FILES")
print("=" * 72)

for root, dirs, files in os.walk(PILOT):
    for filename in sorted(files):
        path = os.path.join(root, filename)
        print(f"{path}  ({os.path.getsize(path):,} bytes)")

        if filename.endswith(".json"):
            try:
                with open(path) as f:
                    data = json.load(f)
                print(json.dumps(data, indent=2)[:5000])
            except Exception as e:
                print("JSON read error:", repr(e))

SAVED TENSOR STRUCTURE
root: list, length=24
  root[0]: dict, 8 keys
    root[0].states: Tensor shape=(1, 4, 24, 8192), dtype=torch.float32, min=-5.22889, max=36.968
    root[0].decay_log: Tensor shape=(1, 24, 4), dtype=torch.float32, min=-44.5382, max=-6.4795
    root[0].initial_states: NoneType = None
    root[0].triton_out: Tensor shape=(1, 4, 24, 8192), dtype=torch.float32, min=-5.17578, max=36.8125
    root[0].triton_final: Tensor shape=(1, 24, 8192), dtype=torch.float32, min=-5.23399, max=37.0041
    root[0].seq_idx_present: bool = False
    root[0].chunk_size: int = 256
    root[0].out_dtype: str = torch.float16

HYBRID PILOT FILES
/kaggle/working/shiftscan_final/state_passing/hybrid_grid_pilot/hybrid_layer_coverage.csv  (3,482 bytes)
/kaggle/working/shiftscan_final/state_passing/hybrid_grid_pilot/hybrid_recurrence_errors.csv  (2,559 bytes)
/kaggle/working/shiftscan_final/state_passing/hybrid_grid_pilot/hybrid_summary.json  (518 bytes)
{
  "total_coefficients": 2304,
  "quantize

In [140]:
import os
import ast
import numpy as np
import pandas as pd
import torch

BASE = "/kaggle/working/shiftscan_final/state_passing"
RAW_PATH = os.path.join(BASE, "state_passing_raw_tensors.pt")
GRID_PATH = "/kaggle/working/shiftscan_final/canonical_dyadic_grid_definitions.csv"

OUT_DIR = os.path.join(BASE, "hybrid_grid_pilot", "error_isolation")
os.makedirs(OUT_DIR, exist_ok=True)

raw = torch.load(RAW_PATH, map_location="cpu", weights_only=False)

grid_df = pd.read_csv(GRID_PATH)
grid_df = grid_df[grid_df["grid"] == "K1_14_T3"].copy()

grid_df["exponents"] = grid_df["canonical_exponents"].apply(
    lambda x: tuple(ast.literal_eval(x))
)

grid_decay = grid_df["decay"].to_numpy(dtype=np.float64)
grid_exp = grid_df["exponents"].tolist()

# Direct power-of-two candidates used by the hybrid policy.
pot_ks = np.arange(4, 31)
pot_values = 2.0 ** (-pot_ks)

def select_hybrid(a):
    """
    Same hybrid policy as the previous pilot:
      mode 1: three-term Dyadic2 for a >= 0.125
      mode 2: direct power-of-two for 2^-30 <= a < 0.125
      mode 3: exact zero
      mode 0: exact fallback
    """
    a = float(a)

    if a == 0.0:
        return 0.0, 3, (), None

    if a >= 0.125:
        d = 1.0 - a
        idx = int(np.argmin(np.abs(np.log(d) - np.log(1.0 - grid_decay))))
        return float(grid_decay[idx]), 1, grid_exp[idx], None

    if a >= 2.0 ** (-30):
        idx = int(np.argmin(np.abs(np.log(a) - np.log(pot_values))))
        return float(pot_values[idx]), 2, (), int(pot_ks[idx])

    return a, 0, (), None


def recurrence(states, decays):
    """
    FP32 state-passing recurrence:
        S_c = decay_c * S_(c-1) + N_c
    states shape: [chunks, heads, state_dim]
    decays shape: [chunks, heads]
    """
    chunks, heads, dim = states.shape
    state = torch.zeros((heads, dim), dtype=torch.float32)
    outputs = []

    for c in range(chunks):
        state = decays[c, :, None] * state + states[c]
        outputs.append(state.clone())

    return torch.stack(outputs, dim=0), state.clone()


rows = []
all_intermediate = []

for layer_idx, record in enumerate(raw):
    N = record["states"][0].float()          # [4, 24, 8192]
    log_decay = record["decay_log"][0].float()  # [24, 4]
    true_decay = torch.exp(log_decay).T.contiguous()  # [4, 24]

    chunks, heads, dim = N.shape

    q = torch.empty_like(true_decay)
    modes = np.zeros((chunks, heads), dtype=np.int32)

    # Reconstruct the exact hybrid selection policy.
    for c in range(chunks):
        for h in range(heads):
            value = float(true_decay[c, h])
            q_value, mode, exponents, shift_k = select_hybrid(value)
            q[c, h] = q_value
            modes[c, h] = mode

    # Two independent FP32 recurrences.
    ref_out, ref_final = recurrence(N, true_decay)
    hybrid_out, hybrid_final = recurrence(N, q)

    # Actual Triton outputs: intermediate output is stored after chunks 0,1,2.
    captured_out = record["triton_out"][0].float()
    captured_final = record["triton_final"][0].float()

    # Compare independent reference against the actual Triton output.
    triton_intermediate_error = (
        ref_out[:3] - captured_out[1:4]
    ).abs()

    triton_final_error = (ref_final - captured_final).abs()

    # Compare hybrid FP32 recurrence against original FP32 recurrence.
    hybrid_error = (hybrid_out - ref_out).abs()

    # Per-layer metrics.
    coefficient_error = (q - true_decay).abs()
    quantized_mask = torch.from_numpy(modes != 0)
    fallback_mask = torch.from_numpy(modes == 0)

    qerr = coefficient_error[quantized_mask]
    ferr = coefficient_error[fallback_mask]

    rows.append({
        "layer": layer_idx,
        "coverage_pct": 100.0 * float(quantized_mask.float().mean()),
        "fallback_pct": 100.0 * float(fallback_mask.float().mean()),
        "dyadic2_pct": 100.0 * float((modes == 1).mean()),
        "pot_pct": 100.0 * float((modes == 2).mean()),
        "zero_pct": 100.0 * float((modes == 3).mean()),

        "coef_mae_all": coefficient_error.mean().item(),
        "coef_mae_quantized_only": qerr.mean().item() if qerr.numel() else 0.0,
        "coef_max_quantized_only": qerr.max().item() if qerr.numel() else 0.0,

        "hybrid_recurrence_mae": hybrid_error.mean().item(),
        "hybrid_recurrence_rmse": hybrid_error.square().mean().sqrt().item(),
        "hybrid_recurrence_max": hybrid_error.max().item(),

        "triton_intermediate_mae": triton_intermediate_error.mean().item(),
        "triton_intermediate_max": triton_intermediate_error.max().item(),
        "triton_final_mae": triton_final_error.mean().item(),
        "triton_final_max": triton_final_error.max().item(),

        "hybrid_final_mae": (hybrid_final - ref_final).abs().mean().item(),
        "hybrid_final_max": (hybrid_final - ref_final).abs().max().item(),
    })

    all_intermediate.append({
        "layer": layer_idx,
        "true_decay": true_decay,
        "hybrid_decay": q,
        "modes": modes,
        "reference": ref_out,
        "hybrid": hybrid_out,
        "captured": captured_out,
        "reference_final": ref_final,
        "hybrid_final": hybrid_final,
        "captured_final": captured_final,
    })

results = pd.DataFrame(rows)

print("=" * 80)
print("HYBRID ERROR ISOLATION")
print("=" * 80)

display(results.round(8))

print("\n" + "=" * 80)
print("AGGREGATE DIAGNOSTICS")
print("=" * 80)

for col in [
    "coverage_pct",
    "coef_mae_all",
    "coef_mae_quantized_only",
    "hybrid_recurrence_mae",
    "hybrid_recurrence_rmse",
    "hybrid_recurrence_max",
    "triton_intermediate_mae",
    "triton_intermediate_max",
    "triton_final_mae",
    "triton_final_max",
]:
    print(f"{col}: {results[col].mean():.10g}")

print("\nWorst hybrid recurrence layer:")
print(
    results.loc[
        results["hybrid_recurrence_max"].idxmax()
    ].to_string()
)

print("\nWorst Triton reference mismatch layer:")
print(
    results.loc[
        results["triton_final_max"].idxmax()
    ].to_string()
)

# Save results.
csv_path = os.path.join(OUT_DIR, "hybrid_error_isolation.csv")
results.to_csv(csv_path, index=False)

torch.save(
    all_intermediate,
    os.path.join(OUT_DIR, "hybrid_error_isolation_tensors.pt")
)

print("\nSaved:")
print(csv_path)
print(os.path.join(OUT_DIR, "hybrid_error_isolation_tensors.pt"))

HYBRID ERROR ISOLATION


,layer,coverage_pct,fallback_pct,dyadic2_pct,pot_pct,zero_pct,coef_mae_all,coef_mae_quantized_only,coef_max_quantized_only,hybrid_recurrence_mae,hybrid_recurrence_rmse,hybrid_recurrence_max,triton_intermediate_mae,triton_intermediate_max,triton_final_mae,triton_final_max,hybrid_final_mae,hybrid_final_max
0,0,95.833331,4.166667,0.000000,95.833333,0.000000,1.722700e-04,1.797600e-04,0.000526,7.570000e-06,0.000031,0.003542,1.054000e-05,0.015533,0.000000e+00,1.200000e-07,9.800000e-06,0.003542
1,1,79.166669,20.833333,14.583333,14.583333,50.000000,4.509540e-03,5.696260e-03,0.058020,8.262700e-04,0.007164,0.503998,2.611000e-05,0.007717,0.000000e+00,1.910000e-06,1.260620e-03,0.407390
2,2,87.500000,12.500000,53.125000,13.541667,20.833333,6.231720e-03,7.121960e-03,0.046557,4.060100e-04,0.002766,0.582973,1.427000e-05,0.007717,0.000000e+00,1.910000e-06,6.607700e-04,0.412075
3,3,70.833331,29.166666,4.166667,29.166667,37.500000,5.416200e-04,7.646400e-04,0.015387,2.712000e-04,0.003431,0.735947,6.162000e-05,0.031197,0.000000e+00,7.630000e-06,3.588000e-04,0.518684
4,4,91.666669,8.333334,65.625000,13.541667,12.500000,3.254150e-03,3.549980e-03,0.059642,1.462300e-03,0.008325,0.792351,7.189000e-05,0.029076,3.000000e-08,9.540000e-06,2.757090e-03,0.792351
5,5,93.750000,6.250000,21.875000,9.375000,62.500000,2.531550e-03,2.700320e-03,0.038528,5.814000e-04,0.004115,0.605705,4.716000e-05,0.015038,1.000000e-08,3.810000e-06,1.125080e-03,0.605705
6,6,84.375000,15.625000,43.750000,31.250000,9.375000,4.953710e-03,5.871060e-03,0.060937,7.200300e-04,0.004049,0.804142,3.426000e-05,0.012466,1.000000e-08,3.810000e-06,9.095300e-04,0.203983
7,7,56.250000,43.750000,3.125000,34.375000,18.750000,3.753400e-04,6.672700e-04,0.011138,2.984500e-04,0.004318,1.289810,1.190600e-04,0.030083,0.000000e+00,7.630000e-06,2.534800e-04,0.576660
8,8,35.416666,64.583331,0.000000,28.125000,7.291667,4.986500e-04,1.407960e-03,0.033932,2.452400e-04,0.005299,1.636654,4.570000e-05,0.030174,0.000000e+00,4.800000e-07,1.141000e-05,0.017796
9,9,76.041669,23.958333,3.125000,5.208333,67.708333,1.280600e-04,1.684100e-04,0.009845,1.612000e-05,0.000646,0.107979,3.679000e-05,0.031189,0.000000e+00,7.630000e-06,3.145000e-05,0.107979



AGGREGATE DIAGNOSTICS
coverage_pct: 76.69270821
coef_mae_all: 0.001914726945
coef_mae_quantized_only: 0.0022860048
hybrid_recurrence_mae: 0.0003012053268
hybrid_recurrence_rmse: 0.002734179534
hybrid_recurrence_max: 0.5265944561
triton_intermediate_mae: 3.504600396e-05
triton_intermediate_max: 0.01950232436
triton_final_mae: 3.024060495e-09
triton_final_max: 2.896607233e-06

Worst hybrid recurrence layer:
layer                      8.000000e+00
coverage_pct               3.541667e+01
fallback_pct               6.458333e+01
dyadic2_pct                0.000000e+00
pot_pct                    2.812500e+01
zero_pct                   7.291667e+00
coef_mae_all               4.986526e-04
coef_mae_quantized_only    1.407960e-03
coef_max_quantized_only    3.393241e-02
hybrid_recurrence_mae      2.452407e-04
hybrid_recurrence_rmse     5.299098e-03
hybrid_recurrence_max      1.636654e+00
triton_intermediate_mae    4.570406e-05
triton_intermediate_max    3.017426e-02
triton_final_mae           3.1

In [141]:
import torch
import pandas as pd
import numpy as np
import os

PATH = (
    "/kaggle/working/shiftscan_final/state_passing/"
    "hybrid_grid_pilot/error_isolation/"
    "hybrid_error_isolation_tensors.pt"
)

data = torch.load(PATH, map_location="cpu", weights_only=False)

candidates = []

for item in data:
    layer = item["layer"]

    ref = item["reference"].float()
    hybrid = item["hybrid"].float()

    err = (hybrid - ref).abs()

    flat_idx = int(err.argmax())
    c, h, d = np.unravel_index(flat_idx, tuple(err.shape))

    candidates.append({
        "layer": layer,
        "chunk": c,
        "head": h,
        "state_dim": d,
        "abs_error": float(err[c, h, d]),
        "reference_state": float(ref[c, h, d]),
        "hybrid_state": float(hybrid[c, h, d]),
        "true_decay": float(item["true_decay"][c, h]),
        "hybrid_decay": float(item["hybrid_decay"][c, h]),
        "mode": int(item["modes"][c, h]),
    })

df = pd.DataFrame(candidates).sort_values(
    "abs_error", ascending=False
)

print("=" * 75)
print("TOP 15 WORST RECURRENCE ERRORS")
print("=" * 75)
display(df.head(15))

print("\n" + "=" * 75)
print("WORST SINGLE LOCATION")
print("=" * 75)

worst = df.iloc[0]
print(worst.to_string())

layer = int(worst["layer"])
chunk = int(worst["chunk"])
head = int(worst["head"])
dim = int(worst["state_dim"])

item = data[layer]

print("\n" + "=" * 75)
print("ERROR TRAJECTORY FOR WORST STATE DIMENSION")
print("=" * 75)

ref = item["reference"][:, head, dim]
hybrid = item["hybrid"][:, head, dim]

trajectory = pd.DataFrame({
    "chunk": np.arange(len(ref)),
    "reference": ref.numpy(),
    "hybrid": hybrid.numpy(),
    "absolute_error": (hybrid - ref).abs().numpy(),
    "true_decay": item["true_decay"][:, head].numpy(),
    "hybrid_decay": item["hybrid_decay"][:, head].numpy(),
    "mode": item["modes"][:, head],
})

display(trajectory)

print("\nSaved diagnostic table:")
out = (
    "/kaggle/working/shiftscan_final/state_passing/"
    "hybrid_grid_pilot/error_isolation/"
    "worst_error_locations.csv"
)
df.to_csv(out, index=False)
print(out)

TOP 15 WORST RECURRENCE ERRORS


,layer,chunk,head,state_dim,abs_error,reference_state,hybrid_state,true_decay,hybrid_decay,mode
8,8,1,22,7115,1.636654,10.866468,9.229815,0.096432,0.062500,2
10,10,3,23,5313,1.585480,220.276367,221.861847,0.022681,0.031250,2
7,7,2,12,2825,1.289810,114.210388,112.920578,0.386138,0.375000,1
11,11,1,7,2342,1.247955,75.541199,74.293243,0.081096,0.062500,2
12,12,2,7,2876,0.854355,38.688095,37.833740,0.519126,0.500000,1
6,6,2,11,6202,0.804142,8.796223,7.992080,0.094904,0.062500,2
4,4,3,5,7938,0.792351,17.484230,16.691879,0.277107,0.250000,1
3,3,2,18,3284,0.735947,43.983555,43.247608,0.765387,0.750000,1
5,5,3,17,6537,0.605705,45.767647,46.373352,0.643667,0.656250,1
15,15,1,16,4360,0.586187,13.109150,12.522963,0.074482,0.062500,2



WORST SINGLE LOCATION
layer                 8.000000
chunk                 1.000000
head                 22.000000
state_dim          7115.000000
abs_error             1.636654
reference_state      10.866468
hybrid_state          9.229815
true_decay            0.096432
hybrid_decay          0.062500
mode                  2.000000

ERROR TRAJECTORY FOR WORST STATE DIMENSION


,chunk,reference,hybrid,absolute_error,true_decay,hybrid_decay,mode
0,0,48.232758,48.232758,0.000000,0.010978,0.007812,2
1,1,10.866468,9.229815,1.636654,0.096432,0.062500,2
2,2,15.000675,15.004859,0.004184,0.052701,0.062500,2
3,3,13.473792,13.465520,0.008272,0.031810,0.031250,2



Saved diagnostic table:
/kaggle/working/shiftscan_final/state_passing/hybrid_grid_pilot/error_isolation/worst_error_locations.csv


In [142]:
import os
import torch
import numpy as np
import pandas as pd

BASE = "/kaggle/working/shiftscan_final/state_passing"
RAW_PATH = os.path.join(BASE, "state_passing_raw_tensors.pt")

data = torch.load(RAW_PATH, map_location="cpu", weights_only=False)

FALLBACK_TOLERANCE = 1e-3

results = []

for layer_idx, record in enumerate(data):

    N = record["states"][0].float()
    true_decay = torch.exp(record["decay_log"][0].float()).T.contiguous()

    chunks, heads, dim = N.shape

    reference = torch.zeros((heads, dim), dtype=torch.float32)
    baseline = torch.zeros_like(reference)
    adaptive = torch.zeros_like(reference)

    baseline_errors = []
    adaptive_errors = []

    quantized_count = 0
    fallback_count = 0

    for c in range(chunks):

        a = true_decay[c]

        # Reference recurrence.
        reference = a[:, None] * reference + N[c]

        # Existing hybrid policy is reconstructed from the saved pilot.
        # The previous error-isolation tensor file contains these coefficients.
        item_path = os.path.join(
            BASE,
            "hybrid_grid_pilot",
            "error_isolation",
            "hybrid_error_isolation_tensors.pt"
        )

        if c == 0:
            saved = torch.load(
                item_path,
                map_location="cpu",
                weights_only=False
            )
            hybrid_coeff = saved[layer_idx]["hybrid_decay"]

        aq = hybrid_coeff[c].float()

        # Existing hybrid recurrence.
        baseline = aq[:, None] * baseline + N[c]

        # State-aware acceptance:
        # keep quantized coefficient only if its predicted local state
        # error is below the tolerance; otherwise use exact decay.
        local_error = (
            (aq - a).abs()[:, None] * adaptive.abs()
        )

        accept = local_error.amax(dim=1) <= FALLBACK_TOLERANCE

        chosen = torch.where(accept, aq, a)

        quantized_count += int(accept.sum())
        fallback_count += int((~accept).sum())

        adaptive = chosen[:, None] * adaptive + N[c]

        baseline_errors.append((baseline - reference).abs())
        adaptive_errors.append((adaptive - reference).abs())

    baseline_errors = torch.stack(baseline_errors)
    adaptive_errors = torch.stack(adaptive_errors)

    results.append({
        "layer": layer_idx,
        "baseline_mae": baseline_errors.mean().item(),
        "adaptive_mae": adaptive_errors.mean().item(),
        "baseline_rmse": baseline_errors.square().mean().sqrt().item(),
        "adaptive_rmse": adaptive_errors.square().mean().sqrt().item(),
        "baseline_max": baseline_errors.max().item(),
        "adaptive_max": adaptive_errors.max().item(),
        "adaptive_coverage_pct": (
            100 * quantized_count / (quantized_count + fallback_count)
        ),
        "adaptive_fallback_pct": (
            100 * fallback_count / (quantized_count + fallback_count)
        )
    })

df = pd.DataFrame(results)

print("=" * 75)
print("STATE-AWARE HYBRID QUANTIZATION")
print("=" * 75)

display(df.round(8))

print("\nAggregate results:")
for col in [
    "baseline_mae",
    "adaptive_mae",
    "baseline_rmse",
    "adaptive_rmse",
    "baseline_max",
    "adaptive_max",
    "adaptive_coverage_pct",
    "adaptive_fallback_pct"
]:
    print(f"{col}: {df[col].mean():.10g}")

OUT = os.path.join(
    BASE,
    "hybrid_grid_pilot",
    "error_isolation",
    "state_aware_results.csv"
)

df.to_csv(OUT, index=False)
print("\nSaved:", OUT)

STATE-AWARE HYBRID QUANTIZATION


,layer,baseline_mae,adaptive_mae,baseline_rmse,adaptive_rmse,baseline_max,adaptive_max,adaptive_coverage_pct,adaptive_fallback_pct
0,0,7.570000e-06,4.130000e-06,0.000031,1.687000e-05,0.003542,9.717900e-04,78.125000,21.875000
1,1,8.262700e-04,0.000000e+00,0.007164,0.000000e+00,0.503998,0.000000e+00,78.125000,21.875000
2,2,4.060100e-04,1.300000e-07,0.002766,2.090000e-06,0.582973,6.003400e-04,57.291667,42.708333
3,3,2.712000e-04,6.200000e-07,0.003431,7.820000e-06,0.735947,8.312500e-04,84.375000,15.625000
4,4,1.462300e-03,2.040000e-06,0.008325,1.379000e-05,0.792351,9.667300e-04,57.291667,42.708333
5,5,5.814000e-04,5.000000e-08,0.004115,1.180000e-06,0.605705,2.441400e-04,81.250000,18.750000
6,6,7.200300e-04,1.040000e-06,0.004049,7.020000e-06,0.804142,9.012200e-04,58.333333,41.666667
7,7,2.984500e-04,2.000000e-07,0.004318,2.550000e-06,1.289810,6.418200e-04,80.208333,19.791667
8,8,2.452400e-04,1.400000e-07,0.005299,2.490000e-06,1.636654,7.400500e-04,93.750000,6.250000
9,9,1.612000e-05,0.000000e+00,0.000646,0.000000e+00,0.107979,0.000000e+00,93.750000,6.250000



Aggregate results:
baseline_mae: 0.0003012053268
adaptive_mae: 6.260638663e-07
baseline_rmse: 0.002734179534
adaptive_rmse: 4.515112264e-06
baseline_max: 0.5265944561
adaptive_max: 0.0005711414851
adaptive_coverage_pct: 79.99131944
adaptive_fallback_pct: 20.00868056

Saved: /kaggle/working/shiftscan_final/state_passing/hybrid_grid_pilot/error_isolation/state_aware_results.csv


In [143]:
import os
import torch
import numpy as np
import pandas as pd

BASE = "/kaggle/working/shiftscan_final/state_passing"

RAW_PATH = os.path.join(
    BASE, "state_passing_raw_tensors.pt"
)

HYBRID_PATH = os.path.join(
    BASE,
    "hybrid_grid_pilot",
    "error_isolation",
    "hybrid_error_isolation_tensors.pt"
)

OUT_DIR = os.path.join(
    BASE,
    "hybrid_grid_pilot",
    "threshold_sweep"
)
os.makedirs(OUT_DIR, exist_ok=True)

raw = torch.load(
    RAW_PATH,
    map_location="cpu",
    weights_only=False
)

hybrid_data = torch.load(
    HYBRID_PATH,
    map_location="cpu",
    weights_only=False
)

THRESHOLDS = [
    1e-4,
    3e-4,
    1e-3,
    3e-3,
    1e-2
]

rows = []

for threshold in THRESHOLDS:

    layer_rows = []

    total_accepted = 0
    total_coefficients = 0

    global_errors = []

    for layer_idx, record in enumerate(raw):

        N = record["states"][0].float()
        true_decay = torch.exp(
            record["decay_log"][0].float()
        ).T.contiguous()

        aq = hybrid_data[layer_idx]["hybrid_decay"].float()

        chunks, heads, dim = N.shape

        reference = torch.zeros(
            (heads, dim), dtype=torch.float32
        )

        adaptive = torch.zeros_like(reference)

        accepted = 0
        total = chunks * heads

        for c in range(chunks):

            a = true_decay[c]
            q = aq[c]

            # Predicted local state error using current adaptive state.
            local_error = (
                (q - a).abs()[:, None] * adaptive.abs()
            )

            accept = local_error.amax(dim=1) <= threshold

            chosen = torch.where(accept, q, a)

            accepted += int(accept.sum())
            total_accepted += int(accept.sum())
            total_coefficients += heads

            reference = a[:, None] * reference + N[c]
            adaptive = chosen[:, None] * adaptive + N[c]

            error = (adaptive - reference).abs()
            global_errors.append(error.reshape(-1))

        final_error = (adaptive - reference).abs()

        layer_rows.append({
            "threshold": threshold,
            "layer": layer_idx,
            "coverage_pct": 100 * accepted / total,
            "fallback_pct": 100 * (total - accepted) / total,
            "final_mae": final_error.mean().item(),
            "final_rmse": final_error.square().mean().sqrt().item(),
            "final_max": final_error.max().item()
        })

    errors = torch.cat(global_errors)

    aggregate_coverage = (
        100 * total_accepted / total_coefficients
    )

    rows.append({
        "threshold": threshold,
        "coverage_pct": aggregate_coverage,
        "fallback_pct": 100 - aggregate_coverage,
        "recurrence_mae": errors.mean().item(),
        "recurrence_rmse": errors.square().mean().sqrt().item(),
        "recurrence_max": errors.max().item()
    })

    print(
        f"Threshold={threshold:g} | "
        f"Coverage={aggregate_coverage:.2f}% | "
        f"MAE={errors.mean().item():.8g} | "
        f"RMSE={errors.square().mean().sqrt().item():.8g} | "
        f"Max={errors.max().item():.8g}"
    )

summary = pd.DataFrame(rows)
layer_df = pd.DataFrame(layer_rows)

# Recompute per-layer rows for every threshold.
# Store a complete table in the next loop's results instead.
summary.to_csv(
    os.path.join(OUT_DIR, "threshold_summary.csv"),
    index=False
)

print("\n" + "=" * 75)
print("THRESHOLD SWEEP SUMMARY")
print("=" * 75)

display(summary)

print("\nSaved:", OUT_DIR)

Threshold=0.0001 | Coverage=73.61% | MAE=3.2780402e-08 | RMSE=4.9050044e-07 | Max=9.7170472e-05
Threshold=0.0003 | Coverage=75.95% | MAE=1.5622956e-07 | RMSE=2.0413065e-06 | Max=0.00045871735
Threshold=0.001 | Coverage=79.99% | MAE=6.2606381e-07 | RMSE=6.4198211e-06 | Max=0.00099024177
Threshold=0.003 | Coverage=84.55% | MAE=2.4829394e-06 | RMSE=2.5462627e-05 | Max=0.0038261414
Threshold=0.01 | Coverage=88.37% | MAE=8.9892465e-06 | RMSE=8.2473191e-05 | Max=0.010036886

THRESHOLD SWEEP SUMMARY


,threshold,coverage_pct,fallback_pct,recurrence_mae,recurrence_rmse,recurrence_max
0,0.0001,73.611111,26.388889,3.278040e-08,4.905004e-07,0.000097
1,0.0003,75.954861,24.045139,1.562296e-07,2.041307e-06,0.000459
2,0.0010,79.991319,20.008681,6.260638e-07,6.419821e-06,0.000990
3,0.0030,84.548611,15.451389,2.482939e-06,2.546263e-05,0.003826
4,0.0100,88.368056,11.631944,8.989246e-06,8.247319e-05,0.010037



Saved: /kaggle/working/shiftscan_final/state_passing/hybrid_grid_pilot/threshold_sweep


In [144]:

import os
import json
import torch
import pandas as pd
from datasets import load_dataset
import importlib

# ---------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------

OUT_DIR = (
    "/kaggle/working/shiftscan_final/"
    "state_passing/multisequence_validation"
)

os.makedirs(OUT_DIR, exist_ok=True)

N_DOCS = 10
SEQ_LENGTHS = [1024, 2048]
SKIP_QUALIFYING_DOCS = 100
MAX_STREAM_DOCS = 20000

DEVICE = "cuda"

assert "model" in globals(), "Load the Mamba-2 model first."
assert "tokenizer" in globals(), "Load the tokenizer first."

model.eval()

# Ensure the fallback Triton SSD path is used.
for layer in model.backbone.layers:
    if hasattr(layer.mixer, "use_mem_eff_path"):
        layer.mixer.use_mem_eff_path = False

# ---------------------------------------------------------
# 1. COLLECT NEW FINEWEB-EDU DOCUMENTS
# ---------------------------------------------------------

print("Loading FineWeb-Edu stream...")

stream = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="sample-10BT",
    split="train",
    streaming=True
)

documents = []
qualifying_seen = 0
stream_seen = 0

for item in stream:

    stream_seen += 1

    if stream_seen > MAX_STREAM_DOCS:
        break

    text = item.get("text", "").strip()

    if len(text) <= 100:
        continue

    if qualifying_seen < SKIP_QUALIFYING_DOCS:
        qualifying_seen += 1
        continue

    encoded = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False
    )

    ids = encoded["input_ids"]

    # Need enough tokens for both sequence lengths.
    if len(ids) < 2048:
        continue

    if max(ids) >= model.config.vocab_size:
        raise ValueError(
            f"Token ID exceeds model vocabulary: {max(ids)}"
        )

    documents.append({
        "doc_id": len(documents),
        "stream_index": stream_seen,
        "text_chars": len(text),
        "token_count": len(ids),
        "input_ids": ids[:2048]
    })

    print(
        f"Collected {len(documents)}/{N_DOCS}: "
        f"{len(ids)} tokens"
    )

    if len(documents) >= N_DOCS:
        break

if len(documents) < N_DOCS:
    raise RuntimeError(
        f"Only collected {len(documents)} documents. "
        "Increase MAX_STREAM_DOCS and rerun."
    )

metadata = pd.DataFrame([
    {
        "doc_id": d["doc_id"],
        "stream_index": d["stream_index"],
        "text_chars": d["text_chars"],
        "token_count": d["token_count"]
    }
    for d in documents
])

metadata.to_csv(
    os.path.join(OUT_DIR, "document_metadata.csv"),
    index=False
)

print("\nCollected documents:")
display(metadata)

# ---------------------------------------------------------
# 2. PATCH THE ACTUAL STATE-PASSING CALL
# ---------------------------------------------------------

ssd_combined = importlib.import_module(
    "mamba_ssm.ops.triton.ssd_combined"
)

original_state_passing = ssd_combined._state_passing_fwd

capture_records = []
active_doc = None
active_seq_len = None

def capture_state_passing(*args, **kwargs):

    states = kwargs.get(
        "states",
        args[0] if len(args) > 0 else None
    )

    decay_log = kwargs.get(
        "dA_chunk_cumsum",
        args[1] if len(args) > 1 else None
    )

    initial_states = kwargs.get(
        "initial_states",
        args[2] if len(args) > 2 else None
    )

    seq_idx = kwargs.get(
        "seq_idx",
        args[3] if len(args) > 3 else None
    )

    chunk_size = kwargs.get(
        "chunk_size",
        args[4] if len(args) > 4 else None
    )

    if states is None or decay_log is None:
        raise RuntimeError(
            "Could not locate state-passing input tensors."
        )

    # Preserve original execution.
    result = original_state_passing(*args, **kwargs)

    # Capture the exact tensors passed to the real Triton path.
    capture_records.append({
        "layer": len(capture_records),
        "doc_id": active_doc,
        "seq_len": active_seq_len,
        "states": states.detach().to(
            device="cpu",
            dtype=torch.float32
        ).contiguous(),
        "decay_log": decay_log.detach().to(
            device="cpu",
            dtype=torch.float32
        ).contiguous(),
        "chunk_size": chunk_size,
        "has_initial_states": initial_states is not None,
        "has_seq_idx": seq_idx is not None
    })

    return result


# ---------------------------------------------------------
# 3. RUN REAL MODEL AND SAVE EACH CAPTURE
# ---------------------------------------------------------

ssd_combined._state_passing_fwd = capture_state_passing

manifest = []

try:

    for seq_len in SEQ_LENGTHS:

        for doc in documents:

            active_doc = doc["doc_id"]
            active_seq_len = seq_len
            capture_records.clear()

            input_ids = torch.tensor(
                doc["input_ids"][:seq_len],
                dtype=torch.long
            ).unsqueeze(0).to(DEVICE)

            print(
                f"\nRunning doc={active_doc}, "
                f"sequence_length={seq_len}"
            )

            with torch.inference_mode():
                output = model(input_ids)

            torch.cuda.synchronize()

            if len(capture_records) != 24:
                raise RuntimeError(
                    f"Expected 24 layer captures, "
                    f"received {len(capture_records)}."
                )

            # Verify expected chunk count.
            expected_chunks = seq_len // 256

            for record in capture_records:

                actual_chunks = record["states"].shape[1]

                if actual_chunks != expected_chunks:
                    raise RuntimeError(
                        f"Unexpected chunk count: "
                        f"{actual_chunks}, expected "
                        f"{expected_chunks}"
                    )

            filename = (
                f"doc_{active_doc:03d}_"
                f"seq_{seq_len}.pt"
            )

            save_path = os.path.join(
                OUT_DIR,
                filename
            )

            torch.save({
                "metadata": {
                    "doc_id": active_doc,
                    "seq_len": seq_len,
                    "expected_chunks": expected_chunks,
                    "n_layers": len(capture_records)
                },
                "records": list(capture_records)
            }, save_path)

            manifest.append({
                "doc_id": active_doc,
                "seq_len": seq_len,
                "layers_captured": len(capture_records),
                "chunks_per_layer": expected_chunks,
                "file": filename
            })

            print(
                f"Saved {filename} | "
                f"{len(capture_records)} layers | "
                f"{expected_chunks} chunks"
            )

            del input_ids, output
            torch.cuda.empty_cache()

finally:

    # Always restore original function.
    ssd_combined._state_passing_fwd = original_state_passing

# ---------------------------------------------------------
# 4. SAVE MANIFEST AND VALIDATE
# ---------------------------------------------------------

manifest_df = pd.DataFrame(manifest)

manifest_df.to_csv(
    os.path.join(OUT_DIR, "capture_manifest.csv"),
    index=False
)

with open(
    os.path.join(OUT_DIR, "capture_config.json"),
    "w"
) as f:
    json.dump({
        "model": "state-spaces/mamba2-130m",
        "dataset": "HuggingFaceFW/fineweb-edu/sample-10BT",
        "n_documents": N_DOCS,
        "sequence_lengths": SEQ_LENGTHS,
        "skipped_qualifying_documents": SKIP_QUALIFYING_DOCS,
        "chunk_size": 256,
        "thresholds_for_next_experiment": [
            0.0003, 0.001, 0.003
        ],
        "capture_type": "actual_mamba2_state_passing_inputs"
    }, f, indent=2)

assert len(manifest_df) == N_DOCS * len(SEQ_LENGTHS)

print("\n" + "=" * 70)
print("MULTI-SEQUENCE CAPTURE COMPLETE")
print("=" * 70)

display(manifest_df)

print("\nOutput directory:", OUT_DIR)
print("Total captures:", len(manifest_df))
print("Total layer records:", len(manifest_df) * 24)


Loading FineWeb-Edu stream...


Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Collected 1/10: 3631 tokens
Collected 2/10: 2726 tokens
Collected 3/10: 3683 tokens
Collected 4/10: 2143 tokens
Collected 5/10: 2306 tokens
Collected 6/10: 14068 tokens
Collected 7/10: 2512 tokens
Collected 8/10: 2939 tokens
Collected 9/10: 3750 tokens
Collected 10/10: 2154 tokens

Collected documents:


,doc_id,stream_index,text_chars,token_count
0,0,119,17227,3631
1,1,142,11898,2726
2,2,148,18639,3683
3,3,169,8027,2143
4,4,192,10348,2306
5,5,195,65092,14068
6,6,201,10554,2512
7,7,212,12970,2939
8,8,218,19335,3750
9,9,219,10878,2154



Running doc=0, sequence_length=1024
Saved doc_000_seq_1024.pt | 24 layers | 4 chunks

Running doc=1, sequence_length=1024
Saved doc_001_seq_1024.pt | 24 layers | 4 chunks

Running doc=2, sequence_length=1024
Saved doc_002_seq_1024.pt | 24 layers | 4 chunks

Running doc=3, sequence_length=1024
Saved doc_003_seq_1024.pt | 24 layers | 4 chunks

Running doc=4, sequence_length=1024
Saved doc_004_seq_1024.pt | 24 layers | 4 chunks

Running doc=5, sequence_length=1024
Saved doc_005_seq_1024.pt | 24 layers | 4 chunks

Running doc=6, sequence_length=1024
Saved doc_006_seq_1024.pt | 24 layers | 4 chunks

Running doc=7, sequence_length=1024
Saved doc_007_seq_1024.pt | 24 layers | 4 chunks

Running doc=8, sequence_length=1024
Saved doc_008_seq_1024.pt | 24 layers | 4 chunks

Running doc=9, sequence_length=1024
Saved doc_009_seq_1024.pt | 24 layers | 4 chunks

Running doc=0, sequence_length=2048
Saved doc_000_seq_2048.pt | 24 layers | 8 chunks

Running doc=1, sequence_length=2048
Saved doc_001_seq

,doc_id,seq_len,layers_captured,chunks_per_layer,file
0,0,1024,24,4,doc_000_seq_1024.pt
1,1,1024,24,4,doc_001_seq_1024.pt
2,2,1024,24,4,doc_002_seq_1024.pt
3,3,1024,24,4,doc_003_seq_1024.pt
4,4,1024,24,4,doc_004_seq_1024.pt
5,5,1024,24,4,doc_005_seq_1024.pt
6,6,1024,24,4,doc_006_seq_1024.pt
7,7,1024,24,4,doc_007_seq_1024.pt
8,8,1024,24,4,doc_008_seq_1024.pt
9,9,1024,24,4,doc_009_seq_1024.pt



Output directory: /kaggle/working/shiftscan_final/state_passing/multisequence_validation
Total captures: 20
Total layer records: 480


In [145]:

import os
import ast
import json
import torch
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# PATHS AND SETTINGS
# ---------------------------------------------------------

BASE = "/kaggle/working/shiftscan_final"

CAPTURE_DIR = os.path.join(
    BASE,
    "state_passing",
    "multisequence_validation"
)

GRID_PATH = os.path.join(
    BASE,
    "canonical_dyadic_grid_definitions.csv"
)

OUT_DIR = os.path.join(
    CAPTURE_DIR,
    "adaptive_evaluation"
)

os.makedirs(OUT_DIR, exist_ok=True)

THRESHOLDS = [0.0003, 0.001, 0.003]

POW2_MIN_K = 4
POW2_MAX_K = 30
MIN_PO2_DECAY = 2.0 ** (-POW2_MAX_K)

# ---------------------------------------------------------
# 1. LOAD AND VALIDATE CANONICAL DYADIC GRID
# ---------------------------------------------------------

grid_df = pd.read_csv(GRID_PATH)

required_columns = {"grid", "decay"}

if not required_columns.issubset(grid_df.columns):
    raise ValueError(
        f"Canonical grid missing columns: {required_columns}"
    )

grid_df = grid_df[grid_df["grid"] == "K1_14_T3"].copy()

dyadic_values = np.sort(
    grid_df["decay"].astype(float).unique()
)

if len(dyadic_values) != 467:
    raise ValueError(
        f"Expected 467 unique K1_14_T3 coefficients; "
        f"found {len(dyadic_values)}."
    )

dyadic_grid = torch.tensor(
    dyadic_values,
    dtype=torch.float32
)

dyadic_log_distance = torch.log(
    1.0 - dyadic_grid
)

dyadic_min = float(dyadic_grid.min())
dyadic_max = float(dyadic_grid.max())

print("Canonical K1_14_T3 grid loaded.")
print("Unique coefficients:", len(dyadic_values))
print("Decay range:", dyadic_min, "to", dyadic_max)

# ---------------------------------------------------------
# 2. HYBRID COEFFICIENT SELECTION
#
# Mode 1: K1_14_T3 three-term Dyadic2
# Mode 2: Direct power of two
# Mode 3: Exact zero
# Mode 0: Exact fallback
# ---------------------------------------------------------

def build_hybrid_coefficients(true_decay):

    a = true_decay.float()

    q = a.clone()
    mode = torch.zeros_like(a, dtype=torch.int8)

    finite = torch.isfinite(a)
    positive = finite & (a > 0)
    zero = finite & (a == 0)

    # Three-term Dyadic2 region.
    dyadic_mask = (
        positive
        & (a >= dyadic_min)
        & (a <= dyadic_max)
    )

    if dyadic_mask.any():

        d = 1.0 - a[dyadic_mask]

        # Same logarithmic distance objective used in
        # the earlier hybrid coefficient experiments.
        distance = torch.abs(
            torch.log(d)[:, None]
            - dyadic_log_distance[None, :]
        )

        indices = distance.argmin(dim=1)

        q[dyadic_mask] = dyadic_grid[indices]
        mode[dyadic_mask] = 1

    # Direct power-of-two region.
    pot_mask = (
        positive
        & (a < dyadic_min)
        & (a >= MIN_PO2_DECAY)
    )

    if pot_mask.any():

        values = a[pot_mask]

        k = torch.round(
            -torch.log2(values)
        ).to(torch.int64)

        k = k.clamp(
            POW2_MIN_K,
            POW2_MAX_K
        )

        q[pot_mask] = torch.pow(
            torch.tensor(2.0),
            -k.float()
        )

        mode[pot_mask] = 2

    # Exact zero representation.
    q[zero] = 0.0
    mode[zero] = 3

    # Non-finite, too-small positive, or out-of-grid
    # values remain mode 0 and use exact fallback.
    return q, mode


# ---------------------------------------------------------
# 3. EVALUATE ONE LAYER RECORD
# ---------------------------------------------------------

def evaluate_layer_record(record, threshold):

    states = record["states"].float()
    decay_log = record["decay_log"].float()

    # Expected captured state shape:
    # [batch=1, chunks, heads, state_dim]
    if states.ndim != 4 or states.shape[0] != 1:
        raise ValueError(
            f"Unexpected states shape: {tuple(states.shape)}"
        )

    # Expected cumulative decay shape:
    # [batch=1, heads, chunks]
    if decay_log.ndim != 3 or decay_log.shape[0] != 1:
        raise ValueError(
            f"Unexpected decay_log shape: {tuple(decay_log.shape)}"
        )

    if record["has_initial_states"]:
        raise NotImplementedError(
            "This evaluation expects no initial_states."
        )

    if record["has_seq_idx"]:
        raise NotImplementedError(
            "This evaluation expects no seq_idx."
        )

    N = states[0].contiguous()
    a = torch.exp(decay_log[0]).T.contiguous()

    chunks, heads, dim = N.shape

    if a.shape != (chunks, heads):
        raise ValueError(
            f"Decay shape {tuple(a.shape)} does not match "
            f"state chunks/heads {(chunks, heads)}"
        )

    q, mode = build_hybrid_coefficients(a)

    # Exact and adaptive FP32 recurrence states.
    reference = torch.zeros(
        (heads, dim), dtype=torch.float32
    )

    adaptive = torch.zeros_like(reference)

    trajectory_errors = []

    accepted_count = 0
    total_count = chunks * heads

    for c in range(chunks):

        true_a = a[c]
        candidate_a = q[c]

        candidate_available = mode[c] != 0

        # Predicted local update error:
        # |aq - a| * max_dimension |adaptive_state|
        local_error = (
            (candidate_a - true_a).abs()[:, None]
            * adaptive.abs()
        )

        predicted_head_error = local_error.amax(dim=1)

        accept = (
            candidate_available
            & (predicted_head_error <= threshold)
        )

        accepted_count += int(accept.sum().item())

        chosen_a = torch.where(
            accept,
            candidate_a,
            true_a
        )

        # Exact reference update.
        reference = (
            true_a[:, None] * reference
            + N[c]
        )

        # Adaptive update.
        adaptive = (
            chosen_a[:, None] * adaptive
            + N[c]
        )

        trajectory_errors.append(
            (adaptive - reference).abs().reshape(-1)
        )

    trajectory_errors = torch.cat(trajectory_errors)

    final_errors = (
        adaptive - reference
    ).abs().reshape(-1)

    return {
        "chunks": chunks,
        "heads": heads,
        "state_dim": dim,
        "total_coefficients": total_count,
        "accepted_coefficients": accepted_count,
        "trajectory_mae": trajectory_errors.mean().item(),
        "trajectory_rmse": trajectory_errors.square().mean().sqrt().item(),
        "trajectory_max": trajectory_errors.max().item(),
        "final_mae": final_errors.mean().item(),
        "final_rmse": final_errors.square().mean().sqrt().item(),
        "final_max": final_errors.max().item()
    }


# ---------------------------------------------------------
# 4. RUN ALL 20 CAPTURES
# ---------------------------------------------------------

manifest_path = os.path.join(
    CAPTURE_DIR,
    "capture_manifest.csv"
)

manifest = pd.read_csv(manifest_path)

layer_rows = []
document_rows = []
aggregate_rows = []

for seq_len in sorted(manifest["seq_len"].unique()):

    subset = manifest[
        manifest["seq_len"] == seq_len
    ]

    for threshold in THRESHOLDS:

        print(
            f"\nEvaluating sequence length={seq_len}, "
            f"threshold={threshold}"
        )

        total_accepted = 0
        total_coefficients = 0

        all_trajectory_errors = []
        all_final_errors = []

        # Process each document.
        for _, item in subset.iterrows():

            doc_id = int(item["doc_id"])
            filename = item["file"]

            path = os.path.join(
                CAPTURE_DIR,
                filename
            )

            saved = torch.load(
                path,
                map_location="cpu",
                weights_only=False
            )

            records = saved["records"]

            if len(records) != 24:
                raise ValueError(
                    f"{filename}: expected 24 layers, "
                    f"found {len(records)}"
                )

            doc_accepted = 0
            doc_total = 0

            doc_trajectory = []
            doc_final = []

            for layer_idx, record in enumerate(records):

                result = evaluate_layer_record(
                    record,
                    threshold
                )

                accepted = result["accepted_coefficients"]
                total = result["total_coefficients"]

                total_accepted += accepted
                total_coefficients += total

                doc_accepted += accepted
                doc_total += total

                layer_rows.append({
                    "doc_id": doc_id,
                    "seq_len": int(seq_len),
                    "layer": layer_idx,
                    "threshold": threshold,
                    **result,
                    "coverage_pct": 100 * accepted / total,
                    "fallback_pct": 100 * (total - accepted) / total
                })

                doc_trajectory.append(
                    result["trajectory_mae"]
                )

                doc_final.append(
                    result["final_mae"]
                )

            document_rows.append({
                "doc_id": doc_id,
                "seq_len": int(seq_len),
                "threshold": threshold,
                "coverage_pct": (
                    100 * doc_accepted / doc_total
                ),
                "fallback_pct": (
                    100 * (doc_total - doc_accepted) / doc_total
                ),
                "mean_layer_trajectory_mae": np.mean(doc_trajectory),
                "mean_layer_final_mae": np.mean(doc_final)
            })

        # Aggregate layer-level metrics.
        current_layers = [
            r for r in layer_rows
            if r["seq_len"] == seq_len
            and r["threshold"] == threshold
        ]

        total_trajectory_elements = 0
        total_final_elements = 0

        # Reconstruct weighted aggregate metrics from
        # per-layer MAE/RMSE and equal tensor sizes.
        trajectory_mae = np.mean([
            r["trajectory_mae"] for r in current_layers
        ])

        trajectory_mse = np.mean([
            r["trajectory_rmse"] ** 2
            for r in current_layers
        ])

        final_mae = np.mean([
            r["final_mae"] for r in current_layers
        ])

        final_mse = np.mean([
            r["final_rmse"] ** 2
            for r in current_layers
        ])

        trajectory_max = max(
            r["trajectory_max"] for r in current_layers
        )

        final_max = max(
            r["final_max"] for r in current_layers
        )

        coverage = (
            100 * total_accepted / total_coefficients
        )

        aggregate_rows.append({
            "seq_len": int(seq_len),
            "threshold": threshold,
            "documents": len(subset),
            "layers": len(current_layers),
            "coverage_pct": coverage,
            "fallback_pct": 100 - coverage,
            "trajectory_mae": trajectory_mae,
            "trajectory_rmse": np.sqrt(trajectory_mse),
            "trajectory_max": trajectory_max,
            "final_mae": final_mae,
            "final_rmse": np.sqrt(final_mse),
            "final_max": final_max
        })

        print(
            f"Coverage={coverage:.2f}% | "
            f"Trajectory MAE={trajectory_mae:.8g} | "
            f"Trajectory max={trajectory_max:.8g} | "
            f"Final MAE={final_mae:.8g} | "
            f"Final max={final_max:.8g}"
        )

# ---------------------------------------------------------
# 5. SAVE RESULTS
# ---------------------------------------------------------

layer_df = pd.DataFrame(layer_rows)
document_df = pd.DataFrame(document_rows)
aggregate_df = pd.DataFrame(aggregate_rows)

layer_df.to_csv(
    os.path.join(OUT_DIR, "per_layer_results.csv"),
    index=False
)

document_df.to_csv(
    os.path.join(OUT_DIR, "per_document_results.csv"),
    index=False
)

aggregate_df.to_csv(
    os.path.join(OUT_DIR, "aggregate_results.csv"),
    index=False
)

with open(
    os.path.join(OUT_DIR, "evaluation_config.json"),
    "w"
) as f:

    json.dump({
        "thresholds": THRESHOLDS,
        "dyadic_grid": "K1_14_T3",
        "dyadic_unique_coefficients": len(dyadic_values),
        "direct_pot_exponents": [
            POW2_MIN_K,
            POW2_MAX_K
        ],
        "minimum_direct_pot_decay": MIN_PO2_DECAY,
        "metric_definitions": {
            "trajectory": "all post-update chunk states",
            "final": "state after final chunk",
            "coverage": "accepted chunk-head coefficients / all chunk-head coefficients"
        }
    }, f, indent=2)

print("\n" + "=" * 80)
print("MULTI-SEQUENCE ADAPTIVE EVALUATION COMPLETE")
print("=" * 80)

display(
    aggregate_df.sort_values(
        ["seq_len", "threshold"]
    ).reset_index(drop=True)
)

print("\nFiles saved to:", OUT_DIR)


Canonical K1_14_T3 grid loaded.
Unique coefficients: 467
Decay range: 0.125 to 0.99981689453125

Evaluating sequence length=1024, threshold=0.0003
Coverage=51.20% | Trajectory MAE=1.1930935e-07 | Trajectory max=0.00043487549 | Final MAE=1.9204344e-07 | Final max=0.00043487549

Evaluating sequence length=1024, threshold=0.001
Coverage=55.27% | Trajectory MAE=6.210804e-07 | Trajectory max=0.0016630292 | Final MAE=9.4032838e-07 | Final max=0.0016630292

Evaluating sequence length=1024, threshold=0.003
Coverage=59.26% | Trajectory MAE=2.1367205e-06 | Trajectory max=0.0051236153 | Final MAE=3.2481224e-06 | Final max=0.0046668053

Evaluating sequence length=2048, threshold=0.0003
Coverage=46.46% | Trajectory MAE=1.5923168e-07 | Trajectory max=0.00052428246 | Final MAE=1.9325081e-07 | Final max=0.00052428246

Evaluating sequence length=2048, threshold=0.001
Coverage=51.16% | Trajectory MAE=8.4911875e-07 | Trajectory max=0.0025274754 | Final MAE=1.1744761e-06 | Final max=0.0025274754

Evaluati

,seq_len,threshold,documents,layers,coverage_pct,fallback_pct,trajectory_mae,trajectory_rmse,trajectory_max,final_mae,final_rmse,final_max
0,1024,0.0003,10,240,51.202257,48.797743,1.193094e-07,0.000002,0.000435,1.920434e-07,0.000002,0.000435
1,1024,0.0010,10,240,55.269097,44.730903,6.210804e-07,0.000006,0.001663,9.403284e-07,0.000008,0.001663
2,1024,0.0030,10,240,59.262153,40.737847,2.136721e-06,0.000020,0.005124,3.248122e-06,0.000025,0.004667
3,2048,0.0003,10,240,46.464844,53.535156,1.592317e-07,0.000002,0.000524,1.932508e-07,0.000002,0.000524
4,2048,0.0010,10,240,51.161024,48.838976,8.491188e-07,0.000008,0.002527,1.174476e-06,0.000010,0.002527
5,2048,0.0030,10,240,55.928819,44.071181,3.034509e-06,0.000025,0.005398,4.019183e-06,0.000030,0.005398



Files saved to: /kaggle/working/shiftscan_final/state_passing/multisequence_validation/adaptive_evaluation


In [146]:

import os
import torch
import numpy as np
import pandas as pd

BASE = "/kaggle/working/shiftscan_final"
CAPTURE_DIR = os.path.join(
    BASE, "state_passing", "multisequence_validation"
)
GRID_PATH = os.path.join(
    BASE, "canonical_dyadic_grid_definitions.csv"
)
OUT_DIR = os.path.join(
    CAPTURE_DIR, "fixed_point_range_audit"
)
os.makedirs(OUT_DIR, exist_ok=True)

THRESHOLD = 0.001
FRACTION_BITS = [16, 18, 20, 22, 24]
INT32_MAX = 2**31 - 1
SAFETY_FACTOR = 2.0

# ---------------------------------------------------------
# LOAD CANONICAL GRID
# ---------------------------------------------------------

grid_df = pd.read_csv(GRID_PATH)
grid_df = grid_df[grid_df["grid"] == "K1_14_T3"]

grid_values = torch.tensor(
    np.sort(grid_df["decay"].unique()),
    dtype=torch.float32
)

grid_log_distance = torch.log(1.0 - grid_values)

a_min = float(grid_values.min())
a_max = float(grid_values.max())

# ---------------------------------------------------------
# RECREATE THE HYBRID COEFFICIENT POLICY
# ---------------------------------------------------------

def build_hybrid(a):

    q = a.clone()
    mode = torch.zeros_like(a, dtype=torch.int8)

    finite = torch.isfinite(a)
    positive = finite & (a > 0)
    zero = finite & (a == 0)

    # Three-term Dyadic2.
    mask = positive & (a >= a_min) & (a <= a_max)

    if mask.any():
        d = 1.0 - a[mask]
        dist = torch.abs(
            torch.log(d)[:, None]
            - grid_log_distance[None, :]
        )
        idx = dist.argmin(dim=1)
        q[mask] = grid_values[idx]
        mode[mask] = 1

    # Direct power-of-two.
    pot_min = 2.0 ** -30
    pot_mask = positive & (a < a_min) & (a >= pot_min)

    if pot_mask.any():
        k = torch.round(
            -torch.log2(a[pot_mask])
        ).to(torch.int64).clamp(4, 30)

        q[pot_mask] = torch.pow(
            torch.tensor(2.0),
            -k.float()
        )
        mode[pot_mask] = 2

    # Exact zero.
    q[zero] = 0.0
    mode[zero] = 3

    return q, mode


# ---------------------------------------------------------
# AUDIT EVERY DOCUMENT AND LAYER
# ---------------------------------------------------------

manifest = pd.read_csv(
    os.path.join(CAPTURE_DIR, "capture_manifest.csv")
)

rows = []

for _, item in manifest.iterrows():

    doc_id = int(item["doc_id"])
    seq_len = int(item["seq_len"])
    path = os.path.join(CAPTURE_DIR, item["file"])

    saved = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    for layer_idx, record in enumerate(saved["records"]):

        N = record["states"][0].float()
        a = torch.exp(
            record["decay_log"][0].float()
        ).T.contiguous()

        q, mode = build_hybrid(a)

        chunks, heads, dim = N.shape

        ref_state = torch.zeros(
            (heads, dim), dtype=torch.float32
        )
        adaptive_state = torch.zeros_like(ref_state)

        max_reference_state = 0.0
        max_adaptive_state = 0.0
        max_input_contribution = float(N.abs().max())

        accepted = 0
        total = chunks * heads

        for c in range(chunks):

            local_error = (
                (q[c] - a[c]).abs()[:, None]
                * adaptive_state.abs()
            )

            accept = (
                (mode[c] != 0)
                & (local_error.amax(dim=1) <= THRESHOLD)
            )

            accepted += int(accept.sum())

            chosen = torch.where(
                accept, q[c], a[c]
            )

            ref_state = (
                a[c, :, None] * ref_state + N[c]
            )

            adaptive_state = (
                chosen[:, None] * adaptive_state + N[c]
            )

            max_reference_state = max(
                max_reference_state,
                float(ref_state.abs().max())
            )

            max_adaptive_state = max(
                max_adaptive_state,
                float(adaptive_state.abs().max())
            )

        rows.append({
            "doc_id": doc_id,
            "seq_len": seq_len,
            "layer": layer_idx,
            "chunks": chunks,
            "heads": heads,
            "state_dim": dim,
            "max_abs_input": max_input_contribution,
            "max_abs_reference_state": max_reference_state,
            "max_abs_adaptive_state": max_adaptive_state,
            "coverage_pct": 100 * accepted / total
        })

audit = pd.DataFrame(rows)

audit.to_csv(
    os.path.join(OUT_DIR, "per_layer_range_audit.csv"),
    index=False
)

# ---------------------------------------------------------
# AGGREGATE BY SEQUENCE LENGTH
# ---------------------------------------------------------

summary_rows = []

for seq_len, group in audit.groupby("seq_len"):

    max_input = group["max_abs_input"].max()
    max_ref = group["max_abs_reference_state"].max()
    max_adaptive = group["max_abs_adaptive_state"].max()

    required_range = SAFETY_FACTOR * max(
        max_input, max_ref, max_adaptive
    )

    row = {
        "seq_len": int(seq_len),
        "documents": group["doc_id"].nunique(),
        "layer_records": len(group),
        "max_abs_input": max_input,
        "max_abs_reference_state": max_ref,
        "max_abs_adaptive_state": max_adaptive,
        "required_real_range_with_2x_margin": required_range
    }

    for F in FRACTION_BITS:

        available_real_range = INT32_MAX / (2**F)

        row[f"F{F}_real_range"] = available_real_range
        row[f"F{F}_safe_2x"] = (
            available_real_range >= required_range
        )

    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)

# Overall maximum across both sequence lengths.
overall = {
    "seq_len": "ALL",
    "documents": audit["doc_id"].nunique(),
    "layer_records": len(audit),
    "max_abs_input": audit["max_abs_input"].max(),
    "max_abs_reference_state": audit[
        "max_abs_reference_state"
    ].max(),
    "max_abs_adaptive_state": audit[
        "max_abs_adaptive_state"
    ].max()
}

overall_required = SAFETY_FACTOR * max(
    overall["max_abs_input"],
    overall["max_abs_reference_state"],
    overall["max_abs_adaptive_state"]
)

overall["required_real_range_with_2x_margin"] = overall_required

for F in FRACTION_BITS:
    available = INT32_MAX / (2**F)
    overall[f"F{F}_real_range"] = available
    overall[f"F{F}_safe_2x"] = available >= overall_required

summary = pd.concat(
    [summary, pd.DataFrame([overall])],
    ignore_index=True
)

summary.to_csv(
    os.path.join(OUT_DIR, "fixed_point_range_summary.csv"),
    index=False
)

print("=" * 80)
print("FIXED-POINT RANGE AUDIT")
print("=" * 80)

display(summary)

print("\nLargest individual layer records:")
display(
    audit.sort_values(
        "max_abs_adaptive_state",
        ascending=False
    ).head(15)
)

print("\nSaved:", OUT_DIR)


FIXED-POINT RANGE AUDIT


,seq_len,documents,layer_records,max_abs_input,max_abs_reference_state,max_abs_adaptive_state,required_real_range_with_2x_margin,F16_real_range,F16_safe_2x,F18_real_range,F18_safe_2x,F20_real_range,F20_safe_2x,F22_real_range,F22_safe_2x,F24_real_range,F24_safe_2x
0,1024,10,240,361.130737,361.130737,361.130737,722.261475,32767.999985,True,8191.999996,True,2047.999999,True,512.0,False,128.0,False
1,2048,10,240,458.807892,458.807892,458.807892,917.615784,32767.999985,True,8191.999996,True,2047.999999,True,512.0,False,128.0,False
2,ALL,10,480,458.807892,458.807892,458.807892,917.615784,32767.999985,True,8191.999996,True,2047.999999,True,512.0,False,128.0,False



Largest individual layer records:


,doc_id,seq_len,layer,chunks,heads,state_dim,max_abs_input,max_abs_reference_state,max_abs_adaptive_state,coverage_pct
450,8,2048,18,8,24,8192,458.807892,458.807892,458.807892,24.479167
371,5,2048,11,8,24,8192,378.209930,378.209930,378.209930,38.541667
322,3,2048,10,8,24,8192,371.831848,372.480133,372.480133,42.187500
251,0,2048,11,8,24,8192,361.237091,361.237091,361.237091,38.020833
11,0,1024,11,4,24,8192,361.130737,361.130737,361.130737,36.458333
58,2,1024,10,4,24,8192,351.577637,353.502899,353.502899,42.708333
298,2,2048,10,8,24,8192,351.499542,353.425415,353.425415,39.583333
418,7,2048,10,8,24,8192,320.857971,330.196350,330.196350,44.791667
186,7,1024,18,4,24,8192,328.630280,328.630280,328.630280,33.333333
426,7,2048,18,8,24,8192,328.610870,328.610870,328.610870,25.520833



Saved: /kaggle/working/shiftscan_final/state_passing/multisequence_validation/fixed_point_range_audit


In [147]:

import os
import ast
import torch
import numpy as np
import pandas as pd

BASE = "/kaggle/working/shiftscan_final"
CAPTURE_DIR = os.path.join(
    BASE, "state_passing", "multisequence_validation"
)
GRID_PATH = os.path.join(
    BASE, "canonical_dyadic_grid_definitions.csv"
)
OUT_DIR = os.path.join(
    CAPTURE_DIR, "fixed_point_integer_validation"
)
os.makedirs(OUT_DIR, exist_ok=True)

THRESHOLD = 0.001
PRECISIONS = [18, 20]

INT32_MIN = -(2**31)
INT32_MAX = 2**31 - 1

# ---------------------------------------------------------
# 1. ROUNDING HELPERS
# ---------------------------------------------------------

def round_away_float_to_int(x):
    """Round float tensor to integer, ties away from zero."""
    x = x.to(torch.float64)
    return (
        torch.sign(x) * torch.floor(torch.abs(x) + 0.5)
    ).to(torch.int64)


def round_shift_signed(x, k):
    """Signed right shift, round-to-nearest, ties away from zero."""
    x = x.to(torch.int64)

    if k == 0:
        return x.clone()

    magnitude = torch.abs(x)
    rounded = (
        magnitude + (1 << (k - 1))
    ) >> k

    return torch.sign(x) * rounded


# ---------------------------------------------------------
# 2. LOAD CANONICAL THREE-TERM GRID AND EXPONENTS
# ---------------------------------------------------------

grid_df = pd.read_csv(GRID_PATH)
grid_df = grid_df[
    grid_df["grid"] == "K1_14_T3"
].copy()

def parse_exponents(value):

    if isinstance(value, (tuple, list)):
        result = tuple(int(v) for v in value)
    else:
        result = tuple(
            int(v) for v in ast.literal_eval(str(value))
        )

    if len(result) != 3:
        raise ValueError(
            f"Expected three exponents, got {result}"
        )

    return result


grid_df["exp_tuple"] = grid_df[
    "canonical_exponents"
].apply(parse_exponents)

grid_df = grid_df.drop_duplicates(
    subset=["decay"]
).sort_values("decay").reset_index(drop=True)

grid_values = torch.tensor(
    grid_df["decay"].to_numpy(),
    dtype=torch.float32
)

grid_exponents = grid_df["exp_tuple"].tolist()

grid_log_distance = torch.log(1.0 - grid_values)

grid_min = float(grid_values.min())
grid_max = float(grid_values.max())

print("Unique grid values:", len(grid_values))
print("Decay range:", grid_min, grid_max)
print("Example exponent tuples:", grid_exponents[:5])


# ---------------------------------------------------------
# 3. HYBRID COEFFICIENT SELECTION
#
# mode 1: Dyadic2, three shift-subtract terms
# mode 2: Direct power of two
# mode 3: Exact zero
# mode 0: Exact fallback
# ---------------------------------------------------------

def build_hybrid(a):

    q = a.clone()
    mode = torch.zeros_like(a, dtype=torch.int8)

    # -1 means no Dyadic2 exponent tuple.
    exp_index = torch.full_like(
        a, -1, dtype=torch.int64
    )

    finite = torch.isfinite(a)
    positive = finite & (a > 0)
    zero = finite & (a == 0)

    dyadic_mask = (
        positive
        & (a >= grid_min)
        & (a <= grid_max)
    )

    if dyadic_mask.any():

        d = 1.0 - a[dyadic_mask]

        distance = torch.abs(
            torch.log(d)[:, None]
            - grid_log_distance[None, :]
        )

        indices = distance.argmin(dim=1)

        q[dyadic_mask] = grid_values[indices]
        mode[dyadic_mask] = 1
        exp_index[dyadic_mask] = indices

    # Direct power-of-two region.
    pot_mask = (
        positive
        & (a < grid_min)
        & (a >= 2.0**-30)
    )

    if pot_mask.any():

        k = torch.round(
            -torch.log2(a[pot_mask])
        ).to(torch.int64).clamp(4, 30)

        q[pot_mask] = torch.pow(
            torch.tensor(2.0),
            -k.float()
        )

        mode[pot_mask] = 2

    q[zero] = 0.0
    mode[zero] = 3

    return q, mode, exp_index


# ---------------------------------------------------------
# 4. INTEGER DECAY UPDATE
# ---------------------------------------------------------

def integer_decay_update(
    state_int,
    true_decay,
    candidate_decay,
    mode,
    exp_index,
    accept
):
    """
    Apply the accepted quantized coefficient using
    integer shifts. Use exact coefficient multiplication
    for rejected or unsupported coefficients.
    """

    heads, dim = state_int.shape

    decayed = torch.empty_like(state_int)

    # Any rejected coefficient or mode-0 coefficient
    # takes exact fallback.
    fallback = (~accept) | (mode == 0)

    if fallback.any():

        # Float64 multiplication is used here to make
        # fallback rounding accurate. This is deliberately
        # NOT claimed to be multiplication-free.
        product = (
            state_int[fallback].to(torch.float64)
            * true_decay[fallback].to(torch.float64)[:, None]
        )

        decayed[fallback] = round_away_float_to_int(
            product
        )

    # Exact zero.
    zero_mask = accept & (mode == 3)

    if zero_mask.any():
        decayed[zero_mask] = 0

    # Direct power-of-two shift.
    pot_mask = accept & (mode == 2)

    if pot_mask.any():

        ks = torch.round(
            -torch.log2(candidate_decay[pot_mask])
        ).to(torch.int64)

        for k in torch.unique(ks).tolist():

            mask_indices = torch.where(pot_mask)[0]
            local_mask = ks == k
            selected_heads = mask_indices[local_mask]

            decayed[selected_heads] = round_shift_signed(
                state_int[selected_heads],
                int(k)
            )

    # Three-term Dyadic2 shift-subtract.
    dyadic_mask = accept & (mode == 1)

    if dyadic_mask.any():

        for idx in torch.unique(
            exp_index[dyadic_mask]
        ).tolist():

            selected_heads = torch.where(
                dyadic_mask & (exp_index == idx)
            )[0]

            exponents = grid_exponents[int(idx)]

            value = state_int[selected_heads].clone()

            for k in exponents:
                value -= round_shift_signed(
                    state_int[selected_heads],
                    int(k)
                )

            decayed[selected_heads] = value

    return decayed


# ---------------------------------------------------------
# 5. EVALUATE ONE LAYER AT ONE FIXED-POINT PRECISION
# ---------------------------------------------------------

def evaluate_fixed_layer(record, F):

    scale = 2**F

    N = record["states"][0].float()
    true_decay = torch.exp(
        record["decay_log"][0].float()
    ).T.contiguous()

    chunks, heads, dim = N.shape

    q, mode, exp_index = build_hybrid(true_decay)

    # Same FP32 reference and adaptive decisions
    # as in the previous experiment.
    reference = torch.zeros(
        (heads, dim), dtype=torch.float32
    )

    adaptive_fp32 = torch.zeros_like(reference)

    # Fixed-point state stored as int64 during calculation.
    fixed_int = torch.zeros(
        (heads, dim), dtype=torch.int64
    )

    trajectory_errors = []
    fixed_vs_adaptive_errors = []

    accepted_count = 0
    overflow_count = 0

    max_abs_integer_state = 0

    for c in range(chunks):

        # Reproduce the FP32 adaptive accept/fallback decision.
        local_error = (
            (q[c] - true_decay[c]).abs()[:, None]
            * adaptive_fp32.abs()
        )

        accept = (
            (mode[c] != 0)
            & (local_error.amax(dim=1) <= THRESHOLD)
        )

        accepted_count += int(accept.sum())

        # Update exact FP32 reference.
        reference = (
            true_decay[c, :, None] * reference
            + N[c]
        )

        # Update FP32 adaptive recurrence.
        chosen_fp32 = torch.where(
            accept, q[c], true_decay[c]
        )

        adaptive_fp32 = (
            chosen_fp32[:, None] * adaptive_fp32
            + N[c]
        )

        # Quantize the new-state contribution.
        N_int = round_away_float_to_int(
            N[c] * scale
        )

        # Apply integer decay.
        decayed_int = integer_decay_update(
            state_int=fixed_int,
            true_decay=true_decay[c],
            candidate_decay=q[c],
            mode=mode[c],
            exp_index=exp_index[c],
            accept=accept
        )

        # Add quantized new-state contribution.
        fixed_int = decayed_int + N_int

        current_max = int(
            fixed_int.abs().max().item()
        )

        max_abs_integer_state = max(
            max_abs_integer_state,
            current_max
        )

        if current_max > INT32_MAX:
            overflow_count += 1

        if int(fixed_int.min()) < INT32_MIN:
            overflow_count += 1

        fixed_float = fixed_int.float() / scale

        trajectory_errors.append(
            (fixed_float - reference).abs().reshape(-1)
        )

        fixed_vs_adaptive_errors.append(
            (fixed_float - adaptive_fp32).abs().reshape(-1)
        )

    trajectory_errors = torch.cat(trajectory_errors)
    fixed_vs_adaptive_errors = torch.cat(
        fixed_vs_adaptive_errors
    )

    final_reference = reference
    final_fixed = fixed_int.float() / scale
    final_adaptive = adaptive_fp32

    return {
        "accepted_coefficients": accepted_count,
        "total_coefficients": chunks * heads,
        "coverage_pct": (
            100 * accepted_count / (chunks * heads)
        ),
        "trajectory_mae_vs_reference": (
            trajectory_errors.mean().item()
        ),
        "trajectory_rmse_vs_reference": (
            trajectory_errors.square().mean().sqrt().item()
        ),
        "trajectory_max_vs_reference": (
            trajectory_errors.max().item()
        ),
        "final_mae_vs_reference": (
            (final_fixed - final_reference).abs().mean().item()
        ),
        "final_rmse_vs_reference": (
            (final_fixed - final_reference).square().mean().sqrt().item()
        ),
        "final_max_vs_reference": (
            (final_fixed - final_reference).abs().max().item()
        ),
        "trajectory_mae_vs_fp32_adaptive": (
            fixed_vs_adaptive_errors.mean().item()
        ),
        "trajectory_rmse_vs_fp32_adaptive": (
            fixed_vs_adaptive_errors.square().mean().sqrt().item()
        ),
        "max_abs_integer_state": max_abs_integer_state,
        "overflow_count": overflow_count
    }


# ---------------------------------------------------------
# 6. RUN ALL CAPTURES
# ---------------------------------------------------------

manifest = pd.read_csv(
    os.path.join(CAPTURE_DIR, "capture_manifest.csv")
)

results = []

for F in PRECISIONS:

    print("\n" + "=" * 75)
    print(f"RUNNING FIXED-POINT F{F}")
    print("=" * 75)

    for _, item in manifest.iterrows():

        doc_id = int(item["doc_id"])
        seq_len = int(item["seq_len"])

        saved = torch.load(
            os.path.join(CAPTURE_DIR, item["file"]),
            map_location="cpu",
            weights_only=False
        )

        for layer_idx, record in enumerate(saved["records"]):

            result = evaluate_fixed_layer(record, F)

            results.append({
                "F": F,
                "doc_id": doc_id,
                "seq_len": seq_len,
                "layer": layer_idx,
                **result
            })

    print(f"Completed F{F}")


# ---------------------------------------------------------
# 7. SAVE AND SUMMARIZE
# ---------------------------------------------------------

results_df = pd.DataFrame(results)

results_df.to_csv(
    os.path.join(OUT_DIR, "fixed_point_per_layer.csv"),
    index=False
)

summary_rows = []

for (F, seq_len), group in results_df.groupby(
    ["F", "seq_len"]
):

    summary_rows.append({
        "F": F,
        "seq_len": seq_len,
        "layer_records": len(group),
        "mean_coverage_pct": group["coverage_pct"].mean(),
        "mean_trajectory_mae": group[
            "trajectory_mae_vs_reference"
        ].mean(),
        "mean_trajectory_rmse": np.sqrt(
            np.mean(
                group["trajectory_rmse_vs_reference"]**2
            )
        ),
        "worst_trajectory_max": group[
            "trajectory_max_vs_reference"
        ].max(),
        "mean_final_mae": group[
            "final_mae_vs_reference"
        ].mean(),
        "mean_final_rmse": np.sqrt(
            np.mean(
                group["final_rmse_vs_reference"]**2
            )
        ),
        "worst_final_max": group[
            "final_max_vs_reference"
        ].max(),
        "mean_fixed_vs_adaptive_mae": group[
            "trajectory_mae_vs_fp32_adaptive"
        ].mean(),
        "max_integer_state": group[
            "max_abs_integer_state"
        ].max(),
        "overflow_count": group[
            "overflow_count"
        ].sum()
    })

summary = pd.DataFrame(summary_rows)

summary.to_csv(
    os.path.join(OUT_DIR, "fixed_point_summary.csv"),
    index=False
)

print("\n" + "=" * 85)
print("INTEGER SHIFT-SUBTRACT VALIDATION SUMMARY")
print("=" * 85)

display(summary)

print("\nSaved results:", OUT_DIR)


Unique grid values: 467
Decay range: 0.125 0.99981689453125
Example exponent tuples: [(1, 2, 3), (1, 2, 4), (1, 2, 5), (1, 2, 6), (1, 2, 7)]

RUNNING FIXED-POINT F18
Completed F18

RUNNING FIXED-POINT F20
Completed F20

INTEGER SHIFT-SUBTRACT VALIDATION SUMMARY


,F,seq_len,layer_records,mean_coverage_pct,mean_trajectory_mae,mean_trajectory_rmse,worst_trajectory_max,mean_final_mae,mean_final_rmse,worst_final_max,mean_fixed_vs_adaptive_mae,max_integer_state,overflow_count
0,18,1024,240,55.269097,1.638715e-06,0.000007,0.001660,0.000002,0.000008,0.001660,1.106326e-06,94668256,0
1,18,2048,240,51.161024,1.886946e-06,0.000008,0.002524,0.000002,0.000010,0.002524,1.167957e-06,120273736,0
2,20,1024,240,55.269097,8.668593e-07,0.000006,0.001663,0.000001,0.000008,0.001663,2.773109e-07,378673024,0
3,20,2048,240,51.161024,1.095208e-06,0.000008,0.002526,0.000001,0.000010,0.002526,2.930480e-07,481094944,0



Saved results: /kaggle/working/shiftscan_final/state_passing/multisequence_validation/fixed_point_integer_validation


In [149]:

import os, glob, torch

ROOT = "/kaggle/working/shiftscan_final/state_passing/multisequence_validation"

files = sorted(glob.glob(os.path.join(ROOT, "doc_*_seq_*.pt")))
print("Capture files:", len(files))

payload = torch.load(files[0], map_location="cpu", weights_only=False)

print("\nFirst file:", os.path.basename(files[0]))
print("Payload type:", type(payload).__name__)

if isinstance(payload, dict):
    print("Top-level keys:", list(payload.keys()))

    records = payload.get("records", payload)

    if isinstance(records, dict):
        print("\nRecords container keys:", list(records.keys()))
        records = records.get("layers", records.get("records", []))

    print("\nRecords type:", type(records).__name__)
    print("Number of records:", len(records))

    if len(records):
        first = records[0]
        print("\nFirst record type:", type(first).__name__)

        if isinstance(first, dict):
            print("First record keys:", list(first.keys()))

            for key, value in first.items():
                if torch.is_tensor(value):
                    print(
                        f"{key}: shape={tuple(value.shape)}, "
                        f"dtype={value.dtype}"
                    )
                else:
                    print(f"{key}: {type(value).__name__} = {value}")


Capture files: 20

First file: doc_000_seq_1024.pt
Payload type: dict
Top-level keys: ['metadata', 'records']

Records type: list
Number of records: 24

First record type: dict
First record keys: ['layer', 'doc_id', 'seq_len', 'states', 'decay_log', 'chunk_size', 'has_initial_states', 'has_seq_idx']
layer: int = 0
doc_id: int = 0
seq_len: int = 1024
states: shape=(1, 4, 24, 8192), dtype=torch.float32
decay_log: shape=(1, 24, 4), dtype=torch.float32
chunk_size: int = 256
has_initial_states: bool = False
has_seq_idx: bool = False


In [151]:

def run_fixed_controller(
    states,
    decay,
    frac_bits,
    threshold=THRESHOLD
):
    """
    Fixed-point-driven adaptive controller.

    states: [chunks, heads, state_dim]
    decay:  [chunks, heads]

    Adaptive decisions use the current fixed-point state.
    FP64 reference state is used only for evaluation.
    """

    scale = 1 << frac_bits

    states = states.detach().cpu().float()
    decay = decay.detach().cpu().double()

    chunks, heads, dim = states.shape

    assert decay.shape == (chunks, heads)

    # Exact FP64 reference state
    ref_state = torch.zeros(
        (heads, dim),
        dtype=torch.float64
    )

    # Fixed-point integer state
    fixed_state = torch.zeros(
        (heads, dim),
        dtype=torch.int64
    )

    # Quantize incoming state contributions
    N_int = round_away_float_to_int(
        states.double() * scale
    )

    trajectory_mae = []
    trajectory_rmse = []
    trajectory_max = []

    accepted_count = 0
    fallback_count = 0

    mode_counts = {0: 0, 1: 0, 2: 0, 3: 0}

    overflow_count = 0
    max_integer_state = 0

    for c in range(chunks):

        # Convert decay coefficients to NumPy for policy selection
        a_np = decay[c].numpy()

        modes, q_values, exponents = propose_decay_modes(a_np)

        # ----------------------------------------------------
        # Reference recurrence: tensor × tensor
        # ----------------------------------------------------

        a_tensor = decay[c].unsqueeze(1)

        ref_state = (
            a_tensor * ref_state
            + states[c].double()
        )

        # ----------------------------------------------------
        # Fixed-point-driven acceptance decision
        # ----------------------------------------------------

        current_real = fixed_state.double() / scale

        current_max = (
            torch.abs(current_real)
            .amax(dim=1)
            .numpy()
        )

        local_bound = (
            np.abs(q_values - a_np) * current_max
        )

        accept = (
            (modes == 3) |
            (
                (modes != 0) &
                np.isfinite(local_bound) &
                (local_bound <= threshold)
            )
        )

        fallback = ~accept

        accepted_modes = modes.copy()
        accepted_modes[fallback] = 0

        decayed = torch.empty_like(fixed_state)

        # ----------------------------------------------------
        # Exact fallback: floating-point multiplication
        # ----------------------------------------------------

        fallback_idx = np.where(fallback)[0]

        if len(fallback_idx):

            idx = torch.tensor(
                fallback_idx,
                dtype=torch.long
            )

            a_fallback = decay[c, idx]

            fallback_values = (
                fixed_state[idx].double()
                * a_fallback[:, None]
            )

            decayed[idx] = round_away_float_to_int(
                fallback_values
            )

        # ----------------------------------------------------
        # Exact zero mode
        # ----------------------------------------------------

        zero_idx = np.where(
            accept & (modes == 3)
        )[0]

        if len(zero_idx):

            idx = torch.tensor(
                zero_idx,
                dtype=torch.long
            )

            decayed[idx] = 0

        # ----------------------------------------------------
        # Direct power-of-two mode
        # ----------------------------------------------------

        pot_idx = np.where(
            accept & (modes == 2)
        )[0]

        for h in pot_idx:

            k = int(exponents[h, 0])

            decayed[h] = round_shift_signed(
                fixed_state[h],
                k
            )

        # ----------------------------------------------------
        # Three-term Dyadic2 shift-subtract mode
        # All shifts use the ORIGINAL state.
        # ----------------------------------------------------

        dyadic_idx = np.where(
            accept & (modes == 1)
        )[0]

        for h in dyadic_idx:

            original = fixed_state[h].clone()
            value = original.clone()

            for k in exponents[h]:

                value -= round_shift_signed(
                    original,
                    int(k)
                )

            decayed[h] = value

        # Add incoming fixed-point state contribution
        fixed_state = decayed + N_int[c]

        # ----------------------------------------------------
        # Integer range check
        # ----------------------------------------------------

        current_min = int(fixed_state.min().item())
        current_max_int = int(fixed_state.max().item())

        if (
            current_min < INT32_MIN or
            current_max_int > INT32_MAX
        ):
            overflow_count += 1

        max_integer_state = max(
            max_integer_state,
            int(torch.abs(fixed_state).max().item())
        )

        # ----------------------------------------------------
        # Compare fixed-point state with FP64 reference
        # ----------------------------------------------------

        fixed_real = fixed_state.double() / scale

        err = fixed_real - ref_state

        trajectory_mae.append(
            float(torch.abs(err).mean().item())
        )

        trajectory_rmse.append(
            float(torch.sqrt(torch.mean(err ** 2)).item())
        )

        trajectory_max.append(
            float(torch.abs(err).max().item())
        )

        accepted_count += int(accept.sum())
        fallback_count += int(fallback.sum())

        for mode in range(4):
            mode_counts[mode] += int(
                np.sum(accepted_modes == mode)
            )

    # --------------------------------------------------------
    # Final metrics
    # --------------------------------------------------------

    fixed_real = fixed_state.double() / scale
    final_err = fixed_real - ref_state

    total = accepted_count + fallback_count

    return {
        "coverage_pct": (
            100.0 * accepted_count / total if total else 0.0
        ),
        "fallback_pct": (
            100.0 * fallback_count / total if total else 0.0
        ),

        "dyadic_count": mode_counts[1],
        "pot_count": mode_counts[2],
        "zero_count": mode_counts[3],
        "fallback_count": fallback_count,

        "trajectory_mae": float(np.mean(trajectory_mae)),
        "trajectory_rmse": float(np.mean(trajectory_rmse)),
        "trajectory_max": float(np.max(trajectory_max)),

        "final_mae": float(
            torch.abs(final_err).mean().item()
        ),
        "final_rmse": float(
            torch.sqrt(torch.mean(final_err ** 2)).item()
        ),
        "final_max": float(
            torch.abs(final_err).max().item()
        ),

        "max_integer_state": max_integer_state,
        "overflow_count": overflow_count,

        "frac_bits": frac_bits,
        "chunks": chunks,
        "heads": heads,
        "state_dim": dim
    }


print("Corrected run_fixed_controller() is ready.")


Corrected run_fixed_controller() is ready.


In [160]:

from pathlib import Path
import pandas as pd
import torch
import json

ROOT = Path(
    "/kaggle/working/shiftscan_final/state_passing/"
    "multisequence_validation"
)

CTRL = ROOT / "fixed_point_controller_validation_v2"

print("Controller files:")
for p in sorted(CTRL.iterdir()):
    if p.is_file():
        print(f"  {p.name}  ({p.stat().st_size:,} bytes)")

print("\nPer-layer controller results:")
per_layer_path = CTRL / "per_layer_controller_results.csv"
df = pd.read_csv(per_layer_path)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head(5))

print("\nController summary:")
summary_path = CTRL / "controller_summary.csv"
summary = pd.read_csv(summary_path)
print("Shape:", summary.shape)
print("Columns:", summary.columns.tolist())
display(summary)

print("\nCapture structure:")
capture_path = ROOT / "multisequence_validation" / "doc_000_seq_1024.pt"

# If the path above does not exist, use the actual capture directory.
if not capture_path.exists():
    capture_path = ROOT / "doc_000_seq_1024.pt"

obj = torch.load(capture_path, map_location="cpu", weights_only=False)

print("Top-level type:", type(obj))
if isinstance(obj, dict):
    print("Top-level keys:", list(obj.keys()))
    for k, v in obj.items():
        if torch.is_tensor(v):
            print(f"{k}: tensor {tuple(v.shape)} {v.dtype}")
        elif isinstance(v, list):
            print(f"{k}: list length {len(v)}")
            if v and isinstance(v[0], dict):
                print("  first record keys:", list(v[0].keys()))
        else:
            print(f"{k}: {type(v)}")
elif isinstance(obj, list):
    print("List length:", len(obj))
    if obj and isinstance(obj[0], dict):
        print("First record keys:", list(obj[0].keys()))

print("\nExperiment configuration:")
config_path = CTRL / "experiment_config.json"
if config_path.exists():
    print(json.dumps(json.loads(config_path.read_text()), indent=2))


Controller files:
  controller_summary.csv  (980 bytes)
  experiment_config.json  (607 bytes)
  per_layer_controller_results.csv  (219,031 bytes)

Per-layer controller results:
Shape: (960, 22)
Columns: ['coverage_pct', 'fallback_pct', 'dyadic_count', 'pot_count', 'zero_count', 'fallback_count', 'trajectory_mae', 'trajectory_rmse', 'trajectory_max', 'final_mae', 'final_rmse', 'final_max', 'max_integer_state', 'overflow_count', 'frac_bits', 'chunks', 'heads', 'state_dim', 'file', 'seq_len', 'doc_id', 'layer']


,coverage_pct,fallback_pct,dyadic_count,pot_count,zero_count,fallback_count,trajectory_mae,trajectory_rmse,trajectory_max,final_mae,...,max_integer_state,overflow_count,frac_bits,chunks,heads,state_dim,file,seq_len,doc_id,layer
0,65.625000,34.375000,0,63,0,33,0.000004,0.000013,0.000972,0.000005,...,9766190,0,18,4,24,8192,doc_000_seq_1024.pt,1024,0,0
1,57.291667,42.708333,0,7,48,41,0.000001,0.000001,0.000005,0.000001,...,6955186,0,18,4,24,8192,doc_000_seq_1024.pt,1024,0,1
2,42.708333,57.291667,14,10,17,55,0.000001,0.000002,0.000309,0.000001,...,9498163,0,18,4,24,8192,doc_000_seq_1024.pt,1024,0,2
3,55.208333,44.791667,1,19,33,43,0.000001,0.000003,0.000563,0.000001,...,30433624,0,18,4,24,8192,doc_000_seq_1024.pt,1024,0,3
4,46.875000,53.125000,21,8,16,51,0.000002,0.000006,0.000773,0.000003,...,22224362,0,18,4,24,8192,doc_000_seq_1024.pt,1024,0,4



Controller summary:
Shape: (4, 13)
Columns: ['frac_bits', 'seq_len', 'layer_records', 'mean_coverage_pct', 'mean_fallback_pct', 'mean_trajectory_mae', 'mean_trajectory_rmse', 'worst_trajectory_max', 'mean_final_mae', 'mean_final_rmse', 'worst_final_max', 'max_integer_state', 'overflow_count']


,frac_bits,seq_len,layer_records,mean_coverage_pct,mean_fallback_pct,mean_trajectory_mae,mean_trajectory_rmse,worst_trajectory_max,mean_final_mae,mean_final_rmse,worst_final_max,max_integer_state,overflow_count
0,18,1024,240,55.269097,44.730903,1.637394e-06,0.000004,0.001660,0.000002,0.000006,0.001660,94668256,0
1,18,2048,240,51.163194,48.836806,1.886074e-06,0.000005,0.002524,0.000002,0.000007,0.002524,120273736,0
2,20,1024,240,55.269097,44.730903,8.655578e-07,0.000004,0.001663,0.000001,0.000006,0.001663,378673024,0
3,20,2048,240,51.163194,48.836806,1.094323e-06,0.000005,0.002526,0.000001,0.000007,0.002526,481094944,0



Capture structure:
Top-level type: <class 'dict'>
Top-level keys: ['metadata', 'records']
metadata: <class 'dict'>
records: list length 24
  first record keys: ['layer', 'doc_id', 'seq_len', 'states', 'decay_log', 'chunk_size', 'has_initial_states', 'has_seq_idx']

Experiment configuration:
{
  "experiment": "fixed_point_driven_adaptive_controller_v2",
  "model": "state-spaces/mamba2-130m",
  "dataset": "FineWeb-Edu",
  "capture_files": 20,
  "layer_records": 960,
  "fractional_bits": [
    18,
    20
  ],
  "threshold": 0.001,
  "dyadic_grid": "K1_14_T3",
  "unique_grid_values": 467,
  "pot_exponents": [
    4,
    30
  ],
  "rounding": "nearest_ties_away_from_zero",
  "integer_intermediate": "int64",
  "int32_range_checked": true,
  "adaptive_decision_state": "current_fixed_point_state",
  "fallback": "float64_exact_decay_multiply",
  "scope": "Mamba-2 state-passing recurrence only"
}


In [159]:

# ============================================================
# SHIFT-SCAN: FIXED-POINT-DRIVEN ADAPTIVE CONTROLLER
# CLEAN, SELF-CONTAINED EXECUTION CELL
#
# Actual Mamba-2 130M state-passing captures
# 10 documents x 2 sequence lengths x 24 layers
#
# Fixed-point formats: F18 and F20
# Adaptive threshold: 0.001
#
# FIXES:
#   1. Correct capture key: decay_log
#   2. Correct tensor layout: [chunks, heads]
#   3. Explicit NumPy -> Torch conversion
#   4. All Dyadic2 shifts use the original state
#   5. Decisions use fixed-point state, not FP32 reference
# ============================================================

import os
import ast
import json
import glob
import numpy as np
import pandas as pd
import torch

# ============================================================
# 1. CONFIGURATION
# ============================================================

ROOT = (
    "/kaggle/working/shiftscan_final/"
    "state_passing/multisequence_validation"
)

GRID_CSV = (
    "/kaggle/working/shiftscan_final/"
    "canonical_dyadic_grid_definitions.csv"
)

OUT = os.path.join(
    ROOT,
    "fixed_point_controller_validation_v2"
)

os.makedirs(OUT, exist_ok=True)

FRAC_BITS_LIST = [18, 20]
THRESHOLD = 0.001

POT_K_MIN = 4
POT_K_MAX = 30

INT32_MIN = -(2**31)
INT32_MAX = 2**31 - 1

# ============================================================
# 2. VALIDATE INPUT FILES
# ============================================================

assert os.path.exists(GRID_CSV), (
    f"Canonical grid CSV not found: {GRID_CSV}"
)

capture_files = sorted(
    glob.glob(
        os.path.join(ROOT, "doc_*_seq_*.pt")
    )
)

assert len(capture_files) == 20, (
    f"Expected 20 capture files, found {len(capture_files)}"
)

print("Capture files:", len(capture_files))

# ============================================================
# 3. LOAD CANONICAL DYADIC2 GRID
# ============================================================

grid_df = pd.read_csv(GRID_CSV)

grid_df = grid_df[
    grid_df["grid"] == "K1_14_T3"
].copy()

grid_df = (
    grid_df
    .drop_duplicates(subset=["decay"])
    .sort_values("decay")
    .reset_index(drop=True)
)

grid_a = grid_df["decay"].to_numpy(
    dtype=np.float64
)

def parse_exponents(value):
    if isinstance(value, str):
        return tuple(ast.literal_eval(value))
    return tuple(value)

grid_exp = [
    parse_exponents(x)
    for x in grid_df["canonical_exponents"]
]

grid_exp_np = np.asarray(
    grid_exp,
    dtype=np.int64
)

assert len(grid_a) == 467
assert grid_exp_np.shape == (467, 3)

assert np.allclose(
    grid_a,
    1.0 - np.sum(
        2.0 ** (-grid_exp_np),
        axis=1
    ),
    atol=1e-12
)

print("Canonical grid values:", len(grid_a))
print("Decay range:", grid_a.min(), grid_a.max())

# ============================================================
# 4. ROUNDING HELPERS
# ============================================================

def round_away_float_to_int(x):
    """Nearest integer, ties away from zero."""
    x = torch.as_tensor(
        x,
        dtype=torch.float64
    )

    return (
        torch.sign(x) *
        torch.floor(torch.abs(x) + 0.5)
    ).to(torch.int64)


def round_shift_signed(x, k):
    """Signed right shift, nearest rounding, ties away."""
    x = x.to(torch.int64)
    k = int(k)

    if k == 0:
        return x.clone()

    divisor = 1 << k

    rounded = (
        torch.abs(x) + divisor // 2
    ) // divisor

    return torch.sign(x) * rounded


# ============================================================
# 5. DECAY REPRESENTATION POLICY
# ============================================================

def propose_decay_modes(decay):
    """
    mode 0 = exact fallback
    mode 1 = three-term Dyadic2
    mode 2 = direct power-of-two
    mode 3 = exact zero
    """

    decay = np.asarray(
        decay,
        dtype=np.float64
    )

    modes = np.zeros(
        decay.shape,
        dtype=np.int64
    )

    q_values = decay.copy()

    exponents = np.zeros(
        decay.shape + (3,),
        dtype=np.int64
    )

    # Exact zero
    zero = (
        np.isfinite(decay) &
        (decay == 0.0)
    )

    modes[zero] = 3
    q_values[zero] = 0.0

    # Three-term Dyadic2
    dyadic = (
        np.isfinite(decay) &
        (decay >= grid_a.min()) &
        (decay <= grid_a.max()) &
        (~zero)
    )

    if np.any(dyadic):

        vals = decay[dyadic]

        distances = np.abs(
            vals[:, None] - grid_a[None, :]
        )

        idx = distances.argmin(axis=1)

        modes[dyadic] = 1
        q_values[dyadic] = grid_a[idx]
        exponents[dyadic] = grid_exp_np[idx]

    # Direct PoT for small positive decay
    pot = (
        np.isfinite(decay) &
        (decay > 0.0) &
        (decay < grid_a.min()) &
        (decay >= 2.0 ** (-POT_K_MAX))
    )

    if np.any(pot):

        vals = decay[pot]

        k = np.clip(
            np.rint(-np.log2(vals)).astype(np.int64),
            POT_K_MIN,
            POT_K_MAX
        )

        modes[pot] = 2
        q_values[pot] = np.exp2(-k.astype(np.float64))
        exponents[pot, 0] = k

    return modes, q_values, exponents


# ============================================================
# 6. FIXED-POINT-DRIVEN CONTROLLER
# ============================================================

def run_fixed_controller(
    states,
    decay,
    frac_bits,
    threshold=THRESHOLD
):

    scale = 1 << frac_bits

    states = states.detach().cpu().float()
    decay = decay.detach().cpu().double()

    chunks, heads, dim = states.shape

    assert decay.shape == (chunks, heads)

    # FP64 reference is only for error evaluation.
    ref_state = torch.zeros(
        (heads, dim),
        dtype=torch.float64
    )

    # Fixed-point state.
    fixed_state = torch.zeros(
        (heads, dim),
        dtype=torch.int64
    )

    # Quantize incoming state contributions.
    N_int = round_away_float_to_int(
        states.double() * scale
    )

    trajectory_mae = []
    trajectory_rmse = []
    trajectory_max = []

    accepted_count = 0
    fallback_count = 0

    mode_counts = {
        0: 0,
        1: 0,
        2: 0,
        3: 0
    }

    overflow_count = 0
    max_integer_state = 0

    for c in range(chunks):

        # ----------------------------------------------------
        # Decay values for current chunk
        # ----------------------------------------------------

        a_tensor = decay[c].clone()

        a_np = a_tensor.numpy()

        modes, q_values, exponents = (
            propose_decay_modes(a_np)
        )

        # ----------------------------------------------------
        # FP64 reference recurrence
        # Both operands are now PyTorch tensors.
        # ----------------------------------------------------

        ref_state = (
            a_tensor.unsqueeze(1) * ref_state
            + states[c].double()
        )

        # ----------------------------------------------------
        # Decision based on fixed-point state only
        # ----------------------------------------------------

        current_real = (
            fixed_state.double() / scale
        )

        current_max = (
            torch.abs(current_real)
            .amax(dim=1)
            .numpy()
        )

        local_bound = (
            np.abs(q_values - a_np) * current_max
        )

        accept = (
            (modes == 3) |
            (
                (modes != 0) &
                np.isfinite(local_bound) &
                (local_bound <= threshold)
            )
        )

        fallback = ~accept

        accepted_modes = modes.copy()
        accepted_modes[fallback] = 0

        decayed = torch.empty_like(
            fixed_state
        )

        # ----------------------------------------------------
        # Mode 0: exact floating-point fallback
        # ----------------------------------------------------

        fallback_idx = np.where(fallback)[0]

        if len(fallback_idx):

            idx = torch.as_tensor(
                fallback_idx,
                dtype=torch.long
            )

            a_fallback = a_tensor[idx]

            fallback_values = (
                fixed_state[idx].double()
                * a_fallback.unsqueeze(1)
            )

            decayed[idx] = round_away_float_to_int(
                fallback_values
            )

        # ----------------------------------------------------
        # Mode 3: exact zero
        # ----------------------------------------------------

        zero_idx = np.where(
            accept & (modes == 3)
        )[0]

        if len(zero_idx):

            idx = torch.as_tensor(
                zero_idx,
                dtype=torch.long
            )

            decayed[idx] = 0

        # ----------------------------------------------------
        # Mode 2: direct power-of-two
        # ----------------------------------------------------

        pot_idx = np.where(
            accept & (modes == 2)
        )[0]

        for h in pot_idx:

            k = int(exponents[h, 0])

            decayed[h] = round_shift_signed(
                fixed_state[h],
                k
            )

        # ----------------------------------------------------
        # Mode 1: three-term Dyadic2
        #
        # Hq = H - round(H/2^k1)
        #        - round(H/2^k2)
        #        - round(H/2^k3)
        #
        # Each shift uses the original H.
        # ----------------------------------------------------

        dyadic_idx = np.where(
            accept & (modes == 1)
        )[0]

        for h in dyadic_idx:

            original = fixed_state[h].clone()

            value = original.clone()

            for k in exponents[h]:

                value -= round_shift_signed(
                    original,
                    int(k)
                )

            decayed[h] = value

        # ----------------------------------------------------
        # Add incoming fixed-point contribution
        # ----------------------------------------------------

        fixed_state = decayed + N_int[c]

        # ----------------------------------------------------
        # Range check
        # ----------------------------------------------------

        current_min = int(
            fixed_state.min().item()
        )

        current_max = int(
            fixed_state.max().item()
        )

        if (
            current_min < INT32_MIN or
            current_max > INT32_MAX
        ):
            overflow_count += 1

        max_integer_state = max(
            max_integer_state,
            int(
                torch.abs(fixed_state).max().item()
            )
        )

        # ----------------------------------------------------
        # Error evaluation
        # ----------------------------------------------------

        fixed_real = (
            fixed_state.double() / scale
        )

        err = fixed_real - ref_state

        trajectory_mae.append(
            float(torch.abs(err).mean().item())
        )

        trajectory_rmse.append(
            float(torch.sqrt(torch.mean(err ** 2)).item())
        )

        trajectory_max.append(
            float(torch.abs(err).max().item())
        )

        # ----------------------------------------------------
        # Counters
        # ----------------------------------------------------

        accepted_count += int(accept.sum())
        fallback_count += int(fallback.sum())

        for mode in range(4):
            mode_counts[mode] += int(
                np.sum(accepted_modes == mode)
            )

    # ========================================================
    # FINAL METRICS
    # ========================================================

    fixed_real = fixed_state.double() / scale

    final_err = fixed_real - ref_state

    total = accepted_count + fallback_count

    return {
        "coverage_pct": (
            100.0 * accepted_count / total
            if total else 0.0
        ),

        "fallback_pct": (
            100.0 * fallback_count / total
            if total else 0.0
        ),

        "dyadic_count": mode_counts[1],
        "pot_count": mode_counts[2],
        "zero_count": mode_counts[3],
        "fallback_count": fallback_count,

        "trajectory_mae": float(
            np.mean(trajectory_mae)
        ),

        "trajectory_rmse": float(
            np.mean(trajectory_rmse)
        ),

        "trajectory_max": float(
            np.max(trajectory_max)
        ),

        "final_mae": float(
            torch.abs(final_err).mean().item()
        ),

        "final_rmse": float(
            torch.sqrt(
                torch.mean(final_err ** 2)
            ).item()
        ),

        "final_max": float(
            torch.abs(final_err).max().item()
        ),

        "max_integer_state": max_integer_state,
        "overflow_count": overflow_count,

        "frac_bits": frac_bits,
        "chunks": chunks,
        "heads": heads,
        "state_dim": dim
    }


# ============================================================
# 7. VALIDATE CAPTURE STRUCTURE
# ============================================================

test_payload = torch.load(
    capture_files[0],
    map_location="cpu",
    weights_only=False
)

assert "records" in test_payload
assert len(test_payload["records"]) == 24

test_record = test_payload["records"][0]

assert "states" in test_record
assert "decay_log" in test_record

print("\nCapture schema validated.")
print("Record keys:", list(test_record.keys()))


# ============================================================
# 8. RUN ALL CAPTURES
# ============================================================

all_results = []

for frac_bits in FRAC_BITS_LIST:

    print("\n" + "=" * 75)
    print(f"RUNNING F{frac_bits}")
    print("=" * 75)

    for file_idx, file_path in enumerate(capture_files):

        payload = torch.load(
            file_path,
            map_location="cpu",
            weights_only=False
        )

        records = payload["records"]

        for record in records:

            states = record["states"]
            decay_log = record["decay_log"]

            # states: [1, chunks, heads, dim]
            if states.ndim == 4:
                assert states.shape[0] == 1
                states = states[0]

            # decay_log: [1, heads, chunks]
            if decay_log.ndim == 3:
                assert decay_log.shape[0] == 1
                decay_log = decay_log[0]

            # Convert cumulative log-decay to multiplier.
            # Result shape: [chunks, heads]
            decay = torch.exp(
                decay_log.float()
            ).T.double()

            assert states.ndim == 3
            assert decay.ndim == 2
            assert states.shape[:2] == decay.shape

            result = run_fixed_controller(
                states=states,
                decay=decay,
                frac_bits=frac_bits,
                threshold=THRESHOLD
            )

            result.update({
                "file": os.path.basename(file_path),
                "seq_len": int(record["seq_len"]),
                "doc_id": str(record["doc_id"]),
                "layer": int(record["layer"])
            })

            all_results.append(result)

        print(
            f"F{frac_bits}: "
            f"{file_idx + 1}/{len(capture_files)} "
            f"completed - {os.path.basename(file_path)}"
        )

# ============================================================
# 9. SAVE PER-LAYER RESULTS
# ============================================================

results_df = pd.DataFrame(all_results)

assert len(results_df) == 960, (
    f"Expected 960 rows, found {len(results_df)}"
)

per_layer_path = os.path.join(
    OUT,
    "per_layer_controller_results.csv"
)

results_df.to_csv(
    per_layer_path,
    index=False
)

# ============================================================
# 10. AGGREGATE RESULTS
# ============================================================

summary = (
    results_df
    .groupby(["frac_bits", "seq_len"])
    .agg(
        layer_records=("layer", "count"),

        mean_coverage_pct=("coverage_pct", "mean"),
        mean_fallback_pct=("fallback_pct", "mean"),

        mean_trajectory_mae=("trajectory_mae", "mean"),
        mean_trajectory_rmse=("trajectory_rmse", "mean"),
        worst_trajectory_max=("trajectory_max", "max"),

        mean_final_mae=("final_mae", "mean"),
        mean_final_rmse=("final_rmse", "mean"),
        worst_final_max=("final_max", "max"),

        max_integer_state=("max_integer_state", "max"),
        overflow_count=("overflow_count", "sum")
    )
    .reset_index()
)

summary_path = os.path.join(
    OUT,
    "controller_summary.csv"
)

summary.to_csv(
    summary_path,
    index=False
)

# ============================================================
# 11. SAVE CONFIGURATION
# ============================================================

config = {
    "experiment": "fixed_point_driven_adaptive_controller_v2",
    "model": "state-spaces/mamba2-130m",
    "dataset": "FineWeb-Edu",
    "capture_files": len(capture_files),
    "layer_records": len(results_df),
    "fractional_bits": FRAC_BITS_LIST,
    "threshold": THRESHOLD,
    "dyadic_grid": "K1_14_T3",
    "unique_grid_values": int(len(grid_a)),
    "pot_exponents": [POT_K_MIN, POT_K_MAX],
    "rounding": "nearest_ties_away_from_zero",
    "integer_intermediate": "int64",
    "int32_range_checked": True,
    "adaptive_decision_state": "current_fixed_point_state",
    "fallback": "float64_exact_decay_multiply",
    "scope": "Mamba-2 state-passing recurrence only"
}

config_path = os.path.join(
    OUT,
    "experiment_config.json"
)

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

# ============================================================
# 12. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 100)
print("FIXED-POINT-DRIVEN ADAPTIVE CONTROLLER RESULTS")
print("=" * 100)

display(
    summary.style.format({
        "mean_coverage_pct": "{:.4f}",
        "mean_fallback_pct": "{:.4f}",
        "mean_trajectory_mae": "{:.8e}",
        "mean_trajectory_rmse": "{:.8e}",
        "worst_trajectory_max": "{:.8e}",
        "mean_final_mae": "{:.8e}",
        "mean_final_rmse": "{:.8e}",
        "worst_final_max": "{:.8e}"
    })
)

print("\nTotal records:", len(results_df))
print("Expected records: 960")

print("\nSaved:")
print(per_layer_path)
print(summary_path)
print(config_path)

print("\nEXPERIMENT COMPLETED.")


Capture files: 20
Canonical grid values: 467
Decay range: 0.125 0.99981689453125

Capture schema validated.
Record keys: ['layer', 'doc_id', 'seq_len', 'states', 'decay_log', 'chunk_size', 'has_initial_states', 'has_seq_idx']

RUNNING F18
F18: 1/20 completed - doc_000_seq_1024.pt
F18: 2/20 completed - doc_000_seq_2048.pt
F18: 3/20 completed - doc_001_seq_1024.pt
F18: 4/20 completed - doc_001_seq_2048.pt
F18: 5/20 completed - doc_002_seq_1024.pt
F18: 6/20 completed - doc_002_seq_2048.pt
F18: 7/20 completed - doc_003_seq_1024.pt
F18: 8/20 completed - doc_003_seq_2048.pt
F18: 9/20 completed - doc_004_seq_1024.pt
F18: 10/20 completed - doc_004_seq_2048.pt
F18: 11/20 completed - doc_005_seq_1024.pt
F18: 12/20 completed - doc_005_seq_2048.pt
F18: 13/20 completed - doc_006_seq_1024.pt
F18: 14/20 completed - doc_006_seq_2048.pt
F18: 15/20 completed - doc_007_seq_1024.pt
F18: 16/20 completed - doc_007_seq_2048.pt
F18: 17/20 completed - doc_008_seq_1024.pt
F18: 18/20 completed - doc_008_seq_2048

,frac_bits,seq_len,layer_records,mean_coverage_pct,mean_fallback_pct,mean_trajectory_mae,mean_trajectory_rmse,worst_trajectory_max,mean_final_mae,mean_final_rmse,worst_final_max,max_integer_state,overflow_count
0,18,1024,240,55.2691,44.7309,1.63739448e-06,4.34805895e-06,1.66049241e-03,1.98643695e-06,5.99015884e-06,1.66049241e-03,94668256,0
1,18,2048,240,51.1632,48.8368,1.88607445e-06,5.44487545e-06,2.52356883e-03,2.22603919e-06,6.96084768e-06,2.52356883e-03,120273736,0
2,20,1024,240,55.2691,44.7309,8.65557848e-07,3.80852460e-06,1.66335344e-03,1.18622445e-06,5.57540859e-06,1.66335344e-03,378673024,0
3,20,2048,240,51.1632,48.8368,1.09432331e-06,4.96423155e-06,2.52642985e-03,1.41757599e-06,6.56434737e-06,2.52642985e-03,481094944,0



Total records: 960
Expected records: 960

Saved:
/kaggle/working/shiftscan_final/state_passing/multisequence_validation/fixed_point_controller_validation_v2/per_layer_controller_results.csv
/kaggle/working/shiftscan_final/state_passing/multisequence_validation/fixed_point_controller_validation_v2/controller_summary.csv
/kaggle/working/shiftscan_final/state_passing/multisequence_validation/fixed_point_controller_validation_v2/experiment_config.json

EXPERIMENT COMPLETED.


In [161]:

from pathlib import Path
import pandas as pd
import numpy as np

CTRL = Path(
    "/kaggle/working/shiftscan_final/state_passing/"
    "multisequence_validation/fixed_point_controller_validation_v2"
)

df = pd.read_csv(CTRL / "per_layer_controller_results.csv")

MODE_COUNTS = [
    "dyadic_count",
    "pot_count",
    "zero_count",
    "fallback_count",
]

MODE_LABELS = {
    "dyadic_count": "Three-term Dyadic2",
    "pot_count": "Direct PoT",
    "zero_count": "Exact zero",
    "fallback_count": "Floating-point fallback",
}

# ---------------------------------------------------------
# 1. Validate that every recurrence update is accounted for
# ---------------------------------------------------------
df["total_updates"] = df[MODE_COUNTS].sum(axis=1)
df["expected_updates"] = df["chunks"] * df["heads"]

bad = df[df["total_updates"] != df["expected_updates"]]

print("VALIDATION")
print("Layer records:", len(df))
print("Records with count mismatch:", len(bad))
print("Total recurrence updates:", int(df["total_updates"].sum()))

if len(bad):
    display(bad[
        ["frac_bits", "seq_len", "doc_id", "layer",
         "total_updates", "expected_updates"] 
    ].head(20))
    raise ValueError("Mode counts do not sum to all recurrence updates.")

# ---------------------------------------------------------
# 2. Aggregate mode counts and percentages
# ---------------------------------------------------------
def mode_breakdown(group, group_name):
    counts = group[MODE_COUNTS].sum()
    total = int(counts.sum())

    result = {
        "group": group_name,
        "total_updates": total,
    }

    for col in MODE_COUNTS:
        count = int(counts[col])
        result[col.replace("_count", "_updates")] = count
        result[col.replace("_count", "_pct")] = (
            100.0 * count / total if total else 0.0
        )

    # Strict shift arithmetic excludes exact-zero handling.
    shift_count = int(counts["dyadic_count"] + counts["pot_count"])
    zero_count = int(counts["zero_count"])
    fallback_count = int(counts["fallback_count"])

    result["shift_arithmetic_updates"] = shift_count
    result["shift_arithmetic_pct"] = 100.0 * shift_count / total
    result["zero_or_shift_pct"] = (
        100.0 * (shift_count + zero_count) / total
    )
    result["fallback_pct_recomputed"] = (
        100.0 * fallback_count / total
    )

    return result


# Overall for each precision and sequence length.
rows = []
for (frac, seq), group in df.groupby(["frac_bits", "seq_len"]):
    rows.append(
        mode_breakdown(group, f"F{frac} / {seq} tokens")
        | {"frac_bits": frac, "seq_len": seq}
    )

summary = pd.DataFrame(rows)

print("\nBREAKDOWN BY PRECISION AND SEQUENCE LENGTH")
display(summary.round(4))

# ---------------------------------------------------------
# 3. Overall breakdown across all 20 captures
# ---------------------------------------------------------
overall_rows = []
for frac, group in df.groupby("frac_bits"):
    overall_rows.append(
        mode_breakdown(group, f"F{frac} / ALL LENGTHS")
        | {"frac_bits": frac, "seq_len": "ALL"}
    )

overall_rows.append(
    mode_breakdown(df, "ALL PRECISIONS / ALL LENGTHS")
    | {"frac_bits": "ALL", "seq_len": "ALL"}
)

overall = pd.DataFrame(overall_rows)

print("\nOVERALL MODE BREAKDOWN")
display(overall.round(4))

# ---------------------------------------------------------
# 4. Per-layer breakdown, aggregated over documents
# ---------------------------------------------------------
layer_rows = []

for (frac, seq, layer), group in df.groupby(
    ["frac_bits", "seq_len", "layer"]
):
    row = mode_breakdown(
        group, f"F{frac} / {seq} / Layer {layer}"
    )
    row.update({
        "frac_bits": frac,
        "seq_len": seq,
        "layer": layer,
        "documents": group["doc_id"].nunique(),
    })
    layer_rows.append(row)

layer_summary = pd.DataFrame(layer_rows)

print("\nPER-LAYER BREAKDOWN — FIRST 30 ROWS")
display(
    layer_summary[
        [
            "frac_bits", "seq_len", "layer", "documents",
            "total_updates",
            "dyadic_updates", "dyadic_pct",
            "pot_updates", "pot_pct",
            "zero_updates", "zero_pct",
            "fallback_updates", "fallback_pct",
            "shift_arithmetic_pct",
        ]
    ].round(3).head(30)
)

# ---------------------------------------------------------
# 5. Compare F18 and F20 mode decisions
# ---------------------------------------------------------
pivot = summary.pivot(
    index="seq_len",
    columns="frac_bits",
    values=[
        "dyadic_pct",
        "pot_pct",
        "zero_pct",
        "fallback_pct",
        "shift_arithmetic_pct",
    ],
)

print("\nF18 VS F20 MODE-PERCENTAGE COMPARISON")
display(pivot.round(5))

# ---------------------------------------------------------
# 6. Save all reports
# ---------------------------------------------------------
OUT = CTRL / "mode_breakdown"
OUT.mkdir(exist_ok=True)

summary.to_csv(OUT / "mode_breakdown_by_length.csv", index=False)
overall.to_csv(OUT / "mode_breakdown_overall.csv", index=False)
layer_summary.to_csv(OUT / "mode_breakdown_per_layer.csv", index=False)

print("\nSaved reports to:", OUT)
print("  mode_breakdown_by_length.csv")
print("  mode_breakdown_overall.csv")
print("  mode_breakdown_per_layer.csv")


VALIDATION
Layer records: 960
Records with count mismatch: 0
Total recurrence updates: 138240

BREAKDOWN BY PRECISION AND SEQUENCE LENGTH


,group,total_updates,dyadic_updates,dyadic_pct,pot_updates,pot_pct,zero_updates,zero_pct,fallback_updates,fallback_pct,shift_arithmetic_updates,shift_arithmetic_pct,zero_or_shift_pct,fallback_pct_recomputed,frac_bits,seq_len
0,F18 / 1024 tokens,23040,1773,7.6953,3525,15.2995,7436,32.2743,10306,44.7309,5298,22.9948,55.2691,44.7309,18,1024
1,F18 / 2048 tokens,46080,2661,5.7747,6271,13.6089,14644,31.7795,22504,48.8368,8932,19.3837,51.1632,48.8368,18,2048
2,F20 / 1024 tokens,23040,1773,7.6953,3525,15.2995,7436,32.2743,10306,44.7309,5298,22.9948,55.2691,44.7309,20,1024
3,F20 / 2048 tokens,46080,2661,5.7747,6271,13.6089,14644,31.7795,22504,48.8368,8932,19.3837,51.1632,48.8368,20,2048



OVERALL MODE BREAKDOWN


,group,total_updates,dyadic_updates,dyadic_pct,pot_updates,pot_pct,zero_updates,zero_pct,fallback_updates,fallback_pct,shift_arithmetic_updates,shift_arithmetic_pct,zero_or_shift_pct,fallback_pct_recomputed,frac_bits,seq_len
0,F18 / ALL LENGTHS,69120,4434,6.4149,9796,14.1725,22080,31.9444,32810,47.4682,14230,20.5874,52.5318,47.4682,18,ALL
1,F20 / ALL LENGTHS,69120,4434,6.4149,9796,14.1725,22080,31.9444,32810,47.4682,14230,20.5874,52.5318,47.4682,20,ALL
2,ALL PRECISIONS / ALL LENGTHS,138240,8868,6.4149,19592,14.1725,44160,31.9444,65620,47.4682,28460,20.5874,52.5318,47.4682,ALL,ALL



PER-LAYER BREAKDOWN — FIRST 30 ROWS


,frac_bits,seq_len,layer,documents,total_updates,dyadic_updates,dyadic_pct,pot_updates,pot_pct,zero_updates,zero_pct,fallback_updates,fallback_pct,shift_arithmetic_pct
0,18,1024,0,10,960,0,0.000,687,71.562,0,0.000,273,28.438,71.562
1,18,1024,1,10,960,4,0.417,70,7.292,480,50.000,406,42.292,7.708
2,18,1024,2,10,960,156,16.250,86,8.958,185,19.271,533,55.521,25.208
3,18,1024,3,10,960,12,1.250,168,17.500,346,36.042,434,45.208,18.750
4,18,1024,4,10,960,243,25.312,69,7.188,135,14.062,513,53.438,32.500
5,18,1024,5,10,960,42,4.375,94,9.792,567,59.062,257,26.771,14.167
6,18,1024,6,10,960,120,12.500,187,19.479,98,10.208,555,57.812,31.979
7,18,1024,7,10,960,0,0.000,156,16.250,169,17.604,635,66.146,16.250
8,18,1024,8,10,960,0,0.000,205,21.354,63,6.562,692,72.083,21.354
9,18,1024,9,10,960,1,0.104,24,2.500,640,66.667,295,30.729,2.604



F18 VS F20 MODE-PERCENTAGE COMPARISON


dyadic_pct            pot_pct            zero_pct            \
frac_bits         18       20        18        20        18        20   
seq_len                                                                 
1024         7.69531  7.69531  15.29948  15.29948  32.27431  32.27431   
2048         5.77474  5.77474  13.60894  13.60894  31.77951  31.77951   

          fallback_pct           shift_arithmetic_pct            
frac_bits           18        20                   18        20  
seq_len                                                          
1024          44.73090  44.73090             22.99479  22.99479  
2048          48.83681  48.83681             19.38368  19.38368


Saved reports to: /kaggle/working/shiftscan_final/state_passing/multisequence_validation/fixed_point_controller_validation_v2/mode_breakdown
  mode_breakdown_by_length.csv
  mode_breakdown_overall.csv
  mode_breakdown_per_layer.csv


In [163]:

from pathlib import Path
import pandas as pd
import numpy as np
import torch

ROOT = Path(
    "/kaggle/working/shiftscan_final/state_passing/"
    "multisequence_validation"
)
CTRL = ROOT / "fixed_point_controller_validation_v2"

saved = pd.read_csv(CTRL / "per_layer_controller_results.csv")
saved = saved[saved.frac_bits == 20].copy()

rows = []

for path in sorted(ROOT.glob("doc_*_seq_*.pt")):
    obj = torch.load(path, map_location="cpu", weights_only=False)

    for rec in obj["records"]:
        log_decay = rec["decay_log"].detach().cpu()

        # Test A: preserve the captured tensor's dtype.
        decay_torch_native = torch.exp(log_decay)
        zeros_torch_native = int((decay_torch_native == 0).sum().item())

        # Test B: explicitly compute exponential in float32.
        decay_torch_f32 = torch.exp(log_decay.float())
        zeros_torch_f32 = int((decay_torch_f32 == 0).sum().item())

        # Test C: compute exponential in float64.
        decay_np_f64 = np.exp(log_decay.double().numpy())
        zeros_numpy_f64 = int((decay_np_f64 == 0).sum())

        # Also count values that are positive in float64 but underflow
        # to zero in float32.
        float32_only_zeros = int(
            ((decay_torch_f32 == 0) &
             (torch.from_numpy(decay_np_f64) > 0)).sum().item()
        )

        rows.append({
            "file": path.name,
            "layer": int(rec["layer"]),
            "doc_id": int(rec["doc_id"]),
            "seq_len": int(rec["seq_len"]),
            "log_dtype": str(log_decay.dtype),
            "zero_native": zeros_torch_native,
            "zero_torch_f32": zeros_torch_f32,
            "zero_numpy_f64": zeros_numpy_f64,
            "float32_only_zeros": float32_only_zeros,
        })

diag = pd.DataFrame(rows)

check = diag.merge(
    saved[
        ["file", "layer", "doc_id", "seq_len", "zero_count"]
    ],
    on=["file", "layer", "doc_id", "seq_len"],
    how="inner",
    validate="one_to_one",
)

print("Records checked:", len(check))

print("\nZERO COUNT AGREEMENT")
print(
    "Native torch matches saved:",
    int((check.zero_native == check.zero_count).sum()),
    "/", len(check)
)
print(
    "Explicit torch FP32 matches saved:",
    int((check.zero_torch_f32 == check.zero_count).sum()),
    "/", len(check)
)
print(
    "NumPy FP64 matches saved:",
    int((check.zero_numpy_f64 == check.zero_count).sum()),
    "/", len(check)
)

print("\nAGGREGATE ZERO COUNTS")
display(
    check[
        [
            "zero_count",
            "zero_native",
            "zero_torch_f32",
            "zero_numpy_f64",
            "float32_only_zeros",
        ]
    ].sum().to_frame("total")
)

print("\nEXAMPLES WHERE FLOAT32 AND FLOAT64 DIFFER")
different = check[
    check.zero_torch_f32 != check.zero_numpy_f64
]
display(different.head(20))

print("\nLOG-DECAY RANGE AND DTYPE")
print("Dtypes:", diag.log_dtype.value_counts().to_dict())

# Save diagnostic.
out = CTRL / "fallback_diagnosis"
out.mkdir(exist_ok=True)
check.to_csv(out / "zero_underflow_precision_diagnostic.csv", index=False)

print("\nSaved:", out / "zero_underflow_precision_diagnostic.csv")


Records checked: 480

ZERO COUNT AGREEMENT
Native torch matches saved: 480 / 480
Explicit torch FP32 matches saved: 480 / 480
NumPy FP64 matches saved: 45 / 480

AGGREGATE ZERO COUNTS


,total
zero_count,22080
zero_native,22080
zero_torch_f32,22080
zero_numpy_f64,4211
float32_only_zeros,17869



EXAMPLES WHERE FLOAT32 AND FLOAT64 DIFFER


,file,layer,doc_id,seq_len,log_dtype,zero_native,zero_torch_f32,zero_numpy_f64,float32_only_zeros,zero_count
1,doc_000_seq_1024.pt,1,0,1024,torch.float32,48,48,0,48,48
2,doc_000_seq_1024.pt,2,0,1024,torch.float32,17,17,4,13,17
3,doc_000_seq_1024.pt,3,0,1024,torch.float32,33,33,0,33,33
4,doc_000_seq_1024.pt,4,0,1024,torch.float32,16,16,12,4,16
5,doc_000_seq_1024.pt,5,0,1024,torch.float32,54,54,0,54,54
6,doc_000_seq_1024.pt,6,0,1024,torch.float32,9,9,4,5,9
7,doc_000_seq_1024.pt,7,0,1024,torch.float32,16,16,0,16,16
8,doc_000_seq_1024.pt,8,0,1024,torch.float32,2,2,0,2,2
9,doc_000_seq_1024.pt,9,0,1024,torch.float32,63,63,0,63,63
10,doc_000_seq_1024.pt,10,0,1024,torch.float32,26,26,11,15,26



LOG-DECAY RANGE AND DTYPE
Dtypes: {'torch.float32': 480}

Saved: /kaggle/working/shiftscan_final/state_passing/multisequence_validation/fixed_point_controller_validation_v2/fallback_diagnosis/zero_underflow_precision_diagnostic.csv


In [165]:

# ================================================================
# SHIFT-SCAN: FALLBACK-REASON DIAGNOSIS — CORRECTED FP32 REPLAY
#
# Replays the F20 fixed-point adaptive controller over all saved
# Mamba-2 state-passing captures.
#
# Critical correction:
#   decay = exp(decay_log) must be evaluated in FP32, matching
#   the original controller. Converting to FP64 before exp()
#   changes underflow-to-zero decisions.
#
# Outputs:
#   1. Exact validation of all four arithmetic-mode counts
#   2. Fallback reason counts overall and by sequence length
#   3. Per-layer diagnostic CSV
#   4. Aggregate diagnostic CSVs
# ================================================================

import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path

# -----------------------------
# 1. Paths and configuration
# -----------------------------

BASE = Path(
    "/kaggle/working/shiftscan_final/state_passing/"
    "multisequence_validation"
)

CAPTURE_DIR = BASE
RESULT_DIR = (
    BASE / "fixed_point_controller_validation_v2"
)

SAVED_RESULTS = RESULT_DIR / "per_layer_controller_results.csv"

OUT_DIR = RESULT_DIR / "fallback_diagnosis_corrected"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FRAC_BITS = 20
SCALE = 1 << FRAC_BITS
ERROR_THRESHOLD = 0.001

GRID_CSV = Path(
    "/kaggle/working/shiftscan_final/"
    "canonical_dyadic_grid_definitions.csv"
)

# The controller's original modes:
# 0 = exact floating-point fallback
# 1 = three-term Dyadic2 shift-subtract
# 2 = direct power-of-two shift
# 3 = exact zero under FP32 exp()

POT_K_MIN = 4
POT_K_MAX = 30

print("Configuration")
print("Fraction bits:", FRAC_BITS)
print("Scale:", SCALE)
print("Error threshold:", ERROR_THRESHOLD)


# -----------------------------
# 2. Load canonical Dyadic2 grid
# -----------------------------

grid_df = pd.read_csv(GRID_CSV)

grid_df = grid_df[grid_df["grid"] == "K1_14_T3"].copy()

if grid_df.empty:
    raise RuntimeError("K1_14_T3 grid was not found.")

# Deduplicate numerical decay values.
grid_df = (
    grid_df.sort_values(
        ["decay", "canonical_exponents"]
    )
    .drop_duplicates(subset=["decay"])
    .reset_index(drop=True)
)

grid_decay = grid_df["decay"].to_numpy(dtype=np.float64)

def parse_exponents(value):
    """
    Supports canonical exponent strings such as:
       (3, 4, 5)
       [3, 4, 5]
       3,4,5
    """
    if isinstance(value, (tuple, list, np.ndarray)):
        return tuple(int(x) for x in value)

    s = str(value).strip()
    s = s.replace("(", "").replace(")", "")
    s = s.replace("[", "").replace("]", "")

    return tuple(
        int(x.strip())
        for x in s.split(",")
        if x.strip()
    )

grid_exponents = np.asarray(
    [
        parse_exponents(v)
        for v in grid_df["canonical_exponents"]
    ],
    dtype=np.int64
)

if grid_exponents.ndim != 2 or grid_exponents.shape[1] != 3:
    raise RuntimeError(
        f"Expected three exponents per K1_14_T3 value; "
        f"got shape {grid_exponents.shape}"
    )

GRID_MIN = float(grid_decay.min())
GRID_MAX = float(grid_decay.max())

print("\nCanonical grid")
print("Unique values:", len(grid_decay))
print("Minimum decay:", GRID_MIN)
print("Maximum decay:", GRID_MAX)
print("Exponent shape:", grid_exponents.shape)


# -----------------------------
# 3. Helpers
# -----------------------------

def round_away_from_zero(x):
    """
    Round to nearest integer, ties away from zero.
    """
    x = np.asarray(x, dtype=np.float64)

    return (
        np.sign(x) * np.floor(np.abs(x) + 0.5)
    ).astype(np.int64)


def quantize_input_contribution(x):
    """
    Match the controller's fixed-point input quantization.
    """
    return round_away_from_zero(
        np.asarray(x, dtype=np.float64) * SCALE
    )


def shift_round_signed(h, k):
    """
    Signed integer right shift with ties away from zero.
    """
    h = np.asarray(h, dtype=np.int64)

    divisor = 1 << int(k)
    abs_h = np.abs(h)

    rounded = (
        abs_h + divisor // 2
    ) // divisor

    return np.sign(h).astype(np.int64) * rounded


def apply_dyadic2_integer(h, exponents):
    """
    Apply:
        H_new = H - sum_i round(H / 2^k_i)

    The three terms are rounded independently, matching
    the shift-subtract arithmetic used by the controller.
    """
    out = np.asarray(h, dtype=np.int64).copy()

    for k in exponents:
        out -= shift_round_signed(h, int(k))

    return out


def choose_nearest_pot(a):
    """
    Choose nearest power-of-two coefficient in log space.

    Returns:
        q, exponent
    """
    if a <= 0:
        return 0.0, None

    log2a = np.log2(a)

    k = int(np.rint(-log2a))
    k = int(np.clip(k, POT_K_MIN, POT_K_MAX))

    return float(2.0 ** (-k)), k


# -----------------------------
# 4. Load saved controller rows
# -----------------------------

saved = pd.read_csv(SAVED_RESULTS)

saved_f20 = saved[
    saved["frac_bits"].astype(int) == FRAC_BITS
].copy()

if len(saved_f20) != 480:
    raise RuntimeError(
        f"Expected 480 F20 layer records, found {len(saved_f20)}"
    )

saved_f20["file"] = saved_f20["file"].astype(str)

saved_lookup = {
    (str(row["file"]), int(row["layer"])): row
    for _, row in saved_f20.iterrows()
}

print("\nSaved F20 records:", len(saved_f20))


# -----------------------------
# 5. Replay all capture records
# -----------------------------

diagnostic_rows = []

total_records = 0
missing_saved_records = []

for capture_path in sorted(
    CAPTURE_DIR.glob("doc_*_seq_*.pt")
):

    obj = torch.load(
        capture_path,
        map_location="cpu",
        weights_only=False
    )

    records = obj["records"]

    for rec in records:

        total_records += 1

        layer = int(rec["layer"])
        key = (capture_path.name, layer)

        if key not in saved_lookup:
            missing_saved_records.append(key)
            continue

        saved_row = saved_lookup[key]

        # ------------------------------------------------
        # A. Load captured state-passing tensors
        # ------------------------------------------------

        states_t = rec["states"].detach().cpu().float()
        log_decay_t = (
            rec["decay_log"]
            .detach()
            .cpu()
            .float()
        )

        # states expected as:
        # [batch, chunks, heads, state_dim]
        #
        # This experiment uses batch size 1.
        if states_t.ndim != 4 or states_t.shape[0] != 1:
            raise RuntimeError(
                f"Unexpected states shape in {key}: "
                f"{tuple(states_t.shape)}"
            )

        states = states_t[0].numpy().astype(np.float64)

        # decay_log expected as:
        # [batch, heads, chunks] or [heads, chunks]
        log_decay = log_decay_t

        if log_decay.ndim == 3:
            if log_decay.shape[0] != 1:
                raise RuntimeError(
                    f"Unexpected decay batch dimension in {key}"
                )
            log_decay = log_decay[0]

        if log_decay.ndim != 2:
            raise RuntimeError(
                f"Unexpected decay_log shape in {key}: "
                f"{tuple(log_decay.shape)}"
            )

        # Convert to [chunks, heads].
        log_decay = log_decay.transpose(0, 1).contiguous()

        chunks, heads, state_dim = states.shape

        if tuple(log_decay.shape) != (chunks, heads):
            raise RuntimeError(
                f"Decay/state shape mismatch in {key}: "
                f"decay={tuple(log_decay.shape)}, "
                f"states={(chunks, heads, state_dim)}"
            )

        # ------------------------------------------------
        # B. CRITICAL: FP32 exponential semantics
        # ------------------------------------------------

        # Do NOT calculate np.exp(log_decay.double()).
        # The original controller computes torch.exp in FP32.
        #
        # Convert to FP64 only AFTER the FP32 exponential,
        # preserving FP32 underflow-to-zero decisions.
        decay_fp32 = torch.exp(log_decay.float())

        decay = (
            decay_fp32
            .cpu()
            .numpy()
            .astype(np.float64)
        )

        # ------------------------------------------------
        # C. Quantize new-state contributions
        # ------------------------------------------------

        N_int = quantize_input_contribution(states)

        # ------------------------------------------------
        # D. Fixed-point adaptive replay
        # ------------------------------------------------

        H_int = np.zeros(
            (heads, state_dim),
            dtype=np.int64
        )

        counts = {
            "dyadic_count": 0,
            "pot_count": 0,
            "zero_count": 0,
            "fallback_count": 0,
        }

        fallback_reasons = {
            "threshold_dyadic": 0,
            "threshold_pot": 0,
            "above_dyadic_max": 0,
            "below_pot_min": 0,
            "invalid_decay": 0,
        }

        # Reconstruct recurrence one chunk at a time.
        for c in range(chunks):

            H_real = H_int.astype(np.float64) / SCALE

            for h in range(heads):

                a = float(decay[c, h])

                # ----------------------------------------
                # Mode 3: exact zero under FP32 exp()
                # ----------------------------------------

                if a == 0.0:
                    counts["zero_count"] += 1

                    # Exact zero decay:
                    # H_new = N
                    H_int[h] = N_int[c, h]
                    continue

                # ----------------------------------------
                # Invalid values
                # ----------------------------------------

                if not np.isfinite(a) or a < 0.0:
                    counts["fallback_count"] += 1
                    fallback_reasons["invalid_decay"] += 1

                    # Exact fallback using FP32 decay value.
                    H_int[h] = round_away_from_zero(
                        a * H_int[h]
                    ) + N_int[c, h]
                    continue

                # Current fixed-point state magnitude.
                state_max = float(
                    np.max(np.abs(H_real[h]))
                )

                selected_mode = None
                selected_q = None
                selected_exponents = None
                selected_k = None

                # ----------------------------------------
                # Mode 1: Dyadic2 grid
                # ----------------------------------------

                if GRID_MIN <= a <= GRID_MAX:

                    idx = int(
                        np.argmin(np.abs(grid_decay - a))
                    )

                    q = float(grid_decay[idx])
                    exponents = grid_exponents[idx]

                    predicted_local_error = (
                        abs(q - a) * state_max
                    )

                    if predicted_local_error <= ERROR_THRESHOLD:
                        selected_mode = "dyadic"
                        selected_q = q
                        selected_exponents = exponents
                    else:
                        fallback_reasons[
                            "threshold_dyadic"
                        ] += 1

                # ----------------------------------------
                # Mode 2: direct PoT for small positive a
                # ----------------------------------------

                elif 0.0 < a < GRID_MIN:

                    pot_min = 2.0 ** (-POT_K_MAX)

                    if a >= pot_min:

                        q, k = choose_nearest_pot(a)

                        predicted_local_error = (
                            abs(q - a) * state_max
                        )

                        if predicted_local_error <= ERROR_THRESHOLD:
                            selected_mode = "pot"
                            selected_q = q
                            selected_k = k
                        else:
                            fallback_reasons[
                                "threshold_pot"
                            ] += 1

                    else:
                        fallback_reasons[
                            "below_pot_min"
                        ] += 1

                # ----------------------------------------
                # Positive decay above grid maximum
                # ----------------------------------------

                elif a > GRID_MAX:
                    fallback_reasons[
                        "above_dyadic_max"
                    ] += 1

                # ----------------------------------------
                # Remaining invalid/out-of-range values
                # ----------------------------------------

                else:
                    fallback_reasons[
                        "invalid_decay"
                    ] += 1

                # ----------------------------------------
                # Apply selected arithmetic mode
                # ----------------------------------------

                if selected_mode == "dyadic":

                    H_int[h] = apply_dyadic2_integer(
                        H_int[h],
                        selected_exponents
                    )

                    counts["dyadic_count"] += 1

                elif selected_mode == "pot":

                    H_int[h] = shift_round_signed(
                        H_int[h],
                        selected_k
                    )

                    counts["pot_count"] += 1

                else:

                    # Exact floating-point fallback.
                    # This is intentionally not shift-only.
                    #
                    # Match the FP32 decay used by the
                    # original controller, then round the
                    # state contribution into fixed-point.
                    decay_for_update = np.float32(a)

                    H_int[h] = round_away_from_zero(
                        float(decay_for_update) * H_int[h]
                    )

                    counts["fallback_count"] += 1

                # Add the new-state contribution.
                H_int[h] += N_int[c, h]

        # ------------------------------------------------
        # E. Validate mode counts against saved result
        # ------------------------------------------------

        row = {
            "file": capture_path.name,
            "layer": layer,
            "doc_id": int(saved_row["doc_id"]),
            "seq_len": int(saved_row["seq_len"]),
            "frac_bits": FRAC_BITS,
            "chunks": chunks,
            "heads": heads,
            "state_dim": state_dim,
            **counts,
            **fallback_reasons,
        }

        expected_columns = [
            "dyadic_count",
            "pot_count",
            "zero_count",
            "fallback_count",
        ]

        mismatches = []

        for col in expected_columns:
            expected = int(saved_row[col])
            actual = int(row[col])

            if expected != actual:
                mismatches.append(
                    f"{col}: replay={actual}, saved={expected}"
                )

        row["mode_counts_match"] = len(mismatches) == 0
        row["mismatch_detail"] = "; ".join(mismatches)

        diagnostic_rows.append(row)


# -----------------------------
# 6. Build diagnostic dataframe
# -----------------------------

diagnostics = pd.DataFrame(diagnostic_rows)

if missing_saved_records:
    raise RuntimeError(
        f"Missing {len(missing_saved_records)} saved records. "
        f"Examples: {missing_saved_records[:5]}"
    )

if len(diagnostics) != 480:
    raise RuntimeError(
        f"Expected 480 replayed records, got {len(diagnostics)}"
    )

diagnostics.to_csv(
    OUT_DIR / "fallback_diagnosis_per_layer.csv",
    index=False
)

print("\nReplay records:", len(diagnostics))


# -----------------------------
# 7. Strict arithmetic-mode validation
# -----------------------------

mode_columns = [
    "dyadic_count",
    "pot_count",
    "zero_count",
    "fallback_count",
]

mismatch_df = diagnostics[
    ~diagnostics["mode_counts_match"]
].copy()

print("\n" + "=" * 65)
print("ARITHMETIC MODE COUNT VALIDATION")
print("=" * 65)

print("Records checked:", len(diagnostics))
print("Records matching all four counts:",
      int(diagnostics["mode_counts_match"].sum()))
print("Records with mismatch:", len(mismatch_df))

if len(mismatch_df):
    print("\nFirst mismatches:")
    display(
        mismatch_df[
            [
                "file",
                "layer",
                "dyadic_count",
                "pot_count",
                "zero_count",
                "fallback_count",
                "mismatch_detail",
            ]
        ].head(20)
    )

    raise RuntimeError(
        "Replay mode counts do not match the saved controller. "
        "Do not interpret fallback reasons until this is resolved."
    )

print("SUCCESS: all four mode counts match in all 480 records.")


# -----------------------------
# 8. Aggregate fallback reasons
# -----------------------------

reason_columns = [
    "threshold_dyadic",
    "threshold_pot",
    "above_dyadic_max",
    "below_pot_min",
    "invalid_decay",
]

total_updates = int(
    diagnostics[
        [
            "dyadic_count",
            "pot_count",
            "zero_count",
            "fallback_count",
        ]
    ].to_numpy().sum()
)

reason_totals = {
    col: int(diagnostics[col].sum())
    for col in reason_columns
}

fallback_total = int(diagnostics["fallback_count"].sum())

reason_summary = pd.DataFrame(
    [
        {
            "fallback_reason": reason,
            "count": count,
            "percent_of_all_updates": (
                100.0 * count / total_updates
            ),
            "percent_of_fallbacks": (
                100.0 * count / fallback_total
                if fallback_total else 0.0
            ),
        }
        for reason, count in reason_totals.items()
    ]
)

reason_summary.to_csv(
    OUT_DIR / "fallback_reason_summary.csv",
    index=False
)

print("\n" + "=" * 65)
print("OVERALL FALLBACK REASONS")
print("=" * 65)

display(reason_summary)


# -----------------------------
# 9. Arithmetic mode totals
# -----------------------------

mode_totals = {
    "Dyadic2 shift-subtract": int(
        diagnostics["dyadic_count"].sum()
    ),
    "Direct PoT shift": int(
        diagnostics["pot_count"].sum()
    ),
    "Exact zero under FP32 exp": int(
        diagnostics["zero_count"].sum()
    ),
    "Floating-point fallback": fallback_total,
}

mode_summary = pd.DataFrame(
    [
        {
            "mode": name,
            "count": count,
            "percent_of_all_updates": (
                100.0 * count / total_updates
            ),
        }
        for name, count in mode_totals.items()
    ]
)

mode_summary.to_csv(
    OUT_DIR / "arithmetic_mode_summary.csv",
    index=False
)

print("\n" + "=" * 65)
print("ARITHMETIC MODE TOTALS")
print("=" * 65)

display(mode_summary)

print("Total recurrence updates:", total_updates)

# Strict shift arithmetic excludes exact-zero handling.
strict_shift_count = (
    mode_totals["Dyadic2 shift-subtract"]
    + mode_totals["Direct PoT shift"]
)

print(
    "Strict shift-arithmetic coverage: "
    f"{100.0 * strict_shift_count / total_updates:.4f}%"
)


# -----------------------------
# 10. Breakdown by sequence length
# -----------------------------

length_rows = []

for seq_len, group in diagnostics.groupby("seq_len"):

    group_updates = int(
        group[mode_columns].to_numpy().sum()
    )

    group_fallback = int(group["fallback_count"].sum())

    for reason in reason_columns:

        count = int(group[reason].sum())

        length_rows.append(
            {
                "seq_len": int(seq_len),
                "fallback_reason": reason,
                "count": count,
                "percent_of_all_updates": (
                    100.0 * count / group_updates
                    if group_updates else 0.0
                ),
                "percent_of_fallbacks": (
                    100.0 * count / group_fallback
                    if group_fallback else 0.0
                ),
            }
        )

length_summary = pd.DataFrame(length_rows)

length_summary.to_csv(
    OUT_DIR / "fallback_reason_by_length.csv",
    index=False
)

print("\n" + "=" * 65)
print("FALLBACK REASONS BY SEQUENCE LENGTH")
print("=" * 65)

display(
    length_summary.pivot(
        index="fallback_reason",
        columns="seq_len",
        values="percent_of_fallbacks"
    ).round(3)
)


# -----------------------------
# 11. Layer-level fallback diagnosis
# -----------------------------

layer_summary = (
    diagnostics
    .groupby("layer", as_index=False)
    .agg(
        updates=("fallback_count", "size"),
        dyadic_count=("dyadic_count", "sum"),
        pot_count=("pot_count", "sum"),
        zero_count=("zero_count", "sum"),
        fallback_count=("fallback_count", "sum"),
        threshold_dyadic=("threshold_dyadic", "sum"),
        threshold_pot=("threshold_pot", "sum"),
        above_dyadic_max=("above_dyadic_max", "sum"),
        below_pot_min=("below_pot_min", "sum"),
        invalid_decay=("invalid_decay", "sum"),
    )
)

layer_summary["total_updates"] = (
    layer_summary["dyadic_count"]
    + layer_summary["pot_count"]
    + layer_summary["zero_count"]
    + layer_summary["fallback_count"]
)

layer_summary["fallback_pct"] = (
    100.0
    * layer_summary["fallback_count"]
    / layer_summary["total_updates"]
)

layer_summary["strict_shift_pct"] = (
    100.0
    * (
        layer_summary["dyadic_count"]
        + layer_summary["pot_count"]
    )
    / layer_summary["total_updates"]
)

layer_summary.to_csv(
    OUT_DIR / "fallback_diagnosis_by_layer.csv",
    index=False
)

print("\n" + "=" * 65)
print("LAYERS WITH HIGHEST FALLBACK PERCENTAGE")
print("=" * 65)

display(
    layer_summary.sort_values(
        "fallback_pct",
        ascending=False
    ).head(12)
)


# -----------------------------
# 12. Save experiment metadata
# -----------------------------

metadata = {
    "experiment": "corrected_fallback_reason_diagnosis",
    "frac_bits": FRAC_BITS,
    "scale": SCALE,
    "error_threshold": ERROR_THRESHOLD,
    "grid": "K1_14_T3",
    "unique_grid_values": int(len(grid_decay)),
    "grid_min_decay": GRID_MIN,
    "grid_max_decay": GRID_MAX,
    "pot_k_min": POT_K_MIN,
    "pot_k_max": POT_K_MAX,
    "records_checked": int(len(diagnostics)),
    "mode_count_mismatches": int(len(mismatch_df)),
    "total_updates": total_updates,
    "fp32_exp_semantics": True,
    "fp64_exp_before_zero_classification": False,
    "all_mode_counts_validated": True,
    "fallback_reason_counts": reason_totals,
}

with open(
    OUT_DIR / "fallback_diagnosis_metadata.json",
    "w"
) as f:
    json.dump(metadata, f, indent=2)


# -----------------------------
# 13. Final report
# -----------------------------

print("\n" + "=" * 65)
print("FALLBACK DIAGNOSIS COMPLETE")
print("=" * 65)

print("All 480 layer records passed mode-count validation.")
print("Total updates:", total_updates)
print("Floating fallback updates:", fallback_total)
print(
    "Fallback percentage:",
    round(100.0 * fallback_total / total_updates, 4)
)

print("\nSaved files:")
for path in sorted(OUT_DIR.glob("*")):
    print(path)

print("\nImportant interpretation:")
print(
    "Exact-zero cases are classified using FP32 exp(decay_log). "
    "They are not counted as floating-point fallbacks."
)
print(
    "The fallback-reason categories are diagnostic classifications "
    "of the controller's decisions; exact fallback still uses "
    "floating-point multiplication."
)


Configuration
Fraction bits: 20
Scale: 1048576
Error threshold: 0.001

Canonical grid
Unique values: 467
Minimum decay: 0.125
Maximum decay: 0.99981689453125
Exponent shape: (467, 3)

Saved F20 records: 480

Replay records: 480

ARITHMETIC MODE COUNT VALIDATION
Records checked: 480
Records matching all four counts: 480
Records with mismatch: 0
SUCCESS: all four mode counts match in all 480 records.

OVERALL FALLBACK REASONS


,fallback_reason,count,percent_of_all_updates,percent_of_fallbacks
0,threshold_dyadic,10973,15.875289,33.444072
1,threshold_pot,4795,6.937211,14.614447
2,above_dyadic_max,0,0.000000,0.000000
3,below_pot_min,17042,24.655671,51.941481
4,invalid_decay,0,0.000000,0.000000



ARITHMETIC MODE TOTALS


,mode,count,percent_of_all_updates
0,Dyadic2 shift-subtract,4434,6.414931
1,Direct PoT shift,9796,14.172454
2,Exact zero under FP32 exp,22080,31.944444
3,Floating-point fallback,32810,47.468171


Total recurrence updates: 69120
Strict shift-arithmetic coverage: 20.5874%

FALLBACK REASONS BY SEQUENCE LENGTH


seq_len,1024,2048
fallback_reason,,
above_dyadic_max,0.000,0.000
below_pot_min,54.444,50.795
invalid_decay,0.000,0.000
threshold_dyadic,31.758,34.216
threshold_pot,13.798,14.988



LAYERS WITH HIGHEST FALLBACK PERCENTAGE


,layer,updates,dyadic_count,pot_count,zero_count,fallback_count,threshold_dyadic,threshold_pot,above_dyadic_max,below_pot_min,invalid_decay,total_updates,fallback_pct,strict_shift_pct
12,12,20,4,85,556,2235,98,74,0,2063,0,2880,77.604167,3.090278
18,18,20,1,152,523,2204,83,34,0,2087,0,2880,76.527778,5.312500
8,8,20,0,593,191,2096,17,183,0,1896,0,2880,72.777778,20.590278
7,7,20,2,397,510,1971,102,585,0,1284,0,2880,68.437500,13.854167
6,6,20,268,510,287,1815,1063,314,0,438,0,2880,63.020833,27.013889
15,15,20,35,319,769,1757,209,362,0,1186,0,2880,61.006944,12.291667
2,2,20,377,213,558,1732,1238,92,0,402,0,2880,60.138889,20.486111
11,11,20,0,131,1033,1716,10,182,0,1524,0,2880,59.583333,4.548611
4,4,20,630,175,401,1674,1293,182,0,199,0,2880,58.125000,27.951389
10,10,20,323,158,771,1628,1017,182,0,429,0,2880,56.527778,16.701389



FALLBACK DIAGNOSIS COMPLETE
All 480 layer records passed mode-count validation.
Total updates: 69120
Floating fallback updates: 32810
Fallback percentage: 47.4682

Saved files:
/kaggle/working/shiftscan_final/state_passing/multisequence_validation/fixed_point_controller_validation_v2/fallback_diagnosis_corrected/arithmetic_mode_summary.csv
/kaggle/working/shiftscan_final/state_passing/multisequence_validation/fixed_point_controller_validation_v2/fallback_diagnosis_corrected/fallback_diagnosis_by_layer.csv
/kaggle/working/shiftscan_final/state_passing/multisequence_validation/fixed_point_controller_validation_v2/fallback_diagnosis_corrected/fallback_diagnosis_metadata.json
/kaggle/working/shiftscan_final/state_passing/multisequence_validation/fixed_point_controller_validation_v2/fallback_diagnosis_corrected/fallback_diagnosis_per_layer.csv
/kaggle/working/shiftscan_final/state_passing/multisequence_validation/fixed_point_controller_validation_v2/fallback_diagnosis_corrected/fallback_re


## Fallback-Reason Diagnosis: Multi-Sequence Mamba-2 Evaluation

### Objective

This experiment investigates why the fixed-point adaptive controller uses floating-point fallback instead of shift-based arithmetic when processing actual Mamba-2 state-passing tensors.

The evaluation uses:
- Mamba-2 130M pretrained checkpoint.
- FineWeb-Edu documents.
- 10 documents evaluated at sequence lengths 1024 and 2048.
- 24 model layers per document and sequence length.
- F20 fixed-point representation.
- K1_14_T3 canonical three-term Dyadic2 grid.
- Adaptive local error threshold of 0.001.

A total of 480 layer-level records and 69,120 recurrence updates were evaluated.

### FP32 exponential semantics

The replay uses FP32 exponential evaluation to reproduce the original controller's underflow behavior:

\[
a = \operatorname{float32}(\exp(\Delta))
\]

This distinction is important because FP64 exponential evaluation preserves very small positive values that underflow to zero in FP32.

The corrected replay matched all four original arithmetic-mode counts in all 480 records.

### Arithmetic-mode distribution

| Arithmetic mode | Count | Percentage |
|---|---:|---:|
| Dyadic2 shift-subtract | 4,434 | 6.41% |
| Direct power-of-two shift | 9,796 | 14.17% |
| Exact zero under FP32 exp | 22,080 | 31.94% |
| Floating-point fallback | 32,810 | 47.47% |

Strict shift-arithmetic coverage is:

\[
\frac{4434+9796}{69120}\times100
=20.5874\%
\]

Exact-zero handling is reported separately and is not included in strict shift-arithmetic coverage.

### Reasons for floating-point fallback

| Fallback reason | Count | Percentage of fallbacks |
|---|---:|---:|
| Below minimum PoT range | 17,042 | 51.94% |
| Dyadic2 rejected by error threshold | 10,973 | 33.44% |
| PoT rejected by error threshold | 4,795 | 14.61% |
| Above maximum Dyadic2 range | 0 | 0% |
| Invalid decay | 0 | 0% |

The dominant fallback reason is the lower representability limit of the direct PoT approximation.

The two state-aware threshold rejection categories together account for 48.06% of fallback cases.

### Sequence-length comparison

Among fallback cases:

| Fallback reason | 1024 tokens | 2048 tokens |
|---|---:|---:|
| Below minimum PoT range | 54.44% | 50.80% |
| Dyadic2 threshold rejection | 31.76% | 34.22% |
| PoT threshold rejection | 13.80% | 14.99% |

The longer sequence has a somewhat higher share of threshold-based rejections and a lower share of below-range cases.

### Interpretation

The fallback analysis identifies two main limitations:

1. **Representability limitation:** positive decay values below the minimum supported PoT coefficient cannot be represented by the current shift-based approximation.
2. **State-dependent accuracy limitation:** some otherwise representable coefficients are rejected because their predicted local state-update error exceeds the threshold of 0.001.

No fallback cases were attributed to coefficients above the Dyadic2 grid maximum or invalid decay values.

These findings suggest that extending the small-decay representation and improving the state-aware approximation policy are the most relevant next research directions.

### Scope and limitations

These results concern the Mamba-2 state-passing recurrence, not every multiplication in the complete SSD scan.

Floating-point fallback still uses multiplication. Therefore, the measured 20.59% strict shift-arithmetic coverage must not be interpreted as a 20.59% reduction in total model multiplications or as a GPU speedup.

The evaluation covers 10 documents at two sequence lengths. Broader evaluation and actual fused-kernel benchmarking remain necessary.


In [166]:

import pandas as pd
from pathlib import Path

ROOT = Path(
    "/kaggle/working/shiftscan_final/state_passing/"
    "multisequence_validation/fixed_point_controller_validation_v2/"
    "fallback_diagnosis_corrected"
)

df = pd.read_csv(
    ROOT / "fallback_diagnosis_per_layer.csv"
)

mode_cols = [
    "dyadic_count",
    "pot_count",
    "zero_count",
    "fallback_count",
]

reason_cols = [
    "threshold_dyadic",
    "threshold_pot",
    "above_dyadic_max",
    "below_pot_min",
    "invalid_decay",
]

# Aggregate actual update counts across all documents and lengths.
layer = (
    df.groupby("layer", as_index=False)[mode_cols + reason_cols]
      .sum()
)

layer["total_updates"] = layer[mode_cols].sum(axis=1)

layer["fallback_pct"] = (
    100 * layer["fallback_count"] / layer["total_updates"]
)

layer["strict_shift_pct"] = (
    100 * (
        layer["dyadic_count"] + layer["pot_count"]
    ) / layer["total_updates"]
)

layer["below_pot_pct_of_fallbacks"] = (
    100 * layer["below_pot_min"] /
    layer["fallback_count"].replace(0, float("nan"))
)

layer["threshold_rejection_pct_of_fallbacks"] = (
    100 * (
        layer["threshold_dyadic"] + layer["threshold_pot"]
    ) / layer["fallback_count"].replace(0, float("nan"))
)

layer = layer.sort_values(
    "fallback_pct", ascending=False
)

layer.to_csv(
    ROOT / "corrected_layer_summary.csv",
    index=False
)

print("Corrected layer-wise fallback summary")
display(
    layer[
        [
            "layer",
            "total_updates",
            "dyadic_count",
            "pot_count",
            "zero_count",
            "fallback_count",
            "fallback_pct",
            "strict_shift_pct",
            "below_pot_pct_of_fallbacks",
            "threshold_rejection_pct_of_fallbacks",
        ]
    ].round(3)
)

print("\nOverall consistency check")

print(
    "Total updates:",
    int(layer["total_updates"].sum())
)

print(
    "Total fallbacks:",
    int(layer["fallback_count"].sum())
)

print(
    "Overall strict shift coverage:",
    round(
        100 * (
            layer["dyadic_count"].sum()
            + layer["pot_count"].sum()
        ) / layer["total_updates"].sum(),
        4
    ),
    "%"
)

print(
    "\nSaved:",
    ROOT / "corrected_layer_summary.csv"
)


Corrected layer-wise fallback summary


,layer,total_updates,dyadic_count,pot_count,zero_count,fallback_count,fallback_pct,strict_shift_pct,below_pot_pct_of_fallbacks,threshold_rejection_pct_of_fallbacks
12,12,2880,4,85,556,2235,77.604,3.090,92.304,7.696
18,18,2880,1,152,523,2204,76.528,5.312,94.691,5.309
8,8,2880,0,593,191,2096,72.778,20.590,90.458,9.542
7,7,2880,2,397,510,1971,68.438,13.854,65.145,34.855
6,6,2880,268,510,287,1815,63.021,27.014,24.132,75.868
15,15,2880,35,319,769,1757,61.007,12.292,67.501,32.499
2,2,2880,377,213,558,1732,60.139,20.486,23.210,76.790
11,11,2880,0,131,1033,1716,59.583,4.549,88.811,11.189
4,4,2880,630,175,401,1674,58.125,27.951,11.888,88.112
10,10,2880,323,158,771,1628,56.528,16.701,26.351,73.649



Overall consistency check
Total updates: 69120
Total fallbacks: 32810
Overall strict shift coverage: 20.5874 %

Saved: /kaggle/working/shiftscan_final/state_passing/multisequence_validation/fixed_point_controller_validation_v2/fallback_diagnosis_corrected/corrected_layer_summary.csv



## Layer-Wise Fallback Heterogeneity

The fallback analysis reveals substantial variation in the arithmetic behavior of different Mamba-2 layers.

Across 24 layers, 69,120 recurrence updates were evaluated using F20 fixed-point arithmetic. The overall floating-point fallback rate was 47.47%, while strict shift-arithmetic coverage was 20.59%.

Two primary sources of fallback were identified:

1. **Representability limitation:** positive decay coefficients below the minimum supported power-of-two range.
2. **State-dependent accuracy limitation:** representable coefficients rejected because their predicted local state-update error exceeds the threshold of 0.001.

The representability limitation dominates in layers 12, 18, and 8, where fallback rates exceed 72%.

In contrast, layers 6, 2, 4, 10, and 20 exhibit a larger contribution from state-dependent threshold rejection.

Layer 23 demonstrates comparatively favorable behavior, with a fallback rate of 7.74% and strict shift-arithmetic coverage of 37.99%.

These findings indicate that decay approximation difficulty is layer-dependent. A uniform quantization policy may therefore leave significant opportunities for improvement.

### Research implication

The results motivate an adaptive, layer-aware approximation policy that distinguishes between representability-limited and accuracy-limited decay coefficients.

The next experiment investigates whether sufficiently small positive decay coefficients can be approximated by zero using shift-only arithmetic, subject to the same local error constraint:

\[
|0-a|\max_j|\hat S_j| \leq \epsilon
\]

where \(\epsilon=0.001\).

This would introduce an approximate-zero mode distinct from exact FP32 underflow-to-zero handling.

The experiment must evaluate both recurrence accuracy and the resulting increase in strict shift-arithmetic coverage.


In [167]:

# ================================================================
# SHIFT-SCAN: STATE-AWARE APPROXIMATE-ZERO EXPERIMENT
#
# Baseline:
#   Existing F20 adaptive controller.
#
# Candidate:
#   Same controller, plus an approximate-zero mode for positive
#   decay coefficients below 2^-30.
#
# Approximate-zero acceptance:
#   a * max(abs(current_fixed_point_state)) <= 0.001
#
# Evaluation:
#   20 real Mamba-2 state-passing captures
#   10 FineWeb-Edu documents
#   Sequence lengths 1024 and 2048
#   24 layers
#   F20 fixed-point arithmetic
#
# Important:
#   FP32 exp is used for controller decisions.
#   FP64 exp is used only for reference-error measurement.
#   Baseline mode counts are validated against saved results.
# ================================================================

import json
import numpy as np
import pandas as pd
import torch

from pathlib import Path


# ------------------------------------------------
# 1. Configuration
# ------------------------------------------------

BASE = Path(
    "/kaggle/working/shiftscan_final/state_passing/"
    "multisequence_validation"
)

RESULT_DIR = (
    BASE / "fixed_point_controller_validation_v2"
)

SAVED_RESULTS = (
    RESULT_DIR / "per_layer_controller_results.csv"
)

GRID_CSV = Path(
    "/kaggle/working/shiftscan_final/"
    "canonical_dyadic_grid_definitions.csv"
)

OUT_DIR = (
    BASE / "approximate_zero_validation_v1"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

FRAC_BITS = 20
SCALE = 1 << FRAC_BITS

ERROR_THRESHOLD = 0.001

POT_K_MIN = 4
POT_K_MAX = 30

POT_MIN = 2.0 ** (-POT_K_MAX)

INT32_MIN = -(2**31)
INT32_MAX = 2**31 - 1

print("Approximate-zero experiment")
print("Fraction bits:", FRAC_BITS)
print("Fixed-point scale:", SCALE)
print("Local error threshold:", ERROR_THRESHOLD)
print("Minimum PoT coefficient:", POT_MIN)


# ------------------------------------------------
# 2. Load and deduplicate canonical Dyadic2 grid
# ------------------------------------------------

grid_df = pd.read_csv(GRID_CSV)

grid_df = grid_df[
    grid_df["grid"] == "K1_14_T3"
].copy()

if grid_df.empty:
    raise RuntimeError("K1_14_T3 grid not found.")

grid_df = (
    grid_df
    .sort_values(
        ["decay", "canonical_exponents"]
    )
    .drop_duplicates(subset=["decay"])
    .reset_index(drop=True)
)


def parse_exponents(value):

    if isinstance(value, (tuple, list, np.ndarray)):
        return tuple(int(x) for x in value)

    s = str(value).strip()
    s = s.replace("(", "").replace(")", "")
    s = s.replace("[", "").replace("]", "")

    return tuple(
        int(x.strip())
        for x in s.split(",")
        if x.strip()
    )


grid_decay = grid_df["decay"].to_numpy(
    dtype=np.float64
)

grid_exponents = np.asarray(
    [
        parse_exponents(v)
        for v in grid_df["canonical_exponents"]
    ],
    dtype=np.int64
)

if grid_exponents.shape != (len(grid_decay), 3):
    raise RuntimeError(
        f"Unexpected grid exponent shape: "
        f"{grid_exponents.shape}"
    )

GRID_MIN = float(grid_decay.min())
GRID_MAX = float(grid_decay.max())

print("\nGrid")
print("Unique coefficients:", len(grid_decay))
print("Minimum:", GRID_MIN)
print("Maximum:", GRID_MAX)


# ------------------------------------------------
# 3. Fixed-point arithmetic helpers
# ------------------------------------------------

def round_away_from_zero(x):

    x = np.asarray(x, dtype=np.float64)

    return (
        np.sign(x) * np.floor(np.abs(x) + 0.5)
    ).astype(np.int64)


def quantize_input(x):

    return round_away_from_zero(
        np.asarray(x, dtype=np.float64) * SCALE
    )


def shift_round_signed(h, k):

    h = np.asarray(h, dtype=np.int64)

    divisor = 1 << int(k)

    rounded = (
        np.abs(h) + divisor // 2
    ) // divisor

    return np.sign(h).astype(np.int64) * rounded


def apply_dyadic2(h, exponents):

    original = np.asarray(
        h, dtype=np.int64
    )

    result = original.copy()

    for k in exponents:
        result -= shift_round_signed(
            original, int(k)
        )

    return result


def nearest_pot(a):

    k = int(np.rint(-np.log2(a)))

    k = int(
        np.clip(k, POT_K_MIN, POT_K_MAX)
    )

    return float(2.0 ** (-k)), k


# ------------------------------------------------
# 4. One controller update
# ------------------------------------------------

def controller_step(
    H_int,
    N_int,
    a_fp32,
    a_reference,
    allow_approx_zero
):
    """
    Apply one state-passing recurrence update.

    Returns:
        updated fixed-point state
        baseline/candidate mode
        local predicted error bound
    """

    H_real = H_int.astype(np.float64) / SCALE

    state_max = float(
        np.max(np.abs(H_real))
    )

    # Mode 3: exact zero under FP32 exp.
    if a_fp32 == 0.0:

        return (
            N_int.copy(),
            "exact_zero",
            0.0
        )

    if not np.isfinite(a_fp32) or a_fp32 < 0.0:

        updated = (
            round_away_from_zero(
                float(a_fp32) * H_int
            )
            + N_int
        )

        return (
            updated,
            "fallback_invalid",
            np.nan
        )

    # Dyadic2 range.
    if GRID_MIN <= a_fp32 <= GRID_MAX:

        idx = int(
            np.argmin(
                np.abs(grid_decay - float(a_fp32))
            )
        )

        q = float(grid_decay[idx])

        local_bound = (
            abs(q - float(a_fp32)) * state_max
        )

        if local_bound <= ERROR_THRESHOLD:

            updated = apply_dyadic2(
                H_int,
                grid_exponents[idx]
            )

            updated += N_int

            return (
                updated,
                "dyadic2",
                local_bound
            )

        updated = (
            round_away_from_zero(
                float(a_fp32) * H_int
            )
            + N_int
        )

        return (
            updated,
            "fallback_threshold_dyadic",
            local_bound
        )

    # Small positive decay: direct PoT range.
    if 0.0 < a_fp32 < GRID_MIN:

        if a_fp32 >= POT_MIN:

            q, k = nearest_pot(
                float(a_fp32)
            )

            local_bound = (
                abs(q - float(a_fp32)) * state_max
            )

            if local_bound <= ERROR_THRESHOLD:

                updated = (
                    shift_round_signed(H_int, k)
                    + N_int
                )

                return (
                    updated,
                    "pot",
                    local_bound
                )

            updated = (
                round_away_from_zero(
                    float(a_fp32) * H_int
                )
                + N_int
            )

            return (
                updated,
                "fallback_threshold_pot",
                local_bound
            )

        # ------------------------------------------------
        # New candidate mode:
        # approximate a small positive decay by zero.
        # ------------------------------------------------

        approximate_zero_bound = (
            float(a_fp32) * state_max
        )

        if (
            allow_approx_zero
            and approximate_zero_bound <= ERROR_THRESHOLD
        ):

            # Shift-only zero-decay update:
            # H_new = 0 * H + N
            return (
                N_int.copy(),
                "approximate_zero",
                approximate_zero_bound
            )

        updated = (
            round_away_from_zero(
                float(a_fp32) * H_int
            )
            + N_int
        )

        if allow_approx_zero:

            return (
                updated,
                "fallback_approx_zero_rejected",
                approximate_zero_bound
            )

        return (
            updated,
            "fallback_below_pot_min",
            np.nan
        )

    # Above the Dyadic2 maximum.
    updated = (
        round_away_from_zero(
            float(a_fp32) * H_int
        )
        + N_int
    )

    return (
        updated,
        "fallback_above_grid",
        np.nan
    )


# ------------------------------------------------
# 5. Load saved baseline results
# ------------------------------------------------

saved = pd.read_csv(SAVED_RESULTS)

saved_f20 = saved[
    saved["frac_bits"].astype(int) == FRAC_BITS
].copy()

if len(saved_f20) != 480:
    raise RuntimeError(
        f"Expected 480 F20 baseline records, "
        f"found {len(saved_f20)}"
    )

saved_lookup = {
    (str(row["file"]), int(row["layer"])): row
    for _, row in saved_f20.iterrows()
}

print("\nSaved baseline records:", len(saved_f20))


# ------------------------------------------------
# 6. Evaluation helpers
# ------------------------------------------------

def summarize_errors(error_values):

    error_values = np.asarray(
        error_values,
        dtype=np.float64
    )

    return {
        "mae": float(
            np.mean(np.abs(error_values))
        ),
        "rmse": float(
            np.sqrt(np.mean(error_values ** 2))
        ),
        "max_abs": float(
            np.max(np.abs(error_values))
        ),
    }


def run_record(
    states,
    decay_fp32,
    decay_reference,
    allow_approx_zero
):
    """
    Evaluate one layer record.

    Reference:
        FP64 recurrence using FP64 exp(log_decay).

    Controller:
        FP32 exp semantics for mode decisions.
    """

    chunks, heads, state_dim = states.shape

    N_int = quantize_input(states)

    H_int = np.zeros(
        (heads, state_dim),
        dtype=np.int64
    )

    H_reference = np.zeros(
        (heads, state_dim),
        dtype=np.float64
    )

    errors = []

    counts = {
        "dyadic2": 0,
        "pot": 0,
        "exact_zero": 0,
        "approximate_zero": 0,
        "fallback_threshold_dyadic": 0,
        "fallback_threshold_pot": 0,
        "fallback_below_pot_min": 0,
        "fallback_approx_zero_rejected": 0,
        "fallback_above_grid": 0,
        "fallback_invalid": 0,
    }

    local_bounds = []

    for c in range(chunks):

        for h in range(heads):

            a32 = float(decay_fp32[c, h])
            a_ref = float(decay_reference[c, h])

            # Update exact FP64 reference.
            H_reference[h] = (
                a_ref * H_reference[h]
                + states[c, h].astype(np.float64)
            )

            # Update controller state.
            H_int[h], mode, local_bound = controller_step(
                H_int[h],
                N_int[c, h],
                a32,
                a_ref,
                allow_approx_zero
            )

            counts[mode] += 1

            if np.isfinite(local_bound):
                local_bounds.append(local_bound)

        H_fixed_real = (
            H_int.astype(np.float64) / SCALE
        )

        errors.append(
            H_fixed_real - H_reference
        )

    errors = np.stack(errors, axis=0)

    trajectory_metrics = summarize_errors(errors)

    final_metrics = summarize_errors(errors[-1])

    max_integer_state = int(
        np.max(np.abs(H_int))
    )

    overflow_count = int(
        np.sum(
            (H_int < INT32_MIN)
            | (H_int > INT32_MAX)
        )
    )

    total_updates = chunks * heads

    strict_shift_count = (
        counts["dyadic2"]
        + counts["pot"]
    )

    nonfallback_count = (
        strict_shift_count
        + counts["exact_zero"]
        + counts["approximate_zero"]
    )

    fallback_count = (
        total_updates - nonfallback_count
    )

    return {
        "trajectory_mae": trajectory_metrics["mae"],
        "trajectory_rmse": trajectory_metrics["rmse"],
        "trajectory_max": trajectory_metrics["max_abs"],
        "final_mae": final_metrics["mae"],
        "final_rmse": final_metrics["rmse"],
        "final_max": final_metrics["max_abs"],
        "max_integer_state": max_integer_state,
        "overflow_count": overflow_count,
        "total_updates": total_updates,
        "strict_shift_count": strict_shift_count,
        "nonfallback_count": nonfallback_count,
        "fallback_count": fallback_count,
        "strict_shift_pct": (
            100.0 * strict_shift_count / total_updates
        ),
        "coverage_pct": (
            100.0 * nonfallback_count / total_updates
        ),
        "fallback_pct": (
            100.0 * fallback_count / total_updates
        ),
        "max_local_bound": (
            max(local_bounds) if local_bounds else 0.0
        ),
        **{
            f"count_{k}": v
            for k, v in counts.items()
        },
    }


# ------------------------------------------------
# 7. Run all captures: baseline and candidate
# ------------------------------------------------

baseline_rows = []
candidate_rows = []

mode_mismatches = []

capture_paths = sorted(
    BASE.glob("doc_*_seq_*.pt")
)

if len(capture_paths) != 20:
    raise RuntimeError(
        f"Expected 20 capture files, found {len(capture_paths)}"
    )

for capture_index, capture_path in enumerate(capture_paths, 1):

    print(
        f"\n[{capture_index}/20] "
        f"Processing {capture_path.name}"
    )

    obj = torch.load(
        capture_path,
        map_location="cpu",
        weights_only=False
    )

    for rec in obj["records"]:

        layer = int(rec["layer"])

        key = (capture_path.name, layer)

        if key not in saved_lookup:
            raise RuntimeError(
                f"Missing saved baseline row: {key}"
            )

        saved_row = saved_lookup[key]

        states_t = (
            rec["states"]
            .detach()
            .cpu()
            .float()
        )

        log_decay_t = (
            rec["decay_log"]
            .detach()
            .cpu()
            .float()
        )

        states = states_t[0].numpy().astype(
            np.float64
        )

        log_decay_fp32 = log_decay_t

        if log_decay_fp32.ndim == 3:
            log_decay_fp32 = log_decay_fp32[0]

        if log_decay_fp32.ndim != 2:
            raise RuntimeError(
                f"Unexpected log-decay shape in {key}"
            )

        # Convert [heads, chunks] to [chunks, heads].
        log_decay_fp32 = (
            log_decay_fp32.transpose(0, 1).contiguous()
        )

        # Controller decisions use FP32 exp.
        decay_fp32 = torch.exp(
            log_decay_fp32.float()
        ).cpu().numpy().astype(np.float64)

        # Reference uses FP64 exp before conversion.
        decay_reference = np.exp(
            log_decay_fp32.double().numpy()
        )

        chunks, heads, state_dim = states.shape

        if decay_fp32.shape != (chunks, heads):
            raise RuntimeError(
                f"Decay shape mismatch in {key}"
            )

        # --------------------------------------------
        # Baseline: original policy
        # --------------------------------------------

        baseline = run_record(
            states,
            decay_fp32,
            decay_reference,
            allow_approx_zero=False
        )

        # --------------------------------------------
        # Candidate: new approximate-zero policy
        # --------------------------------------------

        candidate = run_record(
            states,
            decay_fp32,
            decay_reference,
            allow_approx_zero=True
        )

        # --------------------------------------------
        # Validate baseline mode counts
        # --------------------------------------------

        baseline_count_checks = {
            "count_dyadic2": int(
                saved_row["dyadic_count"]
            ),
            "count_pot": int(
                saved_row["pot_count"]
            ),
            "count_exact_zero": int(
                saved_row["zero_count"]
            ),
            "count_fallback_threshold_dyadic":
                int(saved_row["threshold_dyadic_count"])
                if "threshold_dyadic_count" in saved_row.index
                else None,
        }

        # Validate the four original saved modes.
        # The saved CSV uses these names:
        # dyadic_count, pot_count, zero_count, fallback_count.
        original_mode_checks = {
            "count_dyadic2": int(saved_row["dyadic_count"]),
            "count_pot": int(saved_row["pot_count"]),
            "count_exact_zero": int(saved_row["zero_count"]),
            "fallback_count": int(saved_row["fallback_count"]),
        }

        baseline_fallback = int(
            baseline["fallback_count"]
        )

        actual_mode_checks = {
            "count_dyadic2": int(
                baseline["count_dyadic2"]
            ),
            "count_pot": int(
                baseline["count_pot"]
            ),
            "count_exact_zero": int(
                baseline["count_exact_zero"]
            ),
            "fallback_count": baseline_fallback,
        }

        mismatch = {
            name: {
                "expected": expected,
                "actual": actual_mode_checks[name],
            }
            for name, expected in original_mode_checks.items()
            if expected != actual_mode_checks[name]
        }

        if mismatch:
            mode_mismatches.append(
                {
                    "file": capture_path.name,
                    "layer": layer,
                    "mismatch": mismatch,
                }
            )

        common = {
            "file": capture_path.name,
            "doc_id": int(saved_row["doc_id"]),
            "seq_len": int(saved_row["seq_len"]),
            "layer": layer,
            "chunks": chunks,
            "heads": heads,
            "state_dim": state_dim,
            "frac_bits": FRAC_BITS,
        }

        baseline_rows.append(
            {
                **common,
                "policy": "baseline",
                **baseline,
            }
        )

        candidate_rows.append(
            {
                **common,
                "policy": "approximate_zero",
                **candidate,
            }
        )

    print("Completed:", capture_path.name)


# ------------------------------------------------
# 8. Validate baseline reproduction
# ------------------------------------------------

if len(baseline_rows) != 480:
    raise RuntimeError(
        f"Expected 480 baseline records, got {len(baseline_rows)}"
    )

if len(candidate_rows) != 480:
    raise RuntimeError(
        f"Expected 480 candidate records, got {len(candidate_rows)}"
    )

print("\n" + "=" * 65)
print("BASELINE REPRODUCTION VALIDATION")
print("=" * 65)

print("Baseline records:", len(baseline_rows))
print("Candidate records:", len(candidate_rows))
print("Baseline mode-count mismatches:", len(mode_mismatches))

if mode_mismatches:
    print("Examples:", mode_mismatches[:5])

    raise RuntimeError(
        "Baseline mode counts do not reproduce saved results. "
        "Do not interpret candidate results."
    )

print("SUCCESS: baseline arithmetic-mode counts reproduced.")


# ------------------------------------------------
# 9. Save per-layer outputs
# ------------------------------------------------

baseline_df = pd.DataFrame(baseline_rows)
candidate_df = pd.DataFrame(candidate_rows)

baseline_df.to_csv(
    OUT_DIR / "baseline_per_layer.csv",
    index=False
)

candidate_df.to_csv(
    OUT_DIR / "approximate_zero_per_layer.csv",
    index=False
)

comparison = baseline_df.merge(
    candidate_df,
    on=[
        "file",
        "doc_id",
        "seq_len",
        "layer",
        "chunks",
        "heads",
        "state_dim",
        "frac_bits",
    ],
    suffixes=("_baseline", "_candidate"),
    validate="one_to_one"
)

comparison["trajectory_mae_change"] = (
    comparison["trajectory_mae_candidate"]
    - comparison["trajectory_mae_baseline"]
)

comparison["trajectory_rmse_change"] = (
    comparison["trajectory_rmse_candidate"]
    - comparison["trajectory_rmse_baseline"]
)

comparison["trajectory_max_change"] = (
    comparison["trajectory_max_candidate"]
    - comparison["trajectory_max_baseline"]
)

comparison["final_mae_change"] = (
    comparison["final_mae_candidate"]
    - comparison["final_mae_baseline"]
)

comparison["final_rmse_change"] = (
    comparison["final_rmse_candidate"]
    - comparison["final_rmse_baseline"]
)

comparison["final_max_change"] = (
    comparison["final_max_candidate"]
    - comparison["final_max_baseline"]
)

comparison["coverage_gain_pp"] = (
    comparison["coverage_pct_candidate"]
    - comparison["coverage_pct_baseline"]
)

comparison["fallback_reduction_pp"] = (
    comparison["fallback_pct_baseline"]
    - comparison["fallback_pct_candidate"]
)

comparison.to_csv(
    OUT_DIR / "baseline_vs_approximate_zero.csv",
    index=False
)


# ------------------------------------------------
# 10. Aggregate summary
# ------------------------------------------------

def aggregate_policy(df, policy_name):

    total_updates = int(df["total_updates"].sum())

    total_strict = int(
        df["strict_shift_count"].sum()
    )

    total_coverage = int(
        df["nonfallback_count"].sum()
    )

    total_fallback = int(
        df["fallback_count"].sum()
    )

    return {
        "policy": policy_name,
        "layer_records": len(df),
        "total_updates": total_updates,
        "strict_shift_updates": total_strict,
        "nonfallback_updates": total_coverage,
        "floating_fallback_updates": total_fallback,
        "strict_shift_pct": (
            100.0 * total_strict / total_updates
        ),
        "coverage_pct": (
            100.0 * total_coverage / total_updates
        ),
        "fallback_pct": (
            100.0 * total_fallback / total_updates
        ),
        "mean_trajectory_mae": float(
            df["trajectory_mae"].mean()
        ),
        "mean_trajectory_rmse": float(
            df["trajectory_rmse"].mean()
        ),
        "worst_trajectory_max": float(
            df["trajectory_max"].max()
        ),
        "mean_final_mae": float(
            df["final_mae"].mean()
        ),
        "mean_final_rmse": float(
            df["final_rmse"].mean()
        ),
        "worst_final_max": float(
            df["final_max"].max()
        ),
        "max_integer_state": int(
            df["max_integer_state"].max()
        ),
        "overflow_count": int(
            df["overflow_count"].sum()
        ),
    }


aggregate = pd.DataFrame(
    [
        aggregate_policy(
            baseline_df,
            "baseline"
        ),
        aggregate_policy(
            candidate_df,
            "approximate_zero"
        ),
    ]
)

aggregate.to_csv(
    OUT_DIR / "aggregate_comparison.csv",
    index=False
)

print("\n" + "=" * 65)
print("OVERALL POLICY COMPARISON")
print("=" * 65)

display(aggregate.T)


# ------------------------------------------------
# 11. Arithmetic-mode comparison
# ------------------------------------------------

mode_map = {
    "Dyadic2": "count_dyadic2",
    "Direct PoT": "count_pot",
    "Exact FP32 zero": "count_exact_zero",
    "Approximate zero": "count_approximate_zero",
    "Threshold rejected Dyadic2":
        "count_fallback_threshold_dyadic",
    "Threshold rejected PoT":
        "count_fallback_threshold_pot",
    "Below PoT range fallback":
        "count_fallback_below_pot_min",
    "Rejected approximate zero":
        "count_fallback_approx_zero_rejected",
    "Above grid fallback":
        "count_fallback_above_grid",
    "Invalid fallback":
        "count_fallback_invalid",
}

mode_rows = []

for policy_name, df in [
    ("baseline", baseline_df),
    ("approximate_zero", candidate_df),
]:

    for label, column in mode_map.items():

        mode_rows.append(
            {
                "policy": policy_name,
                "mode": label,
                "count": int(df[column].sum()),
            }
        )

mode_df = pd.DataFrame(mode_rows)

mode_df["percent_of_updates"] = (
    100.0 * mode_df["count"]
    / mode_df["policy"].map(
        {
            "baseline": int(baseline_df["total_updates"].sum()),
            "approximate_zero": int(candidate_df["total_updates"].sum()),
        }
    )
)

mode_df.to_csv(
    OUT_DIR / "arithmetic_mode_comparison.csv",
    index=False
)

print("\nArithmetic mode counts")
display(
    mode_df.pivot(
        index="mode",
        columns="policy",
        values="count"
    ).fillna(0).astype(int)
)


# ------------------------------------------------
# 12. Aggregate by sequence length
# ------------------------------------------------

length_rows = []

for seq_len in sorted(candidate_df["seq_len"].unique()):

    for policy_name, df in [
        ("baseline", baseline_df),
        ("approximate_zero", candidate_df),
    ]:

        subset = df[
            df["seq_len"] == seq_len
        ]

        total_updates = int(
            subset["total_updates"].sum()
        )

        total_strict = int(
            subset["strict_shift_count"].sum()
        )

        total_coverage = int(
            subset["nonfallback_count"].sum()
        )

        total_fallback = int(
            subset["fallback_count"].sum()
        )

        length_rows.append(
            {
                "seq_len": int(seq_len),
                "policy": policy_name,
                "layer_records": len(subset),
                "total_updates": total_updates,
                "strict_shift_pct": (
                    100.0 * total_strict / total_updates
                ),
                "coverage_pct": (
                    100.0 * total_coverage / total_updates
                ),
                "fallback_pct": (
                    100.0 * total_fallback / total_updates
                ),
                "mean_trajectory_mae": float(
                    subset["trajectory_mae"].mean()
                ),
                "mean_trajectory_rmse": float(
                    subset["trajectory_rmse"].mean()
                ),
                "worst_trajectory_max": float(
                    subset["trajectory_max"].max()
                ),
                "mean_final_mae": float(
                    subset["final_mae"].mean()
                ),
                "mean_final_rmse": float(
                    subset["final_rmse"].mean()
                ),
                "worst_final_max": float(
                    subset["final_max"].max()
                ),
                "overflow_count": int(
                    subset["overflow_count"].sum()
                ),
            }
        )

length_df = pd.DataFrame(length_rows)

length_df.to_csv(
    OUT_DIR / "comparison_by_sequence_length.csv",
    index=False
)

print("\n" + "=" * 65)
print("COMPARISON BY SEQUENCE LENGTH")
print("=" * 65)

display(
    length_df.round(8)
)


# ------------------------------------------------
# 13. Layer-wise coverage gains
# ------------------------------------------------

layer_comparison = (
    comparison.groupby("layer", as_index=False)
    .agg(
        baseline_coverage_pct=(
            "coverage_pct_baseline", "mean"
        ),
        candidate_coverage_pct=(
            "coverage_pct_candidate", "mean"
        ),
        baseline_fallback_pct=(
            "fallback_pct_baseline", "mean"
        ),
        candidate_fallback_pct=(
            "fallback_pct_candidate", "mean"
        ),
        baseline_trajectory_mae=(
            "trajectory_mae_baseline", "mean"
        ),
        candidate_trajectory_mae=(
            "trajectory_mae_candidate", "mean"
        ),
        baseline_trajectory_rmse=(
            "trajectory_rmse_baseline", "mean"
        ),
        candidate_trajectory_rmse=(
            "trajectory_rmse_candidate", "mean"
        ),
        baseline_final_mae=(
            "final_mae_baseline", "mean"
        ),
        candidate_final_mae=(
            "final_mae_candidate", "mean"
        ),
        max_candidate_trajectory_error=(
            "trajectory_max_candidate", "max"
        ),
        max_candidate_final_error=(
            "final_max_candidate", "max"
        ),
        max_integer_state=(
            "max_integer_state_candidate", "max"
        ),
        overflow_count=(
            "overflow_count_candidate", "sum"
        ),
    )
)

layer_comparison["coverage_gain_pp"] = (
    layer_comparison["candidate_coverage_pct"]
    - layer_comparison["baseline_coverage_pct"]
)

layer_comparison["trajectory_mae_change"] = (
    layer_comparison["candidate_trajectory_mae"]
    - layer_comparison["baseline_trajectory_mae"]
)

layer_comparison["final_mae_change"] = (
    layer_comparison["candidate_final_mae"]
    - layer_comparison["baseline_final_mae"]
)

layer_comparison.to_csv(
    OUT_DIR / "layer_wise_comparison.csv",
    index=False
)

print("\n" + "=" * 65)
print("LAYER-WISE COVERAGE GAIN")
print("=" * 65)

display(
    layer_comparison[
        [
            "layer",
            "baseline_coverage_pct",
            "candidate_coverage_pct",
            "coverage_gain_pp",
            "trajectory_mae_change",
            "final_mae_change",
            "max_candidate_trajectory_error",
            "overflow_count",
        ]
    ].round(8)
)


# ------------------------------------------------
# 14. Safety checks
# ------------------------------------------------

candidate_overflow = int(
    candidate_df["overflow_count"].sum()
)

candidate_max_integer = int(
    candidate_df["max_integer_state"].max()
)

candidate_int32_safe = (
    candidate_max_integer <= INT32_MAX
)

# Every candidate approximate-zero decision must have
# passed the local error threshold.
candidate_approx_zero_count = int(
    candidate_df["count_approximate_zero"].sum()
)

print("\n" + "=" * 65)
print("SAFETY CHECKS")
print("=" * 65)

print("Candidate approximate-zero updates:",
      candidate_approx_zero_count)

print("Candidate overflow count:",
      candidate_overflow)

print("Maximum absolute integer state:",
      candidate_max_integer)

print("Int32 positive range:",
      INT32_MAX)

print("Int32 range check:",
      "PASS" if candidate_int32_safe and candidate_overflow == 0
      else "CHECK REQUIRED")


# ------------------------------------------------
# 15. Save metadata
# ------------------------------------------------

metadata = {
    "experiment": "state_aware_approximate_zero",
    "baseline": "F20_K1_14_T3_adaptive_controller",
    "candidate": "baseline_plus_state_aware_approximate_zero",
    "frac_bits": FRAC_BITS,
    "scale": SCALE,
    "error_threshold": ERROR_THRESHOLD,
    "approximate_zero_condition": (
        "a_fp32 > 0 and a_fp32 < 2^-30 and "
        "a_fp32 * max(abs(current_fixed_state)) <= 0.001"
    ),
    "controller_exp_dtype": "float32",
    "reference_exp_dtype": "float64",
    "grid": "K1_14_T3",
    "grid_unique_values": int(len(grid_decay)),
    "grid_min": GRID_MIN,
    "grid_max": GRID_MAX,
    "pot_min": POT_MIN,
    "capture_files": len(capture_paths),
    "layer_records": len(candidate_df),
    "baseline_mode_mismatches": len(mode_mismatches),
    "candidate_overflow_count": candidate_overflow,
    "candidate_max_integer_state": candidate_max_integer,
    "candidate_int32_safe": bool(candidate_int32_safe),
    "outputs": [
        "baseline_per_layer.csv",
        "approximate_zero_per_layer.csv",
        "baseline_vs_approximate_zero.csv",
        "aggregate_comparison.csv",
        "arithmetic_mode_comparison.csv",
        "comparison_by_sequence_length.csv",
        "layer_wise_comparison.csv",
    ],
}

with open(
    OUT_DIR / "experiment_config.json",
    "w"
) as f:
    json.dump(metadata, f, indent=2)

print("\n" + "=" * 65)
print("EXPERIMENT COMPLETE")
print("=" * 65)

print("Output directory:", OUT_DIR)

for path in sorted(OUT_DIR.glob("*")):
    print(path.name)


Approximate-zero experiment
Fraction bits: 20
Fixed-point scale: 1048576
Local error threshold: 0.001
Minimum PoT coefficient: 9.313225746154785e-10

Grid
Unique coefficients: 467
Minimum: 0.125
Maximum: 0.99981689453125

Saved baseline records: 480

[1/20] Processing doc_000_seq_1024.pt
Completed: doc_000_seq_1024.pt

[2/20] Processing doc_000_seq_2048.pt
Completed: doc_000_seq_2048.pt

[3/20] Processing doc_001_seq_1024.pt
Completed: doc_001_seq_1024.pt

[4/20] Processing doc_001_seq_2048.pt
Completed: doc_001_seq_2048.pt

[5/20] Processing doc_002_seq_1024.pt
Completed: doc_002_seq_1024.pt

[6/20] Processing doc_002_seq_2048.pt
Completed: doc_002_seq_2048.pt

[7/20] Processing doc_003_seq_1024.pt
Completed: doc_003_seq_1024.pt

[8/20] Processing doc_003_seq_2048.pt
Completed: doc_003_seq_2048.pt

[9/20] Processing doc_004_seq_1024.pt
Completed: doc_004_seq_1024.pt

[10/20] Processing doc_004_seq_2048.pt
Completed: doc_004_seq_2048.pt

[11/20] Processing doc_005_seq_1024.pt
Completed

,0,1
policy,baseline,approximate_zero
layer_records,480,480
total_updates,69120,69120
strict_shift_updates,14230,14230
nonfallback_updates,36310,53352
floating_fallback_updates,32810,15768
strict_shift_pct,20.587384,20.587384
coverage_pct,52.531829,77.1875
fallback_pct,47.468171,22.8125
mean_trajectory_mae,0.000001,0.000001



Arithmetic mode counts


policy,approximate_zero,baseline
mode,,
Above grid fallback,0,0
Approximate zero,17042,0
Below PoT range fallback,0,17042
Direct PoT,9796,9796
Dyadic2,4434,4434
Exact FP32 zero,22080,22080
Invalid fallback,0,0
Rejected approximate zero,0,0
Threshold rejected Dyadic2,10973,10973



COMPARISON BY SEQUENCE LENGTH


,seq_len,policy,layer_records,total_updates,strict_shift_pct,coverage_pct,fallback_pct,mean_trajectory_mae,mean_trajectory_rmse,worst_trajectory_max,mean_final_mae,mean_final_rmse,worst_final_max,overflow_count
0,1024,baseline,240,23040,22.994792,55.269097,44.730903,8.700000e-07,0.000005,0.001663,0.000001,0.000006,0.001663,0
1,1024,approximate_zero,240,23040,22.994792,79.622396,20.377604,8.700000e-07,0.000005,0.001663,0.000001,0.000006,0.001663,0
2,2048,baseline,240,46080,19.383681,51.163194,48.836806,1.090000e-06,0.000006,0.002526,0.000001,0.000007,0.002526,0
3,2048,approximate_zero,240,46080,19.383681,75.970052,24.029948,1.090000e-06,0.000006,0.002526,0.000001,0.000007,0.002526,0



LAYER-WISE COVERAGE GAIN


,layer,baseline_coverage_pct,candidate_coverage_pct,coverage_gain_pp,trajectory_mae_change,final_mae_change,max_candidate_trajectory_error,overflow_count
0,0,69.739583,73.906250,4.166667,0.0,0.0,0.001000,0
1,1,55.859375,76.692708,20.833333,0.0,0.0,0.000995,0
2,2,41.015625,55.000000,13.984375,0.0,0.0,0.001664,0
3,3,53.072917,82.552083,29.479167,0.0,0.0,0.000986,0
4,4,43.046875,49.921875,6.875000,0.0,0.0,0.001387,0
5,5,71.640625,79.322917,7.682292,0.0,0.0,0.000999,0
6,6,38.281250,53.437500,15.156250,0.0,0.0,0.001418,0
7,7,32.135417,76.744792,44.609375,0.0,0.0,0.000988,0
8,8,27.395833,93.203125,65.807292,0.0,0.0,0.000942,0
9,9,68.723958,93.802083,25.078125,0.0,0.0,0.000984,0



SAFETY CHECKS
Candidate approximate-zero updates: 17042
Candidate overflow count: 0
Maximum absolute integer state: 396581856
Int32 positive range: 2147483647
Int32 range check: PASS

EXPERIMENT COMPLETE
Output directory: /kaggle/working/shiftscan_final/state_passing/multisequence_validation/approximate_zero_validation_v1
aggregate_comparison.csv
approximate_zero_per_layer.csv
arithmetic_mode_comparison.csv
baseline_per_layer.csv
baseline_vs_approximate_zero.csv
comparison_by_sequence_length.csv
experiment_config.json
layer_wise_comparison.csv



## State-Aware Approximate-Zero: F20 Validation

### Experimental objective

The baseline adaptive controller uses floating-point fallback for positive decay coefficients below \(2^{-30}\).

A candidate policy was evaluated that replaces these small positive coefficients with zero when:

\[
a\max_j|\hat S_j|\leq 0.001
\]

The experiment used 20 actual Mamba-2 state-passing captures, covering 10 FineWeb-Edu documents at sequence lengths 1024 and 2048, across 24 layers.

### Results

| Metric | Baseline | Approximate-zero |
|---|---:|---:|
| Non-fallback coverage | 52.53% | 77.19% |
| Floating-point fallback | 47.47% | 22.81% |
| Strict shift-arithmetic coverage | 20.59% | 20.59% |
| Overflow count | 0 | 0 |

The candidate converted 17,042 below-range fallback updates into approximate-zero updates.

This represents a 24.66 percentage-point increase in non-fallback coverage and a 51.94% reduction in floating-point fallback updates.

### Numerical accuracy

The reported mean trajectory MAE, trajectory RMSE, maximum trajectory error, final-state MAE, and final-state RMSE were unchanged at the displayed precision.

The worst trajectory error remained 0.002526. The local acceptance threshold of 0.001 does not impose a global trajectory-error bound.

### Fixed-point rounding interpretation

The minimum supported PoT coefficient is:

\[
2^{-30}=9.313225746\times10^{-10}
\]

The maximum observed absolute integer state was 396,581,856.

Therefore:

\[
2^{-30}\times396581856\approx0.369
\]

This is below the 0.5 rounding threshold of the fixed-point decay multiplication.

Consequently, the original fallback multiplication is expected to produce a zero integer decay contribution for the observed states and below-range coefficients.

An update-by-update integer-state equivalence audit is required to confirm this interpretation.

### Scope

Approximate-zero is a separate arithmetic mode. It increases non-fallback coverage but does not increase strict Dyadic2 or direct-PoT shift coverage.

The results do not yet establish a reduction in total Mamba-2 model multiplications or GPU latency. Those require integration into the actual SSD kernels and hardware benchmarking.


In [168]:

# ================================================================
# F20 APPROXIMATE-ZERO: UPDATE-BY-UPDATE INTEGER EQUIVALENCE AUDIT
# ================================================================

import numpy as np
import pandas as pd
import torch
from pathlib import Path

BASE = Path(
    "/kaggle/working/shiftscan_final/state_passing/"
    "multisequence_validation"
)

OUT_DIR = (
    BASE / "approximate_zero_validation_v1"
)

GRID_CSV = Path(
    "/kaggle/working/shiftscan_final/"
    "canonical_dyadic_grid_definitions.csv"
)

FRAC_BITS = 20
SCALE = 1 << FRAC_BITS
THRESHOLD = 0.001
POT_K_MAX = 30
POT_MIN = 2.0 ** (-POT_K_MAX)

# ------------------------------------------------
# Load canonical grid
# ------------------------------------------------

gdf = pd.read_csv(GRID_CSV)

gdf = gdf[
    gdf["grid"] == "K1_14_T3"
].copy()

gdf = (
    gdf.sort_values(
        ["decay", "canonical_exponents"]
    )
    .drop_duplicates("decay")
    .reset_index(drop=True)
)

def parse_exp(v):
    s = str(v).strip()
    s = s.replace("(", "").replace(")", "")
    s = s.replace("[", "").replace("]", "")
    return tuple(
        int(x.strip())
        for x in s.split(",")
        if x.strip()
    )

grid_decay = gdf["decay"].to_numpy(np.float64)

grid_exp = np.asarray(
    [parse_exp(v) for v in gdf["canonical_exponents"]],
    dtype=np.int64
)

GRID_MIN = float(grid_decay.min())
GRID_MAX = float(grid_decay.max())


# ------------------------------------------------
# Fixed-point helpers
# ------------------------------------------------

def round_away(x):
    x = np.asarray(x, dtype=np.float64)
    return (
        np.sign(x) * np.floor(np.abs(x) + 0.5)
    ).astype(np.int64)


def quantize_x(x):
    return round_away(
        np.asarray(x, dtype=np.float64) * SCALE
    )


def shift_round(h, k):
    h = np.asarray(h, dtype=np.int64)
    divisor = 1 << int(k)
    rounded = (
        np.abs(h) + divisor // 2
    ) // divisor
    return np.sign(h).astype(np.int64) * rounded


def dyadic_update(h, exponents):
    h = np.asarray(h, dtype=np.int64)
    out = h.copy()
    for k in exponents:
        out -= shift_round(h, int(k))
    return out


def nearest_pot(a):
    k = int(np.rint(-np.log2(a)))
    k = int(np.clip(k, 4, POT_K_MAX))
    return 2.0 ** (-k), k


# ------------------------------------------------
# Audit counters
# ------------------------------------------------

total_records = 0
total_updates = 0
approx_zero_updates = 0

identical_update_count = 0
different_update_count = 0
different_state_elements = 0

max_abs_raw_decay_product = 0.0
max_abs_rounded_decay_contribution = 0

nonzero_rounded_update_count = 0
nonzero_rounded_element_count = 0

max_local_approx_zero_bound = 0.0

per_layer_rows = []


# ------------------------------------------------
# Replay every capture and layer
# ------------------------------------------------

capture_paths = sorted(
    BASE.glob("doc_*_seq_*.pt")
)

for path in capture_paths:

    obj = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    for rec in obj["records"]:

        total_records += 1

        layer = int(rec["layer"])

        states = (
            rec["states"]
            .detach()
            .cpu()
            .float()[0]
            .numpy()
            .astype(np.float64)
        )

        logd = (
            rec["decay_log"]
            .detach()
            .cpu()
            .float()
        )

        if logd.ndim == 3:
            logd = logd[0]

        logd = logd.transpose(0, 1).contiguous()

        # Match original controller's FP32 exp semantics.
        decay = (
            torch.exp(logd.float())
            .cpu()
            .numpy()
            .astype(np.float64)
        )

        chunks, heads, state_dim = states.shape

        if decay.shape != (chunks, heads):
            raise RuntimeError(
                f"Shape mismatch in {path.name}, layer {layer}"
            )

        N_int = quantize_x(states)

        H_base = np.zeros(
            (heads, state_dim), dtype=np.int64
        )

        H_candidate = np.zeros(
            (heads, state_dim), dtype=np.int64
        )

        layer_approx = 0
        layer_identical = 0
        layer_different = 0
        layer_max_raw = 0.0
        layer_max_rounded = 0
        layer_nonzero_updates = 0
        layer_local_bound_max = 0.0

        for c in range(chunks):

            for h in range(heads):

                a = float(decay[c, h])

                hb = H_base[h].copy()
                hc = H_candidate[h].copy()

                nb = N_int[c, h]
                nc = N_int[c, h]

                # ------------------------------------
                # Baseline controller
                # ------------------------------------

                if a == 0.0:
                    next_base = nb.copy()

                elif GRID_MIN <= a <= GRID_MAX:

                    idx = int(
                        np.argmin(
                            np.abs(grid_decay - a)
                        )
                    )

                    q = float(grid_decay[idx])

                    state_max_b = (
                        np.max(np.abs(hb)) / SCALE
                    )

                    bound = (
                        abs(q - a) * state_max_b
                    )

                    if bound <= THRESHOLD:

                        next_base = (
                            dyadic_update(
                                hb, grid_exp[idx]
                            ) + nb
                        )

                    else:

                        next_base = (
                            round_away(a * hb) + nb
                        )

                elif 0.0 < a < GRID_MIN:

                    if a >= POT_MIN:

                        q, k = nearest_pot(a)

                        state_max_b = (
                            np.max(np.abs(hb)) / SCALE
                        )

                        bound = (
                            abs(q - a) * state_max_b
                        )

                        if bound <= THRESHOLD:

                            next_base = (
                                shift_round(hb, k) + nb
                            )

                        else:

                            next_base = (
                                round_away(a * hb) + nb
                            )

                    else:

                        # Original below-range fallback.
                        next_base = (
                            round_away(a * hb) + nb
                        )

                else:

                    next_base = (
                        round_away(a * hb) + nb
                    )

                # ------------------------------------
                # Candidate controller
                # ------------------------------------

                if a == 0.0:

                    next_candidate = nc.copy()

                elif GRID_MIN <= a <= GRID_MAX:

                    idx = int(
                        np.argmin(
                            np.abs(grid_decay - a)
                        )
                    )

                    q = float(grid_decay[idx])

                    state_max_c = (
                        np.max(np.abs(hc)) / SCALE
                    )

                    bound = (
                        abs(q - a) * state_max_c
                    )

                    if bound <= THRESHOLD:

                        next_candidate = (
                            dyadic_update(
                                hc, grid_exp[idx]
                            ) + nc
                        )

                    else:

                        next_candidate = (
                            round_away(a * hc) + nc
                        )

                elif 0.0 < a < GRID_MIN:

                    if a >= POT_MIN:

                        q, k = nearest_pot(a)

                        state_max_c = (
                            np.max(np.abs(hc)) / SCALE
                        )

                        bound = (
                            abs(q - a) * state_max_c
                        )

                        if bound <= THRESHOLD:

                            next_candidate = (
                                shift_round(hc, k) + nc
                            )

                        else:

                            next_candidate = (
                                round_away(a * hc) + nc
                            )

                    else:

                        # New approximate-zero decision.
                        state_max_c = (
                            np.max(np.abs(hc)) / SCALE
                        )

                        approx_bound = (
                            a * state_max_c
                        )

                        if approx_bound <= THRESHOLD:

                            next_candidate = nc.copy()

                            layer_approx += 1
                            approx_zero_updates += 1

                            layer_local_bound_max = max(
                                layer_local_bound_max,
                                approx_bound
                            )

                            max_local_approx_zero_bound = max(
                                max_local_approx_zero_bound,
                                approx_bound
                            )

                            # Measure the baseline's
                            # unrounded integer decay product.
                            raw_product = np.abs(
                                a * hb.astype(np.float64)
                            )

                            raw_max = float(
                                raw_product.max()
                            )

                            rounded_product = np.abs(
                                round_away(a * hb)
                            )

                            rounded_max = int(
                                rounded_product.max()
                            )

                            max_abs_raw_decay_product = max(
                                max_abs_raw_decay_product,
                                raw_max
                            )

                            max_abs_rounded_decay_contribution = max(
                                max_abs_rounded_decay_contribution,
                                rounded_max
                            )

                            layer_max_raw = max(
                                layer_max_raw,
                                raw_max
                            )

                            layer_max_rounded = max(
                                layer_max_rounded,
                                rounded_max
                            )

                            if np.any(rounded_product != 0):
                                nonzero_rounded_update_count += 1
                                layer_nonzero_updates += 1

                                nonzero_rounded_element_count += int(
                                    np.count_nonzero(
                                        rounded_product
                                    )
                                )

                        else:

                            next_candidate = (
                                round_away(a * hc) + nc
                            )

                else:

                    next_candidate = (
                        round_away(a * hc) + nc
                    )

                # ------------------------------------
                # Compare exact integer states
                # ------------------------------------

                same = np.array_equal(
                    next_base,
                    next_candidate
                )

                if same:
                    identical_update_count += 1
                    layer_identical += 1

                else:
                    different_update_count += 1
                    different_state_elements += int(
                        np.count_nonzero(
                            next_base != next_candidate
                        )
                    )
                    layer_different += 1

                H_base[h] = next_base
                H_candidate[h] = next_candidate

                total_updates += 1

        per_layer_rows.append(
            {
                "file": path.name,
                "layer": layer,
                "seq_len": int(
                    path.stem.split("_")[-1]
                ),
                "approx_zero_updates": layer_approx,
                "identical_updates": layer_identical,
                "different_updates": layer_different,
                "max_raw_integer_decay_product": layer_max_raw,
                "max_rounded_decay_contribution": layer_max_rounded,
                "nonzero_rounded_updates": layer_nonzero_updates,
                "max_local_approx_zero_bound": layer_local_bound_max,
            }
        )


# ------------------------------------------------
# Save and display audit
# ------------------------------------------------

audit_df = pd.DataFrame(per_layer_rows)

audit_df.to_csv(
    OUT_DIR / "integer_state_equivalence_per_layer.csv",
    index=False
)

summary = {
    "records": total_records,
    "recurrence_updates": total_updates,
    "approximate_zero_updates": approx_zero_updates,
    "identical_integer_updates": identical_update_count,
    "different_integer_updates": different_update_count,
    "different_state_elements": different_state_elements,
    "max_unrounded_integer_decay_product": (
        max_abs_raw_decay_product
    ),
    "max_rounded_integer_decay_contribution": (
        max_abs_rounded_decay_contribution
    ),
    "updates_with_nonzero_rounded_decay": (
        nonzero_rounded_update_count
    ),
    "elements_with_nonzero_rounded_decay": (
        nonzero_rounded_element_count
    ),
    "max_local_approx_zero_bound": (
        max_local_approx_zero_bound
    ),
    "all_integer_updates_identical": (
        different_update_count == 0
    ),
}

with open(
    OUT_DIR / "integer_state_equivalence_summary.json",
    "w"
) as f:
    json.dump(summary, f, indent=2)

print("=" * 65)
print("INTEGER STATE EQUIVALENCE AUDIT")
print("=" * 65)

for k, v in summary.items():
    print(f"{k}: {v}")

print("\nPer-layer audit")
display(
    audit_df.sort_values(
        "approx_zero_updates",
        ascending=False
    ).head(20)
)

print("\nSaved:")
print(
    OUT_DIR / "integer_state_equivalence_per_layer.csv"
)
print(
    OUT_DIR / "integer_state_equivalence_summary.json"
)


INTEGER STATE EQUIVALENCE AUDIT
records: 480
recurrence_updates: 69120
approximate_zero_updates: 17042
identical_integer_updates: 69120
different_integer_updates: 0
different_state_elements: 0
max_unrounded_integer_decay_product: 0.037470795470760265
max_rounded_integer_decay_contribution: 0
updates_with_nonzero_rounded_decay: 0
elements_with_nonzero_rounded_decay: 0
max_local_approx_zero_bound: 3.5734935255775705e-08
all_integer_updates_identical: True

Per-layer audit


,file,layer,seq_len,approx_zero_updates,identical_updates,different_updates,max_raw_integer_decay_product,max_rounded_decay_contribution,nonzero_rounded_updates,max_local_approx_zero_bound
282,doc_005_seq_2048.pt,18,2048,153,192,0,0.004783,0,0,4.561284e-09
138,doc_002_seq_2048.pt,18,2048,148,192,0,0.000345,0,0,3.286522e-10
474,doc_009_seq_2048.pt,18,2048,148,192,0,0.009989,0,0,9.525803e-09
186,doc_003_seq_2048.pt,18,2048,147,192,0,0.000112,0,0,1.063449e-10
468,doc_009_seq_2048.pt,12,2048,141,192,0,0.014470,0,0,1.379958e-08
132,doc_002_seq_2048.pt,12,2048,141,192,0,0.004287,0,0,4.088866e-09
228,doc_004_seq_2048.pt,12,2048,140,192,0,0.007227,0,0,6.892619e-09
372,doc_007_seq_2048.pt,12,2048,138,192,0,0.008520,0,0,8.125684e-09
420,doc_008_seq_2048.pt,12,2048,137,192,0,0.000316,0,0,3.014285e-10
426,doc_008_seq_2048.pt,18,2048,137,192,0,0.004519,0,0,4.309895e-09



Saved:
/kaggle/working/shiftscan_final/state_passing/multisequence_validation/approximate_zero_validation_v1/integer_state_equivalence_per_layer.csv
/kaggle/working/shiftscan_final/state_passing/multisequence_validation/approximate_zero_validation_v1/integer_state_equivalence_summary.json



## Exact Integer Equivalence of the Approximate-Zero Policy

### Objective

This experiment tests whether replacing very small positive decay multiplications with zero changes the fixed-point state-passing recurrence.

The baseline controller uses floating-point fallback for positive decay coefficients below the supported PoT range. The candidate controller instead omits the decay contribution when its predicted local state error is below the threshold of 0.001.

### Experimental setup

- Model: Mamba-2 130M
- Data: FineWeb-Edu
- Documents: 10
- Sequence lengths: 1024 and 2048
- Layers: 24
- Fixed-point precision: F20
- Integer representation: signed int32 state, int64 intermediate arithmetic
- Scale: 1,048,576
- Local error threshold: 0.001
- Total layer records: 480
- Total recurrence updates: 69,120

### Results

| Metric | Result |
|---|---:|
| Approximate-zero updates | 17,042 |
| Identical integer updates | 69,120 |
| Different integer updates | 0 |
| Different state elements | 0 |
| Maximum unrounded decay contribution | 0.03747 integer units |
| Maximum rounded decay contribution | 0 |
| Updates with nonzero rounded decay | 0 |
| Maximum local approximation bound | 3.57 × 10⁻⁸ |
| Integer overflow | 0 |

### Interpretation

All 69,120 recurrence updates produced identical integer states under the baseline and approximate-zero policies.

For the 17,042 approximate-zero updates, the maximum unrounded decay contribution was 0.03747 integer units. This is below the 0.5 rounding threshold of the signed fixed-point rounding operation.

Consequently, the original baseline decay multiplication already contributed zero to the integer state update. Replacing it with an explicit zero contribution preserved the integer recurrence exactly in every tested update.

### Coverage improvement

The approximate-zero policy increased non-fallback coverage from 52.53% to 77.19%, reducing floating-point fallback from 47.47% to 22.81%.

Strict Dyadic2 and direct-PoT shift coverage remained 20.59%. Approximate-zero is reported as a separate arithmetic mode.

### Limitations

The result establishes exact equivalence for the tested F20 integer recurrence and captured Mamba-2 state-passing tensors. It does not establish end-to-end multiplication elimination in the full SSD scan, model-level perplexity equivalence for a deployed fixed-point kernel, or GPU speedup.

A Triton implementation and hardware benchmark are required to establish those claims.


In [169]:

# ============================================================
# SHIFT-SCAN: TRITON STATE-PASSING INTERFACE AUDIT
# ============================================================

import inspect
import torch
import triton
import mamba_ssm.ops.triton.ssd_combined as ssd_combined

print("=" * 70)
print("1. ENVIRONMENT")
print("=" * 70)
print("PyTorch:", torch.__version__)
print("Triton:", triton.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))
print("Shared memory/block:",
      torch.cuda.get_device_properties(0).shared_memory_per_block)

print("\n" + "=" * 70)
print("2. ORIGINAL PYTHON FUNCTION")
print("=" * 70)

state_fn = ssd_combined._state_passing_fwd

print("Signature:")
print(inspect.signature(state_fn))

print("\nFunction source:")
print(inspect.getsource(state_fn))

print("\n" + "=" * 70)
print("3. UNDERLYING TRITON KERNEL")
print("=" * 70)

kernel = state_fn.__globals__["_state_passing_fwd_kernel"]

print("Kernel object:", type(kernel))
print("Available configurations:")

for i, cfg in enumerate(kernel.configs):
    print(i, cfg)

print("\nTriton JIT source:")
print(kernel.fn.src)

print("\n" + "=" * 70)
print("4. ORIGINAL OUTPUT CONTRACT")
print("=" * 70)

print("Return annotations:", inspect.signature(state_fn).return_annotation)
print("Kernel argument names:")
print(kernel.fn.arg_names)

print("\n" + "=" * 70)
print("5. CURRENT KERNEL CONFIGURATION")
print("=" * 70)

print("Autotuner best_config:", getattr(kernel, "best_config", None))
print("Autotuner cache size:", len(getattr(kernel, "cache", {})))

print("\nINTERFACE AUDIT COMPLETE")


1. ENVIRONMENT
PyTorch: 2.11.0+cu130
Triton: 3.6.0
CUDA: 13.0
GPU: Tesla T4
Compute capability: (7, 5)
Shared memory/block: 49152

2. ORIGINAL PYTHON FUNCTION
Signature:
(states, dA_chunk_cumsum, initial_states=None, seq_idx=None, chunk_size=None, out_dtype=None)

Function source:
def _state_passing_fwd(states, dA_chunk_cumsum, initial_states=None, seq_idx=None, chunk_size=None,
                       out_dtype=None):
    batch, nchunks, nheads, dim = states.shape
    assert dA_chunk_cumsum.shape == (batch, nheads, nchunks)
    if initial_states is not None:
        assert initial_states.shape == (batch, nheads, dim)
    if seq_idx is not None:
        assert chunk_size is not None
        seqlen = seq_idx.shape[-1]
        assert seq_idx.shape == (batch, seqlen)
    out_dtype = states.dtype if out_dtype is None else out_dtype
    out = torch.empty((batch, nchunks, nheads, dim), device=states.device, dtype=out_dtype)
    final_states = torch.empty((batch, nheads, dim), device=states.de


## Stage 1: Triton Implementation of Shift-Scan State Passing

### Original Mamba-2 recurrence

The original state-passing operation is:

\[
S_c = \exp(\Delta_c)S_{c-1}+N_c
\]

where:
- \(S_c\) is the accumulated state after chunk \(c\).
- \(\Delta_c\) is the cumulative log-decay for that chunk.
- \(N_c\) is the new state contribution.

### Proposed Shift-Scan arithmetic

We use a signed fixed-point representation:

\[
H_c \approx S_c 2^F
\]

For a three-term Dyadic2 coefficient:

\[
a_q=1-2^{-k_1}-2^{-k_2}-2^{-k_3}
\]

the state multiplication can be approximated by:

\[
H_c=H_{c-1}
-\operatorname{RShift}(H_{c-1},k_1)
-\operatorname{RShift}(H_{c-1},k_2)
-\operatorname{RShift}(H_{c-1},k_3)
+N_c^{(int)}
\]

The kernel will support:
- Dyadic2 shift-subtract.
- Direct power-of-two shift.
- Exact-zero decay.
- Floating-point fallback.

### Implementation requirements

1. Preserve the original batch, head, chunk and state-dimension layout.
2. Use signed rounding consistently with the validated integer reference.
3. Use int64 intermediate arithmetic and check int32 output bounds.
4. Preserve the original kernel as an untouched baseline.
5. Validate every state against the Python reference before benchmarking.

The first implementation targets the state-passing operation only. It does not yet replace every decay operation in the SSD scan.


In [170]:

# ============================================================
# SHIFT-SCAN: INSPECT REAL STATE-PASSING CAPTURE
# ============================================================

from pathlib import Path
import torch

BASE = Path(
    "/kaggle/working/shiftscan_final/state_passing/"
    "multisequence_validation"
)

files = sorted(BASE.glob("doc_*_seq_*.pt"))

print("Capture files found:", len(files))
assert len(files) > 0, "No capture files found."

# Inspect one 1024-token and one 2048-token capture
selected = []

for target_len in (1024, 2048):
    match = next(
        (p for p in files if f"seq_{target_len}" in p.name),
        None
    )
    if match is not None:
        selected.append(match)

for path in selected:
    print("\n" + "=" * 75)
    print("FILE:", path.name)
    print("=" * 75)

    obj = torch.load(path, map_location="cpu", weights_only=False)

    print("Top-level type:", type(obj))

    if isinstance(obj, dict):
        print("Top-level keys:", list(obj.keys()))

        for key, value in obj.items():
            if torch.is_tensor(value):
                print(
                    f"{key}: shape={tuple(value.shape)}, "
                    f"dtype={value.dtype}"
                )
            elif isinstance(value, (list, tuple)):
                print(f"{key}: {type(value).__name__}, length={len(value)}")
            else:
                print(f"{key}: {type(value).__name__}")

        records = obj.get("records", None)

        if records is not None:
            print("\nRecord count:", len(records))

            for idx in (0, min(1, len(records)-1)):
                rec = records[idx]
                print(f"\nRecord {idx} keys:", list(rec.keys()))

                for key, value in rec.items():
                    if torch.is_tensor(value):
                        print(
                            f"  {key}: shape={tuple(value.shape)}, "
                            f"dtype={value.dtype}"
                        )
                    else:
                        print(f"  {key}: {value}")

    elif isinstance(obj, (list, tuple)):
        print("Length:", len(obj))
        if len(obj):
            print("First element type:", type(obj[0]))

print("\nCAPTURE STRUCTURE INSPECTION COMPLETE")


Capture files found: 20

FILE: doc_000_seq_1024.pt
Top-level type: <class 'dict'>
Top-level keys: ['metadata', 'records']
metadata: dict
records: list, length=24

Record count: 24

Record 0 keys: ['layer', 'doc_id', 'seq_len', 'states', 'decay_log', 'chunk_size', 'has_initial_states', 'has_seq_idx']
  layer: 0
  doc_id: 0
  seq_len: 1024
  states: shape=(1, 4, 24, 8192), dtype=torch.float32
  decay_log: shape=(1, 24, 4), dtype=torch.float32
  chunk_size: 256
  has_initial_states: False
  has_seq_idx: False

Record 1 keys: ['layer', 'doc_id', 'seq_len', 'states', 'decay_log', 'chunk_size', 'has_initial_states', 'has_seq_idx']
  layer: 1
  doc_id: 0
  seq_len: 1024
  states: shape=(1, 4, 24, 8192), dtype=torch.float32
  decay_log: shape=(1, 24, 4), dtype=torch.float32
  chunk_size: 256
  has_initial_states: False
  has_seq_idx: False

FILE: doc_000_seq_2048.pt
Top-level type: <class 'dict'>
Top-level keys: ['metadata', 'records']
metadata: dict
records: list, length=24

Record count: 24


In [171]:

# ============================================================
# SHIFT-SCAN STAGE 1A
# FIRST TRITON STATE-PASSING PROTOTYPE
#
# Real Mamba-2 capture
# F20 fixed-point arithmetic
# K1_14_T3 canonical grid
# CPU-generated controller decisions
# Exact integer output comparison
# ============================================================

import ast
import math
import numpy as np
import pandas as pd
import torch
import triton
import triton.language as tl

from pathlib import Path

assert torch.cuda.is_available()

DEVICE = "cuda"
F = 20
SCALE = 1 << F
THRESHOLD = 0.001
BLOCK = 128

CAPTURE = Path(
    "/kaggle/working/shiftscan_final/state_passing/"
    "multisequence_validation/doc_000_seq_1024.pt"
)

GRID_FILE = Path(
    "/kaggle/working/shiftscan_final/"
    "canonical_dyadic_grid_definitions.csv"
)

assert CAPTURE.exists(), CAPTURE
assert GRID_FILE.exists(), GRID_FILE

# ------------------------------------------------------------
# 1. LOAD CANONICAL GRID
# ------------------------------------------------------------

gdf = pd.read_csv(GRID_FILE)
gdf = gdf[gdf["grid"] == "K1_14_T3"].copy()

gdf = gdf.drop_duplicates(subset=["decay"])
gdf = gdf.sort_values("decay").reset_index(drop=True)

grid_a = gdf["decay"].to_numpy(dtype=np.float64)
grid_d = gdf["decay_distance"].to_numpy(dtype=np.float64)

grid_exponents = [
    tuple(int(x) for x in ast.literal_eval(str(v)))
    for v in gdf["canonical_exponents"]
]

assert len(grid_a) == 467, f"Unexpected unique grid size: {len(grid_a)}"
assert all(len(x) == 3 for x in grid_exponents)

LOG_GRID_D = np.log(grid_d)

print("Canonical grid size:", len(grid_a))
print("Decay range:", grid_a.min(), grid_a.max())

# ------------------------------------------------------------
# 2. INTEGER ROUNDING HELPERS
# ------------------------------------------------------------

def round_away_np(x):
    x = np.asarray(x, dtype=np.float64)
    return (
        np.sign(x) * np.floor(np.abs(x) + 0.5)
    ).astype(np.int64)


def round_shift_np(x, k):
    x = np.asarray(x, dtype=np.int64)
    ax = np.abs(x)
    q = (ax + (1 << (int(k) - 1))) >> int(k)
    return np.where(x < 0, -q, q).astype(np.int64)


# ------------------------------------------------------------
# 3. LOAD ONE REAL CAPTURE
# ------------------------------------------------------------

obj = torch.load(
    CAPTURE,
    map_location="cpu",
    weights_only=False
)

# Start with layer zero. We will expand after validation.
rec = obj["records"][0]

assert rec["has_initial_states"] is False
assert rec["has_seq_idx"] is False

N_float = rec["states"].detach().cpu().numpy()[0]
log_decay = rec["decay_log"].detach().cpu().float()

chunks, heads, dim = N_float.shape

assert log_decay.shape == (1, heads, chunks)

# Use FP32 exp to reproduce the actual captured decay behavior,
# including FP32 underflow to exact zero.
decay = torch.exp(log_decay).numpy()[0].astype(np.float64)

print("\nCapture:", CAPTURE.name)
print("Layer:", rec["layer"])
print("Sequence length:", rec["seq_len"])
print("N shape:", N_float.shape)
print("Decay shape:", decay.shape)
print("Chunks:", chunks, "Heads:", heads, "State dimension:", dim)

# Quantize new-state contributions once.
N_int = round_away_np(
    N_float.astype(np.float64) * SCALE
)

# ------------------------------------------------------------
# 4. CPU REFERENCE + CONTROLLER DECISIONS
# ------------------------------------------------------------

# Modes:
# 0 = exact floating-point fallback
# 1 = Dyadic2
# 2 = direct PoT
# 3 = exact zero
# 4 = approximate zero

modes = np.zeros((1, heads, chunks), dtype=np.int32)
exponents = np.zeros((1, heads, chunks, 3), dtype=np.int32)
decay_gpu = decay.astype(np.float32)[None, :, :].copy()

reference = np.zeros(
    (chunks, heads, dim),
    dtype=np.int64
)

final_reference = np.zeros(
    (heads, dim),
    dtype=np.int64
)

mode_counts = {
    "fallback": 0,
    "dyadic2": 0,
    "pot": 0,
    "exact_zero": 0,
    "approx_zero": 0
}

max_local_bound = 0.0

for h in range(heads):

    state = np.zeros(dim, dtype=np.int64)

    for c in range(chunks):

        a = float(decay[h, c])
        max_state_real = float(np.max(np.abs(state))) / SCALE

        mode = 0
        chosen_exponents = (0, 0, 0)

        # Exact FP32 zero.
        if a == 0.0:
            mode = 3
            mode_counts["exact_zero"] += 1

        # Three-term Dyadic2 candidate.
        elif (
            grid_a.min() <= a <= grid_a.max()
            and np.isfinite(a)
        ):
            d = 1.0 - a
            idx = int(np.argmin(np.abs(LOG_GRID_D - np.log(d))))

            aq = float(grid_a[idx])
            bound = abs(aq - a) * max_state_real

            if bound <= THRESHOLD:
                mode = 1
                chosen_exponents = grid_exponents[idx]
                mode_counts["dyadic2"] += 1
                max_local_bound = max(max_local_bound, bound)

        # Direct PoT candidate for small positive decays.
        if mode == 0 and 0.0 < a < 0.125 and a >= 2.0**-30:

            ks = np.arange(4, 31)
            pot_values = 2.0 ** (-ks)

            idx = int(
                np.argmin(
                    np.abs(np.log(pot_values) - np.log(a))
                )
            )

            k = int(ks[idx])
            aq = float(pot_values[idx])
            bound = abs(aq - a) * max_state_real

            if bound <= THRESHOLD:
                mode = 2
                chosen_exponents = (k, 0, 0)
                mode_counts["pot"] += 1
                max_local_bound = max(max_local_bound, bound)

        # Approximate-zero candidate.
        if mode == 0 and 0.0 < a < 2.0**-30:

            bound = a * max_state_real

            if bound <= THRESHOLD:
                mode = 4
                mode_counts["approx_zero"] += 1
                max_local_bound = max(max_local_bound, bound)

        # Remaining coefficients use exact floating fallback.
        if mode == 0:
            mode_counts["fallback"] += 1

        modes[0, h, c] = mode
        exponents[0, h, c] = chosen_exponents

        # Apply the selected recurrence to the CPU reference.
        old_state = state.copy()

        if mode == 1:
            decay_sum = np.zeros_like(old_state)

            for k in chosen_exponents:
                decay_sum += round_shift_np(old_state, k)

            state = old_state - decay_sum

        elif mode == 2:
            state = round_shift_np(old_state, chosen_exponents[0])

        elif mode in (3, 4):
            state = np.zeros_like(old_state)

        else:
            # Exact fallback, rounded back to integer state.
            state = round_away_np(a * old_state.astype(np.float64))

        state = state + N_int[c, h]

        reference[c, h] = state

    final_reference[h] = state


print("\nCPU controller mode counts:")
for k, v in mode_counts.items():
    print(f"{k:15s}: {v}")

print("Maximum local bound:", max_local_bound)

# ------------------------------------------------------------
# 5. TRITON INTEGER STATE-PASSING KERNEL
# ------------------------------------------------------------

@triton.jit
def _shift_scan_round_shift(x, k):

    ax = tl.abs(x)

    half = (
        tl.full((), 1, tl.int64)
        << (k - 1)
    )

    q = (ax + half) >> k

    return tl.where(x < 0, -q, q)


@triton.jit
def shift_scan_state_passing_kernel(
    N_ptr,
    MODE_ptr,
    EXP_ptr,
    DECAY_ptr,
    OUT_ptr,
    FINAL_ptr,

    NCHUNKS: tl.constexpr,
    NHEADS: tl.constexpr,
    DIM: tl.constexpr,

    BLOCK_SIZE: tl.constexpr
):

    pid_m = tl.program_id(0)
    pid_b = tl.program_id(1)
    pid_h = tl.program_id(2)

    offs = pid_m * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offs < DIM

    # N is contiguous [batch, chunk, head, dim].
    n_base = (
        pid_b * NCHUNKS * NHEADS * DIM
        + pid_h * DIM
    )

    out_base = n_base

    # Controller arrays:
    # MODE [batch, head, chunk]
    # EXP  [batch, head, chunk, 3]
    # DECAY [batch, head, chunk]

    ctrl_base = (
        pid_b * NHEADS * NCHUNKS
        + pid_h * NCHUNKS
    )

    state = tl.zeros((BLOCK_SIZE,), dtype=tl.int64)

    for c in range(NCHUNKS):

        mode = tl.load(MODE_ptr + ctrl_base + c)

        exp_base = (
            (pid_b * NHEADS * NCHUNKS + pid_h * NCHUNKS + c) * 3
        )

        # Dyadic2: subtract three independently rounded shifts.
        if mode == 1:

            decay_sum = tl.zeros(
                (BLOCK_SIZE,),
                dtype=tl.int64
            )

            for j in tl.static_range(3):
                k = tl.load(EXP_ptr + exp_base + j)

                decay_sum += _shift_scan_round_shift(state, k)

            state = state - decay_sum

        # Direct power-of-two decay.
        elif mode == 2:

            k = tl.load(EXP_ptr + exp_base)
            state = _shift_scan_round_shift(state, k)

        # Exact-zero and approximate-zero decay.
        elif (mode == 3) | (mode == 4):

            state = tl.zeros(
                (BLOCK_SIZE,),
                dtype=tl.int64
            )

        # Floating-point fallback.
        else:

            a = tl.load(DECAY_ptr + ctrl_base + c).to(tl.float64)

            product = state.to(tl.float64) * a

            rounded = tl.floor(
                tl.abs(product) + 0.5
            ).to(tl.int64)

            state = tl.where(
                product < 0,
                -rounded,
                rounded
            )

        # Add pre-quantized new-state contribution.
        n_offset = (
            pid_b * NCHUNKS * NHEADS * DIM
            + c * NHEADS * DIM
            + pid_h * DIM
            + offs
        )

        new_state = tl.load(
            N_ptr + n_offset,
            mask=mask,
            other=0
        ).to(tl.int64)

        state += new_state

        out_offset = (
            pid_b * NCHUNKS * NHEADS * DIM
            + c * NHEADS * DIM
            + pid_h * DIM
            + offs
        )

        tl.store(
            OUT_ptr + out_offset,
            state,
            mask=mask
        )

    final_offset = (
        pid_b * NHEADS * DIM
        + pid_h * DIM
        + offs
    )

    tl.store(
        FINAL_ptr + final_offset,
        state,
        mask=mask
    )


# ------------------------------------------------------------
# 6. LAUNCH TRITON KERNEL
# ------------------------------------------------------------

N_gpu = torch.tensor(
    N_int[None, :, :, :],
    dtype=torch.int64,
    device=DEVICE
)

MODE_gpu = torch.tensor(
    modes,
    dtype=torch.int32,
    device=DEVICE
)

EXP_gpu = torch.tensor(
    exponents,
    dtype=torch.int32,
    device=DEVICE
)

DECAY_gpu = torch.tensor(
    decay_gpu,
    dtype=torch.float32,
    device=DEVICE
)

OUT_gpu = torch.empty_like(N_gpu)

FINAL_gpu = torch.empty(
    (1, heads, dim),
    dtype=torch.int64,
    device=DEVICE
)

grid = (
    triton.cdiv(dim, BLOCK),
    1,
    heads
)

shift_scan_state_passing_kernel[grid](
    N_gpu,
    MODE_gpu,
    EXP_gpu,
    DECAY_gpu,
    OUT_gpu,
    FINAL_gpu,
    chunks,
    heads,
    dim,
    BLOCK_SIZE=BLOCK,
    num_warps=4,
    num_stages=3
)

torch.cuda.synchronize()

# ------------------------------------------------------------
# 7. EXACT INTEGER COMPARISON
# ------------------------------------------------------------

gpu_out = OUT_gpu[0].cpu().numpy()
gpu_final = FINAL_gpu[0].cpu().numpy()

different = gpu_out != reference

different_elements = int(different.sum())
total_elements = int(reference.size)

max_integer_difference = int(
    np.max(np.abs(gpu_out.astype(object) - reference.astype(object)))
)

final_different = int((gpu_final != final_reference).sum())

print("\n" + "=" * 70)
print("TRITON CORRECTNESS RESULTS")
print("=" * 70)

print("Total intermediate state elements:", total_elements)
print("Different intermediate elements:", different_elements)
print("Final state mismatches:", final_different)
print("Maximum integer difference:", max_integer_difference)

print(
    "Intermediate states exactly equal:",
    different_elements == 0
)

print(
    "Final states exactly equal:",
    final_different == 0
)

assert different_elements == 0, "Intermediate state mismatch!"
assert final_different == 0, "Final state mismatch!"

print("\nSUCCESS: TRITON MATCHES THE CPU INTEGER REFERENCE EXACTLY.")


Canonical grid size: 467
Decay range: 0.125 0.99981689453125

Capture: doc_000_seq_1024.pt
Layer: 0
Sequence length: 1024
N shape: (4, 24, 8192)
Decay shape: (24, 4)
Chunks: 4 Heads: 24 State dimension: 8192

CPU controller mode counts:
fallback       : 29
dyadic2        : 0
pot            : 63
exact_zero     : 0
approx_zero    : 4
Maximum local bound: 0.0009714145116791473

TRITON CORRECTNESS RESULTS
Total intermediate state elements: 786432
Different intermediate elements: 0
Final state mismatches: 0
Maximum integer difference: 0
Intermediate states exactly equal: True
Final states exactly equal: True

SUCCESS: TRITON MATCHES THE CPU INTEGER REFERENCE EXACTLY.


In [172]:

# ============================================================
# SHIFT-SCAN STAGE 1B
# FULL 480-RECORD TRITON CORRECTNESS VALIDATION
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import json
import time

BASE = Path(
    "/kaggle/working/shiftscan_final/state_passing/"
    "multisequence_validation"
)

OUT_DIR = BASE / "triton_validation_v1"
OUT_DIR.mkdir(parents=True, exist_ok=True)

files = sorted(BASE.glob("doc_*_seq_*.pt"))

assert len(files) == 20
assert "shift_scan_state_passing_kernel" in globals()

F = 20
SCALE = 1 << F
THRESHOLD = 0.001

# ------------------------------------------------------------
# Controller and reference implementation
# ------------------------------------------------------------

def build_reference_and_modes(rec):

    N_float = rec["states"].cpu().numpy()[0]
    logd = rec["decay_log"].cpu().float()
    decay = torch.exp(logd).numpy()[0].astype(np.float64)

    chunks, heads, dim = N_float.shape

    N_int = round_away_np(
        N_float.astype(np.float64) * SCALE
    )

    modes = np.zeros((1, heads, chunks), dtype=np.int32)
    exps = np.zeros((1, heads, chunks, 3), dtype=np.int32)

    reference = np.zeros(
        (chunks, heads, dim), dtype=np.int64
    )

    counts = {
        "fallback": 0,
        "dyadic2": 0,
        "pot": 0,
        "exact_zero": 0,
        "approx_zero": 0
    }

    max_bound = 0.0

    for h in range(heads):

        state = np.zeros(dim, dtype=np.int64)

        for c in range(chunks):

            a = float(decay[h, c])
            max_state_real = (
                float(np.max(np.abs(state))) / SCALE
            )

            mode = 0
            chosen = (0, 0, 0)

            if a == 0.0:
                mode = 3
                counts["exact_zero"] += 1

            elif grid_a.min() <= a <= grid_a.max():

                d = 1.0 - a
                idx = int(
                    np.argmin(
                        np.abs(LOG_GRID_D - np.log(d))
                    )
                )

                aq = float(grid_a[idx])
                bound = abs(aq - a) * max_state_real

                if bound <= THRESHOLD:
                    mode = 1
                    chosen = grid_exponents[idx]
                    counts["dyadic2"] += 1
                    max_bound = max(max_bound, bound)

            if mode == 0 and 0.0 < a < 0.125 and a >= 2.0**-30:

                ks = np.arange(4, 31)
                values = 2.0**(-ks)

                idx = int(
                    np.argmin(
                        np.abs(np.log(values) - np.log(a))
                    )
                )

                k = int(ks[idx])
                aq = float(values[idx])
                bound = abs(aq - a) * max_state_real

                if bound <= THRESHOLD:
                    mode = 2
                    chosen = (k, 0, 0)
                    counts["pot"] += 1
                    max_bound = max(max_bound, bound)

            if mode == 0 and 0.0 < a < 2.0**-30:

                bound = a * max_state_real

                if bound <= THRESHOLD:
                    mode = 4
                    counts["approx_zero"] += 1
                    max_bound = max(max_bound, bound)

            if mode == 0:
                counts["fallback"] += 1

            modes[0, h, c] = mode
            exps[0, h, c] = chosen

            old = state.copy()

            if mode == 1:
                decay_sum = np.zeros_like(old)
                for k in chosen:
                    decay_sum += round_shift_np(old, k)
                state = old - decay_sum

            elif mode == 2:
                state = round_shift_np(old, chosen[0])

            elif mode in (3, 4):
                state = np.zeros_like(old)

            else:
                state = round_away_np(
                    a * old.astype(np.float64)
                )

            state += N_int[c, h]
            reference[c, h] = state

    return (
        N_int, decay, modes, exps,
        reference, counts, max_bound
    )


# ------------------------------------------------------------
# Run all documents and layers
# ------------------------------------------------------------

rows = []
global_counts = {
    k: 0 for k in [
        "fallback", "dyadic2", "pot",
        "exact_zero", "approx_zero"
    ]
}

total_elements = 0
total_different = 0
total_final_different = 0
global_max_diff = 0
global_max_integer_state = 0
global_max_bound = 0.0
overflow_count = 0

start_all = time.time()

for file_idx, path in enumerate(files):

    obj = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    for rec in obj["records"]:

        N_int, decay, modes, exps, ref, counts, bound = (
            build_reference_and_modes(rec)
        )

        chunks, heads, dim = N_int.shape

        N_gpu = torch.tensor(
            N_int[None],
            dtype=torch.int64,
            device="cuda"
        )

        mode_gpu = torch.tensor(
            modes, dtype=torch.int32, device="cuda"
        )

        exp_gpu = torch.tensor(
            exps, dtype=torch.int32, device="cuda"
        )

        decay_gpu = torch.tensor(
            decay[None].astype(np.float32),
            dtype=torch.float32,
            device="cuda"
        )

        out_gpu = torch.empty_like(N_gpu)

        final_gpu = torch.empty(
            (1, heads, dim),
            dtype=torch.int64,
            device="cuda"
        )

        grid = (
            triton.cdiv(dim, BLOCK),
            1,
            heads
        )

        shift_scan_state_passing_kernel[grid](
            N_gpu,
            mode_gpu,
            exp_gpu,
            decay_gpu,
            out_gpu,
            final_gpu,
            chunks,
            heads,
            dim,
            BLOCK_SIZE=BLOCK,
            num_warps=4,
            num_stages=3
        )

        torch.cuda.synchronize()

        actual = out_gpu[0].cpu().numpy()
        actual_final = final_gpu[0].cpu().numpy()

        diff = np.abs(actual - ref)
        final_ref = ref[-1]

        mismatch = int(np.count_nonzero(diff))
        final_mismatch = int(
            np.count_nonzero(actual_final != final_ref)
        )

        max_diff = int(diff.max())
        max_state = int(np.abs(ref).max())

        total_elements += ref.size
        total_different += mismatch
        total_final_different += final_mismatch

        global_max_diff = max(global_max_diff, max_diff)
        global_max_integer_state = max(
            global_max_integer_state, max_state
        )

        if max_state > np.iinfo(np.int32).max:
            overflow_count += 1

        global_max_bound = max(global_max_bound, bound)

        for k, v in counts.items():
            global_counts[k] += v

        rows.append({
            "file": path.name,
            "doc_id": rec["doc_id"],
            "layer": rec["layer"],
            "seq_len": rec["seq_len"],
            "chunks": chunks,
            "heads": heads,
            "state_dim": dim,
            "state_elements": ref.size,
            "different_elements": mismatch,
            "final_mismatches": final_mismatch,
            "max_integer_difference": max_diff,
            "max_integer_state": max_state,
            "fallback": counts["fallback"],
            "dyadic2": counts["dyadic2"],
            "pot": counts["pot"],
            "exact_zero": counts["exact_zero"],
            "approx_zero": counts["approx_zero"],
            "max_local_bound": bound
        })

    print(
        f"Completed {file_idx + 1}/{len(files)}: "
        f"{path.name}"
    )

# ------------------------------------------------------------
# Summary and save
# ------------------------------------------------------------

df = pd.DataFrame(rows)

elapsed = time.time() - start_all

summary = {
    "capture_files": len(files),
    "layer_records": len(df),
    "total_state_elements": int(total_elements),
    "different_state_elements": int(total_different),
    "final_state_mismatches": int(total_final_different),
    "max_integer_difference": int(global_max_diff),
    "max_integer_state": int(global_max_integer_state),
    "int32_overflow_records": int(overflow_count),
    "max_local_bound": float(global_max_bound),
    "mode_counts": global_counts,
    "elapsed_seconds": float(elapsed),
    "all_intermediate_states_identical": total_different == 0,
    "all_final_states_identical": total_final_different == 0
}

df.to_csv(
    OUT_DIR / "per_layer_correctness.csv",
    index=False
)

with open(OUT_DIR / "correctness_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 75)
print("FULL TRITON VALIDATION SUMMARY")
print("=" * 75)

for k, v in summary.items():
    print(f"{k}: {v}")

print("\nSaved:", OUT_DIR)

assert len(df) == 480
assert total_different == 0
assert total_final_different == 0
assert global_max_diff == 0
assert overflow_count == 0

print("\nSUCCESS: ALL 480 LAYER RECORDS PASSED.")


Completed 1/20: doc_000_seq_1024.pt
Completed 2/20: doc_000_seq_2048.pt
Completed 3/20: doc_001_seq_1024.pt
Completed 4/20: doc_001_seq_2048.pt
Completed 5/20: doc_002_seq_1024.pt
Completed 6/20: doc_002_seq_2048.pt
Completed 7/20: doc_003_seq_1024.pt
Completed 8/20: doc_003_seq_2048.pt
Completed 9/20: doc_004_seq_1024.pt
Completed 10/20: doc_004_seq_2048.pt
Completed 11/20: doc_005_seq_1024.pt
Completed 12/20: doc_005_seq_2048.pt
Completed 13/20: doc_006_seq_1024.pt
Completed 14/20: doc_006_seq_2048.pt
Completed 15/20: doc_007_seq_1024.pt
Completed 16/20: doc_007_seq_2048.pt
Completed 17/20: doc_008_seq_1024.pt
Completed 18/20: doc_008_seq_2048.pt
Completed 19/20: doc_009_seq_1024.pt
Completed 20/20: doc_009_seq_2048.pt

FULL TRITON VALIDATION SUMMARY
capture_files: 20
layer_records: 480
total_state_elements: 566231040
different_state_elements: 0
final_state_mismatches: 0
max_integer_difference: 0
max_integer_state: 481094944
int32_overflow_records: 0
max_local_bound: 0.00099961130545

In [173]:

print("Kernel type:", type(shift_scan_state_passing_kernel))
print("Kernel source:")
print(shift_scan_state_passing_kernel.src)


Kernel type: <class 'triton.runtime.jit.JITFunction'>
Kernel source:
def shift_scan_state_passing_kernel(
    N_ptr,
    MODE_ptr,
    EXP_ptr,
    DECAY_ptr,
    OUT_ptr,
    FINAL_ptr,

    NCHUNKS: tl.constexpr,
    NHEADS: tl.constexpr,
    DIM: tl.constexpr,

    BLOCK_SIZE: tl.constexpr
):

    pid_m = tl.program_id(0)
    pid_b = tl.program_id(1)
    pid_h = tl.program_id(2)

    offs = pid_m * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offs < DIM

    # N is contiguous [batch, chunk, head, dim].
    n_base = (
        pid_b * NCHUNKS * NHEADS * DIM
        + pid_h * DIM
    )

    out_base = n_base

    # Controller arrays:
    # MODE [batch, head, chunk]
    # EXP  [batch, head, chunk, 3]
    # DECAY [batch, head, chunk]

    ctrl_base = (
        pid_b * NHEADS * NCHUNKS
        + pid_h * NCHUNKS
    )

    state = tl.zeros((BLOCK_SIZE,), dtype=tl.int64)

    for c in range(NCHUNKS):

        mode = tl.load(MODE_ptr + ctrl_base + c)

        exp_base = (
            (pi

In [174]:

@triton.jit
def shift_scan_state_passing_kernel(
    N_ptr,
    MODE_ptr,
    EXP_ptr,
    DECAY_ptr,
    OUT_ptr,
    FINAL_ptr,
    OVERFLOW_ptr,

    NCHUNKS: tl.constexpr,
    NHEADS: tl.constexpr,
    DIM: tl.constexpr,
    NBLOCKS: tl.constexpr,

    BLOCK_SIZE: tl.constexpr
):
    pid_m = tl.program_id(0)
    pid_b = tl.program_id(1)
    pid_h = tl.program_id(2)

    offs = pid_m * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offs < DIM

    state = tl.zeros((BLOCK_SIZE,), dtype=tl.int64)
    block_overflow = tl.full((), 0, dtype=tl.int32)

    ctrl_base = (
        pid_b * NHEADS * NCHUNKS
        + pid_h * NCHUNKS
    )

    for c in range(NCHUNKS):

        mode = tl.load(MODE_ptr + ctrl_base + c)

        exp_base = ctrl_base * 3 + c * 3

        # Mode 1: three-term Dyadic2 shift-subtract.
        if mode == 1:
            decay_sum = tl.zeros(
                (BLOCK_SIZE,), dtype=tl.int64
            )

            for j in tl.static_range(3):
                k = tl.load(EXP_ptr + exp_base + j)
                decay_sum += _shift_scan_round_shift(state, k)

            state = state - decay_sum

        # Mode 2: direct power-of-two shift.
        elif mode == 2:
            k = tl.load(EXP_ptr + exp_base)
            state = _shift_scan_round_shift(state, k)

        # Modes 3 and 4: exact-zero and approximate-zero.
        elif (mode == 3) | (mode == 4):
            state = tl.zeros(
                (BLOCK_SIZE,), dtype=tl.int64
            )

        # Mode 0: floating-point exact fallback.
        else:
            a = tl.load(
                DECAY_ptr + ctrl_base + c
            ).to(tl.float64)

            product = state.to(tl.float64) * a

            rounded = tl.floor(
                tl.abs(product) + 0.5
            ).to(tl.int64)

            state = tl.where(
                product < 0,
                -rounded,
                rounded
            )

        # Add pre-quantized new-state contribution.
        n_offset = (
            pid_b * NCHUNKS * NHEADS * DIM
            + c * NHEADS * DIM
            + pid_h * DIM
            + offs
        )

        new_state = tl.load(
            N_ptr + n_offset,
            mask=mask,
            other=0
        ).to(tl.int64)

        state += new_state

        # Check range BEFORE narrowing to int32.
        outside_int32 = (
            (state > 2147483647)
            | (state < -2147483648)
        )

        chunk_overflow = tl.max(
            tl.where(mask & outside_int32, 1, 0),
            axis=0
        )

        block_overflow = tl.maximum(
            block_overflow,
            chunk_overflow
        )

        # OUT_ptr is now int32 storage.
        out_offset = (
            pid_b * NCHUNKS * NHEADS * DIM
            + c * NHEADS * DIM
            + pid_h * DIM
            + offs
        )

        tl.store(
            OUT_ptr + out_offset,
            state.to(tl.int32),
            mask=mask
        )

    # Final state is also stored as int32.
    final_offset = (
        pid_b * NHEADS * DIM
        + pid_h * DIM
        + offs
    )

    tl.store(
        FINAL_ptr + final_offset,
        state.to(tl.int32),
        mask=mask
    )

    # One overflow flag per CTA.
    overflow_offset = (
        (pid_b * NHEADS + pid_h) * NBLOCKS
        + pid_m
    )

    tl.store(
        OVERFLOW_ptr + overflow_offset,
        block_overflow
    )


In [176]:

import inspect

print(inspect.signature(build_reference_and_modes))
print(inspect.getsource(build_reference_and_modes))


(rec)
def build_reference_and_modes(rec):

    N_float = rec["states"].cpu().numpy()[0]
    logd = rec["decay_log"].cpu().float()
    decay = torch.exp(logd).numpy()[0].astype(np.float64)

    chunks, heads, dim = N_float.shape

    N_int = round_away_np(
        N_float.astype(np.float64) * SCALE
    )

    modes = np.zeros((1, heads, chunks), dtype=np.int32)
    exps = np.zeros((1, heads, chunks, 3), dtype=np.int32)

    reference = np.zeros(
        (chunks, heads, dim), dtype=np.int64
    )

    counts = {
        "fallback": 0,
        "dyadic2": 0,
        "pot": 0,
        "exact_zero": 0,
        "approx_zero": 0
    }

    max_bound = 0.0

    for h in range(heads):

        state = np.zeros(dim, dtype=np.int64)

        for c in range(chunks):

            a = float(decay[h, c])
            max_state_real = (
                float(np.max(np.abs(state))) / SCALE
            )

            mode = 0
            chosen = (0, 0, 0)

            if a == 0.0:
                mode = 

In [177]:

import os
import glob
import time
import json
import numpy as np
import pandas as pd
import torch
import triton
import triton.language as tl

# ============================================================
# 1. CONFIGURATION
# ============================================================

BASE_DIR = (
    "/kaggle/working/shiftscan_final/state_passing/"
    "multisequence_validation"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR, "triton_validation_int32_v1"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

FRACTION_BITS = 20
BLOCK = 128
NUM_WARPS = 4
NUM_STAGES = 3

INT32_MAX = 2**31 - 1

# Check required existing helpers and configuration.
required = [
    "round_away_np",
    "round_shift_np",
    "grid_a",
    "grid_exponents",
    "LOG_GRID_D",
    "SCALE",
    "THRESHOLD",
    "build_reference_and_modes",
]

missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f"Missing definitions: {missing}")

assert SCALE == (1 << FRACTION_BITS), (
    f"Unexpected SCALE={SCALE}"
)

# ============================================================
# 2. TRITON INTEGER SHIFT HELPER
# ============================================================

@triton.jit
def _shift_scan_round_shift(x, k):
    ax = tl.abs(x)
    half = 1 << (k - 1)
    q = (ax + half) >> k
    return tl.where(x < 0, -q, q)


# ============================================================
# 3. INT32-STORAGE TRITON KERNEL
#    Internal arithmetic remains int64.
# ============================================================

@triton.jit
def shift_scan_state_passing_kernel(
    N_ptr,
    MODE_ptr,
    EXP_ptr,
    DECAY_ptr,
    OUT_ptr,
    FINAL_ptr,
    OVERFLOW_ptr,

    NCHUNKS: tl.constexpr,
    NHEADS: tl.constexpr,
    DIM: tl.constexpr,
    NBLOCKS: tl.constexpr,

    BLOCK_SIZE: tl.constexpr
):
    pid_m = tl.program_id(0)
    pid_b = tl.program_id(1)
    pid_h = tl.program_id(2)

    offs = pid_m * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offs < DIM

    state = tl.zeros((BLOCK_SIZE,), dtype=tl.int64)
    block_overflow = tl.full((), 0, dtype=tl.int32)

    ctrl_base = (
        pid_b * NHEADS * NCHUNKS
        + pid_h * NCHUNKS
    )

    for c in range(NCHUNKS):

        mode = tl.load(MODE_ptr + ctrl_base + c)
        exp_base = ctrl_base * 3 + c * 3

        # Mode 1: three-term Dyadic2.
        if mode == 1:
            decay_sum = tl.zeros(
                (BLOCK_SIZE,), dtype=tl.int64
            )

            for j in tl.static_range(3):
                k = tl.load(EXP_ptr + exp_base + j)
                decay_sum += _shift_scan_round_shift(state, k)

            state = state - decay_sum

        # Mode 2: direct power-of-two decay.
        elif mode == 2:
            k = tl.load(EXP_ptr + exp_base)
            state = _shift_scan_round_shift(state, k)

        # Mode 3 and 4: exact zero / approximate zero.
        elif (mode == 3) | (mode == 4):
            state = tl.zeros(
                (BLOCK_SIZE,), dtype=tl.int64
            )

        # Mode 0: floating-point fallback.
        else:
            a = tl.load(
                DECAY_ptr + ctrl_base + c
            ).to(tl.float64)

            product = state.to(tl.float64) * a

            rounded = tl.floor(
                tl.abs(product) + 0.5
            ).to(tl.int64)

            state = tl.where(
                product < 0,
                -rounded,
                rounded
            )

        # Add pre-quantized new-state contribution.
        n_offset = (
            pid_b * NCHUNKS * NHEADS * DIM
            + c * NHEADS * DIM
            + pid_h * DIM
            + offs
        )

        new_state = tl.load(
            N_ptr + n_offset,
            mask=mask,
            other=0
        ).to(tl.int64)

        state += new_state

        # Check range before narrowing to int32.
        outside = (
            (state > 2147483647)
            | (state < -2147483648)
        )

        chunk_overflow = tl.max(
            tl.where(mask & outside, 1, 0),
            axis=0
        )

        block_overflow = tl.maximum(
            block_overflow,
            chunk_overflow
        )

        # Intermediate output uses int32 storage.
        out_offset = (
            pid_b * NCHUNKS * NHEADS * DIM
            + c * NHEADS * DIM
            + pid_h * DIM
            + offs
        )

        tl.store(
            OUT_ptr + out_offset,
            state.to(tl.int32),
            mask=mask
        )

    # Final output uses int32 storage.
    final_offset = (
        pid_b * NHEADS * DIM
        + pid_h * DIM
        + offs
    )

    tl.store(
        FINAL_ptr + final_offset,
        state.to(tl.int32),
        mask=mask
    )

    overflow_offset = (
        (pid_b * NHEADS + pid_h) * NBLOCKS
        + pid_m
    )

    tl.store(
        OVERFLOW_ptr + overflow_offset,
        block_overflow
    )


# ============================================================
# 4. LOAD ALL 20 CAPTURE FILES
# ============================================================

capture_files = sorted(
    glob.glob(os.path.join(BASE_DIR, "doc_*_seq_*.pt"))
)

if len(capture_files) != 20:
    raise RuntimeError(
        f"Expected 20 capture files, found {len(capture_files)}"
    )

results = []

total_elements = 0
total_intermediate_mismatches = 0
total_final_mismatches = 0
global_max_difference = 0
global_max_abs_state = 0
global_overflow_flags = 0
global_max_local_bound = 0.0

global_mode_counts = {
    "fallback": 0,
    "dyadic2": 0,
    "pot": 0,
    "exact_zero": 0,
    "approx_zero": 0,
}

start_time = time.time()

# ============================================================
# 5. VALIDATE EVERY LAYER RECORD
# ============================================================

for file_index, path in enumerate(capture_files, start=1):

    payload = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    records = payload["records"]

    for layer_index, rec in enumerate(records):

        # ----------------------------------------------------
        # IMPORTANT FIX:
        # Your helper accepts rec and returns SEVEN values.
        # ----------------------------------------------------

        (
            N_int,
            decay,
            modes,
            exps,
            reference,
            counts,
            max_bound,
        ) = build_reference_and_modes(rec)

        N_int = np.asarray(N_int, dtype=np.int64)
        decay = np.asarray(decay, dtype=np.float64)
        modes = np.asarray(modes, dtype=np.int32)
        exps = np.asarray(exps, dtype=np.int32)
        reference = np.asarray(reference, dtype=np.int64)

        NCHUNKS, NHEADS, DIM = N_int.shape

        assert modes.shape == (1, NHEADS, NCHUNKS)
        assert exps.shape == (1, NHEADS, NCHUNKS, 3)
        assert decay.shape == (NHEADS, NCHUNKS)
        assert reference.shape == (NCHUNKS, NHEADS, DIM)

        # Final reference state is the state after the last chunk.
        reference_final = reference[-1]

        # ----------------------------------------------------
        # CPU reference range check.
        # ----------------------------------------------------

        record_max_abs = int(np.abs(reference).max())

        if record_max_abs > INT32_MAX:
            raise OverflowError(
                f"CPU reference exceeds int32 range: "
                f"{record_max_abs}; file={os.path.basename(path)}, "
                f"layer={layer_index}"
            )

        # ----------------------------------------------------
        # GPU input tensors.
        # ----------------------------------------------------

        N_gpu = torch.as_tensor(
            N_int,
            device="cuda",
            dtype=torch.int64
        ).contiguous().unsqueeze(0)

        mode_gpu = torch.as_tensor(
            modes,
            device="cuda",
            dtype=torch.int32
        ).contiguous()

        exp_gpu = torch.as_tensor(
            exps,
            device="cuda",
            dtype=torch.int32
        ).contiguous()

        decay_gpu = torch.as_tensor(
            decay,
            device="cuda",
            dtype=torch.float64
        ).contiguous().unsqueeze(0)

        # ----------------------------------------------------
        # Allocate int32 outputs.
        # ----------------------------------------------------

        NBLOCKS = triton.cdiv(DIM, BLOCK)

        out_gpu = torch.empty(
            (1, NCHUNKS, NHEADS, DIM),
            device="cuda",
            dtype=torch.int32
        )

        final_gpu = torch.empty(
            (1, NHEADS, DIM),
            device="cuda",
            dtype=torch.int32
        )

        overflow_gpu = torch.zeros(
            (NHEADS * NBLOCKS,),
            device="cuda",
            dtype=torch.int32
        )

        # ----------------------------------------------------
        # Launch kernel.
        # ----------------------------------------------------

        shift_scan_state_passing_kernel[
            (NBLOCKS, 1, NHEADS)
        ](
            N_gpu,
            mode_gpu,
            exp_gpu,
            decay_gpu,
            out_gpu,
            final_gpu,
            overflow_gpu,

            NCHUNKS,
            NHEADS,
            DIM,
            NBLOCKS,

            BLOCK,

            num_warps=NUM_WARPS,
            num_stages=NUM_STAGES
        )

        torch.cuda.synchronize()

        # ----------------------------------------------------
        # Retrieve int32 outputs, widen for exact comparison.
        # ----------------------------------------------------

        actual_intermediate = (
            out_gpu.cpu().to(torch.int64).numpy()[0]
        )

        actual_final = (
            final_gpu.cpu().to(torch.int64).numpy()[0]
        )

        overflow_flags = overflow_gpu.cpu().numpy()
        overflow_count = int(overflow_flags.sum())

        # ----------------------------------------------------
        # Exact comparison against CPU reference.
        # ----------------------------------------------------

        intermediate_diff = np.abs(
            actual_intermediate - reference
        )

        final_diff = np.abs(
            actual_final - reference_final
        )

        intermediate_mismatches = int(
            np.count_nonzero(intermediate_diff)
        )

        final_mismatches = int(
            np.count_nonzero(final_diff)
        )

        max_difference = int(
            max(
                intermediate_diff.max(initial=0),
                final_diff.max(initial=0)
            )
        )

        max_abs_state = int(
            max(
                np.abs(actual_intermediate).max(initial=0),
                np.abs(actual_final).max(initial=0)
            )
        )

        # ----------------------------------------------------
        # Aggregate mode counts.
        # ----------------------------------------------------

        for key in global_mode_counts:
            global_mode_counts[key] += int(counts[key])

        # ----------------------------------------------------
        # Save per-layer result.
        # ----------------------------------------------------

        row = {
            "file": os.path.basename(path),
            "layer": layer_index,
            "chunks": NCHUNKS,
            "heads": NHEADS,
            "state_dim": DIM,
            "state_elements": int(
                actual_intermediate.size + actual_final.size
            ),
            "intermediate_mismatches": intermediate_mismatches,
            "final_mismatches": final_mismatches,
            "max_integer_difference": max_difference,
            "max_abs_integer_state": max_abs_state,
            "overflow_flags": overflow_count,
            "max_local_bound": float(max_bound),
            **{
                f"mode_{key}": int(value)
                for key, value in counts.items()
            },
        }

        results.append(row)

        total_elements += row["state_elements"]
        total_intermediate_mismatches += intermediate_mismatches
        total_final_mismatches += final_mismatches

        global_max_difference = max(
            global_max_difference,
            max_difference
        )

        global_max_abs_state = max(
            global_max_abs_state,
            max_abs_state
        )

        global_overflow_flags += overflow_count

        global_max_local_bound = max(
            global_max_local_bound,
            float(max_bound)
        )

    print(
        f"Completed {file_index}/{len(capture_files)}: "
        f"{os.path.basename(path)}"
    )

# ============================================================
# 6. SAVE VALIDATION RESULTS
# ============================================================

elapsed = time.time() - start_time

df = pd.DataFrame(results)

csv_path = os.path.join(
    OUTPUT_DIR, "per_layer_correctness.csv"
)

df.to_csv(csv_path, index=False)

summary = {
    "capture_files": len(capture_files),
    "layer_records": len(df),
    "total_state_elements": total_elements,
    "different_state_elements": total_intermediate_mismatches,
    "final_state_mismatches": total_final_mismatches,
    "max_integer_difference": global_max_difference,
    "max_integer_state": global_max_abs_state,
    "int32_overflow_flags": global_overflow_flags,
    "max_local_bound": global_max_local_bound,
    "mode_counts": global_mode_counts,
    "elapsed_seconds": elapsed,
    "storage_dtype": "int32",
    "internal_arithmetic_dtype": "int64",
    "fraction_bits": FRACTION_BITS,
    "all_intermediate_states_identical": (
        total_intermediate_mismatches == 0
    ),
    "all_final_states_identical": (
        total_final_mismatches == 0
    ),
}

json_path = os.path.join(
    OUTPUT_DIR, "correctness_summary.json"
)

with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 75)
print("INT32 STORAGE TRITON VALIDATION SUMMARY")
print("=" * 75)

for key, value in summary.items():
    print(f"{key}: {value}")

print("\nSaved:", OUTPUT_DIR)

# ============================================================
# 7. STRICT VALIDATION
# ============================================================

assert len(df) == 480, (
    f"Expected 480 layer records, found {len(df)}"
)

assert total_intermediate_mismatches == 0, (
    "Intermediate state mismatch detected"
)

assert total_final_mismatches == 0, (
    "Final state mismatch detected"
)

assert global_max_difference == 0, (
    f"Nonzero integer difference: {global_max_difference}"
)

assert global_overflow_flags == 0, (
    f"Int32 overflow flags detected: {global_overflow_flags}"
)

assert global_max_abs_state <= INT32_MAX, (
    f"Observed state exceeds int32 range: {global_max_abs_state}"
)

print("\nSUCCESS: ALL 480 RECORDS PASSED WITH INT32 STORAGE.")


Completed 1/20: doc_000_seq_1024.pt
Completed 2/20: doc_000_seq_2048.pt
Completed 3/20: doc_001_seq_1024.pt
Completed 4/20: doc_001_seq_2048.pt
Completed 5/20: doc_002_seq_1024.pt
Completed 6/20: doc_002_seq_2048.pt
Completed 7/20: doc_003_seq_1024.pt
Completed 8/20: doc_003_seq_2048.pt
Completed 9/20: doc_004_seq_1024.pt
Completed 10/20: doc_004_seq_2048.pt
Completed 11/20: doc_005_seq_1024.pt
Completed 12/20: doc_005_seq_2048.pt
Completed 13/20: doc_006_seq_1024.pt
Completed 14/20: doc_006_seq_2048.pt
Completed 15/20: doc_007_seq_1024.pt
Completed 16/20: doc_007_seq_2048.pt
Completed 17/20: doc_008_seq_1024.pt
Completed 18/20: doc_008_seq_2048.pt
Completed 19/20: doc_009_seq_1024.pt
Completed 20/20: doc_009_seq_2048.pt

INT32 STORAGE TRITON VALIDATION SUMMARY
capture_files: 20
layer_records: 480
total_state_elements: 660602880
different_state_elements: 0
final_state_mismatches: 0
max_integer_difference: 0
max_integer_state: 481094944
int32_overflow_flags: 0
max_local_bound: 0.0009996

In [178]:

import os
import glob
import time
import numpy as np
import pandas as pd
import torch
import triton
import triton.language as tl

# ============================================================
# 1. CONFIGURATION
# ============================================================

BASE_DIR = (
    "/kaggle/working/shiftscan_final/state_passing/"
    "multisequence_validation"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR, "gpu_benchmark_v1"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

BLOCK = 128
WARPS = 4
STAGES = 3

FRACTION_BITS = 20
SCALE = 1 << FRACTION_BITS

WARMUP_MS = 30
REPETITION_MS = 100

DOC_INDICES = [0, 4, 8]
LAYER_INDICES = [0, 8, 16, 23]
SEQ_LENGTHS = [1024, 2048]

required = [
    "build_reference_and_modes",
    "shift_scan_state_passing_kernel",
    "round_away_np",
]

missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f"Missing definitions: {missing}")

# ============================================================
# 2. FP32 REFERENCE TRITON KERNEL
# ============================================================

@triton.jit
def fp32_state_passing_reference(
    N_ptr,
    DECAY_ptr,
    OUT_ptr,
    FINAL_ptr,

    NCHUNKS: tl.constexpr,
    NHEADS: tl.constexpr,
    DIM: tl.constexpr,

    BLOCK_SIZE: tl.constexpr
):
    pid_m = tl.program_id(0)
    pid_h = tl.program_id(2)

    offs = pid_m * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offs < DIM

    state = tl.zeros((BLOCK_SIZE,), dtype=tl.float32)

    for c in range(NCHUNKS):

        decay_offset = pid_h * NCHUNKS + c
        a = tl.load(DECAY_ptr + decay_offset).to(tl.float32)

        n_offset = (
            c * NHEADS * DIM
            + pid_h * DIM
            + offs
        )

        new_state = tl.load(
            N_ptr + n_offset,
            mask=mask,
            other=0
        ).to(tl.float32)

        state = a * state + new_state

        out_offset = (
            c * NHEADS * DIM
            + pid_h * DIM
            + offs
        )

        tl.store(
            OUT_ptr + out_offset,
            state,
            mask=mask
        )

    final_offset = pid_h * DIM + offs

    tl.store(
        FINAL_ptr + final_offset,
        state,
        mask=mask
    )


# ============================================================
# 3. LOAD SELECTED CAPTURES
# ============================================================

capture_files = sorted(
    glob.glob(os.path.join(BASE_DIR, "doc_*_seq_*.pt"))
)

if len(capture_files) != 20:
    raise RuntimeError(
        f"Expected 20 capture files, found {len(capture_files)}"
    )

selected_files = []

for doc_idx in DOC_INDICES:
    for seq_len in SEQ_LENGTHS:
        matches = [
            p for p in capture_files
            if f"doc_{doc_idx:03d}_seq_{seq_len}.pt"
            in os.path.basename(p)
        ]

        if len(matches) != 1:
            raise RuntimeError(
                f"Could not uniquely find doc {doc_idx}, "
                f"sequence length {seq_len}"
            )

        selected_files.append(matches[0])

# ============================================================
# 4. BENCHMARK LOOP
# ============================================================

results = []

for file_index, path in enumerate(selected_files, start=1):

    payload = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    records = payload["records"]

    for layer_idx in LAYER_INDICES:

        rec = records[layer_idx]

        # ----------------------------------------------------
        # Rebuild the same controller used in validation.
        # ----------------------------------------------------

        (
            N_int,
            decay,
            modes,
            exps,
            reference,
            counts,
            max_bound,
        ) = build_reference_and_modes(rec)

        N_float = (
            rec["states"].detach().cpu().float().numpy()[0]
        )

        NCHUNKS, NHEADS, DIM = N_float.shape

        # ----------------------------------------------------
        # Allocate GPU inputs.
        # ----------------------------------------------------

        N_fp32_gpu = torch.as_tensor(
            N_float,
            device="cuda",
            dtype=torch.float32
        ).contiguous()

        N_int_gpu = torch.as_tensor(
            N_int,
            device="cuda",
            dtype=torch.int64
        ).contiguous().unsqueeze(0)

        mode_gpu = torch.as_tensor(
            modes,
            device="cuda",
            dtype=torch.int32
        ).contiguous()

        exp_gpu = torch.as_tensor(
            exps,
            device="cuda",
            dtype=torch.int32
        ).contiguous()

        decay_fp64_gpu = torch.as_tensor(
            decay,
            device="cuda",
            dtype=torch.float64
        ).contiguous().unsqueeze(0)

        decay_fp32_gpu = torch.as_tensor(
            decay,
            device="cuda",
            dtype=torch.float32
        ).contiguous()

        # ----------------------------------------------------
        # Allocate outputs.
        # ----------------------------------------------------

        NBLOCKS = triton.cdiv(DIM, BLOCK)

        candidate_out = torch.empty(
            (1, NCHUNKS, NHEADS, DIM),
            device="cuda",
            dtype=torch.int32
        )

        candidate_final = torch.empty(
            (1, NHEADS, DIM),
            device="cuda",
            dtype=torch.int32
        )

        overflow_gpu = torch.zeros(
            (NHEADS * NBLOCKS,),
            device="cuda",
            dtype=torch.int32
        )

        reference_out = torch.empty(
            (NCHUNKS, NHEADS, DIM),
            device="cuda",
            dtype=torch.float32
        )

        reference_final = torch.empty(
            (NHEADS, DIM),
            device="cuda",
            dtype=torch.float32
        )

        grid = (NBLOCKS, 1, NHEADS)

        # ----------------------------------------------------
        # Define launch functions.
        # ----------------------------------------------------

        def run_candidate():
            shift_scan_state_passing_kernel[grid](
                N_int_gpu,
                mode_gpu,
                exp_gpu,
                decay_fp64_gpu,
                candidate_out,
                candidate_final,
                overflow_gpu,

                NCHUNKS,
                NHEADS,
                DIM,
                NBLOCKS,

                BLOCK,

                num_warps=WARPS,
                num_stages=STAGES
            )

        def run_reference():
            fp32_state_passing_reference[grid](
                N_fp32_gpu,
                decay_fp32_gpu,
                reference_out,
                reference_final,

                NCHUNKS,
                NHEADS,
                DIM,

                BLOCK,

                num_warps=WARPS,
                num_stages=STAGES
            )

        # ----------------------------------------------------
        # Compile and warm up both kernels.
        # ----------------------------------------------------

        for _ in range(5):
            run_candidate()
            run_reference()

        torch.cuda.synchronize()

        # ----------------------------------------------------
        # Benchmark using Triton's CUDA-event benchmark.
        # ----------------------------------------------------

        candidate_ms = triton.testing.do_bench(
            run_candidate,
            warmup=WARMUP_MS,
            rep=REPETITION_MS
        )

        reference_ms = triton.testing.do_bench(
            run_reference,
            warmup=WARMUP_MS,
            rep=REPETITION_MS
        )

        torch.cuda.synchronize()

        overflow_count = int(
            overflow_gpu.sum().item()
        )

        speed_ratio = reference_ms / candidate_ms

        row = {
            "file": os.path.basename(path),
            "sequence_length": NCHUNKS * 256,
            "layer": layer_idx,
            "chunks": NCHUNKS,
            "heads": NHEADS,
            "state_dim": DIM,
            "candidate_ms": float(candidate_ms),
            "fp32_reference_ms": float(reference_ms),
            "reference_over_candidate": float(speed_ratio),
            "candidate_overflow_flags": overflow_count,
            "fallback_count": int(counts["fallback"]),
            "dyadic2_count": int(counts["dyadic2"]),
            "pot_count": int(counts["pot"]),
            "exact_zero_count": int(counts["exact_zero"]),
            "approx_zero_count": int(counts["approx_zero"]),
        }

        results.append(row)

        print(
            f"{file_index}/{len(selected_files)} | "
            f"Layer {layer_idx:02d} | "
            f"FP32 {reference_ms:.4f} ms | "
            f"Shift-Scan {candidate_ms:.4f} ms | "
            f"Ratio {speed_ratio:.3f}x"
        )

# ============================================================
# 5. SAVE RESULTS
# ============================================================

df = pd.DataFrame(results)

csv_path = os.path.join(
    OUTPUT_DIR, "kernel_benchmark_results.csv"
)

df.to_csv(csv_path, index=False)

summary = (
    df.groupby("sequence_length")
    .agg(
        records=("layer", "count"),
        median_candidate_ms=("candidate_ms", "median"),
        median_fp32_ms=("fp32_reference_ms", "median"),
        median_speed_ratio=("reference_over_candidate", "median"),
        mean_candidate_ms=("candidate_ms", "mean"),
        mean_fp32_ms=("fp32_reference_ms", "mean"),
        overflow_flags=("candidate_overflow_flags", "sum"),
    )
    .reset_index()
)

summary_path = os.path.join(
    OUTPUT_DIR, "benchmark_summary.csv"
)

summary.to_csv(summary_path, index=False)

print("\n" + "=" * 75)
print("GPU KERNEL BENCHMARK SUMMARY")
print("=" * 75)

print(summary.to_string(index=False))

print("\nOverall median candidate latency:",
      round(df["candidate_ms"].median(), 6), "ms")

print("Overall median FP32 latency:",
      round(df["fp32_reference_ms"].median(), 6), "ms")

print("Overall median reference/candidate ratio:",
      round(df["reference_over_candidate"].median(), 4), "x")

print("Total overflow flags:",
      int(df["candidate_overflow_flags"].sum()))

print("\nSaved:", OUTPUT_DIR)

assert len(df) == 24, (
    f"Expected 24 benchmark records, got {len(df)}"
)

assert int(df["candidate_overflow_flags"].sum()) == 0

print("\nBENCHMARK COMPLETED.")


1/6 | Layer 00 | FP32 0.0361 ms | Shift-Scan 0.0714 ms | Ratio 0.506x
1/6 | Layer 08 | FP32 0.0348 ms | Shift-Scan 0.0547 ms | Ratio 0.636x
1/6 | Layer 16 | FP32 0.0345 ms | Shift-Scan 0.0466 ms | Ratio 0.741x
1/6 | Layer 23 | FP32 0.0338 ms | Shift-Scan 0.0468 ms | Ratio 0.723x
2/6 | Layer 00 | FP32 0.0617 ms | Shift-Scan 0.0865 ms | Ratio 0.713x
2/6 | Layer 08 | FP32 0.0614 ms | Shift-Scan 0.0864 ms | Ratio 0.710x
2/6 | Layer 16 | FP32 0.0613 ms | Shift-Scan 0.0851 ms | Ratio 0.721x
2/6 | Layer 23 | FP32 0.0611 ms | Shift-Scan 0.0860 ms | Ratio 0.711x
3/6 | Layer 00 | FP32 0.0345 ms | Shift-Scan 0.0475 ms | Ratio 0.728x
3/6 | Layer 08 | FP32 0.0343 ms | Shift-Scan 0.0479 ms | Ratio 0.717x
3/6 | Layer 16 | FP32 0.0339 ms | Shift-Scan 0.0467 ms | Ratio 0.725x
3/6 | Layer 23 | FP32 0.0339 ms | Shift-Scan 0.0468 ms | Ratio 0.724x
4/6 | Layer 00 | FP32 0.0607 ms | Shift-Scan 0.0865 ms | Ratio 0.703x
4/6 | Layer 08 | FP32 0.0611 ms | Shift-Scan 0.0864 ms | Ratio 0.708x
4/6 | Layer 16 | FP3

In [179]:

import os
import glob
import time
import numpy as np
import pandas as pd
import torch
import triton
import triton.language as tl

# ============================================================
# 1. CONFIGURATION
# ============================================================

BASE_DIR = (
    "/kaggle/working/shiftscan_final/state_passing/"
    "multisequence_validation"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR, "gpu_benchmark_diagnostic_v2"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

BLOCK = 128
WARPS = 4
STAGES = 3

DOC_INDICES = [0, 4, 8]
SEQ_LENGTHS = [1024, 2048]
LAYER_INDICES = [0, 8, 16, 23]

WARMUP_MS = 30
REPETITION_MS = 100

required = [
    "build_reference_and_modes",
    "shift_scan_state_passing_kernel",
    "round_away_np",
]

missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f"Missing required definitions: {missing}")

# ============================================================
# 2. FP32 REFERENCE KERNEL
# ============================================================

@triton.jit
def fp32_reference_kernel_v2(
    N_ptr,
    DECAY_ptr,
    OUT_ptr,
    FINAL_ptr,
    NCHUNKS: tl.constexpr,
    NHEADS: tl.constexpr,
    DIM: tl.constexpr,
    BLOCK_SIZE: tl.constexpr
):
    pid_m = tl.program_id(0)
    pid_h = tl.program_id(2)

    offs = pid_m * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offs < DIM

    state = tl.zeros((BLOCK_SIZE,), dtype=tl.float32)

    for c in range(NCHUNKS):
        a = tl.load(
            DECAY_ptr + pid_h * NCHUNKS + c
        ).to(tl.float32)

        n_offset = (
            c * NHEADS * DIM
            + pid_h * DIM
            + offs
        )

        new_state = tl.load(
            N_ptr + n_offset,
            mask=mask,
            other=0
        ).to(tl.float32)

        state = a * state + new_state

        tl.store(
            OUT_ptr + n_offset,
            state,
            mask=mask
        )

    final_offset = pid_h * DIM + offs

    tl.store(
        FINAL_ptr + final_offset,
        state,
        mask=mask
    )


# ============================================================
# 3. ISOLATED INTEGER SHIFT-PATH MICROKERNEL
#    Runs only updates assigned to modes 1, 2, 3, or 4.
#    Mode 0 is skipped.
# ============================================================

@triton.jit
def shift_path_microkernel(
    N_ptr,
    MODE_ptr,
    EXP_ptr,
    OUT_ptr,

    NCHUNKS: tl.constexpr,
    NHEADS: tl.constexpr,
    DIM: tl.constexpr,
    TOTAL: tl.constexpr,

    BLOCK_SIZE: tl.constexpr
):
    idx = tl.program_id(0) * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = idx < TOTAL

    chunk = idx // (NHEADS * DIM)
    head = (idx // DIM) % NHEADS

    ctrl = head * NCHUNKS + chunk
    mode = tl.load(MODE_ptr + ctrl, mask=mask, other=0)

    state = tl.load(
        N_ptr + idx,
        mask=mask,
        other=0
    ).to(tl.int64)

    result = tl.zeros((BLOCK_SIZE,), dtype=tl.int64)

    if tl.sum(tl.cast(mask, tl.int32), 0) > 0:

        # Dyadic2: three rounded shifts.
        if tl.sum(tl.cast(mask & (mode == 1), tl.int32), 0) > 0:
            pass

    # Elementwise selection keeps the microbenchmark simple.
    # Compute all three shifts, then select the applicable mode.
    exp_base = ctrl * 3

    k0 = tl.load(EXP_ptr + exp_base, mask=mask, other=0)
    k1 = tl.load(EXP_ptr + exp_base + 1, mask=mask, other=0)
    k2 = tl.load(EXP_ptr + exp_base + 2, mask=mask, other=0)

    ax = tl.abs(state)

    half0 = 1 << (k0 - 1)
    half1 = 1 << (k1 - 1)
    half2 = 1 << (k2 - 1)

    q0 = (ax + half0) >> k0
    q1 = (ax + half1) >> k1
    q2 = (ax + half2) >> k2

    r0 = tl.where(state < 0, -q0, q0)
    r1 = tl.where(state < 0, -q1, q1)
    r2 = tl.where(state < 0, -q2, q2)

    dyadic_result = state - r0 - r1 - r2
    pot_result = r0

    result = tl.where(
        mode == 1,
        dyadic_result,
        result
    )

    result = tl.where(
        mode == 2,
        pot_result,
        result
    )

    # Exact-zero and approximate-zero result in zero.
    result = tl.where(
        (mode == 3) | (mode == 4),
        0,
        result
    )

    tl.store(OUT_ptr + idx, result, mask=mask)


# ============================================================
# 4. ISOLATED FALLBACK MICROKERNEL
#    Performs FP64 multiply + signed rounding only for mode 0.
# ============================================================

@triton.jit
def fallback_microkernel(
    N_ptr,
    MODE_ptr,
    DECAY_ptr,
    OUT_ptr,

    NCHUNKS: tl.constexpr,
    NHEADS: tl.constexpr,
    DIM: tl.constexpr,
    TOTAL: tl.constexpr,

    BLOCK_SIZE: tl.constexpr
):
    idx = tl.program_id(0) * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = idx < TOTAL

    chunk = idx // (NHEADS * DIM)
    head = (idx // DIM) % NHEADS

    ctrl = head * NCHUNKS + chunk

    mode = tl.load(
        MODE_ptr + ctrl,
        mask=mask,
        other=1
    )

    state = tl.load(
        N_ptr + idx,
        mask=mask,
        other=0
    ).to(tl.float64)

    a = tl.load(
        DECAY_ptr + ctrl,
        mask=mask,
        other=0
    ).to(tl.float64)

    product = state * a

    rounded = tl.floor(
        tl.abs(product) + 0.5
    ).to(tl.int64)

    result = tl.where(
        product < 0,
        -rounded,
        rounded
    )

    result = tl.where(
        mode == 0,
        result,
        0
    )

    tl.store(
        OUT_ptr + idx,
        result,
        mask=mask
    )


# ============================================================
# 5. SELECT CAPTURES
# ============================================================

capture_files = sorted(
    glob.glob(os.path.join(BASE_DIR, "doc_*_seq_*.pt"))
)

if len(capture_files) != 20:
    raise RuntimeError(
        f"Expected 20 capture files, found {len(capture_files)}"
    )

selected_files = []

for doc_idx in DOC_INDICES:
    for seq_len in SEQ_LENGTHS:
        matches = [
            p for p in capture_files
            if f"doc_{doc_idx:03d}_seq_{seq_len}.pt"
            in os.path.basename(p)
        ]

        if len(matches) != 1:
            raise RuntimeError(
                f"Capture missing or duplicated: "
                f"doc={doc_idx}, length={seq_len}"
            )

        selected_files.append(matches[0])


# ============================================================
# 6. BENCHMARK
# ============================================================

rows = []

for file_index, path in enumerate(selected_files, start=1):

    payload = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    records = payload["records"]

    for layer_idx in LAYER_INDICES:

        rec = records[layer_idx]

        (
            N_int,
            decay,
            modes,
            exps,
            reference,
            counts,
            max_bound,
        ) = build_reference_and_modes(rec)

        N_float = (
            rec["states"].detach().cpu().float().numpy()[0]
        )

        NCHUNKS, NHEADS, DIM = N_float.shape
        TOTAL = NCHUNKS * NHEADS * DIM

        # GPU inputs.
        N_fp32_gpu = torch.as_tensor(
            N_float,
            device="cuda",
            dtype=torch.float32
        ).contiguous()

        N_int_gpu = torch.as_tensor(
            N_int,
            device="cuda",
            dtype=torch.int64
        ).contiguous().unsqueeze(0)

        mode_gpu = torch.as_tensor(
            modes,
            device="cuda",
            dtype=torch.int32
        ).contiguous()

        exp_gpu = torch.as_tensor(
            exps,
            device="cuda",
            dtype=torch.int32
        ).contiguous()

        decay64_gpu = torch.as_tensor(
            decay,
            device="cuda",
            dtype=torch.float64
        ).contiguous().unsqueeze(0)

        decay32_gpu = torch.as_tensor(
            decay,
            device="cuda",
            dtype=torch.float32
        ).contiguous()

        # Outputs.
        NBLOCKS = triton.cdiv(DIM, BLOCK)

        candidate_out = torch.empty(
            (1, NCHUNKS, NHEADS, DIM),
            device="cuda",
            dtype=torch.int32
        )

        candidate_final = torch.empty(
            (1, NHEADS, DIM),
            device="cuda",
            dtype=torch.int32
        )

        overflow_gpu = torch.zeros(
            (NHEADS * NBLOCKS,),
            device="cuda",
            dtype=torch.int32
        )

        reference_out = torch.empty(
            (NCHUNKS, NHEADS, DIM),
            device="cuda",
            dtype=torch.float32
        )

        reference_final = torch.empty(
            (NHEADS, DIM),
            device="cuda",
            dtype=torch.float32
        )

        shift_out = torch.empty(
            (TOTAL,),
            device="cuda",
            dtype=torch.int64
        )

        fallback_out = torch.empty(
            (TOTAL,),
            device="cuda",
            dtype=torch.int64
        )

        grid = (NBLOCKS, 1, NHEADS)
        flat_grid = (triton.cdiv(TOTAL, BLOCK),)

        # ----------------------------------------------------
        # Four benchmark functions.
        # ----------------------------------------------------

        def run_fp32():
            fp32_reference_kernel_v2[grid](
                N_fp32_gpu,
                decay32_gpu,
                reference_out,
                reference_final,
                NCHUNKS,
                NHEADS,
                DIM,
                BLOCK,
                num_warps=WARPS,
                num_stages=STAGES
            )

        def run_adaptive():
            shift_scan_state_passing_kernel[grid](
                N_int_gpu,
                mode_gpu,
                exp_gpu,
                decay64_gpu,
                candidate_out,
                candidate_final,
                overflow_gpu,
                NCHUNKS,
                NHEADS,
                DIM,
                NBLOCKS,
                BLOCK,
                num_warps=WARPS,
                num_stages=STAGES
            )

        def run_shift_only():
            shift_path_microkernel[flat_grid](
                N_int_gpu,
                mode_gpu,
                exp_gpu,
                shift_out,
                NCHUNKS,
                NHEADS,
                DIM,
                TOTAL,
                BLOCK,
                num_warps=WARPS,
                num_stages=STAGES
            )

        def run_fallback_only():
            fallback_microkernel[flat_grid](
                N_int_gpu,
                mode_gpu,
                decay64_gpu,
                fallback_out,
                NCHUNKS,
                NHEADS,
                DIM,
                TOTAL,
                BLOCK,
                num_warps=WARPS,
                num_stages=STAGES
            )

        # Warmup and compilation.
        for _ in range(5):
            run_fp32()
            run_adaptive()
            run_shift_only()
            run_fallback_only()

        torch.cuda.synchronize()

        # Benchmark each kernel.
        fp32_ms = triton.testing.do_bench(
            run_fp32,
            warmup=WARMUP_MS,
            rep=REPETITION_MS
        )

        adaptive_ms = triton.testing.do_bench(
            run_adaptive,
            warmup=WARMUP_MS,
            rep=REPETITION_MS
        )

        shift_ms = triton.testing.do_bench(
            run_shift_only,
            warmup=WARMUP_MS,
            rep=REPETITION_MS
        )

        fallback_ms = triton.testing.do_bench(
            run_fallback_only,
            warmup=WARMUP_MS,
            rep=REPETITION_MS
        )

        torch.cuda.synchronize()

        overflow_count = int(
            overflow_gpu.sum().item()
        )

        total_updates = NCHUNKS * NHEADS

        strict_shift = (
            counts["dyadic2"] + counts["pot"]
        )

        row = {
            "file": os.path.basename(path),
            "sequence_length": NCHUNKS * 256,
            "layer": layer_idx,
            "chunks": NCHUNKS,
            "heads": NHEADS,
            "state_dim": DIM,

            "fp32_reference_ms": float(fp32_ms),
            "adaptive_shiftscan_ms": float(adaptive_ms),

            "shift_only_microkernel_ms": float(shift_ms),
            "fallback_only_microkernel_ms": float(fallback_ms),

            "adaptive_over_fp32": float(adaptive_ms / fp32_ms),

            "fallback_updates": int(counts["fallback"]),
            "dyadic2_updates": int(counts["dyadic2"]),
            "pot_updates": int(counts["pot"]),
            "exact_zero_updates": int(counts["exact_zero"]),
            "approx_zero_updates": int(counts["approx_zero"]),

            "strict_shift_pct": (
                100.0 * strict_shift / total_updates
            ),

            "fallback_pct": (
                100.0 * counts["fallback"] / total_updates
            ),

            "overflow_flags": overflow_count,
        }

        rows.append(row)

        print(
            f"{file_index}/{len(selected_files)} "
            f"Layer {layer_idx:02d} | "
            f"FP32={fp32_ms:.4f} ms | "
            f"Adaptive={adaptive_ms:.4f} ms | "
            f"Shift-only={shift_ms:.4f} ms | "
            f"Fallback-only={fallback_ms:.4f} ms"
        )


# ============================================================
# 7. SAVE RESULTS
# ============================================================

df = pd.DataFrame(rows)

csv_path = os.path.join(
    OUTPUT_DIR, "four_way_diagnostic_benchmark.csv"
)

df.to_csv(csv_path, index=False)

summary = (
    df.groupby("sequence_length")
    .agg(
        records=("layer", "count"),
        fp32_median_ms=("fp32_reference_ms", "median"),
        adaptive_median_ms=("adaptive_shiftscan_ms", "median"),
        shift_only_median_ms=("shift_only_microkernel_ms", "median"),
        fallback_only_median_ms=("fallback_only_microkernel_ms", "median"),
        median_adaptive_over_fp32=("adaptive_over_fp32", "median"),
        mean_strict_shift_pct=("strict_shift_pct", "mean"),
        mean_fallback_pct=("fallback_pct", "mean"),
        overflow_flags=("overflow_flags", "sum"),
    )
    .reset_index()
)

summary_path = os.path.join(
    OUTPUT_DIR, "four_way_diagnostic_summary.csv"
)

summary.to_csv(summary_path, index=False)

print("\n" + "=" * 80)
print("FOUR-WAY GPU DIAGNOSTIC SUMMARY")
print("=" * 80)

print(summary.to_string(index=False))

print("\nOverall medians:")
for col in [
    "fp32_reference_ms",
    "adaptive_shiftscan_ms",
    "shift_only_microkernel_ms",
    "fallback_only_microkernel_ms",
]:
    print(
        col,
        round(float(df[col].median()), 6),
        "ms"
    )

print("\nTotal overflow flags:",
      int(df["overflow_flags"].sum()))

print("\nSaved:", OUTPUT_DIR)

assert len(df) == 24
assert int(df["overflow_flags"].sum()) == 0

print("\nDIAGNOSTIC BENCHMARK COMPLETED.")


1/6 Layer 00 | FP32=0.0349 ms | Adaptive=0.0713 ms | Shift-only=0.0569 ms | Fallback-only=0.0556 ms
1/6 Layer 08 | FP32=0.0342 ms | Adaptive=0.0479 ms | Shift-only=0.0552 ms | Fallback-only=0.0554 ms
1/6 Layer 16 | FP32=0.0342 ms | Adaptive=0.0470 ms | Shift-only=0.0553 ms | Fallback-only=0.0556 ms
1/6 Layer 23 | FP32=0.0341 ms | Adaptive=0.0471 ms | Shift-only=0.0549 ms | Fallback-only=0.0553 ms
2/6 Layer 00 | FP32=0.0611 ms | Adaptive=0.0865 ms | Shift-only=0.1064 ms | Fallback-only=0.1066 ms
2/6 Layer 08 | FP32=0.0615 ms | Adaptive=0.0869 ms | Shift-only=0.1062 ms | Fallback-only=0.1065 ms
2/6 Layer 16 | FP32=0.0612 ms | Adaptive=0.0853 ms | Shift-only=0.1064 ms | Fallback-only=0.1068 ms
2/6 Layer 23 | FP32=0.0611 ms | Adaptive=0.0854 ms | Shift-only=0.1064 ms | Fallback-only=0.1066 ms
3/6 Layer 00 | FP32=0.0341 ms | Adaptive=0.0471 ms | Shift-only=0.0551 ms | Fallback-only=0.0555 ms
3/6 Layer 08 | FP32=0.0344 ms | Adaptive=0.0478 ms | Shift-only=0.0550 ms | Fallback-only=0.0555 ms


In [180]:

import torch
import triton

print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))

print("\nKernel information:")
print("Name:", shift_scan_state_passing_kernel.__name__)
print("Number of warps:", 4)
print("Number of stages:", 3)

print("\nAvailable compiled kernel metadata:")

if hasattr(shift_scan_state_passing_kernel, "cache"):
    cache = shift_scan_state_passing_kernel.cache
    print("Cache type:", type(cache))

    try:
        print("Cache entries:", len(cache))
    except Exception:
        print("Cache entries: unavailable")

print("\nTriton kernel source:")
print(shift_scan_state_passing_kernel.src)

print("\nGPU memory:")
print(
    "Allocated:",
    round(torch.cuda.memory_allocated() / 1024**2, 2),
    "MiB"
)
print(
    "Reserved:",
    round(torch.cuda.memory_reserved() / 1024**2, 2),
    "MiB"
)


GPU: Tesla T4
Compute capability: (7, 5)

Kernel information:
Name: shift_scan_state_passing_kernel
Number of warps: 4
Number of stages: 3

Available compiled kernel metadata:

Triton kernel source:
def shift_scan_state_passing_kernel(
    N_ptr,
    MODE_ptr,
    EXP_ptr,
    DECAY_ptr,
    OUT_ptr,
    FINAL_ptr,
    OVERFLOW_ptr,

    NCHUNKS: tl.constexpr,
    NHEADS: tl.constexpr,
    DIM: tl.constexpr,
    NBLOCKS: tl.constexpr,

    BLOCK_SIZE: tl.constexpr
):
    pid_m = tl.program_id(0)
    pid_b = tl.program_id(1)
    pid_h = tl.program_id(2)

    offs = pid_m * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offs < DIM

    state = tl.zeros((BLOCK_SIZE,), dtype=tl.int64)
    block_overflow = tl.full((), 0, dtype=tl.int32)

    ctrl_base = (
        pid_b * NHEADS * NCHUNKS
        + pid_h * NCHUNKS
    )

    for c in range(NCHUNKS):

        mode = tl.load(MODE_ptr + ctrl_base + c)
        exp_base = ctrl_base * 3 + c * 3

        # Mode 1: three-term Dyadic2.
        if m

In [181]:

import inspect

helper = shift_scan_state_passing_kernel.__globals__[
    "_shift_scan_round_shift"
]

print("SHIFT ROUNDING HELPER")
print("=" * 70)

try:
    print(inspect.getsource(helper))
except Exception:
    print(getattr(helper, "src", "Source unavailable"))

print("\nKERNEL TYPE")
print(type(shift_scan_state_passing_kernel))

print("\nHELPER TYPE")
print(type(helper))


SHIFT ROUNDING HELPER
def _shift_scan_round_shift(x, k):
    ax = tl.abs(x)
    half = 1 << (k - 1)
    q = (ax + half) >> k
    return tl.where(x < 0, -q, q)


KERNEL TYPE
<class 'triton.runtime.jit.JITFunction'>

HELPER TYPE
<class 'triton.runtime.jit.JITFunction'>


In [182]:

import os
import glob
import numpy as np
import pandas as pd
import torch
import triton
import triton.language as tl

# ============================================================
# 1. INT32 OPTIMIZED KERNEL
# ============================================================

@triton.jit
def shiftscan_round_shift_i32(x, k):
    ax = tl.abs(x)
    half = 1 << (k - 1)
    q = (ax + half) >> k
    return tl.where(x < 0, -q, q)


@triton.jit
def shift_scan_state_passing_i32_kernel(
    N_ptr,
    MODE_ptr,
    EXP_ptr,
    DECAY_ptr,
    OUT_ptr,
    FINAL_ptr,
    OVERFLOW_ptr,

    NCHUNKS: tl.constexpr,
    NHEADS: tl.constexpr,
    DIM: tl.constexpr,
    NBLOCKS: tl.constexpr,

    BLOCK_SIZE: tl.constexpr
):
    pid_m = tl.program_id(0)
    pid_b = tl.program_id(1)
    pid_h = tl.program_id(2)

    offs = pid_m * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offs < DIM

    state = tl.zeros((BLOCK_SIZE,), dtype=tl.int32)
    block_overflow = tl.full((), 0, dtype=tl.int32)

    ctrl_base = (
        pid_b * NHEADS * NCHUNKS
        + pid_h * NCHUNKS
    )

    for c in range(NCHUNKS):

        mode = tl.load(MODE_ptr + ctrl_base + c)
        exp_base = ctrl_base * 3 + c * 3

        # Mode 1: three-term Dyadic2.
        if mode == 1:
            decay_sum = tl.zeros(
                (BLOCK_SIZE,), dtype=tl.int32
            )

            for j in tl.static_range(3):
                k = tl.load(EXP_ptr + exp_base + j)
                decay_sum += shiftscan_round_shift_i32(state, k)

            state = state - decay_sum

        # Mode 2: direct power-of-two.
        elif mode == 2:
            k = tl.load(EXP_ptr + exp_base)
            state = shiftscan_round_shift_i32(state, k)

        # Mode 3 and 4: zero decay.
        elif (mode == 3) | (mode == 4):
            state = tl.zeros(
                (BLOCK_SIZE,), dtype=tl.int32
            )

        # Mode 0: exact floating-point fallback.
        else:
            a = tl.load(
                DECAY_ptr + ctrl_base + c
            ).to(tl.float64)

            product = state.to(tl.float64) * a

            rounded = tl.floor(
                tl.abs(product) + 0.5
            ).to(tl.int64)

            rounded = tl.where(
                product < 0,
                -rounded,
                rounded
            )

            state = rounded.to(tl.int32)

        n_offset = (
            pid_b * NCHUNKS * NHEADS * DIM
            + c * NHEADS * DIM
            + pid_h * DIM
            + offs
        )

        new_state = tl.load(
            N_ptr + n_offset,
            mask=mask,
            other=0
        ).to(tl.int32)

        # Use int64 for safe addition and overflow detection.
        # The recurrent state itself remains int32.
        next64 = (
            state.to(tl.int64)
            + new_state.to(tl.int64)
        )

        outside = (
            (next64 > 2147483647)
            | (next64 < -2147483648)
        )

        chunk_overflow = tl.max(
            tl.where(mask & outside, 1, 0),
            axis=0
        )

        block_overflow = tl.maximum(
            block_overflow,
            chunk_overflow
        )

        state = next64.to(tl.int32)

        out_offset = (
            pid_b * NCHUNKS * NHEADS * DIM
            + c * NHEADS * DIM
            + pid_h * DIM
            + offs
        )

        tl.store(
            OUT_ptr + out_offset,
            state,
            mask=mask
        )

    final_offset = (
        pid_b * NHEADS * DIM
        + pid_h * DIM
        + offs
    )

    tl.store(
        FINAL_ptr + final_offset,
        state,
        mask=mask
    )

    overflow_offset = (
        (pid_b * NHEADS + pid_h) * NBLOCKS
        + pid_m
    )

    tl.store(
        OVERFLOW_ptr + overflow_offset,
        block_overflow
    )


# ============================================================
# 2. LOAD ALL 20 CAPTURES
# ============================================================

BASE_DIR = (
    "/kaggle/working/shiftscan_final/state_passing/"
    "multisequence_validation"
)

OUT_DIR = os.path.join(
    BASE_DIR, "int32_kernel_validation_v1"
)
os.makedirs(OUT_DIR, exist_ok=True)

capture_files = sorted(
    glob.glob(os.path.join(BASE_DIR, "doc_*_seq_*.pt"))
)

assert len(capture_files) == 20, (
    f"Expected 20 capture files; found {len(capture_files)}"
)

BLOCK = 128
WARPS = 4
STAGES = 3

result_rows = []

total_intermediate_mismatch = 0
total_final_mismatch = 0
total_overflow_flags = 0
total_elements = 0
global_max_integer = 0

# ============================================================
# 3. TEST ALL 480 LAYER RECORDS
# ============================================================

for file_idx, path in enumerate(capture_files, start=1):

    payload = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    records = payload["records"]

    for layer_idx, rec in enumerate(records):

        (
            N_int,
            decay,
            modes,
            exps,
            reference,
            counts,
            max_bound
        ) = build_reference_and_modes(rec)

        nchunks, nheads, dim = N_int.shape
        nblocks = triton.cdiv(dim, BLOCK)

        N_gpu = torch.as_tensor(
            N_int,
            device="cuda",
            dtype=torch.int64
        ).contiguous().unsqueeze(0)

        mode_gpu = torch.as_tensor(
            modes,
            device="cuda",
            dtype=torch.int32
        ).contiguous()

        exp_gpu = torch.as_tensor(
            exps,
            device="cuda",
            dtype=torch.int32
        ).contiguous()

        decay_gpu = torch.as_tensor(
            decay,
            device="cuda",
            dtype=torch.float64
        ).contiguous().unsqueeze(0)

        out_gpu = torch.empty(
            (1, nchunks, nheads, dim),
            device="cuda",
            dtype=torch.int32
        )

        final_gpu = torch.empty(
            (1, nheads, dim),
            device="cuda",
            dtype=torch.int32
        )

        overflow_gpu = torch.zeros(
            (nheads * nblocks,),
            device="cuda",
            dtype=torch.int32
        )

        grid = (nblocks, 1, nheads)

        shift_scan_state_passing_i32_kernel[grid](
            N_gpu,
            mode_gpu,
            exp_gpu,
            decay_gpu,
            out_gpu,
            final_gpu,
            overflow_gpu,
            nchunks,
            nheads,
            dim,
            nblocks,
            BLOCK,
            num_warps=WARPS,
            num_stages=STAGES
        )

        torch.cuda.synchronize()

        actual_intermediate = (
            out_gpu[0].cpu().numpy().astype(np.int64)
        )

        actual_final = (
            final_gpu[0].cpu().numpy().astype(np.int64)
        )

        reference = np.asarray(reference, dtype=np.int64)

        intermediate_diff = np.abs(
            actual_intermediate - reference
        )

        final_reference = reference[-1]

        final_diff = np.abs(
            actual_final - final_reference
        )

        intermediate_mismatch = int(
            np.count_nonzero(intermediate_diff)
        )

        final_mismatch = int(
            np.count_nonzero(final_diff)
        )

        overflow_count = int(
            overflow_gpu.sum().item()
        )

        max_integer = int(
            max(
                np.abs(actual_intermediate).max(),
                np.abs(actual_final).max()
            )
        )

        total_intermediate_mismatch += intermediate_mismatch
        total_final_mismatch += final_mismatch
        total_overflow_flags += overflow_count

        total_elements += actual_intermediate.size + actual_final.size
        global_max_integer = max(global_max_integer, max_integer)

        result_rows.append({
            "file": os.path.basename(path),
            "layer": layer_idx,
            "seq_len": nchunks * 256,
            "chunks": nchunks,
            "intermediate_mismatch": intermediate_mismatch,
            "final_mismatch": final_mismatch,
            "max_integer_difference": int(
                max(
                    intermediate_diff.max(),
                    final_diff.max()
                )
            ),
            "max_integer_state": max_integer,
            "overflow_flags": overflow_count,
            "fallback_updates": counts["fallback"],
            "dyadic2_updates": counts["dyadic2"],
            "pot_updates": counts["pot"],
            "exact_zero_updates": counts["exact_zero"],
            "approx_zero_updates": counts["approx_zero"],
        })

    print(
        f"{file_idx}/{len(capture_files)} completed: "
        f"{os.path.basename(path)}"
    )


# ============================================================
# 4. SAVE AND REPORT
# ============================================================

df = pd.DataFrame(result_rows)

csv_path = os.path.join(
    OUT_DIR, "int32_validation_per_layer.csv"
)

df.to_csv(csv_path, index=False)

summary = {
    "capture_files": len(capture_files),
    "layer_records": len(df),
    "total_state_elements_checked": total_elements,
    "intermediate_mismatch": total_intermediate_mismatch,
    "final_mismatch": total_final_mismatch,
    "overflow_flags": total_overflow_flags,
    "global_max_integer_state": global_max_integer,
    "max_integer_difference": int(
        df["max_integer_difference"].max()
    ),
    "all_intermediates_exact": (
        total_intermediate_mismatch == 0
    ),
    "all_finals_exact": (
        total_final_mismatch == 0
    ),
}

summary_path = os.path.join(
    OUT_DIR, "int32_validation_summary.csv"
)

pd.DataFrame([summary]).to_csv(
    summary_path,
    index=False
)

print("\n" + "=" * 75)
print("INT32 KERNEL VALIDATION")
print("=" * 75)

for key, value in summary.items():
    print(f"{key}: {value}")

print("\nSaved:", OUT_DIR)

assert total_intermediate_mismatch == 0
assert total_final_mismatch == 0
assert total_overflow_flags == 0

print("\nALL INT32 VALIDATION CHECKS PASSED.")


1/20 completed: doc_000_seq_1024.pt
2/20 completed: doc_000_seq_2048.pt
3/20 completed: doc_001_seq_1024.pt
4/20 completed: doc_001_seq_2048.pt
5/20 completed: doc_002_seq_1024.pt
6/20 completed: doc_002_seq_2048.pt
7/20 completed: doc_003_seq_1024.pt
8/20 completed: doc_003_seq_2048.pt
9/20 completed: doc_004_seq_1024.pt
10/20 completed: doc_004_seq_2048.pt
11/20 completed: doc_005_seq_1024.pt
12/20 completed: doc_005_seq_2048.pt
13/20 completed: doc_006_seq_1024.pt
14/20 completed: doc_006_seq_2048.pt
15/20 completed: doc_007_seq_1024.pt
16/20 completed: doc_007_seq_2048.pt
17/20 completed: doc_008_seq_1024.pt
18/20 completed: doc_008_seq_2048.pt
19/20 completed: doc_009_seq_1024.pt
20/20 completed: doc_009_seq_2048.pt

INT32 KERNEL VALIDATION
capture_files: 20
layer_records: 480
total_state_elements_checked: 660602880
intermediate_mismatch: 0
final_mismatch: 0
overflow_flags: 0
global_max_integer_state: 481094944
max_integer_difference: 0
all_intermediates_exact: True
all_finals_exa

In [183]:

import os
import glob
import numpy as np
import pandas as pd
import torch
import triton

BASE_DIR = (
    "/kaggle/working/shiftscan_final/state_passing/"
    "multisequence_validation"
)

OUT_DIR = os.path.join(
    BASE_DIR, "int32_vs_int64_benchmark_v1"
)
os.makedirs(OUT_DIR, exist_ok=True)

DOC_INDICES = [0, 4, 8]
SEQ_LENGTHS = [1024, 2048]
LAYERS = [0, 8, 16, 23]

BLOCK = 128
WARPS = 4
STAGES = 3

WARMUP_MS = 50
REP_MS = 200

files = sorted(glob.glob(
    os.path.join(BASE_DIR, "doc_*_seq_*.pt")
))

rows = []

for doc_idx in DOC_INDICES:
    for seq_len in SEQ_LENGTHS:

        matches = [
            p for p in files
            if f"doc_{doc_idx:03d}_seq_{seq_len}.pt"
            in os.path.basename(p)
        ]

        assert len(matches) == 1
        path = matches[0]

        payload = torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )

        for layer_idx in LAYERS:

            rec = payload["records"][layer_idx]

            (
                N_int,
                decay,
                modes,
                exps,
                reference,
                counts,
                max_bound
            ) = build_reference_and_modes(rec)

            nchunks, nheads, dim = N_int.shape
            nblocks = triton.cdiv(dim, BLOCK)

            N_gpu = torch.as_tensor(
                N_int,
                device="cuda",
                dtype=torch.int64
            ).contiguous().unsqueeze(0)

            modes_gpu = torch.as_tensor(
                modes,
                device="cuda",
                dtype=torch.int32
            ).contiguous()

            exps_gpu = torch.as_tensor(
                exps,
                device="cuda",
                dtype=torch.int32
            ).contiguous()

            decay_gpu = torch.as_tensor(
                decay,
                device="cuda",
                dtype=torch.float64
            ).contiguous().unsqueeze(0)

            out64 = torch.empty(
                (1, nchunks, nheads, dim),
                device="cuda",
                dtype=torch.int32
            )

            final64 = torch.empty(
                (1, nheads, dim),
                device="cuda",
                dtype=torch.int32
            )

            overflow64 = torch.zeros(
                (nheads * nblocks,),
                device="cuda",
                dtype=torch.int32
            )

            out32 = torch.empty_like(out64)
            final32 = torch.empty_like(final64)

            overflow32 = torch.zeros_like(overflow64)

            grid = (nblocks, 1, nheads)

            def run_int64():
                shift_scan_state_passing_kernel[grid](
                    N_gpu,
                    modes_gpu,
                    exps_gpu,
                    decay_gpu,
                    out64,
                    final64,
                    overflow64,
                    nchunks,
                    nheads,
                    dim,
                    nblocks,
                    BLOCK,
                    num_warps=WARPS,
                    num_stages=STAGES
                )

            def run_int32():
                shift_scan_state_passing_i32_kernel[grid](
                    N_gpu,
                    modes_gpu,
                    exps_gpu,
                    decay_gpu,
                    out32,
                    final32,
                    overflow32,
                    nchunks,
                    nheads,
                    dim,
                    nblocks,
                    BLOCK,
                    num_warps=WARPS,
                    num_stages=STAGES
                )

            # Compile and warm up both versions.
            for _ in range(8):
                run_int64()
                run_int32()

            torch.cuda.synchronize()

            # Alternate measurement order to reduce systematic bias.
            timings64 = []
            timings32 = []

            for rep in range(5):
                if rep % 2 == 0:
                    t64 = triton.testing.do_bench(
                        run_int64,
                        warmup=WARMUP_MS,
                        rep=REP_MS
                    )
                    t32 = triton.testing.do_bench(
                        run_int32,
                        warmup=WARMUP_MS,
                        rep=REP_MS
                    )
                else:
                    t32 = triton.testing.do_bench(
                        run_int32,
                        warmup=WARMUP_MS,
                        rep=REP_MS
                    )
                    t64 = triton.testing.do_bench(
                        run_int64,
                        warmup=WARMUP_MS,
                        rep=REP_MS
                    )

                timings64.append(float(t64))
                timings32.append(float(t32))

            torch.cuda.synchronize()

            # Correctness check after timing.
            mismatch = int(
                torch.count_nonzero(out64 != out32).item()
                + torch.count_nonzero(final64 != final32).item()
            )

            overflow_count = int(
                overflow32.sum().item()
            )

            row = {
                "file": os.path.basename(path),
                "seq_len": seq_len,
                "layer": layer_idx,
                "int64_median_ms": float(np.median(timings64)),
                "int32_median_ms": float(np.median(timings32)),
                "int32_over_int64": float(
                    np.median(timings32)
                    / np.median(timings64)
                ),
                "int32_speedup_pct": float(
                    100 * (
                        1 - np.median(timings32)
                        / np.median(timings64)
                    )
                ),
                "output_mismatch": mismatch,
                "overflow_flags": overflow_count,
            }

            rows.append(row)

            print(
                f"{os.path.basename(path)} "
                f"L{layer_idx:02d} | "
                f"int64={row['int64_median_ms']:.5f} ms | "
                f"int32={row['int32_median_ms']:.5f} ms | "
                f"ratio={row['int32_over_int64']:.3f}x | "
                f"mismatch={mismatch}"
            )

df = pd.DataFrame(rows)

df.to_csv(
    os.path.join(OUT_DIR, "per_layer_benchmark.csv"),
    index=False
)

summary = (
    df.groupby("seq_len")
    .agg(
        records=("layer", "count"),
        int64_median_ms=("int64_median_ms", "median"),
        int32_median_ms=("int32_median_ms", "median"),
        median_ratio=("int32_over_int64", "median"),
        median_speedup_pct=("int32_speedup_pct", "median"),
        mismatches=("output_mismatch", "sum"),
        overflow_flags=("overflow_flags", "sum"),
    )
    .reset_index()
)

summary.to_csv(
    os.path.join(OUT_DIR, "summary.csv"),
    index=False
)

print("\nSUMMARY")
print(summary.to_string(index=False))

print("\nOverall median latency:")
print("int64:", df["int64_median_ms"].median(), "ms")
print("int32:", df["int32_median_ms"].median(), "ms")

print("\nOverall median int32/int64 ratio:",
      df["int32_over_int64"].median())

print("\nTotal output mismatches:",
      int(df["output_mismatch"].sum()))

print("Total overflow flags:",
      int(df["overflow_flags"].sum()))

print("\nSaved:", OUT_DIR)

assert len(df) == 24
assert int(df["output_mismatch"].sum()) == 0
assert int(df["overflow_flags"].sum()) == 0

print("\nINT32 VS INT64 BENCHMARK COMPLETED.")


doc_000_seq_1024.pt L00 | int64=0.04734 ms | int32=0.04743 ms | ratio=1.002x | mismatch=0
doc_000_seq_1024.pt L08 | int64=0.04776 ms | int32=0.04791 ms | ratio=1.003x | mismatch=0
doc_000_seq_1024.pt L16 | int64=0.04694 ms | int32=0.04683 ms | ratio=0.998x | mismatch=0
doc_000_seq_1024.pt L23 | int64=0.04715 ms | int32=0.04791 ms | ratio=1.016x | mismatch=0
doc_000_seq_2048.pt L00 | int64=0.08660 ms | int32=0.08655 ms | ratio=0.999x | mismatch=0
doc_000_seq_2048.pt L08 | int64=0.08794 ms | int32=0.08753 ms | ratio=0.995x | mismatch=0
doc_000_seq_2048.pt L16 | int64=0.08527 ms | int32=0.08590 ms | ratio=1.007x | mismatch=0
doc_000_seq_2048.pt L23 | int64=0.08591 ms | int32=0.08576 ms | ratio=0.998x | mismatch=0
doc_004_seq_1024.pt L00 | int64=0.04722 ms | int32=0.04756 ms | ratio=1.007x | mismatch=0
doc_004_seq_1024.pt L08 | int64=0.04819 ms | int32=0.04806 ms | ratio=0.997x | mismatch=0
doc_004_seq_1024.pt L16 | int64=0.04668 ms | int32=0.04694 ms | ratio=1.005x | mismatch=0
doc_004_se

In [184]:

import os, glob
import numpy as np
import pandas as pd
import torch
import triton
import triton.language as tl

BASE = "/kaggle/working/shiftscan_final/state_passing/multisequence_validation"
OUT = os.path.join(BASE, "specialized_path_benchmark_v1")
os.makedirs(OUT, exist_ok=True)

BLOCK = 128
WARPS = 4
STAGES = 3

@triton.jit
def specialized_recurrence_kernel(
    N_PTR, DECAY_PTR, EXP_PTR, OUT_PTR, FINAL_PTR,
    NCHUNKS: tl.constexpr,
    NHEADS: tl.constexpr,
    DIM: tl.constexpr,
    BLOCK_SIZE: tl.constexpr,
    MODE_KIND: tl.constexpr
):
    pid_m = tl.program_id(0)
    pid_h = tl.program_id(1)

    offs = pid_m * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offs < DIM

    state = tl.zeros((BLOCK_SIZE,), dtype=tl.int32)

    for c in range(NCHUNKS):
        ctrl = pid_h * NCHUNKS + c

        if MODE_KIND == 1:
            exp_base = ctrl * 3
            decay_sum = tl.zeros((BLOCK_SIZE,), dtype=tl.int32)

            for j in tl.static_range(3):
                k = tl.load(EXP_PTR + exp_base + j)
                ax = tl.abs(state)
                half = 1 << (k - 1)
                q = (ax + half) >> k
                shifted = tl.where(state < 0, -q, q)
                decay_sum += shifted

            state = state - decay_sum

        elif MODE_KIND == 2:
            k = tl.load(EXP_PTR + ctrl * 3)
            ax = tl.abs(state)
            half = 1 << (k - 1)
            q = (ax + half) >> k
            state = tl.where(state < 0, -q, q)

        elif MODE_KIND == 3:
            state = tl.zeros((BLOCK_SIZE,), dtype=tl.int32)

        elif MODE_KIND == 0:
            a = tl.load(DECAY_PTR + ctrl).to(tl.float64)
            product = state.to(tl.float64) * a
            rounded = tl.floor(tl.abs(product) + 0.5).to(tl.int64)
            rounded = tl.where(product < 0, -rounded, rounded)
            state = rounded.to(tl.int32)

        n_offset = (
            c * NHEADS * DIM
            + pid_h * DIM
            + offs
        )

        n = tl.load(
            N_PTR + n_offset,
            mask=mask,
            other=0
        ).to(tl.int32)

        state = state + n

        tl.store(
            OUT_PTR + n_offset,
            state,
            mask=mask
        )

    final_offset = pid_h * DIM + offs
    tl.store(FINAL_PTR + final_offset, state, mask=mask)


# Select the same six captures and four layers as before.
files = sorted(glob.glob(os.path.join(BASE, "doc_*_seq_*.pt")))
selected = []

for doc in [0, 4, 8]:
    for length in [1024, 2048]:
        matches = [
            p for p in files
            if f"doc_{doc:03d}_seq_{length}.pt" in os.path.basename(p)
        ]
        assert len(matches) == 1
        selected.append(matches[0])

mode_names = {
    0: "FP64_fallback_only",
    1: "Dyadic2_only",
    2: "PoT_only",
    3: "Zero_decay_only",
}

rows = []

for path in selected:
    payload = torch.load(path, map_location="cpu", weights_only=False)

    for layer in [0, 8, 16, 23]:
        rec = payload["records"][layer]

        N_int, decay, modes, exps, reference, counts, bound = (
            build_reference_and_modes(rec)
        )

        nchunks, nheads, dim = N_int.shape
        nblocks = triton.cdiv(dim, BLOCK)
        grid = (nblocks, nheads)

        N_gpu = torch.as_tensor(
            N_int, device="cuda", dtype=torch.int32
        ).contiguous()

        decay_gpu = torch.as_tensor(
            decay, device="cuda", dtype=torch.float64
        ).contiguous()

        # Valid exponents for every control entry.
        # These are fixed timing inputs, not the learned controller.
        exp_timing = np.zeros((1, nheads, nchunks, 3), dtype=np.int32)
        exp_timing[..., 0] = 3
        exp_timing[..., 1] = 4
        exp_timing[..., 2] = 5

        exp_gpu = torch.as_tensor(
            exp_timing, device="cuda", dtype=torch.int32
        ).contiguous()

        timings = {}

        for mode, name in mode_names.items():
            out_gpu = torch.empty(
                (nchunks, nheads, dim),
                device="cuda",
                dtype=torch.int32
            )
            final_gpu = torch.empty(
                (nheads, dim),
                device="cuda",
                dtype=torch.int32
            )

            def run():
                specialized_recurrence_kernel[grid](
                    N_gpu,
                    decay_gpu,
                    exp_gpu,
                    out_gpu,
                    final_gpu,
                    nchunks,
                    nheads,
                    dim,
                    BLOCK,
                    mode,
                    num_warps=WARPS,
                    num_stages=STAGES
                )

            for _ in range(5):
                run()

            torch.cuda.synchronize()

            samples = []
            for _ in range(3):
                samples.append(
                    triton.testing.do_bench(
                        run, warmup=50, rep=150
                    )
                )

            timings[name] = float(np.median(samples))

        row = {
            "file": os.path.basename(path),
            "seq_len": nchunks * 256,
            "layer": layer,
            **timings
        }
        rows.append(row)

        print(
            os.path.basename(path),
            f"L{layer:02d}",
            " | ".join(
                f"{name}={value:.5f} ms"
                for name, value in timings.items()
            )
        )

df = pd.DataFrame(rows)
df.to_csv(
    os.path.join(OUT, "per_layer_specialized_paths.csv"),
    index=False
)

summary = (
    df.groupby("seq_len")[list(mode_names.values())]
    .median()
    .reset_index()
)

summary.to_csv(
    os.path.join(OUT, "specialized_path_summary.csv"),
    index=False
)

print("\nSPECIALIZED PATH MEDIANS")
print(summary.to_string(index=False))

print("\nOverall medians:")
for name in mode_names.values():
    print(name, round(float(df[name].median()), 6), "ms")

print("\nSaved:", OUT)


doc_000_seq_1024.pt L00 FP64_fallback_only=0.07175 ms | Dyadic2_only=0.03454 ms | PoT_only=0.03452 ms | Zero_decay_only=0.03459 ms
doc_000_seq_1024.pt L08 FP64_fallback_only=0.03926 ms | Dyadic2_only=0.03503 ms | PoT_only=0.03422 ms | Zero_decay_only=0.03490 ms
doc_000_seq_1024.pt L16 FP64_fallback_only=0.03936 ms | Dyadic2_only=0.03470 ms | PoT_only=0.03416 ms | Zero_decay_only=0.03463 ms
doc_000_seq_1024.pt L23 FP64_fallback_only=0.03956 ms | Dyadic2_only=0.03509 ms | PoT_only=0.03424 ms | Zero_decay_only=0.03492 ms
doc_000_seq_2048.pt L00 FP64_fallback_only=0.07800 ms | Dyadic2_only=0.06169 ms | PoT_only=0.06170 ms | Zero_decay_only=0.06169 ms
doc_000_seq_2048.pt L08 FP64_fallback_only=0.07819 ms | Dyadic2_only=0.06177 ms | PoT_only=0.06164 ms | Zero_decay_only=0.06169 ms
doc_000_seq_2048.pt L16 FP64_fallback_only=0.07814 ms | Dyadic2_only=0.06172 ms | PoT_only=0.06183 ms | Zero_decay_only=0.06160 ms
doc_000_seq_2048.pt L23 FP64_fallback_only=0.07879 ms | Dyadic2_only=0.06176 ms | P

In [185]:

import os, glob
import numpy as np
import pandas as pd
import torch
import triton
import triton.language as tl

BASE = "/kaggle/working/shiftscan_final/state_passing/multisequence_validation"

OUT = os.path.join(BASE, "fp32_fallback_audit_v1")
os.makedirs(OUT, exist_ok=True)

@triton.jit
def shiftscan_fp32_fallback_kernel(
    N_ptr, MODE_ptr, EXP_ptr, DECAY_ptr,
    OUT_ptr, FINAL_ptr,
    NCHUNKS: tl.constexpr,
    NHEADS: tl.constexpr,
    DIM: tl.constexpr,
    NBLOCKS: tl.constexpr,
    BLOCK_SIZE: tl.constexpr
):
    pid_m = tl.program_id(0)
    pid_h = tl.program_id(2)

    offs = pid_m * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offs < DIM

    state = tl.zeros((BLOCK_SIZE,), dtype=tl.int32)

    ctrl_base = pid_h * NCHUNKS

    for c in range(NCHUNKS):
        mode = tl.load(MODE_ptr + ctrl_base + c)
        exp_base = ctrl_base * 3 + c * 3

        if mode == 1:
            decay_sum = tl.zeros((BLOCK_SIZE,), dtype=tl.int32)

            for j in tl.static_range(3):
                k = tl.load(EXP_ptr + exp_base + j)
                ax = tl.abs(state)
                half = 1 << (k - 1)
                q = (ax + half) >> k
                shifted = tl.where(state < 0, -q, q)
                decay_sum += shifted

            state = state - decay_sum

        elif mode == 2:
            k = tl.load(EXP_PTR + exp_base)
            ax = tl.abs(state)
            half = 1 << (k - 1)
            q = (ax + half) >> k
            state = tl.where(state < 0, -q, q)

        elif (mode == 3) | (mode == 4):
            state = tl.zeros((BLOCK_SIZE,), dtype=tl.int32)

        else:
            # Diagnostic change: FP32 instead of FP64.
            a = tl.load(
                DECAY_ptr + ctrl_base + c
            ).to(tl.float32)

            product = state.to(tl.float32) * a

            rounded = tl.floor(
                tl.abs(product) + 0.5
            ).to(tl.int32)

            state = tl.where(
                product < 0,
                -rounded,
                rounded
            )

        n_offset = (
            c * NHEADS * DIM
            + pid_h * DIM
            + offs
        )

        n = tl.load(
            N_ptr + n_offset,
            mask=mask,
            other=0
        ).to(tl.int32)

        state = state + n

        tl.store(
            OUT_ptr + n_offset,
            state,
            mask=mask
        )

    tl.store(
        FINAL_ptr + pid_h * DIM + offs,
        state,
        mask=mask
    )


# ============================================================
# Correct the helper's EXP_PTR typo by defining a clean kernel
# ============================================================

# Recompile a corrected copy using the same implementation.
# This version has EXP_ptr consistently named.

@triton.jit
def shiftscan_fp32_fallback_kernel(
    N_ptr, MODE_ptr, EXP_ptr, DECAY_ptr,
    OUT_ptr, FINAL_ptr,
    NCHUNKS: tl.constexpr,
    NHEADS: tl.constexpr,
    DIM: tl.constexpr,
    NBLOCKS: tl.constexpr,
    BLOCK_SIZE: tl.constexpr
):
    pid_m = tl.program_id(0)
    pid_h = tl.program_id(2)

    offs = pid_m * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offs < DIM

    state = tl.zeros((BLOCK_SIZE,), dtype=tl.int32)
    ctrl_base = pid_h * NCHUNKS

    for c in range(NCHUNKS):
        mode = tl.load(MODE_ptr + ctrl_base + c)
        exp_base = ctrl_base * 3 + c * 3

        if mode == 1:
            decay_sum = tl.zeros((BLOCK_SIZE,), dtype=tl.int32)
            for j in tl.static_range(3):
                k = tl.load(EXP_ptr + exp_base + j)
                ax = tl.abs(state)
                q = (ax + (1 << (k - 1))) >> k
                shifted = tl.where(state < 0, -q, q)
                decay_sum += shifted
            state = state - decay_sum

        elif mode == 2:
            k = tl.load(EXP_ptr + exp_base)
            ax = tl.abs(state)
            q = (ax + (1 << (k - 1))) >> k
            state = tl.where(state < 0, -q, q)

        elif (mode == 3) | (mode == 4):
            state = tl.zeros((BLOCK_SIZE,), dtype=tl.int32)

        else:
            a = tl.load(DECAY_ptr + ctrl_base + c).to(tl.float32)
            product = state.to(tl.float32) * a
            rounded = tl.floor(tl.abs(product) + 0.5).to(tl.int32)
            state = tl.where(product < 0, -rounded, rounded)

        n_offset = c * NHEADS * DIM + pid_h * DIM + offs

        n = tl.load(N_ptr + n_offset, mask=mask, other=0).to(tl.int32)
        state = state + n

        tl.store(OUT_ptr + n_offset, state, mask=mask)

    tl.store(FINAL_ptr + pid_h * DIM + offs, state, mask=mask)


# ============================================================
# Evaluate all 480 records
# ============================================================

files = sorted(glob.glob(os.path.join(BASE, "doc_*_seq_*.pt")))

rows = []
total_mismatch = 0
total_elements = 0
global_max_diff = 0

for file_idx, path in enumerate(files, 1):
    payload = torch.load(path, map_location="cpu", weights_only=False)

    for layer, rec in enumerate(payload["records"]):
        N_int, decay, modes, exps, reference, counts, bound = (
            build_reference_and_modes(rec)
        )

        nchunks, nheads, dim = N_int.shape
        block = 128
        nblocks = triton.cdiv(dim, block)
        grid = (nblocks, 1, nheads)

        N_gpu = torch.as_tensor(
            N_int, device="cuda", dtype=torch.int64
        ).contiguous().unsqueeze(0)

        mode_gpu = torch.as_tensor(
            modes, device="cuda", dtype=torch.int32
        ).contiguous()

        exp_gpu = torch.as_tensor(
            exps, device="cuda", dtype=torch.int32
        ).contiguous()

        decay_gpu = torch.as_tensor(
            decay, device="cuda", dtype=torch.float64
        ).contiguous().unsqueeze(0)

        out = torch.empty(
            (1, nchunks, nheads, dim),
            device="cuda", dtype=torch.int32
        )
        final = torch.empty(
            (1, nheads, dim),
            device="cuda", dtype=torch.int32
        )

        shiftscan_fp32_fallback_kernel[grid](
            N_gpu, mode_gpu, exp_gpu, decay_gpu,
            out, final,
            nchunks, nheads, dim, nblocks, block,
            num_warps=4, num_stages=3
        )

        torch.cuda.synchronize()

        actual = out[0].cpu().numpy().astype(np.int64)
        actual_final = final[0].cpu().numpy().astype(np.int64)

        ref = np.asarray(reference, dtype=np.int64)

        diff = np.abs(actual - ref)
        final_diff = np.abs(actual_final - ref[-1])

        mismatches = int(np.count_nonzero(diff))
        final_mismatches = int(np.count_nonzero(final_diff))
        max_diff = int(max(diff.max(), final_diff.max()))

        total_mismatch += mismatches + final_mismatches
        total_elements += diff.size + final_diff.size
        global_max_diff = max(global_max_diff, max_diff)

        rows.append({
            "file": os.path.basename(path),
            "layer": layer,
            "seq_len": nchunks * 256,
            "fallback_updates": counts["fallback"],
            "intermediate_mismatches": mismatches,
            "final_mismatches": final_mismatches,
            "max_integer_difference": max_diff,
        })

    print(f"{file_idx}/{len(files)} completed")

df = pd.DataFrame(rows)
df.to_csv(
    os.path.join(OUT, "fp32_fallback_per_layer.csv"),
    index=False
)

summary = {
    "records": len(df),
    "total_checked_elements": total_elements,
    "total_mismatches": total_mismatch,
    "global_max_integer_difference": global_max_diff,
    "records_with_mismatch": int(
        (
            (df["intermediate_mismatches"] > 0)
            | (df["final_mismatches"] > 0)
        ).sum()
    ),
}

pd.DataFrame([summary]).to_csv(
    os.path.join(OUT, "summary.csv"),
    index=False
)

print("\nFP32 FALLBACK ACCURACY AUDIT")
for k, v in summary.items():
    print(f"{k}: {v}")

print("\nSaved:", OUT)


1/20 completed
2/20 completed
3/20 completed
4/20 completed
5/20 completed
6/20 completed
7/20 completed
8/20 completed
9/20 completed
10/20 completed
11/20 completed
12/20 completed
13/20 completed
14/20 completed
15/20 completed
16/20 completed
17/20 completed
18/20 completed
19/20 completed
20/20 completed

FP32 FALLBACK ACCURACY AUDIT
records: 480
total_checked_elements: 660602880
total_mismatches: 2275224
global_max_integer_difference: 13
records_with_mismatch: 426

Saved: /kaggle/working/shiftscan_final/state_passing/multisequence_validation/fp32_fallback_audit_v1


In [186]:

import os, glob
import numpy as np
import pandas as pd
import torch
import triton
import triton.language as tl

BASE = "/kaggle/working/shiftscan_final/state_passing/multisequence_validation"
OUT = os.path.join(BASE, "integer_fallback_audit_v1")
os.makedirs(OUT, exist_ok=True)

BLOCK = 128

@triton.jit
def shiftscan_integer_fallback_kernel(
    N_PTR, MODE_PTR, EXP_PTR, DECAY_Q_PTR,
    OUT_PTR, FINAL_PTR,
    NCHUNKS: tl.constexpr,
    NHEADS: tl.constexpr,
    DIM: tl.constexpr,
    BLOCK_SIZE: tl.constexpr,
    Q_BITS: tl.constexpr
):
    pid_m = tl.program_id(0)
    pid_h = tl.program_id(1)

    offs = pid_m * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offs < DIM

    state = tl.zeros((BLOCK_SIZE,), dtype=tl.int32)
    ctrl_base = pid_h * NCHUNKS

    for c in range(NCHUNKS):
        ctrl = ctrl_base + c
        mode = tl.load(MODE_PTR + ctrl)
        exp_base = ctrl * 3

        if mode == 1:
            decay_sum = tl.zeros((BLOCK_SIZE,), dtype=tl.int32)

            for j in tl.static_range(3):
                k = tl.load(EXP_PTR + exp_base + j)
                ax = tl.abs(state)
                q = (ax + (1 << (k - 1))) >> k
                shifted = tl.where(state < 0, -q, q)
                decay_sum += shifted

            state = state - decay_sum

        elif mode == 2:
            k = tl.load(EXP_PTR + exp_base)
            ax = tl.abs(state)
            q = (ax + (1 << (k - 1))) >> k
            state = tl.where(state < 0, -q, q)

        elif (mode == 3) | (mode == 4):
            state = tl.zeros((BLOCK_SIZE,), dtype=tl.int32)

        else:
            # Integer fixed-point fallback:
            # round(state * decay_q / 2^Q_BITS)
            a_q = tl.load(DECAY_Q_PTR + ctrl).to(tl.int64)

            product = state.to(tl.int64) * a_q
            abs_product = tl.abs(product)

            rounded = (
                abs_product + (1 << (Q_BITS - 1))
            ) >> Q_BITS

            rounded = tl.where(
                product < 0,
                -rounded,
                rounded
            )

            state = rounded.to(tl.int32)

        n_offset = (
            c * NHEADS * DIM
            + pid_h * DIM
            + offs
        )

        n = tl.load(
            N_PTR + n_offset,
            mask=mask,
            other=0
        ).to(tl.int32)

        state = state + n

        # Store as int64 for safe comparison with the CPU reference.
        tl.store(
            OUT_PTR + n_offset,
            state.to(tl.int64),
            mask=mask
        )

    tl.store(
        FINAL_PTR + pid_h * DIM + offs,
        state.to(tl.int64),
        mask=mask
    )


# ============================================================
# Run Q30, Q32 and Q33 over all 480 records
# ============================================================

Q_VALUES = [30, 32, 33]
files = sorted(glob.glob(os.path.join(BASE, "doc_*_seq_*.pt")))

all_rows = []

for Q_BITS in Q_VALUES:
    print(f"\n========== Q{Q_BITS} ==========")

    q_scale = 1 << Q_BITS
    q_rows = []

    for file_idx, path in enumerate(files, 1):
        payload = torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )

        for layer, rec in enumerate(payload["records"]):

            N_int, decay, modes, exps, reference, counts, bound = (
                build_reference_and_modes(rec)
            )

            nchunks, nheads, dim = N_int.shape
            nblocks = triton.cdiv(dim, BLOCK)

            # Quantize the FP32 decay coefficient into Q format.
            # decay originates from torch.exp(log_decay.float()).
            decay_q = np.floor(
                np.asarray(decay, dtype=np.float64) * q_scale + 0.5
            ).astype(np.int64)

            N_gpu = torch.as_tensor(
                N_int,
                device="cuda",
                dtype=torch.int32
            ).contiguous().unsqueeze(0)

            mode_gpu = torch.as_tensor(
                modes,
                device="cuda",
                dtype=torch.int32
            ).contiguous()

            exp_gpu = torch.as_tensor(
                exps,
                device="cuda",
                dtype=torch.int32
            ).contiguous()

            decay_q_gpu = torch.as_tensor(
                decay_q,
                device="cuda",
                dtype=torch.int64
            ).contiguous()

            out_gpu = torch.empty(
                (1, nchunks, nheads, dim),
                device="cuda",
                dtype=torch.int64
            )

            final_gpu = torch.empty(
                (1, nheads, dim),
                device="cuda",
                dtype=torch.int64
            )

            grid = (nblocks, nheads)

            shiftscan_integer_fallback_kernel[grid](
                N_gpu,
                mode_gpu,
                exp_gpu,
                decay_q_gpu,
                out_gpu,
                final_gpu,
                nchunks,
                nheads,
                dim,
                BLOCK,
                Q_BITS,
                num_warps=4,
                num_stages=3
            )

            torch.cuda.synchronize()

            actual = out_gpu[0].cpu().numpy()
            actual_final = final_gpu[0].cpu().numpy()

            reference = np.asarray(reference, dtype=np.int64)

            diff = np.abs(actual - reference)
            final_diff = np.abs(actual_final - reference[-1])

            intermediate_mismatches = int(np.count_nonzero(diff))
            final_mismatches = int(np.count_nonzero(final_diff))

            max_diff = int(max(
                diff.max(),
                final_diff.max()
            ))

            max_state = int(max(
                np.abs(actual).max(),
                np.abs(actual_final).max()
            ))

            int32_range_violation = bool(
                max_state > np.iinfo(np.int32).max
            )

            row = {
                "q_bits": Q_BITS,
                "file": os.path.basename(path),
                "layer": layer,
                "seq_len": nchunks * 256,
                "fallback_updates": counts["fallback"],
                "intermediate_mismatches": intermediate_mismatches,
                "final_mismatches": final_mismatches,
                "max_integer_difference": max_diff,
                "max_abs_integer_state": max_state,
                "int32_range_violation": int32_range_violation
            }

            q_rows.append(row)
            all_rows.append(row)

        print(f"Q{Q_BITS}: {file_idx}/{len(files)} files completed")

    q_df = pd.DataFrame(q_rows)

    q_df.to_csv(
        os.path.join(OUT, f"q{Q_BITS}_per_layer.csv"),
        index=False
    )

    summary = {
        "q_bits": Q_BITS,
        "records": len(q_df),
        "total_intermediate_mismatches": int(
            q_df["intermediate_mismatches"].sum()
        ),
        "total_final_mismatches": int(
            q_df["final_mismatches"].sum()
        ),
        "records_with_mismatch": int(
            (
                (q_df["intermediate_mismatches"] > 0)
                | (q_df["final_mismatches"] > 0)
            ).sum()
        ),
        "global_max_integer_difference": int(
            q_df["max_integer_difference"].max()
        ),
        "max_abs_integer_state": int(
            q_df["max_abs_integer_state"].max()
        ),
        "int32_range_violations": int(
            q_df["int32_range_violation"].sum()
        )
    }

    print("\n", summary)


# ============================================================
# Combined comparison
# ============================================================

all_df = pd.DataFrame(all_rows)

all_df.to_csv(
    os.path.join(OUT, "all_q_formats_per_layer.csv"),
    index=False
)

summary_rows = []

for q in Q_VALUES:
    sub = all_df[all_df["q_bits"] == q]

    summary_rows.append({
        "q_bits": q,
        "records": len(sub),
        "intermediate_mismatches": int(
            sub["intermediate_mismatches"].sum()
        ),
        "final_mismatches": int(
            sub["final_mismatches"].sum()
        ),
        "records_with_mismatch": int(
            (
                (sub["intermediate_mismatches"] > 0)
                | (sub["final_mismatches"] > 0)
            ).sum()
        ),
        "max_integer_difference": int(
            sub["max_integer_difference"].max()
        ),
        "max_abs_integer_state": int(
            sub["max_abs_integer_state"].max()
        ),
        "int32_range_violations": int(
            sub["int32_range_violation"].sum()
        )
    })

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(
    os.path.join(OUT, "integer_fallback_summary.csv"),
    index=False
)

print("\n========== FINAL Q-FORMAT COMPARISON ==========")
display(summary_df)

print("\nSaved:", OUT)



========== Q30 ==========
Q30: 1/20 files completed
Q30: 2/20 files completed
Q30: 3/20 files completed
Q30: 4/20 files completed
Q30: 5/20 files completed
Q30: 6/20 files completed
Q30: 7/20 files completed
Q30: 8/20 files completed
Q30: 9/20 files completed
Q30: 10/20 files completed
Q30: 11/20 files completed
Q30: 12/20 files completed
Q30: 13/20 files completed
Q30: 14/20 files completed
Q30: 15/20 files completed
Q30: 16/20 files completed
Q30: 17/20 files completed
Q30: 18/20 files completed
Q30: 19/20 files completed
Q30: 20/20 files completed

 {'q_bits': 30, 'records': 480, 'total_intermediate_mismatches': 1244, 'total_final_mismatches': 246, 'records_with_mismatch': 171, 'global_max_integer_difference': 1, 'max_abs_integer_state': 481094944, 'int32_range_violations': 0}

========== Q32 ==========
Q32: 1/20 files completed
Q32: 2/20 files completed
Q32: 3/20 files completed
Q32: 4/20 files completed
Q32: 5/20 files completed
Q32: 6/20 files completed
Q32: 7/20 files completed

,q_bits,records,intermediate_mismatches,final_mismatches,records_with_mismatch,max_integer_difference,max_abs_integer_state,int32_range_violations
0,30,480,1244,246,171,1,481094944,0
1,32,480,221,40,75,1,481094944,0
2,33,480,80,18,37,1,481094944,0



Saved: /kaggle/working/shiftscan_final/state_passing/multisequence_validation/integer_fallback_audit_v1


In [187]:

import os, glob, time
import numpy as np
import pandas as pd
import torch
import triton
import triton.language as tl

BASE = "/kaggle/working/shiftscan_final/state_passing/multisequence_validation"
OUT_DIR = os.path.join(BASE, "integer_fallback_benchmark_v1")
os.makedirs(OUT_DIR, exist_ok=True)

BLOCK = 128
Q_VALUES = [30, 32, 33]

# Same representative subset as the earlier path benchmark.
SELECT_DOCS = [0, 4, 8]
SELECT_LENGTHS = [1024, 2048]
SELECT_LAYERS = [0, 8, 16, 23]


@triton.jit
def shiftscan_fallback_benchmark_kernel(
    N_PTR, MODE_PTR, EXP_PTR, DECAY_PTR, DECAY_Q_PTR,
    OUT_PTR, FINAL_PTR,
    NCHUNKS: tl.constexpr,
    NHEADS: tl.constexpr,
    DIM: tl.constexpr,
    BLOCK_SIZE: tl.constexpr,
    FALLBACK_KIND: tl.constexpr,
    Q_BITS: tl.constexpr
):
    pid_m = tl.program_id(0)
    pid_h = tl.program_id(1)

    offs = pid_m * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offs < DIM

    state = tl.zeros((BLOCK_SIZE,), dtype=tl.int32)
    ctrl_base = pid_h * NCHUNKS

    for c in range(NCHUNKS):
        ctrl = ctrl_base + c
        mode = tl.load(MODE_PTR + ctrl)
        exp_base = ctrl * 3

        if mode == 1:
            decay_sum = tl.zeros((BLOCK_SIZE,), dtype=tl.int32)

            for j in tl.static_range(3):
                k = tl.load(EXP_PTR + exp_base + j)
                ax = tl.abs(state)
                q = (ax + (1 << (k - 1))) >> k
                shifted = tl.where(state < 0, -q, q)
                decay_sum += shifted

            state = state - decay_sum

        elif mode == 2:
            k = tl.load(EXP_PTR + exp_base)
            ax = tl.abs(state)
            q = (ax + (1 << (k - 1))) >> k
            state = tl.where(state < 0, -q, q)

        elif (mode == 3) | (mode == 4):
            state = tl.zeros((BLOCK_SIZE,), dtype=tl.int32)

        else:
            if FALLBACK_KIND == 0:
                # Exact FP64 fallback.
                a = tl.load(DECAY_PTR + ctrl).to(tl.float64)
                product = state.to(tl.float64) * a
                rounded = tl.floor(
                    tl.abs(product) + 0.5
                ).to(tl.int64)
                rounded = tl.where(product < 0, -rounded, rounded)
                state = rounded.to(tl.int32)

            else:
                # Q-format integer multiplication and rounded shift.
                a_q = tl.load(DECAY_Q_PTR + ctrl).to(tl.int64)
                product = state.to(tl.int64) * a_q

                rounded = (
                    tl.abs(product) + (1 << (Q_BITS - 1))
                ) >> Q_BITS

                rounded = tl.where(
                    product < 0,
                    -rounded,
                    rounded
                )

                state = rounded.to(tl.int32)

        n_offset = c * NHEADS * DIM + pid_h * DIM + offs
        n = tl.load(N_PTR + n_offset, mask=mask, other=0).to(tl.int32)

        state = state + n

        tl.store(
            OUT_PTR + n_offset,
            state.to(tl.int64),
            mask=mask
        )

    tl.store(
        FINAL_PTR + pid_h * DIM + offs,
        state.to(tl.int64),
        mask=mask
    )


# ============================================================
# Load and select records
# ============================================================

files = sorted(glob.glob(os.path.join(BASE, "doc_*_seq_*.pt")))

selected = []

for doc_idx in SELECT_DOCS:
    for seq_len in SELECT_LENGTHS:
        filename = f"doc_{doc_idx:03d}_seq_{seq_len}.pt"
        matches = [p for p in files if os.path.basename(p) == filename]

        if not matches:
            raise FileNotFoundError(filename)

        payload = torch.load(
            matches[0],
            map_location="cpu",
            weights_only=False
        )

        for layer in SELECT_LAYERS:
            selected.append({
                "file": matches[0],
                "doc": doc_idx,
                "seq_len": seq_len,
                "layer": layer,
                "record": payload["records"][layer]
            })

print("Selected layer records:", len(selected))


# ============================================================
# Benchmark each fallback implementation
# ============================================================

timing_rows = []
accuracy_rows = []

for record_idx, item in enumerate(selected, 1):
    rec = item["record"]

    N_int, decay, modes, exps, reference, counts, bound = (
        build_reference_and_modes(rec)
    )

    nchunks, nheads, dim = N_int.shape
    nblocks = triton.cdiv(dim, BLOCK)
    grid = (nblocks, nheads)

    N_gpu = torch.as_tensor(
        N_int, device="cuda", dtype=torch.int32
    ).contiguous()

    mode_gpu = torch.as_tensor(
        modes, device="cuda", dtype=torch.int32
    ).contiguous()

    exp_gpu = torch.as_tensor(
        exps, device="cuda", dtype=torch.int32
    ).contiguous()

    decay_gpu = torch.as_tensor(
        decay, device="cuda", dtype=torch.float64
    ).contiguous()

    out_gpu = torch.empty(
        nchunks * nheads * dim,
        device="cuda",
        dtype=torch.int64
    )

    final_gpu = torch.empty(
        nheads * dim,
        device="cuda",
        dtype=torch.int64
    )

    implementations = [
        ("FP64", 0, 0, None)
    ]

    for q_bits in Q_VALUES:
        q_scale = 1 << q_bits

        decay_q = np.floor(
            np.asarray(decay, dtype=np.float64) * q_scale + 0.5
        ).astype(np.int64)

        decay_q_gpu = torch.as_tensor(
            decay_q, device="cuda", dtype=torch.int64
        ).contiguous()

        implementations.append((
            f"Q{q_bits}", q_bits, q_bits, decay_q_gpu
        ))

    # Warm up all implementations before timing.
    for name, kind, q_bits, decay_q_gpu in implementations:
        shiftscan_fallback_benchmark_kernel[grid](
            N_gpu,
            mode_gpu,
            exp_gpu,
            decay_gpu,
            decay_q_gpu if decay_q_gpu is not None else decay_gpu,
            out_gpu,
            final_gpu,
            nchunks,
            nheads,
            dim,
            BLOCK,
            kind,
            q_bits,
            num_warps=4,
            num_stages=3
        )

    torch.cuda.synchronize()

    # Rotate timing order to reduce systematic order bias.
    timing_values = {name: [] for name, *_ in implementations}

    for rep in range(5):
        order = implementations[rep % len(implementations):] + \
                implementations[:rep % len(implementations)]

        for name, kind, q_bits, decay_q_gpu in order:

            def run_kernel():
                shiftscan_fallback_benchmark_kernel[grid](
                    N_gpu,
                    mode_gpu,
                    exp_gpu,
                    decay_gpu,
                    decay_q_gpu if decay_q_gpu is not None else decay_gpu,
                    out_gpu,
                    final_gpu,
                    nchunks,
                    nheads,
                    dim,
                    BLOCK,
                    kind,
                    q_bits,
                    num_warps=4,
                    num_stages=3
                )

            ms = triton.testing.do_bench(
                run_kernel,
                warmup=30,
                rep=100
            )

            timing_values[name].append(ms)

    # Correctness check after timing.
    reference = np.asarray(reference, dtype=np.int64)

    for name, kind, q_bits, decay_q_gpu in implementations:
        shiftscan_fallback_benchmark_kernel[grid](
            N_gpu,
            mode_gpu,
            exp_gpu,
            decay_gpu,
            decay_q_gpu if decay_q_gpu is not None else decay_gpu,
            out_gpu,
            final_gpu,
            nchunks,
            nheads,
            dim,
            BLOCK,
            kind,
            q_bits,
            num_warps=4,
            num_stages=3
        )

        torch.cuda.synchronize()

        actual = out_gpu.cpu().numpy().reshape(
            nchunks, nheads, dim
        )
        actual_final = final_gpu.cpu().numpy().reshape(
            nheads, dim
        )

        diff = np.abs(actual - reference)
        final_diff = np.abs(actual_final - reference[-1])

        accuracy_rows.append({
            "file": os.path.basename(item["file"]),
            "doc": item["doc"],
            "seq_len": item["seq_len"],
            "layer": item["layer"],
            "implementation": name,
            "intermediate_mismatches": int(np.count_nonzero(diff)),
            "final_mismatches": int(np.count_nonzero(final_diff)),
            "max_integer_difference": int(
                max(diff.max(), final_diff.max())
            )
        })

    for name, values in timing_values.items():
        timing_rows.append({
            "file": os.path.basename(item["file"]),
            "doc": item["doc"],
            "seq_len": item["seq_len"],
            "layer": item["layer"],
            "implementation": name,
            "median_ms": float(np.median(values)),
            "min_ms": float(np.min(values)),
            "max_ms": float(np.max(values))
        })

    print(
        f"{record_idx}/{len(selected)} completed: "
        f"doc {item['doc']}, seq {item['seq_len']}, "
        f"layer {item['layer']}"
    )


# ============================================================
# Save detailed results and summaries
# ============================================================

timing_df = pd.DataFrame(timing_rows)
accuracy_df = pd.DataFrame(accuracy_rows)

timing_df.to_csv(
    os.path.join(OUT_DIR, "per_layer_timings.csv"),
    index=False
)

accuracy_df.to_csv(
    os.path.join(OUT_DIR, "per_layer_accuracy.csv"),
    index=False
)

timing_summary = (
    timing_df.groupby(["seq_len", "implementation"])["median_ms"]
    .agg(["median", "mean", "std"])
    .reset_index()
)

accuracy_summary = (
    accuracy_df.groupby("implementation")
    .agg(
        intermediate_mismatches=("intermediate_mismatches", "sum"),
        final_mismatches=("final_mismatches", "sum"),
        max_integer_difference=("max_integer_difference", "max"),
        records_with_mismatch=(
            "max_integer_difference",
            lambda x: int((x > 0).sum())
        )
    )
    .reset_index()
)

timing_summary.to_csv(
    os.path.join(OUT_DIR, "timing_summary.csv"),
    index=False
)

accuracy_summary.to_csv(
    os.path.join(OUT_DIR, "accuracy_summary.csv"),
    index=False
)

print("\n========== TIMING SUMMARY ==========")
display(timing_summary)

print("\n========== ACCURACY SUMMARY ==========")
display(accuracy_summary)

print("\nSaved:", OUT_DIR)


Selected layer records: 24
1/24 completed: doc 0, seq 1024, layer 0
2/24 completed: doc 0, seq 1024, layer 8
3/24 completed: doc 0, seq 1024, layer 16
4/24 completed: doc 0, seq 1024, layer 23
5/24 completed: doc 0, seq 2048, layer 0
6/24 completed: doc 0, seq 2048, layer 8
7/24 completed: doc 0, seq 2048, layer 16
8/24 completed: doc 0, seq 2048, layer 23
9/24 completed: doc 4, seq 1024, layer 0
10/24 completed: doc 4, seq 1024, layer 8
11/24 completed: doc 4, seq 1024, layer 16
12/24 completed: doc 4, seq 1024, layer 23
13/24 completed: doc 4, seq 2048, layer 0
14/24 completed: doc 4, seq 2048, layer 8
15/24 completed: doc 4, seq 2048, layer 16
16/24 completed: doc 4, seq 2048, layer 23
17/24 completed: doc 8, seq 1024, layer 0
18/24 completed: doc 8, seq 1024, layer 8
19/24 completed: doc 8, seq 1024, layer 16
20/24 completed: doc 8, seq 1024, layer 23
21/24 completed: doc 8, seq 2048, layer 0
22/24 completed: doc 8, seq 2048, layer 8
23/24 completed: doc 8, seq 2048, layer 16
24/24

,seq_len,implementation,median,mean,std
0,1024,FP64,0.056671,0.056753,0.000650
1,1024,Q30,0.056592,0.056581,0.000575
2,1024,Q32,0.056648,0.056599,0.000589
3,1024,Q33,0.056631,0.056582,0.000623
4,2048,FP64,0.102964,0.103158,0.000756
5,2048,Q30,0.103486,0.103517,0.000973
6,2048,Q32,0.103450,0.103547,0.000972
7,2048,Q33,0.103602,0.103546,0.000979



========== ACCURACY SUMMARY ==========


,implementation,intermediate_mismatches,final_mismatches,max_integer_difference,records_with_mismatch
0,FP64,0,0,0,0
1,Q30,68,16,1,12
2,Q32,8,2,1,4
3,Q33,7,1,1,4



Saved: /kaggle/working/shiftscan_final/state_passing/multisequence_validation/integer_fallback_benchmark_v1


In [188]:

import os, glob
import numpy as np
import pandas as pd
import torch
import triton
import triton.language as tl

BASE = "/kaggle/working/shiftscan_final/state_passing/multisequence_validation"
OUT = os.path.join(BASE, "loop_unroll_benchmark_v1")
os.makedirs(OUT, exist_ok=True)

BLOCK = 128

@triton.jit
def recurrence_loop_kernel(
    N_PTR, DECAY_PTR, OUT_PTR, FINAL_PTR,
    NCHUNKS: tl.constexpr,
    NHEADS: tl.constexpr,
    DIM: tl.constexpr,
    BLOCK_SIZE: tl.constexpr,
    STATIC_UNROLL: tl.constexpr
):
    pid_m = tl.program_id(0)
    pid_h = tl.program_id(1)

    offs = pid_m * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offs < DIM

    state = tl.zeros((BLOCK_SIZE,), dtype=tl.int32)

    if STATIC_UNROLL:
        for c in tl.static_range(NCHUNKS):
            ctrl = pid_h * NCHUNKS + c

            a = tl.load(DECAY_PTR + ctrl).to(tl.float64)
            product = state.to(tl.float64) * a

            rounded = tl.floor(
                tl.abs(product) + 0.5
            ).to(tl.int64)

            rounded = tl.where(
                product < 0,
                -rounded,
                rounded
            )

            state = rounded.to(tl.int32)

            n_offset = (
                c * NHEADS * DIM
                + pid_h * DIM
                + offs
            )

            n = tl.load(
                N_PTR + n_offset,
                mask=mask,
                other=0
            ).to(tl.int32)

            state = state + n

            tl.store(
                OUT_PTR + n_offset,
                state.to(tl.int64),
                mask=mask
            )

    else:
        for c in range(NCHUNKS):
            ctrl = pid_h * NCHUNKS + c

            a = tl.load(DECAY_PTR + ctrl).to(tl.float64)
            product = state.to(tl.float64) * a

            rounded = tl.floor(
                tl.abs(product) + 0.5
            ).to(tl.int64)

            rounded = tl.where(
                product < 0,
                -rounded,
                rounded
            )

            state = rounded.to(tl.int32)

            n_offset = (
                c * NHEADS * DIM
                + pid_h * DIM
                + offs
            )

            n = tl.load(
                N_PTR + n_offset,
                mask=mask,
                other=0
            ).to(tl.int32)

            state = state + n

            tl.store(
                OUT_PTR + n_offset,
                state.to(tl.int64),
                mask=mask
            )

    tl.store(
        FINAL_PTR + pid_h * DIM + offs,
        state.to(tl.int64),
        mask=mask
    )


# Select the same representative documents and layers.
files = sorted(glob.glob(os.path.join(BASE, "doc_*_seq_*.pt")))

selected = []

for doc_idx in [0, 4, 8]:
    for seq_len in [1024, 2048]:
        filename = f"doc_{doc_idx:03d}_seq_{seq_len}.pt"
        path = next(
            p for p in files if os.path.basename(p) == filename
        )

        payload = torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )

        for layer in [0, 8, 16, 23]:
            selected.append({
                "file": filename,
                "doc": doc_idx,
                "seq_len": seq_len,
                "layer": layer,
                "rec": payload["records"][layer]
            })


timing_rows = []
accuracy_rows = []

for idx, item in enumerate(selected, 1):
    N_int, decay, modes, exps, reference, counts, bound = (
        build_reference_and_modes(item["rec"])
    )

    nchunks, nheads, dim = N_int.shape
    nblocks = triton.cdiv(dim, BLOCK)
    grid = (nblocks, nheads)

    # This isolates the fallback recurrence:
    # every update uses the exact FP64 coefficient.
    N_gpu = torch.as_tensor(
        N_int,
        device="cuda",
        dtype=torch.int32
    ).contiguous()

    decay_gpu = torch.as_tensor(
        decay,
        device="cuda",
        dtype=torch.float64
    ).contiguous()

    out_gpu = torch.empty(
        nchunks * nheads * dim,
        device="cuda",
        dtype=torch.int64
    )

    final_gpu = torch.empty(
        nheads * dim,
        device="cuda",
        dtype=torch.int64
    )

    implementations = [
        ("dynamic_loop", False),
        ("static_unroll", True)
    ]

    # Warmup.
    for name, static in implementations:
        recurrence_loop_kernel[grid](
            N_gpu,
            decay_gpu,
            out_gpu,
            final_gpu,
            nchunks,
            nheads,
            dim,
            BLOCK,
            static,
            num_warps=4,
            num_stages=3
        )

    torch.cuda.synchronize()

    values = {
        "dynamic_loop": [],
        "static_unroll": []
    }

    # Alternate execution order.
    for rep in range(7):
        order = implementations if rep % 2 == 0 else implementations[::-1]

        for name, static in order:
            def run():
                recurrence_loop_kernel[grid](
                    N_gpu,
                    decay_gpu,
                    out_gpu,
                    final_gpu,
                    nchunks,
                    nheads,
                    dim,
                    BLOCK,
                    static,
                    num_warps=4,
                    num_stages=3
                )

            ms = triton.testing.do_bench(
                run,
                warmup=30,
                rep=100
            )

            values[name].append(ms)

    reference = np.asarray(reference, dtype=np.int64)

    for name, static in implementations:
        recurrence_loop_kernel[grid](
            N_gpu,
            decay_gpu,
            out_gpu,
            final_gpu,
            nchunks,
            nheads,
            dim,
            BLOCK,
            static,
            num_warps=4,
            num_stages=3
        )

        torch.cuda.synchronize()

        actual = out_gpu.cpu().numpy().reshape(
            nchunks, nheads, dim
        )

        actual_final = final_gpu.cpu().numpy().reshape(
            nheads, dim
        )

        diff = np.abs(actual - reference)
        final_diff = np.abs(actual_final - reference[-1])

        accuracy_rows.append({
            "file": item["file"],
            "seq_len": item["seq_len"],
            "layer": item["layer"],
            "implementation": name,
            "intermediate_mismatches": int(np.count_nonzero(diff)),
            "final_mismatches": int(np.count_nonzero(final_diff)),
            "max_integer_difference": int(
                max(diff.max(), final_diff.max())
            )
        })

    for name, timings in values.items():
        timing_rows.append({
            "file": item["file"],
            "seq_len": item["seq_len"],
            "layer": item["layer"],
            "implementation": name,
            "median_ms": float(np.median(timings)),
            "mean_ms": float(np.mean(timings)),
            "std_ms": float(np.std(timings))
        })

    print(f"{idx}/{len(selected)} completed")


timing_df = pd.DataFrame(timing_rows)
accuracy_df = pd.DataFrame(accuracy_rows)

timing_df.to_csv(
    os.path.join(OUT, "per_layer_timings.csv"),
    index=False
)

accuracy_df.to_csv(
    os.path.join(OUT, "per_layer_accuracy.csv"),
    index=False
)

summary = (
    timing_df.groupby(["seq_len", "implementation"])["median_ms"]
    .agg(["median", "mean", "std"])
    .reset_index()
)

accuracy_summary = (
    accuracy_df.groupby("implementation")
    .agg(
        intermediate_mismatches=("intermediate_mismatches", "sum"),
        final_mismatches=("final_mismatches", "sum"),
        max_integer_difference=("max_integer_difference", "max")
    )
    .reset_index()
)

summary.to_csv(
    os.path.join(OUT, "timing_summary.csv"),
    index=False
)

accuracy_summary.to_csv(
    os.path.join(OUT, "accuracy_summary.csv"),
    index=False
)

print("\nTIMING SUMMARY")
display(summary)

print("\nACCURACY SUMMARY")
display(accuracy_summary)

print("\nSaved:", OUT)


1/24 completed
2/24 completed
3/24 completed
4/24 completed
5/24 completed
6/24 completed
7/24 completed
8/24 completed
9/24 completed
10/24 completed
11/24 completed
12/24 completed
13/24 completed
14/24 completed
15/24 completed
16/24 completed
17/24 completed
18/24 completed
19/24 completed
20/24 completed
21/24 completed
22/24 completed
23/24 completed
24/24 completed

TIMING SUMMARY


,seq_len,implementation,median,mean,std
0,1024,dynamic_loop,0.056519,0.056273,0.000602
1,1024,static_unroll,0.056528,0.056257,0.000594
2,2048,dynamic_loop,0.104783,0.104718,0.000464
3,2048,static_unroll,0.104820,0.104701,0.000432



ACCURACY SUMMARY


,implementation,intermediate_mismatches,final_mismatches,max_integer_difference
0,dynamic_loop,3273257,697750,1316
1,static_unroll,3273257,697750,1316



Saved: /kaggle/working/shiftscan_final/state_passing/multisequence_validation/loop_unroll_benchmark_v1


In [189]:

import os, glob, ast
import numpy as np
import pandas as pd
import torch

BASE = "/kaggle/working/shiftscan_final/state_passing/multisequence_validation"

OUT = os.path.join(BASE, "full_arithmetic_ablation_v1")
os.makedirs(OUT, exist_ok=True)

# Fixed experimental settings
FRAC_BITS = 20
SCALE = 1 << FRAC_BITS
THRESHOLD = 0.001

POT_MIN_K = 4
POT_MAX_K = 30
POT_MIN = 2.0 ** (-POT_MAX_K)

GRID_NAME = "K1_14_T3"

# ============================================================
# Load canonical three-term Dyadic2 grid
# ============================================================

grid_path = (
    "/kaggle/working/shiftscan_final/"
    "canonical_dyadic_grid_definitions.csv"
)

grid_df = pd.read_csv(grid_path)
grid_df = grid_df[grid_df["grid"] == GRID_NAME].copy()

if len(grid_df) == 0:
    raise RuntimeError(f"Could not find {GRID_NAME} in canonical grid CSV")

def parse_exponents(value):
    if isinstance(value, (tuple, list, np.ndarray)):
        return tuple(int(x) for x in value)

    parsed = ast.literal_eval(str(value))
    return tuple(int(x) for x in parsed)

grid_df["exp_tuple"] = grid_df["canonical_exponents"].apply(
    parse_exponents
)

grid_a = grid_df["decay"].to_numpy(dtype=np.float64)
grid_exps = list(grid_df["exp_tuple"])

order = np.argsort(grid_a)
grid_a = grid_a[order]
grid_exps = [grid_exps[i] for i in order]

grid_d = 1.0 - grid_a
log_grid_d = np.log(grid_d)

print("Canonical grid:", GRID_NAME)
print("Unique coefficients:", len(grid_a))
print("Decay range:", float(grid_a.min()), float(grid_a.max()))
print("Fraction bits:", FRAC_BITS)
print("Local error threshold:", THRESHOLD)


# ============================================================
# Rounding helpers
# ============================================================

def round_away_np(x):
    x = np.asarray(x, dtype=np.float64)
    rounded = np.floor(np.abs(x) + 0.5)
    return np.where(x < 0, -rounded, rounded).astype(np.int64)


def round_shift_np(x, k):
    x = np.asarray(x, dtype=np.int64)
    ax = np.abs(x)
    q = (ax + (1 << (k - 1))) >> k
    return np.where(x < 0, -q, q).astype(np.int64)


# ============================================================
# Policy definitions
# ============================================================

POLICIES = [
    "Exact_FP32_Decay",
    "Dyadic2_Only",
    "Dyadic2_Plus_PoT",
    "Dyadic2_PoT_ExactZero",
    "Full_Adaptive_ApproxZero"
]


def choose_policy_mode(a, state, policy):
    """
    Returns:
      mode:
        0 = exact multiplication fallback
        1 = three-term Dyadic2
        2 = direct power-of-two shift
        3 = exact zero
        4 = approximate zero

      chosen_decay
      exponent tuple
      local predicted error
    """

    max_state = (
        float(np.max(np.abs(state))) / SCALE
        if state.size else 0.0
    )

    # Exact reference policy: always use original decay.
    if policy == "Exact_FP32_Decay":
        return 0, a, (0, 0, 0), 0.0

    # Exact zero is mathematically represented without multiplication.
    if a == 0.0:
        return 3, 0.0, (0, 0, 0), 0.0

    # Try Dyadic2 for coefficients covered by the canonical grid.
    if grid_a.min() <= a <= grid_a.max():
        d = 1.0 - a

        idx = int(np.argmin(
            np.abs(log_grid_d - np.log(d))
        ))

        aq = float(grid_a[idx])
        local_bound = abs(aq - a) * max_state

        if local_bound <= THRESHOLD:
            return 1, aq, grid_exps[idx], local_bound

    # Direct PoT is available only in policies that include it.
    if policy in [
        "Dyadic2_Plus_PoT",
        "Dyadic2_PoT_ExactZero",
        "Full_Adaptive_ApproxZero"
    ]:
        if 0.0 < a < 0.125 and a >= POT_MIN:
            ks = np.arange(POT_MIN_K, POT_MAX_K + 1)
            values = np.exp2(-ks.astype(np.float64))

            idx = int(np.argmin(
                np.abs(np.log(values) - np.log(a))
            ))

            aq = float(values[idx])
            k = int(ks[idx])
            local_bound = abs(aq - a) * max_state

            if local_bound <= THRESHOLD:
                return 2, aq, (k, 0, 0), local_bound

    # Approximate zero is enabled only in the full policy.
    if policy == "Full_Adaptive_ApproxZero":
        if 0.0 < a < POT_MIN:
            local_bound = a * max_state

            if local_bound <= THRESHOLD:
                return 4, 0.0, (0, 0, 0), local_bound

    # Preserve original decay when approximation is rejected.
    return 0, a, (0, 0, 0), 0.0


# ============================================================
# Run one policy on one actual Mamba-2 layer record
# ============================================================

def evaluate_policy(rec, policy):
    # Actual captured SSD new-state contributions.
    N_float = (
        rec["states"].detach().cpu().float().numpy()[0]
    )

    log_decay = (
        rec["decay_log"].detach().cpu().float()
    )

    # Use FP32 exp to reproduce actual Triton underflow behaviour.
    decay = torch.exp(log_decay).numpy()[0].astype(np.float64)

    chunks, heads, dim = N_float.shape

    # Quantize N once, identically for every policy.
    N_int = round_away_np(N_float.astype(np.float64) * SCALE)

    # Exact FP64-driven integer recurrence reference.
    exact_state = np.zeros((heads, dim), dtype=np.int64)
    exact_trajectory = np.zeros(
        (chunks, heads, dim), dtype=np.int64
    )

    for c in range(chunks):
        for h in range(heads):
            a = float(decay[h, c])
            exact_state[h] = (
                round_away_np(a * exact_state[h].astype(np.float64))
                + N_int[c, h]
            )

        exact_trajectory[c] = exact_state

    # Candidate policy recurrence.
    state = np.zeros((heads, dim), dtype=np.int64)
    trajectory = np.zeros(
        (chunks, heads, dim), dtype=np.int64
    )

    counts = {
        "dyadic2": 0,
        "pot": 0,
        "exact_zero": 0,
        "approx_zero": 0,
        "fallback": 0
    }

    coefficient_errors = []
    local_bounds = []

    for c in range(chunks):
        for h in range(heads):
            a = float(decay[h, c])

            mode, aq, exponents, bound = choose_policy_mode(
                a, state[h], policy
            )

            local_bounds.append(bound)
            coefficient_errors.append(abs(aq - a))

            if mode == 1:
                counts["dyadic2"] += 1
                new_state = state[h].copy()

                for k in exponents:
                    new_state -= round_shift_np(state[h], int(k))

                state[h] = new_state

            elif mode == 2:
                counts["pot"] += 1
                state[h] = round_shift_np(
                    state[h], int(exponents[0])
                )

            elif mode in (3, 4):
                if mode == 3:
                    counts["exact_zero"] += 1
                else:
                    counts["approx_zero"] += 1

                state[h] = np.zeros(dim, dtype=np.int64)

            else:
                counts["fallback"] += 1
                state[h] = round_away_np(
                    a * state[h].astype(np.float64)
                )

            state[h] += N_int[c, h]

        trajectory[c] = state

    # Integer-state errors against the exact FP64-driven recurrence.
    difference = (
        trajectory.astype(np.float64)
        - exact_trajectory.astype(np.float64)
    )

    abs_error = np.abs(difference)

    trajectory_mae = float(abs_error.mean() / SCALE)
    trajectory_rmse = float(
        np.sqrt(np.mean(difference ** 2)) / SCALE
    )
    trajectory_max = float(abs_error.max() / SCALE)

    final_difference = (
        trajectory[-1].astype(np.float64)
        - exact_trajectory[-1].astype(np.float64)
    )

    final_abs = np.abs(final_difference)

    final_mae = float(final_abs.mean() / SCALE)
    final_rmse = float(
        np.sqrt(np.mean(final_difference ** 2)) / SCALE
    )
    final_max = float(final_abs.max() / SCALE)

    total_updates = chunks * heads

    strict_shift = counts["dyadic2"] + counts["pot"]
    zero_updates = counts["exact_zero"] + counts["approx_zero"]
    nonfallback = strict_shift + zero_updates

    return {
        "total_updates": total_updates,
        **counts,
        "strict_shift_updates": strict_shift,
        "zero_updates": zero_updates,
        "nonfallback_updates": nonfallback,

        "strict_shift_pct": 100 * strict_shift / total_updates,
        "coverage_pct": 100 * nonfallback / total_updates,
        "fallback_pct": 100 * counts["fallback"] / total_updates,

        "coefficient_mae": float(np.mean(coefficient_errors)),
        "coefficient_max_error": float(np.max(coefficient_errors)),
        "max_local_bound": float(np.max(local_bounds)),

        "trajectory_mae": trajectory_mae,
        "trajectory_rmse": trajectory_rmse,
        "trajectory_max": trajectory_max,

        "final_mae": final_mae,
        "final_rmse": final_rmse,
        "final_max": final_max,

        "max_integer_state": int(np.max(np.abs(trajectory)))
    }


# ============================================================
# Evaluate all 20 captures and 24 layers
# ============================================================

files = sorted(
    glob.glob(os.path.join(BASE, "doc_*_seq_*.pt"))
)

if len(files) != 20:
    raise RuntimeError(
        f"Expected 20 capture files, found {len(files)}"
    )

rows = []

for file_idx, path in enumerate(files, 1):
    payload = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    filename = os.path.basename(path)

    seq_len = int(filename.split("_seq_")[1].split(".")[0])
    doc_id = int(filename.split("_")[1])

    if len(payload["records"]) != 24:
        raise RuntimeError(
            f"{filename} has {len(payload['records'])} layers, expected 24"
        )

    for layer, rec in enumerate(payload["records"]):
        for policy in POLICIES:
            result = evaluate_policy(rec, policy)

            rows.append({
                "file": filename,
                "doc": doc_id,
                "seq_len": seq_len,
                "layer": layer,
                "policy": policy,
                **result
            })

    print(f"{file_idx}/{len(files)} captures completed")


# ============================================================
# Save per-layer results
# ============================================================

df = pd.DataFrame(rows)

df.to_csv(
    os.path.join(OUT, "per_layer_results.csv"),
    index=False
)


# ============================================================
# Aggregate results
# ============================================================

metric_cols = [
    "strict_shift_pct",
    "coverage_pct",
    "fallback_pct",
    "coefficient_mae",
    "coefficient_max_error",
    "trajectory_mae",
    "trajectory_rmse",
    "trajectory_max",
    "final_mae",
    "final_rmse",
    "final_max"
]

aggregate = (
    df.groupby(["policy", "seq_len"])[metric_cols]
    .agg(["mean", "max"])
)

aggregate.columns = [
    f"{metric}_{stat}"
    for metric, stat in aggregate.columns
]

aggregate = aggregate.reset_index()

aggregate.to_csv(
    os.path.join(OUT, "aggregate_by_length.csv"),
    index=False
)

overall = (
    df.groupby("policy")[metric_cols]
    .agg(["mean", "max"])
)

overall.columns = [
    f"{metric}_{stat}"
    for metric, stat in overall.columns
]

overall = overall.reset_index()

overall.to_csv(
    os.path.join(OUT, "aggregate_overall.csv"),
    index=False
)


# ============================================================
# Arithmetic mode counts across all records
# ============================================================

mode_cols = [
    "dyadic2",
    "pot",
    "exact_zero",
    "approx_zero",
    "fallback",
    "strict_shift_updates",
    "zero_updates",
    "nonfallback_updates",
    "total_updates"
]

mode_summary = (
    df.groupby("policy")[mode_cols]
    .sum()
    .reset_index()
)

for col in [
    "dyadic2",
    "pot",
    "exact_zero",
    "approx_zero",
    "fallback",
    "strict_shift_updates",
    "zero_updates",
    "nonfallback_updates"
]:
    mode_summary[col + "_pct"] = (
        100 * mode_summary[col] / mode_summary["total_updates"]
    )

mode_summary.to_csv(
    os.path.join(OUT, "arithmetic_mode_summary.csv"),
    index=False
)


# ============================================================
# Print final tables
# ============================================================

print("\n========== OVERALL NUMERICAL RESULTS ==========")
display(overall)

print("\n========== ARITHMETIC MODE COUNTS ==========")
display(mode_summary)

print("\n========== RESULTS BY SEQUENCE LENGTH ==========")
display(aggregate)

print("\nLayer-level records:", len(df))
print("Expected records:", 20 * 24 * len(POLICIES))
print("Saved results:", OUT)


Canonical grid: K1_14_T3
Unique coefficients: 467
Decay range: 0.125 0.99981689453125
Fraction bits: 20
Local error threshold: 0.001
1/20 captures completed
2/20 captures completed
3/20 captures completed
4/20 captures completed
5/20 captures completed
6/20 captures completed
7/20 captures completed
8/20 captures completed
9/20 captures completed
10/20 captures completed
11/20 captures completed
12/20 captures completed
13/20 captures completed
14/20 captures completed
15/20 captures completed
16/20 captures completed
17/20 captures completed
18/20 captures completed
19/20 captures completed
20/20 captures completed

========== OVERALL NUMERICAL RESULTS ==========


,policy,strict_shift_pct_mean,strict_shift_pct_max,coverage_pct_mean,coverage_pct_max,fallback_pct_mean,fallback_pct_max,coefficient_mae_mean,coefficient_mae_max,coefficient_max_error_mean,...,trajectory_rmse_mean,trajectory_rmse_max,trajectory_max_mean,trajectory_max_max,final_mae_mean,final_mae_max,final_rmse_mean,final_rmse_max,final_max_mean,final_max_max
0,Dyadic2_Only,6.733941,43.750000,38.760851,81.250000,61.239149,100.000000,0.000256,0.002196,0.010246,...,0.000003,0.000022,0.000436,0.002527,6.270430e-07,0.000012,0.000004,0.000041,0.000305,0.002527
1,Dyadic2_Plus_PoT,21.188151,77.083333,53.215061,95.833333,46.784939,83.333333,0.000483,0.003226,0.019964,...,0.000005,0.000022,0.000697,0.002527,1.062675e-06,0.000012,0.000006,0.000041,0.000461,0.002527
2,Dyadic2_PoT_ExactZero,21.188151,77.083333,53.215061,95.833333,46.784939,83.333333,0.000483,0.003226,0.019964,...,0.000005,0.000022,0.000697,0.002527,1.062675e-06,0.000012,0.000006,0.000041,0.000461,0.002527
3,Exact_FP32_Decay,0.000000,0.000000,0.000000,0.000000,100.000000,100.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
4,Full_Adaptive_ApproxZero,21.188151,77.083333,77.795139,100.000000,22.204861,59.375000,0.000483,0.003226,0.019964,...,0.000005,0.000022,0.000697,0.002527,1.062675e-06,0.000012,0.000006,0.000041,0.000461,0.002527



========== ARITHMETIC MODE COUNTS ==========


,policy,dyadic2,pot,exact_zero,approx_zero,fallback,strict_shift_updates,zero_updates,nonfallback_updates,total_updates,dyadic2_pct,pot_pct,exact_zero_pct,approx_zero_pct,fallback_pct,strict_shift_updates_pct,zero_updates_pct,nonfallback_updates_pct
0,Dyadic2_Only,4433,0,22080,0,42607,4433,22080,26513,69120,6.413484,0.000000,31.944444,0.000000,61.642072,6.413484,31.944444,38.357928
1,Dyadic2_Plus_PoT,4433,9796,22080,0,32811,14229,22080,36309,69120,6.413484,14.172454,31.944444,0.000000,47.469618,20.585938,31.944444,52.530382
2,Dyadic2_PoT_ExactZero,4433,9796,22080,0,32811,14229,22080,36309,69120,6.413484,14.172454,31.944444,0.000000,47.469618,20.585938,31.944444,52.530382
3,Exact_FP32_Decay,0,0,0,0,69120,0,0,0,69120,0.000000,0.000000,0.000000,0.000000,100.000000,0.000000,0.000000,0.000000
4,Full_Adaptive_ApproxZero,4433,9796,22080,17042,15769,14229,39122,53351,69120,6.413484,14.172454,31.944444,24.655671,22.813947,20.585938,56.600116,77.186053



========== RESULTS BY SEQUENCE LENGTH ==========


,policy,seq_len,strict_shift_pct_mean,strict_shift_pct_max,coverage_pct_mean,coverage_pct_max,fallback_pct_mean,fallback_pct_max,coefficient_mae_mean,coefficient_mae_max,...,trajectory_rmse_mean,trajectory_rmse_max,trajectory_max_mean,trajectory_max_max,final_mae_mean,final_mae_max,final_rmse_mean,final_rmse_max,final_max_mean,final_max_max
0,Dyadic2_Only,1024,7.695312,43.750000,39.969618,81.250000,60.030382,100.000000,0.000335,0.002196,...,0.000003,0.000017,0.000363,0.001663,5.106903e-07,0.000005,0.000003,0.000023,0.000286,0.001663
1,Dyadic2_Only,2048,5.772569,38.020833,37.552083,77.604167,62.447917,100.000000,0.000178,0.001105,...,0.000004,0.000022,0.000509,0.002527,7.433958e-07,0.000012,0.000004,0.000041,0.000325,0.002527
2,Dyadic2_Plus_PoT,1024,22.994792,77.083333,55.269097,95.833333,44.730903,83.333333,0.000628,0.003226,...,0.000005,0.000019,0.000590,0.001663,9.453506e-07,0.000007,0.000006,0.000027,0.000440,0.001663
3,Dyadic2_Plus_PoT,2048,19.381510,73.437500,51.161024,95.833333,48.838976,83.333333,0.000338,0.001672,...,0.000006,0.000022,0.000803,0.002527,1.180000e-06,0.000012,0.000007,0.000041,0.000482,0.002527
4,Dyadic2_PoT_ExactZero,1024,22.994792,77.083333,55.269097,95.833333,44.730903,83.333333,0.000628,0.003226,...,0.000005,0.000019,0.000590,0.001663,9.453506e-07,0.000007,0.000006,0.000027,0.000440,0.001663
5,Dyadic2_PoT_ExactZero,2048,19.381510,73.437500,51.161024,95.833333,48.838976,83.333333,0.000338,0.001672,...,0.000006,0.000022,0.000803,0.002527,1.180000e-06,0.000012,0.000007,0.000041,0.000482,0.002527
6,Exact_FP32_Decay,1024,0.000000,0.000000,0.000000,0.000000,100.000000,100.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
7,Exact_FP32_Decay,2048,0.000000,0.000000,0.000000,0.000000,100.000000,100.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
8,Full_Adaptive_ApproxZero,1024,22.994792,77.083333,79.622396,100.000000,20.377604,48.958333,0.000628,0.003226,...,0.000005,0.000019,0.000590,0.001663,9.453506e-07,0.000007,0.000006,0.000027,0.000440,0.001663
9,Full_Adaptive_ApproxZero,2048,19.381510,73.437500,75.967882,100.000000,24.032118,59.375000,0.000338,0.001672,...,0.000006,0.000022,0.000803,0.002527,1.180000e-06,0.000012,0.000007,0.000041,0.000482,0.002527



Layer-level records: 2400
Expected records: 2400
Saved results: /kaggle/working/shiftscan_final/state_passing/multisequence_validation/full_arithmetic_ablation_v1


In [190]:

import os, glob, ast
import numpy as np
import pandas as pd
import torch

BASE = "/kaggle/working/shiftscan_final/state_passing/multisequence_validation"
OUT = os.path.join(BASE, "corrected_independent_ablation_v2")
os.makedirs(OUT, exist_ok=True)

FRAC_BITS = 20
SCALE = 1 << FRAC_BITS
THRESHOLD = 0.001

POT_MIN_K = 4
POT_MAX_K = 30
POT_MIN = 2.0 ** (-POT_MAX_K)

GRID_NAME = "K1_14_T3"

# ------------------------------------------------------------
# Canonical Dyadic2 grid
# ------------------------------------------------------------

grid_path = (
    "/kaggle/working/shiftscan_final/"
    "canonical_dyadic_grid_definitions.csv"
)

grid_df = pd.read_csv(grid_path)
grid_df = grid_df[grid_df["grid"] == GRID_NAME].copy()

if len(grid_df) == 0:
    raise RuntimeError("Canonical grid not found.")

def parse_exponents(value):
    if isinstance(value, (tuple, list, np.ndarray)):
        return tuple(int(x) for x in value)
    return tuple(int(x) for x in ast.literal_eval(str(value)))

grid_df["exp_tuple"] = grid_df["canonical_exponents"].apply(
    parse_exponents
)

grid_a = grid_df["decay"].to_numpy(dtype=np.float64)
grid_exps = list(grid_df["exp_tuple"])

order = np.argsort(grid_a)
grid_a = grid_a[order]
grid_exps = [grid_exps[i] for i in order]

grid_d = 1.0 - grid_a
log_grid_d = np.log(grid_d)

print("Grid:", GRID_NAME)
print("Unique coefficients:", len(grid_a))
print("Decay range:", grid_a.min(), grid_a.max())

# ------------------------------------------------------------
# Rounding
# ------------------------------------------------------------

def round_away_np(x):
    x = np.asarray(x, dtype=np.float64)
    rounded = np.floor(np.abs(x) + 0.5)
    return np.where(x < 0, -rounded, rounded).astype(np.int64)

def round_shift_np(x, k):
    x = np.asarray(x, dtype=np.int64)
    ax = np.abs(x)
    q = (ax + (1 << (k - 1))) >> k
    return np.where(x < 0, -q, q).astype(np.int64)

# ------------------------------------------------------------
# Independent policy switches
# ------------------------------------------------------------

POLICIES = {
    "A_Dyadic2": {
        "pot": False,
        "exact_zero": False,
        "approx_zero": False
    },
    "B_Dyadic2_ExactZero": {
        "pot": False,
        "exact_zero": True,
        "approx_zero": False
    },
    "C_Dyadic2_PoT": {
        "pot": True,
        "exact_zero": False,
        "approx_zero": False
    },
    "D_Dyadic2_PoT_ExactZero": {
        "pot": True,
        "exact_zero": True,
        "approx_zero": False
    },
    "E_Dyadic2_PoT_ApproxZero": {
        "pot": True,
        "exact_zero": False,
        "approx_zero": True
    },
    "F_FullAdaptive": {
        "pot": True,
        "exact_zero": True,
        "approx_zero": True
    }
}

# ------------------------------------------------------------
# Select representation for one decay coefficient
#
# mode 0 = exact multiplication fallback
# mode 1 = three-term Dyadic2
# mode 2 = direct PoT
# mode 3 = exact zero
# mode 4 = approximate zero
# ------------------------------------------------------------

def choose_mode(a, state, switches):

    max_state = (
        float(np.max(np.abs(state))) / SCALE
        if state.size else 0.0
    )

    # Exact zero is independently controlled.
    if a == 0.0:
        if switches["exact_zero"]:
            return 3, 0.0, (0, 0, 0), 0.0
        return 0, a, (0, 0, 0), 0.0

    # Three-term Dyadic2.
    if grid_a.min() <= a <= grid_a.max():
        d = 1.0 - a
        idx = int(np.argmin(np.abs(log_grid_d - np.log(d))))

        aq = float(grid_a[idx])
        exps = grid_exps[idx]
        bound = abs(aq - a) * max_state

        if bound <= THRESHOLD:
            return 1, aq, exps, bound

    # Direct power-of-two representation.
    if switches["pot"]:
        if 0.0 < a < 0.125 and a >= POT_MIN:
            ks = np.arange(POT_MIN_K, POT_MAX_K + 1)
            values = np.exp2(-ks.astype(np.float64))

            idx = int(np.argmin(
                np.abs(np.log(values) - np.log(a))
            ))

            aq = float(values[idx])
            k = int(ks[idx])
            bound = abs(aq - a) * max_state

            if bound <= THRESHOLD:
                return 2, aq, (k, 0, 0), bound

    # Approximate zero is independently controlled.
    if switches["approx_zero"]:
        if 0.0 < a < POT_MIN:
            bound = a * max_state

            if bound <= THRESHOLD:
                return 4, 0.0, (0, 0, 0), bound

    # Exact multiplication fallback.
    return 0, a, (0, 0, 0), 0.0


# ------------------------------------------------------------
# Evaluate one layer under one policy
# ------------------------------------------------------------

def evaluate_policy(rec, policy_name, switches):

    N_float = (
        rec["states"].detach().cpu().float().numpy()[0]
    )

    log_decay = (
        rec["decay_log"].detach().cpu().float()
    )

    # Match the FP32 exponential used by the actual capture.
    decay = torch.exp(log_decay).numpy()[0].astype(np.float64)

    chunks, heads, dim = N_float.shape

    N_int = round_away_np(
        N_float.astype(np.float64) * SCALE
    )

    # Exact-decay integer recurrence control.
    reference = np.zeros(
        (chunks, heads, dim), dtype=np.int64
    )

    ref_state = np.zeros((heads, dim), dtype=np.int64)

    for c in range(chunks):
        for h in range(heads):
            a = float(decay[h, c])

            ref_state[h] = (
                round_away_np(
                    a * ref_state[h].astype(np.float64)
                )
                + N_int[c, h]
            )

        reference[c] = ref_state

    # Candidate recurrence.
    state = np.zeros((heads, dim), dtype=np.int64)
    trajectory = np.zeros(
        (chunks, heads, dim), dtype=np.int64
    )

    counts = {
        "dyadic2": 0,
        "pot": 0,
        "exact_zero": 0,
        "approx_zero": 0,
        "fallback": 0
    }

    coefficient_errors = []
    local_bounds = []

    for c in range(chunks):
        for h in range(heads):

            a = float(decay[h, c])

            mode, aq, exps, bound = choose_mode(
                a, state[h], switches
            )

            coefficient_errors.append(abs(aq - a))
            local_bounds.append(bound)

            if mode == 1:
                counts["dyadic2"] += 1

                updated = state[h].copy()

                for k in exps:
                    updated -= round_shift_np(
                        state[h], int(k)
                    )

                state[h] = updated

            elif mode == 2:
                counts["pot"] += 1
                state[h] = round_shift_np(
                    state[h], int(exps[0])
                )

            elif mode == 3:
                counts["exact_zero"] += 1
                state[h] = np.zeros(dim, dtype=np.int64)

            elif mode == 4:
                counts["approx_zero"] += 1
                state[h] = np.zeros(dim, dtype=np.int64)

            else:
                counts["fallback"] += 1
                state[h] = round_away_np(
                    a * state[h].astype(np.float64)
                )

            state[h] += N_int[c, h]

        trajectory[c] = state

    # --------------------------------------------------------
    # Numerical errors
    # --------------------------------------------------------

    diff = (
        trajectory.astype(np.float64)
        - reference.astype(np.float64)
    )

    abs_diff = np.abs(diff)

    final_diff = (
        trajectory[-1].astype(np.float64)
        - reference[-1].astype(np.float64)
    )

    final_abs = np.abs(final_diff)

    total = chunks * heads
    strict_shift = counts["dyadic2"] + counts["pot"]
    zero_count = counts["exact_zero"] + counts["approx_zero"]

    return {
        "total_updates": total,

        **counts,

        "strict_shift_updates": strict_shift,
        "zero_updates": zero_count,
        "nonfallback_updates": strict_shift + zero_count,

        "strict_shift_pct": 100 * strict_shift / total,
        "coverage_pct": 100 * (strict_shift + zero_count) / total,
        "fallback_pct": 100 * counts["fallback"] / total,

        "coefficient_mae": float(np.mean(coefficient_errors)),
        "coefficient_max_error": float(np.max(coefficient_errors)),
        "max_local_bound": float(np.max(local_bounds)),

        "trajectory_mae": float(abs_diff.mean() / SCALE),
        "trajectory_rmse": float(
            np.sqrt(np.mean(diff ** 2)) / SCALE
        ),
        "trajectory_max": float(abs_diff.max() / SCALE),

        "final_mae": float(final_abs.mean() / SCALE),
        "final_rmse": float(
            np.sqrt(np.mean(final_diff ** 2)) / SCALE
        ),
        "final_max": float(final_abs.max() / SCALE),

        "max_integer_state": int(np.max(np.abs(trajectory)))
    }


# ------------------------------------------------------------
# Load all captures and run
# ------------------------------------------------------------

files = sorted(
    glob.glob(os.path.join(BASE, "doc_*_seq_*.pt"))
)

if len(files) != 20:
    raise RuntimeError(
        f"Expected 20 capture files, found {len(files)}"
    )

rows = []

for file_idx, path in enumerate(files, 1):

    payload = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    filename = os.path.basename(path)
    seq_len = int(filename.split("_seq_")[1].split(".")[0])
    doc_id = int(filename.split("_")[1])

    if len(payload["records"]) != 24:
        raise RuntimeError(
            f"{filename}: expected 24 layer records"
        )

    for layer, rec in enumerate(payload["records"]):

        for policy_name, switches in POLICIES.items():

            result = evaluate_policy(
                rec, policy_name, switches
            )

            rows.append({
                "file": filename,
                "doc": doc_id,
                "seq_len": seq_len,
                "layer": layer,
                "policy": policy_name,
                **result
            })

    print(f"Completed capture {file_idx}/20: {filename}")


# ------------------------------------------------------------
# Save detailed results
# ------------------------------------------------------------

df = pd.DataFrame(rows)

expected = 20 * 24 * len(POLICIES)

if len(df) != expected:
    raise RuntimeError(
        f"Unexpected row count: {len(df)}; expected {expected}"
    )

df.to_csv(
    os.path.join(OUT, "per_layer_results.csv"),
    index=False
)


# ------------------------------------------------------------
# Overall aggregate
# ------------------------------------------------------------

metrics = [
    "strict_shift_pct",
    "coverage_pct",
    "fallback_pct",
    "coefficient_mae",
    "coefficient_max_error",
    "trajectory_mae",
    "trajectory_rmse",
    "trajectory_max",
    "final_mae",
    "final_rmse",
    "final_max"
]

overall = (
    df.groupby("policy")[metrics]
    .agg(["mean", "max"])
)

overall.columns = [
    f"{metric}_{stat}"
    for metric, stat in overall.columns
]

overall = overall.reset_index()

overall.to_csv(
    os.path.join(OUT, "aggregate_overall.csv"),
    index=False
)


# ------------------------------------------------------------
# Results by sequence length
# ------------------------------------------------------------

by_length = (
    df.groupby(["policy", "seq_len"])[metrics]
    .agg(["mean", "max"])
)

by_length.columns = [
    f"{metric}_{stat}"
    for metric, stat in by_length.columns
]

by_length = by_length.reset_index()

by_length.to_csv(
    os.path.join(OUT, "aggregate_by_length.csv"),
    index=False
)


# ------------------------------------------------------------
# Arithmetic mode counts
# ------------------------------------------------------------

mode_cols = [
    "dyadic2",
    "pot",
    "exact_zero",
    "approx_zero",
    "fallback",
    "strict_shift_updates",
    "zero_updates",
    "nonfallback_updates",
    "total_updates"
]

mode_summary = (
    df.groupby("policy")[mode_cols]
    .sum()
    .reset_index()
)

for col in mode_cols[:-1]:
    mode_summary[col + "_pct"] = (
        100 * mode_summary[col]
        / mode_summary["total_updates"]
    )

mode_summary.to_csv(
    os.path.join(OUT, "arithmetic_mode_summary.csv"),
    index=False
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\n========== OVERALL RESULTS ==========")
display(overall)

print("\n========== RESULTS BY SEQUENCE LENGTH ==========")
display(by_length)

print("\n========== ARITHMETIC MODE COUNTS ==========")
display(mode_summary)

print("\nRecords:", len(df))
print("Expected:", expected)
print("Output directory:", OUT)


Grid: K1_14_T3
Unique coefficients: 467
Decay range: 0.125 0.99981689453125
Completed capture 1/20: doc_000_seq_1024.pt
Completed capture 2/20: doc_000_seq_2048.pt
Completed capture 3/20: doc_001_seq_1024.pt
Completed capture 4/20: doc_001_seq_2048.pt
Completed capture 5/20: doc_002_seq_1024.pt
Completed capture 6/20: doc_002_seq_2048.pt
Completed capture 7/20: doc_003_seq_1024.pt
Completed capture 8/20: doc_003_seq_2048.pt
Completed capture 9/20: doc_004_seq_1024.pt
Completed capture 10/20: doc_004_seq_2048.pt
Completed capture 11/20: doc_005_seq_1024.pt
Completed capture 12/20: doc_005_seq_2048.pt
Completed capture 13/20: doc_006_seq_1024.pt
Completed capture 14/20: doc_006_seq_2048.pt
Completed capture 15/20: doc_007_seq_1024.pt
Completed capture 16/20: doc_007_seq_2048.pt
Completed capture 17/20: doc_008_seq_1024.pt
Completed capture 18/20: doc_008_seq_2048.pt
Completed capture 19/20: doc_009_seq_1024.pt
Completed capture 20/20: doc_009_seq_2048.pt

========== OVERALL RESULTS =====

,policy,strict_shift_pct_mean,strict_shift_pct_max,coverage_pct_mean,coverage_pct_max,fallback_pct_mean,fallback_pct_max,coefficient_mae_mean,coefficient_mae_max,coefficient_max_error_mean,...,trajectory_rmse_mean,trajectory_rmse_max,trajectory_max_mean,trajectory_max_max,final_mae_mean,final_mae_max,final_rmse_mean,final_rmse_max,final_max_mean,final_max_max
0,A_Dyadic2,6.733941,43.750000,6.733941,43.750000,93.266059,100.000000,0.000256,0.002196,0.010246,...,0.000003,0.000022,0.000436,0.002527,6.270430e-07,0.000012,0.000004,0.000041,0.000305,0.002527
1,B_Dyadic2_ExactZero,6.733941,43.750000,38.760851,81.250000,61.239149,100.000000,0.000256,0.002196,0.010246,...,0.000003,0.000022,0.000436,0.002527,6.270430e-07,0.000012,0.000004,0.000041,0.000305,0.002527
2,C_Dyadic2_PoT,21.188151,77.083333,21.188151,77.083333,78.811849,98.958333,0.000483,0.003226,0.019964,...,0.000005,0.000022,0.000697,0.002527,1.062675e-06,0.000012,0.000006,0.000041,0.000461,0.002527
3,D_Dyadic2_PoT_ExactZero,21.188151,77.083333,53.215061,95.833333,46.784939,83.333333,0.000483,0.003226,0.019964,...,0.000005,0.000022,0.000697,0.002527,1.062675e-06,0.000012,0.000006,0.000041,0.000461,0.002527
4,E_Dyadic2_PoT_ApproxZero,21.188151,77.083333,45.768229,92.708333,54.231771,93.750000,0.000483,0.003226,0.019964,...,0.000005,0.000022,0.000697,0.002527,1.062675e-06,0.000012,0.000006,0.000041,0.000461,0.002527
5,F_FullAdaptive,21.188151,77.083333,77.795139,100.000000,22.204861,59.375000,0.000483,0.003226,0.019964,...,0.000005,0.000022,0.000697,0.002527,1.062675e-06,0.000012,0.000006,0.000041,0.000461,0.002527



========== RESULTS BY SEQUENCE LENGTH ==========


,policy,seq_len,strict_shift_pct_mean,strict_shift_pct_max,coverage_pct_mean,coverage_pct_max,fallback_pct_mean,fallback_pct_max,coefficient_mae_mean,coefficient_mae_max,...,trajectory_rmse_mean,trajectory_rmse_max,trajectory_max_mean,trajectory_max_max,final_mae_mean,final_mae_max,final_rmse_mean,final_rmse_max,final_max_mean,final_max_max
0,A_Dyadic2,1024,7.695312,43.750000,7.695312,43.750000,92.304688,100.000000,0.000335,0.002196,...,0.000003,0.000017,0.000363,0.001663,5.106903e-07,0.000005,0.000003,0.000023,0.000286,0.001663
1,A_Dyadic2,2048,5.772569,38.020833,5.772569,38.020833,94.227431,100.000000,0.000178,0.001105,...,0.000004,0.000022,0.000509,0.002527,7.433958e-07,0.000012,0.000004,0.000041,0.000325,0.002527
2,B_Dyadic2_ExactZero,1024,7.695312,43.750000,39.969618,81.250000,60.030382,100.000000,0.000335,0.002196,...,0.000003,0.000017,0.000363,0.001663,5.106903e-07,0.000005,0.000003,0.000023,0.000286,0.001663
3,B_Dyadic2_ExactZero,2048,5.772569,38.020833,37.552083,77.604167,62.447917,100.000000,0.000178,0.001105,...,0.000004,0.000022,0.000509,0.002527,7.433958e-07,0.000012,0.000004,0.000041,0.000325,0.002527
4,C_Dyadic2_PoT,1024,22.994792,77.083333,22.994792,77.083333,77.005208,97.916667,0.000628,0.003226,...,0.000005,0.000019,0.000590,0.001663,9.453506e-07,0.000007,0.000006,0.000027,0.000440,0.001663
5,C_Dyadic2_PoT,2048,19.381510,73.437500,19.381510,73.437500,80.618490,98.958333,0.000338,0.001672,...,0.000006,0.000022,0.000803,0.002527,1.180000e-06,0.000012,0.000007,0.000041,0.000482,0.002527
6,D_Dyadic2_PoT_ExactZero,1024,22.994792,77.083333,55.269097,95.833333,44.730903,83.333333,0.000628,0.003226,...,0.000005,0.000019,0.000590,0.001663,9.453506e-07,0.000007,0.000006,0.000027,0.000440,0.001663
7,D_Dyadic2_PoT_ExactZero,2048,19.381510,73.437500,51.161024,95.833333,48.838976,83.333333,0.000338,0.001672,...,0.000006,0.000022,0.000803,0.002527,1.180000e-06,0.000012,0.000007,0.000041,0.000482,0.002527
8,E_Dyadic2_PoT_ApproxZero,1024,22.994792,77.083333,47.348090,92.708333,52.651910,89.583333,0.000628,0.003226,...,0.000005,0.000019,0.000590,0.001663,9.453506e-07,0.000007,0.000006,0.000027,0.000440,0.001663
9,E_Dyadic2_PoT_ApproxZero,2048,19.381510,73.437500,44.188368,90.625000,55.811632,93.750000,0.000338,0.001672,...,0.000006,0.000022,0.000803,0.002527,1.180000e-06,0.000012,0.000007,0.000041,0.000482,0.002527



========== ARITHMETIC MODE COUNTS ==========


,policy,dyadic2,pot,exact_zero,approx_zero,fallback,strict_shift_updates,zero_updates,nonfallback_updates,total_updates,dyadic2_pct,pot_pct,exact_zero_pct,approx_zero_pct,fallback_pct,strict_shift_updates_pct,zero_updates_pct,nonfallback_updates_pct
0,A_Dyadic2,4433,0,0,0,64687,4433,0,4433,69120,6.413484,0.000000,0.000000,0.000000,93.586516,6.413484,0.000000,6.413484
1,B_Dyadic2_ExactZero,4433,0,22080,0,42607,4433,22080,26513,69120,6.413484,0.000000,31.944444,0.000000,61.642072,6.413484,31.944444,38.357928
2,C_Dyadic2_PoT,4433,9796,0,0,54891,14229,0,14229,69120,6.413484,14.172454,0.000000,0.000000,79.414062,20.585938,0.000000,20.585938
3,D_Dyadic2_PoT_ExactZero,4433,9796,22080,0,32811,14229,22080,36309,69120,6.413484,14.172454,31.944444,0.000000,47.469618,20.585938,31.944444,52.530382
4,E_Dyadic2_PoT_ApproxZero,4433,9796,0,17042,37849,14229,17042,31271,69120,6.413484,14.172454,0.000000,24.655671,54.758391,20.585938,24.655671,45.241609
5,F_FullAdaptive,4433,9796,22080,17042,15769,14229,39122,53351,69120,6.413484,14.172454,31.944444,24.655671,22.813947,20.585938,56.600116,77.186053



Records: 2880
Expected: 2880
Output directory: /kaggle/working/shiftscan_final/state_passing/multisequence_validation/corrected_independent_ablation_v2


In [191]:

import os
import pandas as pd
import numpy as np

BASE = (
    "/kaggle/working/shiftscan_final/"
    "state_passing/multisequence_validation"
)

OLD = os.path.join(
    BASE,
    "approximate_zero_validation_v1",
    "baseline_per_layer.csv"
)

NEW = os.path.join(
    BASE,
    "corrected_independent_ablation_v2",
    "per_layer_results.csv"
)

old = pd.read_csv(OLD)
new = pd.read_csv(NEW)

# Select the full adaptive policy from the new experiment.
new_full = new[
    new["policy"] == "F_FullAdaptive"
].copy()

print("Old rows:", len(old))
print("New full-adaptive rows:", len(new_full))

# Inspect available old columns.
print("\nOld columns:")
print(old.columns.tolist())

print("\nNew columns:")
print(new_full.columns.tolist())

# Compare arithmetic counts if matching fields exist.
candidate_pairs = [
    ("dyadic2", "dyadic2"),
    ("pot", "pot"),
    ("exact_zero", "exact_zero"),
    ("approx_zero", "approx_zero"),
    ("fallback", "fallback")
]

old_cols = set(old.columns)
new_cols = set(new_full.columns)

if "file" in old_cols and "layer" in old_cols:
    keys = ["file", "layer"]
elif "doc" in old_cols and "seq_len" in old_cols and "layer" in old_cols:
    keys = ["doc", "seq_len", "layer"]
else:
    raise RuntimeError(
        "Could not identify matching document/layer keys."
    )

if not all(k in new_cols for k in keys):
    raise RuntimeError("New results do not have matching keys.")

merged = old.merge(
    new_full,
    on=keys,
    suffixes=("_old", "_new"),
    how="inner",
    validate="one_to_one"
)

print("\nMatched layer records:", len(merged))

comparison = []

for old_col, new_col in candidate_pairs:
    if old_col not in old_cols or new_col not in new_cols:
        continue

    a = merged[old_col + "_old"].to_numpy()
    b = merged[new_col + "_new"].to_numpy()

    comparison.append({
        "mode": old_col,
        "old_total": int(a.sum()),
        "new_total": int(b.sum()),
        "difference": int(b.sum() - a.sum()),
        "mismatched_layer_records": int(np.sum(a != b))
    })

comparison = pd.DataFrame(comparison)

display(comparison)

if len(comparison):
    print("\nDetailed mismatch rows:")
    for _, row in comparison.iterrows():
        mode = row["mode"]
        old_col = mode + "_old"
        new_col = mode + "_new"

        bad = merged[
            merged[old_col] != merged[new_col]
        ]

        if len(bad):
            print(f"\nMode: {mode}")
            display(
                bad[keys + [old_col, new_col]].head(20)
            )


Old rows: 480
New full-adaptive rows: 480

Old columns:
['file', 'doc_id', 'seq_len', 'layer', 'chunks', 'heads', 'state_dim', 'frac_bits', 'policy', 'trajectory_mae', 'trajectory_rmse', 'trajectory_max', 'final_mae', 'final_rmse', 'final_max', 'max_integer_state', 'overflow_count', 'total_updates', 'strict_shift_count', 'nonfallback_count', 'fallback_count', 'strict_shift_pct', 'coverage_pct', 'fallback_pct', 'max_local_bound', 'count_dyadic2', 'count_pot', 'count_exact_zero', 'count_approximate_zero', 'count_fallback_threshold_dyadic', 'count_fallback_threshold_pot', 'count_fallback_below_pot_min', 'count_fallback_approx_zero_rejected', 'count_fallback_above_grid', 'count_fallback_invalid']

New columns:
['file', 'doc', 'seq_len', 'layer', 'policy', 'total_updates', 'dyadic2', 'pot', 'exact_zero', 'approx_zero', 'fallback', 'strict_shift_updates', 'zero_updates', 'nonfallback_updates', 'strict_shift_pct', 'coverage_pct', 'fallback_pct', 'coefficient_mae', 'coefficient_max_error', 'ma

""


In [195]:

import os
import pandas as pd

BASE = "/kaggle/working/shiftscan_final/state_passing/multisequence_validation"

old_path = os.path.join(
    BASE, "approximate_zero_validation_v1", "baseline_per_layer.csv"
)

new_path = os.path.join(
    BASE, "corrected_independent_ablation_v2", "per_layer_results.csv"
)

old = pd.read_csv(old_path)
new = pd.read_csv(new_path)

print("OLD CSV COLUMNS:")
for i, col in enumerate(old.columns):
    print(i, repr(col))

print("\nNEW CSV COLUMNS:")
for i, col in enumerate(new.columns):
    print(i, repr(col))

print("\nOLD COLUMNS CONTAINING 'fallback':")
print([c for c in old.columns if "fallback" in c.lower()])

print("\nOLD COLUMNS CONTAINING 'count':")
print([c for c in old.columns if "count" in c.lower()])

print("\nOLD FIRST ROW:")
display(old.head(1).T)

print("\nNEW FIRST ROW:")
display(new[new["policy"] == "F_FullAdaptive"].head(1).T)


OLD CSV COLUMNS:
0 'file'
1 'doc_id'
2 'seq_len'
3 'layer'
4 'chunks'
5 'heads'
6 'state_dim'
7 'frac_bits'
8 'policy'
9 'trajectory_mae'
10 'trajectory_rmse'
11 'trajectory_max'
12 'final_mae'
13 'final_rmse'
14 'final_max'
15 'max_integer_state'
16 'overflow_count'
17 'total_updates'
18 'strict_shift_count'
19 'nonfallback_count'
20 'fallback_count'
21 'strict_shift_pct'
22 'coverage_pct'
23 'fallback_pct'
24 'max_local_bound'
25 'count_dyadic2'
26 'count_pot'
27 'count_exact_zero'
28 'count_approximate_zero'
29 'count_fallback_threshold_dyadic'
30 'count_fallback_threshold_pot'
31 'count_fallback_below_pot_min'
32 'count_fallback_approx_zero_rejected'
33 'count_fallback_above_grid'
34 'count_fallback_invalid'

NEW CSV COLUMNS:
0 'file'
1 'doc'
2 'seq_len'
3 'layer'
4 'policy'
5 'total_updates'
6 'dyadic2'
7 'pot'
8 'exact_zero'
9 'approx_zero'
10 'fallback'
11 'strict_shift_updates'
12 'zero_updates'
13 'nonfallback_updates'
14 'strict_shift_pct'
15 'coverage_pct'
16 'fallback_pct'


,0
file,doc_000_seq_1024.pt
doc_id,0
seq_len,1024
layer,0
chunks,4
heads,24
state_dim,8192
frac_bits,20
policy,baseline
trajectory_mae,0.000004



NEW FIRST ROW:


,5
file,doc_000_seq_1024.pt
doc,0
seq_len,1024
layer,0
policy,F_FullAdaptive
total_updates,96
dyadic2,0
pot,63
exact_zero,0
approx_zero,4


In [196]:

import os
import numpy as np
import pandas as pd

BASE = "/kaggle/working/shiftscan_final/state_passing/multisequence_validation"

OLD = os.path.join(
    BASE, "approximate_zero_validation_v1", "baseline_per_layer.csv"
)
NEW = os.path.join(
    BASE, "corrected_independent_ablation_v2", "per_layer_results.csv"
)

old = pd.read_csv(OLD)
new = pd.read_csv(NEW)
new = new[new["policy"] == "F_FullAdaptive"].copy()

# Match records by capture file, sequence length and layer.
keys = ["file", "seq_len", "layer"]

old = old.set_index(keys).sort_index()
new = new.set_index(keys).sort_index()

assert old.index.equals(new.index), (
    "The old and new files do not contain identical layer records."
)

checks = {
    "Dyadic2": (
        old["count_dyadic2"].to_numpy(),
        new["dyadic2"].to_numpy()
    ),
    "Direct PoT": (
        old["count_pot"].to_numpy(),
        new["pot"].to_numpy()
    ),
    "Exact zero": (
        old["count_exact_zero"].to_numpy(),
        new["exact_zero"].to_numpy()
    ),
    "Total updates": (
        old["total_updates"].to_numpy(),
        new["total_updates"].to_numpy()
    ),
    "Strict shift updates": (
        old["strict_shift_count"].to_numpy(),
        new["strict_shift_updates"].to_numpy()
    ),
    "Nonfallback including new approximate-zero": (
        old["nonfallback_count"].to_numpy()
        + new["approx_zero"].to_numpy(),
        new["nonfallback_updates"].to_numpy()
    ),
    "Fallback including new approximate-zero": (
        old["fallback_count"].to_numpy(),
        new["fallback"].to_numpy()
        + new["approx_zero"].to_numpy()
    )
}

rows = []

for name, (a, b) in checks.items():
    a = np.asarray(a, dtype=np.int64)
    b = np.asarray(b, dtype=np.int64)

    rows.append({
        "check": name,
        "old_or_expected_total": int(a.sum()),
        "new_total": int(b.sum()),
        "total_difference": int(b.sum() - a.sum()),
        "mismatched_records": int(np.count_nonzero(a != b)),
        "exact_match": bool(np.array_equal(a, b))
    })

audit = pd.DataFrame(rows)

print("========== CONSISTENCY AUDIT ==========")
display(audit)

print(
    "\nALL CHECKS PASS:",
    bool(audit["exact_match"].all())
)

# Save audit
OUT = os.path.join(
    BASE, "corrected_independent_ablation_v2", "consistency_audit"
)
os.makedirs(OUT, exist_ok=True)

audit.to_csv(
    os.path.join(OUT, "mode_count_consistency_audit.csv"),
    index=False
)

print("\nSaved to:", OUT)


========== CONSISTENCY AUDIT ==========


,check,old_or_expected_total,new_total,total_difference,mismatched_records,exact_match
0,Dyadic2,4434,4433,-1,1,False
1,Direct PoT,9796,9796,0,0,True
2,Exact zero,22080,22080,0,0,True
3,Total updates,69120,69120,0,0,True
4,Strict shift updates,14230,14229,-1,1,False
5,Nonfallback including new approximate-zero,53352,53351,-1,1,False
6,Fallback including new approximate-zero,32810,32811,1,1,False



ALL CHECKS PASS: False

Saved to: /kaggle/working/shiftscan_final/state_passing/multisequence_validation/corrected_independent_ablation_v2/consistency_audit


In [197]:

import os
import pandas as pd
import numpy as np

BASE = "/kaggle/working/shiftscan_final/state_passing/multisequence_validation"

old_path = os.path.join(
    BASE, "approximate_zero_validation_v1", "baseline_per_layer.csv"
)
new_path = os.path.join(
    BASE, "corrected_independent_ablation_v2", "per_layer_results.csv"
)

old = pd.read_csv(old_path)
new = pd.read_csv(new_path)

new = new[new["policy"] == "F_FullAdaptive"].copy()

keys = ["file", "seq_len", "layer"]

merged = old.merge(
    new,
    left_on=keys,
    right_on=keys,
    suffixes=("_old", "_new"),
    validate="one_to_one"
)

mismatch = merged[
    merged["count_dyadic2"] != merged["dyadic2"]
].copy()

print("Number of mismatching layer records:", len(mismatch))

if len(mismatch):
    cols = [
        "file", "seq_len", "layer",
        "count_dyadic2", "dyadic2",
        "count_pot", "pot",
        "count_exact_zero", "exact_zero",
        "count_approximate_zero", "approx_zero",
        "fallback_count", "fallback",
        "total_updates_old", "total_updates_new",
        "max_local_bound_old", "max_local_bound_new"
    ]

    available = [c for c in cols if c in mismatch.columns]
    display(mismatch[available].T)

    print("\nFull old record:")
    display(old[
        (old["file"] == mismatch.iloc[0]["file"]) &
        (old["seq_len"] == mismatch.iloc[0]["seq_len"]) &
        (old["layer"] == mismatch.iloc[0]["layer"])
    ].T)

    print("\nFull new record:")
    display(new[
        (new["file"] == mismatch.iloc[0]["file"]) &
        (new["seq_len"] == mismatch.iloc[0]["seq_len"]) &
        (new["layer"] == mismatch.iloc[0]["layer"])
    ].T)
else:
    print("No Dyadic2 mismatches found.")


Number of mismatching layer records: 1


,139
file,doc_002_seq_2048.pt
seq_len,2048
layer,19
count_dyadic2,39
dyadic2,38
count_pot,8
pot,8
count_exact_zero,85
exact_zero,85
count_approximate_zero,0



Full old record:


,139
file,doc_002_seq_2048.pt
doc_id,2
seq_len,2048
layer,19
chunks,8
heads,24
state_dim,8192
frac_bits,20
policy,baseline
trajectory_mae,0.000001



Full new record:


,839
file,doc_002_seq_2048.pt
doc,2
seq_len,2048
layer,19
policy,F_FullAdaptive
total_updates,192
dyadic2,38
pot,8
exact_zero,85
approx_zero,3


In [198]:

import os, gc, shutil, torch, pandas as pd

BASE = "/kaggle/working/shiftscan_final"
CAPTURE_DIR = os.path.join(
    BASE, "state_passing", "long_context_validation"
)
os.makedirs(CAPTURE_DIR, exist_ok=True)

print("========== SHIFT-SCAN READINESS ==========")

print("\nGPU")
assert torch.cuda.is_available(), "CUDA is unavailable"
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    p = torch.cuda.get_device_properties(i)
    print(
        f"GPU {i}: {p.name} | "
        f"free={free/1024**3:.2f} GB | "
        f"total={total/1024**3:.2f} GB"
    )

print("\nDISK")
u = shutil.disk_usage("/kaggle/working")
print(f"Free: {u.free/1024**3:.2f} GB")

print("\nREQUIRED OBJECTS")
for name in ["model", "tokenizer", "torch", "triton"]:
    print(name, "AVAILABLE" if name in globals() else "MISSING")

assert "model" in globals(), "Load the existing Mamba-2 model first"
assert "tokenizer" in globals(), "Load the GPT-NeoX tokenizer first"

print("\nMODEL")
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Device:", next(model.parameters()).device)
print("Dtype:", next(model.parameters()).dtype)
print("Layers:", len(model.backbone.layers))

print("\nCAPTURE MODULES")
for name in ["ssd_combined", "mamba2_mod"]:
    print(name, "AVAILABLE" if name in globals() else "MISSING")

if "ssd_combined" in globals():
    print(
        "_state_passing_fwd:",
        hasattr(ssd_combined, "_state_passing_fwd")
    )

print("\nEXISTING LONG-CONTEXT FILES")
print(os.listdir(CAPTURE_DIR))

print("\nREADINESS CHECK COMPLETE")


========== SHIFT-SCAN READINESS ==========

GPU
GPU 0: Tesla T4 | free=12.61 GB | total=14.56 GB
GPU 1: Tesla T4 | free=14.46 GB | total=14.56 GB

DISK
Free: 16.88 GB

REQUIRED OBJECTS
model AVAILABLE
tokenizer AVAILABLE
torch AVAILABLE
triton AVAILABLE

MODEL
Parameters: 128989632
Device: cuda:0
Dtype: torch.float16
Layers: 24

CAPTURE MODULES
ssd_combined AVAILABLE
mamba2_mod AVAILABLE
_state_passing_fwd: True

EXISTING LONG-CONTEXT FILES
[]

READINESS CHECK COMPLETE


In [199]:

import os, gc, json, torch, pandas as pd
from datasets import load_dataset

OUT = "/kaggle/working/shiftscan_final/state_passing/long_context_validation"
os.makedirs(OUT, exist_ok=True)

SEQ_LEN = 4096
N_DOCS = 2
STREAM_SKIP = 610

# Ensure the tested Mamba-2 Triton path is active.
for layer in model.backbone.layers:
    if hasattr(layer.mixer, "use_mem_eff_path"):
        layer.mixer.use_mem_eff_path = False

# Find the actual state-passing function used by the SSD forward path.
scan_globals = ssd_combined._mamba_chunk_scan_combined_fwd.__globals__
assert "_state_passing_fwd" in scan_globals

original_state_passing = scan_globals["_state_passing_fwd"]

captured = []
capture_layer = {"index": 0}

def capture_state_passing(states, dA_chunk_cumsum, *args, **kwargs):
    idx = capture_layer["index"]
    capture_layer["index"] += 1

    captured.append({
        "layer": idx,
        "states": states.detach().cpu().contiguous(),
        "decay_log": dA_chunk_cumsum.detach().cpu().contiguous(),
        "chunk_size": kwargs.get("chunk_size", 256),
        "has_initial_states": kwargs.get("initial_states") is not None,
        "has_seq_idx": kwargs.get("seq_idx") is not None
    })

    return original_state_passing(
        states, dA_chunk_cumsum, *args, **kwargs
    )

# Load a fresh FineWeb-Edu stream, beyond the previously sampled documents.
dataset = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="sample-10BT",
    split="train",
    streaming=True
)

saved = []
seen = 0
doc_id = 0

scan_globals["_state_passing_fwd"] = capture_state_passing

try:
    for item in dataset:
        if seen < STREAM_SKIP:
            if len(item.get("text", "").strip()) > 100:
                seen += 1
            continue

        text = item.get("text", "").strip()
        if len(text) < 100:
            continue

        encoded = tokenizer(
            text,
            truncation=True,
            max_length=SEQ_LEN,
            return_tensors="pt"
        )

        if encoded.input_ids.shape[1] != SEQ_LEN:
            continue

        input_ids = encoded.input_ids.to("cuda:0")

        captured.clear()
        capture_layer["index"] = 0

        torch.cuda.empty_cache()
        with torch.inference_mode():
            output = model(input_ids)

        torch.cuda.synchronize()

        assert len(captured) == 24, (
            f"Expected 24 layer records, got {len(captured)}"
        )

        file_name = f"doc_{doc_id:03d}_seq_{SEQ_LEN}.pt"
        file_path = os.path.join(OUT, file_name)

        torch.save({
            "doc_id": doc_id,
            "seq_len": SEQ_LEN,
            "input_ids": input_ids.cpu(),
            "records": captured
        }, file_path)

        for rec in captured:
            assert rec["states"].shape[1] == SEQ_LEN // 256
            assert rec["decay_log"].shape[-1] == SEQ_LEN // 256

        saved.append({
            "file": file_name,
            "doc_id": doc_id,
            "seq_len": SEQ_LEN,
            "layers": len(captured),
            "chunks_per_layer": SEQ_LEN // 256,
            "state_shape": str(tuple(captured[0]["states"].shape)),
            "decay_shape": str(tuple(captured[0]["decay_log"].shape)),
            "file_size_mb": os.path.getsize(file_path) / 1024**2
        })

        print("SAVED:", saved[-1])

        doc_id += 1
        del output, input_ids, encoded
        gc.collect()
        torch.cuda.empty_cache()

        if doc_id >= N_DOCS:
            break

finally:
    scan_globals["_state_passing_fwd"] = original_state_passing

assert len(saved) == N_DOCS, (
    f"Only captured {len(saved)} documents"
)

manifest = pd.DataFrame(saved)
manifest.to_csv(
    os.path.join(OUT, "manifest_4096.csv"),
    index=False
)

with open(os.path.join(OUT, "config_4096.json"), "w") as f:
    json.dump({
        "sequence_length": SEQ_LEN,
        "documents": N_DOCS,
        "stream_skip_qualifying_documents": STREAM_SKIP,
        "model": "state-spaces/mamba2-130m",
        "dataset": "HuggingFaceFW/fineweb-edu/sample-10BT",
        "chunk_size": 256,
        "layers": 24
    }, f, indent=2)

print("\n4096-TOKEN CAPTURE COMPLETE")
display(manifest)


Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

SAVED: {'file': 'doc_000_seq_4096.pt', 'doc_id': 0, 'seq_len': 4096, 'layers': 24, 'chunks_per_layer': 16, 'state_shape': '(1, 16, 24, 8192)', 'decay_shape': '(1, 24, 16)', 'file_size_mb': 288.08123683929443}
SAVED: {'file': 'doc_001_seq_4096.pt', 'doc_id': 1, 'seq_len': 4096, 'layers': 24, 'chunks_per_layer': 16, 'state_shape': '(1, 16, 24, 8192)', 'decay_shape': '(1, 24, 16)', 'file_size_mb': 288.08123683929443}

4096-TOKEN CAPTURE COMPLETE


,file,doc_id,seq_len,layers,chunks_per_layer,state_shape,decay_shape,file_size_mb
0,doc_000_seq_4096.pt,0,4096,24,16,"(1, 16, 24, 8192)","(1, 24, 16)",288.081237
1,doc_001_seq_4096.pt,1,4096,24,16,"(1, 16, 24, 8192)","(1, 24, 16)",288.081237


In [200]:

import os, gc, torch, pandas as pd, numpy as np

BASE = "/kaggle/working/shiftscan_final/state_passing/long_context_validation"

files = [
    os.path.join(BASE, "doc_000_seq_4096.pt"),
    os.path.join(BASE, "doc_001_seq_4096.pt")
]

rows = []

for path in files:
    data = torch.load(path, map_location="cpu", weights_only=False)

    for rec in data["records"]:
        states = rec["states"].float()
        logd = rec["decay_log"].float()
        decay = torch.exp(logd)

        rows.append({
            "file": os.path.basename(path),
            "layer": rec["layer"],
            "chunks": states.shape[1],
            "heads": states.shape[2],
            "state_dim": states.shape[3],
            "decay_min": decay.min().item(),
            "decay_max": decay.max().item(),
            "decay_mean": decay.mean().item(),
            "zero_decay_pct": 100 * (decay == 0).float().mean().item(),
            "state_absmax": states.abs().max().item()
        })

    del data
    gc.collect()

audit = pd.DataFrame(rows)
audit_path = os.path.join(BASE, "capture_audit_4096.csv")
audit.to_csv(audit_path, index=False)

print("CAPTURE AUDIT")
print("Layer records:", len(audit))
print("All chunks correct:", bool((audit.chunks == 16).all()))
print("All heads correct:", bool((audit.heads == 24).all()))
print("All state dimensions correct:", bool((audit.state_dim == 8192).all()))
display(audit.groupby("file").agg({
    "decay_min": "min",
    "decay_max": "max",
    "decay_mean": "mean",
    "zero_decay_pct": "mean",
    "state_absmax": "max"
}).reset_index())

print("\nKERNEL / HELPER OBJECTS STILL IN NOTEBOOK")
for name, obj in list(globals().items()):
    low = name.lower()
    if any(k in low for k in ["shift", "round_away", "build_reference", "grid_exponent"]):
        if callable(obj) or isinstance(obj, (int, float, np.ndarray, list, tuple)):
            print(name, type(obj).__name__)

print("\nEXISTING VALIDATION OUTPUTS")
for folder in [
    "int32_kernel_validation_v1",
    "triton_validation_int32_v1",
    "fixed_point_controller_validation_v2"
]:
    path = os.path.join(
        "/kaggle/working/shiftscan_final/state_passing/multisequence_validation",
        folder
    )
    print(folder, "FOUND" if os.path.exists(path) else "NOT FOUND")

print("\nFREE DISK GB:", round(
    __import__("shutil").disk_usage("/kaggle/working").free / 1024**3, 2
))


CAPTURE AUDIT
Layer records: 48
All chunks correct: True
All heads correct: True
All state dimensions correct: True


,file,decay_min,decay_max,decay_mean,zero_decay_pct,state_absmax
0,doc_000_seq_4096.pt,0.0,0.976634,0.149656,30.208334,368.813232
1,doc_001_seq_4096.pt,0.0,0.969793,0.141911,32.161458,393.908325



KERNEL / HELPER OBJECTS STILL IN NOTEBOOK
round_shift_signed function
shift_subtract_decay function
h_shift ndarray
shift_decay function
round_shift function
h_shift_new ndarray
shift_subtract_kernel JITFunction
denominator_shift_np ndarray
common_shift int
shift_np ndarray
run_shift function
shift_peak_memory int
grid_exponents list
round_away_float_to_int function
round_away_float function
apply_dyadic_shift function
round_away_from_zero function
shift_round_signed function
strict_shift_count int
round_away function
shift_round function
round_away_np function
round_shift_np function
_shift_scan_round_shift JITFunction
shift_scan_state_passing_kernel JITFunction
build_reference_and_modes function
shift_path_microkernel JITFunction
run_shift_only function
shift_ms float
strict_shift int
shiftscan_round_shift_i32 JITFunction
shift_scan_state_passing_i32_kernel JITFunction
shiftscan_fp32_fallback_kernel JITFunction
shiftscan_integer_fallback_kernel JITFunction
shiftscan_fallback_benchma

In [201]:

import inspect

print("REFERENCE BUILDER")
print(inspect.signature(build_reference_and_modes))

print("\nTRITON KERNEL")
print(inspect.signature(shift_scan_state_passing_i32_kernel))

print("\nKERNEL SOURCE")
print(shift_scan_state_passing_i32_kernel.src[:12000])

print("\nREFERENCE HELPER SOURCE")
print(inspect.getsource(build_reference_and_modes))


REFERENCE BUILDER
(rec)

TRITON KERNEL
(*args, **kwargs)

KERNEL SOURCE
def shift_scan_state_passing_i32_kernel(
    N_ptr,
    MODE_ptr,
    EXP_ptr,
    DECAY_ptr,
    OUT_ptr,
    FINAL_ptr,
    OVERFLOW_ptr,

    NCHUNKS: tl.constexpr,
    NHEADS: tl.constexpr,
    DIM: tl.constexpr,
    NBLOCKS: tl.constexpr,

    BLOCK_SIZE: tl.constexpr
):
    pid_m = tl.program_id(0)
    pid_b = tl.program_id(1)
    pid_h = tl.program_id(2)

    offs = pid_m * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offs < DIM

    state = tl.zeros((BLOCK_SIZE,), dtype=tl.int32)
    block_overflow = tl.full((), 0, dtype=tl.int32)

    ctrl_base = (
        pid_b * NHEADS * NCHUNKS
        + pid_h * NCHUNKS
    )

    for c in range(NCHUNKS):

        mode = tl.load(MODE_ptr + ctrl_base + c)
        exp_base = ctrl_base * 3 + c * 3

        # Mode 1: three-term Dyadic2.
        if mode == 1:
            decay_sum = tl.zeros(
                (BLOCK_SIZE,), dtype=tl.int32
            )

            for j 

In [202]:

import os, gc, json, torch, numpy as np, pandas as pd, triton

BASE = "/kaggle/working/shiftscan_final/state_passing/long_context_validation"
OUT_DIR = os.path.join(BASE, "f20_validation")
os.makedirs(OUT_DIR, exist_ok=True)

assert SCALE == 2**20, f"Expected F20, got SCALE={SCALE}"

BLOCK_SIZE = 128
results = []

for file_name in ["doc_000_seq_4096.pt", "doc_001_seq_4096.pt"]:
    path = os.path.join(BASE, file_name)
    data = torch.load(path, map_location="cpu", weights_only=False)

    print(f"\nValidating {file_name}")

    for rec in data["records"]:
        layer = rec["layer"]

        (N_int, decay, modes, exps,
         reference, counts, max_bound) = build_reference_and_modes(rec)

        chunks, heads, dim = N_int.shape
        nblocks = triton.cdiv(dim, BLOCK_SIZE)

        # Ensure the input contributions fit int32.
        assert N_int.min() >= -2147483648
        assert N_int.max() <= 2147483647

        N_gpu = torch.as_tensor(
            N_int.astype(np.int32),
            device="cuda:0"
        ).contiguous()

        MODE_gpu = torch.as_tensor(
            modes.astype(np.int32),
            device="cuda:0"
        ).contiguous()

        EXP_gpu = torch.as_tensor(
            exps.astype(np.int32),
            device="cuda:0"
        ).contiguous()

        DECAY_gpu = torch.as_tensor(
            decay.astype(np.float64),
            device="cuda:0"
        ).contiguous()

        OUT_gpu = torch.empty(
            (1, chunks, heads, dim),
            device="cuda:0",
            dtype=torch.int32
        )

        FINAL_gpu = torch.empty(
            (1, heads, dim),
            device="cuda:0",
            dtype=torch.int32
        )

        OVERFLOW_gpu = torch.zeros(
            (1, heads, nblocks),
            device="cuda:0",
            dtype=torch.int32
        )

        shift_scan_state_passing_i32_kernel[
            (nblocks, 1, heads)
        ](
            N_gpu,
            MODE_gpu,
            EXP_gpu,
            DECAY_gpu,
            OUT_gpu,
            FINAL_gpu,
            OVERFLOW_gpu,
            NCHUNKS=chunks,
            NHEADS=heads,
            DIM=dim,
            NBLOCKS=nblocks,
            BLOCK_SIZE=BLOCK_SIZE
        )

        torch.cuda.synchronize()

        actual_intermediate = OUT_gpu[0].cpu().numpy().astype(np.int64)
        actual_final = FINAL_gpu[0].cpu().numpy().astype(np.int64)

        intermediate_diff = np.abs(
            actual_intermediate - reference
        )
        final_reference = reference[-1]
        final_diff = np.abs(actual_final - final_reference)

        overflow_count = int(OVERFLOW_gpu.sum().item())

        row = {
            "file": file_name,
            "layer": layer,
            "chunks": chunks,
            "heads": heads,
            "state_dim": dim,
            "dyadic2": counts["dyadic2"],
            "pot": counts["pot"],
            "exact_zero": counts["exact_zero"],
            "approx_zero": counts["approx_zero"],
            "fallback": counts["fallback"],
            "coverage_pct": 100.0 * (
                chunks * heads - counts["fallback"]
            ) / (chunks * heads),
            "max_local_bound": max_bound,
            "intermediate_mismatches": int(
                np.count_nonzero(intermediate_diff)
            ),
            "final_mismatches": int(
                np.count_nonzero(final_diff)
            ),
            "max_integer_difference": int(
                max(intermediate_diff.max(), final_diff.max())
            ),
            "max_integer_state": int(
                max(
                    np.abs(actual_intermediate).max(),
                    np.abs(actual_final).max()
                )
            ),
            "overflow_count": overflow_count
        }

        results.append(row)

        del N_gpu, MODE_gpu, EXP_gpu, DECAY_gpu
        del OUT_gpu, FINAL_gpu, OVERFLOW_gpu
        gc.collect()

    del data
    gc.collect()
    torch.cuda.empty_cache()

    print("Completed:", file_name)

df = pd.DataFrame(results)
csv_path = os.path.join(OUT_DIR, "per_layer_results_4096.csv")
df.to_csv(csv_path, index=False)

summary = {
    "files": int(df.file.nunique()),
    "layer_records": len(df),
    "intermediate_mismatches": int(df.intermediate_mismatches.sum()),
    "final_mismatches": int(df.final_mismatches.sum()),
    "max_integer_difference": int(df.max_integer_difference.max()),
    "max_integer_state": int(df.max_integer_state.max()),
    "overflow_count": int(df.overflow_count.sum()),
    "mean_coverage_pct": float(df.coverage_pct.mean()),
    "pooled_coverage_pct": float(
        100 * (df.dyadic2.sum() + df.pot.sum()
               + df.exact_zero.sum() + df.approx_zero.sum())
        / (df.chunks.sum() * df.heads.sum())
    ),
    "mode_counts": {
        "dyadic2": int(df.dyadic2.sum()),
        "pot": int(df.pot.sum()),
        "exact_zero": int(df.exact_zero.sum()),
        "approx_zero": int(df.approx_zero.sum()),
        "fallback": int(df.fallback.sum())
    }
}

with open(os.path.join(OUT_DIR, "summary_4096.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("\n========== 4096 F20 VALIDATION ==========")
print(json.dumps(summary, indent=2))
print("\nPer-document summary:")
display(df.groupby("file").agg({
    "coverage_pct": "mean",
    "intermediate_mismatches": "sum",
    "final_mismatches": "sum",
    "max_integer_difference": "max",
    "max_integer_state": "max",
    "overflow_count": "sum"
}).reset_index())



Validating doc_000_seq_4096.pt
Completed: doc_000_seq_4096.pt

Validating doc_001_seq_4096.pt
Completed: doc_001_seq_4096.pt

========== 4096 F20 VALIDATION ==========
{
  "files": 2,
  "layer_records": 48,
  "intermediate_mismatches": 0,
  "final_mismatches": 0,
  "max_integer_difference": 0,
  "max_integer_state": 416063839,
  "overflow_count": 0,
  "mean_coverage_pct": 73.87152777777779,
  "pooled_coverage_pct": 1.538990162037037,
  "mode_counts": {
    "dyadic2": 890,
    "pot": 2348,
    "exact_zero": 5748,
    "approx_zero": 4630,
    "fallback": 4816
  }
}

Per-document summary:


,file,coverage_pct,intermediate_mismatches,final_mismatches,max_integer_difference,max_integer_state,overflow_count
0,doc_000_seq_4096.pt,73.697917,0,0,0,386728704,0
1,doc_001_seq_4096.pt,74.045139,0,0,0,416063839,0


In [203]:

import os, gc, json, torch, pandas as pd
from datasets import load_dataset

OUT = "/kaggle/working/shiftscan_final/state_passing/long_context_validation"
os.makedirs(OUT, exist_ok=True)

SEQ_LEN = 8192
N_DOCS = 2
STREAM_SKIP = 612

for layer in model.backbone.layers:
    if hasattr(layer.mixer, "use_mem_eff_path"):
        layer.mixer.use_mem_eff_path = False

scan_globals = ssd_combined._mamba_chunk_scan_combined_fwd.__globals__
original_state_passing = scan_globals["_state_passing_fwd"]

captured = []
capture_layer = {"index": 0}

def capture_state_passing(states, dA_chunk_cumsum, *args, **kwargs):
    captured.append({
        "layer": capture_layer["index"],
        "states": states.detach().cpu().contiguous(),
        "decay_log": dA_chunk_cumsum.detach().cpu().contiguous(),
        "chunk_size": kwargs.get("chunk_size", 256),
        "has_initial_states": kwargs.get("initial_states") is not None,
        "has_seq_idx": kwargs.get("seq_idx") is not None
    })
    capture_layer["index"] += 1
    return original_state_passing(states, dA_chunk_cumsum, *args, **kwargs)

dataset = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="sample-10BT",
    split="train",
    streaming=True
)

saved = []
seen = 0
doc_id = 0

scan_globals["_state_passing_fwd"] = capture_state_passing

try:
    for item in dataset:
        text = item.get("text", "").strip()

        if len(text) <= 100:
            continue

        if seen < STREAM_SKIP:
            seen += 1
            continue

        encoded = tokenizer(
            text,
            truncation=True,
            max_length=SEQ_LEN,
            return_tensors="pt"
        )

        if encoded.input_ids.shape[1] != SEQ_LEN:
            continue

        input_ids = encoded.input_ids.to("cuda:0")

        captured.clear()
        capture_layer["index"] = 0
        torch.cuda.empty_cache()

        with torch.inference_mode():
            output = model(input_ids)

        torch.cuda.synchronize()

        assert len(captured) == 24, (
            f"Expected 24 layers, got {len(captured)}"
        )

        for rec in captured:
            assert rec["states"].shape == (1, 32, 24, 8192)
            assert rec["decay_log"].shape == (1, 24, 32)

        file_name = f"doc_{doc_id:03d}_seq_8192.pt"
        file_path = os.path.join(OUT, file_name)

        torch.save({
            "doc_id": doc_id,
            "seq_len": SEQ_LEN,
            "input_ids": input_ids.cpu(),
            "records": captured
        }, file_path)

        row = {
            "file": file_name,
            "doc_id": doc_id,
            "seq_len": SEQ_LEN,
            "layers": len(captured),
            "chunks_per_layer": 32,
            "state_shape": str(tuple(captured[0]["states"].shape)),
            "decay_shape": str(tuple(captured[0]["decay_log"].shape)),
            "file_size_mb": os.path.getsize(file_path) / 1024**2
        }
        saved.append(row)
        print("SAVED:", row)

        doc_id += 1
        del output, input_ids, encoded
        gc.collect()
        torch.cuda.empty_cache()

        if doc_id >= N_DOCS:
            break

finally:
    scan_globals["_state_passing_fwd"] = original_state_passing

assert len(saved) == N_DOCS, f"Captured only {len(saved)} documents"

manifest = pd.DataFrame(saved)
manifest.to_csv(os.path.join(OUT, "manifest_8192.csv"), index=False)

with open(os.path.join(OUT, "config_8192.json"), "w") as f:
    json.dump({
        "sequence_length": SEQ_LEN,
        "documents": N_DOCS,
        "stream_skip_qualifying_documents": STREAM_SKIP,
        "model": "state-spaces/mamba2-130m",
        "dataset": "HuggingFaceFW/fineweb-edu/sample-10BT",
        "chunk_size": 256,
        "layers": 24
    }, f, indent=2)

print("\n8192-TOKEN CAPTURE COMPLETE")
display(manifest)


Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

SAVED: {'file': 'doc_000_seq_8192.pt', 'doc_id': 0, 'seq_len': 8192, 'layers': 24, 'chunks_per_layer': 32, 'state_shape': '(1, 32, 24, 8192)', 'decay_shape': '(1, 24, 32)', 'file_size_mb': 576.1476430892944}
SAVED: {'file': 'doc_001_seq_8192.pt', 'doc_id': 1, 'seq_len': 8192, 'layers': 24, 'chunks_per_layer': 32, 'state_shape': '(1, 32, 24, 8192)', 'decay_shape': '(1, 24, 32)', 'file_size_mb': 576.1476430892944}

8192-TOKEN CAPTURE COMPLETE


,file,doc_id,seq_len,layers,chunks_per_layer,state_shape,decay_shape,file_size_mb
0,doc_000_seq_8192.pt,0,8192,24,32,"(1, 32, 24, 8192)","(1, 24, 32)",576.147643
1,doc_001_seq_8192.pt,1,8192,24,32,"(1, 32, 24, 8192)","(1, 24, 32)",576.147643


In [204]:

import os, gc, json, torch, numpy as np, pandas as pd, triton

BASE = "/kaggle/working/shiftscan_final/state_passing/long_context_validation"
OUT_DIR = os.path.join(BASE, "f20_validation")
os.makedirs(OUT_DIR, exist_ok=True)

assert SCALE == 2**20, f"Expected F20, got SCALE={SCALE}"

BLOCK_SIZE = 128
results = []

for file_name in ["doc_000_seq_8192.pt", "doc_001_seq_8192.pt"]:
    path = os.path.join(BASE, file_name)
    data = torch.load(path, map_location="cpu", weights_only=False)

    print(f"\nValidating {file_name}")

    for rec in data["records"]:
        (N_int, decay, modes, exps,
         reference, counts, max_bound) = build_reference_and_modes(rec)

        chunks, heads, dim = N_int.shape
        nblocks = triton.cdiv(dim, BLOCK_SIZE)

        assert chunks == 32 and heads == 24 and dim == 8192
        assert N_int.min() >= -2147483648
        assert N_int.max() <= 2147483647

        N_gpu = torch.as_tensor(
            N_int.astype(np.int32), device="cuda:0"
        ).contiguous()
        MODE_gpu = torch.as_tensor(
            modes.astype(np.int32), device="cuda:0"
        ).contiguous()
        EXP_gpu = torch.as_tensor(
            exps.astype(np.int32), device="cuda:0"
        ).contiguous()
        DECAY_gpu = torch.as_tensor(
            decay.astype(np.float64), device="cuda:0"
        ).contiguous()

        OUT_gpu = torch.empty(
            (1, chunks, heads, dim),
            device="cuda:0", dtype=torch.int32
        )
        FINAL_gpu = torch.empty(
            (1, heads, dim),
            device="cuda:0", dtype=torch.int32
        )
        OVERFLOW_gpu = torch.zeros(
            (1, heads, nblocks),
            device="cuda:0", dtype=torch.int32
        )

        shift_scan_state_passing_i32_kernel[
            (nblocks, 1, heads)
        ](
            N_gpu, MODE_gpu, EXP_gpu, DECAY_gpu,
            OUT_gpu, FINAL_gpu, OVERFLOW_gpu,
            NCHUNKS=chunks,
            NHEADS=heads,
            DIM=dim,
            NBLOCKS=nblocks,
            BLOCK_SIZE=BLOCK_SIZE
        )

        torch.cuda.synchronize()

        actual_intermediate = (
            OUT_gpu[0].cpu().numpy().astype(np.int64)
        )
        actual_final = (
            FINAL_gpu[0].cpu().numpy().astype(np.int64)
        )

        intermediate_diff = np.abs(actual_intermediate - reference)
        final_diff = np.abs(actual_final - reference[-1])
        overflow_count = int(OVERFLOW_gpu.sum().item())

        results.append({
            "file": file_name,
            "layer": rec["layer"],
            "chunks": chunks,
            "heads": heads,
            "state_dim": dim,
            "dyadic2": counts["dyadic2"],
            "pot": counts["pot"],
            "exact_zero": counts["exact_zero"],
            "approx_zero": counts["approx_zero"],
            "fallback": counts["fallback"],
            "coverage_pct": 100.0 * (
                chunks * heads - counts["fallback"]
            ) / (chunks * heads),
            "max_local_bound": max_bound,
            "intermediate_mismatches": int(
                np.count_nonzero(intermediate_diff)
            ),
            "final_mismatches": int(
                np.count_nonzero(final_diff)
            ),
            "max_integer_difference": int(
                max(intermediate_diff.max(), final_diff.max())
            ),
            "max_integer_state": int(
                max(
                    np.abs(actual_intermediate).max(),
                    np.abs(actual_final).max()
                )
            ),
            "overflow_count": overflow_count
        })

        del N_gpu, MODE_gpu, EXP_gpu, DECAY_gpu
        del OUT_gpu, FINAL_gpu, OVERFLOW_gpu
        del actual_intermediate, actual_final
        del intermediate_diff, final_diff
        del reference, N_int
        gc.collect()

    del data
    gc.collect()
    torch.cuda.empty_cache()
    print("Completed:", file_name)

df = pd.DataFrame(results)
df.to_csv(
    os.path.join(OUT_DIR, "per_layer_results_8192.csv"),
    index=False
)

total_updates = int((df.chunks * df.heads).sum())
nonfallback = int(
    df.dyadic2.sum() + df.pot.sum()
    + df.exact_zero.sum() + df.approx_zero.sum()
)

summary = {
    "files": int(df.file.nunique()),
    "layer_records": len(df),
    "total_updates": total_updates,
    "intermediate_mismatches": int(df.intermediate_mismatches.sum()),
    "final_mismatches": int(df.final_mismatches.sum()),
    "max_integer_difference": int(df.max_integer_difference.max()),
    "max_integer_state": int(df.max_integer_state.max()),
    "overflow_count": int(df.overflow_count.sum()),
    "mean_layer_coverage_pct": float(df.coverage_pct.mean()),
    "pooled_coverage_pct": 100.0 * nonfallback / total_updates,
    "mode_counts": {
        "dyadic2": int(df.dyadic2.sum()),
        "pot": int(df.pot.sum()),
        "exact_zero": int(df.exact_zero.sum()),
        "approx_zero": int(df.approx_zero.sum()),
        "fallback": int(df.fallback.sum())
    }
}

with open(os.path.join(OUT_DIR, "summary_8192.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("\n========== 8192 F20 VALIDATION ==========")
print(json.dumps(summary, indent=2))

print("\nPer-document summary:")
display(df.groupby("file").agg({
    "coverage_pct": "mean",
    "intermediate_mismatches": "sum",
    "final_mismatches": "sum",
    "max_integer_difference": "max",
    "max_integer_state": "max",
    "overflow_count": "sum"
}).reset_index())



Validating doc_000_seq_8192.pt
Completed: doc_000_seq_8192.pt

Validating doc_001_seq_8192.pt
Completed: doc_001_seq_8192.pt

========== 8192 F20 VALIDATION ==========
{
  "files": 2,
  "layer_records": 48,
  "total_updates": 36864,
  "intermediate_mismatches": 0,
  "final_mismatches": 0,
  "max_integer_difference": 0,
  "max_integer_state": 569384607,
  "overflow_count": 0,
  "mean_layer_coverage_pct": 73.3642578125,
  "pooled_coverage_pct": 73.3642578125,
  "mode_counts": {
    "dyadic2": 1334,
    "pot": 4785,
    "exact_zero": 11595,
    "approx_zero": 9331,
    "fallback": 9819
  }
}

Per-document summary:


,file,coverage_pct,intermediate_mismatches,final_mismatches,max_integer_difference,max_integer_state,overflow_count
0,doc_000_seq_8192.pt,73.459201,0,0,0,458672521,0
1,doc_001_seq_8192.pt,73.269314,0,0,0,569384607,0


In [205]:

import os, json
import numpy as np
import pandas as pd

ROOT = "/kaggle/working/shiftscan_final/state_passing"
LONG = os.path.join(ROOT, "long_context_validation")

# Original 1024/2048 fixed-point-driven controller results
short_path = os.path.join(
    ROOT,
    "multisequence_validation",
    "fixed_point_controller_validation_v2",
    "per_layer_controller_results.csv"
)

short = pd.read_csv(short_path)
short = short[short["frac_bits"] == 20].copy()

# Newly completed long-context results
long_paths = {
    4096: os.path.join(
        LONG, "f20_validation", "per_layer_results_4096.csv"
    ),
    8192: os.path.join(
        LONG, "f20_validation", "per_layer_results_8192.csv"
    )
}

long_frames = {}
for length, path in long_paths.items():
    assert os.path.exists(path), f"Missing: {path}"
    long_frames[length] = pd.read_csv(path)

def summarize_group(g, seq_len):
    g = g.copy()

    if "total_updates" in g.columns:
        updates = g["total_updates"].astype(int)
    else:
        updates = g["chunks"].astype(int) * g["heads"].astype(int)

    total = int(updates.sum())

    result = {
        "seq_len": int(seq_len),
        "layer_records": len(g),
        "total_updates": total,
    }

    # Arithmetic modes, when recorded
    for col in ["dyadic2", "pot", "exact_zero", "approx_zero", "fallback"]:
        if col in g.columns:
            result[col + "_updates"] = int(g[col].sum())

    if "coverage_pct" in g.columns:
        if "fallback" in g.columns:
            fallback = int(g["fallback"].sum())
            result["pooled_coverage_pct"] = 100 * (total - fallback) / total
        else:
            result["pooled_coverage_pct"] = (
                100 * (total - int(g["fallback_count"].sum())) / total
                if "fallback_count" in g.columns
                else np.nan
            )

    # Numerical validation fields
    for col in [
        "intermediate_mismatches",
        "final_mismatches",
        "overflow_count"
    ]:
        if col in g.columns:
            result[col] = int(g[col].sum())

    if "max_integer_difference" in g.columns:
        result["max_integer_difference"] = int(
            g["max_integer_difference"].max()
        )

    if "max_integer_state" in g.columns:
        result["max_integer_state"] = int(
            g["max_integer_state"].max()
        )

    # Recurrent-state errors available for the 1024/2048 controller runs
    for col in [
        "trajectory_mae",
        "trajectory_rmse",
        "trajectory_max",
        "final_mae",
        "final_rmse",
        "final_max"
    ]:
        if col in g.columns:
            result["mean_" + col] = float(g[col].mean())
            result["worst_" + col] = float(g[col].max())

    return result

summaries = []

for length in [1024, 2048]:
    g = short[short["seq_len"] == length]
    assert len(g) > 0, f"No F20 rows for {length}"
    summaries.append(summarize_group(g, length))

for length, g in long_frames.items():
    summaries.append(summarize_group(g, length))

final = pd.DataFrame(summaries).sort_values("seq_len")

OUT = os.path.join(LONG, "final_consolidated_results")
os.makedirs(OUT, exist_ok=True)

final.to_csv(
    os.path.join(OUT, "f20_four_length_summary.csv"),
    index=False
)

# Combined long-context exact-kernel validation summary
long_combined = pd.concat(
    [long_frames[4096], long_frames[8192]],
    ignore_index=True
)

long_audit = {
    "sequence_lengths": [4096, 8192],
    "layer_records": len(long_combined),
    "intermediate_mismatches": int(
        long_combined["intermediate_mismatches"].sum()
    ),
    "final_mismatches": int(
        long_combined["final_mismatches"].sum()
    ),
    "overflow_count": int(
        long_combined["overflow_count"].sum()
    ),
    "maximum_integer_difference": int(
        long_combined["max_integer_difference"].max()
    ),
    "maximum_integer_state": int(
        long_combined["max_integer_state"].max()
    )
}

with open(os.path.join(OUT, "long_context_validation_audit.json"), "w") as f:
    json.dump(long_audit, f, indent=2)

print("========== FOUR-LENGTH F20 SUMMARY ==========")
display(final)

print("\n========== LONG-CONTEXT KERNEL AUDIT ==========")
print(json.dumps(long_audit, indent=2))

print("\nSaved to:", OUT)


========== FOUR-LENGTH F20 SUMMARY ==========


,seq_len,layer_records,total_updates,pooled_coverage_pct,overflow_count,max_integer_state,mean_trajectory_mae,worst_trajectory_mae,mean_trajectory_rmse,worst_trajectory_rmse,...,mean_final_max,worst_final_max,dyadic2_updates,pot_updates,exact_zero_updates,approx_zero_updates,fallback_updates,intermediate_mismatches,final_mismatches,max_integer_difference
0,1024,240,23040,55.269097,0,378673024,8.655578e-07,0.000005,0.000004,0.000016,...,0.000438,0.001663,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2048,240,46080,51.163194,0,481094944,1.094323e-06,0.000005,0.000005,0.000020,...,0.000481,0.002526,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4096,48,18432,73.871528,0,416063839,NaN,NaN,NaN,NaN,...,NaN,NaN,890.0,2348.0,5748.0,4630.0,4816.0,0.0,0.0,0.0
3,8192,48,36864,73.364258,0,569384607,NaN,NaN,NaN,NaN,...,NaN,NaN,1334.0,4785.0,11595.0,9331.0,9819.0,0.0,0.0,0.0



========== LONG-CONTEXT KERNEL AUDIT ==========
{
  "sequence_lengths": [
    4096,
    8192
  ],
  "layer_records": 96,
  "intermediate_mismatches": 0,
  "final_mismatches": 0,
  "overflow_count": 0,
  "maximum_integer_difference": 0,
  "maximum_integer_state": 569384607
}

Saved to: /kaggle/working/shiftscan_final/state_passing/long_context_validation/final_consolidated_results


In [206]:

import os, json, glob, hashlib
import pandas as pd
from pathlib import Path

BASE = "/kaggle/working/shiftscan_final"
STATE = os.path.join(BASE, "state_passing")
LONG = os.path.join(STATE, "long_context_validation")
MULTI = os.path.join(STATE, "multisequence_validation")

FINAL = os.path.join(BASE, "final_research_package")
os.makedirs(FINAL, exist_ok=True)

print("=" * 65)
print("SHIFT-SCAN FINAL RESEARCH AUDIT")
print("=" * 65)

# 1. Canonical independent ablation
ABLATION = os.path.join(
    MULTI, "corrected_independent_ablation_v2"
)
ablation_path = os.path.join(ABLATION, "per_layer_results.csv")

assert os.path.exists(ablation_path), "Canonical ablation CSV missing"

ab = pd.read_csv(ablation_path)
print("\nCANONICAL ABLATION")
print("Rows:", len(ab))
print("Policies:", sorted(ab.policy.unique()))
print("Total updates per policy:")
print(ab.groupby("policy").total_updates.sum())

# 2. Long-context kernel validation
long_files = [
    os.path.join(LONG, "f20_validation", "per_layer_results_4096.csv"),
    os.path.join(LONG, "f20_validation", "per_layer_results_8192.csv")
]

long_dfs = []
for path in long_files:
    assert os.path.exists(path), f"Missing: {path}"
    long_dfs.append(pd.read_csv(path))

long_df = pd.concat(long_dfs, ignore_index=True)

print("\nLONG-CONTEXT VALIDATION")
print("Layer records:", len(long_df))
print("Intermediate mismatches:", long_df.intermediate_mismatches.sum())
print("Final mismatches:", long_df.final_mismatches.sum())
print("Overflow count:", long_df.overflow_count.sum())
print("Maximum integer difference:", long_df.max_integer_difference.max())

assert long_df.intermediate_mismatches.sum() == 0
assert long_df.final_mismatches.sum() == 0
assert long_df.overflow_count.sum() == 0
assert long_df.max_integer_difference.max() == 0

long_df.to_csv(
    os.path.join(FINAL, "long_context_4096_8192.csv"),
    index=False
)

# 3. Existing shorter-context consolidated result
short_candidates = glob.glob(
    os.path.join(LONG, "**", "*four*length*.csv"),
    recursive=True
) + glob.glob(
    os.path.join(LONG, "**", "*consolidated*.csv"),
    recursive=True
)

print("\nSHORT-CONTEXT CONSOLIDATED FILE CANDIDATES")
for p in short_candidates:
    print(p)

# 4. Model-level evaluation files
model_eval = os.path.join(
    BASE, "heldout_500", "model_evaluation"
)
print("\nMODEL-LEVEL EVALUATION FILES")
if os.path.exists(model_eval):
    for p in glob.glob(os.path.join(model_eval, "**", "*.csv"), recursive=True):
        print(p)

# 5. Existing experiment summaries and manifests
print("\nEXPERIMENT SUMMARY JSON FILES")
for p in glob.glob(os.path.join(BASE, "**", "*.json"), recursive=True):
    if any(k in p.lower() for k in [
        "summary", "config", "metadata", "manifest", "result"
    ]):
        print(p)

# 6. Copy canonical ablation
ab.to_csv(
    os.path.join(FINAL, "canonical_independent_ablation.csv"),
    index=False
)

# 7. Create artifact inventory with hashes
inventory = []
for p in glob.glob(os.path.join(BASE, "**", "*"), recursive=True):
    if os.path.isfile(p):
        size = os.path.getsize(p)
        if size == 0:
            continue
        h = hashlib.sha256()
        with open(p, "rb") as f:
            for block in iter(lambda: f.read(1024 * 1024), b""):
                h.update(block)
        inventory.append({
            "relative_path": os.path.relpath(p, BASE),
            "size_bytes": size,
            "sha256": h.hexdigest()
        })

inventory_df = pd.DataFrame(inventory)
inventory_df.to_csv(
    os.path.join(FINAL, "artifact_inventory.csv"),
    index=False
)

print("\nFINAL PACKAGE DIRECTORY:", FINAL)
print("Inventory entries:", len(inventory_df))
print("\nAUDIT PASSED: canonical ablation and long-context kernel checks.")


SHIFT-SCAN FINAL RESEARCH AUDIT

CANONICAL ABLATION
Rows: 2880
Policies: ['A_Dyadic2', 'B_Dyadic2_ExactZero', 'C_Dyadic2_PoT', 'D_Dyadic2_PoT_ExactZero', 'E_Dyadic2_PoT_ApproxZero', 'F_FullAdaptive']
Total updates per policy:
policy
A_Dyadic2                   69120
B_Dyadic2_ExactZero         69120
C_Dyadic2_PoT               69120
D_Dyadic2_PoT_ExactZero     69120
E_Dyadic2_PoT_ApproxZero    69120
F_FullAdaptive              69120
Name: total_updates, dtype: int64

LONG-CONTEXT VALIDATION
Layer records: 96
Intermediate mismatches: 0
Final mismatches: 0
Overflow count: 0
Maximum integer difference: 0

SHORT-CONTEXT CONSOLIDATED FILE CANDIDATES
/kaggle/working/shiftscan_final/state_passing/long_context_validation/final_consolidated_results/f20_four_length_summary.csv

MODEL-LEVEL EVALUATION FILES
/kaggle/working/shiftscan_final/heldout_500/model_evaluation/heldout_document_losses.csv
/kaggle/working/shiftscan_final/heldout_500/model_evaluation/heldout_document_losses_partial.csv
/kaggl

In [207]:

import os, json, shutil
import pandas as pd
from pathlib import Path

BASE = Path("/kaggle/working/shiftscan_final")
OUT = BASE / "final_research_package"
OUT.mkdir(parents=True, exist_ok=True)

def read_csv(path):
    path = Path(path)
    if not path.exists():
        print("MISSING:", path)
        return None
    return pd.read_csv(path)

def copy_if_exists(src, dst_name=None):
    src = Path(src)
    if src.exists() and src.is_file():
        dst = OUT / (dst_name or src.name)
        shutil.copy2(src, dst)
        print("PACKAGED:", dst.name)
        return dst
    return None

# ---------------------------------------------------------
# A. Canonical arithmetic ablation
# ---------------------------------------------------------
ab_path = BASE / "state_passing/multisequence_validation/corrected_independent_ablation_v2/per_layer_results.csv"
ab = read_csv(ab_path)
assert ab is not None and len(ab) == 2880

# Pooled arithmetic-mode counts: sum actual recurrence updates
count_cols = ["dyadic2", "pot", "exact_zero", "approx_zero", "fallback"]
ablation_rows = []

for policy, g in ab.groupby("policy"):
    row = {
        "policy": policy,
        "layer_records": len(g),
        "total_updates": int(g["total_updates"].sum()),
    }
    for c in count_cols:
        row[c + "_updates"] = int(g[c].sum())
        row[c + "_pct"] = 100 * g[c].sum() / g["total_updates"].sum()

    row["strict_shift_pct"] = 100 * (
        g["dyadic2"].sum() + g["pot"].sum()
    ) / g["total_updates"].sum()

    row["nonfallback_pct"] = 100 * (
        g["dyadic2"].sum() + g["pot"].sum()
        + g["exact_zero"].sum() + g["approx_zero"].sum()
    ) / g["total_updates"].sum()

    for metric in [
        "coefficient_mae", "trajectory_mae", "trajectory_rmse",
        "trajectory_max", "final_mae", "final_rmse", "final_max"
    ]:
        if metric in g.columns:
            row["mean_layer_" + metric] = g[metric].mean()
            row["worst_layer_" + metric] = g[metric].max()

    row["max_integer_state"] = g["max_integer_state"].max()
    ablation_rows.append(row)

ablation_summary = pd.DataFrame(ablation_rows)
ablation_summary.to_csv(OUT / "ablation_summary_pooled.csv", index=False)
ab.to_csv(OUT / "ablation_per_layer_canonical.csv", index=False)

print("\n========== CANONICAL ABLATION: POOLED UPDATE COUNTS ==========")
print(ablation_summary.to_string(index=False))

# By sequence length
length_rows = []
for (policy, seq_len), g in ab.groupby(["policy", "seq_len"]):
    total = int(g["total_updates"].sum())
    row = {
        "policy": policy,
        "seq_len": int(seq_len),
        "total_updates": total,
    }
    for c in count_cols:
        row[c + "_updates"] = int(g[c].sum())
        row[c + "_pct"] = 100 * g[c].sum() / total
    row["strict_shift_pct"] = 100 * (
        g["dyadic2"].sum() + g["pot"].sum()
    ) / total
    row["nonfallback_pct"] = 100 * (
        g["dyadic2"].sum() + g["pot"].sum()
        + g["exact_zero"].sum() + g["approx_zero"].sum()
    ) / total
    row["mean_trajectory_mae"] = g["trajectory_mae"].mean()
    row["mean_trajectory_rmse"] = g["trajectory_rmse"].mean()
    row["worst_trajectory_max"] = g["trajectory_max"].max()
    row["mean_final_mae"] = g["final_mae"].mean()
    row["mean_final_rmse"] = g["final_rmse"].mean()
    row["worst_final_max"] = g["final_max"].max()
    length_rows.append(row)

by_length = pd.DataFrame(length_rows)
by_length.to_csv(OUT / "ablation_by_sequence_length.csv", index=False)

print("\n========== ABLATION BY SEQUENCE LENGTH ==========")
print(by_length.to_string(index=False))

# ---------------------------------------------------------
# B. Four-length state-passing validation
# ---------------------------------------------------------
four_path = BASE / "state_passing/long_context_validation/final_consolidated_results/f20_four_length_summary.csv"
four = read_csv(four_path)
if four is not None:
    four.to_csv(OUT / "four_length_validation.csv", index=False)
    print("\n========== FOUR-LENGTH VALIDATION ==========")
    print("Columns:", list(four.columns))
    print(four.to_string(index=False))

# Long-context 4096/8192 correctness records
long_dir = BASE / "state_passing/long_context_validation/f20_validation"
for name in ["summary_4096.json", "summary_8192.json"]:
    p = long_dir / name
    if p.exists():
        with open(p) as f:
            obj = json.load(f)
        with open(OUT / name, "w") as f:
            json.dump(obj, f, indent=2)
        print("\n", name, json.dumps(obj, indent=2))

# ---------------------------------------------------------
# C. Held-out 500-document model evaluation
# ---------------------------------------------------------
eval_dir = BASE / "heldout_500/model_evaluation"
eval_names = [
    "heldout_model_summary.csv",
    "heldout_paired_comparisons.csv",
    "heldout_quantization_coverage.csv",
    "token_weighted_bootstrap_20000.csv",
    "heldout_document_losses.csv",
]

print("\n========== HELD-OUT MODEL EVALUATION ==========")
for name in eval_names:
    df = read_csv(eval_dir / name)
    if df is not None:
        copy_if_exists(eval_dir / name)
        print("\nFILE:", name)
        print("Columns:", list(df.columns))
        if len(df) <= 30:
            print(df.to_string(index=False))
        else:
            print(df.head(8).to_string(index=False))

# ---------------------------------------------------------
# D. Final validation and reproducibility metadata
# ---------------------------------------------------------
for rel in [
    "final_validation/final_validation_results.json",
    "final_validation/reproducibility_manifest.json",
    "baseline_metadata.json",
    "heldout_500/experiment_config.json",
    "state_passing/multisequence_validation/fixed_point_controller_validation_v2/experiment_config.json",
]:
    p = BASE / rel
    if p.exists():
        copy_if_exists(p, Path(rel).name)
        try:
            with open(p) as f:
                obj = json.load(f)
            print("\nJSON:", rel)
            print(json.dumps(obj, indent=2)[:6000])
        except Exception as e:
            print("Could not parse", rel, e)

# ---------------------------------------------------------
# E. Package canonical source tables
# ---------------------------------------------------------
copy_if_exists(ab_path, "ablation_per_layer_canonical.csv")
copy_if_exists(four_path, "four_length_validation_source.csv")

manifest = {
    "project": "Shift-Scan",
    "canonical_ablation": str(ab_path),
    "ablation_rows": int(len(ab)),
    "ablation_policies": sorted(ab.policy.unique().tolist()),
    "four_length_summary": str(four_path),
    "model_evaluation_directory": str(eval_dir),
    "scope_note": (
        "The custom Triton kernel is validated for the Mamba-2 state-passing "
        "recurrence on captured tensors. It is not yet integrated into the "
        "complete Mamba-2 forward path."
    )
}
with open(OUT / "final_package_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("\nFINAL PACKAGE:", OUT)
print("Files:", len(list(OUT.iterdir())))



========== CANONICAL ABLATION: POOLED UPDATE COUNTS ==========
                  policy  layer_records  total_updates  dyadic2_updates  dyadic2_pct  pot_updates   pot_pct  exact_zero_updates  exact_zero_pct  approx_zero_updates  approx_zero_pct  fallback_updates  fallback_pct  strict_shift_pct  nonfallback_pct  mean_layer_coefficient_mae  worst_layer_coefficient_mae  mean_layer_trajectory_mae  worst_layer_trajectory_mae  mean_layer_trajectory_rmse  worst_layer_trajectory_rmse  mean_layer_trajectory_max  worst_layer_trajectory_max  mean_layer_final_mae  worst_layer_final_mae  mean_layer_final_rmse  worst_layer_final_rmse  mean_layer_final_max  worst_layer_final_max  max_integer_state
               A_Dyadic2            480          69120             4433     6.413484            0  0.000000                   0        0.000000                    0         0.000000             64687     93.586516          6.413484         6.413484                    0.000256                     0.002196  

In [208]:

import json
from pathlib import Path
import pandas as pd

BASE = Path("/kaggle/working/shiftscan_final")
OUT = BASE / "final_research_package"

# Correct the 4096 JSON using actual mode counts
p4096 = BASE / "state_passing/long_context_validation/f20_validation/summary_4096.json"
with open(p4096) as f:
    s4096 = json.load(f)

counts = s4096["mode_counts"]
total_4096 = sum(counts.values())
nonfallback_4096 = (
    counts["dyadic2"] + counts["pot"]
    + counts["exact_zero"] + counts["approx_zero"]
)
s4096["total_updates"] = total_4096
s4096["pooled_coverage_pct"] = 100 * nonfallback_4096 / total_4096
s4096["pooled_coverage_calculation"] = (
    "100 * (dyadic2 + pot + exact_zero + approx_zero) / total_updates"
)
s4096["metadata_corrected"] = True

with open(p4096, "w") as f:
    json.dump(s4096, f, indent=2)

with open(OUT / "summary_4096_corrected.json", "w") as f:
    json.dump(s4096, f, indent=2)

# Load canonical four-length table
four_path = BASE / "state_passing/long_context_validation/final_consolidated_results/f20_four_length_summary.csv"
four = pd.read_csv(four_path)

# Validate long-context correctness
assert (four["overflow_count"] == 0).all()
assert (four["intermediate_mismatches"].fillna(0) == 0).all()
assert (four["final_mismatches"].fillna(0) == 0).all()
assert (four["max_integer_difference"].fillna(0) == 0).all()

# Correct 4096 coverage must agree with mode counts
csv_cov = float(four.loc[four.seq_len == 4096, "pooled_coverage_pct"].iloc[0])
json_cov = float(s4096["pooled_coverage_pct"])
assert abs(csv_cov - json_cov) < 1e-8

# Freeze a concise research summary
ab = pd.read_csv(OUT / "ablation_summary_pooled.csv")
model = pd.read_csv(OUT / "heldout_model_summary.csv")
boot = pd.read_csv(OUT / "token_weighted_bootstrap_20000.csv")
coverage = pd.read_csv(OUT / "heldout_quantization_coverage.csv")

full = ab.loc[ab.policy == "F_FullAdaptive"].iloc[0]
t2 = model.loc[model.method == "K1_14_T2"].iloc[0]
t3 = model.loc[model.method == "K1_14_T3"].iloc[0]
t3b = model.loc[model.method == "K1_18_T3"].iloc[0]

summary = {
    "project": "Shift-Scan",
    "canonical_results": {
        "state_passing_updates": int(full.total_updates),
        "full_adaptive_dyadic2_updates": int(full.dyadic2_updates),
        "full_adaptive_pot_updates": int(full.pot_updates),
        "full_adaptive_exact_zero_updates": int(full.exact_zero_updates),
        "full_adaptive_approx_zero_updates": int(full.approx_zero_updates),
        "full_adaptive_fallback_updates": int(full.fallback_updates),
        "strict_shift_pct": float(full.strict_shift_pct),
        "nonfallback_pct": float(full.nonfallback_pct),
        "fallback_pct": float(full.fallback_pct),
        "max_integer_state": int(full.max_integer_state)
    },
    "four_length_validation": four.to_dict(orient="records"),
    "heldout_500_model_evaluation": {
        "target_tokens": int(model.target_tokens.iloc[0]),
        "baseline_ce": float(model.loc[model.method == "Baseline", "mean_CE"].iloc[0]),
        "baseline_ppl": float(model.loc[model.method == "Baseline", "PPL"].iloc[0]),
        "K1_14_T2": {
            "mean_CE": float(t2.mean_CE),
            "PPL": float(t2.PPL),
            "delta_CE": float(t2.delta_CE_vs_baseline),
            "relative_ppl_change_pct": float(t2.relative_PPL_change_pct)
        },
        "K1_14_T3": {
            "mean_CE": float(t3.mean_CE),
            "PPL": float(t3.PPL),
            "delta_CE": float(t3.delta_CE_vs_baseline),
            "relative_ppl_change_pct": float(t3.relative_PPL_change_pct)
        },
        "K1_18_T3": {
            "mean_CE": float(t3b.mean_CE),
            "PPL": float(t3b.PPL),
            "delta_CE": float(t3b.delta_CE_vs_baseline),
            "relative_ppl_change_pct": float(t3b.relative_PPL_change_pct)
        },
        "token_weighted_bootstrap": boot.to_dict(orient="records"),
        "quantization_coverage": coverage.to_dict(orient="records")
    },
    "implementation_status": {
        "custom_state_passing_triton_kernel": "validated against CPU integer reference",
        "sequence_lengths_tested": [1024, 2048, 4096, 8192],
        "full_mamba2_ssd_integration": "not implemented",
        "end_to_end_kernel_model_accuracy": "not evaluated",
        "end_to_end_speedup": "not measured"
    },
    "scope_and_limitations": [
        "State-passing kernel results apply to captured Mamba-2 state-passing tensors, not the complete SSD scan.",
        "Exact fallback still uses floating-point decay multiplication.",
        "Strict shift arithmetic covers 20.59% of updates in the full adaptive ablation.",
        "Model-level T3 perplexity differences are small and statistically inconclusive in the tested 500-document evaluation.",
        "No end-to-end speedup claim is made."
    ]
}

with open(OUT / "final_research_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("4096 JSON coverage repaired:", round(json_cov, 6), "%")
print("4096 CSV coverage:", round(csv_cov, 6), "%")
print("Four-length correctness audit: PASS")
print("Full adaptive strict shift coverage:", round(full.strict_shift_pct, 4), "%")
print("Full adaptive non-fallback coverage:", round(full.nonfallback_pct, 4), "%")
print("Final summary saved:", OUT / "final_research_summary.json")


4096 JSON coverage repaired: 73.871528 %
4096 CSV coverage: 73.871528 %
Four-length correctness audit: PASS
Full adaptive strict shift coverage: 20.5859 %
Full adaptive non-fallback coverage: 77.1861 %
Final summary saved: /kaggle/working/shiftscan_final/final_research_package/final_research_summary.json


In [209]:

# SHIFT-SCAN: FULL MAMBA-2 INTEGRATION PREFLIGHT
# Run this in the existing Kaggle notebook.

import os, sys, inspect, importlib
from pathlib import Path

print("=" * 75)
print("SHIFT-SCAN FULL INTEGRATION PREFLIGHT")
print("=" * 75)

# 1. Environment
import torch, triton
print("\n[1] ENVIRONMENT")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Triton:", triton.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

# 2. Mamba installation
print("\n[2] MAMBA INSTALLATION")
try:
    import mamba_ssm
    print("Mamba version:", getattr(mamba_ssm, "__version__", "unknown"))
    print("Mamba path:", inspect.getfile(mamba_ssm))
except Exception as e:
    print("Mamba import error:", repr(e))

# 3. Relevant SSD functions and source locations
print("\n[3] SSD FUNCTION DISCOVERY")
modules_to_check = [
    "mamba_ssm.ops.triton.ssd_combined",
    "mamba_ssm.ops.triton.ssd_chunk_state",
    "mamba_ssm.ops.triton.ssd_chunk_scan",
    "mamba_ssm.ops.triton.ssd_state_passing",
]

for modname in modules_to_check:
    print(f"\n--- {modname} ---")
    try:
        mod = importlib.import_module(modname)
        print("File:", inspect.getfile(mod))
        for name, obj in inspect.getmembers(mod, inspect.isfunction):
            if any(k in name.lower() for k in
                   ["cumsum", "state", "scan", "chunk", "fwd"]):
                try:
                    print("\nFunction:", name)
                    print("Signature:", inspect.signature(obj))
                    print("Source:", inspect.getsourcelines(obj)[1])
                except Exception:
                    pass
    except Exception as e:
        print("Unavailable:", repr(e))

# 4. Find existing Shift-Scan implementation files
print("\n[4] SHIFT-SCAN FILE DISCOVERY")
roots = [
    Path("/kaggle/working/shiftscan_final"),
    Path("/kaggle/working/shiftscan_improved_outputs"),
]

keywords = [
    "triton", "kernel", "state_passing", "f20",
    "adaptive", "ablation", "validation"
]

found = []
for root in roots:
    if root.exists():
        for p in root.rglob("*"):
            if p.is_file() and any(k in p.name.lower() for k in keywords):
                found.append(p)

for p in found[:100]:
    print(p)

print("\nTotal matching files:", len(found))

# 5. Check existing package validation status
print("\n[5] EXISTING VALIDATION MANIFEST")
manifest = Path(
    "/kaggle/working/shiftscan_final/final_research_package/"
    "final_validation_results.json"
)

if manifest.exists():
    try:
        import json
        data = json.loads(manifest.read_text())
        print(json.dumps(data, indent=2)[:12000])
    except Exception as e:
        print("Manifest read error:", repr(e))
else:
    print("Validation manifest not found:", manifest)

print("\n" + "=" * 75)
print("PREFLIGHT COMPLETE — SEND THE OUTPUT")
print("=" * 75)


SHIFT-SCAN FULL INTEGRATION PREFLIGHT

[1] ENVIRONMENT
Python: 3.13.16
PyTorch: 2.11.0+cu130
Triton: 3.6.0
CUDA: 13.0
GPU: Tesla T4

[2] MAMBA INSTALLATION
Mamba version: 2.3.2.post1
Mamba path: /usr/local/lib/python3.13/dist-packages/mamba_ssm/__init__.py

[3] SSD FUNCTION DISCOVERY

--- mamba_ssm.ops.triton.ssd_combined ---
File: /usr/local/lib/python3.13/dist-packages/mamba_ssm/ops/triton/ssd_combined.py

Function: _bmm_chunk_bwd
Signature: (a, dout, residual=None, out=None)
Source: 213

Function: _bmm_chunk_fwd
Signature: (a, b, chunk_size, seq_idx=None, causal=False, output_dtype=None)
Source: 165

Function: _chunk_cumsum_bwd
Signature: (ddA, ddt_out, dt, A, dt_bias=None, dt_softplus=False, dt_limit=(0.0, inf), ddt=None)
Source: 744

Function: _chunk_cumsum_fwd
Signature: (dt, A, chunk_size, dt_bias=None, dt_softplus=False, dt_limit=(0.0, inf))
Source: 718

Function: _chunk_scan_bwd_dC
Signature: (prev_states, dA_cumsum, dout, seq_idx=None, C=None, ngroups=1)
Source: 1451

Functio

In [210]:

# SHIFT-SCAN: RECOVER KERNEL SOURCE + INSPECT SSD FORWARD PATHS
import os, json, inspect, importlib
from pathlib import Path

ROOT = Path("/kaggle/working/shiftscan_final")

print("=" * 80)
print("1. SEARCHING FOR THE ORIGINAL SHIFT-SCAN TRITON SOURCE")
print("=" * 80)

# Search source-like files and JSON artifacts
candidates = []
for p in ROOT.rglob("*"):
    if not p.is_file():
        continue
    if p.suffix.lower() in {".py", ".json", ".txt", ".ipynb"}:
        candidates.append(p)

keywords = [
    "state_passing", "dyadic", "approx_zero", "shift_subtract",
    "adaptive", "triton", "kernel"
]

for p in candidates:
    try:
        if p.stat().st_size > 20_000_000:
            continue
        content = p.read_text(errors="ignore")
        low = content.lower()
        hits = [k for k in keywords if k in low]
        if hits and (
            p.suffix.lower() in {".py", ".ipynb"}
            or ("@triton.jit" in low)
            or ("triton.jit" in low)
        ):
            print(f"\nFILE: {p}")
            print("SIZE:", p.stat().st_size, "| MATCHES:", hits)
            if "@triton.jit" in low or "triton.jit" in low:
                lines = content.splitlines()
                for i, line in enumerate(lines):
                    if "@triton.jit" in line or "triton.jit" in line:
                        start, end = max(0, i-5), min(len(lines), i+55)
                        print("\n".join(
                            f"{j+1}: {lines[j]}" for j in range(start, end)
                        ))
                        break
    except Exception:
        pass

print("\n" + "=" * 80)
print("2. INSPECTING SAVED KERNEL-SOURCE JSON")
print("=" * 80)

json_files = [
    ROOT / "final_validation/extracted_triton_kernel_sources.json",
    ROOT / "final_validation/state_passing_kernel_inspection.json",
    ROOT / "final_validation/state_passing_kernel_location.json",
]

def inspect_json(obj, path="root", depth=0):
    if depth > 5:
        return
    if isinstance(obj, dict):
        for k, v in obj.items():
            key = str(k).lower()
            if isinstance(v, str) and (
                "triton.jit" in v.lower()
                or "@triton" in v.lower()
                or "def " in v and len(v) > 500
            ):
                print(f"\nSOURCE CANDIDATE: {path}.{k}")
                print(v[:14000])
            elif isinstance(v, (dict, list)):
                inspect_json(v, f"{path}.{k}", depth+1)
            elif isinstance(v, str) and any(
                x in key for x in ["source", "code", "kernel", "path"]
            ):
                print(f"{path}.{k}: {v[:1500]}")
    elif isinstance(obj, list):
        for i, v in enumerate(obj[:100]):
            if isinstance(v, (dict, list)):
                inspect_json(v, f"{path}[{i}]", depth+1)

for p in json_files:
    print(f"\n--- {p} ---")
    if p.exists():
        try:
            inspect_json(json.loads(p.read_text()))
        except Exception as e:
            print("Could not parse:", repr(e))
    else:
        print("Not found")

print("\n" + "=" * 80)
print("3. EXACT MAMBA-2 SSD FORWARD SOURCE")
print("=" * 80)

mod = importlib.import_module("mamba_ssm.ops.triton.ssd_combined")

for name in [
    "_mamba_chunk_scan_combined_fwd",
    "_chunk_cumsum_fwd",
    "_chunk_state_fwd",
    "_state_passing_fwd",
    "_chunk_scan_fwd",
]:
    obj = getattr(mod, name, None)
    print(f"\n\n### {name} ###")
    if obj is None:
        print("NOT FOUND")
        continue
    try:
        print("SIGNATURE:", inspect.signature(obj))
        src = inspect.getsource(obj)
        print(src[:18000])
    except Exception as e:
        print("Source inspection error:", repr(e))

print("\n" + "=" * 80)
print("4. SEARCH COMPLETE")
print("Send the complete output, especially any @triton.jit source.")
print("=" * 80)


1. SEARCHING FOR THE ORIGINAL SHIFT-SCAN TRITON SOURCE

2. INSPECTING SAVED KERNEL-SOURCE JSON

--- /kaggle/working/shiftscan_final/final_validation/extracted_triton_kernel_sources.json ---

SOURCE CANDIDATE: root._state_passing_bwd_kernel.source
def _state_passing_bwd_kernel(
    # Pointers to matrices
    dout_ptr, out_ptr, dA_cs_ptr, dfinal_states_ptr, seq_idx_ptr,
    dstates_ptr, ddA_cs_ptr, dinitstates_ptr, states_converted_ptr,
    # Matrix dimensions
    dim, nchunks, seqlen, chunk_size,
    # Strides
    stride_dout_batch, stride_dout_chunk, stride_dout_head, stride_dout_dim,
    stride_out_batch, stride_out_chunk, stride_out_head, stride_out_dim,
    stride_dA_cs_batch, stride_dA_cs_chunk, stride_dA_cs_head,
    stride_dfinal_states_batch, stride_dfinal_states_head, stride_dfinal_states_dim,
    stride_seq_idx_batch, stride_seq_idx_seqlen,
    stride_dstates_batch, stride_dstates_chunk, stride_dstates_head, stride_dstates_dim,
    stride_ddA_cs_batch, stride_ddA_cs_chunk, str

In [211]:

# SHIFT-SCAN: RECOVER THE EXACT CUSTOM F20 KERNEL FROM NOTEBOOK HISTORY

import re, json
from pathlib import Path

terms = [
    "@triton.jit",
    "dyadic2",
    "approx_zero",
    "F20",
    "state_passing",
    "shift_subtract",
]

matches = []

try:
    history = list(In)
except Exception:
    history = []

for idx, source in enumerate(history):
    if not isinstance(source, str):
        continue

    low = source.lower()
    score = sum(term.lower() in low for term in terms)

    if score >= 2 and ("triton" in low or "kernel" in low):
        matches.append((score, idx, source))

matches.sort(reverse=True, key=lambda x: x[0])

print("=" * 75)
print("NOTEBOOK HISTORY MATCHES:", len(matches))
print("=" * 75)

out_dir = Path("/kaggle/working/shiftscan_final/recovered_kernel")
out_dir.mkdir(parents=True, exist_ok=True)

for rank, (score, idx, source) in enumerate(matches[:12]):
    path = out_dir / f"history_cell_{idx}.py"
    path.write_text(source)

    print(f"\n{'='*75}")
    print(f"CELL {idx} | SCORE {score} | SAVED: {path}")
    print("=" * 75)
    print(source[:14000])

# Also list any saved Python files containing custom arithmetic
print("\nSAVED SOURCE FILES")
root = Path("/kaggle/working/shiftscan_final")

for p in root.rglob("*.py"):
    try:
        s = p.read_text(errors="ignore")
        low = s.lower()
        if "dyadic2" in low and ("triton" in low or "kernel" in low):
            print(p)
    except Exception:
        pass

print("\nRECOVERY COMPLETE")


NOTEBOOK HISTORY MATCHES: 25

CELL 211 | SCORE 6 | SAVED: /kaggle/working/shiftscan_final/recovered_kernel/history_cell_211.py
# SHIFT-SCAN: RECOVER THE EXACT CUSTOM F20 KERNEL FROM NOTEBOOK HISTORY

import re, json
from pathlib import Path

terms = [
    "@triton.jit",
    "dyadic2",
    "approx_zero",
    "F20",
    "state_passing",
    "shift_subtract",
]

matches = []

try:
    history = list(In)
except Exception:
    history = []

for idx, source in enumerate(history):
    if not isinstance(source, str):
        continue

    low = source.lower()
    score = sum(term.lower() in low for term in terms)

    if score >= 2 and ("triton" in low or "kernel" in low):
        matches.append((score, idx, source))

matches.sort(reverse=True, key=lambda x: x[0])

print("=" * 75)
print("NOTEBOOK HISTORY MATCHES:", len(matches))
print("=" * 75)

out_dir = Path("/kaggle/working/shiftscan_final/recovered_kernel")
out_dir.mkdir(parents=True, exist_ok=True)

for rank, (score, idx, source) in enume

In [212]:
import os
import inspect
import json
import torch
import triton
from pathlib import Path

OUT = Path("/kaggle/working/shiftscan_final/integration")
OUT.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 1. Confirm custom Shift-Scan implementation is available
# ---------------------------------------------------------
kernel = globals().get("shift_scan_state_passing_i32_kernel")

assert kernel is not None, (
    "Custom kernel is not in the current notebook namespace. "
    "Run the cell that defines shift_scan_state_passing_i32_kernel first."
)

print("CUSTOM KERNEL:", kernel)
print("TYPE:", type(kernel))

try:
    kernel_source = inspect.getsource(kernel)
except Exception:
    kernel_source = (
        "# inspect.getsource could not retrieve the live Triton JIT source.\n"
        + repr(kernel)
    )

(OUT / "shift_scan_kernel_source.txt").write_text(kernel_source)

print("\nCUSTOM KERNEL SOURCE")
print("=" * 75)
print(kernel_source[:18000])

# ---------------------------------------------------------
# 2. Check required arithmetic/controller helpers
# ---------------------------------------------------------
helper_names = [
    "build_reference_and_modes",
    "SCALE",
]

helper_status = {}
for name in helper_names:
    value = globals().get(name)
    helper_status[name] = value is not None
    print(f"{name}:",
          "AVAILABLE" if value is not None else "MISSING",
          "|", repr(value)[:300])

# ---------------------------------------------------------
# 3. Inspect installed Mamba-2 SSD interfaces
# ---------------------------------------------------------
import mamba_ssm.ops.triton.ssd_combined as ssd

function_names = [
    "_mamba_chunk_scan_combined_fwd",
    "_chunk_cumsum_fwd",
    "_chunk_state_fwd",
    "_state_passing_fwd",
    "_chunk_scan_fwd",
]

api_report = {}

for name in function_names:
    fn = getattr(ssd, name, None)

    if fn is None:
        api_report[name] = {"available": False}
        print(f"\n{name}: NOT FOUND")
        continue

    try:
        sig = str(inspect.signature(fn))
    except Exception as e:
        sig = f"Signature unavailable: {e}"

    try:
        src = inspect.getsource(fn)
    except Exception as e:
        src = f"Source unavailable: {e}"

    api_report[name] = {
        "available": True,
        "signature": sig,
        "source": src
    }

    print(f"\n{name}{sig}")
    print("-" * 75)
    print(src[:5000])

# ---------------------------------------------------------
# 4. Save inspection report
# ---------------------------------------------------------
report = {
    "torch_version": torch.__version__,
    "triton_version": triton.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available() else None
    ),
    "scale": repr(globals().get("SCALE")),
    "helpers": helper_status,
    "mamba_ssd_api": {
        name: {
            "available": item["available"],
            "signature": item.get("signature")
        }
        for name, item in api_report.items()
    }
}

with open(OUT / "integration_api_report.json", "w") as f:
    json.dump(report, f, indent=2)

with open(OUT / "mamba_ssd_sources.txt", "w") as f:
    for name, item in api_report.items():
        f.write(f"\n\n{'='*90}\n{name}\n{'='*90}\n")
        f.write(item.get("source", "Unavailable"))

print("\n" + "=" * 75)
print("INSPECTION COMPLETE")
print("Saved:", OUT)
print("Kernel source:", OUT / "shift_scan_kernel_source.txt")
print("Mamba API report:", OUT / "integration_api_report.json")
print("Mamba source:", OUT / "mamba_ssd_sources.txt")

CUSTOM KERNEL: JITFunction(__main__:shift_scan_state_passing_i32_kernel)
TYPE: <class 'triton.runtime.jit.JITFunction'>

CUSTOM KERNEL SOURCE
# inspect.getsource could not retrieve the live Triton JIT source.
JITFunction(__main__:shift_scan_state_passing_i32_kernel)
build_reference_and_modes: AVAILABLE | <function build_reference_and_modes at 0x79bdc0f74fe0>
SCALE: AVAILABLE | 1048576

_mamba_chunk_scan_combined_fwd(x, dt, A, B, C, chunk_size, D=None, z=None, dt_bias=None, initial_states=None, seq_idx=None, cu_seqlens=None, dt_softplus=False, dt_limit=(0.0, inf))
---------------------------------------------------------------------------
def _mamba_chunk_scan_combined_fwd(x, dt, A, B, C, chunk_size, D=None, z=None, dt_bias=None, initial_states=None, seq_idx=None, cu_seqlens=None, dt_softplus=False, dt_limit=(0.0, float("inf"))):
    batch, seqlen, nheads, headdim = x.shape
    _, _, ngroups, dstate = B.shape
    assert nheads % ngroups == 0
    assert B.shape == (batch, seqlen, ngroups

In [213]:
import inspect
import json
from pathlib import Path

OUT = Path("/kaggle/working/shiftscan_final/integration")
OUT.mkdir(parents=True, exist_ok=True)

names = [
    "shift_scan_state_passing_i32_kernel",
    "build_reference_and_modes",
]

report = {}

for name in names:
    obj = globals().get(name)
    print("\n" + "=" * 80)
    print("OBJECT:", name)
    print("=" * 80)

    if obj is None:
        print("NOT FOUND")
        continue

    item = {
        "type": str(type(obj)),
        "repr": repr(obj),
    }

    # Triton JIT source is often stored directly in .src
    src = getattr(obj, "src", None)

    if isinstance(src, str) and src.strip():
        item["src"] = src
        print("SOURCE FOUND IN .src")
        print(src[:30000])
    else:
        print("No usable .src attribute")

    # Try the underlying Python function
    fn = getattr(obj, "fn", obj)

    try:
        py_src = inspect.getsource(fn)
        item["python_source"] = py_src
        print("\nPYTHON FUNCTION SOURCE")
        print(py_src[:20000])
    except Exception as e:
        item["python_source_error"] = repr(e)
        print("Python source unavailable:", repr(e))

    # Capture useful Triton metadata
    for attr in ["arg_names", "constexprs", "params", "used_global_vals"]:
        try:
            value = getattr(obj, attr)
            item[attr] = repr(value)
            print(f"\n{attr}:", repr(value)[:5000])
        except Exception:
            pass

    report[name] = item

# Save complete source report
with open(OUT / "custom_kernel_source_report.json", "w") as f:
    json.dump(report, f, indent=2, default=str)

with open(OUT / "custom_kernel_source_report.txt", "w") as f:
    for name, item in report.items():
        f.write("\n" + "=" * 80 + "\n")
        f.write(name + "\n")
        f.write("=" * 80 + "\n")
        f.write(item.get("src", ""))
        f.write("\n\nPYTHON SOURCE\n")
        f.write(item.get("python_source", ""))

print("\nSaved source report to:", OUT)


OBJECT: shift_scan_state_passing_i32_kernel
SOURCE FOUND IN .src
def shift_scan_state_passing_i32_kernel(
    N_ptr,
    MODE_ptr,
    EXP_ptr,
    DECAY_ptr,
    OUT_ptr,
    FINAL_ptr,
    OVERFLOW_ptr,

    NCHUNKS: tl.constexpr,
    NHEADS: tl.constexpr,
    DIM: tl.constexpr,
    NBLOCKS: tl.constexpr,

    BLOCK_SIZE: tl.constexpr
):
    pid_m = tl.program_id(0)
    pid_b = tl.program_id(1)
    pid_h = tl.program_id(2)

    offs = pid_m * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offs < DIM

    state = tl.zeros((BLOCK_SIZE,), dtype=tl.int32)
    block_overflow = tl.full((), 0, dtype=tl.int32)

    ctrl_base = (
        pid_b * NHEADS * NCHUNKS
        + pid_h * NCHUNKS
    )

    for c in range(NCHUNKS):

        mode = tl.load(MODE_ptr + ctrl_base + c)
        exp_base = ctrl_base * 3 + c * 3

        # Mode 1: three-term Dyadic2.
        if mode == 1:
            decay_sum = tl.zeros(
                (BLOCK_SIZE,), dtype=tl.int32
            )

            for j in tl.

In [217]:
import os
import gc
import json
import numpy as np
import pandas as pd
import torch
import triton
from pathlib import Path

import mamba_ssm.ops.triton.ssd_combined as ssd

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------
BASE = Path("/kaggle/working/shiftscan_final")
CAPTURE_DIR = BASE / "state_passing" / "long_context_validation"
OUT_DIR = BASE / "integration"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SCALE = int(SCALE)
assert SCALE == 2**20
assert torch.cuda.is_available()

kernel = shift_scan_state_passing_i32_kernel
BLOCK_SIZE = 128

# Test a representative 1024-token capture first.
CAPTURE_DIR = Path(
    "/kaggle/working/shiftscan_final/state_passing/multisequence_validation"
)
capture_path = CAPTURE_DIR / "doc_000_seq_1024.pt"

assert capture_path.exists(), f"Missing capture: {capture_path}"

data = torch.load(
    capture_path,
    map_location="cpu",
    weights_only=False
)

print("Capture:", capture_path.name)
print("Records:", len(data["records"]))

# ---------------------------------------------------------
# Run every layer record in this capture
# ---------------------------------------------------------
results = []

for record_index, rec in enumerate(data["records"]):

    print(f"\nTesting record {record_index + 1}/{len(data['records'])}")

    # Build canonical fixed-point inputs and CPU reference.
    (
        N_int,
        decay,
        modes,
        exps,
        reference,
        counts,
        max_bound
    ) = build_reference_and_modes(rec)

    chunks, heads, dim = N_int.shape
    nblocks = triton.cdiv(dim, BLOCK_SIZE)

    assert heads == 24
    assert dim == 8192

    # -----------------------------------------------------
    # A. Original Mamba floating-point state passing
    # -----------------------------------------------------
    states_fp = rec["states"].to(
        device="cuda",
        dtype=torch.float32
    ).contiguous()

    decay_log = rec["decay_log"].to(
        device="cuda",
        dtype=torch.float32
    ).contiguous()

    assert states_fp.shape == (1, chunks, heads, dim)
    assert decay_log.shape == (1, heads, chunks)

    with torch.no_grad():
        mamba_out, mamba_final = ssd._state_passing_fwd(
            states_fp,
            decay_log,
            initial_states=None,
            seq_idx=None,
            chunk_size=None,
            out_dtype=torch.float32
        )

    torch.cuda.synchronize()

    mamba_out_np = mamba_out[0].cpu().numpy()
    mamba_final_np = mamba_final[0].cpu().numpy()

    # -----------------------------------------------------
    # B. Custom Shift-Scan integer state passing
    # -----------------------------------------------------
    N_gpu = torch.as_tensor(
        N_int.astype(np.int32),
        device="cuda",
        dtype=torch.int32
    ).contiguous()

    MODE_gpu = torch.as_tensor(
        modes.astype(np.int32),
        device="cuda",
        dtype=torch.int32
    ).contiguous()

    EXP_gpu = torch.as_tensor(
        exps.astype(np.int32),
        device="cuda",
        dtype=torch.int32
    ).contiguous()

    DECAY_gpu = torch.as_tensor(
        decay.astype(np.float64),
        device="cuda",
        dtype=torch.float64
    ).contiguous()

    OUT_gpu = torch.empty(
        (1, chunks, heads, dim),
        device="cuda",
        dtype=torch.int32
    )

    FINAL_gpu = torch.empty(
        (1, heads, dim),
        device="cuda",
        dtype=torch.int32
    )

    OVERFLOW_gpu = torch.zeros(
        (1, heads, nblocks),
        device="cuda",
        dtype=torch.int32
    )

    kernel[(nblocks, 1, heads)](
        N_gpu,
        MODE_gpu,
        EXP_gpu,
        DECAY_gpu,
        OUT_gpu,
        FINAL_gpu,
        OVERFLOW_gpu,
        NCHUNKS=chunks,
        NHEADS=heads,
        DIM=dim,
        NBLOCKS=nblocks,
        BLOCK_SIZE=BLOCK_SIZE
    )

    torch.cuda.synchronize()

    custom_int = OUT_gpu[0].cpu().numpy().astype(np.int64)
    custom_final_int = FINAL_gpu[0].cpu().numpy().astype(np.int64)

    custom_float = custom_int.astype(np.float64) / SCALE
    custom_final_float = custom_final_int.astype(np.float64) / SCALE

    # -----------------------------------------------------
    # C. Independent integer-reference correctness
    # -----------------------------------------------------
    integer_diff = np.abs(custom_int - reference)
    integer_final_diff = np.abs(
        custom_final_int - reference[-1]
    )

    integer_mismatches = int(np.count_nonzero(integer_diff))
    integer_final_mismatches = int(
        np.count_nonzero(integer_final_diff)
    )

    # -----------------------------------------------------
    # D. Compare against original Mamba FP32 output
    # -----------------------------------------------------
    fp_error = custom_float - mamba_out_np
    final_fp_error = custom_final_float - mamba_final_np

    results.append({
        "record": record_index,
        "layer": rec.get("layer", record_index),
        "chunks": chunks,
        "heads": heads,
        "state_dim": dim,

        "integer_mismatches": integer_mismatches,
        "integer_final_mismatches": integer_final_mismatches,

        "max_integer_difference": int(
            max(integer_diff.max(), integer_final_diff.max())
        ),

        "trajectory_mae_vs_mamba": float(
            np.mean(np.abs(fp_error))
        ),
        "trajectory_rmse_vs_mamba": float(
            np.sqrt(np.mean(fp_error ** 2))
        ),
        "trajectory_max_abs_vs_mamba": float(
            np.max(np.abs(fp_error))
        ),

        "final_mae_vs_mamba": float(
            np.mean(np.abs(final_fp_error))
        ),
        "final_rmse_vs_mamba": float(
            np.sqrt(np.mean(final_fp_error ** 2))
        ),
        "final_max_abs_vs_mamba": float(
            np.max(np.abs(final_fp_error))
        ),

        "mamba_trajectory_max_abs": float(
            np.max(np.abs(mamba_out_np))
        ),
        "custom_trajectory_max_abs": float(
            np.max(np.abs(custom_float))
        ),

        "overflow_count": int(OVERFLOW_gpu.sum().item()),

        "dyadic2": int(counts["dyadic2"]),
        "pot": int(counts["pot"]),
        "exact_zero": int(counts["exact_zero"]),
        "approx_zero": int(counts["approx_zero"]),
        "fallback": int(counts["fallback"]),
        "max_local_bound": float(max_bound)
    })

    del states_fp, decay_log, mamba_out, mamba_final
    del N_gpu, MODE_gpu, EXP_gpu, DECAY_gpu
    del OUT_gpu, FINAL_gpu, OVERFLOW_gpu
    del custom_int, custom_final_int
    del custom_float, custom_final_float
    del mamba_out_np, mamba_final_np
    del fp_error, final_fp_error
    del integer_diff, integer_final_diff
    del reference, N_int

    gc.collect()
    torch.cuda.empty_cache()

# ---------------------------------------------------------
# Save and summarize
# ---------------------------------------------------------

df = pd.DataFrame(results)

csv_path = OUT_DIR / "mamba_state_passing_comparison_1024.csv"
df.to_csv(csv_path, index=False)

summary = {
    "capture": capture_path.name,
    "layer_records": len(df),

    "integer_mismatches": int(df.integer_mismatches.sum()),
    "integer_final_mismatches": int(
        df.integer_final_mismatches.sum()
    ),
    "max_integer_difference": int(
        df.max_integer_difference.max()
    ),
    "overflow_count": int(df.overflow_count.sum()),

    "mean_trajectory_mae_vs_mamba": float(
        df.trajectory_mae_vs_mamba.mean()
    ),
    "mean_trajectory_rmse_vs_mamba": float(
        df.trajectory_rmse_vs_mamba.mean()
    ),
    "worst_trajectory_max_abs_vs_mamba": float(
        df.trajectory_max_abs_vs_mamba.max()
    ),

    "mean_final_mae_vs_mamba": float(
        df.final_mae_vs_mamba.mean()
    ),
    "mean_final_rmse_vs_mamba": float(
        df.final_rmse_vs_mamba.mean()
    ),
    "worst_final_max_abs_vs_mamba": float(
        df.final_max_abs_vs_mamba.max()
    )
}

json_path = OUT_DIR / "mamba_state_passing_comparison_1024.json"
with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 75)
print("SHIFT-SCAN VS ORIGINAL MAMBA STATE PASSING")
print("=" * 75)
print(json.dumps(summary, indent=2))

print("\nPER-LAYER RESULTS")
display(df)

print("\nSaved:")
print(csv_path)
print(json_path)

Capture: doc_000_seq_1024.pt
Records: 24

Testing record 1/24

Testing record 2/24

Testing record 3/24

Testing record 4/24

Testing record 5/24

Testing record 6/24

Testing record 7/24

Testing record 8/24

Testing record 9/24

Testing record 10/24

Testing record 11/24

Testing record 12/24

Testing record 13/24

Testing record 14/24

Testing record 15/24

Testing record 16/24

Testing record 17/24

Testing record 18/24

Testing record 19/24

Testing record 20/24

Testing record 21/24

Testing record 22/24

Testing record 23/24

Testing record 24/24

SHIFT-SCAN VS ORIGINAL MAMBA STATE PASSING
{
  "capture": "doc_000_seq_1024.pt",
  "layer_records": 24,
  "integer_mismatches": 0,
  "integer_final_mismatches": 0,
  "max_integer_difference": 0,
  "overflow_count": 0,
  "mean_trajectory_mae_vs_mamba": 0.17980896461987506,
  "mean_trajectory_rmse_vs_mamba": 0.5597588141589501,
  "worst_trajectory_max_abs_vs_mamba": 321.40765380859375,
  "mean_final_mae_vs_mamba": 9.942299627956083e-07,


,record,layer,chunks,heads,state_dim,integer_mismatches,integer_final_mismatches,max_integer_difference,trajectory_mae_vs_mamba,trajectory_rmse_vs_mamba,...,final_max_abs_vs_mamba,mamba_trajectory_max_abs,custom_trajectory_max_abs,overflow_count,dyadic2,pot,exact_zero,approx_zero,fallback,max_local_bound
0,0,0,4,24,8192,0,0,0,0.017990,0.120545,...,0.000859,37.255058,37.255056,0,0,63,0,4,29,9.714145e-04
1,1,1,4,24,8192,0,0,0,0.114275,0.368551,...,0.000001,26.531927,26.531927,0,0,7,48,20,21,5.015010e-23
2,2,2,4,24,8192,0,0,0,0.069241,0.246428,...,0.000094,36.232616,36.232616,0,14,10,17,15,40,3.111833e-04
3,3,3,4,24,8192,0,0,0,0.294742,0.858945,...,0.000165,116.095062,116.095062,0,1,19,33,29,14,5.630072e-04
4,4,4,4,24,8192,0,0,0,0.332842,1.164935,...,0.000769,84.779205,84.779209,0,21,8,16,4,47,7.699756e-04
5,5,5,4,24,8192,0,0,0,0.230246,0.664030,...,0.000169,55.782688,55.782521,0,3,10,54,11,18,9.209967e-04
6,6,6,4,24,8192,0,0,0,0.172072,0.416543,...,0.000974,34.432686,35.092683,0,13,20,9,15,39,9.741088e-04
7,7,7,4,24,8192,0,0,0,0.637614,1.570430,...,0.000005,82.473305,107.632911,0,0,16,16,43,21,5.907991e-04
8,8,8,4,24,8192,0,0,0,0.236192,0.638887,...,0.000125,61.853985,61.853985,0,0,25,2,64,5,5.059872e-04
9,9,9,4,24,8192,0,0,0,0.174974,0.769250,...,0.000009,135.751099,135.751099,0,0,2,63,25,6,2.327675e-12



Saved:
/kaggle/working/shiftscan_final/integration/mamba_state_passing_comparison_1024.csv
/kaggle/working/shiftscan_final/integration/mamba_state_passing_comparison_1024.json


In [215]:
from pathlib import Path

ROOT = Path("/kaggle/working/shiftscan_final")

print("Searching for 1024-token captures...\n")

matches = list(ROOT.rglob("doc_000_seq_1024.pt"))

if matches:
    for p in matches:
        print("FOUND:", p)
else:
    print("Exact filename not found.")
    print("\nNearby capture files:")
    for p in ROOT.rglob("*seq_1024*.pt"):
        print(p)

print("\nSearching for 2048-token captures...")
for p in ROOT.rglob("*seq_2048*.pt"):
    print(p)

Searching for 1024-token captures...

FOUND: /kaggle/working/shiftscan_final/state_passing/multisequence_validation/doc_000_seq_1024.pt

Searching for 2048-token captures...
/kaggle/working/shiftscan_final/state_passing/multisequence_validation/doc_004_seq_2048.pt
/kaggle/working/shiftscan_final/state_passing/multisequence_validation/doc_009_seq_2048.pt
/kaggle/working/shiftscan_final/state_passing/multisequence_validation/doc_007_seq_2048.pt
/kaggle/working/shiftscan_final/state_passing/multisequence_validation/doc_008_seq_2048.pt
/kaggle/working/shiftscan_final/state_passing/multisequence_validation/doc_003_seq_2048.pt
/kaggle/working/shiftscan_final/state_passing/multisequence_validation/doc_006_seq_2048.pt
/kaggle/working/shiftscan_final/state_passing/multisequence_validation/doc_005_seq_2048.pt
/kaggle/working/shiftscan_final/state_passing/multisequence_validation/doc_000_seq_2048.pt
/kaggle/working/shiftscan_final/state_passing/multisequence_validation/doc_001_seq_2048.pt
/kaggle

In [218]:
import pandas as pd
from pathlib import Path

RESULT = Path(
    "/kaggle/working/shiftscan_final/integration/"
    "mamba_state_passing_comparison_1024.csv"
)

df = pd.read_csv(RESULT)

print("=" * 75)
print("LAYER-WISE TRAJECTORY ERROR DIAGNOSTIC")
print("=" * 75)

cols = [
    "layer",
    "trajectory_mae_vs_mamba",
    "trajectory_rmse_vs_mamba",
    "final_mae_vs_mamba",
    "final_max_abs_vs_mamba",
    "mamba_trajectory_max_abs",
    "custom_trajectory_max_abs",
    "dyadic2",
    "pot",
    "exact_zero",
    "approx_zero",
    "fallback",
]

available = [c for c in cols if c in df.columns]

print("\nLargest trajectory MAE:")
print(
    df.sort_values("trajectory_mae_vs_mamba", ascending=False)
      [available]
      .head(10)
      .to_string(index=False)
)

print("\nSmallest trajectory MAE:")
print(
    df.sort_values("trajectory_mae_vs_mamba")
      [available]
      .head(5)
      .to_string(index=False)
)

print("\nCorrelation between trajectory and final-state MAE:")
print(
    df[
        ["trajectory_mae_vs_mamba", "final_mae_vs_mamba"]
    ].corr().to_string()
)

print("\nSaved comparison CSV:", RESULT)

LAYER-WISE TRAJECTORY ERROR DIAGNOSTIC

Largest trajectory MAE:
 layer  trajectory_mae_vs_mamba  trajectory_rmse_vs_mamba  final_mae_vs_mamba  final_max_abs_vs_mamba  mamba_trajectory_max_abs  custom_trajectory_max_abs  dyadic2  pot  exact_zero  approx_zero  fallback
     7                 0.637614                  1.570430        2.666913e-07                0.000005                 82.473305                 107.632911        0   16          16           43        21
    12                 0.491975                  1.207784        2.447518e-07                0.000003                 89.240601                 102.357239        0    4          18           68         6
    15                 0.466552                  1.299353        2.615693e-07                0.000008                132.424500                 132.424500        2   12          24           40        18
    11                 0.425355                  1.404785        2.432239e-07                0.000003                361

In [219]:
# Recover the previous comparison cell's kernel launch code.
matches = []

for i, cell in enumerate(In):
    if (
        "shift_scan_state_passing_i32_kernel" in cell
        and "mamba_state_passing_comparison" in cell
    ):
        matches.append((i, cell))

if not matches:
    print("Could not find the previous comparison cell in notebook history.")
else:
    idx, source = matches[-1]
    lines = source.splitlines()

    print(f"Found comparison cell In[{idx}].")
    print("\nRelevant kernel-launch and output-comparison code:\n")

    keywords = (
        "shift_scan_state_passing_i32_kernel",
        "BLOCK_SIZE",
        "custom_int",
        "custom_out",
        "OUT",
        "FINAL",
        "mamba_out",
        "grid",
    )

    selected = set()
    for j, line in enumerate(lines):
        if any(k in line for k in keywords):
            selected.update(range(max(0, j - 3), min(len(lines), j + 5)))

    for j in sorted(selected):
        print(f"{j + 1:4d}: {lines[j]}")

Found comparison cell In[219].

Relevant kernel-launch and output-comparison code:

   3: 
   4: for i, cell in enumerate(In):
   5:     if (
   6:         "shift_scan_state_passing_i32_kernel" in cell
   7:         and "mamba_state_passing_comparison" in cell
   8:     ):
   9:         matches.append((i, cell))
  10: 
  18:     print("\nRelevant kernel-launch and output-comparison code:\n")
  19: 
  20:     keywords = (
  21:         "shift_scan_state_passing_i32_kernel",
  22:         "BLOCK_SIZE",
  23:         "custom_int",
  24:         "custom_out",
  25:         "OUT",
  26:         "FINAL",
  27:         "mamba_out",
  28:         "grid",
  29:     )
  30: 
  31:     selected = set()
  32:     for j, line in enumerate(lines):


In [220]:
# Exclude the current diagnostic cell from notebook history.
matches = []

for i, cell in enumerate(In[:-1]):
    if (
        "shift_scan_state_passing_i32_kernel" in cell
        and "mamba_state_passing_comparison_1024.csv" in cell
    ):
        matches.append((i, cell))

if not matches:
    print("Previous comparison cell not found.")
else:
    idx, source = matches[-1]
    print(f"Found original comparison cell: In[{idx}]")
    print("=" * 80)
    print(source)

Found original comparison cell: In[217]
import os
import gc
import json
import numpy as np
import pandas as pd
import torch
import triton
from pathlib import Path

import mamba_ssm.ops.triton.ssd_combined as ssd

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------
BASE = Path("/kaggle/working/shiftscan_final")
CAPTURE_DIR = BASE / "state_passing" / "long_context_validation"
OUT_DIR = BASE / "integration"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SCALE = int(SCALE)
assert SCALE == 2**20
assert torch.cuda.is_available()

kernel = shift_scan_state_passing_i32_kernel
BLOCK_SIZE = 128

# Test a representative 1024-token capture first.
CAPTURE_DIR = Path(
    "/kaggle/working/shiftscan_final/state_passing/multisequence_validation"
)
capture_path = CAPTURE_DIR / "doc_000_seq_1024.pt"

assert capture_path.exists(), f"Missing capture: {capture_path}"

data = torch.load(
    capture_path,
    map_location="cpu"

In [221]:
import numpy as np
import torch
import triton
from pathlib import Path
import mamba_ssm.ops.triton.ssd_combined as ssd

# ---------------------------------------------------------
# Configuration: preserve the original experiment
# ---------------------------------------------------------
SCALE = int(SCALE)
BLOCK_SIZE = 128

CAPTURE = Path(
    "/kaggle/working/shiftscan_final/state_passing/"
    "multisequence_validation/doc_000_seq_1024.pt"
)

data = torch.load(
    CAPTURE,
    map_location="cpu",
    weights_only=False
)

# Select layer 7
rec = next(
    r for r in data["records"]
    if int(r.get("layer", -1)) == 7
)

print("Selected layer:", rec.get("layer"))
print("Scale:", SCALE)

# ---------------------------------------------------------
# A. Rebuild exactly the same fixed-point inputs
# ---------------------------------------------------------
(
    N_int,
    decay,
    modes,
    exps,
    reference,
    counts,
    max_bound
) = build_reference_and_modes(rec)

chunks, heads, dim = N_int.shape
nblocks = triton.cdiv(dim, BLOCK_SIZE)

# ---------------------------------------------------------
# B. Original Mamba state passing
# ---------------------------------------------------------
states_fp = rec["states"].to(
    device="cuda",
    dtype=torch.float32
).contiguous()

decay_log = rec["decay_log"].to(
    device="cuda",
    dtype=torch.float32
).contiguous()

with torch.no_grad():
    mamba_out, mamba_final = ssd._state_passing_fwd(
        states_fp,
        decay_log,
        initial_states=None,
        seq_idx=None,
        chunk_size=None,
        out_dtype=torch.float32
    )

torch.cuda.synchronize()

mamba_np = mamba_out[0].cpu().numpy()
mamba_final_np = mamba_final[0].cpu().numpy()

# ---------------------------------------------------------
# C. Custom Shift-Scan kernel
# ---------------------------------------------------------
N_gpu = torch.as_tensor(
    N_int.astype(np.int32),
    device="cuda",
    dtype=torch.int32
).contiguous()

MODE_gpu = torch.as_tensor(
    modes.astype(np.int32),
    device="cuda",
    dtype=torch.int32
).contiguous()

EXP_gpu = torch.as_tensor(
    exps.astype(np.int32),
    device="cuda",
    dtype=torch.int32
).contiguous()

DECAY_gpu = torch.as_tensor(
    decay.astype(np.float64),
    device="cuda",
    dtype=torch.float64
).contiguous()

OUT_gpu = torch.empty(
    (1, chunks, heads, dim),
    device="cuda",
    dtype=torch.int32
)

FINAL_gpu = torch.empty(
    (1, heads, dim),
    device="cuda",
    dtype=torch.int32
)

OVERFLOW_gpu = torch.zeros(
    (1, heads, nblocks),
    device="cuda",
    dtype=torch.int32
)

shift_scan_state_passing_i32_kernel[
    (nblocks, 1, heads)
](
    N_gpu,
    MODE_gpu,
    EXP_gpu,
    DECAY_gpu,
    OUT_gpu,
    FINAL_gpu,
    OVERFLOW_gpu,
    NCHUNKS=chunks,
    NHEADS=heads,
    DIM=dim,
    NBLOCKS=nblocks,
    BLOCK_SIZE=BLOCK_SIZE
)

torch.cuda.synchronize()

custom_np = (
    OUT_gpu[0].cpu().numpy().astype(np.float64) / SCALE
)

custom_final_np = (
    FINAL_gpu[0].cpu().numpy().astype(np.float64) / SCALE
)

# ---------------------------------------------------------
# D. Per-chunk diagnostics
# ---------------------------------------------------------
print("\n" + "=" * 80)
print("LAYER 7: PER-CHUNK COMPARISON")
print("=" * 80)

rows = []

for c in range(chunks):
    diff = custom_np[c] - mamba_np[c]

    rows.append({
        "chunk": c,
        "mamba_ma": float(np.mean(np.abs(mamba_np[c]))),
        "custom_ma": float(np.mean(np.abs(custom_np[c]))),
        "error_mae": float(np.mean(np.abs(diff))),
        "error_rmse": float(np.sqrt(np.mean(diff ** 2))),
        "error_max": float(np.max(np.abs(diff))),
        "mamba_max": float(np.max(np.abs(mamba_np[c]))),
        "custom_max": float(np.max(np.abs(custom_np[c])))
    })

import pandas as pd
chunk_df = pd.DataFrame(rows)

display(chunk_df)

print("\nFinal state comparison:")
final_diff = custom_final_np - mamba_final_np

print("Final MAE:", np.mean(np.abs(final_diff)))
print("Final RMSE:", np.sqrt(np.mean(final_diff ** 2)))
print("Final maximum absolute error:", np.max(np.abs(final_diff)))

print("\nInteger reference mismatches:",
      np.count_nonzero(
          OUT_gpu[0].cpu().numpy().astype(np.int64) - reference
      ))

print("Overflow count:", int(OVERFLOW_gpu.sum().item()))

# ---------------------------------------------------------
# E. Check whether later decays suppress earlier errors
# ---------------------------------------------------------
print("\nChunk decay coefficients (exp of captured log-decay):")

decay_actual = torch.exp(decay_log).cpu().numpy()[0]

for c in range(chunks):
    print(
        f"Chunk {c}: "
        f"mean={decay_actual[:, c].mean():.8g}, "
        f"min={decay_actual[:, c].min():.8g}, "
        f"max={decay_actual[:, c].max():.8g}"
    )

# Save diagnostic
out_path = Path(
    "/kaggle/working/shiftscan_final/integration/"
    "layer7_per_chunk_diagnostic.csv"
)
out_path.parent.mkdir(parents=True, exist_ok=True)
chunk_df.to_csv(out_path, index=False)

print("\nSaved:", out_path)

Selected layer: 7
Scale: 1048576

LAYER 7: PER-CHUNK COMPARISON


,chunk,mamba_ma,custom_ma,error_mae,error_rmse,error_max,mamba_max,custom_max
0,0,0.000000,0.660961,0.660961,1.810011,77.764572,0.000000,77.764572
1,1,0.660961,0.675913,0.659914,1.568658,79.580476,77.764572,82.473304
2,2,0.675913,0.610475,0.617288,1.440442,56.956856,82.473305,73.313274
3,3,0.610475,0.598133,0.612295,1.432935,68.582832,73.313271,107.632911



Final state comparison:
Final MAE: 2.6669129917176554e-07
Final RMSE: 3.2515280548957375e-07
Final maximum absolute error: 4.76837158203125e-06

Integer reference mismatches: 0
Overflow count: 0

Chunk decay coefficients (exp of captured log-decay):
Chunk 0: mean=0.0033625325, min=0, max=0.064031467
Chunk 1: mean=0.014104903, min=0, max=0.29513067
Chunk 2: mean=0.017793136, min=0, max=0.39685813
Chunk 3: mean=0.013132829, min=0, max=0.30218551

Saved: /kaggle/working/shiftscan_final/integration/layer7_per_chunk_diagnostic.csv


In [222]:
import gc
import json
import numpy as np
import pandas as pd
import torch
import triton
from pathlib import Path
import mamba_ssm.ops.triton.ssd_combined as ssd

BASE = Path("/kaggle/working/shiftscan_final")
CAPTURE = (
    BASE / "state_passing" / "multisequence_validation"
    / "doc_000_seq_1024.pt"
)
OUT_DIR = BASE / "integration"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SCALE = int(SCALE)
BLOCK_SIZE = 128

data = torch.load(CAPTURE, map_location="cpu", weights_only=False)
results = []

for ri, rec in enumerate(data["records"]):
    print(f"Layer {ri+1}/{len(data['records'])}")

    N_int, decay, modes, exps, reference, counts, max_bound = (
        build_reference_and_modes(rec)
    )

    chunks, heads, dim = N_int.shape
    nblocks = triton.cdiv(dim, BLOCK_SIZE)

    states_fp = rec["states"].to(
        device="cuda", dtype=torch.float32
    ).contiguous()

    decay_log = rec["decay_log"].to(
        device="cuda", dtype=torch.float32
    ).contiguous()

    with torch.no_grad():
        mamba_out, mamba_final = ssd._state_passing_fwd(
            states_fp,
            decay_log,
            initial_states=None,
            seq_idx=None,
            chunk_size=None,
            out_dtype=torch.float32
        )

    torch.cuda.synchronize()

    mamba_out_np = mamba_out[0].cpu().numpy()
    mamba_final_np = mamba_final[0].cpu().numpy()

    # Correct boundary alignment:
    # Mamba out[c+1] is compared with custom post-chunk state[c].
    # The last custom chunk is compared with Mamba final state.
    mamba_aligned = np.concatenate(
        [mamba_out_np[1:], mamba_final_np[None, ...]],
        axis=0
    )

    N_gpu = torch.as_tensor(
        N_int.astype(np.int32),
        device="cuda", dtype=torch.int32
    ).contiguous()

    MODE_gpu = torch.as_tensor(
        modes.astype(np.int32),
        device="cuda", dtype=torch.int32
    ).contiguous()

    EXP_gpu = torch.as_tensor(
        exps.astype(np.int32),
        device="cuda", dtype=torch.int32
    ).contiguous()

    DECAY_gpu = torch.as_tensor(
        decay.astype(np.float64),
        device="cuda", dtype=torch.float64
    ).contiguous()

    OUT_gpu = torch.empty(
        (1, chunks, heads, dim),
        device="cuda", dtype=torch.int32
    )

    FINAL_gpu = torch.empty(
        (1, heads, dim),
        device="cuda", dtype=torch.int32
    )

    OVERFLOW_gpu = torch.zeros(
        (1, heads, nblocks),
        device="cuda", dtype=torch.int32
    )

    shift_scan_state_passing_i32_kernel[
        (nblocks, 1, heads)
    ](
        N_gpu, MODE_gpu, EXP_gpu, DECAY_gpu,
        OUT_gpu, FINAL_gpu, OVERFLOW_gpu,
        NCHUNKS=chunks,
        NHEADS=heads,
        DIM=dim,
        NBLOCKS=nblocks,
        BLOCK_SIZE=BLOCK_SIZE
    )

    torch.cuda.synchronize()

    custom_int = OUT_gpu[0].cpu().numpy().astype(np.int64)
    custom_float = custom_int.astype(np.float64) / SCALE

    custom_final_int = FINAL_gpu[0].cpu().numpy().astype(np.int64)

    # Integer reference check
    int_diff = np.abs(custom_int - reference)
    final_int_diff = np.abs(custom_final_int - reference[-1])

    # Correctly aligned floating-point comparison
    err = custom_float - mamba_aligned

    # Per-layer, per-chunk metrics
    for c in range(chunks):
        chunk_err = err[c]

        results.append({
            "layer": int(rec.get("layer", ri)),
            "chunk": c,
            "chunks": chunks,

            "aligned_mae": float(np.mean(np.abs(chunk_err))),
            "aligned_rmse": float(np.sqrt(np.mean(chunk_err ** 2))),
            "aligned_max_abs": float(np.max(np.abs(chunk_err))),

            "custom_mean_abs": float(np.mean(np.abs(custom_float[c]))),
            "mamba_mean_abs": float(np.mean(np.abs(mamba_aligned[c]))),

            "integer_mismatches": int(
                np.count_nonzero(int_diff[c])
            ),
            "final_integer_mismatches": int(
                np.count_nonzero(final_int_diff)
            ),
            "overflow_count": int(OVERFLOW_gpu.sum().item())
        })

    del states_fp, decay_log, mamba_out, mamba_final
    del N_gpu, MODE_gpu, EXP_gpu, DECAY_gpu
    del OUT_gpu, FINAL_gpu, OVERFLOW_gpu
    gc.collect()
    torch.cuda.empty_cache()

# ---------------------------------------------------------
# Save corrected comparison
# ---------------------------------------------------------
aligned_df = pd.DataFrame(results)

csv_path = OUT_DIR / "mamba_state_passing_aligned_1024.csv"
aligned_df.to_csv(csv_path, index=False)

summary = {
    "capture": CAPTURE.name,
    "layer_records": int(aligned_df.layer.nunique()),
    "chunk_comparisons": len(aligned_df),

    "mean_aligned_mae": float(aligned_df.aligned_mae.mean()),
    "mean_aligned_rmse": float(aligned_df.aligned_rmse.mean()),
    "worst_aligned_max_abs": float(aligned_df.aligned_max_abs.max()),

    "integer_mismatches": int(aligned_df.integer_mismatches.sum()),
    "final_integer_mismatches": int(
        aligned_df.final_integer_mismatches.sum()
    ),
    "overflow_count": int(aligned_df.overflow_count.sum())
}

json_path = OUT_DIR / "mamba_state_passing_aligned_1024.json"
with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 75)
print("SHIFT-SCAN VS MAMBA: CHUNK-ALIGNED COMPARISON")
print("=" * 75)
print(json.dumps(summary, indent=2))

print("\nPER-CHUNK AGGREGATE")
display(
    aligned_df.groupby("chunk")[
        ["aligned_mae", "aligned_rmse", "aligned_max_abs"]
    ].agg(["mean", "max"])
)

print("\nWORST 10 LAYER-CHUNK COMPARISONS")
display(
    aligned_df.sort_values("aligned_mae", ascending=False).head(10)
)

print("\nSaved:", csv_path)
print("Saved:", json_path)

Layer 1/24
Layer 2/24
Layer 3/24
Layer 4/24
Layer 5/24
Layer 6/24
Layer 7/24
Layer 8/24
Layer 9/24
Layer 10/24
Layer 11/24
Layer 12/24
Layer 13/24
Layer 14/24
Layer 15/24
Layer 16/24
Layer 17/24
Layer 18/24
Layer 19/24
Layer 20/24
Layer 21/24
Layer 22/24
Layer 23/24
Layer 24/24

SHIFT-SCAN VS MAMBA: CHUNK-ALIGNED COMPARISON
{
  "capture": "doc_000_seq_1024.pt",
  "layer_records": 24,
  "chunk_comparisons": 96,
  "mean_aligned_mae": 7.757806816543309e-07,
  "mean_aligned_rmse": 3.3692553282481436e-06,
  "worst_aligned_max_abs": 0.001359701156616211,
  "integer_mismatches": 0,
  "final_integer_mismatches": 0,
  "overflow_count": 0
}

PER-CHUNK AGGREGATE


aligned_mae                aligned_rmse               aligned_max_abs  \
               mean           max          mean           max            mean   
chunk                                                                           
0      2.378300e-07  2.387628e-07  2.757339e-07  2.768288e-07    4.768372e-07   
1      8.521171e-07  5.148630e-06  3.640210e-06  1.743613e-05    2.750875e-04   
2      1.018946e-06  4.540566e-06  4.877602e-06  1.610854e-05    3.926476e-04   
3      9.942300e-07  4.499240e-06  4.683475e-06  1.614892e-05    3.481756e-04   

                     
                max  
chunk                
0      4.768372e-07  
1      9.498596e-04  
2      1.275867e-03  
3      1.359701e-03


WORST 10 LAYER-CHUNK COMPARISONS


,layer,chunk,chunks,aligned_mae,aligned_rmse,aligned_max_abs,custom_mean_abs,mamba_mean_abs,integer_mismatches,final_integer_mismatches,overflow_count
1,0,1,4,0.000005,0.000017,0.000950,0.058739,0.058736,0,0,0
2,0,2,4,0.000005,0.000016,0.000972,0.059756,0.059754,0,0,0
3,0,3,4,0.000004,0.000016,0.000859,0.059053,0.059052,0,0,0
27,6,3,4,0.000003,0.000016,0.000974,0.214767,0.214768,0,0,0
86,21,2,4,0.000003,0.000015,0.001276,0.009038,0.009037,0,0,0
85,21,1,4,0.000003,0.000016,0.000851,0.008795,0.008794,0,0,0
87,21,3,4,0.000003,0.000011,0.000906,0.008455,0.008454,0,0,0
41,10,1,4,0.000002,0.000010,0.000551,0.231156,0.231155,0,0,0
22,5,2,4,0.000002,0.000013,0.000921,0.273382,0.273383,0,0,0
79,19,3,4,0.000002,0.000014,0.001360,0.020501,0.020500,0,0,0



Saved: /kaggle/working/shiftscan_final/integration/mamba_state_passing_aligned_1024.csv
Saved: /kaggle/working/shiftscan_final/integration/mamba_state_passing_aligned_1024.json


## Final integration test: all 20 held-out captures

This consolidated test compares the custom INT32 Triton state-passing kernel with (1) the notebook's integer reference recurrence and (2) the original Mamba-2 state-passing implementation. It checks 10 held-out documents at sequence lengths 1,024 and 2,048, across 24 layers per capture (480 layer records total).

The floating-point comparison uses the verified chunk-boundary alignment: custom post-chunk state c is compared with Mamba output c+1, and the final custom state is compared with Mamba's returned final state. Integer-reference mismatches and INT32 overflow are hard failures. Floating-point approximation errors are reported separately.

**Prerequisites:** run the earlier setup and kernel-definition cells in the same CUDA/Triton notebook session. This cell has not been executed as part of preparing this clean release.

In [ ]:
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import triton
import mamba_ssm.ops.triton.ssd_combined as ssd

BASE = Path("/kaggle/working/shiftscan_final")
CAPTURE_DIR = BASE / "state_passing" / "multisequence_validation"
OUT_DIR = BASE / "integration" / "all_heldout_capture_validation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SCALE = int(SCALE)
BLOCK_SIZE = 128
assert SCALE == 2**20, f"Expected F20 scale (2**20), got {SCALE}"
assert torch.cuda.is_available(), "This test requires the CUDA/Triton runtime."
assert callable(globals().get("build_reference_and_modes")), "Run the reference-builder setup cell first."
assert callable(globals().get("shift_scan_state_passing_i32_kernel")), "Run the INT32 Triton kernel-definition cell first."

expected = {f"doc_{doc:03d}_seq_{seq}.pt" for doc in range(10) for seq in (1024, 2048)}
paths = sorted(CAPTURE_DIR.glob("doc_*_seq_*.pt"), key=lambda p: (int(p.stem.split("_")[1]), int(p.stem.split("_")[3])))
found = {p.name for p in paths}
missing = sorted(expected - found)
if missing:
    raise FileNotFoundError(f"Missing held-out captures: {missing}")
paths = [p for p in paths if p.name in expected]
assert len(paths) == 20, f"Expected 20 held-out captures, found {len(paths)}"

layer_rows, capture_rows = [], []
for capture_i, path in enumerate(paths, 1):
    print(f"[{capture_i}/20] {path.name}")
    payload = torch.load(path, map_location="cpu", weights_only=False)
    records = payload["records"]
    assert len(records) == 24, f"{path.name}: expected 24 layers, found {len(records)}"
    mismatch_count = final_mismatch_count = overflow_count = 0
    capture_mae, capture_rmse, capture_worst = [], [], 0.0
    doc_id, seq_len = int(path.stem.split("_")[1]), int(path.stem.split("_")[-1])

    for record_i, rec in enumerate(records):
        N_int, decay, modes, exps, reference, counts, max_bound = build_reference_and_modes(rec)
        chunks, heads, dim = N_int.shape
        assert (heads, dim) == (24, 8192), f"{path.name}, layer {record_i}: unexpected state shape {N_int.shape}"

        states = rec["states"].to(device="cuda", dtype=torch.float32).contiguous()
        decay_log = rec["decay_log"].to(device="cuda", dtype=torch.float32).contiguous()
        with torch.no_grad():
            mamba_out, mamba_final = ssd._state_passing_fwd(
                states, decay_log, initial_states=None, seq_idx=None,
                chunk_size=None, out_dtype=torch.float32
            )
        torch.cuda.synchronize()
        # Custom post-chunk state c aligns with Mamba output c+1; append final state.
        mamba_aligned = torch.cat((mamba_out[:, 1:], mamba_final.unsqueeze(1)), dim=1)[0].cpu().numpy()

        nblocks = triton.cdiv(dim, BLOCK_SIZE)
        n_gpu = torch.as_tensor(N_int.astype(np.int32), device="cuda", dtype=torch.int32).contiguous()
        mode_gpu = torch.as_tensor(modes.astype(np.int32), device="cuda", dtype=torch.int32).contiguous()
        exp_gpu = torch.as_tensor(exps.astype(np.int32), device="cuda", dtype=torch.int32).contiguous()
        decay_gpu = torch.as_tensor(decay.astype(np.float64), device="cuda", dtype=torch.float64).contiguous()
        out_gpu = torch.empty((1, chunks, heads, dim), device="cuda", dtype=torch.int32)
        final_gpu = torch.empty((1, heads, dim), device="cuda", dtype=torch.int32)
        overflow_gpu = torch.zeros((1, heads, nblocks), device="cuda", dtype=torch.int32)

        shift_scan_state_passing_i32_kernel[(nblocks, 1, heads)](
            n_gpu, mode_gpu, exp_gpu, decay_gpu, out_gpu, final_gpu, overflow_gpu,
            NCHUNKS=chunks, NHEADS=heads, DIM=dim, NBLOCKS=nblocks,
            BLOCK_SIZE=BLOCK_SIZE
        )
        torch.cuda.synchronize()

        custom_int = out_gpu[0].cpu().numpy().astype(np.int64)
        custom_float = custom_int.astype(np.float64) / SCALE
        custom_final_int = final_gpu[0].cpu().numpy().astype(np.int64)
        reference_int = np.asarray(reference, dtype=np.int64)
        if reference_int.shape != custom_int.shape:
            raise AssertionError(f"Reference shape mismatch in {path.name}: {reference_int.shape} vs {custom_int.shape}")

        integer_diff = np.abs(custom_int - reference_int)
        final_diff = np.abs(custom_final_int - reference_int[-1])
        overflows = int(overflow_gpu.sum().item())
        err = custom_float - mamba_aligned
        mae = float(np.mean(np.abs(err)))
        rmse = float(np.sqrt(np.mean(err ** 2)))
        max_abs = float(np.max(np.abs(err)))
        layer = int(rec.get("layer", record_i))

        mismatch_count += int(np.count_nonzero(integer_diff))
        final_mismatch_count += int(np.count_nonzero(final_diff))
        overflow_count += overflows
        capture_mae.append(mae)
        capture_rmse.append(rmse)
        capture_worst = max(capture_worst, max_abs)
        for chunk_i in range(chunks):
            chunk_err = err[chunk_i]
            layer_rows.append({
                "capture": path.name, "doc_id": doc_id, "seq_len": seq_len,
                "layer": layer, "chunk": chunk_i,
                "aligned_mae": float(np.mean(np.abs(chunk_err))),
                "aligned_rmse": float(np.sqrt(np.mean(chunk_err ** 2))),
                "aligned_max_abs": float(np.max(np.abs(chunk_err))),
                "integer_mismatches": int(np.count_nonzero(integer_diff[chunk_i])),
                "final_integer_mismatches": int(np.count_nonzero(final_diff)) if chunk_i == chunks - 1 else 0,
                "overflow_count": overflows if chunk_i == 0 else 0,
            })

        del states, decay_log, mamba_out, mamba_final, mamba_aligned
        del n_gpu, mode_gpu, exp_gpu, decay_gpu, out_gpu, final_gpu, overflow_gpu
        gc.collect()
        torch.cuda.empty_cache()

    capture_rows.append({
        "capture": path.name, "doc_id": doc_id, "seq_len": seq_len,
        "layer_records": len(records), "mean_layer_aligned_mae": float(np.mean(capture_mae)),
        "mean_layer_aligned_rmse": float(np.mean(capture_rmse)),
        "worst_aligned_max_abs": capture_worst,
        "integer_mismatches": mismatch_count,
        "final_integer_mismatches": final_mismatch_count,
        "overflow_count": overflow_count,
    })
    del payload
    gc.collect()

layer_df = pd.DataFrame(layer_rows)
capture_df = pd.DataFrame(capture_rows)
layer_csv = OUT_DIR / "all_capture_layer_chunk_results.csv"
capture_csv = OUT_DIR / "all_capture_summary.csv"
summary_json = OUT_DIR / "all_capture_validation_summary.json"
layer_df.to_csv(layer_csv, index=False)
capture_df.to_csv(capture_csv, index=False)

summary = {
    "experiment": "all_heldout_capture_mamba_state_passing_integration",
    "capture_count": int(capture_df.capture.nunique()),
    "document_count": int(capture_df.doc_id.nunique()),
    "sequence_lengths": sorted(int(x) for x in capture_df.seq_len.unique()),
    "layer_records": int(capture_df.layer_records.sum()),
    "chunk_comparisons": int(len(layer_df)),
    "mean_aligned_mae": float(layer_df.aligned_mae.mean()),
    "mean_aligned_rmse": float(layer_df.aligned_rmse.mean()),
    "worst_aligned_max_abs": float(layer_df.aligned_max_abs.max()),
    "integer_mismatches": int(layer_df.integer_mismatches.sum()),
    "final_integer_mismatches": int(layer_df.final_integer_mismatches.sum()),
    "overflow_count": int(layer_df.overflow_count.sum()),
    "integer_reference_exact": bool(layer_df.integer_mismatches.sum() == 0 and layer_df.final_integer_mismatches.sum() == 0),
    "no_overflow": bool(layer_df.overflow_count.sum() == 0),
    "alignment": "custom post-chunk state c vs Mamba output c+1; final custom state vs Mamba final state",
    "scale": SCALE, "block_size": BLOCK_SIZE,
}
summary_json.write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print("Saved:", layer_csv, capture_csv, summary_json)

assert summary["capture_count"] == 20
assert summary["layer_records"] == 480
assert summary["integer_reference_exact"], "Integer recurrence mismatch detected."
assert summary["no_overflow"], "INT32 overflow detected."
assert np.isfinite(layer_df[["aligned_mae", "aligned_rmse", "aligned_max_abs"]].to_numpy()).all()
print("PASS: 20 captures, 480 layer records, exact integer-reference agreement, no INT32 overflow.")